# 제조데이터 분석 통합 제출 노트북

데이터 확인 → EDA → Analysis → 모델링과 오류 분석 → 현재 선택 방법 재현 순서로 실행한다. `제출` 폴더 전체를 유지한 채 정규 CPython 3.13 커널에서 **Restart Kernel and Run All Cells**를 선택한다. 네 상세 원고와 압축본에 사용된 Python 코드 및 직접 import 의존 코드만 포함했다.

원문 코드는 아래 셀에 모두 들어 있다. `%%submission_source`는 셀을 원래 모듈로 등록하여 같은 이름의 함수·변수 충돌과 `__file__` 경로 문제를 처리한다. 실행은 뒤의 대목차별 셀에서 순서대로 수행한다. 문서 생성·보존 검사 코드는 원고 작성 근거로 포함했으며 실행 중 원고를 다시 편집하지 않는다.

데이터·EDA·Analysis는 실행 사본에서 계산한다. 과거 모델 탐색은 저장된 고정 결과의 해시·모델·지표를 검증해 재사용하며, 현재 선택 방법은 원자료에서 6회 재학습해 예측을 대조한다. 과거 Optuna 재탐색이나 이미 본 7 ~ 8월을 새 독립 시험으로 해석하지 않는다. 원고의 결과 설명과 이번 실행에서 계산한 결과를 구분해 표시한다.

목차: 0. 실행 환경 / 1. 데이터 확인 / 2. EDA / 3. Analysis / 4. 모델링 / 5. 실행 확인. 결과와 로그는 매번 새로운 `실행결과/실행시각/` 폴더에 저장한다.

## 0. 실행 환경과 코드 등록

`requirements.txt`의 패키지를 준비한다. 입력 해시를 먼저 확인하고, 원고·참조 결과의 복사본에서 계산한다. 오류가 나면 해당 셀에서 중단하며 완료로 표시하지 않는다.

In [1]:
from pathlib import Path
from datetime import datetime
import contextlib, hashlib, importlib, importlib.metadata, io, json, os, shutil, sys, time, types
import numpy as np
import pandas as pd
from IPython.core.magic import register_cell_magic
from IPython.display import display, Markdown, Image
assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled(), "정규 CPython 3.13 커널을 선택하세요."
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MPLBACKEND", "Agg")
PACKAGE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "선별_파일목록.json").is_file())
inventory = json.loads((PACKAGE / "선별_파일목록.json").read_text(encoding="utf-8"))
def digest(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert all(digest(PACKAGE/r['path']) == r['sha256'] for r in inventory['files']), "제출 입력이 변경되었습니다."
for stage in ["EDA", "Analysis", "Modeling"]:
    sys.path.insert(0, str(PACKAGE / stage / "scripts"))
RUN = PACKAGE / "실행결과" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
WORK = RUN / "재계산"
WORK.mkdir(parents=True)
for stage in ["data", "EDA", "Analysis", "Modeling"]:
    shutil.copytree(PACKAGE/stage, WORK/stage, ignore=shutil.ignore_patterns('__pycache__'))
shutil.copy2(PACKAGE/'README.md',WORK/'README.md')
MODULES = {}
RECORDS = []
@register_cell_magic
def submission_source(line, cell):
    """원문을 그대로 표시하고 모듈로 등록한다. CLI main은 뒤의 실행 셀에서 호출한다."""
    relative = line.strip()
    path = PACKAGE / relative
    assert cell.rstrip() == path.read_text(encoding='utf-8-sig').rstrip(), relative
    name = path.stem
    module = sys.modules.get(name) or types.ModuleType(name)
    sys.modules[name] = module
    module.__file__ = str(path)
    module.__name__ = name
    exec(compile(cell, str(path), 'exec'), module.__dict__)
    MODULES[name] = module
    display(Markdown(f"코드 등록 완료: `{relative}`"))

@contextlib.contextmanager
def computation_paths():
    """모든 모듈의 경로를 같은 실행 사본으로 옮겨 원본·대조 결과를 보존한다."""
    previous=[]
    for m in list(sys.modules.values()):
        path=getattr(m,'__file__',None)
        if not path or not str(path).startswith(str(PACKAGE)): continue
        for k,v in list(vars(m).items()):
            if isinstance(v,Path) and v.is_absolute() and v.is_relative_to(PACKAGE):
                previous.append((m,k,v));setattr(m,k,WORK/v.relative_to(PACKAGE))
    try:yield
    finally:
        for m,k,v in previous:setattr(m,k,v)

def execute_step(name, callback):
    started=time.monotonic();stream=io.StringIO()
    try:
        with contextlib.redirect_stdout(stream):callback()
    except BaseException:
        (RUN/(name+'.log')).write_text(stream.getvalue(),encoding='utf-8')
        raise
    (RUN/(name+'.log')).write_text(stream.getvalue(),encoding='utf-8')
    row={'step':name,'status':'passed','seconds':round(time.monotonic()-started,3)}
    RECORDS.append(row)
    display(Markdown(f"### 실행 결과: {name}\n\n정상 종료. 소요 시간 {row['seconds']:.1f}초. 상세 출력: `{(RUN/(name+'.log')).relative_to(PACKAGE).as_posix()}`."))

def cli(name,*args):
    old=sys.argv
    try:sys.argv=[str(PACKAGE/'Modeling/scripts'/f'{name}.py'),*args];MODULES[name].main()
    finally:sys.argv=old

def show_table(relative, limit=16):
    p=WORK/relative
    data=pd.read_csv(p,encoding='utf-8-sig')
    view=data.head(limit)
    display(Markdown(f"### 실행 결과표: `{relative}`\n\n전체 {len(data):,}행. 아래는 {len(view):,}행이며, 전체 표는 실행결과 폴더에 저장합니다.\n\n"+view.to_markdown(index=False)))

display(Markdown(f"정규 Python {sys.version.split()[0]}. 선별 코드 {inventory['code_count']}개와 입력 {len(inventory['files'])}개 해시 확인. 이번 출력 폴더: `{RUN.relative_to(PACKAGE).as_posix()}`."))

정규 Python 3.13.1. 선별 코드 97개와 입력 994개 해시 확인. 이번 출력 폴더: `실행결과/20261006_100859_563535`.

## EDA 원고에 사용한 전체 코드

아래 셀은 코드 정의를 등록한다. 계산 순서와 이번 실행 결과는 뒤의 1 ~ 5장에 정리한다.

### `EDA/scripts/data_overview.py`

원고 연결: EDA\01_데이터_0.5페이지용_압축본.md; EDA\01_데이터_원고.md; EDA\01_데이터_원고.md; EDA\01_데이터_원고.md; EDA\01_데이터_원고.md; EDA\01_데이터_원고.md; EDA\02_EDA_원고.md.

코드 설명: Data manuscript reproduction and section 2.1 hourly summaries.

Adapted from the moved eda_analysis.py prepare/narrative calculations.
Does not train a model or change the original CSV. No plots in data chapter.

주요 함수: `main()`.

In [2]:
%%submission_source EDA/scripts/data_overview.py
"""Data manuscript reproduction and section 2.1 hourly summaries.

Adapted from the moved eda_analysis.py prepare/narrative calculations.
Does not train a model or change the original CSV. No plots in data chapter.
"""
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT=Path(__file__).resolve().parents[2]
OUT=ROOT/'EDA/tables'
SOURCE=ROOT/'data/origin/okm_augumented_2021.csv'
SHA='8f7af2e49366c93e1d6f5fdef4b5e350066c1792ac463c2c2886e370f4674830'
SLOTS=['15분','30분','45분','60분']

def main():
    OUT.mkdir(parents=True,exist_ok=True)
    assert hashlib.sha256(SOURCE.read_bytes()).hexdigest()==SHA
    raw=pd.read_csv(SOURCE,encoding='utf-8-sig')
    scope=raw.loc[raw['날짜'].lt(20210901)].copy()
    valid=scope.loc[scope['시간'].between(0,23)].copy()
    assert raw.shape==(6168,18) and len(scope)==5832 and len(valid)==5784
    assert raw['날짜'].nunique()==257 and valid['날짜'].nunique()==241
    assert len(raw.loc[raw['날짜'].ge(20210901)])==336
    assert valid.groupby('날짜').size().eq(24).all()
    np.testing.assert_array_equal(raw['평균'],np.floor(raw[SLOTS].mean(axis=1)+.5))
    ratio=raw['생산량']/raw[SLOTS].sum(axis=1)
    available=raw['공장인원'].notna()
    # The existing variable audit reports max absolute storage error ~4.94e-9
    # (data/origin/데이터_변수표.md, section 6). Do not assume fixed 9 decimals.
    ratio_error=float((raw.loc[available,'공장인원']-ratio.loc[available]).abs().max())
    np.testing.assert_allclose(raw.loc[available,'공장인원'],ratio.loc[available],rtol=0,atol=5e-9)
    missing=raw.isna().sum().rename_axis('variable').reset_index(name='missing_n')
    missing.to_csv(OUT/'data_missing.csv',index=False,encoding='utf-8-sig')
    assert missing.missing_n.sum()==21
    assert raw['풍속'].isna().sum()==3 and raw['강수량'].isna().sum()==1 and (~available).sum()==17
    raw.loc[raw.isna().any(axis=1),['날짜','시간','풍속','강수량','공장인원']].to_csv(
        OUT/'data_missing_records.csv',index=False,encoding='utf-8-sig')
    outliers=[]
    for variable in ['생산량','강수량']:
        v=scope[variable].dropna();q1,q3=v.quantile([.25,.75]);iqr=q3-q1
        outliers.append(dict(variable=variable,n=len(v),zero_n=int(v.eq(0).sum()),
            zero_share=float(v.eq(0).mean()),q1=float(q1),q3=float(q3),iqr=float(iqr),
            outside_iqr_n=int((v.lt(q1-1.5*iqr)|v.gt(q3+1.5*iqr)).sum()),maximum=float(v.max())))
    pd.DataFrame(outliers).to_csv(OUT/'data_distribution.csv',index=False,encoding='utf-8-sig')
    assert outliers[0]['outside_iqr_n']==533 and outliers[0]['maximum']==9830
    assert outliers[1]['outside_iqr_n']==1442 and outliers[1]['iqr']==0
    invalid=scope.loc[~scope['시간'].between(0,23)]
    assert len(invalid)==48 and set(invalid['날짜'])=={20210713,20210715}
    assert invalid['시간'].min()==70 and invalid['시간'].max()==188
    invalid[['날짜','시간']].to_csv(OUT/'data_invalid_times.csv',index=False,encoding='utf-8-sig')
    valid=valid.sort_values(['날짜','시간'])
    profiles=valid.groupby('날짜')[SLOTS].apply(lambda x:tuple(x.to_numpy().ravel()))
    multiplicity=profiles.value_counts()
    assert len(multiplicity)==126 and int(multiplicity.loc[multiplicity.gt(1)].sum())==160
    prod_unique=valid.groupby('날짜')['생산량'].nunique()
    assert prod_unique.gt(1).sum()==181 and not raw.duplicated().any()
    assert raw['생산량'].eq(0).sum()==2657
    assert (raw['생산량'].eq(0)&raw['평균'].gt(0)).sum()==2640
    assert raw[SLOTS].eq(0).all(axis=1).sum()==17
    dates=pd.to_datetime(valid['날짜'].astype(str),format='%Y%m%d')
    valid['weekend']=dates.dt.dayofweek.ge(5)
    summaries=[]
    for group,flag in [('weekday',False),('weekend',True)]:
        for hour,part in valid.loc[valid.weekend.eq(flag)].groupby('시간'):
            summaries.append(dict(population=group,hour=int(hour),n=len(part),
                production_mean=float(part['생산량'].mean()),power_mean=float(part['평균'].mean())))
    pd.DataFrame(summaries).to_csv(OUT/'hourly_overview.csv',index=False,encoding='utf-8-sig')
    # Check availability only; actual model choices remain in archived model evidence.
    series=pd.Series(valid['평균'].to_numpy(),index=dates+pd.to_timedelta(valid['시간'],unit='h'))
    grid=series.reindex(pd.date_range('2021-01-01','2021-08-31 23:00',freq='h'))
    eligible=grid.shift(1).rolling(24,min_periods=24).count().eq(24)
    for lag in (1,2,3,24,168):eligible &= grid.shift(lag).notna()
    assert int(eligible.reindex(series.index).sum())==5520
    facts=dict(status='passed',source_sha256=SHA,raw_rows=6168,columns=18,
        scope_rows=5832,valid_hours=5784,valid_days=241,missing_cells=21,
        headcount_formula_checked_rows=int(available.sum()),headcount_formula_max_abs_error=ratio_error,unique_96slot_profiles=126,
        dates_in_repeated_profiles=160,multiple_production_value_dates=181,
        eligible_history_hours=5520,model_training_performed=False)
    (OUT/'data_overview_manifest.json').write_text(json.dumps(facts,ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps(facts,ensure_ascii=False))

if __name__=='__main__':main()


코드 등록 완료: `EDA/scripts/data_overview.py`

### `EDA/scripts/eda_daily_pattern.py`

원고 연결: EDA\01_데이터_0.5페이지용_압축본.md; EDA\01_데이터_원고.md; EDA\02_EDA_3페이지용_압축본.md; EDA\02_EDA_원고.md.

코드 설명: Restart EDA section 2.1 only: weekday/weekend daily production/power lines.

Reuse hourly_overview; audit its 48 selected means against the supplied CSV.
Generate the two-panel section 2.1 PNG; image review is separate.

주요 함수: `main()`.

In [3]:
%%submission_source EDA/scripts/eda_daily_pattern.py
"""Restart EDA section 2.1 only: weekday/weekend daily production/power lines.

Reuse hourly_overview; audit its 48 selected means against the supplied CSV.
Generate the two-panel section 2.1 PNG; image review is separate.
"""
# %% Source and scope
from pathlib import Path
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
import eda_figures as style

ROOT = Path(__file__).resolve().parents[2]
REPORT = ROOT/'EDA'
SOURCE = ROOT/'data/origin/okm_augumented_2021.csv'
SHA = '8f7af2e49366c93e1d6f5fdef4b5e350066c1792ac463c2c2886e370f4674830'


def main():
    # %% Reuse existing summaries; independently check selected means and counts
    assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == SHA
    slots = ['15분','30분','45분','60분']
    raw = pd.read_csv(SOURCE,encoding='utf-8-sig',usecols=['날짜','시간','생산량','평균',*slots])
    valid = raw.loc[raw['날짜'].lt(20210901) & raw['시간'].between(0,23)].copy()
    assert len(valid) == 5784 and valid['날짜'].nunique() == 241
    assert valid.groupby('날짜').size().eq(24).all()
    np.testing.assert_array_equal(valid['평균'],np.floor(valid[slots].mean(axis=1)+.5))
    valid['weekend'] = pd.to_datetime(valid['날짜'].astype(str),format='%Y%m%d').dt.dayofweek.ge(5)
    table = pd.read_csv(style.TABLES/'hourly_overview.csv',encoding='utf-8-sig',float_precision='round_trip')
    selected = table.loc[table.population.isin(['weekday','weekend']),
                         ['population','hour','n','production_mean','power_mean']].copy()
    assert len(selected) == 48
    facts = {}
    for population,weekend,days in [('weekday',False,171),('weekend',True,70)]:
        part = selected.loc[selected.population.eq(population)].sort_values('hour')
        expected = valid.loc[valid.weekend.eq(weekend)].groupby('시간').agg(
            n=('평균','size'),production_mean=('생산량','mean'),power_mean=('평균','mean'))
        np.testing.assert_array_equal(part.hour,np.arange(24))
        np.testing.assert_array_equal(part.n,np.full(24,days))
        np.testing.assert_allclose(part[['n','production_mean','power_mean']],expected,
                                   rtol=1e-12,atol=1e-12)
        facts[population] = dict(days=days,
                                 production_peak_hour=int(part.loc[part.production_mean.idxmax(),'hour']),
                                 power_peak_hour=int(part.loc[part.power_mean.idxmax(),'hour']))
    weekday=selected.loc[selected.population.eq('weekday')].set_index('hour')
    for metric in ['production_mean','power_mean']:
        assert weekday.loc[12,metric] < weekday.loc[11,metric]
        assert weekday.loc[13,metric] > weekday.loc[12,metric]
    weekend_19=valid.loc[valid.weekend & valid['시간'].eq(19)]
    assert weekend_19['생산량'].eq(0).all()
    assert round(float(weekend_19['평균'].mean()),2)==41.11
    selected.to_csv(style.TABLES/'daily_pattern_restart.csv',index=False,encoding='utf-8-sig')
    # %% One shared time axis; two independent measurement scales
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        font = style.setup()
        fig,axes = style.plt.subplots(2,1,figsize=(11.5,7.2),sharex=True)
        for ax,metric,title,label in zip(axes,['production_mean','power_mean'],
                                       ['시간대별 평균 생산량','시간대별 평균 전력'],
                                       ['평균 생산량','평균 전력']):
            for population,color,ls,name in [('weekday',style.BLUE,'-','평일 (171일)'),
                                            ('weekend',style.ORANGE,'--','주말 (70일)')]:
                part = selected.loc[selected.population.eq(population)].sort_values('hour')
                ax.plot(part.hour,part[metric],color=color,linestyle=ls,linewidth=2.4,
                        marker='o',markersize=3.5,label=name)
            ax.set(title=title,ylabel=label,ylim=(0,None))
            ax.grid(axis='y',alpha=.18,linewidth=.7)
            ax.set_axisbelow(True)
            ax.legend(loc='upper left',fontsize=10,ncol=2,frameon=False)
        axes[1].set(xticks=np.arange(24),xlim=(-.25,23.25),xlabel='시간대 (시)')
        style.finish(fig,'restart_01_daily_pattern_clean','평일과 주말의 생산량·전력 하루 패턴',
                     '2021년 1~8월 · 각 시각의 평일 171일 / 주말 70일 평균 · 공휴일 포함\n'
                     '전력은 CSV 평균 열(한 시간의 네 전력값을 평균·반올림한 값) · 두 그래프의 세로축 단위와 범위는 다름')
    important = [str(w.message) for w in caught if 'Glyph' in str(w.message) or 'layout' in str(w.message).lower()]
    assert not important,important
    image_file=style.FIGURES/'restart_01_daily_pattern_clean.png'
    assert image_file.stat().st_size > 10000
    manifest=dict(status='passed',scope='Restarted EDA first section only',source_sha256=SHA,
                  checked_rows=48,checked_numeric_columns=3,groups=facts,
                  figure=image_file.name,bytes=image_file.stat().st_size,
                  figure_sha256=hashlib.sha256(image_file.read_bytes()).hexdigest(),font=font,
                  warnings=important,image_analysis_performed=False,visual_review=False,
                  script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())
    (style.TABLES/'daily_pattern_restart_manifest.json').write_text(
        json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps(dict(status='passed',checked_rows=48,figure=image_file.name,
                         font=font,warnings=important,image_analysis_performed=False)))


if __name__ == '__main__':
    main()


코드 등록 완료: `EDA/scripts/eda_daily_pattern.py`

### `EDA/scripts/eda_daily_repetition.py`

원고 연결: EDA\01_데이터_원고.md; EDA\02_EDA_3페이지용_압축본.md; EDA\02_EDA_원고.md.

코드 설명: EDA 2.2 analysis: day-by-day hourly mean power, no manuscript authoring.

Question: is the aggregate weekday/weekend daily shape present on actual dates?
Previous evidence: 2.1 shows 11->12 down and 12->13 up on weekday means.
Existing E018 concerns slot boundaries; its counts are not hourly-mean counts.

주요 함수: `json_save()`, `csv_save()`, `main()`.

In [4]:
%%submission_source EDA/scripts/eda_daily_repetition.py
"""EDA 2.2 analysis: day-by-day hourly mean power, no manuscript authoring.

Question: is the aggregate weekday/weekend daily shape present on actual dates?
Previous evidence: 2.1 shows 11->12 down and 12->13 up on weekday means.
Existing E018 concerns slot boundaries; its counts are not hourly-mean counts.
"""
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT=Path(__file__).resolve().parents[2]
SOURCE=ROOT/'data/origin/okm_augumented_2021.csv'
SHA='8f7af2e49366c93e1d6f5fdef4b5e350066c1792ac463c2c2886e370f4674830'
TAB=ROOT/'EDA/tables/daily_repetition'
FIG=ROOT/'EDA/figures/daily_repetition'
MANUSCRIPT=ROOT/'EDA/02_EDA_원고.md'

def json_save(name,obj):
    (TAB/(name+'.json')).write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8')

def csv_save(name,frame):
    frame.to_csv(TAB/(name+'.csv'),index=False,encoding='utf-8-sig')

def main():
    TAB.mkdir(parents=True,exist_ok=True);FIG.mkdir(parents=True,exist_ok=True)
    before=hashlib.sha256(MANUSCRIPT.read_bytes()).hexdigest()
    json_save('plan',{
        'previous_evidence':'2.1 weekday/weekend hourly means: weekday 11->12 decreases and 12->13 increases. E018 counts different, 15-minute boundary values.',
        'question':'How often does the same shape occur on individual dates, and how do level and spread change across dates?',
        'scope':'Jan-Aug 2021, valid hours; weekday includes public holidays. No training/test partition.',
        'target':'CSV 平均, hourly four-slot arithmetic mean rounded half up, same target as section 2.1.',
        'criteria':'For each date: M12<M11 and M13>M12. Strict comparisons; ties explicitly distinguished. No peak threshold or inference test.',
        'outputs':'Per-date changes and production context; hourly distributions and monthly summaries. Plotting is separate.',
        'follow_up_rule':'If a shape is not present every day, inspect month/day-type composition and ties. If exact daily profiles repeat, compare one-vote-per-profile summaries descriptively.',
        'manuscript_edited':False})
    assert hashlib.sha256(SOURCE.read_bytes()).hexdigest()==SHA
    cols=['15분','30분','45분','60분']
    raw=pd.read_csv(SOURCE,encoding='utf-8-sig')
    period=raw.loc[raw['날짜'].between(20210101,20210831)]
    v=period.loc[period['시간'].between(0,23)].copy()
    v['date']=pd.to_datetime(v['날짜'].astype(str),format='%Y%m%d')
    v=v.sort_values(['date','시간'])
    assert len(v)==5784 and v.date.nunique()==241
    assert v.groupby('date').size().eq(24).all() and not v.duplicated(['date','시간']).any()
    np.testing.assert_array_equal(v['평균'],np.floor(v[cols].mean(axis=1)+.5))
    matrix=v.pivot(index='date',columns='시간',values='평균').sort_index()
    assert list(matrix.columns)==list(range(24))
    days=pd.DataFrame(index=matrix.index)
    days['group']=np.where(days.index.dayofweek<5,'weekday','weekend')
    days['month']=days.index.month
    days['mean_power']=matrix.mean(axis=1)
    days['M11']=matrix[11];days['M12']=matrix[12];days['M13']=matrix[13]
    days['drop_change']=days.M12-days.M11;days['rise_change']=days.M13-days.M12
    days['dip_rebound']=days.drop_change.lt(0)&days.rise_change.gt(0)
    days['returned_to_M11']=days.M13.ge(days.M11)
    days['positive_production_hours']=v.groupby('date')['생산량'].apply(lambda x:x.gt(0).sum())
    # Exact 96-slot identities are auxiliary record structure, not process IDs.
    slot_matrix=v.set_index(['date','시간'])[cols].unstack('시간').sort_index()
    profile_ids=pd.factorize(pd.Series([tuple(x) for x in slot_matrix.to_numpy()]))[0]
    days['profile_id']=profile_ids
    csv_save('daily_values',days.reset_index())
    csv_save('hourly_matrix',matrix.reset_index())
    dist=[];group_summary=[];monthly=[];signs=[];profiles=[];production_context=[]
    old=pd.read_csv(ROOT/'EDA/tables/daily_pattern_restart.csv',encoding='utf-8-sig')
    for group in ['weekday','weekend']:
        subset=days.loc[days.group.eq(group)]
        a=matrix.loc[subset.index]
        expected=old.loc[old.population.eq(group)].sort_values('hour')
        np.testing.assert_allclose(a.mean(axis=0),expected.power_mean,rtol=1e-12,atol=1e-12)
        for h in range(24):
            dist.append({'group':group,'hour':h,'n':len(a),'mean':float(a[h].mean()),
                'median':float(a[h].median()),'p10':float(a[h].quantile(.1)),
                'p90':float(a[h].quantile(.9)),'min':int(a[h].min()),'max':int(a[h].max())})
        stat={'group':group,'n':len(subset),'drop':int(subset.drop_change.lt(0).sum()),
            'rise':int(subset.rise_change.gt(0).sum()),'both':int(subset.dip_rebound.sum()),
            'both_rate':float(subset.dip_rebound.mean()),'median_drop':float(subset.drop_change.median()),
            'median_rise':float(subset.rise_change.median()),
            'ties_at_one_or_both':int((subset.drop_change.eq(0)|subset.rise_change.eq(0)).sum()),
            'returned_among_both':int((subset.dip_rebound&subset.returned_to_M11).sum())}
        group_summary.append(stat)
        for positive in [False,True]:
            part=subset.loc[subset.positive_production_hours.gt(0).eq(positive)]
            production_context.append({'group':group,'any_positive_production':positive,'n':len(part),
                'both':int(part.dip_rebound.sum()),'both_rate':float(part.dip_rebound.mean()),
                'median_drop':float(part.drop_change.median()),'median_rise':float(part.rise_change.median())})
        for m,part in subset.groupby('month'):
            monthly.append({'group':group,'month':int(m),'n':len(part),'both':int(part.dip_rebound.sum()),
                'both_rate':float(part.dip_rebound.mean()),'drop_median':float(part.drop_change.median()),
                'rise_median':float(part.rise_change.median()),'day_mean_median':float(part.mean_power.median()),
                'zero_production_days':int(part.positive_production_hours.eq(0).sum())})
        for x in [-1,0,1]:
            for y in [-1,0,1]:
                signs.append({'group':group,'drop_sign':x,'rise_sign':y,
                    'n':int((np.sign(subset.drop_change).eq(x)&np.sign(subset.rise_change).eq(y)).sum())})
        unique=subset.drop_duplicates('profile_id')
        profiles.append({'group':group,'all_days':len(subset),'unique_profiles':len(unique),
            'all_both_rate':float(subset.dip_rebound.mean()),'unique_both':int(unique.dip_rebound.sum()),
            'unique_both_rate':float(unique.dip_rebound.mean())})
    dist=pd.DataFrame(dist);monthly=pd.DataFrame(monthly)
    csv_save('hourly_distribution',dist);csv_save('group_summary',pd.DataFrame(group_summary))
    csv_save('monthly_summary',monthly);csv_save('sign_combinations',pd.DataFrame(signs))
    csv_save('profile_sensitivity',pd.DataFrame(profiles))
    csv_save('production_context',pd.DataFrame(production_context))
    # Independent direct per-date reconstruction of strict inequalities.
    counts={'weekday':0,'weekend':0}
    for date,part in v.groupby('date'):
        mm=part.set_index('시간')['평균']
        if mm[12]<mm[11] and mm[13]>mm[12]:
            counts['weekday' if date.dayofweek<5 else 'weekend']+=1
    assert all(counts[s['group']]==s['both'] for s in group_summary)
    assert sum(s['n'] for s in group_summary)==241
    assert sum(s['n'] for s in signs)==241
    # Calendar rows preserve invalid dates as gray, not compressed away.
    calendar=pd.date_range('2021-01-01','2021-08-31',freq='D')
    full=matrix.reindex(calendar)
    missing=full.index[full.isna().all(axis=1)]
    assert list(missing.strftime('%Y-%m-%d'))==['2021-07-13','2021-07-15']

    figures=[]  # Selected manuscript plots are generated by separate scripts.
    summary={'source_sha256':SHA,'scope':{'hours':5784,'normal_dates':241,'weekday':171,'weekend':70,
        'calendar_dates':len(calendar),'invalid_dates':list(missing.strftime('%Y-%m-%d'))},
        'groups':group_summary,'monthly':monthly.to_dict('records'),'profile_sensitivity':profiles,
        'production_context':production_context,
        'unique_hourly_mean_profiles':int(matrix.drop_duplicates().shape[0]),'unique_96slot_profiles':int(days.profile_id.nunique()),
        'august_zero_production_dates':list(days.loc[days.month.eq(8)&days.positive_production_hours.eq(0)].index.strftime('%Y-%m-%d')),
        'figures':figures,'verification':'passed: original hash; 24 hours per valid date; exact rounding identity; all 48 group-hour means reconciled with 2.1; direct per-date sign count; missing dates preserved',
        'quantile_band':'Across observed dates, linear-interpolated P10/P90, not a confidence interval or forecast interval.',
        'manuscript_sha256_before':before,'manuscript_sha256_after':hashlib.sha256(MANUSCRIPT.read_bytes()).hexdigest(),
        'manuscript_edited':False,'visual_review':False}
    assert summary['manuscript_sha256_after']==before
    json_save('summary',summary)
    print(json.dumps({k:summary[k] for k in ['scope','groups','production_context','profile_sensitivity','unique_hourly_mean_profiles','unique_96slot_profiles','verification','manuscript_edited']},ensure_ascii=False,indent=2))

if __name__=='__main__':main()


코드 등록 완료: `EDA/scripts/eda_daily_repetition.py`

### `EDA/scripts/eda_figures.py`

원고 연결: EDA\02_EDA_3페이지용_압축본.md; EDA\02_EDA_원고.md; EDA\02_EDA_원고.md; EDA\02_EDA_원고.md; EDA\02_EDA_원고.md.

코드 설명: Shared plot styling for the manuscript: font, colors, and PNG export.

주요 함수: `read()`, `setup()`, `finish()`, `base()`.

In [5]:
%%submission_source EDA/scripts/eda_figures.py
"""Shared plot styling for the manuscript: font, colors, and PNG export."""
# %% Plot settings
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import PercentFormatter

REPORT = Path(__file__).resolve().parents[1]
TABLES, FIGURES = REPORT/"tables", REPORT/"figures"
BLUE, ORANGE, TEAL, GRAY = "#285B8C", "#D07834", "#25857A", "#7B8794"
COLORS = [BLUE, ORANGE]
GENERATED = []
STYLE_CHECKS = []


def read(name):
    return pd.read_csv(TABLES/f"{name}.csv", encoding="utf-8-sig")


def setup():
    names = {f.name for f in font_manager.fontManager.ttflist}
    selected = next((x for x in ["Malgun Gothic", "Noto Sans CJK KR", "NanumGothic"] if x in names), None)
    if selected is None:
        raise RuntimeError("Korean font required: Malgun Gothic / Noto Sans CJK KR / NanumGothic")
    plt.rcParams.update({"font.family": selected, "axes.unicode_minus": False,
                         "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
                         "axes.spines.top": False, "axes.spines.right": False,
                         "figure.facecolor": "white", "axes.facecolor": "white",
                         "savefig.facecolor": "white", "axes.labelcolor": "#283747",
                         "xtick.color": "#455563", "ytick.color": "#455563"})
    FIGURES.mkdir(parents=True, exist_ok=True)
    return selected


def finish(fig, name, title, note):
    # Report-wide user preference: titles/captions belong in Markdown.
    # Preserve panel titles, axes, legends and data annotations.
    assert fig._suptitle is None and not fig.texts, "No whole-figure title or footer"
    for ax in fig.axes:
        for label in [ax.get_xlabel(), ax.get_ylabel(), ax.get_title()]:
            assert "원자료 기록값" not in label and "원자료 단위" not in label
    fig.tight_layout(pad=1.5)
    path = FIGURES/f"{name}.png"
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        fig.savefig(path, dpi=180)
    missing = [str(w.message) for w in captured if "Glyph" in str(w.message)]
    plt.close(fig)
    if missing:
        raise RuntimeError(f"Missing font glyphs in {name}: {missing[:3]}")
    GENERATED.append(dict(file=path.name, bytes=path.stat().st_size,
                          title=title, caption=note, visual_review=False))
    STYLE_CHECKS.append(dict(path=str(path), whole_figure_title=False,
                             footer=False, original_value_label=False))


def base(ncols=1, nrows=1, size=(10, 5.4)):
    return plt.subplots(nrows, ncols, figsize=size)




코드 등록 완료: `EDA/scripts/eda_figures.py`

### `EDA/scripts/eda_slot_time_patterns.py`

원고 연결: EDA\01_데이터_0.5페이지용_압축본.md; EDA\01_데이터_원고.md; EDA\02_EDA_3페이지용_압축본.md; EDA\02_EDA_원고.md.

코드 설명: Whole-dataset EDA: recurring four-slot profiles at each hour of day.

Jan-Aug valid dates, all records; weekday/weekend summaries, no peak threshold.
Selected heatmap: EDA/figures/slot_time_patterns.png. The 96-slot line is a preview.
The script never edits the manuscript.

주요 함수: `load_data()`, `summarize()`, `verify_independently()`, `save_figure()`, `plot_daily_96()`, `plot_centered_heatmap()`, `main()`.

In [6]:
%%submission_source EDA/scripts/eda_slot_time_patterns.py
"""Whole-dataset EDA: recurring four-slot profiles at each hour of day.

Jan-Aug valid dates, all records; weekday/weekend summaries, no peak threshold.
Selected heatmap: EDA/figures/slot_time_patterns.png. The 96-slot line is a preview.
The script never edits the manuscript.
"""
from pathlib import Path
import csv
import argparse
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
from PIL import Image

from eda_variable_relations import SOURCE, SHA
import eda_figures as style

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / 'EDA/tables/slot_time_patterns'
PREVIEW = ROOT / 'tmp/eda_slot_time_patterns'
FIGURE = ROOT / 'EDA/figures/slot_time_patterns.png'
SLOTS = ['15분', '30분', '45분', '60분']


def load_data():
    assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == SHA
    raw = pd.read_csv(SOURCE, encoding='utf-8-sig',
                      usecols=['날짜', '시간', '평균', *SLOTS])
    data = raw.loc[raw['날짜'].lt(20210901) & raw['시간'].between(0, 23)].copy()
    data = data.sort_values(['날짜', '시간']).reset_index(drop=True)
    assert len(data) == 5784 and data['날짜'].nunique() == 241
    assert not data.duplicated(['날짜', '시간']).any()
    assert data.groupby('날짜')['시간'].apply(list).map(lambda x: x == list(range(24))).all()
    dates = pd.to_datetime(data['날짜'].astype(str), format='%Y%m%d')
    data['group'] = np.where(dates.dt.dayofweek.ge(5), 'weekend', 'weekday')
    data['exact_mean'] = data[SLOTS].mean(axis=1)
    assert np.array_equal(np.floor(data.exact_mean + .5), data['평균'])
    assert data.groupby('group')['날짜'].nunique().to_dict() == {'weekday':171, 'weekend':70}
    return data


def summarize(data):
    cells, directions, profiles = [], [], []
    for (group, hour), part in data.groupby(['group', '시간'], sort=True):
        values = part[SLOTS].to_numpy()
        centered = values - part.exact_mean.to_numpy()[:, None]
        unique_values = part[SLOTS].drop_duplicates().to_numpy()
        unique_centered = unique_values - unique_values.mean(axis=1)[:, None]
        for i, slot in enumerate(SLOTS):
            delta = centered[:, i]
            cells.append(dict(group=group, hour=int(hour), slot=slot, slot_index=i,
                days=len(part), power_mean=float(values[:, i].mean()),
                hourly_exact_mean=float(part.exact_mean.mean()),
                mean_deviation=float(delta.mean()), median_deviation=float(np.median(delta)),
                above_hours=int((delta > 0).sum()), equal_hours=int((delta == 0).sum()),
                below_hours=int((delta < 0).sum()), above_fraction=float((delta > 0).mean()),
                below_fraction=float((delta < 0).mean()),
                unique_four_value_rows=len(unique_values),
                unique_mean_deviation=float(unique_centered[:, i].mean())))
        comparisons = [('60분-15분', 3, 0), ('30분-15분', 1, 0),
                       ('45분-30분', 2, 1), ('60분-45분', 3, 2)]
        for label, last, first in comparisons:
            delta = values[:, last] - values[:, first]
            unique_delta = unique_values[:, last] - unique_values[:, first]
            directions.append(dict(group=group, hour=int(hour), comparison=label,
                days=len(part), mean_change=float(delta.mean()), median_change=float(np.median(delta)),
                positive_days=int((delta > 0).sum()), zero_days=int((delta == 0).sum()),
                negative_days=int((delta < 0).sum()), positive_fraction=float((delta > 0).mean()),
                negative_fraction=float((delta < 0).mean()),
                unique_rows=len(unique_values), unique_positive_fraction=float((unique_delta > 0).mean()),
                unique_negative_fraction=float((unique_delta < 0).mean())))
        def profile_counts(array):
            return dict(first_strict_lowest=int((array[:, 0] < array[:, 1:].min(axis=1)).sum()),
                middle_below_both_ends=int((array[:, 1:3].max(axis=1) < array[:, [0, 3]].min(axis=1)).sum()),
                late_above_both_early=int((array[:, 2:4].min(axis=1) > array[:, :2].max(axis=1)).sum()),
                strictly_increasing=int(np.all(np.diff(array, axis=1) > 0, axis=1).sum()))
        counts = profile_counts(values)
        unique_counts = profile_counts(unique_values)
        profiles.append(dict(group=group, hour=int(hour), days=len(part), **counts,
            unique_rows=len(unique_values), **{f'unique_{k}':v for k,v in unique_counts.items()}))
    cells, directions = pd.DataFrame(cells), pd.DataFrame(directions)
    assert len(cells) == 192 and len(directions) == 192
    assert np.allclose(cells.groupby(['group', 'hour']).mean_deviation.sum(), 0)
    assert (cells.above_hours + cells.equal_hours + cells.below_hours).eq(cells.days).all()
    assert (directions.positive_days + directions.zero_days + directions.negative_days).eq(directions.days).all()
    return cells, directions, pd.DataFrame(profiles)


def verify_independently(cells, directions, profiles):
    buckets = {}
    with SOURCE.open(encoding='utf-8-sig', newline='') as handle:
        for row in csv.DictReader(handle):
            date, hour = int(row['날짜']), int(row['시간'])
            if date >= 20210901 or not 0 <= hour <= 23:
                continue
            group = 'weekend' if pd.Timestamp(str(date)).dayofweek >= 5 else 'weekday'
            buckets.setdefault((group, hour), []).append([int(row[x]) for x in SLOTS])
    for row in cells.itertuples(index=False):
        values = buckets[row.group, row.hour]
        mean = sum(x[row.slot_index] for x in values) / len(values)
        deviations = [x[row.slot_index] - sum(x) / 4 for x in values]
        assert len(values) == row.days
        assert abs(mean - row.power_mean) < 1e-10
        assert abs(sum(deviations) / len(values) - row.mean_deviation) < 1e-10
        assert sum(x > 0 for x in deviations) == row.above_hours
        assert sum(x < 0 for x in deviations) == row.below_hours
    for row in directions.itertuples(index=False):
        last, first = [SLOTS.index(x) for x in row.comparison.split('-')]
        delta = [x[last] - x[first] for x in buckets[row.group, row.hour]]
        assert sum(x > 0 for x in delta) == row.positive_days
        assert sum(x == 0 for x in delta) == row.zero_days
        assert sum(x < 0 for x in delta) == row.negative_days
        assert abs(sum(delta) / len(delta) - row.mean_change) < 1e-10
    for row in profiles.itertuples(index=False):
        values = buckets[row.group, row.hour]
        assert sum(x[0] < min(x[1:]) for x in values) == row.first_strict_lowest
        assert sum(max(x[1:3]) < min(x[0], x[3]) for x in values) == row.middle_below_both_ends
        assert sum(min(x[2:]) > max(x[:2]) for x in values) == row.late_above_both_early
        assert sum(x[0] < x[1] < x[2] < x[3] for x in values) == row.strictly_increasing


def save_figure(fig, name):
    assert fig._suptitle is None and not fig.texts
    fig.tight_layout(pad=1.5)
    target = PREVIEW / f'{name}.png'
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        fig.savefig(target, dpi=180)
    assert not [str(x.message) for x in caught if 'Glyph' in str(x.message) or 'layout' in str(x.message)]
    style.plt.close(fig)
    return target


def plot_daily_96(cells):
    style.setup()
    fig, ax = style.plt.subplots(figsize=(12, 7.5))
    for group, label, color in zip(['weekday', 'weekend'],
        ['평일 (171일)', '주말 (70일)'], [style.BLUE, style.ORANGE]):
        part = cells.loc[cells.group.eq(group)].sort_values(['hour', 'slot_index'])
        x = part.hour.to_numpy() + part.slot_index.to_numpy() / 4
        ax.plot(x, part.power_mean, color=color, lw=1.7, marker='o', markersize=2.5,
                label=label)
    ax.set(ylabel='평균 전력', xlim=(-.2, 24), ylim=(0, 210))
    ax.grid(axis='y', alpha=.15)
    ax.legend(loc='upper right', frameon=False, fontsize=11)
    ax.set(xticks=np.arange(24) + .375, xticklabels=[str(h) for h in range(24)],
                 xlabel='시간대 (시)')
    return save_figure(fig, 'daily_96_slot_means')


def plot_centered_heatmap(cells):
    fig, axes = style.plt.subplots(1, 2, figsize=(11.6, 9), sharey=True, layout='constrained')
    bound = float(np.ceil(cells.mean_deviation.abs().max()))
    for ax, group, label in zip(axes, ['weekday', 'weekend'], ['평일 (171일)', '주말 (70일)']):
        matrix = cells.loc[cells.group.eq(group)].pivot(index='hour', columns='slot',
            values='mean_deviation').reindex(columns=SLOTS)
        art = ax.imshow(matrix.to_numpy(), cmap='RdBu_r', vmin=-bound, vmax=bound, aspect='auto')
        ax.set(title=label, xticks=range(4), xticklabels=SLOTS,
               yticks=range(24), yticklabels=[f'{h:02d}시' for h in range(24)])
        ax.tick_params(axis='both', length=0)
        for y in range(24):
            for x in range(4):
                value = matrix.iloc[y, x]
                number = '0.0' if abs(value) < .05 else f'{value:+.1f}'
                ax.text(x, y, number, ha='center', va='center', fontsize=10,
                         color='white' if abs(value) > .60 * bound else '#253746')
    axes[0].set_ylabel('시간대')
    fig.colorbar(art, ax=axes, shrink=.92, label='15분 구간 전력 - 같은 시간의 네 구간 평균')
    assert fig._suptitle is None and not fig.texts
    target = FIGURE
    target.parent.mkdir(parents=True, exist_ok=True)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        fig.savefig(target, dpi=180)
    assert not [str(x.message) for x in caught if 'Glyph' in str(x.message) or 'layout' in str(x.message)]
    style.plt.close(fig)
    return target


def main(figures_only=False):
    OUT.mkdir(parents=True, exist_ok=True)
    PREVIEW.mkdir(parents=True, exist_ok=True)
    manuscript = ROOT / 'EDA/02_EDA_원고.md'
    before = hashlib.sha256(manuscript.read_bytes()).hexdigest()
    if figures_only:
        assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == SHA
        table_paths = sorted(OUT.glob('*.csv'))
        table_hashes = {str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in table_paths}
        cells = pd.read_csv(OUT / 'hour_slot_summary.csv', encoding='utf-8-sig')
        assert len(cells) == 192
        assert cells.loc[cells.group.eq('weekday'), 'days'].eq(171).all()
        assert cells.loc[cells.group.eq('weekend'), 'days'].eq(70).all()
        assert np.allclose(cells.groupby(['group','hour']).mean_deviation.sum(), 0)
        style.setup()
        target = plot_centered_heatmap(cells)
        with Image.open(target) as image:
            image.resize((image.width // 2, image.height // 2), Image.Resampling.LANCZOS).save(
                PREVIEW / 'heatmap_50.png')
        result = json.loads((OUT / 'summary.json').read_text(encoding='utf-8'))
        result.pop('heatmap_figure_group', None)
        result['heatmap_figure_groups'] = ['weekday','weekend']
        for item in result['figures']:
            if Path(item['path']).name in ['hour_slot_mean_deviations.png', target.name]:
                item['path'] = str(target)
                item['sha256'] = hashlib.sha256(target.read_bytes()).hexdigest()
        result['visual_review'] = False
        result['plot_revision'] = dict(date='2026-10-03', figures_only=True,
            result_tables_unchanged=True, common_color_limits=[-float(np.ceil(cells.mean_deviation.abs().max())),
                float(np.ceil(cells.mean_deviation.abs().max()))])
        assert table_hashes == {str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in table_paths}
        assert hashlib.sha256(manuscript.read_bytes()).hexdigest() == before
        (OUT / 'summary.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
        print(json.dumps(dict(status='passed', figures_only=True, reused_cells=192,
            result_tables_unchanged=True, manuscript_edited=False, figure=str(target)), ensure_ascii=False))
        return
    data = load_data()
    cells, directions, profiles = summarize(data)
    verify_independently(cells, directions, profiles)
    cells.to_csv(OUT / 'hour_slot_summary.csv', index=False, encoding='utf-8-sig')
    directions.to_csv(OUT / 'within_hour_directions.csv', index=False, encoding='utf-8-sig')
    profiles.to_csv(OUT / 'within_hour_profiles.csv', index=False, encoding='utf-8-sig')
    # Rank every hour by the spread of its four mean values; no selected dates.
    ranks = cells.groupby(['group', 'hour']).agg(days=('days', 'first'),
        smallest_slot_mean=('power_mean', 'min'), largest_slot_mean=('power_mean', 'max'))
    ranks['spread_of_slot_means'] = ranks.largest_slot_mean - ranks.smallest_slot_mean
    ranks.reset_index().sort_values(['group', 'spread_of_slot_means'], ascending=[True, False]).to_csv(
        OUT / 'hourly_slot_mean_spread.csv', index=False, encoding='utf-8-sig')
    figures = [plot_daily_96(cells), plot_centered_heatmap(cells)]
    thumbs = []
    for target in figures:
        with Image.open(target) as image:
            thumbs.append(image.convert('RGB').resize((image.width // 2, image.height // 2),
                                                     Image.Resampling.LANCZOS))
    contact = Image.new('RGB', (sum(x.width for x in thumbs) + 16, max(x.height for x in thumbs)), 'white')
    x = 0
    for thumb in thumbs:
        contact.paste(thumb, (x, 0))
        x += thumb.width + 16
    contact.save(PREVIEW / 'contact_50.png')
    result = dict(status='passed', source_sha256=SHA, rows=5784, days=241,
        weekday_days=171, weekend_days=70, checked_mean_deviation_cells=192,
        checked_direction_cells=192, checked_profile_conditions=192,
        line_figure_groups=['weekday','weekend'], heatmap_figure_groups=['weekday','weekend'],
        definition='Each slot minus the exact arithmetic mean of its own four-slot row; average across all dates at the same hour and group.',
        figure_x_definition='Four slots are placed in column order within each hour, not verified exact sensor timestamps.',
        figures=[dict(path=str(p), sha256=hashlib.sha256(p.read_bytes()).hexdigest()) for p in figures],
        manuscript_edited=False, visual_review=False, p_values_used=False,
        note='Date frequencies are descriptive; repeated four-value profiles are not independent experiments.')
    assert hashlib.sha256(manuscript.read_bytes()).hexdigest() == before
    (OUT / 'summary.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    top = ranks.reset_index().sort_values(['group', 'spread_of_slot_means'], ascending=[True, False]).groupby('group').head(4)
    print(top.to_json(orient='records', force_ascii=False))
    hours = sorted(set(top.hour.tolist() + [4]))
    print(directions.loc[directions.comparison.eq('60분-15분') & directions.hour.isin([4,7])].to_json(
        orient='records', force_ascii=False))
    print(profiles.loc[profiles.hour.isin([7,10,12,17])].to_json(orient='records', force_ascii=False))


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--figures-only', action='store_true', help='Reuse verified tables and redraw only the heatmap')
    main(figures_only=parser.parse_args().figures_only)


코드 등록 완료: `EDA/scripts/eda_slot_time_patterns.py`

### `EDA/scripts/eda_variable_relations.py`

원고 연결: EDA\01_데이터_0.5페이지용_압축본.md; EDA\01_데이터_원고.md; EDA\01_데이터_원고.md; EDA\02_EDA_3페이지용_압축본.md; EDA\02_EDA_원고.md; EDA\02_EDA_원고.md.

코드 설명: Restart EDA 2.3: distribution/method diagnostics, then observational relations.

Supplied Jan-Aug valid-hour records only. No image reading or model evaluation.
Run --diagnose before --figures. All hours and extreme values remain in outputs.

주요 함수: `save()`, `source_frame()`, `diagnose()`, `relations()`, `figures()`.

In [7]:
%%submission_source EDA/scripts/eda_variable_relations.py
"""Restart EDA 2.3: distribution/method diagnostics, then observational relations.

Supplied Jan-Aug valid-hour records only. No image reading or model evaluation.
Run --diagnose before --figures. All hours and extreme values remain in outputs.
"""
# %% Fixed scope and diagnostics chosen before examining new results
from pathlib import Path
import argparse
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import eda_figures as style

REPORT=Path(__file__).resolve().parents[1]
ROOT=REPORT.parent
OUT=REPORT/'tables'
SOURCE=ROOT/'data/origin/okm_augumented_2021.csv'
SHA='8f7af2e49366c93e1d6f5fdef4b5e350066c1792ac463c2c2886e370f4674830'
INPUTS=['생산량','기온','풍속','습도','강수량']
VARIABLES=['평균 전력',*INPUTS]


def save(table,name):
    table.to_csv(OUT/f'{name}.csv',index=False,encoding='utf-8-sig')


def source_frame():
    assert hashlib.sha256(SOURCE.read_bytes()).hexdigest()==SHA
    raw=pd.read_csv(SOURCE,encoding='utf-8-sig',usecols=['날짜','시간','평균',*INPUTS])
    data=raw.loc[raw['날짜'].lt(20210901) & raw['시간'].between(0,23)].copy()
    assert len(data)==5784 and data['날짜'].nunique()==241
    data=data.rename(columns={'평균':'평균 전력'})
    dates=pd.to_datetime(data['날짜'].astype(str),format='%Y%m%d')
    data['month']=dates.dt.month
    data['weekend']=dates.dt.dayofweek.ge(5)
    return data


def diagnose(data):
    distributions=[]
    for variable in VARIABLES:
        values=data[variable].dropna()
        normal=stats.normaltest(values)
        distributions.append(dict(variable=variable,n=len(values),missing=len(data)-len(values),
                                  zero_n=int(values.eq(0).sum()),zero_share=values.eq(0).mean(),
                                  unique=values.nunique(),mode_share=values.value_counts().max()/len(values),
                                  skew=values.skew(),excess_kurtosis=values.kurt(),
                                  min=values.min(),q25=values.quantile(.25),median=values.median(),
                                  q75=values.quantile(.75),q99=values.quantile(.99),max=values.max(),
                                  normaltest_statistic=normal.statistic,normaltest_naive_iid_p=normal.pvalue))
    save(pd.DataFrame(distributions),'restart_relations_distribution')
    summaries=[]
    for variable in INPUTS:
        part=data[[variable,'평균 전력']].dropna()
        # A predeclared sensitivity ONLY, never a cleaned analysis population.
        cutoff=part[variable].quantile(.99)
        inner=part.loc[part[variable].le(cutoff)]
        summaries.append(dict(variable=variable,n=len(part),x_p99=cutoff,
                              diagnostic_inner_n=len(inner),diagnostic_omitted_n=len(part)-len(inner),
                              pearson_all=part[variable].corr(part['평균 전력']),
                              spearman_all=part[variable].corr(part['평균 전력'],method='spearman'),
                              pearson_below_x_p99=inner[variable].corr(inner['평균 전력']),
                              spearman_below_x_p99=inner[variable].corr(inner['평균 전력'],method='spearman')))
    save(pd.DataFrame(summaries),'restart_relations_sensitivity')
    # Bin summaries diagnose shape numerically; these are not regression curves.
    bins=[]
    for variable in INPUTS:
        part=data[[variable,'평균 전력']].dropna().copy()
        if variable in ['생산량','강수량']:
            part['band']='0'
            positive=part[variable].gt(0)
            part.loc[positive,'band']=pd.qcut(part.loc[positive,variable],q=5,duplicates='drop').astype(str)
        else:
            part['band']=pd.qcut(part[variable],q=5,duplicates='drop').astype(str)
        for band,cell in part.groupby('band'):
            bins.append(dict(variable=variable,band=band,n=len(cell),
                             x_min=cell[variable].min(),x_max=cell[variable].max(),
                             x_median=cell[variable].median(),power_mean=cell['평균 전력'].mean(),
                             power_median=cell['평균 전력'].median()))
    save(pd.DataFrame(bins).sort_values(['variable','x_min']),'restart_relations_bins')
    print(pd.DataFrame(distributions)[['variable','n','zero_share','skew','excess_kurtosis','normaltest_naive_iid_p']].to_json(orient='records',force_ascii=True))
    print(pd.DataFrame(summaries).to_json(orient='records',force_ascii=True))


def relations(data):
    # The main estimand is ordinal co-movement, not proportional/linear effect.
    # This choice is supported by zero masses, skew and extreme-value sensitivity,
    # not by an automatic normality-test switch. Conventional p-values are unused.
    rows=[]
    for x in VARIABLES:
        for y in VARIABLES:
            part=data[[x,y]].dropna() if x!=y else data[[x]].dropna()
            coefficient=part[x].corr(part[y],method='spearman') if x!=y else 1.
            rows.append(dict(x=x,y=y,n=len(part),spearman=coefficient))
    overall=pd.DataFrame(rows)
    save(overall,'restart_relations_spearman')
    monthly=[]
    for variable in INPUTS:
        for month,cell in data.groupby('month'):
            part=cell[[variable,'평균 전력']].dropna()
            monthly.append(dict(variable=variable,month=month,n=len(part),
                                days=cell['날짜'].nunique(),spearman=part[variable].corr(part['평균 전력'],method='spearman')))
    monthly=pd.DataFrame(monthly)
    save(monthly,'restart_relations_monthly')
    # Existing evidence audit and independent average-tie rank correlation.
    for row in overall.itertuples():
        if row.x != row.y:
            part=data[[row.x,row.y]].dropna()
            independent=np.corrcoef(stats.rankdata(part[row.x]),stats.rankdata(part[row.y]))[0,1]
            np.testing.assert_allclose(row.spearman,independent,atol=1e-12)
    for row in monthly.itertuples():
        part=data.loc[data.month.eq(row.month),[row.variable,'평균 전력']].dropna()
        independent=np.corrcoef(stats.rankdata(part[row.variable]),stats.rankdata(part['평균 전력']))[0,1]
        np.testing.assert_allclose(row.spearman,independent,atol=1e-12)
    # Follow the observed month-dependent temperature sign with the E007 August
    # production-state example. Same method; no centered residuals or new model.
    august=data.loc[data.month.eq(8)]
    contexts=[]
    for population,part in [('all',august),('positive',august.loc[august['생산량'].gt(0)]),
                            ('zero',august.loc[august['생산량'].eq(0)])]:
        pair=part[['기온','평균 전력']].dropna()
        coefficient=pair['기온'].corr(pair['평균 전력'],method='spearman')
        independent=np.corrcoef(stats.rankdata(pair['기온']),stats.rankdata(pair['평균 전력']))[0,1]
        np.testing.assert_allclose(coefficient,independent,atol=1e-12)
        contexts.append(dict(population=population,n=len(pair),days=part['날짜'].nunique(),
                             spearman=coefficient,power_mean=pair['평균 전력'].mean(),temperature_mean=pair['기온'].mean()))
    save(pd.DataFrame(contexts),'restart_relations_august_context')
    week=august.loc[august['날짜'].between(20210802,20210808)]
    assert len(week)==168 and week['생산량'].eq(0).all()
    week_row=dict(start=20210802,end=20210808,n=len(week),days=week['날짜'].nunique(),
                  power_mean=week['평균 전력'].mean(),temperature_mean=week['기온'].mean())
    save(pd.DataFrame([week_row]),'restart_relations_august_week')
    return overall,monthly


def figures(data):
    assert (OUT/'restart_relations_distribution.csv').exists(),'Run diagnostics first'
    overall,monthly=relations(data)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        font=style.setup()
        matrix=overall.pivot(index='x',columns='y',values='spearman').loc[VARIABLES,VARIABLES].to_numpy()
        fig,ax=style.plt.subplots(figsize=(8.8,7.2))
        masked=np.ma.array(matrix,mask=np.triu(np.ones_like(matrix),0).astype(bool))
        image=ax.imshow(masked,cmap='RdBu_r',vmin=-1,vmax=1)
        for i in range(len(VARIABLES)):
            for j in range(i):
                ax.text(j,i,f'{matrix[i,j]:.2f}',ha='center',va='center',fontsize=12,
                        color='white' if abs(matrix[i,j])>.65 else '#283747')
        ax.set(xticks=range(6),xticklabels=VARIABLES,yticks=range(6),yticklabels=VARIABLES)
        ax.tick_params(axis='x',rotation=35)
        fig.colorbar(image,ax=ax,pad=.03,label='Spearman 상관계수')
        style.finish(fig,'restart_02_variable_relations','생산량·날씨·전력의 관계',
                     '정상 5,784시간, 쌍별 결측 제외 · Spearman, 동률 평균 순위 · 유의성·인과효과 검정 아님')
        # Same color scale, one coefficient per month, independent variables included.
        values=monthly.pivot(index='variable',columns='month',values='spearman').loc[INPUTS]
        fig,ax=style.plt.subplots(figsize=(11,5.4))
        image=ax.imshow(values,cmap='RdBu_r',vmin=-1,vmax=1,aspect='auto')
        for i in range(5):
            for j in range(8):
                value=values.iloc[i,j]
                ax.text(j,i,f'{value:.2f}',ha='center',va='center',fontsize=11,
                        color='white' if abs(value)>.65 else '#283747')
        ax.set(xticks=range(8),xticklabels=[f'{m}월' for m in range(1,9)],
               yticks=range(5),yticklabels=INPUTS,xlabel='월')
        fig.colorbar(image,ax=ax,pad=.02,label='평균 전력과의 Spearman 상관계수')
        style.finish(fig,'restart_02_monthly_relations','월에 따라 달라지는 전력 관계',
                     '전체 정상 기록의 월별 상관 · 생산량 양수만의 결과와 구분 · n과 날짜 수는 월별 표에 보존')
    important=[str(w.message) for w in caught if 'Glyph' in str(w.message) or 'layout' in str(w.message).lower()]
    assert not important,important
    manifest=dict(status='passed',rows=len(data),scope='Restarted EDA 2.3 only',method='Spearman',
                  estimand='ordinal monotonic co-movement; no proportional or causal effect',
                  method_basis=['distribution: zero masses/ties and skewed production/rainfall','purpose: ordinal co-movement without proportional effect','sensitivity comparison is diagnostic only, not a method-selection rule'],
                  normality_diagnostic='D Agostino K2; naive iid p-values only, not calibrated for time dependence; not a selection switch',
                  main_population='All valid Jan-Aug hours; no tail exclusions',
                  checked_rank_rows=len(overall)+len(monthly)+3,
                  checked_independent_august_groups=3,pvalue_inference=False,
                  source_sha256=SHA,font=font,warnings=important,image_analysis_performed=False,
                  images=[dict(file=item['file'],bytes=item['bytes'],sha256=hashlib.sha256((style.FIGURES/item['file']).read_bytes()).hexdigest()) for item in style.GENERATED],
                  script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())
    (OUT/'restart_relations_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps({key:value for key,value in manifest.items() if key in ['status','method','checked_rank_rows','warnings','image_analysis_performed']}))
    print(monthly.loc[monthly.variable.eq('기온')].to_json(orient='records',force_ascii=True))


if __name__=='__main__':
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--diagnose',action='store_true')
    parser.add_argument('--figures',action='store_true')
    args=parser.parse_args()
    if not args.diagnose and not args.figures: parser.error('Choose --diagnose and/or --figures')
    frame=source_frame()
    if args.diagnose: diagnose(frame)
    if args.figures: figures(frame)


코드 등록 완료: `EDA/scripts/eda_variable_relations.py`

### `EDA/scripts/eda_weekday_power_levels.py`

원고 연결: EDA\02_EDA_3페이지용_압축본.md; EDA\02_EDA_원고.md.

코드 설명: Final EDA 2.4: weekday distribution and all-date low-hour patterns.

Run from project root: python EDA/scripts/eda_weekday_power_levels.py
Output tables: EDA/tables/weekday_power_levels; one manuscript histogram:
EDA/figures/weekday_power_levels.png. No discarded preview plots are generated.

주요 함수: `save()`, `summarize_patterns()`, `plot_distribution()`, `main()`.

In [8]:
%%submission_source EDA/scripts/eda_weekday_power_levels.py
"""Final EDA 2.4: weekday distribution and all-date low-hour patterns.

Run from project root: python EDA/scripts/eda_weekday_power_levels.py
Output tables: EDA/tables/weekday_power_levels; one manuscript histogram:
EDA/figures/weekday_power_levels.png. No discarded preview plots are generated.
"""
from pathlib import Path
import hashlib
import json
import warnings

import numpy as np
import pandas as pd

from eda_variable_relations import source_frame, SHA
import eda_figures as style

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / 'EDA/tables/weekday_power_levels'
FIGURE = ROOT / 'EDA/figures/weekday_power_levels.png'
POWER = '평균 전력'


def save(table, name):
    table.to_csv(OUT / f'{name}.csv', index=False, encoding='utf-8-sig')


def summarize_patterns(data, cutoff):
    """Enumerate exact low-hour sets for every complete weekday, not just counts."""
    daily = []
    for date, cell in data.groupby('날짜', sort=True):
        cell = cell.sort_values('시간')
        assert cell['시간'].tolist() == list(range(24))
        hours = tuple(cell.loc[cell[POWER].le(cutoff), '시간'].astype(int))
        daily.append(dict(date=int(date), low_hours=len(hours),
                          low_hour_pattern=','.join(map(str, hours)) if hours else 'none',
                          daily_any_production=bool(cell['생산량'].gt(0).any())))
    daily = pd.DataFrame(daily)
    counts = daily.groupby('low_hour_pattern').size().rename('days')
    patterns = [tuple(range(7)), tuple(range(8, 24)), tuple(range(24)), ()]
    labels = ['07시부터 49 이상', '08시부터 낮은 구간', '하루 내내 낮은 구간', '하루 내내 49 이상']
    records = []
    expected_keys = set()
    for pattern, label in zip(patterns, labels):
        key = ','.join(map(str, pattern)) if pattern else 'none'
        expected_keys.add(key)
        values = ['20~26' if hour in pattern else '49 이상' for hour in [0, 7, 8]]
        records.append(dict(pattern=label, low_hour_pattern=key,
                            hours_00_06=values[0], hour_07=values[1], hours_08_23=values[2],
                            days=int(counts.get(key, 0)), low_hours_per_day=len(pattern)))
    assert set(counts.index) == expected_keys, 'Manuscript requires revising for additional patterns'
    summary = pd.DataFrame(records)
    assert summary.days.sum() == len(daily) == 171
    assert int((summary.days * summary.low_hours_per_day).sum()) == int(data[POWER].le(cutoff).sum())
    return daily, summary


def plot_distribution(data, cutoff):
    """Edit histogram bin width, colors, axes and legend here."""
    style.setup()
    fig, ax = style.plt.subplots(figsize=(9, 4.6))
    bins = np.arange(19.5, 214.6, 5)
    low = data.loc[data[POWER].le(cutoff), POWER]
    other = data.loc[data[POWER].gt(cutoff), POWER]
    assert bins[0] <= data[POWER].min() and bins[-1] > data[POWER].max()
    counts, _ = np.histogram(data[POWER], bins=bins)
    assert counts.sum() == len(data)
    ax.hist([low, other], bins=bins, stacked=True,
            color=[style.ORANGE, style.BLUE], edgecolor='white', linewidth=.5,
            label=['20~26', '49~208'])
    ax.set(xlabel='시간별 평균 전력', ylabel='기록 수 (시간)', xlim=(15, 215))
    ax.legend(title='평일 전력 구간')
    ax.grid(axis='y', alpha=.16)
    style.finish(fig, 'weekday_power_levels', '평일 전력 분포',
                 '5 간격 히스토그램; 경계는 정수값별 실제 빈 구간에서 확인')
    return counts, bins


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    data = source_frame().loc[lambda frame: ~frame.weekend].copy()
    assert len(data) == 4104 and data['날짜'].nunique() == 171
    assert data[['날짜', '시간']].duplicated().sum() == 0
    support = np.sort(data[POWER].unique())
    largest = int(np.argmax(np.diff(support)))
    cutoff, next_value = int(support[largest]), int(support[largest + 1])
    assert (int(support.min()), cutoff, next_value, int(support.max())) == (20, 26, 49, 208)
    assert not data[POWER].between(27, 48).any()
    for boundary in range(27, 49):
        assert data[POWER].le(cutoff).equals(data[POWER].lt(boundary))
    frequencies = data[POWER].value_counts().reindex(range(20, 209), fill_value=0)
    save(frequencies.rename_axis('power').reset_index(name='hours'), 'power_frequency')
    daily, patterns = summarize_patterns(data, cutoff)
    save(daily, 'daily_hour_patterns')
    save(patterns, 'pattern_summary')
    data['low'] = data[POWER].le(cutoff)
    production_context = data.groupby('low').agg(
        hours=(POWER, 'size'), positive_production_hours=('생산량', lambda x: int(x.gt(0).sum()))).reset_index()
    save(production_context, 'production_context')
    # Independent cross-check of the exact daily masks using a wide array.
    matrix = data.pivot(index='날짜', columns='시간', values=POWER).sort_index()
    low_mask = matrix.le(cutoff).to_numpy()
    independently_counted = {}
    for row in low_mask:
        key = ','.join(map(str, np.flatnonzero(row))) if row.any() else 'none'
        independently_counted[key] = independently_counted.get(key, 0) + 1
    assert independently_counted == patterns.set_index('low_hour_pattern').days.to_dict()
    np.testing.assert_array_equal(low_mask.sum(axis=1), daily.low_hours.to_numpy())
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        histogram_counts, histogram_bins = plot_distribution(data, cutoff)
    important = [str(w.message) for w in caught if 'Glyph' in str(w.message) or 'layout' in str(w.message).lower()]
    assert not important, important
    save(pd.DataFrame(dict(left=histogram_bins[:-1], right=histogram_bins[1:], hours=histogram_counts)), 'histogram_bins')
    result = dict(status='passed', source_sha256=SHA, rows=len(data), days=171,
                  cutoff=cutoff, next_observed_value=next_value, empty_integer_values=[27, 48],
                  low_hours=int(data.low.sum()), low_fraction=float(data.low.mean()),
                  low_production_positive=int(data.loc[data.low, '생산량'].gt(0).sum()),
                  patterns=patterns.to_dict('records'), checked_daily_masks=171,
                  histogram_hours=int(histogram_counts.sum()), warnings=important,
                  image_analysis_performed=False, image=str(FIGURE),
                  image_sha256=hashlib.sha256(FIGURE.read_bytes()).hexdigest(),
                  script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())
    (OUT / 'summary.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps({key:value for key,value in result.items() if key not in ['source_sha256', 'script_sha256', 'image_sha256']}, ensure_ascii=False))


if __name__ == '__main__':
    main()


코드 등록 완료: `EDA/scripts/eda_weekday_power_levels.py`

### `EDA/scripts/p01_peak_shapes.py`

원고 연결: EDA\02_EDA_원고.md.

코드 설명: P01 descriptive slot shape audit; no fit, manuscript edits or plots.
Use exact-hour timestamps, reuse verified low/zero episodes, retain every valid row.

주요 함수: `sha()`, `dump()`, `save()`, `main()`.

In [9]:
%%submission_source EDA/scripts/p01_peak_shapes.py
"""P01 descriptive slot shape audit; no fit, manuscript edits or plots.
Use exact-hour timestamps, reuse verified low/zero episodes, retain every valid row.
"""
import csv, hashlib, json, sys
from datetime import datetime, timedelta, timezone
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path(__file__).resolve().parents[2]
SOURCE = ROOT/'data/origin/okm_augumented_2021.csv'
OUT = ROOT/'EDA/tables/p01_peak_shapes'
SLOTS = ['15분','30분','45분','60분']
def sha(p): return hashlib.sha256(p.read_bytes()).hexdigest()
def dump(x,p): p.write_text(json.dumps(x,ensure_ascii=False,indent=2),encoding='utf-8')
def save(d,name): d.to_csv(OUT/(name+'.csv'),index=False,encoding='utf-8-sig')
def main():
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    OUT.mkdir(parents=True,exist_ok=True)
    assert not (OUT/'run.json').exists(), 'Do not overwrite completed analysis'
    parent=json.loads((ROOT/'Analysis/tables/a06_power_state_transitions/summary.json').read_text(encoding='utf-8'))
    assert sha(SOURCE)==parent['source_sha256']
    for name,h in parent['outputs_sha256'].items():
        assert sha(ROOT/'Analysis/tables/a06_power_state_transitions'/name)==h
    c={'version':'p01-v1','recorded_at':datetime.now(timezone(timedelta(hours=9))).isoformat(timespec='minutes'),
       'source_sha256':sha(SOURCE),'script_sha256':sha(Path(__file__)),
       'question':'Does within-hour shape and duration justify history and four-output comparisons?',
       'scope':'All 5784 valid Jan-Aug hours; descriptive only; already observed data',
       'high_definition':'January valid hourly maximum linear q95; fixed before output inspection; slot >= threshold',
       'shape_measures':['maximum-mean','maximum-minimum','maximum-second_largest','tied maxima','high slots count'],
       'episode_definition':'Consecutive recorded slots in column order, connect hours only when exact-hour consecutive; no interpolation',
       'future_rule':'No full-day label or completed episode duration may be a prediction input',
       'reuse':'a06 episodes.csv for low/zero; slot_time_patterns for mean slot shapes',
       'no_manuscript_edit':True}
    dump(c,OUT/'contract.json')
    r=pd.read_csv(SOURCE,encoding='utf-8-sig')
    d=r.loc[r['날짜'].between(20210101,20210831)&r['시간'].between(0,23)].sort_values(['날짜','시간']).copy()
    d['timestamp']=pd.to_datetime(d['날짜'].astype(str),format='%Y%m%d')+pd.to_timedelta(d['시간'],unit='h')
    assert len(d)==5784 and d.timestamp.is_unique
    v=d[SLOTS].to_numpy(float); mx=v.max(1); mn=v.min(1); av=v.mean(1)
    threshold=float(np.quantile(mx[d.timestamp.dt.month.eq(1)],.95))
    o=pd.DataFrame({'timestamp':d.timestamp,'date':d.timestamp.dt.normalize(),'month':d.timestamp.dt.month,'hour':d['시간'],
        'maximum':mx,'mean':av,'range':mx-mn,'max_mean_gap':mx-av,'top_margin':mx-np.sort(v,axis=1)[:,-2],
        'max_ties':(v==mx[:,None]).sum(1),'high_slots':(v>=threshold).sum(1)})
    for i in range(4): o['max_slot_'+str(i+1)]=(v[:,i]==mx).astype(int)
    o['peak_mask']=[''.join('1' if a==max(row) else '0' for a in row) for row in v]
    hashes={day:hashlib.sha256(g[SLOTS].to_numpy(dtype=np.int64).tobytes()).hexdigest() for day,g in d.groupby(d.timestamp.dt.normalize())}
    o['profile']=o.date.map(hashes);freq=pd.Series(hashes).value_counts();o['profile_weight']=1/o.profile.map(freq)
    o['repeated_profile']=o.profile.map(freq).gt(1)
    dm=o.groupby('date').maximum.transform('max'); flag=o.maximum.eq(dm)
    o['daily_maximum_weight']=flag/flag.groupby(o.date).transform('sum')
    records=[]
    def summary(g,label,weight=None):
        w=np.ones(len(g)) if weight is None else np.asarray(weight)
        return {'scope':label,'hours':len(g),'dates':g.date.nunique(),'weight_sum':float(w.sum()),
            **{x:float(np.average(g[x],weights=w)) for x in ['maximum','range','max_mean_gap','top_margin','max_ties','high_slots']},
            'unique_max_fraction':float(np.average(g.max_ties.eq(1),weights=w)),
            **{'slot_'+str(i)+'_share':float(np.average(g['max_slot_'+str(i)]/g.max_ties,weights=w)) for i in range(1,5)}}
    records.extend([summary(o,'all'),summary(o,'profile_weighted',o.profile_weight),summary(o.loc[flag],'daily_peak',o.loc[flag,'daily_maximum_weight']),summary(o.loc[o.high_slots.gt(0)],'high')])
    for field in ['month','hour','repeated_profile','high_slots']:
        for val,g in o.groupby(field):records.append(summary(g,f'{field}:{val}'))
    save(o,'hourly_shapes');save(pd.DataFrame(records),'shape_summary')
    save(o.groupby(['month','peak_mask','high_slots']).size().rename('hours').reset_index(),'shape_counts')
    episodes=[];start=None;count=0;prev=None;last=None
    for ts,row in zip(d.timestamp,v):
        for i,value in enumerate(row):
            ordinal=int(ts.value//3_600_000_000_000)*4+i
            if start is not None and (ordinal!=prev+1 or value<threshold):
                episodes.append({'start_hour':start[0],'start_slot':start[1]+1,'end_hour':last[0],'end_slot':last[1]+1,'slots':count,'left_censored':left,'right_censored':ordinal!=prev+1})
                start=None;count=0
            if value>=threshold:
                if start is None:start=(ts,i);left=prev is None or ordinal!=prev+1
                count+=1;last=(ts,i)
            prev=ordinal
    if start is not None:episodes.append({'start_hour':start[0],'start_slot':start[1]+1,'end_hour':last[0],'end_slot':last[1]+1,'slots':count,'left_censored':left,'right_censored':True})
    ep=pd.DataFrame(episodes);save(ep,'high_episodes')
    assert ep.slots.sum()==int((v>=threshold).sum())
    # Independent stdlib raw-row reconstruction of every derived shape value.
    raw={}
    with SOURCE.open(encoding='utf-8-sig',newline='') as f:
        for q in csv.DictReader(f):
            day=int(q['날짜']);hour=float(q['시간'])
            if 20210101<=day<=20210831 and 0<=hour<=23:
                ts=datetime.strptime(str(day),'%Y%m%d')+timedelta(hours=hour)
                raw[ts]=[float(q[s]) for s in SLOTS]
    for q in o.to_dict('records'):
        a=raw[q['timestamp'].to_pydatetime()];maximum=max(a)
        assert q['maximum']==maximum and q['mean']==sum(a)/4 and q['range']==maximum-min(a)
        assert q['max_mean_gap']==maximum-sum(a)/4 and q['top_margin']==maximum-sorted(a)[-2]
        assert q['max_ties']==a.count(maximum) and q['high_slots']==sum(x>=threshold for x in a)
        assert q['peak_mask']==''.join('1' if x==maximum else '0' for x in a)
    # Check episode records from a distinct list-based run-length reconstruction.
    high_ord=sorted(int(pd.Timestamp(ts).value//3_600_000_000_000)*4+i for ts,a in raw.items() for i,x in enumerate(a) if x>=threshold)
    groups=[]
    for x in high_ord:
        if not groups or x!=groups[-1][-1]+1:groups.append([x])
        else:groups[-1].append(x)
    assert ep.slots.tolist()==list(map(len,groups))
    verify={'status':'passed','raw_rows':len(raw),'high_episodes_checked':len(groups),'outputs_sha256':{p.name:sha(p) for p in OUT.glob('*.csv')}}
    dump(verify,OUT/'verification.json')
    counts=o.loc[o.high_slots.gt(0)].high_slots.value_counts().sort_index().to_dict()
    result={'status':'completed','high_threshold_january_q95':threshold,'rows':len(o),'days':o.date.nunique(),'profiles':len(freq),
        'high_hours':int(o.high_slots.gt(0).sum()),'high_slot_counts':{str(k):int(n) for k,n in counts.items()},
        'high_episodes':len(ep),'high_episode_slots_median':float(ep.slots.median()),'high_episode_slots_max':int(ep.slots.max()),
        'key_summaries':records[:4],'existing_low_zero_episodes':parent['episodes'],'contract_sha256':sha(OUT/'contract.json'),
        'outputs_sha256':verify['outputs_sha256'],'verified':True,'model_fit':False,'manuscript_edited':False}
    dump(result,OUT/'run.json');print(json.dumps(result,ensure_ascii=False),flush=True)
if __name__=='__main__':main()

코드 등록 완료: `EDA/scripts/p01_peak_shapes.py`

### `EDA/scripts/plot_daily_repetition_simple.py`

원고 연결: EDA\02_EDA_원고.md.

코드 설명: Plot aggregated EDA 2.2 curves with axes and legends only.

주요 함수: `main()`.

In [10]:
%%submission_source EDA/scripts/plot_daily_repetition_simple.py
"""Plot aggregated EDA 2.2 curves with axes and legends only."""
from pathlib import Path
import hashlib
import json
import warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT=Path(__file__).resolve().parents[2]
TAB=ROOT/'EDA/tables/daily_repetition'
OUT=ROOT/'EDA/figures/daily_repetition'
MANUSCRIPT=ROOT/'EDA/02_EDA_원고.md'

def main():
    before=hashlib.sha256(MANUSCRIPT.read_bytes()).hexdigest()
    data=pd.read_csv(TAB/'hourly_distribution.csv',encoding='utf-8-sig')
    counts=pd.read_csv(TAB/'group_summary.csv',encoding='utf-8-sig').set_index('group')
    plt.rcParams.update({'font.family':'Malgun Gothic','axes.unicode_minus':False,
        'font.size':12,'axes.titlesize':17,'axes.titleweight':'bold','axes.labelsize':13,
        'axes.spines.top':False,'axes.spines.right':False,
        'figure.facecolor':'white','savefig.facecolor':'white'})
    fig,axes=plt.subplots(1,2,figsize=(14.6,6.5),sharey=True,layout='constrained')
    for ax,group,label,color in zip(axes,['weekday','weekend'],['평일','주말'],['#285B8C','#C46E2C']):
        q=data.loc[data.group.eq(group)].sort_values('hour').set_index('hour')
        stat=counts.loc[group]
        assert len(q)==24 and q['n'].eq(stat['n']).all()
        ax.fill_between(q.index,q.p10,q.p90,color=color,alpha=.15,
            label='10~90백분위 범위')
        ax.plot(q.index,q['mean'],c='#4c5963',ls='--',lw=2,label='평균',zorder=3)
        ax.plot(q.index,q['median'],c=color,lw=2.6,label='중앙값',zorder=4)
        ax.set_title(label,pad=12)
        ax.set(xlabel='시간대 (시)',xlim=(0,23),ylim=(0,225),xticks=[0,3,6,9,12,15,18,21,23],
            yticks=[0,50,100,150,200])
        ax.tick_params(axis='x',labelsize=10)
        ax.grid(axis='y',alpha=.13)
        ax.legend(loc='upper left',fontsize=10,frameon=False,handlelength=2.5)
        ax.set_ylabel('시간 평균 전력')
    assert fig._suptitle is None and not fig.texts and all(not ax.texts for ax in axes)
    target=OUT/'04_daily_pattern_simple.png'
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always');fig.savefig(target,dpi=180,bbox_inches='tight')
    bad=[str(w.message) for w in caught if 'Glyph' in str(w.message) or 'layout' in str(w.message).lower()]
    assert not bad,bad
    plt.close(fig)
    # Preserve monthly descriptive values for the independent source audit.
    matrix=pd.read_csv(TAB/'hourly_matrix.csv',encoding='utf-8-sig',index_col='date')
    matrix.index=pd.to_datetime(matrix.index)
    matrix.columns=matrix.columns.astype(int)
    dates=pd.read_csv(TAB/'daily_values.csv',encoding='utf-8-sig',index_col='date')
    dates.index=pd.to_datetime(dates.index)
    monthly=[]
    for (month,group),part in dates.groupby(['month','group']):
        values=matrix.loc[part.index]
        for hour in range(24):
            monthly.append({'month':month,'group':group,'hour':hour,'n':len(values),
                'median':float(values[hour].median()),'p10':float(values[hour].quantile(.1)),
                'p90':float(values[hour].quantile(.9))})
    monthly=pd.DataFrame(monthly)
    assert monthly.groupby(['month','group']).n.first().sum()==241
    monthly.to_csv(TAB/'monthly_hourly_distribution.csv',index=False,encoding='utf-8-sig')
    after=hashlib.sha256(MANUSCRIPT.read_bytes()).hexdigest();assert before==after
    (TAB/'simple_figure_manifest.json').write_text(json.dumps({
        'figure':str(target),'change':'Removed all in-plot annotations, arrows, and outcome counts. Whole-period figure: mean, median and P10-P90 only. Monthly descriptive table retained for independent validation; the separate heatmap script generates the selected monthly plot.',
        'period':'2021-01-01 through 2021-08-31 valid dates; holidays included by day of week',
        'criterion':'M12<M11 and M13>M12, same date; no minimum amplitude cutoff',
        'band':'P10-P90 across dates, not confidence or prediction interval',
        'manuscript_edited':False,'manuscript_sha256':after,'visual_review':False,
        'numeric_result_changed':False,'warnings':bad},ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps({'figure':str(target),'warnings':bad,'manuscript_edited':False},ensure_ascii=False))

if __name__=='__main__':main()


코드 등록 완료: `EDA/scripts/plot_daily_repetition_simple.py`

### `EDA/scripts/plot_monthly_power_heatmap.py`

원고 연결: EDA\02_EDA_3페이지용_압축본.md; EDA\02_EDA_원고.md.

코드 설명: Monthly weekday/weekend summary heatmap: eight rows per panel, no annotations.

주요 함수: `main()`.

In [11]:
%%submission_source EDA/scripts/plot_monthly_power_heatmap.py
"""Monthly weekday/weekend summary heatmap: eight rows per panel, no annotations."""
from pathlib import Path
import argparse
import hashlib
import json
import warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

ROOT=Path(__file__).resolve().parents[2]
TAB=ROOT/'EDA/tables/daily_repetition'
OUT=ROOT/'EDA/figures/daily_repetition'
MANUSCRIPT=ROOT/'EDA/02_EDA_원고.md'

def main(statistic='mean'):
    before=hashlib.sha256(MANUSCRIPT.read_bytes()).hexdigest()
    source=TAB/('independent_monthly_cells.csv' if statistic=='mean' else 'monthly_hourly_distribution.csv')
    d=pd.read_csv(source,encoding='utf-8-sig')
    if statistic=='mean':
        d=d.rename(columns={'n_raw':'n','mean_raw':'value'})
        reference=pd.read_csv(TAB/'hourly_distribution.csv',encoding='utf-8-sig').set_index(['group','hour'])
        for (group,hour),part in d.groupby(['group','hour']):
            combined=np.average(part['value'],weights=part['n'])
            assert np.isclose(combined,reference.loc[(group,hour),'mean'],rtol=1e-12,atol=1e-12)
    else:
        d=d.rename(columns={'median':'value'})
    assert len(d)==384 and not d.duplicated(['month','group','hour']).any()
    plt.rcParams.update({'font.family':'Malgun Gothic','axes.unicode_minus':False,
        'font.size':12,'axes.titlesize':15,'axes.labelsize':12,
        'figure.facecolor':'white','savefig.facecolor':'white'})
    fig,axes=plt.subplots(1,2,figsize=(13.6,4.6),layout='constrained')
    norm=Normalize(vmin=0,vmax=210)
    for ax,group,label in zip(axes,['weekday','weekend'],['평일','주말']):
        p=d.loc[d.group.eq(group)].pivot(index='month',columns='hour',values='value')
        assert list(p.index)==list(range(1,9)) and list(p.columns)==list(range(24))
        assert not p.isna().any().any()
        assert p.to_numpy().min()>=norm.vmin and p.to_numpy().max()<=norm.vmax
        im=ax.imshow(p.to_numpy(),aspect='auto',interpolation='nearest',cmap='YlOrRd',norm=norm)
        ax.set(title=label,xlabel='시간대 (시)',ylabel='월',xticks=[0,3,6,9,12,15,18,21,23],
            yticks=np.arange(8),yticklabels=[f'{month}월' for month in range(1,9)])
        ax.set_yticks(np.arange(.5,7.5,1),minor=True)
        ax.grid(which='minor',axis='y',color='white',lw=1)
        ax.tick_params(which='minor',left=False)
        for spine in ax.spines.values():spine.set_visible(False)
    colorbar=fig.colorbar(im,ax=axes,shrink=.9,pad=.025,ticks=[0,50,100,150,200])
    colorbar.set_label('평균 전력' if statistic=='mean' else '시간 평균 전력의 중앙값')
    assert fig._suptitle is None and not fig.texts and all(not ax.texts for ax in axes)
    output=OUT/('07_monthly_power_mean_heatmap.png' if statistic=='mean' else '06_monthly_power_heatmap.png')
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always');fig.savefig(output,dpi=180,bbox_inches='tight')
    bad=[str(w.message) for w in caught if 'Glyph' in str(w.message) or 'layout' in str(w.message).lower()]
    assert not bad,bad
    plt.close(fig)
    after=hashlib.sha256(MANUSCRIPT.read_bytes()).hexdigest();assert before==after
    manifest={
        'question':'Does month comparison add information beyond the full-period weekday/weekend curve?',
        'previous_evidence':'Monthly small-multiple curves show differences, but occupy eight panels. User requests assessment and an eight-row monthly heatmap if informative.',
        'decision':'Retain a compact monthly comparison candidate; replace eight-panel monthly line figure as preferred candidate with two eight-row summary heatmaps.',
        'reasons':['Weekday 08 medians: June 171.5, July 187.5.','Weekend 10 medians: January 85.5, March 91.5, February 24, July 23, August 21. Whole-period curves conceal this heterogeneity.'],
        'aggregation':'Arithmetic mean of CSV hourly 平均 over all dates in each month/day-type/hour, including zero-production dates.' if statistic=='mean' else 'Median over dates separately for each month, day type and hour; CSV hourly 平均 used.',
        'statistic':statistic,
        'rows_per_panel':8,'columns_per_panel':24,'checked_cells':384,'color_range':[0,210],
        'month_group_date_counts':d.groupby(['month','group']).n.first().reset_index().to_dict('records'),
        'limit':'Weekend monthly summaries contain only 8-10 dates and may shift with date composition. Descriptive period differences, not proof of a seasonal/weather cause or model gain.',
        'next_or_stop':'Current presentation question answered: preserve monthly differences compactly. Do not add further monthly plots without a new question. Production composition and other-variable explanations belong to separate analysis.',
        'source_file':str(source),'source_sha256':hashlib.sha256(source.read_bytes()).hexdigest(),
        'figure':str(output),'warnings':bad,'visual_review':False,
        'manuscript_sha256':after,'manuscript_edited':False}
    if statistic=='mean':
        manifest['question']='How do monthly average levels and hourly shapes differ within weekdays and weekends?'
        manifest['previous_evidence']='User chose an arithmetic mean view after verifying that low July/August weekend values are genuine and discussing the mean-versus-median purpose.'
        manifest['decision']='Use a mean heatmap as the primary monthly comparison candidate; retain the median version as a supplementary calculation.'
        manifest['reasons']=['Consistent with section 2.1 arithmetic means.', 'August weekday 08: mean 142.6818 versus median 172.5; retain genuine low-power dates.']
        manifest['validation']='384 source-derived means retained; weighted recombination over months reconciled all 48 full-period day-type/hour means with section 2.1; common untruncated color range 0..210.'
        manifest['proposed_section_order']=['2.1 weekday/weekend daily pattern','2.2 monthly power changes','2.3 production/weather relationships']
    (TAB/('monthly_mean_heatmap_manifest.json' if statistic=='mean' else 'monthly_heatmap_manifest.json')).write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps({'figure':str(output),'cells':384,'warnings':bad,'manuscript_edited':False},ensure_ascii=False))

if __name__=='__main__':
    parser=argparse.ArgumentParser()
    parser.add_argument('--statistic',choices=['mean','median'],default='mean')
    main(parser.parse_args().statistic)


코드 등록 완료: `EDA/scripts/plot_monthly_power_heatmap.py`

### `EDA/scripts/plot_report_daily_patterns.py`

원고 연결: EDA\02_EDA_3페이지용_압축본.md.

코드 설명: Combine validated daily means and date ranges for the compact EDA manuscript.

Run from the project root with regular CPython 3.13. Existing tables, figures,
and the detailed manuscript are read-only. Review thumbnails go to tmp.

주요 함수: `sha()`, `load_validated_tables()`, `plot_combined()`, `main()`.

In [12]:
%%submission_source EDA/scripts/plot_report_daily_patterns.py
"""Combine validated daily means and date ranges for the compact EDA manuscript.

Run from the project root with regular CPython 3.13. Existing tables, figures,
and the detailed manuscript are read-only. Review thumbnails go to tmp.
"""
from pathlib import Path
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
from PIL import Image

import eda_figures as style

ROOT = Path(__file__).resolve().parents[2]
BASE = ROOT / "EDA"
DETAIL = BASE / "02_EDA_원고.md"
TARGET = BASE / "figures/report_compact/daily_pattern_distribution.png"
TABLES = [BASE / "tables/daily_pattern_restart.csv",
          BASE / "tables/daily_repetition/hourly_distribution.csv"]


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def load_validated_tables():
    means = pd.read_csv(TABLES[0], encoding="utf-8-sig")
    ranges = pd.read_csv(TABLES[1], encoding="utf-8-sig")
    assert len(means) == len(ranges) == 48
    for group, days in [("weekday", 171), ("weekend", 70)]:
        m = means.loc[means.population.eq(group)].sort_values("hour")
        r = ranges.loc[ranges.group.eq(group)].sort_values("hour")
        np.testing.assert_array_equal(m.hour, np.arange(24))
        np.testing.assert_array_equal(r.hour, np.arange(24))
        assert m.n.eq(days).all() and r.n.eq(days).all()
        np.testing.assert_allclose(m.power_mean, r["mean"], rtol=1e-12)
        assert (r["min"] <= r.p10).all()
        assert (r.p10 <= r.p90).all() and (r.p90 <= r["max"]).all()
    return means, ranges


def plot_combined(means, ranges):
    """Edit panel geometry, lines, ranges, labels, and legends here."""
    style.setup()
    fig = style.plt.figure(figsize=(11.4, 6.1), layout="constrained")
    grid = fig.add_gridspec(2, 2, height_ratios=[0.85, 1])
    production = fig.add_subplot(grid[0, :])
    weekday = fig.add_subplot(grid[1, 0])
    weekend = fig.add_subplot(grid[1, 1], sharey=weekday)
    specs = [("weekday", "평일 (171일)", style.BLUE, "-"),
             ("weekend", "주말 (70일)", style.ORANGE, "--")]
    for group, label, color, line in specs:
        m = means.loc[means.population.eq(group)].sort_values("hour")
        production.plot(m.hour, m.production_mean, color=color, linestyle=line,
                        linewidth=2.2, label=label)
    production.set(ylabel="평균 생산량", ylim=(0, None))
    production.legend(loc="upper left", ncol=2, frameon=False)
    for ax, (group, label, color, line) in zip([weekday, weekend], specs):
        r = ranges.loc[ranges.group.eq(group)].sort_values("hour")
        ax.fill_between(r.hour, r.p10, r.p90, color=color, alpha=0.16,
                        label="날짜별 10~90백분위")
        ax.plot(r.hour, r["mean"], color=color, linestyle=line,
                linewidth=2.2, label="평균")
        ax.set(title=label, ylabel="시간 평균 전력", ylim=(0, 225),
               yticks=[0, 50, 100, 150, 200])
        ax.legend(loc="upper left", fontsize=9, frameon=False)
    for ax in [production, weekday, weekend]:
        ax.set(xlim=(0, 23), xticks=[0, 3, 6, 9, 12, 15, 18, 21, 23],
               xlabel="시간대 (시)")
        ax.grid(axis="y", alpha=0.15)
        ax.set_axisbelow(True)
    assert fig._suptitle is None and not fig.texts
    assert all(not ax.texts for ax in fig.axes)
    TARGET.parent.mkdir(parents=True, exist_ok=True)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        fig.savefig(TARGET, dpi=180)
    style.plt.close(fig)
    bad = [str(w.message) for w in caught
           if "Glyph" in str(w.message) or "layout" in str(w.message).lower()]
    assert not bad, bad


def main():
    protected = [DETAIL, *TABLES]
    before = {str(p): sha(p) for p in protected}
    means, ranges = load_validated_tables()
    plot_combined(means, ranges)
    qa = ROOT / "tmp/eda_report_compact"
    qa.mkdir(parents=True, exist_ok=True)
    with Image.open(TARGET) as image:
        image.convert("RGB").resize((image.width // 2, image.height // 2),
                                    Image.Resampling.LANCZOS).save(qa / "daily_50.png")
    assert before == {str(p): sha(p) for p in protected}
    result = dict(status="passed", checked_hour_group_cells=48,
                  mean_series_match=True, input_tables_and_detailed_manuscript_unchanged=True,
                  input_sha256=before, figure=str(TARGET), figure_sha256=sha(TARGET),
                  visual_review=False,
                  display="Production means; separate weekday/weekend power means and P10-P90; no median lines")
    output = BASE / "tables/report_compact"
    output.mkdir(parents=True, exist_ok=True)
    (output / "daily_pattern_manifest.json").write_text(
        json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps({k: v for k, v in result.items() if k != "input_sha256"}, ensure_ascii=False))


if __name__ == "__main__":
    main()


코드 등록 완료: `EDA/scripts/plot_report_daily_patterns.py`

### `EDA/scripts/verify_monthly_power_heatmap.py`

원고 연결: EDA\02_EDA_3페이지용_압축본.md; EDA\02_EDA_원고.md.

코드 설명: Independently audit monthly plotted medians against the original CSV.

주요 함수: `save()`, `main()`.

In [13]:
%%submission_source EDA/scripts/verify_monthly_power_heatmap.py
"""Independently audit monthly plotted medians against the original CSV."""
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT=Path(__file__).resolve().parents[2]
SOURCE=ROOT/'data/origin/okm_augumented_2021.csv'
TAB=ROOT/'EDA/tables/daily_repetition'
SHA='8f7af2e49366c93e1d6f5fdef4b5e350066c1792ac463c2c2886e370f4674830'
MANUSCRIPT=ROOT/'EDA/02_EDA_원고.md'

def save(name,d):
    d.to_csv(TAB/(name+'.csv'),index=False,encoding='utf-8-sig')

def main():
    before=hashlib.sha256(MANUSCRIPT.read_bytes()).hexdigest()
    assert hashlib.sha256(SOURCE.read_bytes()).hexdigest()==SHA
    raw=pd.read_csv(SOURCE,encoding='utf-8-sig')
    v=raw.loc[raw['날짜'].between(20210101,20210831)&raw['시간'].between(0,23)].copy()
    slots=['15분','30분','45분','60분']
    assert len(v)==5784 and not v.duplicated(['날짜','시간']).any()
    assert v.groupby('날짜').size().eq(24).all()
    np.testing.assert_array_equal(np.floor(v[slots].to_numpy().mean(axis=1)+.5),v['평균'])
    date=pd.to_datetime(v['날짜'].astype(str),format='%Y%m%d')
    v['month']=date.dt.month
    v['group']=np.where(date.dt.dayofweek<5,'weekday','weekend')
    v['hour_max']=v[slots].max(axis=1)
    v['date_positive']=v.groupby('날짜')['생산량'].transform(lambda x:x.gt(0).any())
    # Fresh grouping directly from source rows, no matrix/table helper imports.
    cells=[]
    for (m,g,h),p in v.groupby(['month','group','시간']):
        values=p['평균'].to_numpy()
        cells.append({'month':int(m),'group':g,'hour':int(h),'n_raw':len(values),
            'median_raw':float(np.median(values)),'mean_raw':float(np.mean(values)),
            'min_raw':int(values.min()),'max_raw':int(values.max()),
            'max_four_slots':int(p.hour_max.max()),
            'raw_ge_200':int(p['평균'].ge(200).sum())})
    cells=pd.DataFrame(cells)
    stored=pd.read_csv(TAB/'monthly_hourly_distribution.csv',encoding='utf-8-sig')
    check=cells.merge(stored,on=['month','group','hour'],how='outer',validate='one_to_one',indicator=True)
    assert len(check)==384 and check['_merge'].eq('both').all()
    assert check.n_raw.eq(check.n).all()
    np.testing.assert_array_equal(check.median_raw,check['median'])
    save('independent_monthly_cells',cells)
    month_rows=[];hours=[];production=[]
    for m in [7,8]:
        for g in ['all','weekday','weekend']:
            p=v.loc[v.month.eq(m)]
            if g!='all':p=p.loc[p.group.eq(g)]
            month_rows.append({'month':m,'group':g,'days':p['날짜'].nunique(),'hours':len(p),
                'all_hours_average_power':float(p['평균'].mean()),
                'all_hours_median_power':float(p['평균'].median()),
                'largest_hourly_average':int(p['평균'].max()),
                'largest_15minute_value':int(p[slots].to_numpy().max()),
                'hours_with_average_ge_200':int(p['평균'].ge(200).sum())})
        for g in ['weekday','weekend']:
            for h in [8,9,10,11,12,13,14]:
                p=v.loc[v.month.eq(m)&v.group.eq(g)&v['시간'].eq(h)]
                hours.append({'month':m,'group':g,'hour':h,'n':len(p),'arithmetic_mean':float(p['평균'].mean()),
                    'plotted_median':float(np.median(p['평균'])),'maximum_hour_average':int(p['평균'].max()),
                    'maximum_15minute_value':int(p[slots].to_numpy().max())})
        for positive in [False,True]:
            for h in [8,10,13]:
                p=v.loc[v.month.eq(m)&v.group.eq('weekday')&v['시간'].eq(h)&v.date_positive.eq(positive)]
                production.append({'month':m,'hour':h,'any_positive_production_on_date':positive,'days':len(p),
                    'mean_hour_average':float(p['평균'].mean()) if len(p) else None,
                    'median_hour_average':float(p['평균'].median()) if len(p) else None})
    save('july_august_levels',pd.DataFrame(month_rows))
    save('july_august_hour_examples',pd.DataFrame(hours))
    save('july_august_production_composition',pd.DataFrame(production))
    maxima=[]
    for m in [7,8]:
        p=v.loc[v.month.eq(m)]
        row=p.loc[p['평균'].idxmax()]
        maxima.append({'month':m,'date':int(row['날짜']),'hour':int(row['시간']),
            'four_values':[int(row[x]) for x in slots],'hour_average':int(row['평균'])})
    result={'status':'passed','source_sha256':SHA,'raw_valid_hours':len(v),
        'independently_checked_heatmap_cells':len(check),'max_absolute_median_error':float((check.median_raw-check['median']).abs().max()),
        'month_levels':month_rows,'hour_08_examples':[x for x in hours if x['hour']==8],
        'hour_average_maximum_examples':maxima,
        'weekday_production_composition':[x for x in production if x['hour']==8],
        'july_august_weekend_by_date':v.loc[v.month.isin([7,8])&v.group.eq('weekend')].groupby('날짜')['평균'].mean().to_dict(),
        'judgment':'All 384 displayed monthly/hour/day-type medians and counts match the source. Values near or above 200 do occur; a maximum observation, monthly/hourly arithmetic mean, and monthly/hourly median describe different quantities. August all-hour means include low-power dates; do not silently remove them.',
        'manuscript_edited':False}
    assert hashlib.sha256(MANUSCRIPT.read_bytes()).hexdigest()==before
    (TAB/'independent_heatmap_verification.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
    result.pop('july_august_weekend_by_date')
    print(json.dumps(result,ensure_ascii=False,indent=2))

if __name__=='__main__':main()


코드 등록 완료: `EDA/scripts/verify_monthly_power_heatmap.py`

## Analysis 원고에 사용한 전체 코드

아래 셀은 코드 정의를 등록한다. 계산 순서와 이번 실행 결과는 뒤의 1 ~ 5장에 정리한다.

### `Analysis/scripts/a01_production_changes.py`

원고 연결: Analysis\03_Analysis_3페이지용_압축본.md; Analysis\03_Analysis_원고.md.

코드 설명: Approved Analysis 3.1: production changes and observed power variability.
No oracle maximum reconstruction, forecast fitting, backup, or manuscript.

주요 함수: `save()`, `load()`, `summaries()`, `contrast()`, `verify()`, `followup()`, `main()`.

In [14]:
%%submission_source Analysis/scripts/a01_production_changes.py
"""Approved Analysis 3.1: production changes and observed power variability.
No oracle maximum reconstruction, forecast fitting, backup, or manuscript.
"""
from pathlib import Path
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
import csv
import hashlib
import json
import numpy as np
import pandas as pd
from scipy.stats import rankdata

ROOT=Path(__file__).resolve().parents[2]
SOURCE=ROOT/'data/origin/okm_augumented_2021.csv'
OUT=ROOT/'Analysis/tables/a01_production_changes'
SHA='8f7af2e49366c93e1d6f5fdef4b5e350066c1792ac463c2c2886e370f4674830'
SLOTS=['15분','30분','45분','60분']
METRICS=['power_delta','abs_power_delta','slot_range']
PAIRS=[('increase','maintain'),('decrease','maintain'),('increase','decrease')]


def save(frame,name):
    frame.to_csv(OUT/f'{name}.csv',index=False,encoding='utf-8-sig')


def load():
    assert hashlib.sha256(SOURCE.read_bytes()).hexdigest()==SHA
    raw=pd.read_csv(SOURCE,encoding='utf-8-sig')
    data=raw.loc[raw['날짜'].between(20210101,20210831)&raw['시간'].between(0,23)].copy()
    data['date']=pd.to_datetime(data['날짜'].astype(str),format='%Y%m%d')
    data['timestamp']=data.date+pd.to_timedelta(data['시간'],unit='h')
    data=data.sort_values('timestamp').reset_index(drop=True)
    assert len(data)==5784 and not data.timestamp.duplicated().any()
    data['level']=data[SLOTS].mean(axis=1)
    data['slot_range']=data[SLOTS].max(axis=1)-data[SLOTS].min(axis=1)
    daily={date:hashlib.sha256(cell[SLOTS].to_numpy(dtype=np.int64).tobytes()).hexdigest()
           for date,cell in data.groupby('date')}
    data['profile']=data.date.map(daily)
    data['month']=data.date.dt.month
    data['weekend']=data.date.dt.dayofweek.ge(5).astype(int)
    data['week']=data.date-pd.to_timedelta(data.date.dt.dayofweek,unit='D')
    old=data.set_index('timestamp').reindex(data.timestamp-pd.Timedelta(hours=1))
    data['past_production']=old['생산량'].to_numpy()
    data['past_level']=old.level.to_numpy()
    data['past_profile']=old.profile.to_numpy()
    part=data.dropna(subset=['past_production','past_level']).copy().reset_index(drop=True)
    part['production_delta']=part['생산량']-part.past_production
    part['power_delta']=part.level-part.past_level
    part['abs_power_delta']=part.power_delta.abs()
    part['change']=np.select([part.production_delta.gt(0),part.production_delta.lt(0)],['increase','decrease'],default='maintain')
    current=part['생산량'].gt(0)
    previous=part.past_production.gt(0)
    part['transition']=np.select([~previous&current,previous&~current,previous&current],
                                ['zero_to_positive','positive_to_zero','positive_to_positive'],default='zero_to_zero')
    # Joint current/previous day profile reflects the boundary used by differences.
    part['pair_profile']=part.past_profile+'|'+part.profile
    counts=part[['date','pair_profile']].drop_duplicates().pair_profile.value_counts()
    part['profile_weight']=1/part.pair_profile.map(counts)
    return data,part


def summaries(part):
    rows=[]
    for label,keys in [('change',['change']),('transition',['transition']),
                       ('change_transition',['change','transition']),
                       ('hour_change',['weekend','시간','change']),('month_change',['month','change'])]:
        for key,cell in part.groupby(keys,sort=True):
            key=key if isinstance(key,tuple) else (key,)
            for metric in METRICS:
                rows.append(dict(scope=label,**dict(zip(keys,key)),metric=metric,n=len(cell),days=cell.date.nunique(),
                    mean=cell[metric].mean(),median=cell[metric].median(),
                    q10=cell[metric].quantile(.1),q90=cell[metric].quantile(.9)))
    return pd.DataFrame(rows)


def contrast(part,a,b,metric,mode,boot=False):
    sample=part.loc[part.change.isin([a,b])].copy()
    # Only observed matched conditions; no regression or extrapolation.
    cells=sample.groupby(['month','weekend','시간','change']).date.nunique().unstack('change',fill_value=0)
    if a not in cells or b not in cells:
        return None
    support=cells.index[(cells[a]>=2)&(cells[b]>=2)]
    cell_keys=list(zip(sample.month,sample.weekend,sample['시간']))
    support_map={key:i for i,key in enumerate(support)}
    keep=np.array([key in support_map for key in cell_keys])
    sample=sample.loc[keep].copy()
    if sample.empty:
        return None
    cell_ids=np.array([support_map[key] for key,k in zip(cell_keys,keep) if k])
    side=sample.change.eq(b).to_numpy().astype(int)
    slot=cell_ids*2+side
    n_cells=len(support)
    base_w=np.ones(len(sample)) if mode=='date_hour' else sample.profile_weight.to_numpy()
    value=sample[metric].to_numpy()
    # Common calendar composition: pooled eligible-hour weight within each cell.
    fixed_cell_w=np.bincount(cell_ids,weights=base_w,minlength=n_cells)
    def estimate(extra=None):
        w=base_w if extra is None else base_w*extra
        count=np.bincount(slot,weights=w,minlength=n_cells*2).reshape(-1,2)
        total=np.bincount(slot,weights=w*value,minlength=n_cells*2).reshape(-1,2)
        ok=(count>0).all(axis=1)
        if not ok.any():
            return np.nan,np.nan
        means=total[ok]/count[ok]
        averaged=np.average(means,axis=0,weights=fixed_cell_w[ok])
        return float(averaged[0]),float(averaged[1])
    ma,mb=estimate()
    result=dict(a=a,b=b,metric=metric,weighting=mode,matched_cells=n_cells,
        n_a=int((side==0).sum()),n_b=int((side==1).sum()),
        eligible_a=int(part.change.eq(a).sum()),eligible_b=int(part.change.eq(b).sum()),
        days_a=sample.loc[side==0,'date'].nunique(),days_b=sample.loc[side==1,'date'].nunique(),
        adjusted_a=ma,adjusted_b=mb,difference=ma-mb,
        relative_difference=(ma/mb-1) if metric!='power_delta' and mb>0 else np.nan)
    if boot:
        week_id=pd.factorize(sample.week)[0]
        n_weeks=week_id.max()+1
        rng=np.random.default_rng(3103)
        estimates=[]
        for _ in range(800):
            multiplicity=rng.multinomial(n_weeks,np.full(n_weeks,1/n_weeks))
            va,vb=estimate(multiplicity[week_id])
            estimates.append(va-vb)
        result['week_boot_q025'],result['week_boot_q975']=np.nanquantile(estimates,[.025,.975])
        result['bootstrap_finite']=int(np.isfinite(estimates).sum())
    return result


def verify(data,part):
    independent={}
    with SOURCE.open(encoding='utf-8-sig',newline='') as stream:
        for row in csv.DictReader(stream):
            date,hour=int(row['날짜']),int(row['시간'])
            if 20210101<=date<=20210831 and 0<=hour<=23:
                stamp=pd.Timestamp(str(date))+pd.Timedelta(hours=hour)
                independent[stamp]=(int(row['생산량']),tuple(int(row[c]) for c in SLOTS))
    checked=0
    for row in part.itertuples():
        prod,q=independent[row.timestamp]
        past,pq=independent[row.timestamp-pd.Timedelta(hours=1)]
        assert row.production_delta==prod-past
        assert row.power_delta==(sum(q)-sum(pq))/4
        assert row.slot_range==max(q)-min(q)
        checked+=1
    expected=sum(stamp-pd.Timedelta(hours=1) in independent for stamp in independent)
    assert checked==expected and len(data)==5784
    return dict(raw_independent_pairs=checked,unpaired_hours=len(data)-len(part),source_preserved=True)


def followup(part):
    """Observed confounding: unchanged-positive has only four observations."""
    transition=part.copy()
    transition['change']=transition.transition
    rows=[]
    comparisons=[('positive_to_positive','zero_to_zero'),('zero_to_positive','positive_to_positive'),
                 ('positive_to_zero','positive_to_positive')]
    for a,b in comparisons:
        for metric in METRICS:
            for mode in ('date_hour','profile_balanced'):
                row=contrast(transition,a,b,metric,mode,boot=True)
                if row:
                    rows.append(row)
    correlations=[]
    positive=part.loc[part.transition.eq('positive_to_positive')].copy()
    for scope,cell in [('positive_both',positive),('same_day_positive_both',positive.loc[positive['시간'].ne(0)])]:
        ids=pd.factorize(pd.MultiIndex.from_frame(cell[['month','weekend','시간']]))[0]
        for mode in ('date_hour','profile_balanced'):
            w=np.ones(len(cell)) if mode=='date_hour' else cell.profile_weight.to_numpy()
            for metric,xcol in [('power_delta','production_delta'),('abs_power_delta','abs_production_delta'),
                                ('slot_range','abs_production_delta')]:
                x=cell.production_delta.to_numpy()
                if xcol=='abs_production_delta':
                    x=abs(x)
                rx,ry=rankdata(x),rankdata(cell[metric])
                for value in (rx,ry):
                    means=np.bincount(ids,weights=value*w)/np.bincount(ids,weights=w)
                    value-=means[ids]
                rho=float(np.sum(w*rx*ry)/np.sqrt(np.sum(w*rx**2)*np.sum(w*ry**2)))
                correlations.append(dict(scope=scope,weighting=mode,metric=metric,n=len(cell),
                    calendar_centered_rank_correlation=rho))
    save(pd.DataFrame(rows),'transition_matched_contrasts')
    save(pd.DataFrame(correlations),'positive_production_correlations')
    stability=[]
    for label,subset in [('same_day',transition.loc[transition['시간'].ne(0)])]+[
            (f'exclude_month_{m}',transition.loc[transition.month.ne(m)]) for m in range(1,9)]:
        for a,b in comparisons:
            for metric in ('abs_power_delta','slot_range'):
                row=contrast(subset,a,b,metric,'date_hour')
                if row:
                    stability.append(dict(sensitivity=label,**row))
    save(pd.DataFrame(stability),'transition_stability')
    # Independent per-cell grouping directly from the observation CSV.
    raw_groups={}
    with (OUT/'observations.csv').open(encoding='utf-8-sig',newline='') as stream:
        for row in csv.DictReader(stream):
            key=(int(row['month']),int(row['weekend']),int(row['시간']))
            cell=raw_groups.setdefault(key,{})
            cell.setdefault(row['transition'],[]).append((row['date'],float(row['abs_power_delta'])))
    for result in rows:
        if result['weighting']!='date_hour' or result['metric']!='abs_power_delta':
            continue
        a,b=result['a'],result['b']
        wa=wb=total=0.0
        na=nb=0
        for cell in raw_groups.values():
            ca,cb=cell.get(a,[]),cell.get(b,[])
            if len({x[0] for x in ca})<2 or len({x[0] for x in cb})<2:
                continue
            weight=len(ca)+len(cb)
            wa+=weight*sum(x[1] for x in ca)/len(ca)
            wb+=weight*sum(x[1] for x in cb)/len(cb)
            total+=weight
            na+=len(ca)
            nb+=len(cb)
        assert na==result['n_a'] and nb==result['n_b']
        np.testing.assert_allclose([wa/total,wb/total],[result['adjusted_a'],result['adjusted_b']],rtol=1e-12)
    follow=dict(previous_result='2292 of2296 unchanged-production hours have zero in both records; only4 unchanged-positive. Delta-only effect confounded.',
        questions=['Which zero/nonzero transitions account for observed volatility?',
                   'Within positive production, is the magnitude/direction of production change connected to power change after calendar control?'],
        comparison='Same fixed calendar matching, plus rank associations centered within exact month-hour-weekend cells.',
        not_causal=True,no_threshold=True,no_forecast=True,unchanged_positive_conclusion='Too few records for stable matched contrast; do not infer equivalence.')
    (OUT/'followup_contract.json').write_text(json.dumps(follow,indent=2),encoding='utf-8')


def main():
    OUT.mkdir(parents=True,exist_ok=True)
    contract=dict(question='Within the same month/hour/weekday-weekend, do production increases/decreases/unchanged records have different observed power changes?',
        previous_evidence='Current EDA: weekday noon drop/13h rise; production peak19h versus power8h; recurring within-hour differences.',
        scope='2021 Jan-Aug current EDA valid241 days; exact 1h pairing only, including valid midnight boundaries.',
        metrics=METRICS,definitions={'power_delta':'current unrounded four-slot mean minus exact 1h past mean',
        'abs_power_delta':'absolute value of power_delta','slot_range':'current max-min across four slots'},
        production_groups='sign of exact production difference: >0,<0,=0; even 1-unit production difference retained, actual magnitudes described',
        matching='month x hour x weekend; at least two dates in EACH compared group; pooled common-cell weights, fixed prior to bootstrap',
        repeat_sensitivity='Inverse number of dates sharing joint current/previous day power profile',
        zero_sensitivity='0->positive,positive->0,positive->positive,0->0; not operational closure labels',
        robustness='same-day only; both production records positive; leave each month out; week cluster bootstrap800 seed3103',
        intervals='Exploratory week-resampling sensitivity, not proof of independent weeks or causal/statistical significance; duplicated profiles retained',
        pvalues=False,forecast_training=False,oracle_maximum_reconstruction=False,peak_threshold=None,manuscript=False,
        backup_access=False,stop='Resolve matched relationships and confounding, no desired-result selection.',source_sha256=SHA)
    (OUT/'contract.json').write_text(json.dumps(contract,ensure_ascii=False,indent=2),encoding='utf-8')
    data,part=load()
    save(part[['timestamp','date','month','weekend','시간','생산량','past_production',
        'production_delta','change','transition','level','past_level',*METRICS,'profile_weight']],'observations')
    save(summaries(part),'summaries')
    rows=[]
    for scope,cell in [('all',part),('same_day',part.loc[part['시간'].ne(0)]),
                       ('positive_both',part.loc[part.transition.eq('positive_to_positive')])]:
        for a,b in PAIRS:
            for metric in METRICS:
                for mode in ('date_hour','profile_balanced'):
                    row=contrast(cell,a,b,metric,mode,boot=scope=='all')
                    if row:
                        rows.append(dict(scope=scope,**row))
    results=pd.DataFrame(rows)
    save(results,'matched_contrasts')
    stability=[]
    for month in range(1,9):
        cell=part.loc[part.month.ne(month)]
        for a,b in PAIRS:
            for metric in METRICS:
                row=contrast(cell,a,b,metric,'date_hour')
                if row:
                    stability.append(dict(excluded_month=month,**row))
    save(pd.DataFrame(stability),'leave_month_out')
    followup(part)
    verification=verify(data,part)
    verification.update(status='passed',n=len(part),days=part.date.nunique(),
        production_group_counts=part.change.value_counts().to_dict(),
        transition_counts=part.transition.value_counts().to_dict(),
        transition_matching_independently_verified=True,
        source_sha256=SHA,script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        outputs_sha256={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.glob('*.csv')})
    (OUT/'verification.json').write_text(json.dumps(verification,ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps(verification,ensure_ascii=False))
    print('Approved production-change analysis and confounding followup complete.')


if __name__=='__main__':
    main()


코드 등록 완료: `Analysis/scripts/a01_production_changes.py`

### `Analysis/scripts/a01_relative_changes.py`

원고 연결: Analysis\03_Analysis_3페이지용_압축본.md; Analysis\03_Analysis_원고.md.

코드 설명: Scoped relative-change check for approved A01 observation comparisons.

주요 함수: `save()`, `main()`.

In [15]:
%%submission_source Analysis/scripts/a01_relative_changes.py
"""Scoped relative-change check for approved A01 observation comparisons."""
from pathlib import Path
import csv
import hashlib
import json
import numpy as np
import pandas as pd
from a01_production_changes import ROOT,SOURCE,SHA,OUT,contrast,SLOTS

PREFIX='relative_'
A='zero_to_positive'
B='positive_to_positive'


def save(frame,name):
    frame.to_csv(OUT/f'{PREFIX}{name}.csv',index=False,encoding='utf-8-sig')


def main():
    source_obs=OUT/'observations.csv'
    prior_verification=json.loads((OUT/'verification.json').read_text(encoding='utf-8'))
    assert hashlib.sha256(source_obs.read_bytes()).hexdigest()==prior_verification['outputs_sha256']['observations.csv']
    assert hashlib.sha256(SOURCE.read_bytes()).hexdigest()==SHA
    part=pd.read_csv(source_obs,encoding='utf-8-sig')
    part['timestamp']=pd.to_datetime(part.timestamp)
    part['date']=pd.to_datetime(part.date)
    part['week']=part.date-pd.to_timedelta(part.date.dt.dayofweek,unit='D')
    part['change']=part.transition
    part['relative_change_pct']=np.where(part.past_level.gt(0),
        part.abs_power_delta/part.past_level.replace(0,np.nan)*100,np.nan)
    pair=part.loc[part.transition.isin([A,B])].copy()
    support=pair.groupby(['month','weekend','시간','transition']).date.nunique().unstack(fill_value=0)
    support=support.index[(support[A]>=2)&(support[B]>=2)]
    matched=pair.set_index(['month','weekend','시간']).loc[support].reset_index()
    assert len(support)==46
    assert matched.transition.eq(A).sum()==150 and matched.transition.eq(B).sum()==567
    contract=dict(previous='Absolute matched mean changes37.9208 versus30.2138; user asked whether prior power levels influence interpretation.',
        scope='Keep same717 matched records and46 month-hour-weekday/weekend conditions for primary comparison.',
        metric='100*abs(current four-slot mean minus exact1h past mean)/past mean',
        zero_denominator='Undefined, no epsilon imputation; count and retain separately in source observations.',
        primary='Absolute change remains main; relative percent only checks conclusion.',
        low_denominator='Use existing currentEDA low-power band20..26 as sensitivity; no new threshold search or peak definition. Prior mean>26 is supplementary.',
        fixed_tests='Basic; profile balanced; same day; exclude each month; prior mean>26.',
        not_causal=True,forecast=False,manuscript=False,backup=False,source_sha256=SHA,
        observations_sha256=prior_verification['outputs_sha256']['observations.csv'])
    (OUT/'relative_contract.json').write_text(json.dumps(contract,indent=2),encoding='utf-8')
    rows=[]
    cases=[('primary',matched),('same_day',matched.loc[matched['시간'].ne(0)]),
           ('outside_existing_low_band',matched.loc[matched.past_level.gt(26)])]+[
           (f'exclude_month_{m}',matched.loc[matched.month.ne(m)]) for m in range(1,9)]
    for label,sample in cases:
        finite=sample.loc[sample.past_level.gt(0)]
        for metric in ('abs_power_delta','relative_change_pct'):
            for mode in ('date_hour','profile_balanced') if label in ('primary','outside_existing_low_band') else ('date_hour',):
                result=contrast(finite,A,B,metric,mode,boot=label=='primary')
                if result:
                    rows.append(dict(check=label,undefined_excluded=len(sample)-len(finite),**result))
    results=pd.DataFrame(rows)
    save(results,'contrasts')
    detail=[]
    for label,cell in [('all_pair_candidates',pair),('matched_primary',matched)]:
        for group,subset in cell.groupby('transition'):
            valid=subset.loc[subset.past_level.gt(0)]
            low=valid.past_level.between(20,26)
            detail.append(dict(scope=label,transition=group,n=len(subset),zero_denominator=int(subset.past_level.eq(0).sum()),
                past_min=valid.past_level.min(),past_q10=valid.past_level.quantile(.1),past_median=valid.past_level.median(),
                past_q90=valid.past_level.quantile(.9),relative_mean=valid.relative_change_pct.mean(),
                relative_median=valid.relative_change_pct.median(),relative_q90=valid.relative_change_pct.quantile(.9),
                prior_existing_low_band_hours=int(low.sum()),
                low_band_relative_sum_share=float(valid.loc[low,'relative_change_pct'].sum()/valid.relative_change_pct.sum())))
    save(pd.DataFrame(detail),'denominators')
    save(matched[['timestamp','date','month','weekend','시간','transition','past_level','level','abs_power_delta','relative_change_pct','profile_weight']], 'matched_observations')
    # Independent raw integer CSV arithmetic for each relative change.
    raw={}
    with SOURCE.open(encoding='utf-8-sig',newline='') as stream:
        for row in csv.DictReader(stream):
            date,hour=int(row['날짜']),int(row['시간'])
            if 20210101<=date<=20210831 and 0<=hour<=23:
                stamp=pd.Timestamp(str(date))+pd.Timedelta(hours=hour)
                raw[stamp]=sum(int(row[c]) for c in SLOTS)
    for row in matched.itertuples():
        now=raw[row.timestamp]
        old=raw[row.timestamp-pd.Timedelta(hours=1)]
        expected=abs(now-old)/old*100 if old>0 else np.nan
        np.testing.assert_allclose(row.relative_change_pct,expected,rtol=1e-12,equal_nan=True)
    # independent primary same-calendar percent averaging, using Python groups
    cells={}
    with (OUT/'relative_matched_observations.csv').open(encoding='utf-8-sig',newline='') as stream:
        for row in csv.DictReader(stream):
            if float(row['past_level'])<=0:
                continue
            key=(row['month'],row['weekend'],row['시간'])
            cells.setdefault(key,{A:[],B:[]})[row['transition']].append(float(row['relative_change_pct']))
    num_a=num_b=den=0.0
    for cell in cells.values():
        assert cell[A] and cell[B]
        w=len(cell[A])+len(cell[B])
        num_a+=w*sum(cell[A])/len(cell[A])
        num_b+=w*sum(cell[B])/len(cell[B])
        den+=w
    result=results.loc[(results.check=='primary')&(results.metric=='relative_change_pct')&(results.weighting=='date_hour')].iloc[0]
    np.testing.assert_allclose([num_a/den,num_b/den],[result.adjusted_a,result.adjusted_b],rtol=1e-12)
    # Preserve exact original absolute scores when denominators did not exclude rows.
    original=pd.read_csv(OUT/'transition_matched_contrasts.csv',encoding='utf-8-sig')
    for mode in ('date_hour','profile_balanced'):
        prior=original.loc[(original.a==A)&(original.b==B)&(original.metric=='abs_power_delta')&(original.weighting==mode)].iloc[0]
        current=results.loc[(results.check=='primary')&(results.metric=='abs_power_delta')&(results.weighting==mode)].iloc[0]
        if not current.undefined_excluded:
            np.testing.assert_allclose([prior.adjusted_a,prior.adjusted_b],[current.adjusted_a,current.adjusted_b],rtol=1e-12)
    verification=dict(status='passed',original_observations_preserved=True,source_sha256=SHA,
        script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        matched_rows=len(matched),matched_cells=len(support),
        all_zero_denominator_hours=int(part.past_level.eq(0).sum()),
        primary_zero_denominator_hours=int(matched.past_level.eq(0).sum()),
        independent_raw_rows_checked=len(matched),primary_percent_grouping_verified=True,
        outputs_sha256={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.glob('relative_*.csv')})
    (OUT/'relative_verification.json').write_text(json.dumps(verification,indent=2),encoding='utf-8')
    print(results.to_json(orient='records'))
    print(pd.DataFrame(detail).to_json(orient='records'))
    print(json.dumps(verification))


if __name__=='__main__':
    main()


코드 등록 완료: `Analysis/scripts/a01_relative_changes.py`

### `Analysis/scripts/a02_prior_power_signals.py`

원고 연결: Analysis\03_Analysis_원고.md.

코드 설명: Approved 3.2: observed past-power signals for the next hourly change.
Exploratory associations only; no forecast fitting, threshold search or manuscript.

주요 함수: `save()`, `build()`, `weighted_corr()`, `residuals()`, `association()`, `scopes()`, `independent_check()`, `main()`.

In [16]:
%%submission_source Analysis/scripts/a02_prior_power_signals.py
"""Approved 3.2: observed past-power signals for the next hourly change.
Exploratory associations only; no forecast fitting, threshold search or manuscript.
"""
from pathlib import Path
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
import csv
import hashlib
import json
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from a01_production_changes import ROOT, SOURCE, SHA, SLOTS, load, contrast

OUT = ROOT / 'Analysis/tables/a02_prior_power_signals'
FEATURES = ['prior_slot_trend', 'recent_level_delta']


def save(frame, name):
    frame.to_csv(OUT / f'{name}.csv', index=False, encoding='utf-8-sig')


def build():
    data, part = load()
    index = data.set_index('timestamp')
    previous = index.reindex(part.timestamp - pd.Timedelta(hours=1))
    older = index.reindex(part.timestamp - pd.Timedelta(hours=2))
    part['older_level'] = older.level.to_numpy()
    part['prior_slot_trend'] = (previous['60분'] - previous['15분']).to_numpy()
    part['recent_level_delta'] = part.past_level - part.older_level
    part['input_timestamp'] = part.timestamp - pd.Timedelta(hours=1)
    part['older_timestamp'] = part.timestamp - pd.Timedelta(hours=2)
    part['known_production_positive'] = part.past_production.gt(0).astype(int)
    part['prior_profile'] = previous.profile.to_numpy()
    part['older_profile'] = older.profile.to_numpy()
    part = part.dropna(subset=['older_level']).copy().reset_index(drop=True)
    part['triple_profile'] = part.older_profile + '|' + part.prior_profile + '|' + part.profile
    counts = part[['date', 'triple_profile']].drop_duplicates().triple_profile.value_counts()
    part['profile_weight'] = 1 / part.triple_profile.map(counts)
    assert len(part) == 5778
    return data, part


def weighted_corr(x, y, weights):
    x = x - np.average(x, weights=weights)
    y = y - np.average(y, weights=weights)
    denominator = np.sqrt(np.sum(weights*x*x) * np.sum(weights*y*y))
    return float(np.sum(weights*x*y)/denominator) if denominator > 1e-15 else np.nan


def residuals(values, cells, weights, controls=None):
    values = np.asarray(values, dtype=float)
    if values.ndim == 1:
        values = values[:, None]
    n = cells.max()+1
    sums = np.zeros((n, values.shape[1]))
    np.add.at(sums, cells, weights[:, None]*values)
    mass = np.bincount(cells, weights=weights, minlength=n)
    centered = values - (sums/mass[:, None])[cells]
    if controls is None or not controls.shape[1]:
        return centered
    centered_controls = residuals(controls, cells, weights)
    weighted_controls = centered_controls*np.sqrt(weights[:, None])
    coefficients = np.linalg.lstsq(weighted_controls,
                                   centered*np.sqrt(weights[:, None]), rcond=None)[0]
    return centered-centered_controls@coefficients


def association(sample, feature, adjustment, mode):
    sample = sample.copy()
    # All available observations remain in raw summaries. Adjusted associations
    # require at least three distinct dates within each calendar cell.
    cell_keys = ['month', 'weekend', '시간']
    if adjustment != 'raw':
        counts = sample.groupby(cell_keys).date.transform('nunique')
        sample = sample.loc[counts.ge(3)]
    if len(sample) < 10:
        return None
    weights = np.ones(len(sample)) if mode == 'date_hour' else sample.profile_weight.to_numpy()
    x = rankdata(sample[feature].to_numpy())/len(sample)
    y = rankdata(sample.power_delta.to_numpy())/len(sample)
    if adjustment == 'raw':
        value = weighted_corr(x, y, weights)
    else:
        cells = pd.factorize(pd.MultiIndex.from_frame(sample[cell_keys]))[0]
        controls = None
        if adjustment in ('calendar_level', 'calendar_level_both'):
            # Rank-level cubic permits a smooth non-linear adjustment without
            # introducing thresholds or treating the target mean as a control.
            level = rankdata(sample.past_level.to_numpy())/len(sample)
            controls = np.column_stack([level, level**2, level**3,
                                        sample.known_production_positive.to_numpy()])
            if adjustment == 'calendar_level_both':
                other = FEATURES[1] if feature == FEATURES[0] else FEATURES[0]
                controls = np.column_stack([controls, rankdata(sample[other].to_numpy())/len(sample)])
        res = residuals(np.column_stack([x, y]), cells, weights, controls)
        value = weighted_corr(res[:, 0], res[:, 1], weights)
    return dict(feature=feature, adjustment=adjustment, weighting=mode,
                n=len(sample), days=sample.date.nunique(), correlation=value)


def scopes(part):
    return {
        'all': part,
        'known_production_zero': part.loc[part.past_production.eq(0)],
        'known_production_positive': part.loc[part.past_production.gt(0)],
        'zero_to_zero_posthoc': part.loc[part.transition.eq('zero_to_zero')],
        'zero_to_positive_posthoc': part.loc[part.transition.eq('zero_to_positive')],
    }


def independent_check(part):
    raw = {}
    with SOURCE.open(encoding='utf-8-sig', newline='') as stream:
        for row in csv.DictReader(stream):
            date, hour = int(row['날짜']), int(row['시간'])
            if 20210101 <= date <= 20210831 and 0 <= hour <= 23:
                stamp = pd.Timestamp(str(date)) + pd.Timedelta(hours=hour)
                raw[stamp] = (tuple(int(row[c]) for c in SLOTS), int(row['생산량']))
    for row in part.itertuples():
        now, production = raw[row.timestamp]
        prior, prior_production = raw[row.input_timestamp]
        older, _ = raw[row.older_timestamp]
        expected = [sum(now)/4-sum(prior)/4, prior[-1]-prior[0],
                    sum(prior)/4-sum(older)/4, sum(prior)/4]
        np.testing.assert_allclose([row.power_delta, row.prior_slot_trend,
                                    row.recent_level_delta, row.past_level], expected, rtol=0, atol=0)
        assert row.past_production == prior_production
        assert row.input_timestamp < row.timestamp and row.older_timestamp < row.input_timestamp
    a01_path = ROOT/'Analysis/tables/a01_production_changes/observations.csv'
    a01_verification = json.loads((a01_path.parent/'verification.json').read_text(encoding='utf-8'))
    assert hashlib.sha256(a01_path.read_bytes()).hexdigest() == a01_verification['outputs_sha256']['observations.csv']
    a01 = pd.read_csv(a01_path, encoding='utf-8-sig').set_index('timestamp')
    joined = a01.reindex(part.timestamp.astype(str))
    np.testing.assert_allclose(joined.power_delta, part.power_delta, rtol=0, atol=0)


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    contract = dict(
        previous='A01 zero-to-positive absolute change37.92 vs30.21; EDA2.5 shows within-hour patterns.',
        question='Do the last observed hour slot trend and recent mean change precede next-hour mean changes after calendar and observed level adjustment?',
        input_time='Hour t-1 has completely finished and all four slots/production are assumed observed; actual reporting latency is unidentified.',
        target='Unrounded mean(t) minus mean(t-1), signed; never used in controls.',
        features={'prior_slot_trend': 'last minus first slot at t-1',
                  'recent_level_delta': 'mean(t-1) minus mean(t-2)'},
        scope='Same Jan-Aug valid241 days; exact consecutive t-2,t-1,t required; no gaps bridged.',
        primary='Rank associations: raw, calendar, calendar+observed rank-level cubic+known production0/positive, then other-feature adjustment.',
        calendar='Target month x hour x weekday/weekend fixed effects, adjusted scopes need >=3distinct dates per cell.',
        level_adjustment='Cubic of observed rank(mean(t-1)), no target levels, no threshold or parameter search; partial association is descriptive, not a fitted future forecast.',
        sensitivity='Inverse joint3-day profile frequency, same date for all3hours, leave each month out; no pvalue or causal significance.',
        transition='Actual production(t) used only to split posthoc diagnostic groups; cannot be a predictor.',
        next_action='If calendar-only and level-adjusted differ, diagnose scale/coupling; if conditional signal survives use as forecasting candidate, no claim of improvement.',
        forecast_training=False, manuscript=False, backup_access=False,
        source_sha256=SHA)
    (OUT/'contract.json').write_text(json.dumps(contract, indent=2), encoding='utf-8')
    _, part = build()
    independent_check(part)
    selected = ['timestamp', 'input_timestamp', 'older_timestamp', 'date', 'month', 'weekend', '시간',
                'past_production', '생산량', 'known_production_positive', 'transition',
                'older_level', 'past_level', 'level', 'prior_slot_trend', 'recent_level_delta',
                'power_delta', 'abs_power_delta', 'profile_weight']
    save(part[selected], 'observations')
    rows=[]
    for scope, sample in scopes(part).items():
        for feature in FEATURES:
            for adjustment in ('raw', 'calendar', 'calendar_level', 'calendar_level_both'):
                for mode in ('date_hour', 'profile_balanced'):
                    result=association(sample, feature, adjustment, mode)
                    if result:
                        rows.append(dict(scope=scope, **result))
    results=pd.DataFrame(rows)
    save(results, 'associations')
    stability=[]
    for scope, sample in scopes(part).items():
        for label, subset in [('same_date', sample.loc[sample['시간'].ge(2)])]+[
            (f'exclude_month_{month}', sample.loc[sample.month.ne(month)]) for month in range(1,9)]:
            for feature in FEATURES:
                for adjustment in ('calendar_level', 'calendar_level_both'):
                    result=association(subset, feature, adjustment, 'date_hour')
                    if result:
                        stability.append(dict(scope=scope, check=label, **result))
    save(pd.DataFrame(stability), 'stability')
    distributions=[]
    for scope, sample in scopes(part).items():
        for metric in FEATURES+['past_level', 'power_delta']:
            distributions.append(dict(scope=scope, metric=metric, n=len(sample),
                mean=sample[metric].mean(), median=sample[metric].median(),
                q10=sample[metric].quantile(.1), q90=sample[metric].quantile(.9)))
    save(pd.DataFrame(distributions), 'distributions')
    # Match calendar composition to compare observable antecedents among hours
    # whose known production is zero. This is a posthoc diagnostic, not detection.
    risk=part.loc[part.past_production.eq(0)].copy()
    risk['change']=risk.transition
    matched=[]
    for metric in FEATURES+['past_level', 'power_delta']:
        for mode in ('date_hour', 'profile_balanced'):
            result=contrast(risk, 'zero_to_positive', 'zero_to_zero', metric, mode)
            if result:
                matched.append(result)
    save(pd.DataFrame(matched), 'zero_production_antecedents')
    # Projection check against explicit weighted dummy least squares on a
    # deterministic small subset, avoiding duplicate full model computation.
    tiny=part.loc[part.month.eq(1)&part['시간'].isin([7,8,12,13])].copy()
    cells=pd.factorize(pd.MultiIndex.from_frame(tiny[['month','weekend','시간']]))[0]
    w=tiny.profile_weight.to_numpy()
    lev=rankdata(tiny.past_level)/len(tiny)
    controls=np.column_stack([lev, lev**2, lev**3, tiny.known_production_positive])
    values=np.column_stack([rankdata(tiny.prior_slot_trend), rankdata(tiny.power_delta)])/len(tiny)
    computed=residuals(values,cells,w,controls)
    design=np.column_stack([np.eye(cells.max()+1)[cells],controls])
    beta=np.linalg.lstsq(design*np.sqrt(w[:,None]),values*np.sqrt(w[:,None]),rcond=None)[0]
    direct=values-design@beta
    np.testing.assert_allclose(computed,direct,atol=1e-10,rtol=1e-10)
    verification=dict(status='passed',source_sha256=SHA,n=len(part),
        exact_raw_triples_checked=len(part),a01_delta_reproduced=True,
        input_timestamps_precede_target=True,zero_values_retained=True,
        calendar_projection_vs_explicit_dummies=True,
        transition_counts=part.transition.value_counts().to_dict(),
        script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        outputs_sha256={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.glob('*.csv')})
    (OUT/'verification.json').write_text(json.dumps(verification,indent=2),encoding='utf-8')
    print(results.loc[(results.weighting=='date_hour') & results.scope.isin(['all','known_production_zero','zero_to_positive_posthoc'])].to_json(orient='records'))
    print(pd.DataFrame(matched).to_json(orient='records'))
    print(json.dumps(verification))


if __name__=='__main__':
    main()


코드 등록 완료: `Analysis/scripts/a02_prior_power_signals.py`

### `Analysis/scripts/a02_signal_diagnostics.py`

원고 연결: Analysis\03_Analysis_원고.md.

코드 설명: Minimal follow-up: shared-level arithmetic and exact-level sign comparisons.

주요 함수: `matched_sign()`, `main()`.

In [17]:
%%submission_source Analysis/scripts/a02_signal_diagnostics.py
"""Minimal follow-up: shared-level arithmetic and exact-level sign comparisons."""
from pathlib import Path
import csv
import hashlib
import json
import numpy as np
import pandas as pd
from a02_prior_power_signals import OUT, FEATURES, residuals, weighted_corr


def matched_sign(sample, feature, mode):
    # Fixed zero is a direction, not a searched event-size threshold.
    sample=sample.copy()
    sample['side']=np.sign(sample[feature]).astype(int)
    keys=['month','weekend','시간','past_level','known_production_positive']
    sample=sample.loc[sample.side.ne(0)]
    dates=sample.groupby(keys+['side']).date.nunique().unstack(fill_value=0)
    if -1 not in dates or 1 not in dates:
        return None,None
    support=dates.index[(dates[-1]>=2)&(dates[1]>=2)]
    if not len(support):
        return None,None
    ids={key:i for i,key in enumerate(support)}
    allkeys=list(sample[keys].itertuples(index=False,name=None))
    keep=np.array([key in ids for key in allkeys])
    sample=sample.loc[keep].copy()
    cell=np.array([ids[key] for key,k in zip(allkeys,keep) if k])
    side=sample.side.eq(1).to_numpy().astype(int)
    slot=2*cell+side
    w=np.ones(len(sample)) if mode=='date_hour' else sample.profile_weight.to_numpy()
    cell_weight=np.bincount(cell,weights=w,minlength=len(support))
    mass=np.bincount(slot,weights=w,minlength=2*len(support)).reshape(-1,2)
    result=dict(feature=feature,weighting=mode,cells=len(support),n_negative=int((side==0).sum()),
        n_positive=int((side==1).sum()),days=sample.date.nunique())
    for metric,value in [('next_delta',sample.power_delta.to_numpy()),
                         ('next_rise_rate',sample.power_delta.gt(0).to_numpy().astype(float)*100),
                         ('prior_feature',sample[feature].to_numpy()),
                         ('past_level',sample.past_level.to_numpy())]:
        total=np.bincount(slot,weights=w*value,minlength=2*len(support)).reshape(-1,2)
        means=np.average(total/mass,axis=0,weights=cell_weight)
        result[f'{metric}_negative']=means[0]
        result[f'{metric}_positive']=means[1]
    assert abs(result['past_level_negative']-result['past_level_positive'])<1e-10
    return result,sample


def main():
    prior=json.loads((OUT/'verification.json').read_text(encoding='utf-8'))
    path=OUT/'observations.csv'
    assert hashlib.sha256(path.read_bytes()).hexdigest()==prior['outputs_sha256']['observations.csv']
    part=pd.read_csv(path,encoding='utf-8-sig',parse_dates=['timestamp','date'])
    contract=dict(previous='A02 adjusted rank correlation .305 for prior within-hour trend but only .013 within posthoc onset group; onset/zero-continuing prior means81.85 vs39.57.',
        reason='Recent differences share mean(t-1); prior-level group imbalance requires transparent conditioning, not an onset detection claim.',
        checks='Numeric-scale partial correlation with exact observed level polynomial; target-level and target-change residual identity; exact prior-level/calendar/known-production sign comparisons.',
        scope='Reuse immutable A02 observations only, no forecasting or threshold search.',
        exact_matching='month x upcoming hour x weekend x exact mean(t-1) x known production0/positive; >=2 dates in each sign; zero trend retained in source but outside opposite-sign comparison.',
        decision='If exact matching has little coverage report it; do not relax definitions for larger differences. If sign relationship reverses under repeat weighting do not promote it.',
        manuscript=False,backup=False,forecast=False)
    (OUT/'diagnostic_contract.json').write_text(json.dumps(contract,indent=2),encoding='utf-8')
    associations=[]
    form_sensitivity=[]
    for scope,sample in [('all',part),('known_production_zero',part.loc[part.past_production.eq(0)]),
                         ('zero_to_positive_posthoc',part.loc[part.transition.eq('zero_to_positive')])]:
        counts=sample.groupby(['month','weekend','시간']).date.transform('nunique')
        sample=sample.loc[counts.ge(3)].copy()
        cells=pd.factorize(pd.MultiIndex.from_frame(sample[['month','weekend','시간']]))[0]
        level=sample.past_level.to_numpy()
        level=(level-level.mean())/level.std()
        controls=np.column_stack([level,level**2,level**3,sample.known_production_positive])
        # Test whether the primary finding relies on the cubic adjustment.
        for form in ('linear_rank','cubic_rank','linear_numeric','cubic_numeric'):
            if 'rank' in form:
                from scipy.stats import rankdata
                lev=rankdata(sample.past_level.to_numpy())/len(sample)
            else:
                lev=level
            parts=[lev, sample.known_production_positive.to_numpy()]
            if 'cubic' in form:
                parts.extend([lev**2,lev**3])
            c=np.column_stack(parts)
            for feature in FEATURES:
                if 'rank' in form:
                    values=np.column_stack([rankdata(sample[feature]),rankdata(sample.power_delta)])/len(sample)
                else:
                    values=np.column_stack([sample[feature],sample.power_delta])
                res=residuals(values,cells,np.ones(len(sample)),c)
                form_sensitivity.append(dict(scope=scope,feature=feature,form=form,n=len(sample),
                    correlation=weighted_corr(res[:,0],res[:,1],np.ones(len(sample)))))
        for mode in ('date_hour','profile_balanced'):
            w=np.ones(len(sample)) if mode=='date_hour' else sample.profile_weight.to_numpy()
            for feature in FEATURES:
                for both in (False,True):
                    c=controls
                    if both:
                        other=FEATURES[1] if feature==FEATURES[0] else FEATURES[0]
                        c=np.column_stack([c,sample[other]])
                    values=np.column_stack([sample[feature],sample.power_delta,sample.level])
                    res=residuals(values,cells,w,c)
                    # mean(t)-mean(t-1) and mean(t) residuals must coincide when
                    # the exact observed mean(t-1) is included in the controls.
                    np.testing.assert_allclose(res[:,1],res[:,2],atol=1e-9,rtol=1e-9)
                    associations.append(dict(scope=scope,feature=feature,both_features=both,weighting=mode,
                        n=len(sample),correlation=weighted_corr(res[:,0],res[:,1],w),
                        next_level_correlation=weighted_corr(res[:,0],res[:,2],w)))
    pd.DataFrame(associations).to_csv(OUT/'numeric_associations.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(form_sensitivity).to_csv(OUT/'adjustment_form_sensitivity.csv',index=False,encoding='utf-8-sig')
    comparisons=[]
    for scope,sample in [('all',part),('known_production_zero',part.loc[part.past_production.eq(0)])]:
        for feature in FEATURES:
            for mode in ('date_hour','profile_balanced'):
                result,_=matched_sign(sample,feature,mode)
                if result:
                    comparisons.append(dict(scope=scope,**result))
    pd.DataFrame(comparisons).to_csv(OUT/'exact_level_sign_comparisons.csv',index=False,encoding='utf-8-sig')
    # Independent grouping arithmetic for the main exact-matching contrast.
    mainrow=next((row for row in comparisons if row['scope']=='all' and row['feature']==FEATURES[0] and row['weighting']=='date_hour'),None)
    groups={}
    with path.open(encoding='utf-8-sig',newline='') as stream:
        for row in csv.DictReader(stream):
            value=float(row[FEATURES[0]])
            if not value:
                continue
            key=(row['month'],row['weekend'],row['시간'],row['past_level'],row['known_production_positive'])
            side=int(value>0)
            groups.setdefault(key,[[],[]])[side].append((row['date'],float(row['power_delta'])))
    numerator=np.zeros(2)
    denominator=0
    totals=np.zeros(2,dtype=int)
    for sides in groups.values():
        if not all(len({date for date,val in side})>=2 for side in sides):
            continue
        w=sum(len(side) for side in sides)
        numerator+=w*np.array([sum(v for d,v in side)/len(side) for side in sides])
        denominator+=w
        totals+=np.array([len(side) for side in sides])
    if mainrow:
        np.testing.assert_allclose(numerator/denominator,[mainrow['next_delta_negative'],mainrow['next_delta_positive']],atol=1e-10)
        assert list(totals)==[mainrow['n_negative'],mainrow['n_positive']]
    else:
        assert denominator==0 and totals.sum()==0
    support=[]
    for scope,sample in [('all',part),('known_production_zero',part.loc[part.past_production.eq(0)])]:
        for feature in FEATURES:
            sample=sample.copy()
            sample['side']=np.sign(sample[feature]).astype(int)
            keys=['month','weekend','시간','past_level','known_production_positive']
            dates=sample.loc[sample.side.ne(0)].groupby(keys+['side']).date.nunique().unstack(fill_value=0)
            neg=dates[-1] if -1 in dates else pd.Series(0,index=dates.index)
            pos=dates[1] if 1 in dates else pd.Series(0,index=dates.index)
            support.append(dict(scope=scope,feature=feature,opposite_sign_common_cells=int(((neg>0)&(pos>0)).sum()),
                minimum_two_dates_cells=int(((neg>=2)&(pos>=2)).sum()),direction_zero_hours=int(sample.side.eq(0).sum())))
    pd.DataFrame(support).to_csv(OUT/'exact_level_support.csv',index=False,encoding='utf-8-sig')
    verification=dict(status='passed',a02_observations_unchanged=True,
        exact_level_residual_identity_verified=True,exact_sign_comparison_independent=bool(mainrow),
        main_sign_stop_reason=None if mainrow else 'No exact-level common cells with at least2dates in each opposite-sign group; no threshold relaxation.',
        main_sign_n=int(totals.sum()),script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        outputs_sha256={name:hashlib.sha256((OUT/name).read_bytes()).hexdigest()
            for name in ['numeric_associations.csv','exact_level_sign_comparisons.csv','exact_level_support.csv','adjustment_form_sensitivity.csv']})
    (OUT/'diagnostic_verification.json').write_text(json.dumps(verification,indent=2),encoding='utf-8')
    print(pd.DataFrame(associations).to_json(orient='records'))
    print(pd.DataFrame(comparisons).to_json(orient='records'))
    print(pd.DataFrame(support).to_json(orient='records'))
    print(pd.DataFrame(form_sensitivity).to_json(orient='records'))
    print(json.dumps(verification))


if __name__=='__main__':
    main()


코드 등록 완료: `Analysis/scripts/a02_signal_diagnostics.py`

### `Analysis/scripts/a02_transition_forecast.py`

원고 연결: Analysis\03_Analysis_3페이지용_압축본.md; Analysis\03_Analysis_원고.md.

코드 설명: Approved extension: chronological onset discrimination and power forecasting.
No future production features or in-sample stacking probabilities.

주요 함수: `write_json()`, `load()`, `matrix()`, `classifier()`, `threshold()`, `classify_metrics()`, `class_validation()`, `probability_history()`, `regression_features()`, `regressor()`, `soft_mixture()`, `mixture_validation()`, `power_metrics()`, `power_validation()`, `final_evaluation()`, `verify()`, `main()`.

In [18]:
%%submission_source Analysis/scripts/a02_transition_forecast.py
"""Approved extension: chronological onset discrimination and power forecasting.
No future production features or in-sample stacking probabilities.
"""
from pathlib import Path
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('OMP_NUM_THREADS','1')
import argparse
import hashlib
import json
import csv
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, precision_recall_curve

ROOT=Path(__file__).resolve().parents[2]
INPUT=ROOT/'Analysis/tables/a02_prior_power_signals/observations.csv'
OUT=ROOT/'Analysis/tables/a02_transition_forecast'
SETS={'calendar':[], 'level':['past_level'], 'shape':['past_level','prior_slot_trend']}
FAMILIES=['logistic','hgb']


def write_json(name,value):
    (OUT/f'{name}.json').write_text(json.dumps(value,indent=2,ensure_ascii=False),encoding='utf-8')


def load():
    manifest=json.loads((INPUT.parent/'verification.json').read_text(encoding='utf-8'))
    assert hashlib.sha256(INPUT.read_bytes()).hexdigest()==manifest['outputs_sha256']['observations.csv']
    part=pd.read_csv(INPUT,encoding='utf-8-sig',parse_dates=['timestamp','input_timestamp','older_timestamp','date'])
    assert len(part)==5778
    assert (part.timestamp-part.input_timestamp).eq(pd.Timedelta(hours=1)).all()
    assert (part.input_timestamp-part.older_timestamp).eq(pd.Timedelta(hours=1)).all()
    part['onset']=part.transition.eq('zero_to_positive').astype(int)
    assert (part.loc[part.onset.eq(1),'past_production']==0).all()
    part['known_zero']=part.past_production.eq(0).astype(int)
    part['past_production_log']=np.log1p(part.past_production)
    return part


def matrix(frame,features):
    # Fixed calendar categories, no fitted target-derived transformations.
    hours=np.eye(24)[frame['시간'].to_numpy(dtype=int)]
    return np.column_stack([hours,frame.weekend,frame.month]+[frame[f] for f in features])


def classifier(family):
    if family=='logistic':
        return make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=2000,random_state=3203))
    return HistGradientBoostingClassifier(max_iter=150,max_leaf_nodes=7,min_samples_leaf=20,
        l2_regularization=1,learning_rate=.05,early_stopping=False,random_state=3203)


def threshold(y,prob):
    precision,recall,cuts=precision_recall_curve(y,prob)
    f1=2*precision[:-1]*recall[:-1]/np.maximum(precision[:-1]+recall[:-1],1e-12)
    best=np.flatnonzero(np.isclose(f1,f1.max()))[-1]
    return float(cuts[best])


def classify_metrics(y,prob,cut):
    y=np.asarray(y,dtype=int)
    pred=np.asarray(prob)>=cut
    tp=int((pred & (y==1)).sum()); fp=int((pred & (y==0)).sum()); fn=int((~pred & (y==1)).sum())
    precision=tp/(tp+fp) if tp+fp else 0
    recall=tp/(tp+fn) if tp+fn else 0
    return dict(n=len(y),events=int(y.sum()),prevalence=float(y.mean()),
        ap=float(average_precision_score(y,prob)),roc_auc=float(roc_auc_score(y,prob)) if len(np.unique(y))>1 else None,
        brier=float(brier_score_loss(y,prob)),threshold=cut,tp=tp,fp=fp,fn=fn,
        precision=precision,recall=recall,f1=2*precision*recall/(precision+recall) if precision+recall else 0)


def class_validation(part):
    risk=part.loc[part.known_zero.eq(1)].copy()
    train=risk.loc[risk.month.le(4)]
    valid=risk.loc[risk.month.between(5,6)]
    rows=[]; predictions=valid[['timestamp','date','month','onset','profile_weight']].copy()
    for family in FAMILIES:
        for group,features in SETS.items():
            model=classifier(family)
            model.fit(matrix(train,features),train.onset)
            prob=model.predict_proba(matrix(valid,features))[:,1]
            cut=threshold(valid.onset,prob)
            rows.append(dict(family=family,features=group,**classify_metrics(valid.onset,prob,cut)))
            predictions[f'{family}_{group}']=prob
    results=pd.DataFrame(rows)
    results.to_csv(OUT/'classification_validation.csv',index=False,encoding='utf-8-sig')
    predictions.to_csv(OUT/'classification_validation_predictions.csv',index=False,encoding='utf-8-sig')
    # Select a common family using richer-input validation AP, then keep family
    # constant in the ablation. No July/August outcome consulted here.
    best=results.loc[results.features.eq('shape')].sort_values('ap',ascending=False).iloc[0]
    chosen=results.loc[results.family.eq(best.family)]
    best_ap=chosen.ap.max()
    selected=next(name for name in SETS if float(chosen.loc[chosen.features.eq(name),'ap'].iloc[0])>=best_ap-.01)
    baseline=chosen.loc[chosen.features.eq('calendar')].iloc[0]
    selected_row=chosen.loc[chosen.features.eq(selected)].iloc[0]
    gate=selected!='calendar' and selected_row.ap>baseline.ap and selected_row.brier<=1.05*baseline.brier
    frozen=dict(family=str(best.family),selected_features=selected,power_gate=bool(gate),
        reason='Validation AP within0.01 of best favors fewer inputs; same classifier family for ablations. Power followup requires AP above calendar and Brier no worse than5%.',
        thresholds={name:float(chosen.loc[chosen.features.eq(name),'threshold'].iloc[0]) for name in SETS},
        validated_before_test=True,test_consulted=False,
        classifier_parameters='Fixed logistic C1 or HGB150iterations/7leaves/min20/l2=1/lr.05; no tuning grid.',
        train_n=len(train),train_events=int(train.onset.sum()),validation_n=len(valid),validation_events=int(valid.onset.sum()))
    write_json('frozen',frozen)
    print(results.to_json(orient='records')); print(json.dumps(frozen))


def probability_history(part,frozen):
    risk=part.loc[part.known_zero.eq(1)]
    probabilities=pd.Series(0.,index=part.index)
    audit=[]
    features=SETS[frozen['selected_features']]
    for month in range(3,9):
        # March-June training meta-features see strictly earlier months.
        # July/August both use a final classifier fitted through June.
        cutoff=month-1 if month<=6 else 6
        train=risk.loc[risk.month.le(cutoff)]
        target=risk.loc[risk.month.eq(month)]
        model=classifier(frozen['family'])
        model.fit(matrix(train,features),train.onset)
        probabilities.loc[target.index]=model.predict_proba(matrix(target,features))[:,1]
        assert train.timestamp.max()<target.timestamp.min()
        audit.append(dict(target_month=month,train_last_timestamp=str(train.timestamp.max()),
            target_first_timestamp=str(target.timestamp.min()),train_n=len(train),train_events=int(train.onset.sum()),target_n=len(target)))
    pd.DataFrame(audit).to_csv(OUT/'probability_time_audit.csv',index=False,encoding='utf-8-sig')
    return probabilities


def regression_features():
    base=['past_level','older_level','past_production_log','known_zero']
    return {'baseline':base,'direct_shape':base+['prior_slot_trend'],
            'transition_probability':base+['prior_slot_trend','onset_probability']}


def regressor():
    return HistGradientBoostingRegressor(loss='squared_error',max_iter=200,max_leaf_nodes=15,
        min_samples_leaf=20,l2_regularization=1,learning_rate=.05,early_stopping=False,random_state=3203)


def soft_mixture(train,target):
    # Actual target production labels are used to train experts only; branch
    # selection at inference uses known past production and a past-only score.
    features=regression_features()['direct_shape']
    general=regressor(); general.fit(matrix(train,features),train.level)
    pred=general.predict(matrix(target,features))
    knownzero=target.known_zero.eq(1).to_numpy()
    experts=[]
    for event in (0,1):
        sample=train.loc[train.known_zero.eq(1)&train.onset.eq(event)]
        assert len(sample)>=20
        model=regressor(); model.fit(matrix(sample,features),sample.level)
        experts.append(model.predict(matrix(target.loc[knownzero],features)))
    p=target.loc[knownzero,'onset_probability'].to_numpy()
    pred[knownzero]=(1-p)*experts[0]+p*experts[1]
    return pred


def mixture_validation(part,frozen):
    assert not (OUT/'classification_test.csv').exists()
    assert frozen['power_gate']
    part=part.copy(); part['onset_probability']=probability_history(part,frozen)
    train=part.loc[part.month.between(3,4)]; valid=part.loc[part.month.between(5,6)]
    pred=soft_mixture(train,valid)
    predictions=pd.read_csv(OUT/'power_validation_predictions.csv',encoding='utf-8-sig')
    assert list(pd.to_datetime(predictions.timestamp))==list(valid.timestamp)
    predictions['soft_mixture']=pred
    predictions.to_csv(OUT/'power_validation_predictions.csv',index=False,encoding='utf-8-sig')
    results=pd.read_csv(OUT/'power_validation.csv',encoding='utf-8-sig')
    rows=[]
    for scope,keep in [('all',np.ones(len(valid),dtype=bool)),('onset',valid.onset.eq(1).to_numpy()),('known_zero',valid.known_zero.eq(1).to_numpy())]:
        rows.append(dict(model='soft_mixture',scope=scope,**power_metrics(valid.loc[keep],pred[keep])))
    results=pd.concat([results,pd.DataFrame(rows)],ignore_index=True)
    results.to_csv(OUT/'power_validation.csv',index=False,encoding='utf-8-sig')
    freeze=json.loads((OUT/'power_frozen.json').read_text(encoding='utf-8'))
    freeze['models'].append('soft_mixture')
    baseline=results.loc[results.model.eq('baseline')].set_index('scope')
    mixture=results.loc[results.model.eq('soft_mixture')].set_index('scope')
    freeze['validation_mixture_support']=bool(mixture.loc['all','mae']<baseline.loc['all','mae'] and mixture.loc['onset','mae']<baseline.loc['onset','mae'] and mixture.loc['onset','mean_under_amount']<baseline.loc['onset','mean_under_amount'])
    freeze['mixture_rationale']='Validation score-as-feature worsened onsetMAE9.95vs8.51; one minimum structural change: zero-continuing/onset regression experts mixed by prior-only probabilities. Same fixed regressor; no further search.'
    freeze['report_rule']='Report all four fixed models, including worsening; no changes after July/August evaluation.'
    write_json('power_frozen',freeze)
    print(pd.DataFrame(rows).to_json(orient='records')); print(json.dumps(freeze))


def power_metrics(frame,pred):
    error=pred-frame.level.to_numpy()
    w=frame.profile_weight.to_numpy()
    return dict(n=len(frame),mae=float(np.abs(error).mean()),rmse=float(np.sqrt(np.mean(error**2))),
        mean_error=float(error.mean()),mean_under_amount=float(np.maximum(-error,0).mean()),
        under_rate=float((error<0).mean()),profile_weighted_mae=float(np.average(np.abs(error),weights=w)))


def power_validation(part,frozen):
    if not frozen['power_gate']:
        write_json('power_frozen',dict(run=False,reason='Classifier validation gate not met; no favorable-result tuning.'))
        return
    part=part.copy(); part['onset_probability']=probability_history(part,frozen)
    train=part.loc[part.month.between(3,4)]
    valid=part.loc[part.month.between(5,6)]
    rows=[]; predictions=valid[['timestamp','date','month','level','onset','known_zero','profile_weight','onset_probability']].copy()
    for name,features in regression_features().items():
        model=regressor(); model.fit(matrix(train,features),train.level)
        pred=model.predict(matrix(valid,features)); predictions[name]=pred
        for scope,keep in [('all',np.ones(len(valid),dtype=bool)),('onset',valid.onset.eq(1).to_numpy()),('known_zero',valid.known_zero.eq(1).to_numpy())]:
            rows.append(dict(model=name,scope=scope,**power_metrics(valid.loc[keep],pred[keep])))
    results=pd.DataFrame(rows)
    results.to_csv(OUT/'power_validation.csv',index=False,encoding='utf-8-sig')
    predictions.to_csv(OUT/'power_validation_predictions.csv',index=False,encoding='utf-8-sig')
    # Freeze all three comparisons, including an unhelpful probability addition.
    write_json('power_frozen',dict(run=True,models=list(regression_features()),
        final_training_months=[3,4,5,6],final_test_months=[7,8],test_consulted=False,
        reason='Report all fixed ablations; no model or feature changes after final test. All probability training features were prior-month predictions.'))
    print(results.to_json(orient='records'))


def final_evaluation(part,frozen):
    risk=part.loc[part.known_zero.eq(1)]
    train=risk.loc[risk.month.le(6)]; test=risk.loc[risk.month.ge(7)]
    rows=[]; predictions=test[['timestamp','date','month','onset','profile_weight']].copy()
    for group,features in SETS.items():
        model=classifier(frozen['family']); model.fit(matrix(train,features),train.onset)
        prob=model.predict_proba(matrix(test,features))[:,1]
        predictions[group]=prob
        for scope,keep in [('all',np.ones(len(test),dtype=bool)),('month7',test.month.eq(7).to_numpy()),('month8',test.month.eq(8).to_numpy())]:
            rows.append(dict(features=group,scope=scope,**classify_metrics(test.loc[keep,'onset'],prob[keep],frozen['thresholds'][group])))
    pd.DataFrame(rows).to_csv(OUT/'classification_test.csv',index=False,encoding='utf-8-sig')
    predictions.to_csv(OUT/'classification_test_predictions.csv',index=False,encoding='utf-8-sig')
    print(pd.DataFrame(rows).to_json(orient='records'))
    power_frozen=json.loads((OUT/'power_frozen.json').read_text(encoding='utf-8'))
    if power_frozen['run']:
        part=part.copy(); part['onset_probability']=probability_history(part,frozen)
        train=part.loc[part.month.between(3,6)]; test=part.loc[part.month.ge(7)]
        predictions=test[['timestamp','date','month','level','onset','known_zero','profile_weight','onset_probability']].copy()
        rows=[]
        for name,features in regression_features().items():
            model=regressor(); model.fit(matrix(train,features),train.level)
            pred=model.predict(matrix(test,features)); predictions[name]=pred
            for scope,keep in [('all',np.ones(len(test),dtype=bool)),('onset',test.onset.eq(1).to_numpy()),
                ('known_zero',test.known_zero.eq(1).to_numpy()),('month7',test.month.eq(7).to_numpy()),('month8',test.month.eq(8).to_numpy())]:
                rows.append(dict(model=name,scope=scope,**power_metrics(test.loc[keep],pred[keep])))
        if 'soft_mixture' in power_frozen['models']:
            pred=soft_mixture(train,test); predictions['soft_mixture']=pred
            for scope,keep in [('all',np.ones(len(test),dtype=bool)),('onset',test.onset.eq(1).to_numpy()),
                ('known_zero',test.known_zero.eq(1).to_numpy()),('month7',test.month.eq(7).to_numpy()),('month8',test.month.eq(8).to_numpy())]:
                rows.append(dict(model='soft_mixture',scope=scope,**power_metrics(test.loc[keep],pred[keep])))
        pd.DataFrame(rows).to_csv(OUT/'power_test.csv',index=False,encoding='utf-8-sig')
        predictions.to_csv(OUT/'power_test_predictions.csv',index=False,encoding='utf-8-sig')
        print(pd.DataFrame(rows).to_json(orient='records'))
    verify(part)


def verify(part):
    with (OUT/'classification_test_predictions.csv').open(encoding='utf-8-sig',newline='') as stream:
        records=list(csv.DictReader(stream))
    for group in SETS:
        expected=sum((float(row[group])-int(row['onset']))**2 for row in records)/len(records)
        scores=pd.read_csv(OUT/'classification_test.csv',encoding='utf-8-sig')
        got=scores.loc[scores.features.eq(group)&scores.scope.eq('all'),'brier'].iloc[0]
        np.testing.assert_allclose(expected,got,atol=1e-12)
    if (OUT/'power_test_predictions.csv').exists():
        with (OUT/'power_test_predictions.csv').open(encoding='utf-8-sig',newline='') as stream:
            records=list(csv.DictReader(stream))
        scores=pd.read_csv(OUT/'power_test.csv',encoding='utf-8-sig')
        models=list(regression_features())+(['soft_mixture'] if 'soft_mixture' in records[0] else [])
        for model in models:
            for scope in ('all','onset'):
                rows=[row for row in records if scope=='all' or int(row['onset'])==1]
                mae=sum(abs(float(row[model])-float(row['level'])) for row in rows)/len(rows)
                got=scores.loc[scores.model.eq(model)&scores.scope.eq(scope),'mae'].iloc[0]
                np.testing.assert_allclose(mae,got,atol=1e-12)
    write_json('verification',dict(status='passed',input_sha256=hashlib.sha256(INPUT.read_bytes()).hexdigest(),
        input_timestamps_checked=len(part),n=len(part),target_production_not_a_feature=True,
        independent_csv_brier_and_mae=True,probability_training_strictly_past=True,
        script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        outputs_sha256={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.glob('*.csv')}))


def main():
    parser=argparse.ArgumentParser(); parser.add_argument('stage',choices=['validation','mixture','final']); args=parser.parse_args()
    OUT.mkdir(parents=True,exist_ok=True)
    part=load()
    if args.stage=='validation':
        write_json('contract',dict(previous='A02 within-onset correlation.013 does not test onset/no-onset separation; matched prior means81.85 vs39.57.',
            question='Can past information discriminate next-hour production onset among known-zero production hours, then improve next mean-power forecasts?',
            source='Current verified A02 observations only, no backup or previous models.',
            split='Class1-4train/5-6validation/7-8final chronological retrospective evaluation; all months were previously explored in EDA.',
            class_models='Fixed logistic and fixed shallow HGB; calendar -> observed mean -> observed slot trend; select common family on validation shapeAP.',
            class_selection='Highest validation AP within.01 favors simpler features; threshold maximizes validation F1, high threshold breaks ties.',
            probability_gate='Selected noncalendar AP better than calendar and Brier<=1.05calendar on validation.',
            power_models='Fixed HGB baseline past means/calendar/known past production; add direct slot trend; then add soft onset probability.',
            power_probability='March-June probabilities from strictly earlier month data; July/August probabilities from January-June only; positives set0 plus explicit known_zero feature.',
            power_split='3-4train/5-6validation; final3-6train/7-8test; same records across three fixed models.',
            power_metrics='All-hour and actual onset MAE plus average underprediction amount (including0 for overprediction), monthly and profile-weighted sensitivity.',
            no_target_inputs=True,no_peak_threshold=True,no_manuscript=True,stop='After final evaluation no feature/model retuning for favorable scores.'))
        class_validation(part)
        frozen=json.loads((OUT/'frozen.json').read_text(encoding='utf-8'))
        power_validation(part,frozen)
    elif args.stage=='mixture':
        frozen=json.loads((OUT/'frozen.json').read_text(encoding='utf-8'))
        mixture_validation(part,frozen)
    else:
        assert (OUT/'frozen.json').exists() and (OUT/'power_frozen.json').exists()
        frozen=json.loads((OUT/'frozen.json').read_text(encoding='utf-8'))
        final_evaluation(part,frozen)


if __name__=='__main__':
    main()


코드 등록 완료: `Analysis/scripts/a02_transition_forecast.py`

### `Analysis/scripts/a02_transition_review.py`

원고 연결: Analysis\03_Analysis_원고.md; Analysis\03_Analysis_원고.md.

코드 설명: Post-evaluation sensitivity/diagnosis and a readable count plot; no refitting.

주요 함수: `main()`.

In [19]:
%%submission_source Analysis/scripts/a02_transition_review.py
"""Post-evaluation sensitivity/diagnosis and a readable count plot; no refitting."""
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score,brier_score_loss
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from a02_transition_forecast import ROOT,INPUT,OUT,load
from a01_production_changes import load as raw_load


def main():
    part=load(); _,raw=raw_load()
    # Daily profile is a retrospective repetition diagnostic only, never a feature.
    raw=raw.set_index('timestamp')
    profile_train=set(raw.loc[raw.month.le(6),'profile'])
    frame=pd.read_csv(OUT/'classification_test_predictions.csv',encoding='utf-8-sig',parse_dates=['timestamp'])
    risk=part.set_index('timestamp').loc[frame.timestamp]
    target_profile=raw.loc[frame.timestamp,'profile']
    unseen=~target_profile.isin(profile_train).to_numpy()
    rows=[]
    for group in ('calendar','level','shape'):
        for scope,mask in [('all',np.ones(len(frame),dtype=bool)),('unseen_target_daily_profile',unseen),('seen_target_daily_profile',~unseen)]:
            if mask.sum()==0 or frame.loc[mask,'onset'].nunique()<2:
                continue
            y=frame.loc[mask,'onset']; prob=frame.loc[mask,group]
            rows.append(dict(features=group,scope=scope,n=int(mask.sum()),events=int(y.sum()),
                ap=average_precision_score(y,prob),brier=brier_score_loss(y,prob),
                profile_weighted_ap=average_precision_score(y,prob,sample_weight=frame.loc[mask,'profile_weight']),
                profile_weighted_brier=brier_score_loss(y,prob,sample_weight=frame.loc[mask,'profile_weight'])))
    pd.DataFrame(rows).to_csv(OUT/'repetition_sensitivity.csv',index=False,encoding='utf-8-sig')
    # Diagnose failed experts via their training event counts and the validated
    # probabilities on actual onset vs no-onset records. No new models fitted.
    diag=[]
    for phase in ('validation','test'):
        prediction=pd.read_csv(OUT/f'power_{phase}_predictions.csv',encoding='utf-8-sig')
        for event in (0,1):
            sample=prediction.loc[prediction.known_zero.eq(1)&prediction.onset.eq(event)]
            diag.append(dict(phase=phase,event=event,n=len(sample),mean_probability=sample.onset_probability.mean(),
                mean_actual_power=sample.level.mean(),mean_mixture_prediction=sample.soft_mixture.mean(),
                mean_direct_prediction=sample.direct_shape.mean()))
    pd.DataFrame(diag).to_csv(OUT/'mixture_diagnosis.csv',index=False,encoding='utf-8-sig')
    splits=[]
    for scope,mask in [('class_train',part.month.le(4)),('class_valid',part.month.between(5,6)),('class_finaltrain',part.month.le(6)),
        ('power_validtrain',part.month.between(3,4)),('power_finaltrain',part.month.between(3,6)),('final_test',part.month.ge(7))]:
        subset=part.loc[mask]; risk=subset.loc[subset.known_zero.eq(1)]
        splits.append(dict(scope=scope,n=len(subset),risk_hours=len(risk),events=int(risk.onset.sum())))
    pd.DataFrame(splits).to_csv(OUT/'split_counts.csv',index=False,encoding='utf-8-sig')
    # One simple figure: detected true onsets and false detections, with total52
    # actual events explicitly shown. No accuracy or undefined peak criteria.
    scores=pd.read_csv(OUT/'classification_test.csv',encoding='utf-8-sig')
    scores=scores.loc[scores.scope.eq('all')].set_index('features').loc[['calendar','level','shape']]
    plt.rcParams.update({'font.family':'Malgun Gothic','axes.unicode_minus':False,'font.size':12})
    fig,ax=plt.subplots(figsize=(9,4.8),layout='constrained')
    x=np.arange(3); width=.32
    a=ax.bar(x-width/2,scores.tp,width,label='실제 전환을 찾아낸 시간',color='#24788A')
    b=ax.bar(x+width/2,scores.fp,width,label='전환으로 잘못 예상한 시간',color='#D78947')
    ax.bar_label(a,padding=4); ax.bar_label(b,padding=4)
    ax.set_xticks(x,['달력만','달력 + 직전 평균 전력','달력 + 직전 평균 전력\n+ 시간 안의 변화'])
    ax.set_ylabel('시간 수'); ax.set_ylim(0,66)
    ax.axhline(52,color='#777777',linestyle=':',linewidth=1)
    ax.text(2.45,53,'실제 전환 52시간',ha='right',fontsize=10,color='#666666')
    ax.legend(loc='upper center',ncol=2,frameon=False)
    ax.spines[['top','right']].set_visible(False)
    figures=ROOT/'Analysis/figures'; figures.mkdir(exist_ok=True)
    path=figures/'a02_transition_detection.png'; fig.savefig(path,dpi=160); plt.close(fig)
    with Image.open(path) as img:
        img.resize((img.width//2,img.height//2),Image.Resampling.LANCZOS).save(figures/'a02_transition_detection_review.png')
    audit=pd.read_csv(OUT/'probability_time_audit.csv')
    assert (pd.to_datetime(audit.train_last_timestamp)<pd.to_datetime(audit.target_first_timestamp)).all()
    frozen_hashes={name:hashlib.sha256((OUT/name).read_bytes()).hexdigest() for name in ['frozen.json','power_frozen.json']}
    assert frozen_hashes['frozen.json']=='5cc22daeba82fc8f9c276c386215c8dc4021e17f0bfe0c0fd83340bb31c22b4e'
    assert frozen_hashes['power_frozen.json']=='a2f0d13883678e19d5b5625cb700cf265ca8c01ac00bde25ea8a66f2ef30229d'
    verification=dict(status='passed',frozen_conditions_unchanged=frozen_hashes,
        daily_profiles_used_only_for_posthoc_diagnostic=True,figure_path=str(path.relative_to(ROOT)),
        figure_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),figure_reviewed=False,
        outputs_sha256={name:hashlib.sha256((OUT/name).read_bytes()).hexdigest() for name in ['repetition_sensitivity.csv','mixture_diagnosis.csv','split_counts.csv']})
    (OUT/'review_verification.json').write_text(json.dumps(verification,indent=2),encoding='utf-8')
    print(pd.DataFrame(rows).to_json(orient='records'));print(pd.DataFrame(diag).to_json(orient='records'));print(pd.DataFrame(splits).to_json(orient='records'))


if __name__=='__main__':
    main()


코드 등록 완료: `Analysis/scripts/a02_transition_review.py`

### `Analysis/scripts/a03_conditioned_weather.py`

원고 연결: Analysis\03_Analysis_3페이지용_압축본.md; Analysis\03_Analysis_원고.md.

코드 설명: Approved 3.3: contemporaneous weather-power associations by calendar/production.
No forecasting, new peak definition, backup, or manuscript.

주요 함수: `save()`, `prepare()`, `association()`, `scopes()`, `reproduce_eda()`, `raw_check()`, `main()`.

In [20]:
%%submission_source Analysis/scripts/a03_conditioned_weather.py
"""Approved 3.3: contemporaneous weather-power associations by calendar/production.
No forecasting, new peak definition, backup, or manuscript.
"""
from pathlib import Path
import csv
import hashlib
import json
import numpy as np
import pandas as pd
from scipy.stats import rankdata,spearmanr
from a01_production_changes import ROOT,SOURCE,SHA,SLOTS,load
from a02_prior_power_signals import residuals,weighted_corr

OUT=ROOT/'Analysis/tables/a03_conditioned_weather'
WEATHER=['기온','풍속','습도','강수량']


def save(frame,name):
    frame.to_csv(OUT/f'{name}.csv',index=False,encoding='utf-8-sig')


def prepare():
    data,_=load()
    data['production_positive']=data['생산량'].gt(0).astype(int)
    freq=data[['date','profile']].drop_duplicates().profile.value_counts()
    data['profile_weight']=1/data.profile.map(freq)
    return data


def association(sample,feature,adjustment,mode):
    needed=[feature,'level','생산량']+ (WEATHER if adjustment=='joint' else [])
    sample=sample.dropna(subset=list(dict.fromkeys(needed))).copy()
    keys=['month','weekend','시간']
    if adjustment!='raw':
        counts=sample.groupby(keys).date.transform('nunique')
        sample=sample.loc[counts.ge(3)].copy()
    if len(sample)<10:
        return None
    w=np.ones(len(sample)) if mode=='date_hour' else sample.profile_weight.to_numpy()
    x=rankdata(sample[feature])/len(sample); y=rankdata(sample.level)/len(sample)
    if adjustment=='raw':
        value=weighted_corr(x,y,w)
    else:
        cell=pd.factorize(pd.MultiIndex.from_frame(sample[keys]))[0]
        controls=None
        if adjustment in ('production','joint'):
            rankprod=rankdata(sample['생산량'])/len(sample)
            controls=np.column_stack([sample.production_positive,rankprod,rankprod**2,rankprod**3])
        if adjustment=='joint':
            other=[rankdata(sample[c])/len(sample) for c in WEATHER if c!=feature]
            controls=np.column_stack([controls]+other)
        res=residuals(np.column_stack([x,y]),cell,w,controls)
        value=weighted_corr(res[:,0],res[:,1],w)
    return dict(variable=feature,adjustment=adjustment,weighting=mode,n=len(sample),
        days=sample.date.nunique(),correlation=value)


def scopes(data):
    weekday=data.loc[data.weekend.eq(0)]
    return {'weekday':weekday,'weekday_positive':weekday.loc[weekday.production_positive.eq(1)],
        'weekday_zero':weekday.loc[weekday.production_positive.eq(0)],
        'all':data,'all_positive':data.loc[data.production_positive.eq(1)],
        'august_all':data.loc[data.month.eq(8)],
        'august_positive':data.loc[data.month.eq(8)&data.production_positive.eq(1)]}


def reproduce_eda(data):
    table=pd.read_csv(ROOT/'EDA/tables/restart_relations_august_context.csv',encoding='utf-8-sig').set_index('population')
    subset=data.loc[data.month.eq(8)]
    for label,sample in [('all',subset),('positive',subset.loc[subset.production_positive.eq(1)]),
                         ('zero',subset.loc[subset.production_positive.eq(0)])]:
        r=spearmanr(sample['기온'],sample['평균']).statistic
        assert len(sample)==table.loc[label,'n']
        np.testing.assert_allclose(r,table.loc[label,'spearman'],atol=1e-12)


def raw_check(data):
    reference={}
    with SOURCE.open(encoding='utf-8-sig',newline='') as stream:
        for row in csv.DictReader(stream):
            date,hour=int(row['날짜']),int(row['시간'])
            if 20210101<=date<=20210831 and 0<=hour<=23:
                stamp=pd.Timestamp(str(date))+pd.Timedelta(hours=hour)
                reference[stamp]=row
    for row in data.to_dict('records'):
        source=reference[row['timestamp']]
        assert row['level']==sum(int(source[c]) for c in SLOTS)/4
        for column in WEATHER+['생산량']:
            expected=float(source[column]) if source[column].strip() else np.nan
            np.testing.assert_allclose(row[column],expected,rtol=0,atol=0,equal_nan=True)


def main():
    OUT.mkdir(parents=True,exist_ok=True)
    contract=dict(previous='Current EDA August temp-power rho-.076 overall vs+.599 production-positive383hours; weekday low-power747hours include92production-positive.',
        question='Does weather-power association remain within calendar and observed production conditions? What explains aggregate/subgroup difference?',
        period='Jan-Aug2021 normal241days5784hours; primary weekday171days4104hours; no zero/low-value filtering.',
        outcome='Unrounded mean of four15min values at same time; descriptive contemporaneous association only.',
        variables=WEATHER,calendar='month x hour x weekday/weekend; adjusted groups require>=3distinct dates per cell.',
        adjustment='Rank correlations raw -> same-calendar residual -> production0/positive plus cubic rank-production -> other three rank-weather variables.',
        sample='Each variable uses its observed pairs except joint analyses use common complete four-weather records; count missing per scope.',
        sensitivity='Daily array inverse-frequency; all/weekdays/positive/zero; per-month and leave-month-out; no pvalues or forecasting.',
        decision='If monthly signs or adjustments differ, diagnose composition/production/weather co-variation before interpreting; report mixed results without seeking stronger correlations.',
        next_scope='Analysis only, do not start power prediction; any Modeling inputs must respect information time.',
        backup=False,forecast=False,manuscript=False,new_peak_threshold=False,source_sha256=SHA)
    (OUT/'contract.json').write_text(json.dumps(contract,indent=2,ensure_ascii=False),encoding='utf-8')
    data=prepare(); raw_check(data); reproduce_eda(data)
    save(data[['timestamp','date','month','weekend','시간','level','생산량','production_positive']+WEATHER+['profile_weight']], 'observations')
    rows=[]
    for scope,sample in scopes(data).items():
        for feature in WEATHER:
            for adjustment in ('raw','calendar','production','joint'):
                for mode in ('date_hour','profile_balanced'):
                    result=association(sample,feature,adjustment,mode)
                    if result:
                        rows.append(dict(scope=scope,**result))
    results=pd.DataFrame(rows); save(results,'associations')
    monthly=[]
    for scope,sample in scopes(data).items():
        if scope not in ('weekday','weekday_positive','weekday_zero','all_positive'):
            continue
        for month,subset in sample.groupby('month'):
            for feature in WEATHER:
                for adjustment in ('raw','production','joint'):
                    result=association(subset,feature,adjustment,'date_hour')
                    if result:
                        monthly.append(dict(scope=scope,month=month,**result))
    save(pd.DataFrame(monthly),'monthly')
    stability=[]
    for scope,sample in scopes(data).items():
        if scope not in ('weekday','weekday_positive','weekday_zero'):
            continue
        for month in range(1,9):
            for feature in WEATHER:
                result=association(sample.loc[sample.month.ne(month)],feature,'joint','date_hour')
                if result:
                    stability.append(dict(scope=scope,exclude_month=month,**result))
    save(pd.DataFrame(stability),'leave_month_out')
    counts=[]
    for scope,sample in scopes(data).items():
        counts.append(dict(scope=scope,n=len(sample),days=sample.date.nunique(),complete=int(sample[WEATHER].notna().all(axis=1).sum()),
            **{f'{column}_missing':int(sample[column].isna().sum()) for column in WEATHER}))
    save(pd.DataFrame(counts),'sample_counts')
    correlations=data[WEATHER].corr(method='spearman'); save(correlations.reset_index(names='variable'),'weather_correlations')
    # Independent explicit-dummy residualization on January weekdays verifies
    # the method without interpreting any model as a future prediction.
    sample=data.loc[data.month.eq(1)&data.weekend.eq(0)].dropna(subset=WEATHER)
    cell=pd.factorize(pd.MultiIndex.from_frame(sample[['month','weekend','시간']]))[0]
    prod=rankdata(sample['생산량'])/len(sample)
    controls=np.column_stack([sample.production_positive,prod,prod**2,prod**3]+
        [rankdata(sample[c])/len(sample) for c in WEATHER if c!='기온'])
    values=np.column_stack([rankdata(sample['기온']),rankdata(sample.level)])/len(sample)
    w=sample.profile_weight.to_numpy()
    res=residuals(values,cell,w,controls)
    design=np.column_stack([np.eye(cell.max()+1)[cell],controls])
    beta=np.linalg.lstsq(design*np.sqrt(w[:,None]),values*np.sqrt(w[:,None]),rcond=None)[0]
    np.testing.assert_allclose(res,values-design@beta,atol=1e-10)
    verification=dict(status='passed',source_sha256=SHA,raw_hours_checked=len(data),eda_august_correlations_reproduced=True,
        joint_calendar_residual_vs_explicit_dummy_verified=True,zero_records_preserved=True,
        script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        outputs_sha256={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.glob('*.csv')})
    (OUT/'verification.json').write_text(json.dumps(verification,indent=2),encoding='utf-8')
    print(results.loc[results.scope.isin(['weekday','weekday_positive','weekday_zero','august_positive'])&results.weighting.eq('date_hour')].to_json(orient='records'))
    print(pd.DataFrame(counts).to_json(orient='records'))
    print(json.dumps(verification))


if __name__=='__main__':
    main()


코드 등록 완료: `Analysis/scripts/a03_conditioned_weather.py`

### `Analysis/scripts/a03_weather_diagnostics.py`

원고 연결: Analysis\03_Analysis_3페이지용_압축본.md; Analysis\03_Analysis_원고.md.

코드 설명: Minimal followup: common rows, monthly differences, and date-level variation.

주요 함수: `adjusted()`, `main()`.

In [21]:
%%submission_source Analysis/scripts/a03_weather_diagnostics.py
"""Minimal followup: common rows, monthly differences, and date-level variation."""
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from a03_conditioned_weather import OUT,WEATHER,association,save
from a02_prior_power_signals import residuals,weighted_corr


def adjusted(sample,date_control=False,prod_form='cubic',other_weather=True):
    sample=sample.dropna(subset=WEATHER).copy()
    counts=sample.groupby(['month','weekend','시간']).date.transform('nunique')
    sample=sample.loc[counts.ge(3)]
    cells=pd.factorize(pd.MultiIndex.from_frame(sample[['month','weekend','시간']]))[0]
    p=rankdata(sample['생산량'])/len(sample)
    controls=np.column_stack([sample.production_positive,p])
    if prod_form=='cubic':
        controls=np.column_stack([controls,p**2,p**3])
    if other_weather:
        controls=np.column_stack([controls]+[rankdata(sample[c])/len(sample) for c in WEATHER if c!='기온'])
    if date_control:
        date_ids=pd.factorize(sample.date)[0]
        # Date-level nuisance controls identify within-date contrasts, not the
        # same estimand as across-date thermal/weather load association.
        controls=np.column_stack([controls,np.eye(date_ids.max()+1)[date_ids]])
    values=np.column_stack([rankdata(sample['기온']),rankdata(sample.level)])/len(sample)
    w=np.ones(len(sample))
    res=residuals(values,cells,w,controls)
    return dict(n=len(sample),days=sample.date.nunique(),correlation=weighted_corr(res[:,0],res[:,1],w))


def main():
    manifest=json.loads((OUT/'verification.json').read_text(encoding='utf-8'))
    source=OUT/'observations.csv'
    assert hashlib.sha256(source.read_bytes()).hexdigest()==manifest['outputs_sha256']['observations.csv']
    data=pd.read_csv(source,encoding='utf-8-sig',parse_dates=['timestamp','date'])
    data['week']=data.date-pd.to_timedelta(data.date.dt.dayofweek,unit='D')
    weekday=data.loc[data.weekend.eq(0)]
    rows=[]
    for scope,sample in [('weekday',weekday),('weekday_positive',weekday.loc[weekday.production_positive.eq(1)])]:
        sample=sample.dropna(subset=WEATHER)
        sample=sample.loc[sample.groupby(['month','weekend','시간']).date.transform('nunique').ge(3)]
        for feature in WEATHER:
            for adjustment in ('raw','calendar','production','joint'):
                result=association(sample,feature,adjustment,'date_hour')
                rows.append(dict(scope=scope,**result))
    save(pd.DataFrame(rows),'common_row_associations')
    monthly=[]
    positive=weekday.loc[weekday.production_positive.eq(1)]
    for month,sample in positive.groupby('month'):
        for label,date_control,prod_form in [('joint',False,'cubic'),('linear_production',False,'linear'),('within_date_joint',True,'cubic')]:
            monthly.append(dict(month=month,check=label,**adjusted(sample,date_control,prod_form)))
    save(pd.DataFrame(monthly),'temperature_monthly_diagnostics')
    weeks=[]
    # Temperature is the predeclared focal variable. All eight months are
    # checked; do not keep only a favorable summer/month result.
    for month,sample in positive.groupby('month'):
        for week in sample.week.unique():
            subset=sample.loc[sample.week.ne(week)]
            weeks.append(dict(month=month,excluded_week=str(pd.Timestamp(week).date()),**adjusted(subset)))
    save(pd.DataFrame(weeks),'temperature_leave_week_out')
    weather=[]
    for month,sample in positive.groupby('month'):
        sample=sample.dropna(subset=WEATHER)
        sample=sample.loc[sample.groupby(['month','weekend','시간']).date.transform('nunique').ge(3)]
        cells=pd.factorize(pd.MultiIndex.from_frame(sample[['month','weekend','시간']]))[0]
        p=rankdata(sample['생산량'])/len(sample)
        c=np.column_stack([p,p**2,p**3])
        values=np.column_stack([rankdata(sample[name]) for name in WEATHER])/len(sample)
        res=residuals(values,cells,np.ones(len(sample)),c)
        for j,name in enumerate(WEATHER[1:],1):
            weather.append(dict(month=month,other_weather=name,n=len(sample),
                correlation=weighted_corr(res[:,0],res[:,j],np.ones(len(sample)))))
    save(pd.DataFrame(weather),'conditional_temperature_weather')
    # Day-level diagnostics retain only daily averages during positive-production
    # observations; summarize selected-hour composition, not daily total power.
    daily=positive.groupby('date').agg(month=('month','first'),hours=('level','size'),
        power=('level','mean'),temperature=('기온','mean'),production=('생산량','mean')).reset_index()
    save(daily,'positive_day_means')
    ds=[]
    from scipy.stats import spearmanr
    for month,sample in daily.groupby('month'):
        ds.append(dict(month=month,days=len(sample),temperature_power_rho=spearmanr(sample.temperature,sample.power).statistic))
    save(pd.DataFrame(ds),'daily_temperature_relations')
    contract=dict(previous='Weekday temperature rho+.227 falls to-.004 after calendar/production/otherweather; August positive+.286 survives; monthly signs differ.',
        checks='Fixed common-row comparisons; all8monthly positive groups with date fixed effects and linear-vs-cubic production; leave each week out; conditional weather co-variation; daily positive-record means.',
        cautions='Date-fixed relation is within-date variation, cannot erase genuine between-day weather effect; daily positive means have different selected hours; no causal or predictive claims.',
        no_forecast=True,no_threshold_search=True,no_backup=True)
    (OUT/'diagnostic_contract.json').write_text(json.dumps(contract,indent=2),encoding='utf-8')
    verify=dict(status='passed',source_observations_unchanged=True,
        script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        outputs_sha256={name:hashlib.sha256((OUT/f'{name}.csv').read_bytes()).hexdigest() for name in [
            'common_row_associations','temperature_monthly_diagnostics','temperature_leave_week_out','conditional_temperature_weather','positive_day_means','daily_temperature_relations']})
    (OUT/'diagnostic_verification.json').write_text(json.dumps(verify,indent=2),encoding='utf-8')
    print(pd.DataFrame(monthly).to_json(orient='records'));print(pd.DataFrame(ds).to_json(orient='records'))
    print(pd.DataFrame(weeks).groupby('month').correlation.agg(['min','max']).to_json())
    print(pd.DataFrame(weather).to_json(orient='records'))


if __name__=='__main__':
    main()


코드 등록 완료: `Analysis/scripts/a03_weather_diagnostics.py`

### `Analysis/scripts/a04_maximum_conditions.py`

원고 연결: Analysis\03_Analysis_3페이지용_압축본.md; Analysis\03_Analysis_원고.md.

코드 설명: 3.4: observed next-hour maximum and available antecedents, no forecast fitting.

주요 함수: `save()`, `build()`, `control_array()`, `association()`, `matched()`, `independent_check()`, `main()`.

In [22]:
%%submission_source Analysis/scripts/a04_maximum_conditions.py
"""3.4: observed next-hour maximum and available antecedents, no forecast fitting."""
import csv
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from a01_production_changes import ROOT, SOURCE, SHA, SLOTS, load, contrast
from a02_prior_power_signals import residuals, weighted_corr

OUT = ROOT / 'Analysis/tables/a04_maximum_conditions'
BASE = ['past_level', 'prior_maximum', 'prior_last']
SHAPE = ['prior_slot_trend', 'prior_last_above_mean', 'prior_maximum_excess',
         'prior_range', 'recent_level_delta']
TARGETS = ['target_maximum', 'level', 'target_excess']
KEYS = ['month', 'weekend', '시간']


def save(frame, name):
    frame.to_csv(OUT / f'{name}.csv', index=False, encoding='utf-8-sig')


def build():
    data, part = load()
    index = data.set_index('timestamp')
    previous = index.reindex(part.timestamp - pd.Timedelta(hours=1))
    older = index.reindex(part.timestamp - pd.Timedelta(hours=2))
    part['input_timestamp'] = part.timestamp - pd.Timedelta(hours=1)
    part['older_timestamp'] = part.timestamp - pd.Timedelta(hours=2)
    part['older_level'] = older.level.to_numpy()
    part['target_maximum'] = part[SLOTS].max(axis=1)
    part['target_excess'] = part.target_maximum - part.level
    part['target_relative_excess'] = part.target_excess / part.level.where(part.level.ne(0))
    part['prior_maximum'] = previous[SLOTS].max(axis=1).to_numpy()
    part['prior_last'] = previous[SLOTS[-1]].to_numpy()
    part['prior_first'] = previous[SLOTS[0]].to_numpy()
    part['prior_slot_trend'] = part.prior_last - part.prior_first
    part['prior_last_above_mean'] = part.prior_last - part.past_level
    part['prior_maximum_excess'] = part.prior_maximum - part.past_level
    part['prior_range'] = previous.slot_range.to_numpy()
    part['recent_level_delta'] = part.past_level - part.older_level
    part['known_production_positive'] = part.past_production.gt(0).astype(int)
    part['prior_profile'] = previous.profile.to_numpy()
    part['older_profile'] = older.profile.to_numpy()
    triples = part.loc[part.older_level.notna()].copy().reset_index(drop=True)
    triples['triple_profile'] = triples.older_profile + '|' + triples.prior_profile + '|' + triples.profile
    counts = triples[['date', 'triple_profile']].drop_duplicates().triple_profile.value_counts()
    triples['profile_weight'] = 1 / triples.triple_profile.map(counts)
    assert len(part) == 5781 and len(triples) == 5778
    return data, part, triples


def control_array(sample, include_level=True, degree=3):
    arrays = [sample.known_production_positive.to_numpy()]
    production = rankdata(sample.past_production) / len(sample)
    arrays.extend(production ** power for power in range(1, degree + 1))
    if include_level:
        level = rankdata(sample.past_level) / len(sample)
        arrays.extend(level ** power for power in range(1, degree + 1))
    return np.column_stack(arrays)


def association(sample, feature, target, adjustment, mode):
    sample = sample.copy()
    if adjustment != 'raw':
        sample = sample.loc[sample.groupby(KEYS).date.transform('nunique').ge(3)].copy()
    if len(sample) < 20:
        return None
    weights = np.ones(len(sample)) if mode == 'date_hour' else sample.profile_weight.to_numpy()
    values = np.column_stack([rankdata(sample[feature]) / len(sample),
                              rankdata(sample[target]) / len(sample)])
    if adjustment == 'raw':
        res = values
    else:
        cells = pd.factorize(pd.MultiIndex.from_frame(sample[KEYS]))[0]
        controls = control_array(sample, include_level=adjustment == 'calendar_level_production')
        res = residuals(values, cells, weights, controls)
    return dict(feature=feature, target=target, adjustment=adjustment, weighting=mode,
                n=len(sample), days=sample.date.nunique(),
                correlation=weighted_corr(res[:, 0], res[:, 1], weights))


def matched(part, sensitivity='main', boot=False):
    clone = part.copy()
    clone['change'] = clone.transition
    metrics = ['target_maximum', 'level', 'target_excess', 'target_relative_excess', 'abs_power_delta']
    pairs = [('zero_to_positive', 'positive_to_positive'),
             ('zero_to_positive', 'zero_to_zero'),
             ('positive_to_zero', 'positive_to_positive')]
    rows = []
    for a, b in pairs:
        for metric in metrics:
            for mode in ['date_hour', 'profile_balanced']:
                eligible = clone.loc[clone[metric].notna()]
                row = contrast(eligible, a, b, metric, mode, boot=boot and mode == 'date_hour')
                if row:
                    row['sensitivity'] = sensitivity
                    row['undefined_metric_hours'] = int(clone[metric].isna().sum())
                    rows.append(row)
    return rows


def independent_check(data, pairs, triples):
    raw = {}
    with SOURCE.open(encoding='utf-8-sig', newline='') as handle:
        for row in csv.DictReader(handle):
            date, hour = int(row['날짜']), int(row['시간'])
            if 20210101 <= date <= 20210831 and 0 <= hour <= 23:
                timestamp = pd.to_datetime(str(date), format='%Y%m%d') + pd.Timedelta(hours=hour)
                raw[timestamp] = ([float(row[key]) for key in SLOTS], float(row['생산량']))
    for row in pairs.itertuples():
        current, prod = raw[row.timestamp]
        previous, prior_prod = raw[row.input_timestamp]
        mean, past = sum(current) / 4, sum(previous) / 4
        relative = (max(current) - mean) / mean if mean else np.nan
        expected = [max(current), mean, max(current) - mean, relative,
                    past, max(previous), previous[-1], previous[0], previous[-1] - previous[0],
                    previous[-1] - past, max(previous) - past, max(previous) - min(previous)]
        got = [row.target_maximum, row.level, row.target_excess, row.target_relative_excess,
               row.past_level, row.prior_maximum, row.prior_last, row.prior_first,
               row.prior_slot_trend, row.prior_last_above_mean, row.prior_maximum_excess, row.prior_range]
        np.testing.assert_allclose(got, expected, rtol=0, atol=0)
        assert row.input_timestamp < row.timestamp
        assert row.past_production == prior_prod and getattr(row, '생산량') == prod
    for row in triples.itertuples():
        older, _ = raw[row.older_timestamp]
        assert row.recent_level_delta == row.past_level - sum(older) / 4
        assert row.older_timestamp < row.input_timestamp < row.timestamp
    tiny = triples.loc[(triples.month == 1) & (triples.weekend == 0)].copy()
    cells = pd.factorize(pd.MultiIndex.from_frame(tiny[KEYS]))[0]
    controls = control_array(tiny)
    values = np.column_stack([rankdata(tiny.prior_slot_trend), rankdata(tiny.target_maximum)])
    weights = tiny.profile_weight.to_numpy()
    computed = residuals(values, cells, weights, controls)
    dummies = np.eye(cells.max() + 1)[cells]
    design = np.column_stack([dummies, controls])
    fitted = np.linalg.lstsq(design * np.sqrt(weights[:, None]),
                             values * np.sqrt(weights[:, None]), rcond=None)[0]
    np.testing.assert_allclose(computed, values - design @ fitted, atol=1e-9)
    assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == SHA
    return dict(raw_pairs_checked=len(pairs), raw_triples_checked=len(triples),
                explicit_dummy_residual_check=True, input_timestamps_precede_target=True)


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    contract = dict(
        previous='3.1 matched absolute mean-change37.92vs30.21;3.2 power-change association+.305 but power forecast improvement unstable;3.3 weather weak.',
        question='Are production transitions high maximum conditions, does mean conceal maximum differences, and which observed power features precede next-hour maximum?',
        outcome='Maximum of the four15min recorded values in next fully observed hour, continuous; mean and maximum-minus-mean for interpretation.',
        period='Normal Jan-Aug241days; exact1h5781pairs and exact2h5778triples; all zeros/extremes retained.',
        comparison='Same target month-hour-weekday/weekend, each production group>=2dates, common pooled-hour weights.',
        signals='Available at completed t-1; raw and calendar+known production, then observed prior mean rank cubic; never control/input target mean or future production.',
        antecedents=BASE + SHAPE, outcomes=TARGETS,
        sensitivity='Daily-array inverse-frequency, same-day pairs, month omission; week-resampling800 exploratory ranges for main matched contrasts.',
        decision='Separate absolute change from maximum level; if conditional shape signal weak or monthly heterogeneous diagnose observed-level and target-gap distinction, no favorable row or threshold search.',
        boundaries='Descriptive rank associations only; no model training, forecast scoring, peak classification, oracle maximum reconstruction, manuscript, backup.',
        zero_mean='Zero means retained for absolute/maximum metrics; only relative excess is undefined for zero denominator and omitted metric-wise with count.',
        source_sha256=SHA)
    (OUT / 'contract.json').write_text(json.dumps(contract, ensure_ascii=False, indent=2), encoding='utf-8')
    data, pairs, triples = build()
    columns = ['timestamp', 'input_timestamp', 'older_timestamp', 'date', 'month', 'weekend', '시간',
               '생산량', 'past_production', 'known_production_positive', 'transition', 'level',
               'target_maximum', 'target_excess', 'target_relative_excess', 'past_level', 'older_level',
               *BASE[1:], 'prior_first', *SHAPE, 'abs_power_delta', 'power_delta', 'profile_weight']
    save(pairs[columns], 'observations_pairs')
    save(triples[columns], 'observations_triples')
    rows = []
    for label, keys in [('all', []), ('transition', ['transition']),
                        ('month_transition', ['month', 'transition']), ('hour', ['weekend', '시간'])]:
        groups = [((), pairs)] if not keys else pairs.groupby(keys)
        for key, cell in groups:
            key = key if isinstance(key, tuple) else (key,)
            for metric in ['target_maximum', 'level', 'target_excess', 'target_relative_excess']:
                rows.append(dict(scope=label, **dict(zip(keys, key)), metric=metric, n=len(cell),
                                 days=cell.date.nunique(), mean=cell[metric].mean(),
                                 median=cell[metric].median(), q10=cell[metric].quantile(.1),
                                 q90=cell[metric].quantile(.9), maximum=cell[metric].max()))
    save(pd.DataFrame(rows), 'descriptive')
    contrasts = matched(pairs, boot=True)
    save(pd.DataFrame(contrasts), 'matched_contrasts')
    sensitivities = matched(pairs.loc[pairs.date.eq(pairs.input_timestamp.dt.normalize())], 'same_day')
    for month in range(1, 9):
        sensitivities.extend(matched(pairs.loc[pairs.month.ne(month)], f'leave_month_{month}'))
    save(pd.DataFrame(sensitivities), 'matched_sensitivity')
    scopes = dict(all=triples, weekday=triples.loc[triples.weekend.eq(0)],
                  known_production_zero=triples.loc[triples.past_production.eq(0)],
                  known_production_positive=triples.loc[triples.past_production.gt(0)])
    signals = []
    for scope, sample in scopes.items():
        for target in TARGETS:
            for feature in BASE + SHAPE:
                adjustments = ['raw', 'calendar_production']
                if feature != 'past_level':
                    adjustments.append('calendar_level_production')
                for adjustment in adjustments:
                    for mode in ['date_hour', 'profile_balanced']:
                        row = association(sample, feature, target, adjustment, mode)
                        if row:
                            row['scope'] = scope
                            signals.append(row)
    save(pd.DataFrame(signals), 'antecedent_associations')
    monthly = []
    for month, sample in triples.groupby('month'):
        for target in TARGETS:
            for feature in ['past_level', *SHAPE]:
                adjustment = 'calendar_production' if feature == 'past_level' else 'calendar_level_production'
                row = association(sample, feature, target, adjustment, 'date_hour')
                if row:
                    row['month'] = month
                    monthly.append(row)
    save(pd.DataFrame(monthly), 'monthly_associations')
    audit = independent_check(data, pairs, triples)
    original = pd.read_csv(ROOT / 'Analysis/tables/a01_production_changes/transition_matched_contrasts.csv', encoding='utf-8-sig')
    recreated = pd.DataFrame(contrasts)
    old = original.loc[(original.a == 'zero_to_positive') & (original.b == 'positive_to_positive') &
                       (original.metric == 'abs_power_delta') & (original.weighting == 'date_hour')].iloc[0]
    new = recreated.loc[(recreated.a == old.a) & (recreated.b == old.b) &
                        (recreated.metric == old.metric) & (recreated.weighting == old.weighting)].iloc[0]
    np.testing.assert_allclose([new.adjusted_a, new.adjusted_b, new.matched_cells, new.n_a, new.n_b],
                               [old.adjusted_a, old.adjusted_b, old.matched_cells, old.n_a, old.n_b], rtol=0, atol=1e-12)
    audit.update(status='passed', a01_matched_contrast_reproduced=True,
                 source_sha256=SHA, script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
                 outputs_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.glob('*.csv'))})
    (OUT / 'verification.json').write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(dict(status='passed', pairs=len(pairs), triples=len(triples)), ensure_ascii=False))
    print(recreated.loc[(recreated.weighting == 'date_hour') & recreated.metric.isin(['target_maximum','level','target_excess'])].to_json(orient='records'))
    signal_frame = pd.DataFrame(signals)
    print(signal_frame.loc[(signal_frame.scope == 'all') & (signal_frame.weighting == 'date_hour') &
                           (signal_frame.adjustment != 'raw')].to_json(orient='records'))


if __name__ == '__main__':
    main()


코드 등록 완료: `Analysis/scripts/a04_maximum_conditions.py`

### `Analysis/scripts/a04_maximum_diagnostics.py`

원고 연결: Analysis\03_Analysis_원고.md.

코드 설명: Follow-up to A04: observed daily priorities and robustness of terminal slot.

주요 함수: `daily_priorities()`, `slot_group_associations()`, `maximum_context()`, `main()`.

In [23]:
%%submission_source Analysis/scripts/a04_maximum_diagnostics.py
"""Follow-up to A04: observed daily priorities and robustness of terminal slot."""
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from a01_production_changes import SLOTS, load, SOURCE, SHA
from a02_prior_power_signals import residuals, weighted_corr
from a04_maximum_conditions import OUT, KEYS, association, control_array, save


def daily_priorities(data):
    rows = []
    for date, cell in data.groupby('date'):
        maximum = cell[SLOTS].max(axis=1)
        best_maximum = maximum.max()
        top_mean = cell.level.eq(cell.level.max())
        top_maximum = maximum.eq(best_maximum)
        chosen = maximum.loc[top_mean]
        rows.append(dict(date=date, month=int(cell.month.iloc[0]), weekend=int(cell.weekend.iloc[0]),
                         daily_maximum=best_maximum, highest_hourly_mean=cell.level.max(),
                         n_top_mean=int(top_mean.sum()), n_top_maximum=int(top_maximum.sum()),
                         common_top_hours=int((top_mean & top_maximum).sum()),
                         no_common_top=int(not (top_mean & top_maximum).any()),
                         top_mean_hours='|'.join(cell.loc[top_mean, '시간'].astype(str)),
                         top_maximum_hours='|'.join(cell.loc[top_maximum, '시간'].astype(str)),
                         maximum_in_mean_top_best=chosen.max(), maximum_in_mean_top_worst=chosen.min(),
                         minimum_missed_maximum=best_maximum - chosen.max(),
                         maximum_missed_maximum=best_maximum - chosen.min(),
                         equal_tie_average_gap=best_maximum - chosen.mean()))
    frame = pd.DataFrame(rows)
    save(frame, 'daily_priority_diagnostics')
    summaries = []
    scopes = dict(all=frame, weekday=frame.loc[frame.weekend.eq(0)], weekend=frame.loc[frame.weekend.eq(1)],
                  disjoint=frame.loc[frame.no_common_top.eq(1)])
    for scope, cell in scopes.items():
        summaries.append(dict(scope=scope, n_days=len(cell), disjoint_days=int(cell.no_common_top.sum()),
                              mean_minimum_gap=cell.minimum_missed_maximum.mean(),
                              median_minimum_gap=cell.minimum_missed_maximum.median(),
                              q90_minimum_gap=cell.minimum_missed_maximum.quantile(.9),
                              largest_minimum_gap=cell.minimum_missed_maximum.max(),
                              tie_days=int(cell.n_top_mean.gt(1).sum())))
    for month, cell in frame.groupby('month'):
        summaries.append(dict(scope=f'month_{month}', n_days=len(cell), disjoint_days=int(cell.no_common_top.sum()),
                              mean_minimum_gap=cell.minimum_missed_maximum.mean(),
                              median_minimum_gap=cell.minimum_missed_maximum.median(),
                              q90_minimum_gap=cell.minimum_missed_maximum.quantile(.9),
                              largest_minimum_gap=cell.minimum_missed_maximum.max(),
                              tie_days=int(cell.n_top_mean.gt(1).sum())))
    save(pd.DataFrame(summaries), 'daily_priority_summary')
    return frame, summaries


def slot_group_associations(sample, mode='date_hour', degree=3, add_dates=False):
    sample = sample.loc[sample.groupby(KEYS).date.transform('nunique').ge(3)].copy()
    weights = np.ones(len(sample)) if mode == 'date_hour' else sample.profile_weight.to_numpy()
    features = ['prior_first', 'prior_second', 'prior_third', 'prior_last',
                'prior_slot_trend', 'prior_last_above_mean']
    targets = ['target_maximum', 'level', 'target_excess']
    values = np.column_stack([rankdata(sample[column]) / len(sample) for column in features + targets])
    controls = control_array(sample, degree=degree)
    if add_dates:
        dates = pd.factorize(sample.date)[0]
        controls = np.column_stack([controls, np.eye(dates.max() + 1)[dates]])
    cells = pd.factorize(pd.MultiIndex.from_frame(sample[KEYS]))[0]
    result = residuals(values, cells, weights, controls)
    rows = []
    for i, feature in enumerate(features):
        for j, target in enumerate(targets):
            rows.append(dict(feature=feature, target=target, n=len(sample), days=sample.date.nunique(),
                             correlation=weighted_corr(result[:, i], result[:, len(features) + j], weights)))
    return rows


def maximum_context(data, pairs):
    context = data.copy()
    context['target_maximum'] = context[SLOTS].max(axis=1)
    context['target_excess'] = context.target_maximum - context.level
    context = context.merge(pairs[['timestamp', 'transition']], on='timestamp', how='left', validate='one_to_one')
    context['transition'] = context.transition.fillna('prior_unavailable')
    context['daily_top_maximum'] = context.target_maximum.eq(context.groupby('date').target_maximum.transform('max'))
    context['daily_top_mean'] = context.level.eq(context.groupby('date')['level'].transform('max'))
    context['daily_maximum_share'] = context.daily_top_maximum / context.groupby('date').daily_top_maximum.transform('sum')
    summaries = []
    for keys in [['transition'], ['weekend', '시간'], ['weekend', 'transition']]:
        for key, cell in context.groupby(keys):
            key = key if isinstance(key, tuple) else (key,)
            summaries.append(dict(scope='_'.join(keys), **dict(zip(keys, key)), n_hours=len(cell),
                                  n_days=cell.date.nunique(), maximum_mean=cell.target_maximum.mean(),
                                  excess_mean=cell.target_excess.mean(),
                                  daily_maximum_days_equivalent=cell.daily_maximum_share.sum(),
                                  daily_maximum_hours=int(cell.daily_top_maximum.sum())))
    save(pd.DataFrame(summaries), 'daily_maximum_context')
    return context


def main():
    observed_path = OUT / 'observations_triples.csv'
    old_hash = hashlib.sha256(observed_path.read_bytes()).hexdigest()
    contract = dict(
        reason='Main A04 onset maximum135.73vs135.32(nearzero); terminal slot partialrank+.434 versus mean+.431, target-excess relation weaker. Need observed priority differences and stability, not new tuning.',
        daily='All241complete24h dates; compare observed maximum-per-hour top set vs observed hourly-mean top set, all ties retained. Report unavoidable maximum gap if sets disjoint, continuous size; this is not a forecast score.',
        terminal='Compare all four past positions together using same controls; linear vs cubic rank level/production, add date effects, inverse-profile, same-date, monthly/leave-month-out.',
        boundaries='No peak threshold, highest-mean oracle restoration, predictive training or risk classifier. Actual targets used for descriptive context only.',
        stop='If terminal association remains but excess/rank-disagreement is weak, use direct maximum candidate with cautious scope; do not seek a different cutoff or winning month.')
    (OUT / 'diagnostic_contract.json').write_text(json.dumps(contract, ensure_ascii=False, indent=2), encoding='utf-8')
    sample = pd.read_csv(observed_path, encoding='utf-8-sig', parse_dates=['timestamp', 'input_timestamp', 'older_timestamp', 'date'])
    pairs = pd.read_csv(OUT / 'observations_pairs.csv', encoding='utf-8-sig', parse_dates=['timestamp', 'date'])
    data, _ = load()
    previous = data.set_index('timestamp').reindex(sample.input_timestamp)
    sample['prior_second'] = previous[SLOTS[1]].to_numpy()
    sample['prior_third'] = previous[SLOTS[2]].to_numpy()
    frame, summaries = daily_priorities(data)
    maximum_context(data, pairs)
    robust = []
    checks = dict(main=sample, same_date=sample.loc[sample.date.eq(sample.input_timestamp.dt.normalize())],
                  weekday=sample.loc[sample.weekend.eq(0)], known_production_zero=sample.loc[sample.past_production.eq(0)],
                  known_production_positive=sample.loc[sample.past_production.gt(0)])
    for name, cell in checks.items():
        for mode in ['date_hour', 'profile_balanced']:
            for row in slot_group_associations(cell, mode):
                robust.append(dict(check=name, weighting=mode, **row))
    for row in slot_group_associations(sample, degree=1):
        robust.append(dict(check='linear_controls', weighting='date_hour', **row))
    for row in slot_group_associations(sample, add_dates=True):
        robust.append(dict(check='date_controls', weighting='date_hour', **row))
    for month in range(1, 9):
        for name, cell in [(f'month_{month}', sample.loc[sample.month.eq(month)]),
                           (f'leave_month_{month}', sample.loc[sample.month.ne(month)])]:
            for row in slot_group_associations(cell):
                robust.append(dict(check=name, weighting='date_hour', **row))
    result = pd.DataFrame(robust)
    save(result, 'terminal_slot_robustness')
    # Independent daily calculation from the four raw columns, retaining ties.
    for row in frame.itertuples():
        cell = data.loc[data.date.eq(row.date)]
        maxima = [max(values) for values in cell[SLOTS].to_numpy()]
        means = [sum(values) / 4 for values in cell[SLOTS].to_numpy()]
        top = [maximum for maximum, mean in zip(maxima, means) if mean == max(means)]
        assert row.minimum_missed_maximum == max(maxima) - max(top)
        assert row.maximum_missed_maximum == max(maxima) - min(top)
        assert row.no_common_top == int(max(top) != max(maxima))
        assert len(cell) == 24
    assert old_hash == hashlib.sha256(observed_path.read_bytes()).hexdigest()
    assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == SHA
    audit = dict(status='passed', daily_tie_aware_diagnostics_checked=241, observations_unchanged=True,
                 source_sha256=SHA, script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
                 outputs_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.glob('*.csv'))})
    (OUT / 'diagnostic_verification.json').write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding='utf-8')
    print(pd.DataFrame(summaries).to_json(orient='records'))
    print(result.loc[(result.feature == 'prior_last') & (result.weighting == 'date_hour') &
                     (result.target == 'target_maximum')].to_json(orient='records'))
    print(result.loc[(result.check == 'main') & (result.weighting == 'date_hour')].to_json(orient='records'))


if __name__ == '__main__':
    main()


코드 등록 완료: `Analysis/scripts/a04_maximum_diagnostics.py`

### `Analysis/scripts/a04_support_diagnostics.py`

원고 연결: Analysis\03_Analysis_3페이지용_압축본.md; Analysis\03_Analysis_원고.md.

코드 설명: A04 final diagnosis: exact-mean support composition, not a threshold search.

주요 함수: `main()`.

In [24]:
%%submission_source Analysis/scripts/a04_support_diagnostics.py
"""A04 final diagnosis: exact-mean support composition, not a threshold search."""
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
from a01_production_changes import load
from a04_maximum_conditions import OUT, KEYS, association, save
from a04_terminal_unique import compare


def main():
    source = OUT / 'observations_triples.csv'
    source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
    (OUT / 'support_contract.json').write_text(json.dumps(dict(
        reason='Exact prior-mean matching has1039of5778hours, Jan-Apr r-.009 vs whole-control r+.394. Check support and observed low-power composition rather than alter matching bins.',
        comparison='Exact same calendar, prior unrounded mean and known0/positive state; require3dates. Describe retained/omitted records and cells with varying terminal values. Compare same selected subset under original controls.',
        low_power='Use existing EDA original rounded average20..26 at observed t-1, including zeros/extremes in other group. No new peak threshold or selection on future outcomes.',
        stop='Report support restriction and remaining heterogeneity; no more cutoff/transformation searches.'), ensure_ascii=False, indent=2), encoding='utf-8')
    sample = pd.read_csv(source, encoding='utf-8-sig', parse_dates=['timestamp','date','input_timestamp'])
    data, _ = load()
    previous = data.set_index('timestamp').reindex(sample.input_timestamp)
    sample['known_low_power'] = previous['평균'].between(20,26).to_numpy()
    exact_keys = KEYS + ['past_level','known_production_positive']
    keep = sample.groupby(exact_keys).date.transform('nunique').ge(3)
    vary = sample.groupby(exact_keys).prior_last.transform('nunique').gt(1)
    sample['exact_supported'] = keep
    sample['terminal_varies'] = vary
    support = []
    for period, scope in dict(all=sample, jan_apr=sample.loc[sample.month.le(4)],
                             may_jun=sample.loc[sample.month.between(5,6)],
                             jul_aug=sample.loc[sample.month.ge(7)]).items():
        for group, cell in [('available',scope),('exact_mean_supported',scope.loc[scope.exact_supported]),
                            ('supported_terminal_varies',scope.loc[scope.exact_supported & scope.terminal_varies])]:
            support.append(dict(period=period,group=group,n=len(cell),days=cell.date.nunique(),
                                known_low_hours=int(cell.known_low_power.sum()),
                                known_low_fraction=cell.known_low_power.mean(),
                                known_zero_hours=int(cell.past_production.eq(0).sum()),
                                prior_mean_median=cell.past_level.median(),prior_mean_q90=cell.past_level.quantile(.9),
                                target_maximum_median=cell.target_maximum.median(),
                                target_maximum_q90=cell.target_maximum.quantile(.9)))
    save(pd.DataFrame(support),'exact_mean_support')
    results=[]
    groups = dict(exact_supported=sample.loc[keep], known_low=sample.loc[sample.known_low_power],
                  known_other=sample.loc[~sample.known_low_power])
    for scope, cell in groups.items():
        for mode in ['date_hour','profile_balanced']:
            for row in compare(cell, mode, adjustment='mean_maximum'):
                results.append(dict(scope=scope,**row))
            for row in compare(cell, mode, adjustment='exact_mean'):
                results.append(dict(scope=scope,**row))
    for scope, cell in dict(jan_apr_exact=sample.loc[keep & sample.month.le(4)],
                            may_jun_exact=sample.loc[keep & sample.month.between(5,6)],
                            jul_aug_exact=sample.loc[keep & sample.month.ge(7)]).items():
        for row in compare(cell,adjustment='mean_maximum'):
            results.append(dict(scope=scope,**row))
    frame = pd.DataFrame(results)
    save(frame,'support_associations')
    assert len(sample) == 5778 and int(keep.sum()) == 1039
    assert source_hash == hashlib.sha256(source.read_bytes()).hexdigest()
    audit = dict(status='passed',source_observations_unchanged=True,
                 support_hours=1039,total_hours=5778,
                 script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
                 outputs_sha256={name:hashlib.sha256((OUT/f'{name}.csv').read_bytes()).hexdigest()
                                 for name in ['exact_mean_support','support_associations']})
    (OUT/'support_verification.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
    print(pd.DataFrame(support).to_json(orient='records'))
    print(frame.loc[frame.feature.eq('prior_last')].to_json(orient='records'))


if __name__ == '__main__':
    main()


코드 등록 완료: `Analysis/scripts/a04_support_diagnostics.py`

### `Analysis/scripts/a04_terminal_unique.py`

원고 연결: Analysis\03_Analysis_3페이지용_압축본.md; Analysis\03_Analysis_원고.md.

코드 설명: A04 third comparison: terminal-slot information beyond observed mean/max.

주요 함수: `compare()`, `main()`.

In [25]:
%%submission_source Analysis/scripts/a04_terminal_unique.py
"""A04 third comparison: terminal-slot information beyond observed mean/max."""
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from a02_prior_power_signals import residuals, weighted_corr
from a04_maximum_conditions import OUT, KEYS, control_array, save

FEATURES = ['prior_last', 'prior_slot_trend', 'prior_last_above_mean', 'prior_maximum_excess']


def compare(sample, mode='date_hour', adjustment='mean_maximum'):
    keys = KEYS if adjustment != 'exact_mean' else KEYS + ['past_level', 'known_production_positive']
    sample = sample.loc[sample.groupby(keys).date.transform('nunique').ge(3)].copy()
    if sample.empty:
        return []
    weights = np.ones(len(sample)) if mode == 'date_hour' else sample.profile_weight.to_numpy()
    values = np.column_stack([rankdata(sample[key]) / len(sample) for key in FEATURES + ['target_maximum']])
    cells = pd.factorize(pd.MultiIndex.from_frame(sample[keys]))[0]
    controls = control_array(sample, include_level=adjustment != 'exact_mean')
    if adjustment in ['mean_maximum', 'mean_maximum_trend']:
        maximum = rankdata(sample.prior_maximum) / len(sample)
        controls = np.column_stack([controls, maximum, maximum ** 2, maximum ** 3])
    if adjustment == 'mean_maximum_trend':
        controls = np.column_stack([controls, rankdata(sample.prior_slot_trend) / len(sample)])
    res = residuals(values, cells, weights, controls)
    return [dict(feature=feature, adjustment=adjustment, weighting=mode, n=len(sample),
                 days=sample.date.nunique(), cells=int(cells.max() + 1),
                 correlation=weighted_corr(res[:, i], res[:, -1], weights))
            for i, feature in enumerate(FEATURES)
            if not (adjustment == 'mean_maximum_trend' and feature == 'prior_slot_trend')]


def main():
    source = OUT / 'observations_triples.csv'
    source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
    (OUT / 'unique_contract.json').write_text(json.dumps(dict(
        reason='Terminal slot association robust, but also associated with next mean. Need check redundancy with prior maximum and mean-adjustment assumptions before forecasting candidate.',
        comparison='Retain continuous next maximum; add observed prior maximum cubic ranks to fixed controls, then prior trend; exact prior mean+production0state matched cells as non-binned sensitivity.',
        periods='Jan-Apr,May-Jun,Jul-Aug separate descriptive associations only, no fitting or independent test claim.',
        stop='Candidate retained only as observed antecedent if direction survives; no forecast training/threshold change or favorable subgroup selection.'), ensure_ascii=False, indent=2), encoding='utf-8')
    sample = pd.read_csv(source, encoding='utf-8-sig', parse_dates=['timestamp','date','input_timestamp'])
    rows = []
    for mode in ['date_hour','profile_balanced']:
        for adjustment in ['mean_maximum', 'mean_maximum_trend', 'exact_mean']:
            for row in compare(sample, mode, adjustment):
                rows.append(dict(scope='all', **row))
    for scope, cell in dict(jan_apr=sample.loc[sample.month.le(4)],
                            may_jun=sample.loc[sample.month.between(5,6)],
                            jul_aug=sample.loc[sample.month.ge(7)]).items():
        for adjustment in ['mean_maximum', 'exact_mean']:
            for row in compare(cell, adjustment=adjustment):
                rows.append(dict(scope=scope, **row))
    result = pd.DataFrame(rows)
    save(result, 'terminal_unique_information')
    # Verify the new mean+maximum residual calculation against explicit controls.
    tiny = sample.loc[(sample.month == 1) & (sample.weekend == 0)].copy()
    cells = pd.factorize(pd.MultiIndex.from_frame(tiny[KEYS]))[0]
    maximum = rankdata(tiny.prior_maximum) / len(tiny)
    controls = np.column_stack([control_array(tiny), maximum, maximum ** 2, maximum ** 3])
    values = np.column_stack([rankdata(tiny.prior_last), rankdata(tiny.target_maximum)])
    weights = tiny.profile_weight.to_numpy()
    computed = residuals(values, cells, weights, controls)
    design = np.column_stack([np.eye(cells.max()+1)[cells], controls])
    fitted = np.linalg.lstsq(design*np.sqrt(weights[:,None]), values*np.sqrt(weights[:,None]), rcond=None)[0]
    np.testing.assert_allclose(computed, values-design@fitted, atol=1e-9)
    assert source_hash == hashlib.sha256(source.read_bytes()).hexdigest()
    audit = dict(status='passed', source_observations_unchanged=True, explicit_dummy_maximum_control_verified=True,
                 script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
                 output_sha256=hashlib.sha256((OUT/'terminal_unique_information.csv').read_bytes()).hexdigest())
    (OUT/'unique_verification.json').write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding='utf-8')
    print(result.to_json(orient='records'))


if __name__ == '__main__':
    main()


코드 등록 완료: `Analysis/scripts/a04_terminal_unique.py`

### `Analysis/scripts/a06_power_state_transitions.py`

원고 연결: EDA\02_EDA_3페이지용_압축본.md; EDA\02_EDA_원고.md; Analysis\03_Analysis_3페이지용_압축본.md; Analysis\03_Analysis_원고.md.

코드 설명: Descriptive power transitions for existing manuscripts; no model fitting.

Question: do the low-power periods seen in EDA correspond to production
transitions, and how much of the observed hour-to-hour change occurs there?
Reuse EDA's rounded-mean 20..26 definition; zero is a separate state.
Positive values below 20 are separated from above-26 values after the first
pass found weekend 19 values. Crossing either boundary is not a peak label.

주요 함수: `load()`, `independent_verify()`, `episodes()`, `summarize()`, `main()`.

In [26]:
%%submission_source Analysis/scripts/a06_power_state_transitions.py
"""Descriptive power transitions for existing manuscripts; no model fitting.

Question: do the low-power periods seen in EDA correspond to production
transitions, and how much of the observed hour-to-hour change occurs there?
Reuse EDA's rounded-mean 20..26 definition; zero is a separate state.
Positive values below 20 are separated from above-26 values after the first
pass found weekend 19 values. Crossing either boundary is not a peak label.
"""
import csv
import hashlib
import json
from collections import Counter
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
SOURCE = ROOT / "data/origin/okm_augumented_2021.csv"
OUT = ROOT / "Analysis/tables/a06_power_state_transitions"
SLOTS = ["15분", "30분", "45분", "60분"]


def load():
    raw = pd.read_csv(SOURCE, encoding="utf-8-sig")
    dates = pd.to_datetime(raw["날짜"].astype(str), format="%Y%m%d")
    keep = dates.dt.month.between(1, 8) & raw["시간"].between(0, 23)
    d = raw.loc[keep].copy()
    d["timestamp"] = dates[keep] + pd.to_timedelta(d["시간"], unit="h")
    d = d.sort_values("timestamp").set_index("timestamp")
    assert d.index.is_unique and len(d) == 5784
    assert d[SLOTS].ge(0).all().all()
    d["mean"] = d[SLOTS].mean(axis=1)
    d["maximum"] = d[SLOTS].max(axis=1)
    d["state"] = np.select([d.maximum.eq(0), d["평균"].lt(20), d["평균"].between(20, 26)],
                           ["zero", "below20", "low"], default="above26")
    d["month"] = d.index.month
    d["hour"] = d.index.hour
    d["daytype"] = np.where(d.index.dayofweek < 5, "weekday", "weekend")
    for col in ["mean", "maximum", "state", "생산량", "60분"]:
        d["prior_" + col] = d[col].reindex(d.index - pd.Timedelta(hours=1)).to_numpy()
    d["delta_mean"] = d["mean"] - d.prior_mean
    d["delta_maximum"] = d.maximum - d.prior_maximum
    d["prior_last_minus_mean"] = d["prior_60분"] - d.prior_mean
    return d


def independent_verify(d, transitions):
    # Separate stdlib reconstruction checks all rows and exact-hour pairs.
    records = {}
    with SOURCE.open(encoding="utf-8-sig", newline="") as fh:
        for r in csv.DictReader(fh):
            day = datetime.strptime(r["날짜"], "%Y%m%d")
            hour = float(r["시간"])
            if day.month > 8 or not 0 <= hour <= 23:
                continue
            ts = day + timedelta(hours=hour)
            values = [float(r[s]) for s in SLOTS]
            state = "zero" if max(values) == 0 else "below20" if float(r["평균"]) < 20 else "low" if float(r["평균"]) <= 26 else "above26"
            records[ts] = (sum(values) / 4, max(values), state)
    counts = Counter()
    for ts, row in d.iterrows():
        m, maximum, state = records[ts.to_pydatetime()]
        assert (row["mean"], row.maximum, row.state) == (m, maximum, state)
        prior = records.get(ts.to_pydatetime() - timedelta(hours=1))
        if prior is not None:
            counts[(prior[2], state)] += 1
            assert row.delta_mean == m - prior[0]
            assert row.delta_maximum == maximum - prior[1]
    assert counts == Counter(transitions.groupby(["prior_state", "state"]).size().to_dict())
    assert sum(counts.values()) == 5781
    # The existing figures use M01's full valid timeline, not its model subsets.
    frame = pd.read_csv(ROOT / "Modeling/tables/m01/hourly_frame.csv", encoding="utf-8-sig")
    assert pd.to_datetime(frame.timestamp).tolist() == list(d.index)
    assert np.array_equal(frame.target_mean, d["mean"])
    assert np.array_equal(frame.target_maximum, d.maximum)
    return {"raw_rows_checked": len(records), "exact_hour_pairs_checked": sum(counts.values()),
            "existing_plot_frame_matches_raw": True}


def episodes(d):
    contiguous = d.index.to_series().diff().eq(pd.Timedelta(hours=1))
    group = (~(contiguous & d.state.eq(d.state.shift()))).cumsum()
    rows = []
    for _, part in d.groupby(group):
        if part.state.iloc[0] not in ("low", "zero"):
            continue
        start, end = part.index[0], part.index[-1]
        rows.append({"state": part.state.iloc[0], "start": start, "end": end,
                     "hours": len(part), "left_complete": start - pd.Timedelta(hours=1) in d.index,
                     "right_complete": end + pd.Timedelta(hours=1) in d.index,
                     "positive_production_hours": int(part["생산량"].gt(0).sum())})
    return pd.DataFrame(rows)


def summarize(d):
    t = d.loc[d.prior_state.notna()].copy()
    rows = []
    abs_total = t.delta_maximum.abs().sum()
    for (prior, state), g in t.groupby(["prior_state", "state"]):
        rows.append({"prior_state": prior, "state": state, "hours": len(g),
                     "dates": g.index.normalize().nunique(),
                     "median_delta_mean": g.delta_mean.median(),
                     "median_delta_maximum": g.delta_maximum.median(),
                     "negative_delta_maximum_hours": int(g.delta_maximum.lt(0).sum()),
                     "positive_delta_maximum_hours": int(g.delta_maximum.gt(0).sum()),
                     "mean_abs_delta_maximum": g.delta_maximum.abs().mean(),
                     "share_total_abs_delta_maximum": g.delta_maximum.abs().sum() / abs_total,
                     "minimum_delta_maximum": g.delta_maximum.min(),
                     "maximum_delta_maximum": g.delta_maximum.max(),
                     "production_zero_to_positive": int((g["prior_생산량"].eq(0) & g["생산량"].gt(0)).sum()),
                     "production_positive_to_zero": int((g["prior_생산량"].gt(0) & g["생산량"].eq(0)).sum()),
                     "prior_last_minus_mean_median": g.prior_last_minus_mean.median()})
    return t, pd.DataFrame(rows)


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    d = load()
    t, contrasts = summarize(d)
    verified = independent_verify(d, t)
    e = episodes(d)
    assert int(e.hours.sum()) == int(d.state.isin(["low", "zero"]).sum())
    assert len(e.loc[e.state.eq("zero")]) == 1
    tables = {
        "transition_summary": contrasts,
        "state_counts": d.groupby(["daytype", "state"]).size().rename("hours").reset_index(),
        "monthly_state_counts": d.groupby(["month", "state"]).size().rename("hours").reset_index(),
        "transition_hours": t.groupby(["prior_state", "state", "daytype", "hour"]).size().rename("hours").reset_index(),
        "episodes": e,
        "boundary_records": t.loc[t.prior_state.ne(t.state)].reset_index(),
        "largest_changes": pd.concat([t.nsmallest(5, "delta_maximum").assign(direction="drop"),
                                       t.nlargest(5, "delta_maximum").assign(direction="rise")]).reset_index(),
    }
    for name, table in tables.items():
        table.to_csv(OUT / f"{name}.csv", index=False, encoding="utf-8-sig")
    episode_summary = {}
    for state, g in e.groupby("state"):
        complete = g.loc[g.left_complete & g.right_complete]
        episode_summary[state] = {"observed_runs": len(g), "complete_runs": len(complete),
                                  "observed_hours": int(g.hours.sum()),
                                  "complete_duration_min": int(complete.hours.min()),
                                  "complete_duration_median": float(complete.hours.median()),
                                  "complete_duration_max": int(complete.hours.max())}
    summary = {"source_sha256": hashlib.sha256(SOURCE.read_bytes()).hexdigest(),
               "rows": len(d), "pairs": len(t), "state_counts": d.state.value_counts().to_dict(),
               "episodes": episode_summary, "verification": verified,
               "largest_changes_both_production_zero": bool((tables["largest_changes"]["생산량"].eq(0) & tables["largest_changes"]["prior_생산량"].eq(0)).all()),
               "definition": "zero: maximum=0; below20: positive and rounded mean<20; low: rounded mean20..26; above26: rounded mean>26",
               "revision": "Initial other group mixed weekend values below20 with above26. Separated at existing lower bound20 to avoid interpreting every low-range exit as a rise; no records excluded.",
               "scope": "all valid Jan-Aug hours, no model split, no new peak threshold, no model fit",
               "inference": "descriptive counts only; temporal and repeated-profile dependence retained",
               "outputs_sha256": {f"{name}.csv": hashlib.sha256((OUT / f"{name}.csv").read_bytes()).hexdigest() for name in tables}}
    (OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps({k: summary[k] for k in ["rows", "pairs", "state_counts", "episodes", "verification"]}, ensure_ascii=False))
    print(contrasts.to_string(index=False))


if __name__ == "__main__":
    main()


코드 등록 완료: `Analysis/scripts/a06_power_state_transitions.py`

### `Analysis/scripts/plot_a01_production_changes.py`

원고 연결: Analysis\03_Analysis_원고.md; Analysis\03_Analysis_원고.md.

코드 설명: Simple observed-change figure from verified matched comparison table.

주요 함수: `main()`.

In [27]:
%%submission_source Analysis/scripts/plot_a01_production_changes.py
"""Simple observed-change figure from verified matched comparison table."""
from pathlib import Path
import json
import hashlib
import warnings
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from PIL import Image

ROOT=Path(__file__).resolve().parents[2]
TABLE=ROOT/'Analysis/tables/a01_production_changes/transition_matched_contrasts.csv'
OUT=ROOT/'Analysis/figures'


def main():
    table=pd.read_csv(TABLE,encoding='utf-8-sig')
    row=table.loc[(table.weighting=='date_hour')&(table.a=='zero_to_positive')
                  &(table.b=='positive_to_positive')&(table.metric=='abs_power_delta')].iloc[0]
    vals=[row.adjusted_b,row.adjusted_a]
    font_manager.fontManager.addfont('C:/Windows/Fonts/malgun.ttf')
    plt.rcParams.update({'font.family':'Malgun Gothic','axes.unicode_minus':False,
        'font.size':12,'axes.spines.top':False,'axes.spines.right':False})
    fig,ax=plt.subplots(figsize=(7.4,4.6))
    bars=ax.bar([0,1],vals,color=['#7F9BB7','#D07834'],width=.5)
    ax.set_xticks([0,1],['직전·현재 모두 생산량 양수','직전 생산량 0 → 현재 양수'])
    ax.set_ylabel('직전 시간 대비 전력 변화 크기')
    ax.set_ylim(0,max(vals)*1.25)
    for bar,v in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,v+1.1,f'{v:.2f}',ha='center',fontsize=14)
    fig.tight_layout(pad=1.5)
    OUT.mkdir(exist_ok=True,parents=True)
    path=OUT/'a01_production_transition.png'
    with warnings.catch_warnings(record=True) as ws:
        warnings.simplefilter('always')
        fig.savefig(path,dpi=180,facecolor='white')
    plt.close(fig)
    assert not any('Glyph' in str(w.message) for w in ws)
    with Image.open(path) as img:
        img.resize((img.width//2,img.height//2)).save(OUT/'a01_production_transition_review.png')
    verification=dict(status='passed',input_sha256=hashlib.sha256(TABLE.read_bytes()).hexdigest(),
        script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),values=vals,
        actual_observations=True,forecast_or_oracle=False,visual_review=False)
    (ROOT/'Analysis/tables/a01_production_changes/figure_verification.json').write_text(json.dumps(verification,indent=2),encoding='utf-8')
    print(json.dumps(verification))


if __name__=='__main__':
    main()


코드 등록 완료: `Analysis/scripts/plot_a01_production_changes.py`

### `Analysis/scripts/plot_a03_weather.py`

원고 연결: Analysis\03_Analysis_3페이지용_압축본.md; Analysis\03_Analysis_원고.md; Analysis\03_Analysis_원고.md.

코드 설명: Plot fixed common-row weather comparisons; no new analysis/model fitting.

주요 함수: `main()`.

In [28]:
%%submission_source Analysis/scripts/plot_a03_weather.py
"""Plot fixed common-row weather comparisons; no new analysis/model fitting."""
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from a03_conditioned_weather import ROOT,OUT,WEATHER


def main():
    manifest=json.loads((OUT/'diagnostic_verification.json').read_text(encoding='utf-8'))
    source=OUT/'common_row_associations.csv'
    assert hashlib.sha256(source.read_bytes()).hexdigest()==manifest['outputs_sha256']['common_row_associations']
    data=pd.read_csv(source,encoding='utf-8-sig')
    scope=data.loc[data.scope.eq('weekday')]
    raw=scope.loc[scope.adjustment.eq('raw')].set_index('variable').loc[WEATHER]
    adjusted=scope.loc[scope.adjustment.eq('joint')].set_index('variable').loc[WEATHER]
    assert (raw.n==4102).all() and (adjusted.n==4102).all()
    plt.rcParams.update({'font.family':'Malgun Gothic','axes.unicode_minus':False,'font.size':12})
    fig,ax=plt.subplots(figsize=(8.7,4.8),layout='constrained')
    x=np.arange(4); width=.34
    a=ax.bar(x-width/2,raw.correlation,width,label='단순 관계',color='#A3B6C2')
    b=ax.bar(x+width/2,adjusted.correlation,width,label='월·시간대·생산량·다른 날씨를 고려',color='#267D8D')
    for bars in (a,b):
        ax.bar_label(bars,labels=[f'{bar.get_height():+.3f}' for bar in bars],padding=4,fontsize=11)
    ax.set_xticks(x,WEATHER); ax.set_ylim(-.18,.33)
    ax.set_ylabel('전력과의 순위 관계'); ax.axhline(0,color='#777777',linewidth=.8)
    ax.legend(loc='upper center',ncol=1,frameon=False)
    ax.spines[['top','right']].set_visible(False)
    figures=ROOT/'Analysis/figures'; path=figures/'a03_weather_conditions.png'
    fig.savefig(path,dpi=160); plt.close(fig)
    with Image.open(path) as img:
        img.resize((img.width//2,img.height//2),Image.Resampling.LANCZOS).save(figures/'a03_weather_conditions_review.png')
    verify=dict(status='passed',same4102weekday_records=True,
        note='Raw Spearman vs adjusted rank-residual partial association; neither is accuracy or causal effect.',
        path=str(path.relative_to(ROOT)),image_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
        visually_reviewed=False)
    (OUT/'figure_verification.json').write_text(json.dumps(verify,indent=2),encoding='utf-8')


if __name__=='__main__':
    main()


코드 등록 완료: `Analysis/scripts/plot_a03_weather.py`

### `Analysis/scripts/plot_a04_terminal_slots.py`

원고 연결: Analysis\03_Analysis_3페이지용_압축본.md; Analysis\03_Analysis_원고.md; Analysis\03_Analysis_원고.md.

코드 설명: Plot verified descriptive conditional associations, not forecast accuracy.

주요 함수: `main()`.

In [29]:
%%submission_source Analysis/scripts/plot_a04_terminal_slots.py
"""Plot verified descriptive conditional associations, not forecast accuracy."""
import hashlib
import json
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from a04_maximum_conditions import ROOT, OUT


def main():
    table = pd.read_csv(OUT/'terminal_slot_robustness.csv', encoding='utf-8-sig')
    rows = table.loc[(table.check == 'main') & (table.weighting == 'date_hour') &
                     (table.target == 'target_maximum')].set_index('feature')
    features = ['prior_first', 'prior_second', 'prior_third', 'prior_last']
    values = rows.loc[features, 'correlation'].to_numpy()
    assert rows.loc[features, 'n'].eq(5778).all()
    plt.rcParams.update({'font.family':'Malgun Gothic', 'axes.unicode_minus':False,
                         'font.size':13, 'axes.spines.top':False, 'axes.spines.right':False})
    fig, ax = plt.subplots(figsize=(8.8,4.5))
    x = np.arange(4)
    ax.bar(x, values, width=.55, color=['#98A3AF']*3+['#227F78'], zorder=3)
    ax.axhline(0,color='#6B7280',linewidth=1)
    ax.set_xticks(x, ['15분 값', '30분 값', '45분 값', '60분 값\n(마지막)'])
    ax.set_xlabel('관측을 마친 직전 시간의 전력값', labelpad=12)
    ax.set_ylabel('다음 시간 최댓값과의 순위 관계', labelpad=12)
    ax.set_ylim(-.22,.52)
    ax.set_yticks([-.2,0,.2,.4])
    ax.grid(axis='y',alpha=.16,zorder=0)
    for xi, value in zip(x,values):
        ax.text(xi,value+(.025 if value>=0 else -.03), f'{value:+.3f}',
                ha='center', va='bottom' if value>=0 else 'top', fontsize=14)
    fig.tight_layout(pad=1.6)
    output = ROOT/'Analysis/figures/a04_terminal_slots.png'
    output.parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(output,dpi=180,facecolor='white')
    fig.set_size_inches(4.4,2.25)
    # Preserve layout and text exactly in a 50% pixel review, via Pillow.
    from PIL import Image
    with Image.open(output) as source:
        source.resize((source.width//2,source.height//2),Image.Resampling.LANCZOS).save(output.with_name('a04_terminal_slots_review.png'))
    plt.close(fig)
    audit=dict(status='passed',same5778records=True,
               controls='Target calendar and known production plus observed prior mean cubic ranks; no target mean/production controls.',
               interpretation='Partial rank association, not accuracy, percentage improvement or causal effect.',
               path=str(output.relative_to(ROOT)),image_sha256=hashlib.sha256(output.read_bytes()).hexdigest(),visually_reviewed=False)
    (OUT/'figure_verification.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps(dict(status='passed',path=str(output)),ensure_ascii=False))


if __name__ == '__main__':
    main()


코드 등록 완료: `Analysis/scripts/plot_a04_terminal_slots.py`

### `Analysis/scripts/plot_analysis_transition_summary.py`

원고 연결: Analysis\03_Analysis_3페이지용_압축본.md.

코드 설명: Combine two distinct, verified transition analyses without recomputing them.

주요 함수: `main()`.

In [30]:
%%submission_source Analysis/scripts/plot_analysis_transition_summary.py
"""Combine two distinct, verified transition analyses without recomputing them."""
from pathlib import Path
import hashlib
import json

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]


def main():
    change_path = ROOT / 'Analysis/tables/a01_production_changes/transition_matched_contrasts.csv'
    class_path = ROOT / 'Analysis/tables/a02_transition_forecast/classification_test.csv'
    changes = pd.read_csv(change_path, encoding='utf-8-sig')
    row = changes.loc[(changes.weighting == 'date_hour') &
                      (changes.a == 'zero_to_positive') &
                      (changes.b == 'positive_to_positive') &
                      (changes.metric == 'abs_power_delta')]
    assert len(row) == 1
    row = row.iloc[0]
    assert int(row.n_a) == 150 and int(row.n_b) == 567
    scores = pd.read_csv(class_path, encoding='utf-8-sig')
    scores = scores.loc[scores.scope == 'all'].set_index('features').loc[['calendar', 'level', 'shape']]
    assert scores.n.eq(617).all() and scores.events.eq(52).all()
    assert scores.tp.tolist() == [35, 50, 50] and scores.fp.tolist() == [51, 25, 25]
    plt.rcParams.update({'font.family': 'Malgun Gothic', 'axes.unicode_minus': False,
                         'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={'width_ratios': [1, 1.6]})
    left, right = axes
    bars = left.bar([0, 1], [row.adjusted_b, row.adjusted_a], width=.5,
                    color=['#7F9BB7', '#D07834'])
    left.bar_label(bars, fmt='%.2f', padding=5)
    left.set_xticks([0, 1], ['양수 지속\n567시간', '0→양수 전환\n150시간'])
    left.set_ylabel('시간 사이 평균 전력 변화 크기')
    left.set_ylim(0, 47)
    x = np.arange(3)
    width = .32
    hits = right.bar(x-width/2, scores.tp, width, color='#24788A', label='실제 전환 적중')
    false = right.bar(x+width/2, scores.fp, width, color='#D78947', label='전환 오탐')
    right.bar_label(hits, padding=4)
    right.bar_label(false, padding=4)
    right.axhline(52, color='#777777', linestyle=':', linewidth=1)
    right.text(2.48, 53, '실제 전환 52시간', ha='right', fontsize=10, color='#666666')
    right.set_xticks(x, ['달력만', '달력 +\n직전 평균', '달력 + 직전 평균\n+ 마지막-첫 값'])
    right.set_ylabel('시간 수')
    right.set_ylim(0, 69)
    right.legend(loc='upper center', ncol=2, frameon=False, fontsize=11)
    fig.tight_layout(pad=1.0, w_pad=2)
    output = ROOT / 'Analysis/figures/a01_a02_transition_summary.png'
    fig.savefig(output, dpi=180, facecolor='white')
    plt.close(fig)
    manifest = {'status': 'passed', 'panels_use_distinct_samples_and_metrics': True,
                'left_n': 717, 'right_n': 617, 'right_events': 52,
                'source_sha256': {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
                                  for p in (change_path, class_path)},
                'image_sha256': hashlib.sha256(output.read_bytes()).hexdigest()}
    audit_path = ROOT / 'Analysis/tables/transition_summary_figure_verification.json'
    audit_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(manifest, ensure_ascii=False))


if __name__ == '__main__':
    main()


코드 등록 완료: `Analysis/scripts/plot_analysis_transition_summary.py`

## Modeling 원고에 사용한 전체 코드

아래 셀은 코드 정의를 등록한다. 계산 순서와 이번 실행 결과는 뒤의 1 ~ 5장에 정리한다.

### `Modeling/scripts/error_warning.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Observed-input ambiguity and chronological large-underprediction warning study.
Normal HGB B forecasts remain fixed; risk scores use past OOF error labels only.

주요 함수: `csvout()`, `read()`, `diagnose()`, `freeze()`, `scores()`, `metrics()`, `evaluate()`, `run()`.

In [31]:
%%submission_source Modeling/scripts/error_warning.py
"""Observed-input ambiguity and chronological large-underprediction warning study.
Normal HGB B forecasts remain fixed; risk scores use past OOF error labels only.
"""
import argparse,json,os,sys
from pathlib import Path
for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']:os.environ.setdefault(k,'4')
import numpy as np,pandas as pd,joblib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from m01_prepare import ROOT,sha
from profile_history import load,FEATURES
from redevelopment_r01 import now,dump
OUT=ROOT/'Modeling/tables/error_warning';MODELS=ROOT/'Modeling/models/error_warning'
RISK_FEATURES=FEATURES['B']+['prediction','neighbor_std','neighbor_range','neighbor_distance','neighbor_mean_minus_prediction']
def csvout(d,name):d.to_csv(OUT/(name+'.csv'),index=False,encoding='utf-8-sig')
def read(p):return pd.read_csv(p,encoding='utf-8-sig',float_precision='round_trip',parse_dates=['timestamp','date'])
def diagnose():
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    OUT.mkdir(parents=True,exist_ok=True);assert not (OUT/'diagnosis.json').exists()
    paths=['Modeling/scripts/error_warning.py','Modeling/scripts/profile_history.py','Modeling/tables/profile_history/run.json','Modeling/tables/profile_history/independent_verification.json','Modeling/tables/profile_history/inner_predictions.csv','Modeling/tables/profile_history/predictions.csv','Modeling/tables/m01/hourly_frame.csv','Modeling/config/m01_contract.json','data/origin/okm_augumented_2021.csv']
    contract={'recorded_at':now(),'inputs_sha256':{p:sha(ROOT/p) for p in paths},'scope':'Jan-Jun only; full B and H6 exact groups including calendar, then past-only nearest20 diagnostic',
        'neighbors':'training StandardScaler; Euclidean; exactly20 prior-month eligible rows; repeated neighbors retained and distinct-date/profile support recorded; no future evaluation outcomes',
        'exact':'all input columns equal; report within-group target variation, empirical median absolute deviation; not a population irreducible-error bound',
        'followup':'if full-input conflicts exist OR historical neighbor outcomes vary, test warning model; neither implies inevitable failure or guarantees error predictability'}
    dump(contract,OUT/'diagnosis_contract.json')
    ver=json.loads((ROOT/'Modeling/tables/profile_history/independent_verification.json').read_text(encoding='utf-8'));assert ver['status']=='passed' and ver['run_sha256']==sha(ROOT/'Modeling/tables/profile_history/run.json')
    f=load();f=f.loc[f.eligible_history & f.month.le(6)].copy()
    exact=[];summaries=[]
    for name in ['B','H6']:
        selected=f[FEATURES[name]+['target_maximum','timestamp','date']].copy()
        count=0;conflict_rows=0;conflicts=0;mad_sum=0;groupid=0
        for key,g in selected.groupby(FEATURES[name],sort=False):
            if len(g)<2:continue
            target=g.target_maximum.to_numpy();spread=float(np.ptp(target));mad=float(abs(target-np.median(target)).mean())
            exact.append({'inputs':name,'group':groupid,'rows':len(g),'dates':g.date.nunique(),'first':str(g.timestamp.min()),'last':str(g.timestamp.max()),'target_min':float(target.min()),'target_max':float(target.max()),'range':spread,'MAD_to_group_median':mad})
            groupid+=1;count+=len(g);mad_sum+=mad*len(g)
            if spread>0:conflicts+=1;conflict_rows+=len(g)
        summaries.append({'inputs':name,'eligible_rows':len(f),'repeated_input_groups':groupid,'repeated_input_rows':count,'conflicting_groups':conflicts,'conflicting_rows':conflict_rows,'within_repeated_empirical_MAD':mad_sum/count if count else None})
    csvout(pd.DataFrame(exact),'exact_groups')
    old=read(ROOT/'Modeling/tables/profile_history/inner_predictions.csv');old=old.loc[old.group.eq('B')]
    outer=read(ROOT/'Modeling/tables/profile_history/predictions.csv');outer=outer.loc[outer.method.eq('fixed_HGB_B')&outer.month.eq(6)]
    point=pd.concat([old[['timestamp','date','month','actual','prediction','daily_maximum_weight','train_profile_overlap']],outer[['timestamp','date','month','actual','prediction','daily_maximum_weight','train_profile_overlap']]],ignore_index=True)
    records=[];neighbor_records=[];audits=[]
    for month in range(2,7):
        tr=f.loc[f.month.lt(month)];ev=f.loc[f.month.eq(month)];q=point.loc[point.month.eq(month)].copy()
        assert ev.timestamp.tolist()==q.timestamp.tolist()
        for name in ['B','H6']:
            scaler=StandardScaler().fit(tr[FEATURES[name]]);tx=scaler.transform(tr[FEATURES[name]]);ex=scaler.transform(ev[FEATURES[name]])
            nn=NearestNeighbors(n_neighbors=20,algorithm='brute',n_jobs=1).fit(tx);dist,idx=nn.kneighbors(ex)
            y=tr.target_maximum.to_numpy()[idx];dates=tr.date.to_numpy()[idx];profiles=tr.diag_target_profile.to_numpy()[idx]
            data={'neighbor_std':np.std(y,axis=1),'neighbor_range':np.ptp(y,axis=1),'neighbor_distance':dist[:,-1],
                'neighbor_mean_minus_prediction':y.mean(1)-q.prediction.to_numpy(),'neighbor_dates':np.array([len(set(a)) for a in dates]),'neighbor_profiles':np.array([len(set(a)) for a in profiles])}
            if name=='B':
                r=q.merge(ev[['timestamp']+FEATURES['B']],on='timestamp',validate='one_to_one')
                for col,val in data.items():r[col]=val
                records.append(r)
            audits.append({'month':month,'inputs':name,'train_hours':len(tr),'eval_hours':len(ev),'mean_neighbor_std':float(np.std(y,axis=1).mean()),'mean_distinct_neighbor_dates':float(data['neighbor_dates'].mean()),'mean_distinct_neighbor_profiles':float(data['neighbor_profiles'].mean())})
            for i,ts in enumerate(ev.timestamp):
                for rank,j in enumerate(idx[i]):neighbor_records.append({'inputs':name,'timestamp':ts,'rank':rank+1,'neighbor_timestamp':tr.timestamp.iloc[j],'distance':float(dist[i,rank]),'neighbor_actual':float(y[i,rank])})
    d=pd.concat(records,ignore_index=True);d['under_amount']=(d.actual-d.prediction).clip(lower=0)
    csvout(d,'risk_frame');csvout(pd.DataFrame(neighbor_records),'neighbors');csvout(pd.DataFrame(audits),'neighbor_summary')
    result={'status':'completed','contract_sha256':sha(OUT/'diagnosis_contract.json'),'exact_summaries':summaries,
        'eligible_for_warning_trial':any(r['conflicting_rows']>0 for r in summaries) or d.neighbor_std.gt(0).any().item(),
        'note':'Exact conflicts describe finite recorded cases; nearest-state variability is not proof of an irreducible limit or operational risk.',
        'outputs_sha256':{p.name:sha(p) for p in OUT.glob('*.csv')}}
    dump(result,OUT/'diagnosis.json');print(json.dumps(result,ensure_ascii=False),flush=True)
def freeze():
    assert not (OUT/'contract.json').exists();diag=json.loads((OUT/'diagnosis.json').read_text(encoding='utf-8'));assert diag['eligible_for_warning_trial']
    d=read(OUT/'risk_frame.csv');feb=d.loc[d.month.eq(2)];threshold=float(np.quantile(feb.under_amount,.9));assert threshold>0
    c={'version':'error-warning-v1','recorded_at':now(),'entrypoint_sha256':sha(Path(__file__)),'diagnosis_sha256':sha(OUT/'diagnosis.json'),'frame_sha256':sha(OUT/'risk_frame.csv'),
        'event_threshold':threshold,'event_definition':'actual-fixed B prediction >= February OOF positive-or-zero underamount 90th percentile, ties included; diagnostic severity, not operational cost',
        'features':RISK_FEATURES,'point_model':'existing monthly fixed HGB B chronological predictions; no in-sample regression errors used',
        'methods':['high_prediction','prior_maximum','neighbor_std','Logistic','HGB'],
        'Logistic':{'C':1.0,'max_iter':3000,'random_state':42},'HGB':{'max_leaf_nodes':7,'max_iter':100,'learning_rate':.05,'min_samples_leaf':20,'l2_regularization':1.0,'early_stopping':False,'random_state':42},
        'splits':{'4':{'train':[2],'calibration':3},'5':{'train':[2,3],'calibration':4},'6':{'train':[2,3,4],'calibration':5}},
        'threshold':'calibration score linear q90; strictly greater to limit calibration alarms to <=10%; fixed for next month; no ranking with future hours in deployed alarm',
        'ranking_budget':'retrospective top floor(.1*n) per month, stable timestamp tie break; diagnostics only, not an online selection policy',
        'selection':'calibration AP max among Logistic,HGB; tie Logistic; best simple reference chosen by calibration AP among high_prediction,prior_maximum,neighbor_std, tie declaration order',
        'gate':'selected ML must beat calibration-selected simple reference: event-weighted monthly AP >=1.05x; retrospective fixed-budget TP >=reference; online pooled TP>=reference+1; online warnings<=reference and <=10% of all eval rows; each month online TP>=reference; each month AP>=.95x; otherwise reject',
        'scope':'4-6 development already reused, no new independent test. Warning review alone does not correct forecast or prove savings.',
        'no_tuning':True,'max_new_fits':6,'no_manuscript_edit':True,'later':'only if eligible, freeze latest policy for later already-observed evaluation; not run automatically within current dev code'}
    dump(c,OUT/'contract.json');print('Warning contract frozen; threshold='+str(threshold),flush=True)
def scores(m,x):return m.predict_proba(x)[:,1]
def metrics(g):
    y=g.event.to_numpy(bool);a=g.alarm.to_numpy(bool);t=g.top_budget.to_numpy(bool);e=g.under_amount.to_numpy()
    tp=int((y&a).sum());fp=int((~y&a).sum());n=int(y.sum())
    return {'hours':len(g),'events':n,'AP':float(average_precision_score(y,g.score)) if n else 0.,'warnings':int(a.sum()),'warning_fraction':float(a.mean()),'TP':tp,'FP':fp,'FN':n-tp,
        'precision':tp/int(a.sum()) if a.any() else 0.,'recall':tp/n if n else 0.,'top_budget_TP':int((y&t).sum()),'unwarned_events':int((y&~a).sum()),
        'unwarned_under_mean':float(e[~a].mean()) if (~a).any() else None,'warned_under_share':float(e[a].sum()/e.sum()) if e.sum() else 0.}
def evaluate(p):
    rows=[]
    for (method,month),g in p.groupby(['method','month']):rows.append({'method':method,'period':str(month),**metrics(g)})
    for method,g in p.groupby('method'):
        r=metrics(g);sub=[x for x in rows if x['method']==method];r['AP']=sum(x['AP']*x['events'] for x in sub)/sum(x['events'] for x in sub);rows.append({'method':method,'period':'pooled','AP_definition':'event-weighted monthly AP',**r})
    table=pd.DataFrame(rows);i=table.set_index(['method','period']);a=i.loc[('selected_ML','pooled')];b=i.loc[('selected_simple','pooled')]
    flags={'AP_5percent':bool(a.AP>=b.AP*1.05),'ranking_same_budget':bool(a.top_budget_TP>=b.top_budget_TP),'online_TP_gain':bool(a.TP>=b.TP+1),'no_more_warnings':bool(a.warnings<=b.warnings),'warning_cap':bool(a.warning_fraction<=.1),
        'monthly_TP':all(i.loc[('selected_ML',str(m)),'TP']>=i.loc[('selected_simple',str(m)),'TP'] for m in [4,5,6]),
        'monthly_AP':all(i.loc[('selected_ML',str(m)),'AP']>=i.loc[('selected_simple',str(m)),'AP']*.95 for m in [4,5,6])}
    return table,{'eligible':all(flags.values()),'gates':flags}
def run():
    assert not (OUT/'run.json').exists();c=json.loads((OUT/'contract.json').read_text(encoding='utf-8'))
    assert c['entrypoint_sha256']==sha(Path(__file__)) and c['frame_sha256']==sha(OUT/'risk_frame.csv')
    d=read(OUT/'risk_frame.csv');d['event']=d.under_amount.ge(c['event_threshold']);MODELS.mkdir(parents=True,exist_ok=True)
    ps=[];cs=[];models=[];selections=[];fits=0
    for month in [4,5,6]:
        spec=c['splits'][str(month)];tr=d.loc[d.month.isin(spec['train'])];cal=d.loc[d.month.eq(spec['calibration'])];ev=d.loc[d.month.eq(month)]
        assert tr.timestamp.max()<cal.timestamp.min()<ev.timestamp.min() and tr.event.nunique()==2
        candidates={};calibrations={}
        for name in c['methods']:
            if name in ['Logistic','HGB']:
                m=make_pipeline(StandardScaler(),LogisticRegression(**c['Logistic'])) if name=='Logistic' else HistGradientBoostingClassifier(**c['HGB'])
                m.fit(tr[c['features']],tr.event);fits+=1;path=MODELS/f'month{month}_{name}.joblib';joblib.dump(m,path,compress=3)
                cp=scores(m,cal[c['features']]);ep=scores(m,ev[c['features']]);assert np.allclose(scores(joblib.load(path),ev[c['features']]),ep,atol=1e-12)
                models.append({'month':month,'method':name,'file':str(path.relative_to(ROOT)),'sha256':sha(path),'train_rows':len(tr),'calibration_rows':len(cal),'eval_rows':len(ev),'train_end':str(tr.timestamp.max())})
            else:
                col={'high_prediction':'prediction','prior_maximum':'lag1_maximum','neighbor_std':'neighbor_std'}[name];cp=cal[col].to_numpy();ep=ev[col].to_numpy()
            boundary=float(np.quantile(cp,.9))
            def output(frame,score):
                z=frame[['timestamp','date','month','actual','prediction','under_amount','event','daily_maximum_weight','train_profile_overlap']].copy();z['method']=name;z['score']=score;z['threshold']=boundary;z['alarm']=z.score.gt(boundary)
                z['top_budget']=False;order=np.argsort(-np.asarray(score),kind='stable')[:int(.1*len(z))];z.iloc[order,z.columns.get_loc('top_budget')]=True;return z
            cc=output(cal,cp);cc['outer_month']=month;ee=output(ev,ep);cs.append(cc);ps.append(ee);candidates[name]=ee
            calibrations[name]=float(average_precision_score(cal.event,cp));assert cc.alarm.mean()<=.1+1e-12
        simple=max(c['methods'][:3],key=lambda n:calibrations[n]);ml=max(c['methods'][3:],key=lambda n:calibrations[n]);selections.append({'month':month,'simple':simple,'ML':ml,'calibration_AP':calibrations})
        for selected,label in [(simple,'selected_simple'),(ml,'selected_ML')]:z=candidates[selected].copy();z['method']=label;ps.append(z)
        print(f'Warning month={month}: reference={simple}, ML={ml}, train={len(tr)}, cal={len(cal)}, eval={len(ev)}',flush=True)
    p=pd.concat(ps,ignore_index=True);cp=pd.concat(cs,ignore_index=True);table,decision=evaluate(p)
    csvout(p,'predictions');csvout(cp,'calibration_predictions');csvout(table,'metrics');csvout(pd.DataFrame(models),'model_manifest');dump(selections,OUT/'selection.json');dump(decision,OUT/'decision.json')
    result={'status':'completed','contract_sha256':sha(OUT/'contract.json'),'new_fits':fits,'development_hours':2184,'event_threshold':c['event_threshold'],'events':int(p.loc[p.method.eq('selected_ML'),'event'].sum()),'decision':decision,
        'later_evaluated':False,'manuscripts_edited':False,'outputs_sha256':{x.name:sha(x) for x in [OUT/'predictions.csv',OUT/'calibration_predictions.csv',OUT/'metrics.csv',OUT/'model_manifest.csv',OUT/'selection.json',OUT/'decision.json']}}
    dump(result,OUT/'run.json');print(table.loc[table.period.eq('pooled')].to_string(index=False));print(json.dumps(result,ensure_ascii=False),flush=True)
if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('stage',choices=['diagnose','freeze','run']);a=p.parse_args();globals()[a.stage]()

코드 등록 완료: `Modeling/scripts/error_warning.py`

### `Modeling/scripts/error_warning_fit.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Observed-input ambiguity and chronological large-underprediction warning study.
Normal HGB B forecasts remain fixed; risk scores use past OOF error labels only.

주요 함수: `csvout()`, `read()`, `freeze()`, `scores()`, `metrics()`, `evaluate()`, `run()`.

In [32]:
%%submission_source Modeling/scripts/error_warning_fit.py
"""Observed-input ambiguity and chronological large-underprediction warning study.
Normal HGB B forecasts remain fixed; risk scores use past OOF error labels only.
"""
import argparse,json,os,sys
from pathlib import Path
for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']:os.environ.setdefault(k,'4')
import numpy as np,pandas as pd,joblib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from m01_prepare import ROOT,sha
from profile_history import load,FEATURES
from redevelopment_r01 import now,dump
OUT=ROOT/'Modeling/tables/error_warning';MODELS=ROOT/'Modeling/models/error_warning'
RISK_FEATURES=FEATURES['B']+['prediction','neighbor_std','neighbor_range','neighbor_distance','neighbor_mean_minus_prediction']
def csvout(d,name):d.to_csv(OUT/(name+'.csv'),index=False,encoding='utf-8-sig')
def read(p):
    d=pd.read_csv(p,encoding='utf-8-sig',float_precision='round_trip',parse_dates=['timestamp','date'])
    # Diagnostic join stored both equal month columns. Normalize only in memory;
    # preserve frozen diagnostic artifact and verify its two sources agree.
    if 'month_x' in d and 'month_y' in d:
        assert d.month_x.equals(d.month_y) and np.array_equal(d.month_x,d.timestamp.dt.month)
        d=d.rename(columns={'month_x':'month'}).drop(columns='month_y')
    return d
def freeze():
    assert not (OUT/'contract.json').exists();diag=json.loads((OUT/'diagnosis.json').read_text(encoding='utf-8'));assert diag['eligible_for_warning_trial']
    d=read(OUT/'risk_frame.csv');feb=d.loc[d.month.eq(2)];threshold=float(np.quantile(feb.under_amount,.9));assert threshold>0
    c={'version':'error-warning-v1','recorded_at':now(),'entrypoint_sha256':sha(Path(__file__)),'diagnosis_sha256':sha(OUT/'diagnosis.json'),'frame_sha256':sha(OUT/'risk_frame.csv'),
        'event_threshold':threshold,'event_definition':'actual-fixed B prediction >= February OOF positive-or-zero underamount 90th percentile, ties included; diagnostic severity, not operational cost',
        'features':RISK_FEATURES,'point_model':'existing monthly fixed HGB B chronological predictions; no in-sample regression errors used',
        'methods':['high_prediction','prior_maximum','neighbor_std','Logistic','HGB'],
        'Logistic':{'C':1.0,'max_iter':3000,'random_state':42},'HGB':{'max_leaf_nodes':7,'max_iter':100,'learning_rate':.05,'min_samples_leaf':20,'l2_regularization':1.0,'early_stopping':False,'random_state':42},
        'splits':{'4':{'train':[2],'calibration':3},'5':{'train':[2,3],'calibration':4},'6':{'train':[2,3,4],'calibration':5}},
        'threshold':'calibration ascending score at index n-floor(.1*n)-1; strictly greater, hence <=floor(.1*n) alerts including ties; fixed for next month; no future-hour ranking',
        'ranking_budget':'retrospective top floor(.1*n) per month, stable timestamp tie break; diagnostics only, not an online selection policy',
        'selection':'calibration AP max among Logistic,HGB; tie Logistic; best simple reference chosen by calibration AP among high_prediction,prior_maximum,neighbor_std, tie declaration order',
        'gate':'selected ML must beat calibration-selected simple reference: event-weighted monthly AP >=1.05x; retrospective fixed-budget TP >=reference; online pooled TP>=reference+1; online warnings<=reference and <=10% of all eval rows; each month online TP>=reference; each month AP>=.95x; otherwise reject',
        'scope':'4-6 development already reused, no new independent test. Warning review alone does not correct forecast or prove savings.',
        'preflight_schema':'diagnostic month_x/month_y verified equal and normalized in memory before any warning fit; frozen diagnosis unchanged','no_tuning':True,'max_new_fits':6,'no_manuscript_edit':True,'later':'only if eligible, freeze latest policy for later already-observed evaluation; not run automatically within current dev code'}
    dump(c,OUT/'contract.json');print('Warning contract frozen; threshold='+str(threshold),flush=True)
def scores(m,x):return m.predict_proba(x)[:,1]
def metrics(g):
    y=g.event.to_numpy(bool);a=g.alarm.to_numpy(bool);t=g.top_budget.to_numpy(bool);e=g.under_amount.to_numpy()
    tp=int((y&a).sum());fp=int((~y&a).sum());n=int(y.sum())
    return {'hours':len(g),'events':n,'AP':float(average_precision_score(y,g.score)) if n else 0.,'warnings':int(a.sum()),'warning_fraction':float(a.mean()),'TP':tp,'FP':fp,'FN':n-tp,
        'precision':tp/int(a.sum()) if a.any() else 0.,'recall':tp/n if n else 0.,'top_budget_TP':int((y&t).sum()),'unwarned_events':int((y&~a).sum()),
        'unwarned_under_mean':float(e[~a].mean()) if (~a).any() else None,'warned_under_share':float(e[a].sum()/e.sum()) if e.sum() else 0.}
def evaluate(p):
    rows=[]
    for (method,month),g in p.groupby(['method','month']):rows.append({'method':method,'period':str(month),**metrics(g)})
    for method,g in p.groupby('method'):
        r=metrics(g);sub=[x for x in rows if x['method']==method];r['AP']=sum(x['AP']*x['events'] for x in sub)/sum(x['events'] for x in sub);rows.append({'method':method,'period':'pooled','AP_definition':'event-weighted monthly AP',**r})
    table=pd.DataFrame(rows);i=table.set_index(['method','period']);a=i.loc[('selected_ML','pooled')];b=i.loc[('selected_simple','pooled')]
    flags={'AP_5percent':bool(a.AP>=b.AP*1.05),'ranking_same_budget':bool(a.top_budget_TP>=b.top_budget_TP),'online_TP_gain':bool(a.TP>=b.TP+1),'no_more_warnings':bool(a.warnings<=b.warnings),'warning_cap':bool(a.warning_fraction<=.1),
        'monthly_TP':all(i.loc[('selected_ML',str(m)),'TP']>=i.loc[('selected_simple',str(m)),'TP'] for m in [4,5,6]),
        'monthly_AP':all(i.loc[('selected_ML',str(m)),'AP']>=i.loc[('selected_simple',str(m)),'AP']*.95 for m in [4,5,6])}
    return table,{'eligible':all(flags.values()),'gates':flags}
def run():
    assert not (OUT/'run.json').exists();c=json.loads((OUT/'contract.json').read_text(encoding='utf-8'))
    assert c['entrypoint_sha256']==sha(Path(__file__)) and c['frame_sha256']==sha(OUT/'risk_frame.csv')
    d=read(OUT/'risk_frame.csv');d['event']=d.under_amount.ge(c['event_threshold']);MODELS.mkdir(parents=True,exist_ok=True)
    ps=[];cs=[];models=[];selections=[];fits=0
    for month in [4,5,6]:
        spec=c['splits'][str(month)];tr=d.loc[d.month.isin(spec['train'])];cal=d.loc[d.month.eq(spec['calibration'])];ev=d.loc[d.month.eq(month)]
        assert tr.timestamp.max()<cal.timestamp.min()<ev.timestamp.min() and tr.event.nunique()==2
        candidates={};calibrations={}
        for name in c['methods']:
            if name in ['Logistic','HGB']:
                m=make_pipeline(StandardScaler(),LogisticRegression(**c['Logistic'])) if name=='Logistic' else HistGradientBoostingClassifier(**c['HGB'])
                m.fit(tr[c['features']],tr.event);fits+=1;path=MODELS/f'month{month}_{name}.joblib';joblib.dump(m,path,compress=3)
                cp=scores(m,cal[c['features']]);ep=scores(m,ev[c['features']]);assert np.allclose(scores(joblib.load(path),ev[c['features']]),ep,atol=1e-12)
                models.append({'month':month,'method':name,'file':str(path.relative_to(ROOT)),'sha256':sha(path),'train_rows':len(tr),'calibration_rows':len(cal),'eval_rows':len(ev),'train_end':str(tr.timestamp.max())})
            else:
                col={'high_prediction':'prediction','prior_maximum':'lag1_maximum','neighbor_std':'neighbor_std'}[name];cp=cal[col].to_numpy();ep=ev[col].to_numpy()
            boundary=float(np.sort(cp)[len(cp)-int(.1*len(cp))-1])
            def output(frame,score):
                z=frame[['timestamp','date','month','actual','prediction','under_amount','event','daily_maximum_weight','train_profile_overlap']].copy();z['method']=name;z['score']=score;z['threshold']=boundary;z['alarm']=z.score.gt(boundary)
                z['top_budget']=False;order=np.argsort(-np.asarray(score),kind='stable')[:int(.1*len(z))];z.iloc[order,z.columns.get_loc('top_budget')]=True;return z
            cc=output(cal,cp);cc['outer_month']=month;ee=output(ev,ep);cs.append(cc);ps.append(ee);candidates[name]=ee
            calibrations[name]=float(average_precision_score(cal.event,cp));assert cc.alarm.mean()<=.1+1e-12
        simple=max(c['methods'][:3],key=lambda n:calibrations[n]);ml=max(c['methods'][3:],key=lambda n:calibrations[n]);selections.append({'month':month,'simple':simple,'ML':ml,'calibration_AP':calibrations})
        for selected,label in [(simple,'selected_simple'),(ml,'selected_ML')]:z=candidates[selected].copy();z['method']=label;ps.append(z)
        print(f'Warning month={month}: reference={simple}, ML={ml}, train={len(tr)}, cal={len(cal)}, eval={len(ev)}',flush=True)
    p=pd.concat(ps,ignore_index=True);cp=pd.concat(cs,ignore_index=True);table,decision=evaluate(p)
    csvout(p,'predictions');csvout(cp,'calibration_predictions');csvout(table,'metrics');csvout(pd.DataFrame(models),'model_manifest');dump(selections,OUT/'selection.json');dump(decision,OUT/'decision.json')
    result={'status':'completed','contract_sha256':sha(OUT/'contract.json'),'new_fits':fits,'development_hours':2184,'event_threshold':c['event_threshold'],'events':int(p.loc[p.method.eq('selected_ML'),'event'].sum()),'decision':decision,
        'later_evaluated':False,'manuscripts_edited':False,'outputs_sha256':{x.name:sha(x) for x in [OUT/'predictions.csv',OUT/'calibration_predictions.csv',OUT/'metrics.csv',OUT/'model_manifest.csv',OUT/'selection.json',OUT/'decision.json']}}
    dump(result,OUT/'run.json');print(table.loc[table.period.eq('pooled')].to_string(index=False));print(json.dumps(result,ensure_ascii=False),flush=True)
if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('stage',choices=['freeze','run']);a=p.parse_args();globals()[a.stage]()

코드 등록 완료: `Modeling/scripts/error_warning_fit.py`

### `Modeling/scripts/error_warning_verify.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Independent audit: exact groups, past-only neighbors, warnings, AP and refits.

주요 함수: `sha()`, `dump()`, `close()`, `read()`, `ap()`, `main()`.

In [33]:
%%submission_source Modeling/scripts/error_warning_verify.py
"""Independent audit: exact groups, past-only neighbors, warnings, AP and refits."""
import os,json,hashlib,sys
from collections import defaultdict
from pathlib import Path
for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']:os.environ.setdefault(k,'4')
import numpy as np,pandas as pd,joblib
from sklearn.ensemble import HistGradientBoostingClassifier
from profile_history import load,FEATURES
ROOT=Path(__file__).resolve().parents[2];OUT=ROOT/'Modeling/tables/error_warning'
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def dump(x,p):p.write_text(json.dumps(x,ensure_ascii=False,indent=2),encoding='utf-8')
def close(a,b):assert np.allclose(a,b,rtol=1e-9,atol=1e-8),(a,b)
def read(name):return pd.read_csv(OUT/(name+'.csv'),encoding='utf-8-sig',float_precision='round_trip')
def ap(y,s):
    pairs=sorted(zip(s,y),reverse=True);total=sum(y)
    if total==0:return 0.
    hits=0;count=0;result=0.;i=0
    while i<len(pairs):
        j=i
        while j<len(pairs) and pairs[j][0]==pairs[i][0]:j+=1
        added=sum(int(v) for _,v in pairs[i:j]);hits+=added;count+=j-i;result+=added/total*hits/count;i=j
    return result

def main():
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    run=json.loads((OUT/'run.json').read_text(encoding='utf-8'));c=json.loads((OUT/'contract.json').read_text(encoding='utf-8'))
    diag=json.loads((OUT/'diagnosis.json').read_text(encoding='utf-8'));dc=json.loads((OUT/'diagnosis_contract.json').read_text(encoding='utf-8'))
    assert run['contract_sha256']==sha(OUT/'contract.json') and c['entrypoint_sha256']==sha(ROOT/'Modeling/scripts/error_warning_fit.py')
    assert c['diagnosis_sha256']==sha(OUT/'diagnosis.json') and diag['contract_sha256']==sha(OUT/'diagnosis_contract.json')
    for p,h in dc['inputs_sha256'].items():assert sha(ROOT/p)==h,p
    for p,h in {**diag['outputs_sha256'],**run['outputs_sha256']}.items():assert sha(OUT/p)==h,p
    f=load();f=f.loc[f.eligible_history & f.month.le(6)].copy()
    # Cross-check reconstructed raw features against independently verified earlier audit.
    parent=ROOT/'Modeling/tables/profile_history';pv=json.loads((parent/'independent_verification.json').read_text(encoding='utf-8'));pr=json.loads((parent/'run.json').read_text(encoding='utf-8'))
    assert pv['status']=='passed' and pv['run_sha256']==sha(parent/'run.json')
    saved=pd.read_csv(parent/'feature_audit.csv',encoding='utf-8-sig',float_precision='round_trip',parse_dates=['timestamp']);saved=saved.loc[saved.timestamp.dt.month.le(6)]
    key=next(k for k in pr['outputs_sha256'] if k.endswith('feature_audit.csv'));assert sha(parent/'feature_audit.csv')==pr['outputs_sha256'][key]
    assert f.timestamp.tolist()==saved.timestamp.tolist();cols=list(dict.fromkeys(FEATURES['B']+FEATURES['H6']+['target_maximum']));close(f[cols],saved[cols])
    for name in ['B','H6']:
        groups=defaultdict(list)
        for row,y in zip(f[FEATURES[name]].to_numpy(),f.target_maximum):groups[tuple(row)].append(y)
        repeated=[v for v in groups.values() if len(v)>1];conflicts=[v for v in repeated if max(v)>min(v)];d=next(x for x in diag['exact_summaries'] if x['inputs']==name)
        assert d['repeated_input_groups']==len(repeated) and d['repeated_input_rows']==sum(map(len,repeated))
        assert d['conflicting_groups']==len(conflicts) and d['conflicting_rows']==sum(map(len,conflicts))
        close(d['within_repeated_empirical_MAD'],sum(sum(abs(np.array(v)-np.median(v))) for v in repeated)/sum(map(len,repeated)))
    risk=read('risk_frame');risk.timestamp=pd.to_datetime(risk.timestamp)
    assert risk.month_x.equals(risk.month_y) and np.array_equal(risk.month_x,risk.timestamp.dt.month)
    risk=risk.rename(columns={'month_x':'month'}).drop(columns='month_y')
    neighbors=read('neighbors');neighbors.timestamp=pd.to_datetime(neighbors.timestamp);neighbors.neighbor_timestamp=pd.to_datetime(neighbors.neighbor_timestamp)
    checked=0
    for month in range(2,7):
        tr=f.loc[f.month.lt(month)].copy();ev=f.loc[f.month.eq(month)].copy();rr=risk.loc[risk.month.eq(month)]
        assert rr.timestamp.tolist()==ev.timestamp.tolist()
        for name in ['B','H6']:
            tx=tr[FEATURES[name]].to_numpy();ex=ev[FEATURES[name]].to_numpy();mean=tx.mean(0);scale=tx.std(0);scale[scale==0]=1.;tx=(tx-mean)/scale;ex=(ex-mean)/scale
            ns=neighbors.loc[neighbors.inputs.eq(name)&neighbors.timestamp.dt.month.eq(month)].sort_values(['timestamp','rank'])
            assert len(ns)==len(ev)*20 and ns.groupby('timestamp')['rank'].apply(list).map(lambda x:x==list(range(1,21))).all()
            assert ns.groupby('timestamp').neighbor_timestamp.nunique().eq(20).all() and ns.neighbor_timestamp.max()<ev.timestamp.min()
            ix=pd.Index(tr.timestamp).get_indexer(ns.neighbor_timestamp);assert (ix>=0).all();ix=ix.reshape(len(ev),20)
            distance=np.linalg.norm(tx[ix]-ex[:,None,:],axis=2);close(distance,ns.distance.to_numpy().reshape(len(ev),20))
            # Check nearest membership while allowing exact tied distances.
            for start in range(0,len(ev),64):
                x=ex[start:start+64];sq=np.maximum((x*x).sum(1)[:,None]+(tx*tx).sum(1)[None,:]-2*x@tx.T,0)
                smallest=np.sqrt(np.partition(sq,19,axis=1)[:,:20].max(1));close(smallest,distance[start:start+64,-1])
            ys=tr.target_maximum.to_numpy()[ix];close(ys,ns.neighbor_actual.to_numpy().reshape(len(ev),20));checked+=len(ns)
            if name=='B':
                close(rr.neighbor_std,ys.std(1));close(rr.neighbor_range,np.ptp(ys,axis=1));close(rr.neighbor_distance,distance[:,-1]);close(rr.neighbor_mean_minus_prediction,ys.mean(1)-rr.prediction)
                close(rr[FEATURES['B']],ev[FEATURES['B']])
    # Normal predictions remain the verified chronological point forecasts.
    pi=pd.read_csv(parent/'inner_predictions.csv',encoding='utf-8-sig',float_precision='round_trip',parse_dates=['timestamp']);po=pd.read_csv(parent/'predictions.csv',encoding='utf-8-sig',float_precision='round_trip',parse_dates=['timestamp'])
    refs=pd.concat([pi.loc[pi.group.eq('B'),['timestamp','actual','prediction']],po.loc[po.method.eq('fixed_HGB_B')&po.month.eq(6),['timestamp','actual','prediction']]])
    assert refs.timestamp.tolist()==risk.timestamp.tolist();close(refs[['actual','prediction']],risk[['actual','prediction']])
    close(risk.under_amount,np.maximum(risk.actual-risk.prediction,0));close(np.quantile(risk.loc[risk.month.eq(2),'under_amount'],.9),c['event_threshold'])
    risk['event']=risk.under_amount.ge(c['event_threshold'])
    p=read('predictions');p.timestamp=pd.to_datetime(p.timestamp);cal=read('calibration_predictions');cal.timestamp=pd.to_datetime(cal.timestamp)
    manifest=read('model_manifest');models=0;refits=0;prediction_checks=0
    for mr in manifest.to_dict('records'):
        mth=mr['month'];spec=c['splits'][str(mth)];tr=risk.loc[risk.month.isin(spec['train'])];ca=risk.loc[risk.month.eq(spec['calibration'])];ev=risk.loc[risk.month.eq(mth)]
        assert tr.timestamp.max()<ca.timestamp.min()<ev.timestamp.min() and len(tr)==mr['train_rows']
        path=ROOT/mr['file'];assert sha(path)==mr['sha256'];m=joblib.load(path)
        for g,z in [(ca,cal.loc[cal.outer_month.eq(mth)&cal.method.eq(mr['method'])]),(ev,p.loc[p.month.eq(mth)&p.method.eq(mr['method'])])]:
            assert z.timestamp.tolist()==g.timestamp.tolist();close(m.predict_proba(g[c['features']])[:,1],z.score);prediction_checks+=len(g)
        if mr['method']=='HGB':
            fresh=HistGradientBoostingClassifier(**c['HGB']).fit(tr[c['features']],tr.event);close(fresh.predict_proba(ev[c['features']])[:,1],m.predict_proba(ev[c['features']])[:,1]);refits+=1
        models+=1
    selections=json.loads((OUT/'selection.json').read_text(encoding='utf-8'));mt=read('metrics');computed={}
    for month in [4,5,6]:
        ca=cal.loc[cal.outer_month.eq(month)];aps={}
        for name in c['methods']:
            cc=ca.loc[ca.method.eq(name)];ee=p.loc[p.month.eq(month)&p.method.eq(name)];a=np.sort(cc.score.to_numpy());boundary=a[len(a)-int(.1*len(a))-1]
            close(cc.threshold,boundary);close(ee.threshold,boundary)
            for g in [cc,ee]:
                assert np.array_equal(g.event,g.under_amount.ge(c['event_threshold']));assert np.array_equal(g.alarm,g.score.gt(boundary))
                flag=np.zeros(len(g),bool);flag[np.argsort(-g.score.to_numpy(),kind='stable')[:int(.1*len(g))]]=True;assert np.array_equal(flag,g.top_budget)
            assert cc.alarm.sum()<=int(.1*len(cc));aps[name]=ap(cc.event.tolist(),cc.score.tolist())
            if name in c['methods'][:3]:
                col={'high_prediction':'prediction','prior_maximum':'lag1_maximum','neighbor_std':'neighbor_std'}[name]
                for g,mm in [(cc,int(cc.month.iloc[0])),(ee,month)]:close(g.score,risk.loc[risk.month.eq(mm),col])
        sr=next(x for x in selections if x['month']==month);assert sr['simple']==max(c['methods'][:3],key=lambda n:aps[n]);assert sr['ML']==max(c['methods'][3:],key=lambda n:aps[n])
        for label,name in [('selected_ML',sr['ML']),('selected_simple',sr['simple'])]:
            x=p.loc[p.month.eq(month)&p.method.eq(label)];y=p.loc[p.month.eq(month)&p.method.eq(name)];close(x.score,y.score);assert x.alarm.tolist()==y.alarm.tolist()
    for (method,month),g in p.groupby(['method','month']):
        ev=g.event.to_numpy(bool);alarm=g.alarm.to_numpy(bool);under=g.under_amount.to_numpy();row=mt.loc[mt.method.eq(method)&mt.period.eq(str(month))].iloc[0]
        vals={'events':int(ev.sum()),'warnings':int(alarm.sum()),'TP':int((ev&alarm).sum()),'FP':int((~ev&alarm).sum()),'FN':int((ev&~alarm).sum()),'top_budget_TP':int((ev&g.top_budget).sum()),'AP':ap(ev.tolist(),g.score.tolist())}
        for k,v in vals.items():close(row[k],v)
        computed[(method,month)]=vals
    pooled={}
    for method,g in p.groupby('method'):
        sub=[computed[(method,m)] for m in [4,5,6]];v={k:sum(r[k] for r in sub) for k in ['events','warnings','TP','FP','FN','top_budget_TP']};v['AP']=sum(r['AP']*r['events'] for r in sub)/v['events'];pooled[method]=v
        row=mt.loc[mt.method.eq(method)&mt.period.eq('pooled')].iloc[0]
        for k,val in v.items():close(row[k],val)
    a=pooled['selected_ML'];b=pooled['selected_simple'];flags={'AP_5percent':a['AP']>=1.05*b['AP'],'ranking_same_budget':a['top_budget_TP']>=b['top_budget_TP'],'online_TP_gain':a['TP']>=b['TP']+1,'no_more_warnings':a['warnings']<=b['warnings'],'warning_cap':a['warnings']/2184<=.1,
        'monthly_TP':all(computed[('selected_ML',m)]['TP']>=computed[('selected_simple',m)]['TP'] for m in [4,5,6]),'monthly_AP':all(computed[('selected_ML',m)]['AP']>=.95*computed[('selected_simple',m)]['AP'] for m in [4,5,6])}
    assert flags==run['decision']['gates'] and all(flags.values())==run['decision']['eligible']
    result={'status':'passed','run_sha256':sha(OUT/'run.json'),'diagnosis_sha256':sha(OUT/'diagnosis.json'),'verifier_sha256':sha(Path(__file__)),
        'exact_input_rows_checked':len(f),'neighbors_checked':checked,'point_forecasts_checked':len(risk),'warning_models_reloaded':models,'score_values_checked':prediction_checks,'independent_HGB_refits':refits,'evaluated_rows':len(p),'monthly_metric_rows':len(computed),'gate_decision':run['decision'],'later_evaluated':False}
    dump(result,OUT/'independent_verification.json');print(json.dumps(result,ensure_ascii=False),flush=True)
if __name__=='__main__':main()

코드 등록 완료: `Modeling/scripts/error_warning_verify.py`

### `Modeling/scripts/m01_prepare.py`

원고 연결: Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: M01: exact-time feature construction and audit, without model fitting.

Run from project root: regular Python 3.13 Modeling/scripts/m01_prepare.py
Use load_frame() and feature_columns() for subsequent development comparisons.

주요 함수: `sha()`, `profile()`, `read_contract()`, `feature_columns()`, `load_frame()`, `independent_check()`, `save()`, `main()`.

In [34]:
%%submission_source Modeling/scripts/m01_prepare.py
"""M01: exact-time feature construction and audit, without model fitting.

Run from project root: regular Python 3.13 Modeling/scripts/m01_prepare.py
Use load_frame() and feature_columns() for subsequent development comparisons.
"""
import csv
import hashlib
import importlib.metadata
import json
import platform
import sys
from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
CONTRACT = ROOT / "Modeling/config/m01_contract.json"
OUT = ROOT / "Modeling/tables/m01"
SLOTS = ["15분", "30분", "45분", "60분"]
WEATHER = {"temperature": "기온", "wind": "풍속", "humidity": "습도", "rain": "강수량"}


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def profile(values):
    # Stable across numpy byte order and independently reproducible with csv.
    return hashlib.sha256(",".join(format(float(v), ".17g") for v in values).encode()).hexdigest()


def read_contract():
    return json.loads(CONTRACT.read_text(encoding="utf-8"))


def feature_columns(group, contract=None):
    contract = contract or read_contract()
    columns = contract["calendar_features"] + contract["feature_groups"]["A"]
    if group in ("B", "C_B", "D"):
        columns += contract["feature_groups"]["B_add"]
    if group in ("C_A", "C_B", "D"):
        columns += contract["feature_groups"]["C_add"]
    if group == "D":
        columns += contract["feature_groups"]["D_add"]
    if group not in ("A", "B", "C_A", "C_B", "D"):
        raise ValueError(group)
    return columns


def load_frame(contract=None):
    c = contract or read_contract()
    source = ROOT / c["source"]
    assert sha(source) == c["source_sha256"], "Source changed: review contract before execution"
    raw = pd.read_csv(source, encoding="utf-8-sig")
    scoped = raw.loc[raw["날짜"].between(*c["period"])].copy()
    invalid = scoped.loc[~scoped["시간"].between(*c["valid_hour"])].copy()
    valid = scoped.loc[scoped["시간"].between(*c["valid_hour"])].copy()
    valid["timestamp"] = pd.to_datetime(valid["날짜"].astype(str), format="%Y%m%d") + pd.to_timedelta(valid["시간"], unit="h")
    valid = valid.sort_values("timestamp").reset_index(drop=True)
    assert not valid.timestamp.duplicated().any()
    assert valid.groupby("날짜")["시간"].apply(lambda s: set(s) == set(range(24))).all()
    assert valid[SLOTS + ["생산량"]].notna().all().all()
    f = pd.DataFrame({"timestamp": valid.timestamp})
    f["date"] = f.timestamp.dt.normalize()
    f["target_maximum"] = valid[SLOTS].max(axis=1)
    f["target_mean"] = valid[SLOTS].mean(axis=1)
    f["diag_target_production"] = valid["생산량"]
    f["diag_target_rounded_mean"] = valid["평균"]
    f["month"] = f.timestamp.dt.month
    f["weekend"] = (f.timestamp.dt.dayofweek >= 5).astype(int)
    f["hour_sin"] = np.sin(2 * np.pi * f.timestamp.dt.hour / 24)
    f["hour_cos"] = np.cos(2 * np.pi * f.timestamp.dt.hour / 24)
    for dow in range(7):
        f[f"dow_{dow}"] = (f.timestamp.dt.dayofweek == dow).astype(int)
    valid["mean"] = f.target_mean
    valid["maximum"] = f.target_maximum
    valid["last"] = valid[SLOTS[-1]]
    index = valid.set_index("timestamp")
    for lag in c["lags_hours"]:
        times = f.timestamp - pd.Timedelta(hours=lag)
        prior = index.reindex(pd.DatetimeIndex(times))
        f[f"lag{lag}_timestamp"] = times.where(times.isin(index.index))
        f[f"available_lag{lag}"] = times.isin(index.index)
        for metric in ("mean", "maximum", "last"):
            f[f"lag{lag}_{metric}"] = prior[metric].to_numpy()
        f[f"lag{lag}_production"] = prior["생산량"].to_numpy()
    f["lag1_production_zero"] = (f.lag1_production == 0).astype(float).where(f.available_lag1)
    for name, source_col in WEATHER.items():
        f[f"lag1_{name}"] = index.reindex(pd.DatetimeIndex(f.timestamp - pd.Timedelta(hours=1)))[source_col].to_numpy()
    for pool, lags in c["pools"].items():
        f[f"eligible_{pool}"] = f[[f"available_lag{lag}" for lag in lags]].all(axis=1)
    daily = {date: profile(cell[SLOTS].to_numpy().reshape(-1)) for date, cell in valid.groupby("날짜")}
    f["diag_target_profile"] = valid["날짜"].map(daily)
    previous_positive = f.lag1_production > 0
    current_positive = f.diag_target_production > 0
    f["diag_production_transition"] = np.select(
        [~f.available_lag1, ~previous_positive & current_positive,
         previous_positive & ~current_positive, previous_positive & current_positive],
        ["unavailable", "zero_to_positive", "positive_to_zero", "positive_to_positive"],
        default="zero_to_zero")
    return raw, invalid, valid, f


def independent_check(c, raw, invalid, valid, f):
    # Independent standard-library source parse; no reuse of pandas aggregates.
    source = ROOT / c["source"]
    observations, excluded = {}, []
    with source.open(encoding="utf-8-sig", newline="") as handle:
        rows = list(csv.DictReader(handle))
    for row in rows:
        date, hour = int(row["날짜"]), int(row["시간"])
        if not c["period"][0] <= date <= c["period"][1]:
            continue
        if not 0 <= hour <= 23:
            excluded.append((date, hour))
            continue
        ts = datetime.strptime(str(date), "%Y%m%d") + timedelta(hours=hour)
        assert ts not in observations
        slots = [float(row[s]) for s in SLOTS]
        assert int(row["평균"]) == int(sum(slots) / 4 + 0.5)
        observations[ts] = {"slots": slots, "production": float(row["생산량"]),
                            "weather": {name: float(row[col]) if row[col] else np.nan for name, col in WEATHER.items()}}
    checked_values = 0
    for _, row in f.iterrows():
        ts = row.timestamp.to_pydatetime()
        own = observations[ts]
        assert row.target_maximum == max(own["slots"])
        assert row.target_mean == sum(own["slots"]) / 4
        for lag in c["lags_hours"]:
            prior_ts = ts - timedelta(hours=lag)
            exists = prior_ts in observations
            assert bool(row[f"available_lag{lag}"]) == exists
            if exists:
                prior = observations[prior_ts]
                assert row[f"lag{lag}_timestamp"] == prior_ts < ts
                for metric, expected in (("mean", sum(prior["slots"]) / 4), ("maximum", max(prior["slots"])),
                                         ("last", prior["slots"][-1]), ("production", prior["production"])):
                    assert row[f"lag{lag}_{metric}"] == expected
                    checked_values += 1
                if lag == 1:
                    for name, expected in prior["weather"].items():
                        got = row[f"lag1_{name}"]
                        assert (pd.isna(got) and np.isnan(expected)) or got == expected
            else:
                assert pd.isna(row[f"lag{lag}_timestamp"])
                assert pd.isna(row[f"lag{lag}_mean"])
        for pool, lags in c["pools"].items():
            assert bool(row[f"eligible_{pool}"]) == all(ts - timedelta(hours=x) in observations for x in lags)
    independent_profiles = {}
    for day in sorted({t.date() for t in observations}):
        slots = [v for h in range(24) for v in observations[datetime.combine(day, datetime.min.time()) + timedelta(hours=h)]["slots"]]
        independent_profiles[day] = profile(slots)
    assert all(row.diag_target_profile == independent_profiles[row.timestamp.date()] for row in f.itertuples())
    for key, got in {"raw_rows": len(raw), "scope_rows": len(valid) + len(invalid), "invalid_rows": len(excluded),
                     "valid_rows": len(f), "valid_days": f.date.nunique(), "lag1_rows": int(f.available_lag1.sum()),
                     "lag1_and_lag2_rows": int(f.eligible_core.sum()), "unique_profiles": len(set(independent_profiles.values())),
                     "repeated_profile_days": sum(n for n in Counter(independent_profiles.values()).values() if n > 1)}.items():
        assert got == c["expected_scope"][key], (key, got)
    for group in ("A", "B", "C_A", "C_B", "D"):
        features = feature_columns(group, c)
        assert not set(features) & {col for col in f if col.startswith(("diag_", "target_", "eligible_", "available_"))}
        assert f.loc[f.eligible_common, feature_columns("C_B", c)].notna().all().all()
    return {"independent_source_rows": len(rows), "targets_checked": len(f), "lag_feature_values_checked": checked_values,
            "source_time_lookup_checked": True, "feature_allowlist_checked": True, "profile_values_checked": len(independent_profiles)}


def save(frame, name):
    frame.to_csv(OUT / f"{name}.csv", index=False, encoding="utf-8-sig")


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled(), "Use regular CPython 3.13"
    c = read_contract()
    OUT.mkdir(parents=True, exist_ok=True)
    runtime = {"executable": sys.executable, "version": sys.version, "gil_enabled": sys._is_gil_enabled(), "platform": platform.platform(), "packages": {}}
    for package in ("numpy", "pandas", "scipy", "scikit-learn"):
        runtime["packages"][package] = importlib.metadata.version(package)
    from sklearn.ensemble import HistGradientBoostingRegressor
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import StandardScaler
    runtime["required_model_imports"] = [Ridge.__name__, HistGradientBoostingRegressor.__name__, StandardScaler.__name__]
    raw, invalid, valid, f = load_frame(c)
    verification = independent_check(c, raw, invalid, valid, f)
    save(invalid[["날짜", "시간"]].assign(reason="invalid_hour_unidentified_actual_time"), "excluded_rows")
    save(f, "hourly_frame")
    coverage = []
    for label, subset in [("all", f)] + [(f"month_{m}", g) for m, g in f.groupby("month")]:
        for lag in c["lags_hours"]:
            coverage.append({"scope": label, "rule": f"lag{lag}", "total_hours": len(subset), "eligible_hours": int(subset[f"available_lag{lag}"].sum())})
        for pool in c["pools"]:
            coverage.append({"scope": label, "rule": pool, "total_hours": len(subset), "eligible_hours": int(subset[f"eligible_{pool}"].sum())})
    save(pd.DataFrame(coverage), "lag_coverage")
    unavailable = []
    earliest = f.timestamp.min()
    for lag in c["lags_hours"]:
        for row in f.loc[~f[f"available_lag{lag}"]].itertuples():
            requested = row.timestamp - pd.Timedelta(hours=lag)
            unavailable.append({"target_timestamp": row.timestamp, "lag_hours": lag, "requested_timestamp": requested,
                                "reason": "before_scope_start" if requested < earliest else "excluded_invalid_date"})
    save(pd.DataFrame(unavailable), "unavailable_lags")
    splits, repetition, daily_rows, memberships, fallback = [], [], [], [], []
    for split in c["splits"]:
        train_mask = f.timestamp <= pd.Timestamp(split["train_end"])
        eval_mask = f.timestamp.between(pd.Timestamp(split["eval_start"]), pd.Timestamp(split["eval_end"]))
        assert f.loc[train_mask].timestamp.max() < f.loc[eval_mask].timestamp.min()
        for pool in c["pools"]:
            train = f.loc[train_mask & f[f"eligible_{pool}"]]
            evaluate = f.loc[eval_mask & f[f"eligible_{pool}"]]
            row = {"split": split["name"], "role": split["role"], "pool": pool, "train_hours": len(train), "eval_hours": len(evaluate),
                   "train_dates": train.date.nunique(), "eval_dates": evaluate.date.nunique(),
                   "eval_available_target_hours": int(eval_mask.sum()), "eval_excluded_hours": int(eval_mask.sum()) - len(evaluate)}
            profiles = set(train.diag_target_profile)
            diag = evaluate[["timestamp", "date", "diag_target_profile", "target_maximum"]].copy()
            diag["train_profile_overlap"] = diag.diag_target_profile.isin(profiles)
            counts = diag[["date", "diag_target_profile"]].drop_duplicates().diag_target_profile.value_counts()
            diag["profile_weight"] = 1 / diag.diag_target_profile.map(counts)
            diag["daily_maximum_weight"] = 0.0
            full_days = set()
            for day, group in diag.groupby("date"):
                complete = len(group) == 24
                daily_rows.append({"split": split["name"], "pool": pool, "date": day.date(), "eligible_hours": len(group),
                                   "complete_day": complete, "observed_daily_maximum": float(f.loc[f.date == day, "target_maximum"].max())})
                if complete:
                    full_days.add(day)
                    is_max = group.target_maximum == group.target_maximum.max()
                    diag.loc[group.index[is_max], "daily_maximum_weight"] = 1 / int(is_max.sum())
            assert abs(diag.daily_maximum_weight.sum() - len(full_days)) < 1e-10
            row["complete_eval_days"] = len(full_days)
            row["incomplete_eval_days"] = int(f.loc[eval_mask].date.nunique()) - len(full_days)
            row["daily_maximum_eval_hours"] = int((diag.daily_maximum_weight > 0).sum())
            splits.append(row)
            repetition.append({"split": split["name"], "pool": pool, "eval_days": diag.date.nunique(), "eval_profiles": len(counts),
                               "overlap_days": diag.loc[diag.train_profile_overlap].date.nunique(), "overlap_hours": int(diag.train_profile_overlap.sum()),
                               "new_profile_hours": int((~diag.train_profile_overlap).sum())})
            diag["split"], diag["pool"] = split["name"], pool
            memberships.append(diag)
        ev = f.loc[eval_mask]
        for baseline, order in c["missing_policy"]["secondary_baseline_fallback"].items():
            selected = pd.Series("abstain", index=ev.index)
            for lag in order:
                selected.loc[(selected == "abstain") & ev[f"available_lag{lag}"]] = f"lag{lag}"
            for used, n in selected.value_counts().items():
                fallback.append({"split": split["name"], "baseline": baseline, "used": used, "hours": int(n)})
    save(pd.DataFrame(splits), "split_counts")
    save(pd.DataFrame(repetition), "profile_overlap")
    save(pd.DataFrame(daily_rows), "daily_coverage")
    save(pd.concat(memberships, ignore_index=True), "evaluation_diagnostics")
    save(pd.DataFrame(fallback), "baseline_fallback_coverage")
    weather_missing = [{"feature": col, "pool": pool, "missing_hours": int(f.loc[f[f"eligible_{pool}"], col].isna().sum())}
                       for pool in c["pools"] for col in c["feature_groups"]["D_add"]]
    save(pd.DataFrame(weather_missing), "weather_missing")
    features = [{"group": group, "feature": col, "availability": "target calendar known" if col in c["calendar_features"] else "observed lag only"}
                for group in ("A", "B", "C_A", "C_B", "D") for col in feature_columns(group, c)]
    save(pd.DataFrame(features), "feature_dictionary")
    assert sha(ROOT / c["source"]) == c["source_sha256"]
    verification.update({"status": "passed", "contract_sha256": sha(CONTRACT), "script_sha256": sha(Path(__file__)),
                         "source_sha256": c["source_sha256"], "model_fitting_performed": False,
                         "timestamp": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
                         "outputs_sha256": {p.name: sha(p) for p in sorted(OUT.glob("*.csv"))}})
    (OUT / "runtime.json").write_text(json.dumps(runtime, ensure_ascii=False, indent=2), encoding="utf-8")
    (OUT / "verification.json").write_text(json.dumps(verification, ensure_ascii=False, indent=2), encoding="utf-8")
    summary = {"valid_hours": len(f), "lag_counts": {str(lag): int(f[f"available_lag{lag}"].sum()) for lag in c["lags_hours"]},
               "core_hours": int(f.eligible_core.sum()), "common_hours": int(f.eligible_common.sum()),
               "split_counts": splits, "profile_overlap": repetition, "weather_missing": weather_missing,
               "model_fitting_performed": False}
    (OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m01_prepare.py`

### `Modeling/scripts/m01_verify.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Independent standard-library verification of M01 saved split artifacts.

주요 함수: `rows()`, `main()`.

In [35]:
%%submission_source Modeling/scripts/m01_verify.py
"""Independent standard-library verification of M01 saved split artifacts."""
import csv
import hashlib
import json
import math
from collections import Counter, defaultdict
from datetime import datetime, timedelta
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/m01"


def rows(name):
    with (OUT / f"{name}.csv").open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def main():
    c = json.loads((ROOT / "Modeling/config/m01_contract.json").read_text(encoding="utf-8"))
    source = ROOT / c["source"]
    assert hashlib.sha256(source.read_bytes()).hexdigest() == c["source_sha256"]
    source_days = defaultdict(dict)
    observed, bad = {}, []
    with source.open(encoding="utf-8-sig", newline="") as stream:
        for raw in csv.DictReader(stream):
            date, hour = int(raw["날짜"]), int(raw["시간"])
            if not c["period"][0] <= date <= c["period"][1]:
                continue
            if not 0 <= hour <= 23:
                bad.append((date, hour))
                continue
            ts = datetime.strptime(str(date), "%Y%m%d") + timedelta(hours=hour)
            slots = [float(raw[key]) for key in ("15분", "30분", "45분", "60분")]
            observed[ts] = slots
            source_days[ts.date()][hour] = slots
    profiles = {}
    for day, hours in source_days.items():
        assert set(hours) == set(range(24))
        profiles[day] = hashlib.sha256(",".join(format(value, ".17g") for hour in range(24) for value in hours[hour]).encode()).hexdigest()
    assert sorted((int(r["날짜"]), int(r["시간"])) for r in rows("excluded_rows")) == sorted(bad)
    hourly = {datetime.fromisoformat(r["timestamp"]): r for r in rows("hourly_frame")}
    assert set(hourly) == set(observed)
    for ts, r in hourly.items():
        assert float(r["target_maximum"]) == max(observed[ts])
        assert float(r["target_mean"]) == sum(observed[ts]) / 4
        for lag in c["lags_hours"]:
            assert (r[f"available_lag{lag}"] == "True") == (ts - timedelta(hours=lag) in observed)
    split_counts = {(r["split"], r["pool"]): r for r in rows("split_counts")}
    overlap = {(r["split"], r["pool"]): r for r in rows("profile_overlap")}
    diagnostics = defaultdict(list)
    for r in rows("evaluation_diagnostics"):
        diagnostics[r["split"], r["pool"]].append(r)
    checks = []
    for split in c["splits"]:
        end = datetime.fromisoformat(split["train_end"])
        start, stop = (datetime.fromisoformat(split[key]) for key in ("eval_start", "eval_end"))
        all_eval = {ts for ts in observed if start <= ts <= stop}
        for pool, lags in c["pools"].items():
            eligible = {ts for ts in observed if all(ts - timedelta(hours=lag) in observed for lag in lags)}
            train = {ts for ts in eligible if ts <= end}
            evaluation = {ts for ts in eligible if start <= ts <= stop}
            assert max(train) < min(evaluation)
            count = split_counts[split["name"], pool]
            assert int(count["train_hours"]) == len(train)
            assert int(count["eval_hours"]) == len(evaluation)
            assert int(count["eval_excluded_hours"]) == len(all_eval - evaluation)
            diag = diagnostics[split["name"], pool]
            assert {datetime.fromisoformat(r["timestamp"]) for r in diag} == evaluation
            assert len(diag) == len(evaluation)
            train_profiles = {profiles[ts.date()] for ts in train}
            eval_days = {ts.date() for ts in evaluation}
            overlaps = {day for day in eval_days if profiles[day] in train_profiles}
            record = overlap[split["name"], pool]
            assert int(record["overlap_days"]) == len(overlaps)
            assert int(record["overlap_hours"]) == sum(ts.date() in overlaps for ts in evaluation)
            complete_days = {day for day in eval_days if sum(ts.date() == day for ts in evaluation) == 24}
            assert int(count["complete_eval_days"]) == len(complete_days)
            assert int(count["incomplete_eval_days"]) == len({ts.date() for ts in all_eval} - complete_days)
            profile_counts = Counter(profiles[day] for day in eval_days)
            weight_by_day = defaultdict(float)
            for r in diag:
                ts = datetime.fromisoformat(r["timestamp"])
                assert r["diag_target_profile"] == profiles[ts.date()]
                assert (r["train_profile_overlap"] == "True") == (ts.date() in overlaps)
                assert math.isclose(float(r["profile_weight"]), 1 / profile_counts[profiles[ts.date()]], abs_tol=1e-14)
                daily_max = max(max(source_days[ts.date()][hour]) for hour in range(24))
                ties = sum(max(source_days[ts.date()][hour]) == daily_max for hour in range(24))
                expected = 1 / ties if ts.date() in complete_days and max(observed[ts]) == daily_max else 0
                assert math.isclose(float(r["daily_maximum_weight"]), expected, abs_tol=1e-14)
                weight_by_day[ts.date()] += float(r["daily_maximum_weight"])
            for day, weight in weight_by_day.items():
                assert math.isclose(weight, 1 if day in complete_days else 0, abs_tol=1e-12)
            checks.append({"split": split["name"], "pool": pool, "train_hours": len(train), "eval_hours": len(evaluation), "complete_days": len(complete_days)})
    original = json.loads((OUT / "verification.json").read_text(encoding="utf-8"))
    assert hashlib.sha256((ROOT / "Modeling/config/m01_contract.json").read_bytes()).hexdigest() == original["contract_sha256"]
    assert hashlib.sha256((ROOT / "Modeling/scripts/m01_prepare.py").read_bytes()).hexdigest() == original["script_sha256"]
    for name, digest in original["outputs_sha256"].items():
        assert hashlib.sha256((OUT / name).read_bytes()).hexdigest() == digest
    result = {"status": "passed", "independent_standard_library_verification": True, "hourly_rows": len(hourly),
              "split_and_daily_weight_checks": checks, "source_and_output_hashes_match": True, "model_fitting_performed": False,
              "verifier_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    (OUT / "independent_verification.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps({"status": "passed", "hourly_rows": len(hourly), "split_pools": len(checks), "model_fitting_performed": False}))


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m01_verify.py`

### `Modeling/scripts/m022_compare.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: M02-2: nested expanding-month Optuna and honest outer-month ensembles.

주요 함수: `save()`, `suggest()`, `make_model()`, `study_rows()`, `summarize()`, `add_errors()`, `main()`.

In [36]:
%%submission_source Modeling/scripts/m022_compare.py
"""M02-2: nested expanding-month Optuna and honest outer-month ensembles."""
import json
import os
import sys
import time
from pathlib import Path

for key in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ.setdefault(key, "4")
import joblib
import numpy as np
import pandas as pd
import optuna
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from m01_prepare import ROOT, feature_columns, read_contract, sha
from m02_compare import metric
from m03_ablation import now, subsets

OUT = ROOT / "Modeling/tables/m022"
MODELS = ROOT / "Modeling/models/m022"
CONTRACT = ROOT / "Modeling/config/m022_contract.json"


def save(df, name, directory=OUT):
    df.to_csv(directory / f"{name}.csv", index=False, encoding="utf-8-sig")


def suggest(trial, family, c):
    result = {}
    categorical = {"max_leaf_nodes", "num_leaves", "max_depth"} if family == "ExtraTrees" else {"max_leaf_nodes", "num_leaves"}
    integers = {"max_iter", "n_estimators", "iterations", "max_depth", "depth", "min_samples_leaf", "min_child_samples"}
    for name, spec in c["search_spaces"][family].items():
        if name in categorical:
            result[name] = trial.suggest_categorical(name, spec)
        elif name in integers:
            result[name] = trial.suggest_int(name, spec[0], spec[1], step=spec[2] if len(spec) == 3 else 1)
        else:
            result[name] = trial.suggest_float(name, spec[0], spec[1], log=len(spec) == 3 and spec[2] == "log")
    return result


def make_model(family, params):
    if family == "HGB":
        return HistGradientBoostingRegressor(**params, early_stopping=False, random_state=42)
    if family == "XGBoost":
        return XGBRegressor(**params, objective="reg:squarederror", tree_method="hist", random_state=42, n_jobs=4)
    if family == "LightGBM":
        return LGBMRegressor(**params, objective="regression", subsample_freq=1, random_state=42, n_jobs=4,
                             deterministic=True, force_col_wise=True, verbosity=-1)
    if family == "CatBoost":
        return CatBoostRegressor(**params, loss_function="RMSE", random_seed=42, thread_count=4,
                                 verbose=False, allow_writing_files=False, task_type="CPU")
    if family == "ExtraTrees":
        return ExtraTreesRegressor(**params, random_state=42, n_jobs=4, bootstrap=False)
    raise ValueError(family)


def study_rows(study, split, family):
    return [{"split": split, "family": family, "trial": t.number, "state": t.state.name,
             "value": t.value, "parameters": json.dumps(t.params), "seconds": t.duration.total_seconds() if t.duration else None,
             "best_so_far": min(x.value for x in study.trials[:t.number+1] if x.value is not None)} for t in study.trials]


def summarize(p, directory=OUT):
    comparison, conditions, daily = [], [], []
    for (target, model), g in p.groupby(["target", "model"]):
        comparison.append({"target": target, "model": model, "split": "pooled_development", **metric(g)})
        comparison.extend({"target": target, "model": model, "split": s, **metric(part)} for s, part in g.groupby("split"))
        for kind, value, part, w in subsets(g):
            if len(part):
                conditions.append({"target": target, "model": model, "condition": kind, "value": value, **metric(part, w),
                                   "share_total_absolute_error_unweighted": float(part.absolute_error.sum() / g.absolute_error.sum())})
        daily.extend({"target": target, "model": model, "date": date, **metric(part)} for date, part in g.groupby("date"))
    comparison = pd.DataFrame(comparison)
    save(comparison, "comparison", directory)
    save(pd.DataFrame(conditions), "condition_errors", directory)
    save(pd.DataFrame(daily), "daily_errors", directory)
    return comparison


def add_errors(p):
    p["signed_error"] = p.prediction - p.actual
    p["absolute_error"] = p.signed_error.abs()
    p["under_amount"] = (-p.signed_error).clip(lower=0)
    p["over_amount"] = p.signed_error.clip(lower=0)
    return p


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    c = json.loads(CONTRACT.read_text(encoding="utf-8"))
    parent = read_contract()
    OUT.mkdir(parents=True, exist_ok=True)
    MODELS.mkdir(parents=True, exist_ok=True)
    v1 = json.loads((ROOT / "Modeling/tables/m01/verification.json").read_text(encoding="utf-8"))
    olddir = ROOT / "Modeling/tables/m03/c"
    oldrun = json.loads((olddir / "run.json").read_text(encoding="utf-8"))
    oldverify = json.loads((olddir / "independent_verification.json").read_text(encoding="utf-8"))
    assert oldverify["status"] == "passed" and oldverify["run_sha256"] == sha(olddir / "run.json")
    assert sha(ROOT / parent["source"]) == parent["source_sha256"]
    inputs = {"Modeling/config/m022_contract.json": sha(CONTRACT), parent["source"]: sha(ROOT / parent["source"])}
    for rel, expected in [("Modeling/tables/m01/hourly_frame.csv", v1["outputs_sha256"]["hourly_frame.csv"]),
                          ("Modeling/tables/m03/c/predictions.csv", oldrun["outputs_sha256"]["predictions.csv"])]:
        assert sha(ROOT / rel) == expected
        inputs[rel] = expected
    f = pd.read_csv(ROOT / "Modeling/tables/m01/hourly_frame.csv", encoding="utf-8-sig", float_precision="round_trip", parse_dates=["timestamp"])
    f = f.loc[f.eligible_common & f.timestamp.lt("2021-07-01")].copy()
    old = pd.read_csv(olddir / "predictions.csv", encoding="utf-8-sig", float_precision="round_trip", parse_dates=["timestamp"])
    old = old.loc[old.group.eq("B") & old.target.eq(c["target"])].copy()
    features = feature_columns("B", parent)
    assert len(features) == 18 and not any(x.startswith(("target_", "diag_")) for x in features)
    run = {"status": "running", "started": now(), "input_hashes": inputs, "features": features,
           "environment": json.loads((OUT / "environment.json").read_text(encoding="utf-8")),
           "july_august_evaluated": False, "selection_scope": "outer development family choice; not final test"}
    (OUT / "run.json").write_text(json.dumps(run, indent=2), encoding="utf-8")
    all_predictions, trials, fold_scores, audit, configs, manifest, inner_predictions, correlations = [], [], [], [], [], [], [], []
    started = time.perf_counter()
    for outer in parent["splits"]:
        split = outer["name"]
        if split not in c["outer_splits"]:
            continue
        tr = f.loc[f.timestamp.le(outer["train_end"])]
        ev = f.loc[f.timestamp.between(outer["eval_start"], outer["eval_end"])]
        base = old.loc[old.split.eq(split)].copy().drop(columns=["group"])
        assert len(ev) == len(base) and base.timestamp.tolist() == ev.timestamp.tolist()
        assert np.array_equal(base.actual, ev[c["target"]]) and tr.timestamp.max() < ev.timestamp.min()
        for name, values in [("HGB_B_m03", base.prediction.to_numpy()), ("previous_hour", ev.lag1_maximum.to_numpy()),
                             ("previous_day", ev.lag24_maximum.to_numpy()), ("previous_week", ev.lag168_maximum.to_numpy())]:
            record = base.copy()
            record["model"], record["prediction"] = name, values
            all_predictions.append(record)
        folds = []
        for month in c["inner_validation_months"][split]:
            start = pd.Timestamp(2021, month, 1)
            end = start + pd.offsets.MonthBegin(1)
            train = tr.loc[tr.timestamp.lt(start)]
            val = tr.loc[tr.timestamp.ge(start) & tr.timestamp.lt(end)]
            assert len(train) > 300 and len(val) > 300
            assert train.timestamp.max() < val.timestamp.min() and val.timestamp.max() < ev.timestamp.min()
            folds.append((month, train, val))
            audit.append({"outer_split": split, "kind": "inner", "month": month, "train_hours": len(train), "validation_hours": len(val),
                          "train_last": str(train.timestamp.max()), "validation_first": str(val.timestamp.min()), "validation_last": str(val.timestamp.max()),
                          "outer_evaluation_first": str(ev.timestamp.min())})
        chosen_inner, chosen_outer, inner_maes = {}, {}, {}
        inner_actual = np.concatenate([val[c["target"]].to_numpy() for _, _, val in folds])
        inner_times = np.concatenate([val.timestamp.to_numpy() for _, _, val in folds])
        for family in c["families"]:
            cache = {}
            def objective(trial):
                params = suggest(trial, family, c)
                predictions = []
                for month, train, val in folds:
                    model = make_model(family, params)
                    model.fit(train[features], train[c["target"]])
                    predicted = model.predict(val[features])
                    assert np.isfinite(predicted).all()
                    error = np.abs(predicted - val[c["target"]].to_numpy())
                    fold_scores.append({"split": split, "family": family, "trial": trial.number, "month": month,
                                        "train_hours": len(train), "validation_hours": len(val), "MAE": float(error.mean()), "absolute_error_sum": float(error.sum())})
                    predictions.append(predicted)
                pred = np.concatenate(predictions)
                value = float(np.mean(np.abs(pred - inner_actual)))
                cache[trial.number] = pred
                return value
            study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=10), pruner=optuna.pruners.NopPruner())
            if family == "HGB":
                study.enqueue_trial({"max_leaf_nodes": 15, "max_iter": 150, "learning_rate": 0.05, "min_samples_leaf": 20, "l2_regularization": 1.0})
            study.optimize(objective, n_trials=c["optuna"]["base_trials_per_study"], n_jobs=1)
            first20 = min(t.value for t in study.trials[:20])
            extended = study.best_value <= first20 * .99
            if extended:
                study.optimize(objective, n_trials=c["optuna"]["max_trials_per_study"] - len(study.trials), n_jobs=1)
            params = study.best_params
            model = make_model(family, params)
            model.fit(tr[features], tr[c["target"]])
            pred = model.predict(ev[features])
            model_path = MODELS / f"{split}_{family}.joblib"
            joblib.dump(model, model_path, compress=3)
            assert np.allclose(joblib.load(model_path).predict(ev[features]), pred, atol=1e-9, rtol=0)
            manifest.append({"split": split, "model": family, "file": str(model_path.relative_to(ROOT)), "sha256": sha(model_path), "reload_predictions_checked": True})
            chosen_inner[family], chosen_outer[family] = cache[study.best_trial.number], pred
            inner_maes[family] = study.best_value
            inner_predictions.append(pd.DataFrame({"outer_split": split, "model": family, "timestamp": inner_times, "actual": inner_actual, "prediction": chosen_inner[family]}))
            configs.append({"split": split, "model": family, "best_trial": study.best_trial.number, "trials": len(study.trials), "extended": extended,
                            "inner_MAE": study.best_value, "parameters": params, "train_hours": len(tr), "eval_hours": len(ev)})
            trials.extend(study_rows(study, split, family))
            record = base.copy()
            record["model"], record["prediction"] = family, pred
            all_predictions.append(record)
            save(pd.DataFrame(trials), "trials")
            save(pd.DataFrame(fold_scores), "trial_fold_scores")
            (OUT / "selected_configurations.json").write_text(json.dumps(configs, indent=2), encoding="utf-8")
            print(f"{split} {family}: trials={len(study.trials)} inner_MAE={study.best_value:.4f} outer_MAE={np.mean(np.abs(pred-base.actual)):.4f}", flush=True)
        mean_pred = np.mean(list(chosen_outer.values()), axis=0)
        record = base.copy()
        record["model"], record["prediction"] = "mean5", mean_pred
        all_predictions.append(record)
        configs.append({"split": split, "model": "mean5", "members": c["families"], "weights": [0.2] * 5})
        members = sorted(inner_maes, key=inner_maes.get)[:3]
        matrix = np.column_stack([chosen_inner[x] for x in members])
        def weight_objective(trial):
            w = np.array([trial.suggest_float(f"weight_{i}", 0, 1) for i in range(3)])
            w = w / w.sum() if w.sum() else np.ones(3) / 3
            return float(np.mean(np.abs(matrix @ w - inner_actual)))
        ws = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.NopPruner())
        ws.enqueue_trial({f"weight_{i}": 1/3 for i in range(3)})
        for member in range(3):
            ws.enqueue_trial({f"weight_{i}": float(i == member) for i in range(3)})
        ws.optimize(weight_objective, n_trials=50, n_jobs=1)
        w = np.array([ws.best_params[f"weight_{i}"] for i in range(3)])
        w = w / w.sum() if w.sum() else np.ones(3) / 3
        pred = np.column_stack([chosen_outer[x] for x in members]) @ w
        record = base.copy()
        record["model"], record["prediction"] = "weighted_top3", pred
        all_predictions.append(record)
        configs.append({"split": split, "model": "weighted_top3", "members": members, "weights": w.tolist(), "inner_MAE": ws.best_value})
        trials.extend(study_rows(ws, split, "weighted_top3"))
        corr = pd.DataFrame({name: values - inner_actual for name, values in chosen_inner.items()}).corr()
        for i, a in enumerate(c["families"]):
            for b in c["families"][i+1:]:
                correlations.append({"split": split, "model_a": a, "model_b": b, "inner_residual_correlation": corr.loc[a, b]})
        save(add_errors(pd.concat(all_predictions, ignore_index=True)), "predictions")
        save(pd.DataFrame(trials), "trials")
        save(pd.DataFrame(audit), "fold_audit")
        save(pd.DataFrame(manifest), "model_manifest")
        save(pd.concat(inner_predictions, ignore_index=True), "inner_selected_predictions")
        save(pd.DataFrame(correlations), "inner_residual_correlations")
        (OUT / "selected_configurations.json").write_text(json.dumps(configs, indent=2), encoding="utf-8")
    p = add_errors(pd.concat(all_predictions, ignore_index=True))
    assert p.timestamp.max() < pd.Timestamp("2021-07-01")
    comparison = summarize(p)
    pooled = comparison.loc[comparison.split.eq("pooled_development")].copy()
    near = pooled.loc[pooled.MAE.le(pooled.MAE.min() * 1.01)].copy()
    preference = {"HGB_B_m03": 0, "HGB": 1, "XGBoost": 2, "LightGBM": 3, "CatBoost": 4, "ExtraTrees": 5, "mean5": 6, "weighted_top3": 7,
                  "previous_hour": 0, "previous_day": 0, "previous_week": 0}
    near["preference"] = near.model.map(preference)
    selected = near.sort_values(["preference", "MAE"]).iloc[0].to_dict()
    oldmae = float(pooled.loc[pooled.model.eq("HGB_B_m03"), "MAE"].iloc[0])
    selection = {"selected": selected, "point_best": pooled.sort_values("MAE").iloc[0].to_dict(),
                 "old_HGB_B_MAE": oldmae, "relative_MAE_improvement": 1 - selected["MAE"] / oldmae,
                 "M03_repeat_required": selected["model"] != "HGB_B_m03", "development_only": True}
    (OUT / "selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
    run.update({"status": "completed", "finished": now(), "seconds": time.perf_counter()-started,
                "tuning_fits": len(fold_scores), "outer_model_fits": len(manifest), "trial_count_including_ensemble": len(trials),
                "prediction_rows": len(p), "outer_hours_per_method": 2184,
                "outputs_sha256": {x.name: sha(x) for x in OUT.glob("*.csv")},
                "configuration_sha256": sha(OUT / "selected_configurations.json"), "selection_sha256": sha(OUT / "selection.json"),
                "script_sha256": sha(Path(__file__))})
    assert inputs["Modeling/config/m022_contract.json"] == sha(CONTRACT)
    (OUT / "run.json").write_text(json.dumps(run, indent=2), encoding="utf-8")
    print(pooled[["model", "MAE", "RMSE", "mean_under", "mean_over"]].sort_values("MAE").to_string(index=False), flush=True)
    print(json.dumps(selection), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m022_compare.py`

### `Modeling/scripts/m022_environment.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Read-only regular CPython package startup check for authorized M02-2.

주요 함수: .

In [37]:
%%submission_source Modeling/scripts/m022_environment.py
"""Read-only regular CPython package startup check for authorized M02-2."""
import importlib
import importlib.metadata
import json
import sys
from pathlib import Path

assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
result = {"executable": sys.executable, "version": sys.version, "packages": {}}
for name in ["numpy", "pandas", "sklearn", "joblib", "optuna", "xgboost", "lightgbm", "catboost"]:
    try:
        module = importlib.import_module(name)
        result["packages"][name] = {"status": "ok", "version": getattr(module, "__version__", "unknown")}
    except Exception as exc:
        result["packages"][name] = {"status": "error", "type": type(exc).__name__, "message": str(exc)}
out = Path(__file__).resolve().parents[2] / "Modeling/tables/m022"
out.mkdir(parents=True, exist_ok=True)
(out / "environment.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
print(json.dumps(result), flush=True)


{"executable": "C:\\Program Files\\Python313\\python.exe", "version": "3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]", "packages": {"numpy": {"status": "ok", "version": "2.2.6"}, "pandas": {"status": "ok", "version": "3.0.1"}, "sklearn": {"status": "ok", "version": "1.7.1"}, "joblib": {"status": "ok", "version": "1.5.1"}, "optuna": {"status": "ok", "version": "4.7.0"}, "xgboost": {"status": "ok", "version": "3.1.3"}, "lightgbm": {"status": "ok", "version": "4.6.0"}, "catboost": {"status": "ok", "version": "1.2.10"}}}


코드 등록 완료: `Modeling/scripts/m022_environment.py`

### `Modeling/scripts/m02_compare.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: M02: versioned development comparison and first M04 error diagnostics.

Regular CPython 3.13; run from project root. No July/August evaluation.
Only A features. Standardization is fit on each training split.

주요 함수: `candidates()`, `state()`, `metric()`, `save()`, `main()`.

In [38]:
%%submission_source Modeling/scripts/m02_compare.py
"""M02: versioned development comparison and first M04 error diagnostics.

Regular CPython 3.13; run from project root. No July/August evaluation.
Only A features. Standardization is fit on each training split.
"""
import hashlib
import importlib.metadata
import itertools
import json
import math
import os
import platform
import sys
import time
import warnings
from datetime import datetime, timedelta, timezone
from pathlib import Path

# Bound numerical worker counts before importing numerical libraries.
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

from m01_prepare import ROOT, feature_columns, read_contract, sha

CONTRACT = ROOT / "Modeling/config/m02_contract.json"
OUT = ROOT / "Modeling/tables/m02"
MODELS = ROOT / "Modeling/models/m02"


def candidates(c):
    classes = {"Ridge": Ridge, "ElasticNet": ElasticNet, "SVR": SVR, "HGB": HistGradientBoostingRegressor}
    result = []
    for family, spec in c["grids"].items():
        varying = {k: v for k, v in spec.items() if isinstance(v, list)}
        fixed = {k: v for k, v in spec.items() if not isinstance(v, list)}
        for order, values in enumerate(itertools.product(*varying.values())):
            params = {**fixed, **dict(zip(varying, values))}
            estimator = classes[family](**params)
            if family != "HGB":
                estimator = make_pipeline(StandardScaler(), estimator)
            if family in ("ElasticNet", "SVR"):
                estimator = TransformedTargetRegressor(regressor=estimator, transformer=StandardScaler())
            result.append((f"{family.lower()}_{order:02d}", family, order, params, estimator))
    return result


def state(maximum, rounded):
    return np.select([maximum.eq(0), rounded.lt(20), rounded.between(20, 26)],
                     ["zero", "below20", "low"], default="above26")


def metric(g, weights=None):
    w = np.ones(len(g)) if weights is None else np.asarray(weights, dtype=float)
    assert len(g) and w.sum() > 0 and np.isfinite(w).all()
    error = g.prediction.to_numpy() - g.actual.to_numpy()
    absolute = np.abs(error)
    under, over = np.maximum(-error, 0), np.maximum(error, 0)
    return {"hours": len(g), "weight_sum": float(w.sum()),
            "MAE": float(np.average(absolute, weights=w)),
            "RMSE": float(np.sqrt(np.average(error ** 2, weights=w))),
            "bias": float(np.average(error, weights=w)),
            "mean_under": float(np.average(under, weights=w)),
            "mean_over": float(np.average(over, weights=w)),
            "under_fraction": float(np.average(error < 0, weights=w)),
            "over_fraction": float(np.average(error > 0, weights=w)),
            "under_when_present": float(np.average(under[under > 0], weights=w[under > 0])) if (under > 0).any() else 0.0,
            "over_when_present": float(np.average(over[over > 0], weights=w[over > 0])) if (over > 0).any() else 0.0,
            "absolute_error_sum": float(absolute.sum()),
            "mean_actual": float(np.average(g.actual, weights=w)),
            "negative_prediction_hours": int((g.prediction < 0).sum())}


def save(df, name):
    df.to_csv(OUT / f"{name}.csv", index=False, encoding="utf-8-sig")


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    c = json.loads(CONTRACT.read_text(encoding="utf-8"))
    parent = read_contract()
    OUT.mkdir(parents=True, exist_ok=True)
    MODELS.mkdir(parents=True, exist_ok=True)
    verification = json.loads((ROOT / "Modeling/tables/m01/verification.json").read_text(encoding="utf-8"))
    assert sha(ROOT / parent["source"]) == parent["source_sha256"]
    files = ["hourly_frame.csv", "evaluation_diagnostics.csv", "split_counts.csv"]
    for name in files:
        assert sha(ROOT / "Modeling/tables/m01" / name) == verification["outputs_sha256"][name]
    f = pd.read_csv(ROOT / "Modeling/tables/m01/hourly_frame.csv", encoding="utf-8-sig", float_precision="round_trip", parse_dates=["timestamp", "date"])
    diagnostics = pd.read_csv(ROOT / "Modeling/tables/m01/evaluation_diagnostics.csv", encoding="utf-8-sig", parse_dates=["timestamp", "date"])
    expected = pd.read_csv(ROOT / "Modeling/tables/m01/split_counts.csv", encoding="utf-8-sig")
    features = feature_columns("A", parent)
    assert "lag1_last" not in features and not any(x.startswith(("target_", "diag_")) for x in features)
    f["power_state"] = state(f.target_maximum, f.diag_target_rounded_mean)
    # M01 verified integer half-up averages, so rounding the lag arithmetic mean
    # reproduces the prior CSV 'average' used by A06.
    f["prior_power_state"] = state(f.lag1_maximum, np.floor(f.lag1_mean + 0.5))
    f["power_transition"] = f.prior_power_state + "->" + f.power_state
    specs = candidates(c)
    descriptions = [{"model": name, "family": family, "order": order, "parameters": json.dumps(params, ensure_ascii=False)} for name, family, order, params, _ in specs]
    descriptions += [{"model": name, "family": "Baseline", "order": i, "parameters": json.dumps({"lag": lag})} for i, (name, lag) in enumerate(c["baselines"].items())]
    save(pd.DataFrame(descriptions), "candidates")
    inputs = {"m01_contract": sha(ROOT / c["parent"]), "m02_contract": sha(CONTRACT), "source": sha(ROOT / parent["source"]),
              **{name: sha(ROOT / "Modeling/tables/m01" / name) for name in files}}
    run = {"started": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"), "status": "running", "input_hashes": inputs,
           "features": features, "candidate_settings": len(specs), "planned_fits": len(specs) * len(c["targets"]) * len(c["splits"]),
           "runtime": {"executable": sys.executable, "version": sys.version, "platform": platform.platform(),
                       "packages": {p: importlib.metadata.version(p) for p in ["numpy", "pandas", "scikit-learn", "joblib"]}}}
    (OUT / "run.json").write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding="utf-8")
    predictions, audits, fitted = [], [], {}
    total_started = time.perf_counter()
    for split in parent["splits"]:
        if split["name"] not in c["splits"]:
            continue
        train = f.loc[f.eligible_common & (f.timestamp <= pd.Timestamp(split["train_end"]))]
        ev = f.loc[f.eligible_common & f.timestamp.between(pd.Timestamp(split["eval_start"]), pd.Timestamp(split["eval_end"]))]
        expect = expected.loc[expected.split.eq(split["name"]) & expected.pool.eq("common")].iloc[0]
        assert len(train) == expect.train_hours and len(ev) == expect.eval_hours
        assert train.timestamp.max() < ev.timestamp.min()
        assert train[features].notna().all().all() and ev[features].notna().all().all()
        diag = diagnostics.loc[diagnostics.split.eq(split["name"]) & diagnostics.pool.eq("common")]
        cols = ["timestamp", "date", "month", "weekend", "power_transition", "diag_production_transition", "diag_target_profile"]
        base = ev[cols].merge(diag[["timestamp", "train_profile_overlap", "profile_weight", "daily_maximum_weight"]], on="timestamp", validate="one_to_one")
        base["hour"] = base.timestamp.dt.hour
        assert base.timestamp.tolist() == ev.timestamp.tolist()
        for target in c["targets"]:
            for name, lag in c["baselines"].items():
                record = base.copy()
                record["split"], record["target"], record["model"], record["family"] = split["name"], target, name, "Baseline"
                record["actual"] = ev[target].to_numpy()
                metric_name = "maximum" if target == "target_maximum" else "mean"
                record["prediction"] = ev[f"lag{lag}_{metric_name}"].to_numpy()
                predictions.append(record)
            for name, family, order, params, estimator in candidates(c):
                start = time.perf_counter()
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always", ConvergenceWarning)
                    estimator.fit(train[features], train[target])
                if any(issubclass(w.category, ConvergenceWarning) for w in caught):
                    raise RuntimeError(f"Convergence failure: {name}, {split['name']}: {[str(w.message) for w in caught]}")
                predicted = estimator.predict(ev[features])
                assert np.isfinite(predicted).all()
                # Directly verify fitted scaler statistics against training rows.
                reg = estimator.regressor_ if isinstance(estimator, TransformedTargetRegressor) else estimator
                if family != "HGB":
                    scaler = reg.steps[0][1]
                    assert scaler.n_samples_seen_ == len(train)
                    assert np.allclose(scaler.mean_, train[features].mean().to_numpy())
                if isinstance(estimator, TransformedTargetRegressor):
                    assert np.allclose(estimator.transformer_.mean_, [train[target].mean()])
                    assert estimator.transformer_.n_samples_seen_ == len(train)
                elapsed = time.perf_counter() - start
                record = base.copy()
                record["split"], record["target"], record["model"], record["family"] = split["name"], target, name, family
                record["actual"], record["prediction"] = ev[target].to_numpy(), predicted
                predictions.append(record)
                audits.append({"split": split["name"], "target": target, "model": name, "family": family, "train_hours": len(train), "eval_hours": len(ev),
                               "train_last_timestamp": str(train.timestamp.max()), "eval_first_timestamp": str(ev.timestamp.min()),
                               "seconds": elapsed, "warnings": json.dumps([str(w.message) for w in caught]),
                               "training_only_scalers_verified": True, "minimum_prediction": float(predicted.min())})
                fitted[(split["name"], target, name)] = estimator
            print(f"completed {split['name']} {target}: {len(specs)} fits; {len(ev)} evaluation hours", flush=True)
    p = pd.concat(predictions, ignore_index=True)
    p["signed_error"] = p.prediction - p.actual
    p["absolute_error"] = p.signed_error.abs()
    p["under_amount"] = (-p.signed_error).clip(lower=0)
    p["over_amount"] = p.signed_error.clip(lower=0)
    save(p, "predictions")
    save(pd.DataFrame(audits), "training_audit")
    scores = []
    for (target, model, family), g in p.groupby(["target", "model", "family"]):
        scores.append({"target": target, "model": model, "family": family, "split": "pooled_development", **metric(g)})
        for split, part in g.groupby("split"):
            scores.append({"target": target, "model": model, "family": family, "split": split, **metric(part)})
    scores = pd.DataFrame(scores)
    save(scores, "all_candidate_metrics")
    selected = []
    order_map = {name: order for name, _, order, _, _ in specs}
    for (target, family), group in scores.loc[scores.split.eq("pooled_development")].groupby(["target", "family"]):
        if family == "Baseline":
            chosen = group.sort_values(["MAE", "model"]).iloc[0]
        else:
            near = group.loc[group.MAE <= group.MAE.min() * 1.01].copy()
            near["preference"] = near.model.map(order_map)
            chosen = near.sort_values("preference").iloc[0]
        selected.append({**chosen.to_dict(), "family_best_MAE": float(group.MAE.min())})
    selected = pd.DataFrame(selected)
    save(selected, "family_selection")
    # Retain all three baselines in readable comparison and condition tables.
    selected_pairs = set(zip(selected.target, selected.model))
    q = p.loc[[family == "Baseline" or (target, model) in selected_pairs for target, model, family in zip(p.target, p.model, p.family)]].copy()
    save(q, "selected_predictions")
    comparison = scores.loc[[family == "Baseline" or (target, model) in selected_pairs for target, model, family in zip(scores.target, scores.model, scores.family)]]
    save(comparison, "comparison")
    condition_rows = []
    for (target, model, family), g in q.groupby(["target", "model", "family"]):
        total_error = g.absolute_error.sum()
        subsets = [("all", "all", g, None), ("profile_reweighted", "all", g, g.profile_weight)]
        peak = g.loc[g.daily_maximum_weight > 0]
        subsets.append(("daily_maximum", "all", peak, peak.daily_maximum_weight))
        for column in ["split", "month", "weekend", "hour", "power_transition", "diag_production_transition", "train_profile_overlap"]:
            for value, part in g.groupby(column):
                subsets.append((column, str(value), part, None))
        for kind, label, subset, weights in subsets:
            if subset.empty:
                continue
            condition_rows.append({"target": target, "model": model, "family": family, "condition": kind, "value": label,
                                   **metric(subset, weights), "share_total_absolute_error_unweighted": float(subset.absolute_error.sum() / total_error)})
    condition = pd.DataFrame(condition_rows)
    save(condition, "condition_errors")
    daily = []
    for (target, model, date), g in q.groupby(["target", "model", "date"]):
        daily.append({"target": target, "model": model, "date": date, **metric(g)})
    save(pd.DataFrame(daily), "daily_errors")
    save(q.loc[q.target.eq("target_maximum")].sort_values("absolute_error", ascending=False).groupby("model").head(20), "largest_errors")
    model_manifest = []
    for row in selected.loc[selected.family.ne("Baseline")].itertuples():
        for split in c["splits"]:
            estimator = fitted[(split, row.target, row.model)]
            path = MODELS / f"{split}_{row.target}_{row.model}.joblib"
            joblib.dump(estimator, path, compress=3)
            reloaded = joblib.load(path)
            ev = f.loc[f.eligible_common & f.timestamp.isin(q.loc[q.split.eq(split)].timestamp)]
            expected_pred = q.loc[q.target.eq(row.target) & q.model.eq(row.model) & q.split.eq(split)].prediction.to_numpy()
            assert np.allclose(reloaded.predict(ev[features]), expected_pred, rtol=0, atol=1e-10)
            model_manifest.append({"split": split, "target": row.target, "model": row.model, "file": str(path.relative_to(ROOT)), "sha256": sha(path), "reload_predictions_checked": True})
    save(pd.DataFrame(model_manifest), "model_manifest")
    best = {}
    priority = {"Baseline": 0, "Ridge": 1, "ElasticNet": 2, "HGB": 3, "SVR": 4}
    for target, group in selected.groupby("target"):
        near = group.loc[group.MAE <= group.MAE.min() * 1.01].copy()
        near["priority"] = near.family.map(priority)
        best[target] = near.sort_values("priority").iloc[0][["model", "family", "MAE", "RMSE"]].to_dict()
    assert p.timestamp.max() < pd.Timestamp("2021-07-01")
    assert sha(CONTRACT) == inputs["m02_contract"]
    run.update({"status": "completed", "finished": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
                "fits_completed": len(audits), "prediction_rows": len(p), "seconds": time.perf_counter() - total_started,
                "development_hours_per_method_target": 2184, "provisional_candidates": best,
                "july_august_evaluated": False, "last_slot_used": False,
                "outputs_sha256": {x.name: sha(x) for x in OUT.glob("*.csv")}, "script_sha256": sha(Path(__file__))})
    (OUT / "run.json").write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding="utf-8")
    print(selected[["target", "family", "model", "MAE", "RMSE", "mean_under", "mean_over"]].to_string(index=False), flush=True)
    print(json.dumps({"status": "completed", "fits": len(audits), "best": best, "seconds": run["seconds"]}, ensure_ascii=False), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m02_compare.py`

### `Modeling/scripts/m02_rerun.py`

원고 연결: Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Expanded M02: A input, nested chronological two-objective model selection.

주요 함수: `save()`, `peak_weights()`, `objectives()`, `balance()`, `choose_trial()`, `selection_table()`, `select_method()`, `main()`.

In [39]:
%%submission_source Modeling/scripts/m02_rerun.py
"""Expanded M02: A input, nested chronological two-objective model selection."""
import json
import sys
import time
import importlib.metadata
from pathlib import Path
from m022_compare import (make_model, suggest, summarize, add_errors, optuna,
                         np, pd, joblib, ROOT, feature_columns, read_contract, sha, now)

OUT = ROOT / 'Modeling/tables/m02_rerun'
MODELS = ROOT / 'Modeling/models/m02_rerun'
CONTRACT = ROOT / 'Modeling/config/m02_rerun_contract.json'


def save(df, name):
    df.to_csv(OUT / f'{name}.csv', index=False, encoding='utf-8-sig')


def peak_weights(frame):
    """One total weight per complete date, split evenly across tied maxima."""
    date = frame.timestamp.dt.strftime('%Y-%m-%d')
    counts = frame.groupby(date).timestamp.transform('count')
    highest = frame.groupby(date).target_maximum.transform('max')
    indicator = frame.target_maximum.eq(highest) & counts.eq(24)
    ties = indicator.groupby(date).transform('sum')
    return np.where(indicator, 1 / ties.clip(lower=1), 0).astype(float)


def objectives(actual, predicted, weights):
    error = np.abs(np.asarray(actual) - np.asarray(predicted))
    assert np.isfinite(error).all() and np.sum(weights) > 0
    return float(error.mean()), float(np.average(error, weights=weights))


def balance(values, denominator):
    return max(a / b for a, b in zip(values, denominator))


def choose_trial(study, denominator):
    # Keep all Pareto outcomes; the compromise is specified before the run.
    return min(study.best_trials, key=lambda t: (balance(t.values, denominator), t.number))


def selection_table(predictions):
    rows = []
    for model, g in predictions.groupby('model'):
        overall, peak = objectives(g.actual, g.prediction, g.daily_maximum_weight)
        rows.append({'model': model, 'overall_MAE': overall, 'daily_maximum_MAE': peak})
    result = pd.DataFrame(rows)
    baseline = result.loc[result.model.eq('previous_hour')].iloc[0]
    result['relative_overall_MAE'] = result.overall_MAE / baseline.overall_MAE
    result['relative_daily_maximum_MAE'] = result.daily_maximum_MAE / baseline.daily_maximum_MAE
    result['balanced_score'] = result[['relative_overall_MAE', 'relative_daily_maximum_MAE']].max(axis=1)
    result['nondominated'] = [not (((result.overall_MAE <= r.overall_MAE) &
                                  (result.daily_maximum_MAE <= r.daily_maximum_MAE)) &
                                 ((result.overall_MAE < r.overall_MAE) |
                                  (result.daily_maximum_MAE < r.daily_maximum_MAE))).any()
                               for r in result.itertuples()]
    return result


def select_method(table):
    near = table.loc[table.nondominated & table.balanced_score.le(table.balanced_score.min() * 1.01)].copy()
    order = ['previous_hour', 'previous_day', 'previous_week', 'HGB_initial', 'Ridge_initial',
             'ElasticNet_initial', 'SVR_initial', 'HGB', 'XGBoost', 'LightGBM', 'CatBoost',
             'ExtraTrees', 'mean5', 'weighted_top3']
    near['preference'] = near.model.map({name: i for i, name in enumerate(order)})
    return near.sort_values(['preference', 'balanced_score']).iloc[0].to_dict()


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OUT.mkdir(parents=True, exist_ok=True)
    MODELS.mkdir(parents=True, exist_ok=True)
    assert not (OUT / 'run.json').exists(), 'Use a new version for a repeated executed run.'
    c = json.loads(CONTRACT.read_text(encoding='utf-8-sig'))
    parent = read_contract()
    previous = ROOT / 'Modeling/tables/m02'
    oldrun = json.loads((previous / 'run.json').read_text(encoding='utf-8'))
    verified = json.loads((previous / 'independent_verification.json').read_text(encoding='utf-8'))
    assert verified['status'] == 'passed'
    v1 = json.loads((ROOT / 'Modeling/tables/m01/verification.json').read_text(encoding='utf-8'))
    assert sha(ROOT / parent['source']) == parent['source_sha256']
    paths = {'Modeling/config/m02_rerun_contract.json': sha(CONTRACT),
             parent['source']: sha(ROOT / parent['source'])}
    for rel, expected in [('Modeling/tables/m01/hourly_frame.csv', v1['outputs_sha256']['hourly_frame.csv']),
                          ('Modeling/tables/m01/evaluation_diagnostics.csv', v1['outputs_sha256']['evaluation_diagnostics.csv']),
                          ('Modeling/tables/m02/selected_predictions.csv', oldrun['outputs_sha256']['selected_predictions.csv'])]:
        assert sha(ROOT / rel) == expected
        paths[rel] = expected
    frame = pd.read_csv(ROOT / 'Modeling/tables/m01/hourly_frame.csv', encoding='utf-8-sig', float_precision='round_trip', parse_dates=['timestamp'])
    frame = frame.loc[frame.eligible_common & frame.timestamp.lt('2021-07-01')].copy()
    old = pd.read_csv(previous / 'selected_predictions.csv', encoding='utf-8-sig', float_precision='round_trip', parse_dates=['timestamp'])
    old = old.loc[old.target.eq(c['target'])].copy()
    features = feature_columns('A', parent)
    assert len(features) == 17 and 'lag1_last' not in features
    runtime = {'executable': sys.executable, 'version': sys.version,
               'packages': {p: importlib.metadata.version(p) for p in ['numpy', 'pandas', 'scikit-learn', 'optuna', 'xgboost', 'lightgbm', 'catboost', 'joblib']}}
    run = {'status': 'running', 'started': now(), 'input_hashes': paths, 'features': features,
           'environment': runtime, 'july_august_evaluated': False,
           'selection_scope': 'retrospectively redesigned outer development; not independent final test'}
    (OUT / 'run.json').write_text(json.dumps(run, indent=2), encoding='utf-8')
    predictions, trials, fold_scores, audit, configs, manifest, inners, correlations = [], [], [], [], [], [], [], []
    started = time.perf_counter()
    for outer in parent['splits']:
        split = outer['name']
        if split not in c['outer_splits']:
            continue
        train_outer = frame.loc[frame.timestamp.le(outer['train_end'])]
        ev = frame.loc[frame.timestamp.between(outer['eval_start'], outer['eval_end'])]
        base = old.loc[old.split.eq(split) & old.model.eq('previous_hour')].copy()
        assert base.timestamp.tolist() == ev.timestamp.tolist()
        assert np.array_equal(base.actual, ev[c['target']])
        names = {'hgb_01': 'HGB_initial', 'ridge_00': 'Ridge_initial', 'elasticnet_04': 'ElasticNet_initial', 'svr_03': 'SVR_initial'}
        for record in [old.loc[old.split.eq(split) & old.model.eq(name)].copy() for name in ['previous_hour', 'previous_day', 'previous_week', *names]]:
            assert len(record) == len(ev)
            record['model'] = record.model.map(lambda n: names.get(n, n))
            predictions.append(record)
        folds = []
        for month in c['inner_validation_months'][split]:
            start = pd.Timestamp(2021, month, 1)
            train = train_outer.loc[train_outer.timestamp.lt(start)]
            val = train_outer.loc[train_outer.timestamp.ge(start) & train_outer.timestamp.lt(start + pd.offsets.MonthBegin(1))]
            assert len(train) > 300 and len(val) > 300
            assert train.timestamp.max() < val.timestamp.min() and val.timestamp.max() < ev.timestamp.min()
            weight = peak_weights(val)
            folds.append((month, train, val, weight))
            audit.append({'outer_split': split, 'kind': 'inner', 'month': month, 'train_hours': len(train),
                          'validation_hours': len(val), 'peak_weight_sum': float(weight.sum()),
                          'train_last': str(train.timestamp.max()), 'validation_first': str(val.timestamp.min()),
                          'validation_last': str(val.timestamp.max()), 'outer_evaluation_first': str(ev.timestamp.min())})
        actual = np.concatenate([v[c['target']].to_numpy() for _, _, v, _ in folds])
        times = np.concatenate([v.timestamp.to_numpy() for _, _, v, _ in folds])
        weights = np.concatenate([w for _, _, _, w in folds])
        lagpred = np.concatenate([v.lag1_maximum.to_numpy() for _, _, v, _ in folds])
        denominator = objectives(actual, lagpred, weights)
        chosen_inner, chosen_outer, inner_scores = {}, {}, {}
        def log_study(study, family):
            pareto = {t.number for t in study.best_trials}
            for t in study.trials:
                trials.append({'split': split, 'family': family, 'trial': t.number, 'state': t.state.name,
                               'overall_MAE': t.values[0], 'daily_maximum_MAE': t.values[1],
                               'baseline_overall_MAE': denominator[0], 'baseline_daily_maximum_MAE': denominator[1],
                               'balanced_score': balance(t.values, denominator), 'pareto': t.number in pareto,
                               'parameters': json.dumps(t.params), 'seconds': t.duration.total_seconds()})
        for family in c['families']:
            cache = {}
            def objective(trial):
                params = suggest(trial, family, c)
                parts = []
                for month, train, val, w in folds:
                    model = make_model(family, params)
                    model.fit(train[features], train[c['target']])
                    pred = model.predict(val[features])
                    metrics = objectives(val[c['target']], pred, w)
                    fold_scores.append({'split': split, 'family': family, 'trial': trial.number, 'month': month,
                                        'train_hours': len(train), 'validation_hours': len(val),
                                        'peak_weight_sum': float(w.sum()), 'overall_MAE': metrics[0], 'daily_maximum_MAE': metrics[1]})
                    parts.append(pred)
                predicted = np.concatenate(parts)
                cache[trial.number] = predicted
                return objectives(actual, predicted, weights)
            study = optuna.create_study(directions=['minimize', 'minimize'],
                                       sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=10),
                                       pruner=optuna.pruners.NopPruner())
            if family == 'HGB':
                study.enqueue_trial({'max_leaf_nodes': 15, 'max_iter': 150, 'learning_rate': .05, 'min_samples_leaf': 20, 'l2_regularization': 1.0})
            study.optimize(objective, n_trials=30, n_jobs=1)
            first20 = min(balance(t.values, denominator) for t in study.trials[:20])
            extended = balance(choose_trial(study, denominator).values, denominator) <= first20 * .99
            if extended:
                study.optimize(objective, n_trials=20, n_jobs=1)
            selected = choose_trial(study, denominator)
            model = make_model(family, selected.params)
            model.fit(train_outer[features], train_outer[c['target']])
            pred = model.predict(ev[features])
            path = MODELS / f'{split}_{family}.joblib'
            joblib.dump(model, path, compress=3)
            assert np.allclose(joblib.load(path).predict(ev[features]), pred, atol=1e-9, rtol=0)
            manifest.append({'split': split, 'model': family, 'file': str(path.relative_to(ROOT)), 'sha256': sha(path), 'reload_predictions_checked': True})
            chosen_inner[family], chosen_outer[family] = cache[selected.number], pred
            inner_scores[family] = balance(selected.values, denominator)
            inners.append(pd.DataFrame({'outer_split': split, 'model': family, 'timestamp': times, 'actual': actual,
                                       'prediction': chosen_inner[family], 'daily_maximum_weight': weights, 'previous_hour_prediction': lagpred}))
            configs.append({'split': split, 'model': family, 'best_trial': selected.number, 'trials': len(study.trials),
                            'extended': extended, 'inner_overall_MAE': selected.values[0], 'inner_daily_maximum_MAE': selected.values[1],
                            'inner_balanced_score': inner_scores[family], 'parameters': selected.params,
                            'train_hours': len(train_outer), 'eval_hours': len(ev)})
            log_study(study, family)
            record = base.copy()
            record['model'], record['family'], record['prediction'] = family, family, pred
            predictions.append(record)
            save(pd.DataFrame(trials), 'trials')
            save(pd.DataFrame(fold_scores), 'trial_fold_scores')
            (OUT / 'selected_configurations.json').write_text(json.dumps(configs, indent=2), encoding='utf-8')
            print(f'{split} {family}: {len(study.trials)} trials, inner score={inner_scores[family]:.4f}, outer MAEs={objectives(base.actual, pred, base.daily_maximum_weight)}', flush=True)
        for ensemble in ['mean5', 'weighted_top3']:
            members = c['families'] if ensemble == 'mean5' else sorted(inner_scores, key=inner_scores.get)[:3]
            matrix = np.column_stack([chosen_inner[m] for m in members])
            if ensemble == 'mean5':
                w = np.ones(5) / 5
            else:
                def weighted_objective(trial):
                    raw = np.array([trial.suggest_float(f'weight_{i}', 0, 1) for i in range(3)])
                    w = raw / raw.sum() if raw.sum() else np.ones(3) / 3
                    return objectives(actual, matrix @ w, weights)
                ws = optuna.create_study(directions=['minimize', 'minimize'], sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.NopPruner())
                ws.enqueue_trial({f'weight_{i}': 1/3 for i in range(3)})
                for member in range(3):
                    ws.enqueue_trial({f'weight_{i}': float(i == member) for i in range(3)})
                ws.optimize(weighted_objective, n_trials=50, n_jobs=1)
                selected = choose_trial(ws, denominator)
                raw = np.array([selected.params[f'weight_{i}'] for i in range(3)])
                w = raw / raw.sum() if raw.sum() else np.ones(3) / 3
                log_study(ws, ensemble)
            inner_pred = matrix @ w
            pred = np.column_stack([chosen_outer[m] for m in members]) @ w
            record = base.copy()
            record['model'], record['family'], record['prediction'] = ensemble, 'Ensemble', pred
            predictions.append(record)
            configs.append({'split': split, 'model': ensemble, 'members': members, 'weights': w.tolist(),
                            'inner_overall_MAE': objectives(actual, inner_pred, weights)[0],
                            'inner_daily_maximum_MAE': objectives(actual, inner_pred, weights)[1],
                            'inner_balanced_score': balance(objectives(actual, inner_pred, weights), denominator)})
            inners.append(pd.DataFrame({'outer_split': split, 'model': ensemble, 'timestamp': times, 'actual': actual,
                                       'prediction': inner_pred, 'daily_maximum_weight': weights, 'previous_hour_prediction': lagpred}))
        corr = pd.DataFrame({m: p - actual for m, p in chosen_inner.items()}).corr()
        for i, a in enumerate(c['families']):
            for b in c['families'][i+1:]:
                correlations.append({'split': split, 'model_a': a, 'model_b': b, 'inner_residual_correlation': corr.loc[a, b]})
        save(add_errors(pd.concat(predictions, ignore_index=True)), 'predictions')
        save(pd.DataFrame(trials), 'trials')
        save(pd.DataFrame(audit), 'fold_audit')
        save(pd.DataFrame(manifest), 'model_manifest')
        save(pd.concat(inners, ignore_index=True), 'inner_selected_predictions')
        save(pd.DataFrame(correlations), 'inner_residual_correlations')
        (OUT / 'selected_configurations.json').write_text(json.dumps(configs, indent=2), encoding='utf-8')
    predictions = add_errors(pd.concat(predictions, ignore_index=True))
    assert predictions.timestamp.max() < pd.Timestamp('2021-07-01')
    summarize(predictions, OUT)
    table = selection_table(predictions)
    save(table, 'selection_metrics')
    selected = select_method(table)
    selection = {'selected': selected, 'point_best': table.sort_values('balanced_score').iloc[0].to_dict(),
                 'M03_repeat_required': selected['model'] != 'HGB_initial', 'development_only': True,
                 'normalized_against': 'previous_hour', 'selection_rule': c['provisional_selection']}
    (OUT / 'selection.json').write_text(json.dumps(selection, indent=2), encoding='utf-8')
    run.update({'status': 'completed', 'finished': now(), 'seconds': time.perf_counter()-started,
                'tuning_fits': len(fold_scores), 'outer_model_fits': len(manifest), 'trials_including_ensemble': len(trials),
                'prediction_rows': len(predictions), 'outer_hours_per_method': 2184,
                'outputs_sha256': {p.name: sha(p) for p in OUT.glob('*.csv')},
                'configuration_sha256': sha(OUT / 'selected_configurations.json'), 'selection_sha256': sha(OUT / 'selection.json'),
                'script_sha256': sha(Path(__file__)), 'helper_sha256': sha(ROOT / 'Modeling/scripts/m022_compare.py')})
    assert paths['Modeling/config/m02_rerun_contract.json'] == sha(CONTRACT)
    (OUT / 'run.json').write_text(json.dumps(run, indent=2), encoding='utf-8')
    print(table.sort_values('balanced_score').to_string(index=False), flush=True)
    print(json.dumps(selection), flush=True)


if __name__ == '__main__':
    main()


코드 등록 완료: `Modeling/scripts/m02_rerun.py`

### `Modeling/scripts/m02_rerun_document_verify.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Verify current links and every displayed expanded-M02 numeric table row.

주요 함수: `main()`.

In [40]:
%%submission_source Modeling/scripts/m02_rerun_document_verify.py
"""Verify current links and every displayed expanded-M02 numeric table row."""
import ast
import json
import re
from datetime import datetime, timezone, timedelta
from pathlib import Path
from m03_verify import ROOT, rows, sha
from verify_m03_documents import main as verify_previous_documents


def main():
    out = ROOT / 'Modeling/tables/m02_rerun'
    v = json.loads((out / 'independent_verification.json').read_text(encoding='utf-8'))
    assert v['status'] == 'passed' and v['run_sha256'] == sha(out / 'run.json')
    docs = [ROOT / rel for rel in ['README.md', 'Modeling/README.md', 'Modeling/04_Modeling_원고.md',
                                  'Modeling/10.04_M02_확장모델비교_실행기록.md',
                                  'Modeling/10.04_M02_기본모형비교_실행기록.md']]
    links = 0
    for path in docs:
        text = path.read_text(encoding='utf-8-sig')
        if path == ROOT / 'README.md':
            text = text.split('## 이전 jsw 작업 단계 기록')[0]
        assert text.count('```') % 2 == 0
        for target in re.findall(r'!?\[[^\]]*\]\(([^)]+)\)', text):
            if target.startswith(('https:', 'http:', '#', 'app:')):
                continue
            assert (path.parent / target.split('#')[0].strip('<>')).exists(), (path, target)
            links += 1
    manuscript = docs[2].read_text(encoding='utf-8')
    section = manuscript.split('## 4.4 M02 확장 재실행:')[1]
    assert manuscript.count('## 4.4 M02 확장 재실행:') == 1
    labels = {'previous_hour': '직전 값', 'previous_day': '전날 같은 시각', 'previous_week': '전주 같은 시각',
              'Ridge_initial': '기존 Ridge', 'ElasticNet_initial': '기존 Elastic Net', 'SVR_initial': '기존 RBF SVR',
              'HGB_initial': '기존 HGB', 'HGB': 'Optuna HGB', 'mean5': '5모델 동일 평균', 'weighted_top3': '내부 상위 3모델 가중 평균'}
    checks = 0
    for r in rows(out / 'selection_metrics.csv'):
        line = '| '+labels.get(r['model'], r['model'])+' | '+' | '.join(f'{float(r[k]):.3f}' for k in ['overall_MAE', 'daily_maximum_MAE'])+f" | {float(r['balanced_score']):.4f} |"
        assert line in section and line in docs[3].read_text(encoding='utf-8'), line
        checks += 1
    comparisons = {(r['model'], r['split']): r for r in rows(out / 'comparison.csv')}
    conditions = {(r['model'], r['condition'], r['value']): r for r in rows(out / 'condition_errors.csv')}
    for model in ['HGB_initial', 'ExtraTrees', 'weighted_top3']:
        numbers = [float(comparisons[(model, split)]['MAE']) for split in ['dev_apr', 'dev_may', 'dev_jun']]
        numbers.append(float(conditions[(model, 'train_profile_overlap', 'False')]['MAE']))
        line = '| '+labels.get(model, model)+' | '+' | '.join(f'{x:.3f}' for x in numbers)+' |'
        assert line in section, line
        checks += 1
    for value, label in [('low->above26', '낮은 구간→26 초과'), ('above26->low', '26 초과→낮은 구간'),
                         ('low->low', '낮은 구간 유지'), ('above26->above26', '26 초과 유지')]:
        a, b = (conditions[(m, 'power_transition', value)] for m in ['HGB_initial', 'ExtraTrees'])
        line = f"| {label} | {a['hours']} | {float(a['MAE']):.3f} | {float(b['MAE']):.3f} |"
        assert line in section, line
        checks += 1
    hgb = json.loads((ROOT / 'Modeling/config/m02_contract.json').read_text(encoding='utf-8'))['grids']['HGB']
    assert hgb['max_leaf_nodes'] == [7, 15]
    fixed = {'max_leaf_nodes': 15, 'max_iter': 150, 'learning_rate': .05, 'min_samples_leaf': 20, 'l2_regularization': 1.0}
    names = {'max_leaf_nodes': '최대 잎 수', 'max_iter': '반복 횟수', 'learning_rate': '학습률',
             'min_samples_leaf': '잎의 최소 표본 수', 'l2_regularization': 'L2 규제'}
    for key, value in fixed.items():
        assert f'| {names[key]} (`{key}`) | {value} |' in section
        if key != 'max_leaf_nodes':
            assert hgb[key] == value
        checks += 1
    trials = [r for r in rows(out / 'trials.csv') if r['family'] == 'HGB']
    assert len(trials) == 90
    assert {split: sum(r['split'] == split for r in trials) for split in ['dev_apr', 'dev_may', 'dev_jun']} == {'dev_apr': 30, 'dev_may': 30, 'dev_jun': 30}
    for model in ['HGB_initial', 'HGB']:
        r = next(r for r in rows(out / 'selection_metrics.csv') if r['model'] == model)
        assert f"| {float(r['overall_MAE']):.3f} | {float(r['daily_maximum_MAE']):.3f} |" in section
        checks += 1
    memory = (ROOT / 'AGENT_MEMORY.md').read_text(encoding='utf-8')
    assert memory.count('- `session:20261004-1056`') == 1
    assert memory.count('- `decision:modeling-m02-rerun-metric-first`') == 1
    for name in ['m02_rerun.py', 'm02_rerun_verify.py', 'm02_rerun_documents.py', 'm02_rerun_document_verify.py']:
        ast.parse((ROOT / 'Modeling/scripts' / name).read_text(encoding='utf-8'))
    verify_previous_documents()
    result = {'status': 'passed', 'verified_at': datetime.now(timezone(timedelta(hours=9))).isoformat(timespec='minutes'),
              'links_checked': links, 'expanded_M02_table_rows_checked': checks,
              'HGB_tuning_trials_checked': len(trials),
              'documents_sha256': {str(p.relative_to(ROOT)): sha(p) for p in docs},
              'no_new_fitting': True, 'script_sha256': sha(Path(__file__))}
    (out / 'document_verification.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(result, ensure_ascii=False), flush=True)


if __name__ == '__main__':
    main()


코드 등록 완료: `Modeling/scripts/m02_rerun_document_verify.py`

### `Modeling/scripts/m02_rerun_documents.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Integrate verified expanded M02 into the detailed manuscript and READMEs.

주요 함수: `main()`.

In [41]:
%%submission_source Modeling/scripts/m02_rerun_documents.py
"""Integrate verified expanded M02 into the detailed manuscript and READMEs."""
import json
from pathlib import Path
from datetime import datetime, timezone, timedelta
from m03_verify import ROOT, rows, sha

OUT = ROOT / 'Modeling/tables/m02_rerun'


def main():
    verification = json.loads((OUT / 'independent_verification.json').read_text(encoding='utf-8'))
    assert verification['status'] == 'passed' and verification['run_sha256'] == sha(OUT / 'run.json')
    run = json.loads((OUT / 'run.json').read_text(encoding='utf-8'))
    for name, digest in run['outputs_sha256'].items():
        assert sha(OUT / name) == digest
    selection = json.loads((OUT / 'selection.json').read_text(encoding='utf-8'))
    assert selection['selected']['model'] == 'HGB_initial' and not selection['M03_repeat_required']
    values = {r['model']: r for r in rows(OUT / 'selection_metrics.csv')}
    comparison = {(r['model'], r['split']): r for r in rows(OUT / 'comparison.csv')}
    conditions = {(r['model'], r['condition'], r['value']): r for r in rows(OUT / 'condition_errors.csv')}
    order = ['previous_hour', 'previous_day', 'previous_week', 'Ridge_initial', 'ElasticNet_initial', 'SVR_initial',
             'HGB_initial', 'HGB', 'XGBoost', 'LightGBM', 'CatBoost', 'ExtraTrees', 'mean5', 'weighted_top3']
    labels = {'previous_hour': '직전 값', 'previous_day': '전날 같은 시각', 'previous_week': '전주 같은 시각',
              'Ridge_initial': '기존 Ridge', 'ElasticNet_initial': '기존 Elastic Net', 'SVR_initial': '기존 RBF SVR',
              'HGB_initial': '기존 HGB', 'HGB': 'Optuna HGB', 'mean5': '5모델 동일 평균', 'weighted_top3': '내부 상위 3모델 가중 평균'}
    lines = ['| 방법 | 전체 MAE | 실제 일별 최대 시간 MAE | 선택 점수 |', '|---|---:|---:|---:|']
    for model in order:
        r = values[model]
        lines.append(f"| {labels.get(model, model)} | {float(r['overall_MAE']):.3f} | {float(r['daily_maximum_MAE']):.3f} | {float(r['balanced_score']):.4f} |")
    table = '\n'.join(lines)
    monthly = ['| 방법 | 4월 MAE | 5월 MAE | 6월 MAE | 학습에 없는 배열 MAE |', '|---|---:|---:|---:|---:|']
    for model in ['HGB_initial', 'ExtraTrees', 'weighted_top3']:
        numbers = [float(comparison[(model, split)]['MAE']) for split in ['dev_apr', 'dev_may', 'dev_jun']]
        numbers.append(float(conditions[(model, 'train_profile_overlap', 'False')]['MAE']))
        monthly.append('| '+labels.get(model, model)+' | '+' | '.join(f'{x:.3f}' for x in numbers)+' |')
    monthly_table = '\n'.join(monthly)
    errors = ['| 조건 | 평가 시간 | 기존 HGB MAE | ExtraTrees MAE |', '|---|---:|---:|---:|']
    for condition, label in [('low->above26', '낮은 구간→26 초과'), ('above26->low', '26 초과→낮은 구간'),
                              ('low->low', '낮은 구간 유지'), ('above26->above26', '26 초과 유지')]:
        a = conditions[('HGB_initial', 'power_transition', condition)]
        b = conditions[('ExtraTrees', 'power_transition', condition)]
        errors.append(f"| {label} | {a['hours']} | {float(a['MAE']):.3f} | {float(b['MAE']):.3f} |")
    error_table = '\n'.join(errors)
    section = f'''

## 4.4 M02 확장 재실행: Optuna·시간순 검증·앙상블 비교 후의 판단

### 기존 HGB를 다른 모델과 다시 비교한 이유

4.2의 M02에서는 HGB가 기본 A 입력에서 가장 좋았고, 4.3의 M03에서는 마지막 값을 추가한 B의 개선을 확인했다. 그러나 이 비교만으로 XGBoost·LightGBM·CatBoost나 앙상블을 포함한 후보 중에서도 HGB를 유지할 수 있는지는 확인하지 못했다. 또한 상승 전환과 낮은 구간 유지의 오류가 남아 있었으므로, 모델을 바꾸면 그 오류가 줄어드는지도 함께 확인할 필요가 있었다.

사용자 요청에 따라 M02 자체를 확장 재실행했다. 모델 차이와 입력 효과를 분리하기 위해 **마지막 값이 없는 기본 A 입력에서** 다시 비교했고, 선택 모델이나 설정이 바뀌면 그 설정으로 M03을 반복하도록 정했다. 이전 M02·M03은 당시의 결과와 판단을 보여주는 근거로 유지한다. 이미 개발 결과를 보고 정한 확장 설계이므로 이번 점수를 새로운 독립 시험의 성적으로 표현하지 않는다.

### 학습과 예측 구간: 모든 적격 시간의 순차 비교

| 외부 개발 평가 | 학습 기간 | 예측·평가 기간 | 학습 시간 | 평가 시간 |
|---|---|---|---:|---:|
| 1차 | 1~3월 | 4월 전체 | 1,992 | 720 |
| 2차 | 1~4월 | 5월 전체 | 2,712 | 744 |
| 3차 | 1~5월 | 6월 전체 | 3,456 | 720 |

이 표의 시간 수는 정확한 1·2·24·168시간 시차가 모두 있는 common 조건의 수다. 평가월의 유리한 날만 고른 것이 아니라 조건을 충족하는 모든 시각 2,184개를 같은 순서로 비교했다. 1~3월은 학습과 내부 검증에 쓰이고, 4~6월은 모델·설정의 개발 선택에 쓰인다. 후속 M05에는 1~6월 학습→7~8월 평가를 남겼다. 7~8월은 이번 실행의 튜닝·선택에 쓰지 않았지만 과거 관찰 이력이 있어 완전히 보지 않은 독립 시험으로 부르지 않는다. 9월은 기존 분석 범위 밖이므로 편입하지 않는다.

각 외부 평가월의 설정은 그 이전 자료 안에서만 선택했다. 예를 들어 4월 평가의 Optuna는 `1월 학습→2월 검증`과 `1~2월 학습→3월 검증`을 사용한다. 5월 평가에는 4월 내부 검증이, 6월 평가에는 5월 내부 검증이 추가된다. 각 내부 학습 시각은 검증 시각보다 이르고 내부 검증의 마지막 시각은 외부 평가보다 이르다. 시각 오류로 생긴 공백 때문에 단순한 행 개수 분할 대신 실제 달력 경계를 사용했다.

외부 월이 시작되면 선택한 설정으로 이전 자료 전체를 학습하고 그 월 동안 모델을 고정한다. 관측이 끝난 입력은 매시간 갱신한다. 따라서 이 결과는 **매시간 다음 한 시간을 예측한 결과**이며, 월 전체를 월초에 한 번에 예측하거나 하루 최대 시각을 미리 지정한 결과가 아니다.

### 모델과 튜닝 범위

핵심 튜닝 대상은 HGB·XGBoost·LightGBM·CatBoost·ExtraTrees의 5계열이다. 부스팅의 서로 다른 학습 방식과 무작위 분할 트리 평균을 비교한다. 기존 Ridge·Elastic Net·RBF SVR·HGB는 초기 결과의 참조로 유지하며 직전·전날·전주 단순 기준을 포함한다. 선형·커널 참조 모델을 이번 Optuna에서 다시 튜닝하지는 않았다. 따라서 모든 계열에 동일한 탐색량을 부여한 대규모 최적화라는 주장은 하지 않는다.

각 계열과 외부 월 조합에 기본 30회, 내부 선택 점수가 최초 20회의 최선보다 1% 이상 개선된 경우에만 50회 탐색했다. 학습률·트리 수·깊이 또는 잎 수·규제 범위는 [실행 전 조건](config/m02_rerun_contract.json)에 고정했다. 외부 결과를 보고 탐색 범위나 횟수를 늘리지 않았다. 각 시도는 모든 내부 월을 검증했고, 외부 평가값으로 조기 종료하거나 예측값을 사후 보정하지 않았다.

앙상블은 튜닝한 5계열의 동일 가중 평균과 내부 검증 점수 상위 3계열의 비음수 가중 평균을 비교했다. 가중 합은 1이고 구성원·가중치는 외부 평가 전에 내부 예측만으로 선택했다. 내부 예측은 설정·가중치 선택에 사용했으므로 독립적인 성능 추정치로 해석하지 않는다. 복잡한 스태킹은 이번 비교에 포함하지 않았다.

실제 실행은 Optuna 총 {run['trials_including_ensemble']:,}회, 내부 모델 학습 {run['tuning_fits']:,}회, 외부 모델 학습 {run['outer_model_fits']}회였다. 모든 시도는 정규 CPython 3.13에서 수행됐고 총 실행 시간은 약 {run['seconds']/60:.1f}분이었다. 기존 참조 예측은 해시를 대조한 후 재사용했다.

### 전체 정확도와 최대 시간 정확도의 공동 평가

전체 시간 MAE와 **실제 일별 최대 시간 MAE**를 두 주지표로 사용했다. 후자는 예측이 끝난 뒤 실제 일별 최대가 나타난 시각의 수치 오차를 평가하며, 같은 날 최대가 여러 번이면 그 날의 합계 가중치를 1로 나눠 91일을 같은 비중으로 비교한다. 내부 검증에서는 적격 24시간이 모두 있는 날짜만 이 지표에 포함한다. 실제 최대 시각은 사후 평가용이며 입력에 들어가지 않는다.

두 목적값을 Optuna에 각각 전달하고 어느 한 지표를 개선하려면 다른 지표를 양보해야 하는 파레토 후보를 보존했다. 최종 한 후보를 정할 때는 사전에 다음 규칙을 기록했다.

`선택 점수 = max(전체 MAE / 직전 기준 전체 MAE, 최대 시간 MAE / 직전 기준 최대 시간 MAE)`

이 규칙은 두 상대오차 중 더 큰 쪽을 줄이는 절충이다. 분모는 항상 같은 기록의 직전 값 기준이다. 현장 손실비용을 알 수 없는 상태에서 임의의 비용 가중치를 만들지 않고, 두 원래 지표도 각각 공개했다. 최종 개발 선택에서는 다른 방법에 두 지표 모두 뒤지는 후보를 제외하고, 최선 점수의 1% 이내이면 기존 방법과 단일 모델을 우선한다. 실측한 운영 손실함수나 최적의 경제적 판단을 뜻하지는 않는다.

주 목표는 전력 크기의 연속값이므로 MAE와 RMSE로 크기 오차를 평가한다. AUROC·AUPRC는 고전력 사건의 정의와 예측 점수가 있는 분류·순위 문제에서 따로 평가할 수 있다. 이번에는 분류 기준을 새로 만들지 않았으며, 최대 시간 MAE가 낮다는 사실을 최대 시각 탐지 성과로 바꾸어 해석하지 않는다.

### 전체 비교 결과와 HGB 유지 판단

{table}

*학습 모델은 기본 A 입력을 사용하며, 단순 기준을 포함한 모든 방법은 같은 2,184시간을 평가한다. 최대 시간 지표는 동률을 나눈 91일 가중 평균이다. 기존 모델 행은 과거 개발에서 선택한 설정의 보존 참조이며, 새 5계열은 각 외부 월 이전 내부 검증으로 설정을 선택했다.*

정한 규칙의 최선은 **기존 HGB**였다. ExtraTrees는 전체 MAE가 6.825로 기존 6.738보다 약 1.3% 컸지만 최대 시간 MAE는 11.540으로 기존 11.751보다 약 1.8% 작았다. 두 방법만 서로 한쪽 지표에서 앞서는 파레토 후보로 남았다. 기존 HGB의 선택 점수 0.4885가 ExtraTrees의 0.4948보다 작아 기존 모델과 설정을 유지했다. 이 자료에서는 두 방법 모두 최대 시간의 상대오차보다 전체 상대오차가 커서, 최종 절충 점수의 결정 요인은 전체 MAE였다. **최대 시간만을 가장 우선한다면 ExtraTrees를 선호할 수 있으므로 목적에 따른 차이를 함께 밝힌다.**

Optuna HGB·XGBoost·LightGBM·CatBoost와 두 앙상블은 이번 범위에서 기존 HGB보다 두 주지표가 모두 컸다. 따라서 알고리즘의 이름이나 복잡성을 이유로 교체하지 않았다. 이는 이번 입력·기간·탐색 범위의 결론이며, 해당 모델의 모든 설정이나 입력에서 기존 HGB가 더 좋다는 뜻은 아니다.

### 월별 변화와 남은 실패조건

{monthly_table}

가중 앙상블은 5·6월 전체 MAE를 기존 HGB보다 낮췄지만 4월에는 8.974로 기존 8.308보다 컸다. 학습에 없는 하루 배열 1,152시간에서도 7.957로 기존 7.558보다 컸다. 4월은 모든 평가 날짜가 학습에 없는 배열이라는 M01 결과와 함께 보면, 후반 개발월의 개선만으로 전체 안정성이 좋아졌다고 주장하기 어렵다. 이 차이를 배열 반복이 오류의 원인이라는 인과 설명으로 단정하지는 않는다.

{error_table}

ExtraTrees는 26 초과 유지에서 MAE를 낮췄으나 상승 전환 20시간에서는 60.550으로 기존 47.866보다 컸고 낮은 구간 유지 625시간에서도 5.162로 기존 3.766보다 컸다. 최대 시간에서 평균 과소예측은 기존 11.260→10.897로 줄었지만 평균 과대예측은 0.491→0.643으로 늘었다. 최대 시간 MAE의 작은 개선만으로 급상승·낮은 유지 문제까지 해결했다고 판단할 수 없다.

### 기존 M03과의 연결 및 다음 질문

모델과 설정을 바꾸는 조건이 충족되지 않아 **M03을 다시 실행하지 않았다.** 기존 HGB의 A→B·C 비교는 같은 모델·설정·표본의 검증으로 계속 유효하다. 현재 후보는 그 비교에서 선택한 B이며, 최대전력 MAE 6.191과 최대 시간 MAE 7.589라는 기존 결과를 유지한다. 이번 확장 M02는 A 입력의 모델 비교이므로 모든 모델에 마지막 값을 추가했을 때의 순위를 검증한 결과는 아니다. 특히 ExtraTrees B가 HGB B보다 나쁘다는 결론은 낼 수 없다.

후속 우선순위는 기존 M03의 남은 질문으로 돌아간다. 직전 낮은 상태는 예측 시점에 알 수 있고, B에서는 해당 650시간의 MAE가 5.466으로 A 5.117보다 컸다. 다음 최소 M04는 그 조건에서 직전 값 기준을 사용하는 규칙이 낮은 유지 오차를 줄이는 대신 상승 과소예측을 얼마나 늘리는지 비교하는 것이다. 규칙과 허용 범위를 먼저 정하고 같은 시간의 전체·최대 시간·상승 오류로 채택 또는 기각한다. 이번 회차에서는 규칙과 M05를 실행하지 않았다. 새로운 근거 없이 모델 탐색만 계속 늘리지 않는다.

### 실행 코드와 검증 근거

- [M02 확장 실행 기록](10.04_M02_확장모델비교_실행기록.md), [실행 전 조건](config/m02_rerun_contract.json)
- [학습·시간순 내부 검증·두 목적 튜닝·앙상블 실행 코드](scripts/m02_rerun.py), [모델 생성과 탐색 보조 코드](scripts/m022_compare.py)
- [전체 비교표](tables/m02_rerun/comparison.csv), [두 주지표와 선택 점수](tables/m02_rerun/selection_metrics.csv), [조건별 오류](tables/m02_rerun/condition_errors.csv)
- [월별 설정과 앙상블 가중치](tables/m02_rerun/selected_configurations.json), [Optuna 모든 시도](tables/m02_rerun/trials.csv), [시간순 분할 감사](tables/m02_rerun/fold_audit.csv)
- [시각별 예측](tables/m02_rerun/predictions.csv), [개발 선택 판단](tables/m02_rerun/selection.json), [실행 환경·해시](tables/m02_rerun/run.json)
- [독립 검증 코드](scripts/m02_rerun_verify.py), [독립 검증 결과](tables/m02_rerun/independent_verification.json)

독립 검증은 표준 라이브러리로 원본 시차값 33,408개, 예측 30,576행, 지표 1,988행, 튜닝 700회, 설정·앙상블 21개를 대조했다. 시간순 학습 경계, 동일 2,184시각, 91일 최대 가중치, 저장 모델 해시·앙상블 계산·최종 선택의 일치가 확인됐고 종료 코드는 0이었다. 원고 압축과 후반기 평가는 별도 단계로 남긴다.
'''
    manuscript = ROOT / 'Modeling/04_Modeling_원고.md'
    text = manuscript.read_text(encoding='utf-8')
    assert '## 4.4 M02 확장 재실행:' not in text
    text = text.replace('이번 원고는 **M01의 예측 설계·자료 검증, M02의 기본 모델 비교, M03의 마지막 값·계절 시차 입력 비교**를 다룬다.',
                        '이번 원고는 **M01의 예측 설계·자료 검증, M02의 기본 및 확장 모델 비교, M03의 마지막 값·계절 시차 입력 비교**를 다룬다.')
    manuscript.write_text(text + section, encoding='utf-8')
    record = ROOT / 'Modeling/10.04_M02_확장모델비교_실행기록.md'
    appendix = f'''\n\n## 실행 결과와 판단 갱신\n\n{table}\n\n기존 HGB를 유지했다. ExtraTrees의 최대 시간 MAE는 약1.8% 작지만 전체 MAE는 약1.3% 크며, 정한 선택 점수는 기존 HGB가 더 낮다. 상승20시간·낮은 유지625시간에서도 ExtraTrees의 MAE가 더 컸다. 가중 앙상블의5·6월 개선은4월·새 배열 악화와 함께 제시한다. 두 지표가 모두 더 좋은 새 후보는 없었다.\n\n기존 HGB의 설정과 입력A가 그대로이므로 M03 재실행 조건이 충족되지 않았다. 기존 M03 결과와 B 개발 후보를 유지하며 ExtraTrees B의 우열은 미검증이다. 다음 최소 질문은 예측 시점에 알려진 직전 낮은 상태에서 직전 기준을 쓰는 규칙의 이익과 상승 과소예측 손실이다. M04 규칙·M05는 이번에 실행하지 않았다.\n\nOptuna700회·내부학습1630회·외부학습15회·약5.6분,예측30576행·지표1988행·원본시차33408값·700시도·21설정 독립검증종료0. 원본과 기존M01/M02/M03해시를 보존했다. 모든결과는 개발선택이며 후반기 독립시험 성과가 아니다.\n\n[전체 비교](tables/m02_rerun/comparison.csv) · [공동 지표·선택](tables/m02_rerun/selection_metrics.csv) · [오류](tables/m02_rerun/condition_errors.csv) · [설정](tables/m02_rerun/selected_configurations.json) · [선택 판단](tables/m02_rerun/selection.json) · [실행 기록](tables/m02_rerun/run.json) · [독립 검증](tables/m02_rerun/independent_verification.json) · [상세 원고4.4](04_Modeling_원고.md)\n'''
    record.write_text(record.read_text(encoding='utf-8') + appendix, encoding='utf-8')
    modeling = ROOT / 'Modeling/README.md'
    text = modeling.read_text(encoding='utf-8')
    text = text.replace('2026-10-04 M03 실행 갱신.', '2026-10-04 M02 확장 재실행 갱신.')
    text = text.replace('M01 검증, M02 네 계열 비교, M03 A/B·C 입력 비교와 기본 M04 진단 완료',
                        'M01 검증, M02 기본·확장 모델 비교, M03 A/B·C 입력 비교와 기본 M04 진단 완료')
    marker = '### 참고: 실제 시간순 전력과 변화량'
    update = '''### M02 확장 재실행과 현재 선택

[확장 실행 기록](10.04_M02_확장모델비교_실행기록.md) · [상세 원고4.4](04_Modeling_원고.md) · [사전 조건](config/m02_rerun_contract.json) · [두 주지표](tables/m02_rerun/selection_metrics.csv) · [독립 검증](tables/m02_rerun/independent_verification.json).

사용자 요청으로 A입력에서 Optuna·시간순 내부 검증을 적용해 HGB/XGBoost/LightGBM/CatBoost/ExtraTrees와 평균·가중 앙상블을 확장 비교했다. 기존 세 선형·커널 모형과 HGB, 단순기준3개를 포함한14방법이다. 전체 MAE·실제 일별 최대 시간 MAE를 각각 공개하고 직전 기준 대비 상대오차 중 더 큰 값을 줄이는 규칙을 실행 전에 고정했다.

**기존 HGB를 유지한다.** 전체/최대시간MAE는기존6.738/11.751,ExtraTrees6.825/11.540,가중앙상블6.872/13.035였다. ExtraTrees의최대시간개선과전체·상승·낮은유지악화를구분한다. 새5계열은정한범위에서두지표모두개선한후보가없었다. 기존HGB는초기개발에서설정을선택한참조이고새5계열은평가월이전내부검증으로선택했다는차이를밝힌다.

M02의외부예측은1~3월→4월720시간,1~4월→5월744시간,1~5월→6월720시간으로모든적격시각을평가한다. 내부에서는1월→2월,1~2월→3월등으로확장한다. 평가월모델은고정하고관측입력만매시간갱신한다. 7~8월은후속M05이며이번튜닝·선택에쓰지않았으나과거관찰이력이있어완전한독립시험으로부르지않는다.

Optuna700회·내부학습1630회·외부학습15회,30576예측행·1988지표행독립검증완료다. 모델/설정이그대로이므로M03재실행조건은충족되지않았다. 기존M03의B후보와결과를유지하며ExtraTrees B의우열은미검증이다. 다음최소질문은기존M04의직전낮은상태규칙의상충이며새모델탐색·규칙·M05를실행한것으로확대하지않는다.

'''
    assert marker in text
    text = text.replace(marker, update + marker)
    text = text.replace('4.3은 마지막 값과 계절 시차의 추가 효과·상충과 다음 질문을 설명한다.',
                        '4.3은 마지막 값과 계절 시차의 추가 효과·상충을, 4.4는 Optuna·시간순 검증·확장 모델과 HGB 유지 판단을 설명한다.')
    modeling.write_text(text, encoding='utf-8')
    rootreadme = ROOT / 'README.md'
    text = rootreadme.read_text(encoding='utf-8')
    marker = '같은 수의 관리시간으로 높은 최대전력 시간을 더 잘 찾는 성과는 별도 검증이 필요하다.'
    update = '''[M02 확장 재실행](Modeling/10.04_M02_확장모델비교_실행기록.md): 기본A입력에서HGB/XGBoost/LightGBM/CatBoost/ExtraTrees의Optuna시간순튜닝과두앙상블을비교했다. 전체·실제일별최대시간MAE를함께평가한규칙에서기존HGB6.738/11.751을유지했고,ExtraTrees6.825/11.540의상충을공개했다. 모델·설정이같아M03재실행조건은충족되지않았으며기존B후보를유지한다. Optuna700회·독립검증완료,상세원고4.4에결과를연결했다. M02는1~3월→4월,1~4월→5월,1~5월→6월의모든적격2184시간이며내부검증은1월→2월부터순차확장한다. 7~8월은후속M05구간으로이번선택에미사용이며과거관찰이력은유지한다.

'''
    assert marker in text
    text = text.replace(marker, update + marker)
    text = text.replace('다음 최소 질문은 직전 낮은 상태에서 단순 기준을 쓰는 규칙이',
                        '확장 M02에서도 기존 HGB를 유지했고, 다음 최소 질문은 직전 낮은 상태에서 단순 기준을 쓰는 규칙이')
    rootreadme.write_text(text, encoding='utf-8')
    # Fresh-read immediately before upserting durable project memory.
    memory_path = ROOT / 'AGENT_MEMORY.md'
    text = memory_path.read_text(encoding='utf-8')
    minute = datetime.now(timezone(timedelta(hours=9))).isoformat(timespec='minutes')
    import re
    text = re.sub(r'(?m)^- Last updated: .*$', f'- Last updated: {minute}', text, count=1)
    start = text.index('- `decision:modeling-m02-rerun-metric-first`')
    end = text.index('\n- `decision:', start+1)
    old = text[start:end]
    created = re.search(r'  - Created: (.*)', old).group(1)
    new = f'''- `decision:modeling-m02-rerun-metric-first`
  - Created: {created}
  - Updated: {minute}
  - Status: active
  - Content: 사용자 승인으로 A입력14방법의 확장M02를 실행했다. 핵심5계열은 Optuna 두MAE·시간순중첩검증, 직전 기준 상대오차의 최댓값 선택 규칙을 사전 기록했다. 700시도·내부1630/외부15학습·30576예측/1988지표 독립검증통과. 기존HGB6.738/11.751유지, ExtraTrees6.825/11.540의상충공개; 모델/설정불변으로M03재실행조건미충족·기존B유지·ExtraTrees B미검증. 외부1~3월→4월/1~4월→5월/1~5월→6월모든적격2184시간, 내부1월→2월부터확장. 7~8월이번선택미사용·과거관찰이력유지. 중단m022부분결과는선택근거제외.
  - Evidence: Modeling/config/m02_rerun_contract.json, Modeling/tables/m02_rerun/selection.json, Modeling/tables/m02_rerun/independent_verification.json, Modeling/04_Modeling_원고.md4.4.
'''
    text = text[:start] + new + text[end:]
    start = text.index('- `handoff:current`')
    end = text.index('\n## Session log', start)
    text = text[:start] + f'''- `handoff:current`
  - Updated: {minute}
  - Current state: M01·초기M02·M03와 확장M02 완료. 확장A14방법·700시도·내부1630/외부15학습·30576예측/1988지표독립검증종료0. 기존HGB6.738/11.751유지·ExtraTrees6.825/11.540상충, HGB설정불변으로M03재실행불필요·기존B최대MAE6.191/최대시간MAE7.589유지. 원고4.4·확장기록·두README에평가구간/선택이력/결과연결반영. 새압축본/PDF/M05미실행.
  - Next step: 기존M03에서직전낮은상태650시간B MAE5.466>A5.117이었다. 그조건에서직전기준을쓰는최소M04규칙의낮은유지개선과상승과소예측악화를같은시각에서비교하고채택/기각한다. 새로운실행조건과허용범위는실행전기록한다. 후반기M05는1~6월→7~8월이며과거관찰이력상완전한독립시험으로부르지않는다.
  - Blockers: 없음. 원본/기존M01~M03보존, 추가자료대기없음. 중단m022는미완료참고기록이고현재결과아님.
''' + text[end:]
    session = text.index('- `session:20261004-1056`')
    session_end = text.index('\n- `session:', session+1)
    block = text[session:session_end]
    block = re.sub(r'  - Last activity: .*', f'  - Last activity: {minute}', block, count=1)
    block += f'\n  - Expanded M02 execution update: 사용자최신승인으로A14방법·두목적Optuna700시도·시간순중첩비교를실행해기존HGB유지를판단했다. 30576예측/1988지표·원본33408시차값·21설정/앙상블·시간순경계·최종선택 독립검증종료0. ExtraTrees최대시간개선/전체·상승·낮은유지악화를공개하고M03재실행조건미충족을기록했다. 원고4.4·실행기록·두README를갱신했으며사용자질문에학습/내부검증/외부평가기간과모든적격시간예측을구분해설명했다. 규칙M04/M05/압축/PDF미실행.\n'
    text = text[:session] + block + text[session_end:]
    memory_path.write_text(text, encoding='utf-8')
    print(json.dumps({'documents_written': 5, 'selected': selection['selected']['model'], 'updated_at': minute}), flush=True)


if __name__ == '__main__':
    main()


코드 등록 완료: `Modeling/scripts/m02_rerun_documents.py`

### `Modeling/scripts/m02_rerun_verify.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Independent stdlib verification of chronological folds and both M02 objectives.

주요 함수: `two_metrics()`, `score()`, `check_peak_weights()`, `main()`.

In [42]:
%%submission_source Modeling/scripts/m02_rerun_verify.py
"""Independent stdlib verification of chronological folds and both M02 objectives."""
import json
import math
import sys
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from m03_verify import ROOT, rows, sha, close, verify_metric, select

OUT = ROOT / 'Modeling/tables/m02_rerun'


def two_metrics(records):
    errors = [abs(float(r['prediction']) - float(r['actual'])) for r in records]
    weights = [float(r['daily_maximum_weight']) for r in records]
    return math.fsum(errors) / len(errors), math.fsum(e*w for e, w in zip(errors, weights)) / math.fsum(weights)


def score(values, denominator):
    return max(a/b for a, b in zip(values, denominator))


def check_peak_weights(g, complete_only=False):
    days = defaultdict(list)
    for r in g:
        days[r['timestamp'][:10]].append(r)
    for day in days.values():
        maximum = max(float(r['actual']) for r in day)
        ties = sum(float(r['actual']) == maximum for r in day)
        for r in day:
            value = 1 / ties if float(r['actual']) == maximum and len(day) == 24 else 0
            close(r['daily_maximum_weight'], value)
        if not complete_only:
            assert len(day) == 24
    return sum(len(day) == 24 for day in days.values())


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    run = json.loads((OUT / 'run.json').read_text(encoding='utf-8'))
    contract = json.loads((ROOT / 'Modeling/config/m02_rerun_contract.json').read_text(encoding='utf-8-sig'))
    assert run['status'] == 'completed' and not run['july_august_evaluated']
    assert len(run['features']) == 17 and 'lag1_last' not in run['features']
    for rel, digest in run['input_hashes'].items():
        assert sha(ROOT / rel) == digest
    for name, digest in run['outputs_sha256'].items():
        assert sha(OUT / name) == digest
    assert sha(OUT / 'selected_configurations.json') == run['configuration_sha256']
    assert sha(OUT / 'selection.json') == run['selection_sha256']
    m1 = json.loads((ROOT / 'Modeling/config/m01_contract.json').read_text(encoding='utf-8'))
    raw = {}
    for r in rows(ROOT / m1['source']):
        date, hour = int(r['날짜']), float(r['시간'])
        if 20210101 <= date <= 20210630 and hour.is_integer() and 0 <= hour <= 23:
            ts = datetime.strptime(str(date), '%Y%m%d') + timedelta(hours=int(hour))
            slots = [float(r[k]) for k in ['15분', '30분', '45분', '60분']]
            raw[ts] = {'last': slots[-1], 'mean': math.fsum(slots)/4, 'maximum': max(slots)}
    frame = {datetime.fromisoformat(r['timestamp']): r for r in rows(ROOT / 'Modeling/tables/m01/hourly_frame.csv')}
    lag_checks = 0
    for ts, r in frame.items():
        if ts >= datetime(2021, 7, 1) or r['eligible_common'] != 'True':
            continue
        for lag in [1, 2, 24, 168]:
            for name in ['mean', 'maximum']:
                close(r[f'lag{lag}_{name}'], raw[ts - timedelta(hours=lag)][name])
                lag_checks += 1
    p = rows(OUT / 'predictions.csv')
    grouped, daily, keyed = defaultdict(list), defaultdict(list), {}
    for r in p:
        ts = datetime.fromisoformat(r['timestamp'])
        assert datetime(2021, 4, 1) <= ts < datetime(2021, 7, 1)
        close(r['actual'], raw[ts]['maximum'])
        key = (r['model'], r['split'], r['timestamp'])
        assert key not in keyed
        keyed[key] = r
        e = float(r['prediction']) - float(r['actual'])
        assert math.isfinite(e)
        for k, v in {'signed_error': e, 'absolute_error': abs(e), 'under_amount': max(-e, 0), 'over_amount': max(e, 0)}.items():
            close(r[k], v)
        if r['model'].startswith('previous_'):
            lag = {'previous_hour': 1, 'previous_day': 24, 'previous_week': 168}[r['model']]
            close(r['prediction'], raw[ts - timedelta(hours=lag)]['maximum'])
        grouped[(r['target'], r['model'])].append(r)
        daily[(r['target'], r['model'], r['date'])].append(r)
    assert len(grouped) == 14 and len(p) == 14 * 2184
    expected_keys = {(r['split'], r['timestamp']) for r in grouped[('target_maximum', 'previous_hour')]}
    for g in grouped.values():
        assert len(g) == 2184 and {(r['split'], r['timestamp']) for r in g} == expected_keys
        assert check_peak_weights(g) == 91
    metric_checks = 0
    for table in ['comparison', 'condition_errors', 'daily_errors']:
        for r in rows(OUT / f'{table}.csv'):
            g = grouped[(r['target'], r['model'])]
            if table == 'comparison':
                kind = 'all' if r['split'] == 'pooled_development' else 'split'
                g = select(g, kind, r['split'])
            elif table == 'condition_errors':
                kind = r['condition']
                g = select(g, kind, r['value'])
            else:
                kind = 'all'
                g = daily[(r['target'], r['model'], r['date'])]
            verify_metric(r, g, kind)
            metric_checks += 1
    references = {'hgb_01': 'HGB_initial', 'ridge_00': 'Ridge_initial', 'elasticnet_04': 'ElasticNet_initial', 'svr_03': 'SVR_initial'}
    for r in rows(ROOT / 'Modeling/tables/m02/selected_predictions.csv'):
        if r['target'] == 'target_maximum' and r['model'] in references:
            close(keyed[(references[r['model']], r['split'], r['timestamp'])]['prediction'], r['prediction'])
    inner = defaultdict(list)
    for r in rows(OUT / 'inner_selected_predictions.csv'):
        inner[(r['outer_split'], r['model'])].append(r)
        ts = datetime.fromisoformat(r['timestamp'])
        close(r['actual'], raw[ts]['maximum'])
        close(r['previous_hour_prediction'], raw[ts-timedelta(hours=1)]['maximum'])
    for (split, model), g in inner.items():
        check_peak_weights(g, complete_only=True)
        assert len({r['timestamp'] for r in g}) == len(g)
        cutoff = {'dev_apr': '2021-04-01', 'dev_may': '2021-05-01', 'dev_jun': '2021-06-01'}[split]
        assert all('2021-02-01' <= r['timestamp'] < cutoff for r in g)
    trials = defaultdict(list)
    for r in rows(OUT / 'trials.csv'):
        close(r['balanced_score'], score((float(r['overall_MAE']), float(r['daily_maximum_MAE'])),
                                       (float(r['baseline_overall_MAE']), float(r['baseline_daily_maximum_MAE']))))
        trials[(r['split'], r['family'])].append(r)
    configs = json.loads((OUT / 'selected_configurations.json').read_text(encoding='utf-8'))
    for c in configs:
        split, model = c['split'], c['model']
        g = inner[(split, model)]
        values = two_metrics(g)
        close(values[0], c['inner_overall_MAE'])
        close(values[1], c['inner_daily_maximum_MAE'])
        b = [{**r, 'prediction': r['previous_hour_prediction']} for r in g]
        denominator = two_metrics(b)
        close(c['inner_balanced_score'], score(values, denominator))
        if model in contract['families']:
            group = trials[(split, model)]
            assert len(group) == c['trials'] == (50 if c['extended'] else 30)
            first20 = min(float(r['balanced_score']) for r in group[:20])
            first30 = min(float(r['balanced_score']) for r in group[:30])
            assert c['extended'] == (first30 <= first20 * .99)
            best = min(group, key=lambda r: (float(r['balanced_score']), int(r['trial'])))
            assert int(best['trial']) == c['best_trial']
            assert json.loads(best['parameters']) == c['parameters']
            close(best['overall_MAE'], values[0])
            close(best['daily_maximum_MAE'], values[1])
        else:
            close(sum(c['weights']), 1)
            assert all(w >= 0 for w in c['weights'])
            bytime = {m: {r['timestamp']: r for r in inner[(split, m)]} for m in c['members']}
            for r in g:
                close(r['prediction'], math.fsum(w*float(bytime[m][r['timestamp']]['prediction']) for m, w in zip(c['members'], c['weights'])))
            for r in grouped[('target_maximum', model)]:
                if r['split'] == split:
                    close(r['prediction'], math.fsum(w*float(keyed[(m, split, r['timestamp'])]['prediction']) for m, w in zip(c['members'], c['weights'])))
    for r in rows(OUT / 'fold_audit.csv'):
        assert r['train_last'] < r['validation_first'] <= r['validation_last'] < r['outer_evaluation_first']
        assert datetime.fromisoformat(r['validation_first']).month == int(r['month'])
        cutoff = r['validation_first'][:7] + '-01'
        expected_train = sum(ts.isoformat(sep=' ') < cutoff and row['eligible_common'] == 'True' for ts, row in frame.items())
        assert expected_train == int(r['train_hours'])
    trial_folds = defaultdict(list)
    for r in rows(OUT / 'trial_fold_scores.csv'):
        trial_folds[(r['split'], r['family'], r['trial'])].append(r)
    for (split, family), group in trials.items():
        for r in group:
            if family not in contract['families']:
                continue
            parts = trial_folds[(split, family, r['trial'])]
            assert len(parts) == len(contract['inner_validation_months'][split])
            close(r['overall_MAE'], math.fsum(float(x['overall_MAE'])*int(x['validation_hours']) for x in parts)/sum(int(x['validation_hours']) for x in parts))
            close(r['daily_maximum_MAE'], math.fsum(float(x['daily_maximum_MAE'])*float(x['peak_weight_sum']) for x in parts)/math.fsum(float(x['peak_weight_sum']) for x in parts))
    table = rows(OUT / 'selection_metrics.csv')
    denom = two_metrics(grouped[('target_maximum', 'previous_hour')])
    for r in table:
        values = two_metrics(grouped[('target_maximum', r['model'])])
        close(r['overall_MAE'], values[0])
        close(r['daily_maximum_MAE'], values[1])
        close(r['balanced_score'], score(values, denom))
        nondominated = not any(float(x['overall_MAE']) <= values[0] and float(x['daily_maximum_MAE']) <= values[1] and
                              (float(x['overall_MAE']) < values[0] or float(x['daily_maximum_MAE']) < values[1]) for x in table)
        assert r['nondominated'] == str(nondominated)
    selection = json.loads((OUT / 'selection.json').read_text(encoding='utf-8'))
    near = [r for r in table if r['nondominated'] == 'True' and float(r['balanced_score']) <= min(float(x['balanced_score']) for x in table)*1.01]
    order = ['previous_hour', 'previous_day', 'previous_week', 'HGB_initial', 'Ridge_initial', 'ElasticNet_initial', 'SVR_initial', 'HGB', 'XGBoost', 'LightGBM', 'CatBoost', 'ExtraTrees', 'mean5', 'weighted_top3']
    chosen = min(near, key=lambda r: (order.index(r['model']), float(r['balanced_score'])))
    assert selection['selected']['model'] == chosen['model']
    for r in rows(OUT / 'model_manifest.csv'):
        assert sha(ROOT / r['file']) == r['sha256'] and r['reload_predictions_checked'] == 'True'
    result = {'status': 'passed', 'verified_at': datetime.now(timezone(timedelta(hours=9))).isoformat(timespec='minutes'),
              'prediction_rows': len(p), 'metric_rows': metric_checks, 'raw_lag_values_checked': lag_checks,
              'inner_configurations_checked': len(configs), 'trials_checked': sum(map(len, trials.values())),
              'identical_outer_timestamps': 2184, 'daily_maximum_dates': 91,
              'selected_model': chosen['model'], 'july_august_evaluated': False,
              'run_sha256': sha(OUT / 'run.json'), 'verifier_sha256': sha(Path(__file__))}
    (OUT / 'independent_verification.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    print(json.dumps(result), flush=True)


if __name__ == '__main__':
    main()


코드 등록 완료: `Modeling/scripts/m02_rerun_verify.py`

### `Modeling/scripts/m02_verify.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Independent stdlib verification of M02 saved forecasts, scores and weights.

주요 함수: `rows()`, `close()`, `power_state()`, `main()`.

In [43]:
%%submission_source Modeling/scripts/m02_verify.py
"""Independent stdlib verification of M02 saved forecasts, scores and weights."""
import csv
import hashlib
import json
import math
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/m02"


def rows(path):
    with path.open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def close(a, b):
    assert math.isclose(float(a), float(b), rel_tol=1e-10, abs_tol=1e-9), (a, b)


def power_state(maximum, rounded):
    if float(maximum) == 0:
        return "zero"
    if float(rounded) < 20:
        return "below20"
    return "low" if float(rounded) <= 26 else "above26"


def main():
    run = json.loads((OUT / "run.json").read_text(encoding="utf-8"))
    assert run["status"] == "completed" and run["fits_completed"] == 150
    assert run["july_august_evaluated"] is False and run["last_slot_used"] is False
    parent_path = ROOT / "Modeling/config/m01_contract.json"
    parent = json.loads(parent_path.read_text(encoding="utf-8"))
    source = ROOT / parent["source"]
    for key, path in {"m01_contract": parent_path, "m02_contract": ROOT / "Modeling/config/m02_contract.json", "source": source,
                      **{n: ROOT / "Modeling/tables/m01" / n for n in ["hourly_frame.csv", "evaluation_diagnostics.csv", "split_counts.csv"]}}.items():
        assert hashlib.sha256(path.read_bytes()).hexdigest() == run["input_hashes"][key]
    assert run["features"] == parent["calendar_features"] + parent["feature_groups"]["A"]
    for name, digest in run["outputs_sha256"].items():
        assert hashlib.sha256((OUT / name).read_bytes()).hexdigest() == digest
    f = {r["timestamp"]: r for r in rows(ROOT / "Modeling/tables/m01/hourly_frame.csv")}
    diags = {(r["split"], r["timestamp"]): r for r in rows(ROOT / "Modeling/tables/m01/evaluation_diagnostics.csv") if r["pool"] == "common"}
    c = json.loads((ROOT / "Modeling/config/m02_contract.json").read_text(encoding="utf-8"))
    pred = rows(OUT / "predictions.csv")
    groups, membership = defaultdict(list), defaultdict(set)
    for r in pred:
        own = f[r["timestamp"]]
        assert own["eligible_common"] == "True"
        assert datetime.fromisoformat(r["timestamp"]) < datetime(2021, 7, 1)
        close(r["actual"], own[r["target"]])
        prior_state = power_state(own["lag1_maximum"], math.floor(float(own["lag1_mean"]) + 0.5))
        current_state = power_state(own["target_maximum"], own["diag_target_rounded_mean"])
        assert r["power_transition"] == prior_state + "->" + current_state
        assert r["diag_production_transition"] == own["diag_production_transition"]
        error = float(r["prediction"]) - float(r["actual"])
        close(error, r["signed_error"])
        close(abs(error), r["absolute_error"])
        close(max(-error, 0), r["under_amount"])
        close(max(error, 0), r["over_amount"])
        diag = diags[(r["split"], r["timestamp"])]
        for col in ["profile_weight", "daily_maximum_weight"]:
            close(r[col], diag[col])
        assert r["train_profile_overlap"] == diag["train_profile_overlap"]
        if r["model"] in c["baselines"]:
            lag = c["baselines"][r["model"]]
            metric_name = "maximum" if r["target"] == "target_maximum" else "mean"
            close(r["prediction"], own[f"lag{lag}_{metric_name}"])
        key = (r["target"], r["model"])
        groups[key].append(r)
        membership[key].add(r["timestamp"])
    common_times = next(iter(membership.values()))
    assert len(common_times) == 2184
    assert all(v == common_times for v in membership.values())
    assert all(len(v) == 2184 for v in groups.values())
    checked_scores = 0
    for s in rows(OUT / "all_candidate_metrics.csv"):
        g = groups[(s["target"], s["model"])]
        if s["split"] != "pooled_development":
            g = [r for r in g if r["split"] == s["split"]]
        errors = [float(r["prediction"]) - float(r["actual"]) for r in g]
        assert len(g) == int(s["hours"])
        close(sum(abs(e) for e in errors) / len(g), s["MAE"])
        close(math.sqrt(sum(e * e for e in errors) / len(g)), s["RMSE"])
        close(sum(max(-e, 0) for e in errors) / len(g), s["mean_under"])
        close(sum(max(e, 0) for e in errors) / len(g), s["mean_over"])
        checked_scores += 1
    peaks = 0
    for s in rows(OUT / "condition_errors.csv"):
        if s["condition"] != "daily_maximum":
            continue
        g = [r for r in groups[(s["target"], s["model"])] if float(r["daily_maximum_weight"]) > 0]
        w = [float(r["daily_maximum_weight"]) for r in g]
        close(sum(w), 91)
        close(sum(wi * abs(float(r["prediction"]) - float(r["actual"])) for wi, r in zip(w, g)) / sum(w), s["MAE"])
        close(sum(wi * max(float(r["actual"]) - float(r["prediction"]), 0) for wi, r in zip(w, g)) / sum(w), s["mean_under"])
        peaks += 1
    for r in rows(OUT / "model_manifest.csv"):
        assert hashlib.sha256((ROOT / r["file"]).read_bytes()).hexdigest() == r["sha256"]
        assert r["reload_predictions_checked"] == "True"
    for r in rows(OUT / "training_audit.csv"):
        assert datetime.fromisoformat(r["train_last_timestamp"]) < datetime.fromisoformat(r["eval_first_timestamp"])
        assert r["training_only_scalers_verified"] == "True"
        assert json.loads(r["warnings"]) == []
    result = {"status": "passed", "prediction_rows_checked": len(pred), "metrics_recomputed": checked_scores,
              "same_evaluation_timestamps": len(common_times), "daily_maximum_metrics_recomputed": peaks,
              "daily_maximum_weighted_dates": 91, "later_period_absent": True,
              "baseline_predictions_checked": True, "errors_and_input_labels_checked": True,
              "outputs_and_model_hashes_checked": True}
    (OUT / "independent_verification.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result, ensure_ascii=False))


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m02_verify.py`

### `Modeling/scripts/m03_ablation.py`

원고 연결: Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: M03 fixed-HGB input ablation; verified parent predictions reused, dev only.

주요 함수: `now()`, `subsets()`, `main()`.

In [44]:
%%submission_source Modeling/scripts/m03_ablation.py
"""M03 fixed-HGB input ablation; verified parent predictions reused, dev only."""
import argparse
import importlib.metadata
import json
import os
import platform
import sys
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from m01_prepare import ROOT, feature_columns, read_contract, sha
from m02_compare import metric, state


def now():
    return datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes")


def subsets(g):
    result = [("all", "all", g, None), ("profile_reweighted", "all", g, g.profile_weight)]
    peak = g.loc[g.daily_maximum_weight.gt(0)]
    result.append(("daily_maximum", "all", peak, peak.daily_maximum_weight))
    for col in ["split", "month", "weekend", "hour", "power_transition", "diag_production_transition", "train_profile_overlap"]:
        result.extend((col, str(value), part, None) for value, part in g.groupby(col))
    return result


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--stage", choices=["ab", "c"], required=True)
    stage = parser.parse_args().stage
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    contract_path = ROOT / f"Modeling/config/m03_{stage}_contract.json"
    c = json.loads(contract_path.read_text(encoding="utf-8"))
    parent = read_contract()
    out = ROOT / f"Modeling/tables/m03/{stage}"
    model_dir = ROOT / f"Modeling/models/m03/{stage}"
    out.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)
    def save(df, name):
        df.to_csv(out / f"{name}.csv", index=False, encoding="utf-8-sig")
    v1 = json.loads((ROOT / "Modeling/tables/m01/verification.json").read_text(encoding="utf-8"))
    assert sha(ROOT / parent["source"]) == parent["source_sha256"]
    inputs = {"source": sha(ROOT / parent["source"]), "contract": sha(contract_path),
              "m01_contract": sha(ROOT / "Modeling/config/m01_contract.json")}
    for name in ["hourly_frame.csv", "evaluation_diagnostics.csv", "split_counts.csv"]:
        path = ROOT / "Modeling/tables/m01" / name
        assert sha(path) == v1["outputs_sha256"][name]
        inputs[str(path.relative_to(ROOT))] = sha(path)
    reuse_dir = ROOT / ("Modeling/tables/m02" if stage == "ab" else "Modeling/tables/m03/ab")
    reuse_run = json.loads((reuse_dir / "run.json").read_text(encoding="utf-8"))
    assert reuse_run["status"] == "completed"
    verification = json.loads((reuse_dir / "independent_verification.json").read_text(encoding="utf-8"))
    assert verification["status"] == "passed"
    reuse_name = "selected_predictions.csv" if stage == "ab" else "predictions.csv"
    for name in [reuse_name, "model_manifest.csv"]:
        path = reuse_dir / name
        assert sha(path) == reuse_run["outputs_sha256"][name]
        inputs[str(path.relative_to(ROOT))] = sha(path)
    inputs[str((reuse_dir / "run.json").relative_to(ROOT))] = sha(reuse_dir / "run.json")
    if stage == "ab":
        assert inputs["m01_contract"] == reuse_run["input_hashes"]["m01_contract"]
        assert sha(ROOT / c["parent"]) == reuse_run["input_hashes"]["m02_contract"]
        assert c["hgb_parameters"] == next(x[3] for x in __import__("m02_compare").candidates(json.loads((ROOT / c["parent"]).read_text(encoding="utf-8"))) if x[0] == "hgb_01")
    else:
        assert c["hgb_parameters"] == json.loads((ROOT / c["parent"]).read_text(encoding="utf-8"))["hgb_parameters"]
    f = pd.read_csv(ROOT / "Modeling/tables/m01/hourly_frame.csv", encoding="utf-8-sig", float_precision="round_trip", parse_dates=["timestamp", "date"])
    diag = pd.read_csv(ROOT / "Modeling/tables/m01/evaluation_diagnostics.csv", encoding="utf-8-sig", parse_dates=["timestamp", "date"])
    expected = pd.read_csv(ROOT / "Modeling/tables/m01/split_counts.csv", encoding="utf-8-sig")
    reused = pd.read_csv(reuse_dir / reuse_name, encoding="utf-8-sig", float_precision="round_trip", parse_dates=["timestamp", "date"])
    manifest = pd.read_csv(reuse_dir / "model_manifest.csv", encoding="utf-8-sig")
    if stage == "ab":
        reused = reused.loc[reused.model.eq("hgb_01")].copy()
        reused["group"] = "A"
    f["power_state"] = state(f.target_maximum, f.diag_target_rounded_mean)
    f["prior_power_state"] = state(f.lag1_maximum, np.floor(f.lag1_mean + 0.5))
    f["power_transition"] = f.prior_power_state + "->" + f.power_state
    features = {group: feature_columns(group, parent) for group in c["groups"]}
    assert features["B"] == features["A"] + ["lag1_last"]
    for group, cols in features.items():
        assert not any(x.startswith(("target_", "diag_")) for x in cols)
    run = {"status": "running", "started": now(), "stage": stage, "input_hashes": inputs,
           "features": features, "parameters": c["hgb_parameters"],
           "runtime": {"executable": sys.executable, "version": sys.version, "platform": platform.platform(),
                       "packages": {x: importlib.metadata.version(x) for x in ["numpy", "pandas", "scikit-learn", "joblib"]}}}
    (out / "run.json").write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding="utf-8")
    begin = time.perf_counter()
    records, audits, models = [], [], []
    for split in parent["splits"]:
        if split["name"] not in c["splits"]:
            continue
        tr = f.loc[f.eligible_common & (f.timestamp <= pd.Timestamp(split["train_end"]))]
        ev = f.loc[f.eligible_common & f.timestamp.between(pd.Timestamp(split["eval_start"]), pd.Timestamp(split["eval_end"]))]
        count = expected.loc[expected.split.eq(split["name"]) & expected.pool.eq("common")].iloc[0]
        assert len(tr) == count.train_hours and len(ev) == count.eval_hours
        assert tr.timestamp.max() < ev.timestamp.min()
        d = diag.loc[diag.split.eq(split["name"]) & diag.pool.eq("common")]
        base = ev[["timestamp", "date", "month", "weekend", "power_transition", "diag_production_transition", "diag_target_profile"]].merge(
            d[["timestamp", "train_profile_overlap", "profile_weight", "daily_maximum_weight"]], on="timestamp", validate="one_to_one")
        base["hour"] = base.timestamp.dt.hour
        assert base.timestamp.tolist() == ev.timestamp.tolist()
        for target in c["targets"]:
            for group in c["groups"]:
                cols = features[group]
                assert tr[cols].notna().all().all() and ev[cols].notna().all().all()
                start = time.perf_counter()
                if group in c["new_groups"]:
                    estimator = HistGradientBoostingRegressor(**c["hgb_parameters"])
                    estimator.fit(tr[cols], tr[target])
                    pred = estimator.predict(ev[cols])
                    path = model_dir / f"{split['name']}_{target}_{group}.joblib"
                    joblib.dump(estimator, path, compress=3)
                    assert np.allclose(joblib.load(path).predict(ev[cols]), pred, rtol=0, atol=1e-10)
                    source = "new_fit"
                else:
                    mr = manifest.loc[manifest.split.eq(split["name"]) & manifest.target.eq(target) &
                                      (manifest.model.eq("hgb_01") if stage == "ab" else manifest.group.eq(group))].iloc[0]
                    path = ROOT / mr.file
                    assert sha(path) == mr.sha256
                    estimator = joblib.load(path)
                    pred = estimator.predict(ev[cols])
                    old = reused.loc[reused.split.eq(split["name"]) & reused.target.eq(target) & reused.group.eq(group)]
                    assert old.timestamp.tolist() == ev.timestamp.tolist()
                    assert np.array_equal(old.actual.to_numpy(), ev[target].to_numpy())
                    assert np.allclose(old.prediction.to_numpy(), pred, rtol=0, atol=1e-10)
                    source = "verified_parent_model"
                assert list(estimator.feature_names_in_) == cols
                assert all(estimator.get_params()[k] == v for k, v in c["hgb_parameters"].items())
                assert np.isfinite(pred).all()
                rec = base.copy()
                rec["split"], rec["target"], rec["group"] = split["name"], target, group
                rec["actual"], rec["prediction"] = ev[target].to_numpy(), pred
                records.append(rec)
                models.append({"split": split["name"], "target": target, "group": group, "file": str(path.relative_to(ROOT)),
                               "sha256": sha(path), "source": source, "reload_predictions_checked": True})
                audits.append({"split": split["name"], "target": target, "group": group, "train_hours": len(tr), "eval_hours": len(ev),
                               "features": len(cols), "source": source, "train_end": str(tr.timestamp.max()), "eval_start": str(ev.timestamp.min()),
                               "seconds": time.perf_counter() - start})
        print(f"completed {stage}: {split['name']}", flush=True)
    p = pd.concat(records, ignore_index=True)
    p["signed_error"] = p.prediction - p.actual
    p["absolute_error"] = p.signed_error.abs()
    p["under_amount"] = (-p.signed_error).clip(lower=0)
    p["over_amount"] = p.signed_error.clip(lower=0)
    assert p.timestamp.max() < pd.Timestamp("2021-07-01")
    save(p, "predictions")
    save(pd.DataFrame(audits), "training_audit")
    save(pd.DataFrame(models), "model_manifest")
    scores, conditions, daily = [], [], []
    for (target, group), g in p.groupby(["target", "group"]):
        scores.append({"target": target, "group": group, "split": "pooled_development", **metric(g)})
        scores.extend({"target": target, "group": group, "split": s, **metric(part)} for s, part in g.groupby("split"))
        for kind, value, part, weights in subsets(g):
            if len(part):
                conditions.append({"target": target, "group": group, "condition": kind, "value": value, **metric(part, weights),
                                   "share_total_absolute_error_unweighted": float(part.absolute_error.sum() / g.absolute_error.sum())})
        daily.extend({"target": target, "group": group, "date": date, **metric(part)} for date, part in g.groupby("date"))
    scores = pd.DataFrame(scores)
    save(scores, "comparison")
    save(pd.DataFrame(conditions), "condition_errors")
    save(pd.DataFrame(daily), "daily_errors")
    pairs, deltas, daily_deltas = [], [], []
    keys = ["split", "target", "timestamp"]
    for before, after in c["comparisons"]:
        a = p.loc[p.group.eq(before)].drop(columns="group")
        b = p.loc[p.group.eq(after), keys + ["actual", "prediction"]]
        pair = a.merge(b, on=keys, suffixes=("_before", "_after"), validate="one_to_one")
        assert np.array_equal(pair.actual_before, pair.actual_after)
        pair["before"], pair["after"] = before, after
        for label in ["before", "after"]:
            error = pair[f"prediction_{label}"] - pair.actual_before
            pair[f"abs_{label}"] = error.abs()
            pair[f"under_{label}"] = (-error).clip(lower=0)
            pair[f"over_{label}"] = error.clip(lower=0)
        for kind in ["abs", "under", "over"]:
            pair[f"delta_{kind}"] = pair[f"{kind}_after"] - pair[f"{kind}_before"]
        pairs.append(pair)
        for target, g in pair.groupby("target"):
            for kind, value, part, weights in subsets(g):
                if part.empty:
                    continue
                w = np.ones(len(part)) if weights is None else np.asarray(weights)
                deltas.append({"target": target, "before": before, "after": after, "condition": kind, "value": value,
                               "hours": len(part), "weight_sum": float(w.sum()),
                               **{f"delta_{x}": float(np.average(part[f"delta_{x}"], weights=w)) for x in ["abs", "under", "over"]},
                               "improved_hours": int(part.delta_abs.lt(-1e-10).sum()), "worsened_hours": int(part.delta_abs.gt(1e-10).sum())})
            for date, part in g.groupby("date"):
                daily_deltas.append({"target": target, "before": before, "after": after, "date": date, "hours": len(part),
                                     **{f"delta_{x}": float(part[f"delta_{x}"].mean()) for x in ["abs", "under", "over"]}})
    save(pd.concat(pairs, ignore_index=True), "paired_predictions")
    save(pd.DataFrame(deltas), "paired_deltas")
    save(pd.DataFrame(daily_deltas), "daily_deltas")
    chosen = {}
    for target, g in scores.loc[scores.split.eq("pooled_development")].groupby("target"):
        near = g.loc[g.MAE <= g.MAE.min() * 1.01].copy()
        near["size"] = near.group.map(lambda x: len(features[x]))
        chosen[target] = near.sort_values(["size", "MAE"]).iloc[0][["group", "MAE", "RMSE"]].to_dict()
    assert sha(contract_path) == inputs["contract"]
    run.update({"status": "completed", "finished": now(), "seconds": time.perf_counter() - begin,
                "fits_completed": sum(x["source"] == "new_fit" for x in audits), "reused_models_verified": sum(x["source"] != "new_fit" for x in audits),
                "prediction_rows": len(p), "july_august_evaluated": False, "provisional_candidates": chosen,
                "outputs_sha256": {x.name: sha(x) for x in out.glob("*.csv")},
                "script_sha256": sha(Path(__file__)), "helper_sha256": {x: sha(ROOT / f"Modeling/scripts/{x}") for x in ["m01_prepare.py", "m02_compare.py"]}})
    (out / "run.json").write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding="utf-8")
    print(scores.loc[scores.split.eq("pooled_development"), ["target", "group", "MAE", "RMSE", "mean_under", "mean_over"]].to_string(index=False), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m03_ablation.py`

### `Modeling/scripts/m03_error_followup.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Minimum M04 follow-up explaining maximum-hour versus rise errors; no fitting.

주요 함수: `main()`.

In [45]:
%%submission_source Modeling/scripts/m03_error_followup.py
"""Minimum M04 follow-up explaining maximum-hour versus rise errors; no fitting."""
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path
import pandas as pd
from m01_prepare import ROOT, sha
from m02_compare import metric
from m03_verify import rows, verify_metric


def main():
    tables = ROOT / "Modeling/tables/m03/c"
    out = ROOT / "Modeling/tables/m03/diagnostics"
    out.mkdir(parents=True, exist_ok=True)
    run = json.loads((tables / "run.json").read_text(encoding="utf-8"))
    v = json.loads((tables / "independent_verification.json").read_text(encoding="utf-8"))
    assert v["status"] == "passed" and v["run_sha256"] == sha(tables / "run.json")
    for name in ["predictions.csv", "paired_predictions.csv"]:
        assert sha(tables / name) == run["outputs_sha256"][name]
    p = pd.read_csv(tables / "predictions.csv", encoding="utf-8-sig", float_precision="round_trip")
    p = p.loc[p.target.eq("target_maximum") & p.group.isin(["A", "B"])].copy()
    p["prior_state"] = p.power_transition.str.split("->").str[0]
    prior, peak = [], []
    for group, g in p.groupby("group"):
        for state, part in g.groupby("prior_state"):
            prior.append({"group": group, "prior_state": state, **metric(part)})
        maximum = g.loc[g.daily_maximum_weight.gt(0)]
        for transition, part in maximum.groupby("power_transition"):
            peak.append({"group": group, "power_transition": transition, **metric(part, part.daily_maximum_weight)})
    pd.DataFrame(prior).to_csv(out / "prior_state_errors.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(peak).to_csv(out / "maximum_transition_errors.csv", index=False, encoding="utf-8-sig")
    paired = pd.read_csv(tables / "paired_predictions.csv", encoding="utf-8-sig", float_precision="round_trip")
    rise = paired.loc[paired.target.eq("target_maximum") & paired.before.eq("A") & paired.after.eq("B") & paired.power_transition.eq("low->above26")].copy()
    assert len(rise) == 20 and rise.timestamp.nunique() == 20
    cols = ["timestamp", "date", "split", "actual_before", "prediction_before", "prediction_after", "under_before", "under_after", "delta_abs", "daily_maximum_weight"]
    rise[cols].to_csv(out / "rise_cases.csv", index=False, encoding="utf-8-sig")
    summary = {"status": "completed", "finished": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
               "no_fitting": True, "question": "Why daily maximum underprediction improved while low->above26 did not; examine overlap and observable prior-state bias.",
               "rise_hours": len(rise), "rise_dates": rise.date.nunique(),
               "rise_maximum_hours": int(rise.daily_maximum_weight.gt(0).sum()),
               "rise_maximum_dates": int(rise.loc[rise.daily_maximum_weight.gt(0), "date"].nunique()),
               "rise_maximum_weight": float(rise.daily_maximum_weight.sum()),
               "maximum_total_hours": int(p.loc[p.group.eq("B") & p.daily_maximum_weight.gt(0)].shape[0]),
               "maximum_total_date_weight": float(p.loc[p.group.eq("B"), "daily_maximum_weight"].sum()),
               "rise_by_month": rise.groupby("split").size().to_dict(),
               "rise_B_underpredicted_hours": int(rise.under_after.gt(0).sum()),
               "input_sha256": {name: sha(tables / name) for name in ["predictions.csv", "paired_predictions.csv"]},
               "outputs_sha256": {x.name: sha(x) for x in out.glob("*.csv")}, "script_sha256": sha(Path(__file__))}
    assert summary["maximum_total_date_weight"] == 91
    # Independent direct differences check against saved columns, no model call.
    for r in rise.itertuples():
        assert abs(r.under_after - max(r.actual_before-r.prediction_after, 0)) < 1e-9
        assert abs(r.delta_abs - (abs(r.prediction_after-r.actual_before)-abs(r.prediction_before-r.actual_before))) < 1e-9
    summary["rise_cases_arithmetic_verified"] = True
    csv_predictions = rows(tables / "predictions.csv")
    independent_checks = 0
    for row in rows(out / "prior_state_errors.csv"):
        subset = [r for r in csv_predictions if r["target"] == "target_maximum" and r["group"] == row["group"]
                  and r["power_transition"].split("->")[0] == row["prior_state"]]
        verify_metric(row, subset)
        independent_checks += 1
    for row in rows(out / "maximum_transition_errors.csv"):
        subset = [r for r in csv_predictions if r["target"] == "target_maximum" and r["group"] == row["group"]
                  and r["power_transition"] == row["power_transition"] and float(r["daily_maximum_weight"]) > 0]
        verify_metric(row, subset, "daily_maximum")
        independent_checks += 1
    summary["independently_recomputed_metric_rows"] = independent_checks
    (out / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False), flush=True)
    print(pd.DataFrame(prior)[["group", "prior_state", "hours", "MAE", "mean_under", "mean_over"]].to_string(index=False), flush=True)
    print(pd.DataFrame(peak)[["group", "power_transition", "hours", "weight_sum", "MAE", "mean_under", "mean_over"]].to_string(index=False), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m03_error_followup.py`

### `Modeling/scripts/m03_verify.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Independent standard-library checks of M03 inputs, metrics and paired deltas.

주요 함수: `sha()`, `rows()`, `close()`, `select()`, `weights()`, `verify_metric()`, `main()`.

In [46]:
%%submission_source Modeling/scripts/m03_verify.py
"""Independent standard-library checks of M03 inputs, metrics and paired deltas."""
import argparse
import csv
import hashlib
import json
import math
import sys
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def rows(path):
    with path.open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def close(a, b):
    assert math.isclose(float(a), float(b), rel_tol=1e-11, abs_tol=1e-9), (a, b)


def select(g, kind, value):
    if kind in ("all", "profile_reweighted"):
        return g
    if kind == "daily_maximum":
        return [r for r in g if float(r["daily_maximum_weight"]) > 0]
    return [r for r in g if r[kind] == value]


def weights(g, kind):
    column = {"daily_maximum": "daily_maximum_weight", "profile_reweighted": "profile_weight"}.get(kind)
    return [float(r[column]) if column else 1.0 for r in g]


def verify_metric(row, g, kind="all"):
    assert len(g) == int(row["hours"])
    w = weights(g, kind)
    total = math.fsum(w)
    close(row["weight_sum"], total)
    errors = [float(r["prediction"]) - float(r["actual"]) for r in g]
    values = {"MAE": [abs(e) for e in errors], "bias": errors,
              "mean_under": [max(-e, 0) for e in errors], "mean_over": [max(e, 0) for e in errors],
              "under_fraction": [float(e < 0) for e in errors], "over_fraction": [float(e > 0) for e in errors],
              "mean_actual": [float(r["actual"]) for r in g]}
    for name, numbers in values.items():
        close(row[name], math.fsum(x * weight for x, weight in zip(numbers, w)) / total)
    close(row["RMSE"], math.sqrt(math.fsum(e * e * weight for e, weight in zip(errors, w)) / total))
    close(row["absolute_error_sum"], math.fsum(abs(e) for e in errors))
    assert int(row["negative_prediction_hours"]) == sum(float(r["prediction"]) < 0 for r in g)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--stage", choices=["ab", "c"], required=True)
    stage = parser.parse_args().stage
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    out = ROOT / f"Modeling/tables/m03/{stage}"
    run = json.loads((out / "run.json").read_text(encoding="utf-8"))
    cpath = ROOT / f"Modeling/config/m03_{stage}_contract.json"
    c = json.loads(cpath.read_text(encoding="utf-8"))
    m1 = json.loads((ROOT / "Modeling/config/m01_contract.json").read_text(encoding="utf-8"))
    assert run["status"] == "completed" and not run["july_august_evaluated"]
    assert sha(cpath) == run["input_hashes"]["contract"]
    assert sha(ROOT / "Modeling/config/m01_contract.json") == run["input_hashes"]["m01_contract"]
    assert sha(ROOT / m1["source"]) == run["input_hashes"]["source"] == m1["source_sha256"]
    for name, digest in run["outputs_sha256"].items():
        assert sha(out / name) == digest
    for name, digest in run["input_hashes"].items():
        if name not in ("source", "contract", "m01_contract"):
            assert sha(ROOT / name) == digest
    raw = {}
    for r in rows(ROOT / m1["source"]):
        date = int(r["날짜"])
        hour = float(r["시간"])
        if 20210101 <= date <= 20210630 and hour.is_integer() and 0 <= hour <= 23:
            ts = datetime.strptime(str(date), "%Y%m%d") + timedelta(hours=int(hour))
            slots = [float(r[k]) for k in ["15분", "30분", "45분", "60분"]]
            raw[ts] = {"last": slots[-1], "mean": math.fsum(slots) / 4, "maximum": max(slots)}
    frame = {datetime.fromisoformat(r["timestamp"]): r for r in rows(ROOT / "Modeling/tables/m01/hourly_frame.csv")}
    lag_checks = 0
    for ts, r in frame.items():
        if ts >= datetime(2021, 7, 1) or r["eligible_common"] != "True":
            continue
        for lag, names in [(1, ["last"]), (24, ["mean", "maximum"]), (168, ["mean", "maximum"])]:
            previous = ts - timedelta(hours=lag)
            assert previous in raw and previous < ts
            for name in names:
                close(r[f"lag{lag}_{name}"], raw[previous][name])
                lag_checks += 1
    p = rows(out / "predictions.csv")
    assert len(p) == len(c["groups"]) * len(c["targets"]) * 2184
    keyed = {}
    grouped = defaultdict(list)
    daily = defaultdict(list)
    for r in p:
        key = (r["target"], r["group"], r["split"], r["timestamp"])
        assert key not in keyed
        keyed[key] = r
        ts = datetime.fromisoformat(r["timestamp"])
        assert datetime(2021, 4, 1) <= ts < datetime(2021, 7, 1)
        close(r["actual"], raw[ts]["maximum" if r["target"] == "target_maximum" else "mean"])
        e = float(r["prediction"]) - float(r["actual"])
        assert math.isfinite(e)
        for name, value in {"signed_error": e, "absolute_error": abs(e), "under_amount": max(-e, 0), "over_amount": max(e, 0)}.items():
            close(r[name], value)
        grouped[(r["target"], r["group"])].append(r)
        daily[(r["target"], r["group"], r["date"])].append(r)
    expected_keys = {(r["split"], r["timestamp"]) for r in p if r["target"] == "target_maximum" and r["group"] == "A"}
    assert len(expected_keys) == 2184
    for target in c["targets"]:
        for group in c["groups"]:
            g = grouped[(target, group)]
            assert {(r["split"], r["timestamp"]) for r in g} == expected_keys and len(g) == 2184
            assert {s: sum(r["split"] == s for r in g) for s in c["splits"]} == {"dev_apr": 720, "dev_may": 744, "dev_jun": 720}
            date_weights = defaultdict(float)
            for r in g:
                date_weights[r["date"]] += float(r["daily_maximum_weight"])
            assert len(date_weights) == 91
            for date, weight in date_weights.items():
                close(weight, 1)
                day = daily[(target, group, date)]
                assert len(day) == 24
                maximum = max(float(frame[datetime.fromisoformat(r["timestamp"])]["target_maximum"]) for r in day)
                ties = [r for r in day if float(frame[datetime.fromisoformat(r["timestamp"])]["target_maximum"]) == maximum]
                for r in day:
                    close(r["daily_maximum_weight"], 1 / len(ties) if r in ties else 0)
    score_checks = 0
    for table in ["comparison", "condition_errors", "daily_errors"]:
        for r in rows(out / f"{table}.csv"):
            g = grouped[(r["target"], r["group"])]
            if table == "comparison":
                kind = "all" if r["split"] == "pooled_development" else "split"
                g = select(g, kind, r["split"])
            elif table == "condition_errors":
                kind = r["condition"]
                g = select(g, kind, r["value"])
                denominator = math.fsum(float(x["absolute_error"]) for x in grouped[(r["target"], r["group"])])
                close(r["share_total_absolute_error_unweighted"], math.fsum(float(x["absolute_error"]) for x in g) / denominator)
            else:
                kind = "all"
                g = daily[(r["target"], r["group"], r["date"])]
            verify_metric(r, g, kind)
            score_checks += 1
    pairs = rows(out / "paired_predictions.csv")
    pairgroups = defaultdict(list)
    pairdays = defaultdict(list)
    assert len(pairs) == len(c["comparisons"]) * 2 * 2184
    for r in pairs:
        for label in ["before", "after"]:
            ref = keyed[(r["target"], r[label], r["split"], r["timestamp"])]
            close(r[f"prediction_{label}"], ref["prediction"])
            close(r[f"actual_{label}"], ref["actual"])
            for k, source in [("abs", "absolute_error"), ("under", "under_amount"), ("over", "over_amount")]:
                close(r[f"{k}_{label}"], ref[source])
        for k in ["abs", "under", "over"]:
            close(r[f"delta_{k}"], float(r[f"{k}_after"]) - float(r[f"{k}_before"]))
        pairgroups[(r["target"], r["before"], r["after"])].append(r)
        pairdays[(r["target"], r["before"], r["after"], r["date"])].append(r)
    for table in ["paired_deltas", "daily_deltas"]:
        for r in rows(out / f"{table}.csv"):
            if table == "paired_deltas":
                g = select(pairgroups[(r["target"], r["before"], r["after"])], r["condition"], r["value"])
                w = weights(g, r["condition"])
                close(r["weight_sum"], math.fsum(w))
                assert int(r["improved_hours"]) == sum(float(x["delta_abs"]) < -1e-10 for x in g)
                assert int(r["worsened_hours"]) == sum(float(x["delta_abs"]) > 1e-10 for x in g)
            else:
                g = pairdays[(r["target"], r["before"], r["after"], r["date"])]
                w = [1.0] * len(g)
            assert int(r["hours"]) == len(g)
            for k in ["abs", "under", "over"]:
                close(r[f"delta_{k}"], math.fsum(float(x[f"delta_{k}"]) * weight for x, weight in zip(g, w)) / math.fsum(w))
    for r in rows(out / "model_manifest.csv"):
        assert sha(ROOT / r["file"]) == r["sha256"]
        assert r["reload_predictions_checked"] == "True"
    audits = rows(out / "training_audit.csv")
    for r in audits:
        assert datetime.fromisoformat(r["train_end"]) < datetime.fromisoformat(r["eval_start"])
        assert int(r["features"]) == len(run["features"][r["group"]])
    result = {"status": "passed", "verified_at": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
              "stage": stage, "prediction_rows": len(p), "paired_rows": len(pairs), "metric_rows": score_checks,
              "raw_source_lag_checks": lag_checks, "csv_hashes_checked": len(run["outputs_sha256"]),
              "models_checked": len(audits), "july_august_evaluated": False,
              "identical_evaluation_timestamps": len(expected_keys), "daily_maximum_dates": 91,
              "run_sha256": sha(out / "run.json"), "verifier_sha256": sha(Path(__file__))}
    (out / "independent_verification.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result, ensure_ascii=False), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m03_verify.py`

### `Modeling/scripts/m04_compare.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: M04: one predeclared observable-state rule and paired error diagnosis; no fitting.

주요 함수: `now()`, `read_csv()`, `save()`, `subsets()`, `paired()`, `main()`.

In [47]:
%%submission_source Modeling/scripts/m04_compare.py
"""M04: one predeclared observable-state rule and paired error diagnosis; no fitting."""
import json
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from m01_prepare import ROOT, sha
from m02_compare import metric, state

CONTRACT = ROOT / "Modeling/config/m04_contract.json"
OUT = ROOT / "Modeling/tables/m04"


def now():
    return datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes")


def read_csv(path):
    return pd.read_csv(path, encoding="utf-8-sig", float_precision="round_trip")


def save(frame, name):
    frame.to_csv(OUT / name, index=False, encoding="utf-8-sig")


def subsets(g):
    yield "all", "all", g, None
    peak = g.loc[g.daily_maximum_weight.gt(0)]
    yield "daily_maximum", "all", peak, peak.daily_maximum_weight
    yield "profile_reweighted", "all", g, g.profile_weight
    for column in ["split", "prior_state", "power_transition", "diag_production_transition",
                   "train_profile_overlap", "weekend", "hour"]:
        for value, part in g.groupby(column):
            yield column, str(value), part, None
    for column in ["split", "power_transition"]:
        for value, part in peak.groupby(column):
            yield "peak_" + column, str(value), part, part.daily_maximum_weight


def paired(before, after, name):
    fields = ["timestamp", "date", "split", "power_transition", "prior_state",
              "daily_maximum_weight", "train_profile_overlap"]
    q = before[fields + ["actual", "prediction", "absolute_error", "under_amount", "over_amount"]].merge(
        after[["timestamp", "actual", "prediction", "absolute_error", "under_amount", "over_amount"]],
        on="timestamp", suffixes=("_before", "_after"), validate="one_to_one")
    assert np.array_equal(q.actual_before, q.actual_after)
    q["comparison"] = name
    for original, label in [("absolute_error", "abs"), ("under_amount", "under"), ("over_amount", "over")]:
        q["delta_" + label] = q[original + "_after"] - q[original + "_before"]
    return q


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    assert not (OUT / "run.json").exists(), "Preserve completed M04; version a new comparison instead."
    c = json.loads(CONTRACT.read_text(encoding="utf-8"))
    for key in ["parent_run", "parent_predictions", "frame"]:
        assert sha(ROOT / c[key]) == c[key + "_sha256"], key
    parent = json.loads((ROOT / c["parent_run"]).read_text(encoding="utf-8"))
    verified = json.loads((ROOT / "Modeling/tables/m03/c/independent_verification.json").read_text(encoding="utf-8"))
    assert verified["status"] == "passed" and verified["run_sha256"] == sha(ROOT / c["parent_run"])
    assert parent["outputs_sha256"]["predictions.csv"] == c["parent_predictions_sha256"]
    m1 = json.loads((ROOT / "Modeling/tables/m01/verification.json").read_text(encoding="utf-8"))
    assert m1["status"] == "passed" and m1["outputs_sha256"]["hourly_frame.csv"] == c["frame_sha256"]
    p = read_csv(ROOT / c["parent_predictions"])
    p = p.loc[p.target.eq(c["target"]) & p.group.isin(["A", "B"])].copy()
    f = read_csv(ROOT / c["frame"])
    features = [x for x in c["origin_input_diagnostics"] if x not in ["hour", "lag1_last_minus_mean", "lag1_last_minus_lag2_last"]]
    f = f[["timestamp", "lag1_timestamp"] + features]
    p = p.merge(f, on="timestamp", validate="many_to_one")
    p["prior_state"] = state(p.lag1_maximum, np.floor(p.lag1_mean + 0.5))
    assert (p.prior_state == p.power_transition.str.split("->").str[0]).all()
    p["lag1_last_minus_mean"] = p.lag1_last - p.lag1_mean
    p["lag1_last_minus_lag2_last"] = p.lag1_last - p.lag2_last
    p["rule_applied"] = False
    a, b = [p.loc[p.group.eq(group)].sort_values("timestamp").copy() for group in ["A", "B"]]
    assert len(b) == 2184 and b.timestamp.nunique() == 2184 and b.date.nunique() == 91
    assert set(b.split) == {"dev_apr", "dev_may", "dev_jun"}
    rule = b.copy()
    rule["group"] = c["rule"]["name"]
    rule["rule_applied"] = rule.prior_state.eq("low")
    rule["prediction"] = np.where(rule.rule_applied, rule.lag1_maximum, rule.prediction)
    baseline = b.copy()
    baseline["group"] = "lag1"
    baseline["prediction"] = baseline.lag1_maximum
    p = pd.concat([a, b, rule, baseline], ignore_index=True)
    p["signed_error"] = p.prediction - p.actual
    p["absolute_error"] = p.signed_error.abs()
    p["under_amount"] = (-p.signed_error).clip(lower=0)
    p["over_amount"] = p.signed_error.clip(lower=0)
    # All paired and case outputs must use the recalculated rule errors.
    a, b, rule = [p.loc[p.group.eq(group)].copy() for group in ["A", "B", c["rule"]["name"]]]
    OUT.mkdir(parents=True, exist_ok=True)
    save(p, "predictions.csv")
    metrics = []
    for group, g in p.groupby("group"):
        total_error = g.absolute_error.sum()
        for kind, value, part, weights in subsets(g):
            metrics.append({"group": group, "kind": kind, "value": value,
                            **metric(part, weights),
                            "absolute_error_share": float(part.absolute_error.sum() / total_error)})
    m = pd.DataFrame(metrics)
    save(m, "metrics.csv")
    pairs = pd.concat([paired(a, b, "A_to_B"), paired(b, rule, "B_to_rule")], ignore_index=True)
    save(pairs, "paired_predictions.csv")
    q = pairs.loc[pairs.comparison.eq("B_to_rule")]
    daily = q.groupby(["split", "date"], as_index=False).agg(
        hours=("timestamp", "size"), MAE_B=("absolute_error_before", "mean"),
        MAE_rule=("absolute_error_after", "mean"), delta_MAE=("delta_abs", "mean"),
        delta_under=("delta_under", "mean"), delta_over=("delta_over", "mean"))
    save(daily, "daily_errors.csv")
    low = b.loc[b.prior_state.eq("low")].merge(
        rule[["timestamp", "prediction", "absolute_error", "under_amount", "over_amount"]],
        on="timestamp", suffixes=("_B", "_rule"), validate="one_to_one")
    save(low, "prior_low_cases.csv")
    summary = []
    for transition, part in low.groupby("power_transition"):
        for feature in c["origin_input_diagnostics"]:
            values = part[feature]
            summary.append({"power_transition": transition, "feature": feature, "hours": len(part),
                            "mean": values.mean(), "min": values.min(), "q25": values.quantile(.25),
                            "median": values.median(), "q75": values.quantile(.75), "max": values.max()})
    save(pd.DataFrame(summary), "origin_feature_summary.csv")
    counts = low.groupby(["split", "hour", "power_transition"], as_index=False).size()
    save(counts, "origin_hour_counts.csv")
    # Complete observed B input signatures; posthoc collisions cannot be deployed as a rule.
    low["day_of_week"] = pd.to_datetime(low.timestamp).dt.dayofweek
    signatures = ["month", "day_of_week", "weekend", "hour", "lag1_production", "lag1_mean", "lag1_maximum",
                  "lag2_mean", "lag2_maximum", "lag1_last"]
    low["signature"] = low[signatures].astype(str).agg("|".join, axis=1)
    collisions = low.groupby("signature").agg(hours=("timestamp", "size"), dates=("date", "nunique"),
        actual_min=("actual", "min"), actual_max=("actual", "max"), transitions=("power_transition", "nunique"))
    collisions = collisions.loc[collisions.transitions.gt(1)].reset_index()
    save(collisions, "origin_signature_collisions.csv")
    def cell(group, kind, value="all"):
        r = m.loc[m.group.eq(group) & m.kind.eq(kind) & m.value.eq(value)]
        assert len(r) == 1
        return r.iloc[0]
    tol = c["selection"]["numeric_tolerance"]
    rule_name = c["rule"]["name"]
    gates = []
    for kind, value, field, strict in [("all", "all", "MAE", True),
            ("daily_maximum", "all", "MAE", False), ("daily_maximum", "all", "mean_under", False),
            ("power_transition", "low->above26", "mean_under", False)]:
        ref, candidate = float(cell("B", kind, value)[field]), float(cell(rule_name, kind, value)[field])
        delta = candidate - ref
        gates.append({"kind": kind, "value": value, "metric": field, "B": ref, "rule": candidate,
                      "delta": delta, "strict_decrease": strict, "passed": delta < -tol if strict else delta <= tol})
    accepted = all(g["passed"] for g in gates)
    selection = {"status": "accepted" if accepted else "rejected", "selected": rule_name if accepted else "B",
        "gates": gates, "rule_applied_hours": int(rule.rule_applied.sum()),
        "unchanged_hours": int((~rule.rule_applied).sum()),
        "days_improved": int(daily.delta_MAE.lt(-tol).sum()), "days_worsened": int(daily.delta_MAE.gt(tol).sum()),
        "days_unchanged": int(daily.delta_MAE.abs().le(tol).sum()),
        "origin_signature_collisions": len(collisions),
        "note": "A smaller aggregate error does not override a failed predeclared safety-of-accuracy gate; no operational safety claim."}
    (OUT / "selection.json").write_text(json.dumps(selection, ensure_ascii=False, indent=2), encoding="utf-8")
    run = {"status": "completed", "finished": now(), "no_fitting": True, "no_july_august_evaluation": True,
        "contract_sha256": sha(CONTRACT), "script_sha256": sha(Path(__file__)),
        "inputs_sha256": {c[key]: sha(ROOT / c[key]) for key in ["parent_run", "parent_predictions", "frame"]},
        "outputs_sha256": {x.name: sha(x) for x in sorted(OUT.iterdir()) if x.suffix in [".csv", ".json"] and x.name != "run.json"},
        "prediction_rows": len(p), "paired_rows": len(pairs), "metric_rows": len(m),
        "unique_evaluation_hours": len(b), "daily_maximum_hours": int(b.daily_maximum_weight.gt(0).sum()),
        "daily_maximum_date_weight": float(b.daily_maximum_weight.sum()),
        "runtime": {"executable": sys.executable, "version": sys.version}}
    (OUT / "run.json").write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(selection, ensure_ascii=False), flush=True)
    print(m.loc[m.kind.isin(["all", "daily_maximum"]) | (m.kind.eq("power_transition") & m.value.isin(["low->low", "low->above26"]))][
        ["group", "kind", "value", "hours", "MAE", "RMSE", "mean_under", "mean_over"]].to_string(index=False), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m04_compare.py`

### `Modeling/scripts/m04_rise_compare.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: M04 rise development: dynamics, weighted HGB, probability-gated expert; chronological Optuna.

주요 함수: `now()`, `read_csv()`, `save()`, `build_frame()`, `stats()`, `score()`, `classifier()`, `fit_probability()`, `probability()`, `alarm_stats()`, `gated()`, `main()`.

In [48]:
%%submission_source Modeling/scripts/m04_rise_compare.py
"""M04 rise development: dynamics, weighted HGB, probability-gated expert; chronological Optuna."""
import os
os.environ["OMP_NUM_THREADS"] = "1"
import json
import sys
import time
import warnings
from datetime import datetime, timedelta, timezone
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from m01_prepare import ROOT, SLOTS, feature_columns, read_contract, sha
from m02_compare import metric, state
from m02_rerun import peak_weights
from m04_compare import subsets

OUT = ROOT / "Modeling/tables/m04_rise"
MODELS = ROOT / "Modeling/models/m04_rise"
CONTRACT = ROOT / "Modeling/config/m04_rise_contract.json"


def now():
    return datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes")


def read_csv(path, dates=None):
    return pd.read_csv(path, encoding="utf-8-sig", float_precision="round_trip", parse_dates=dates)


def save(frame, name):
    frame.to_csv(OUT / name, index=False, encoding="utf-8-sig")


def build_frame(c):
    parent = read_contract()
    source = ROOT / parent["source"]
    frame_path = ROOT / "Modeling/tables/m01/hourly_frame.csv"
    assert sha(source) == c["source_sha256"] == parent["source_sha256"]
    assert sha(frame_path) == c["frame_sha256"]
    f = read_csv(frame_path, ["timestamp", "date"])
    f = f.loc[f.timestamp.lt("2021-07-01")].copy()
    raw = read_csv(source)
    raw = raw.loc[raw["날짜"].between(20210101, 20210630) & raw["시간"].between(0, 23)].copy()
    raw["timestamp"] = pd.to_datetime(raw["날짜"].astype(str), format="%Y%m%d") + pd.to_timedelta(raw["시간"], unit="h")
    raw = raw.set_index("timestamp")
    assert raw.index.is_unique
    slots = raw[SLOTS].reindex(pd.DatetimeIndex(f.timestamp - pd.Timedelta(hours=1))).to_numpy()
    f["prev_slot_rise"] = slots[:, 3] - slots[:, 0]
    f["prev_slot_range"] = np.max(slots, axis=1) - np.min(slots, axis=1)
    f["prev_slot_slope"] = slots @ np.array([-1.5, -.5, .5, 1.5]) / 5
    f["last_delta"] = f.lag1_last - f.lag2_last
    f["mean_delta"] = f.lag1_mean - f.lag2_mean
    f["maximum_delta"] = f.lag1_maximum - f.lag2_maximum
    f["production_delta"] = f.lag1_production - f.lag2_production
    f["prior_state"] = state(f.lag1_maximum, np.floor(f.lag1_mean + .5))
    f["power_state"] = state(f.target_maximum, f.diag_target_rounded_mean)
    f["power_transition"] = f.prior_state + "->" + f.power_state
    f["prior_low"] = f.prior_state.eq("low")
    f["rise_event"] = f.power_transition.eq("low->above26")
    full_state = dict(zip(f.timestamp, f.power_state))
    durations, gaps = [], []
    for timestamp in f.timestamp:
        duration, gap = 0, 0
        for lag in range(1, 7):
            previous = full_state.get(timestamp - pd.Timedelta(hours=lag))
            if previous is None:
                gap = 1
                break
            if previous != "low":
                break
            duration += 1
        durations.append(duration)
        gaps.append(gap)
    f["low_run_length6"], f["low_history_gap6"] = durations, gaps
    f = f.loc[f.eligible_common].copy().reset_index(drop=True)
    f["hour"] = f.timestamp.dt.hour
    base = feature_columns("B", parent)
    dynamic = base + c["features"]["add"]
    assert len(base) == 18 and len(dynamic) == 27 and f[dynamic].notna().all().all()
    assert not any(x.startswith(("target_", "diag_")) for x in dynamic)
    return f, base, dynamic


def stats(frame, prediction):
    e = np.asarray(prediction) - frame.target_maximum.to_numpy()
    low = frame.prior_low.to_numpy()
    event = frame.rise_event.to_numpy()
    stay = frame.power_transition.eq("low->low").to_numpy()
    peak = peak_weights(frame)
    assert event.any() and stay.any() and peak.sum() > 0
    return {"overall_MAE": float(np.abs(e).mean()), "daily_maximum_MAE": float(np.average(np.abs(e), weights=peak)),
        "rise_MAE": float(np.abs(e[event]).mean()), "rise_under": float(np.maximum(-e[event], 0).mean()),
        "low_stay_MAE": float(np.abs(e[stay]).mean()), "low_stay_over": float(np.maximum(e[stay], 0).mean()),
        "prior_low_hours": int(low.sum()), "rise_hours": int(event.sum())}


def score(values, reference, limits, false_positive_fraction=None):
    ratios = {key: values[key] / reference[key] for key in ["overall_MAE", "daily_maximum_MAE", "rise_MAE", "low_stay_MAE", "low_stay_over"]}
    penalty = sum(max(ratios[key] / limit - 1, 0) for key, limit in [
        ("overall_MAE", limits["overall_MAE_ratio_max"]), ("daily_maximum_MAE", limits["daily_maximum_MAE_ratio_max"]),
        ("low_stay_MAE", limits["low_stay_MAE_ratio_max"]), ("low_stay_over", limits["low_stay_mean_over_ratio_max"])])
    if false_positive_fraction is not None:
        penalty += max(false_positive_fraction / limits["gate_false_positive_fraction_max"] - 1, 0)
    return max(ratios[key] for key in ["overall_MAE", "daily_maximum_MAE", "rise_MAE"]) + 10 * penalty


def classifier(family, params):
    if family == "Logistic":
        return make_pipeline(StandardScaler(), LogisticRegression(C=params["C"], max_iter=2000, random_state=42))
    return HistGradientBoostingClassifier(max_leaf_nodes=params["max_leaf_nodes"], min_samples_leaf=params["min_samples_leaf"],
        l2_regularization=params["l2_regularization"], max_iter=150, learning_rate=.05, early_stopping=False, random_state=42)


def fit_probability(family, params, tr, ev, features):
    low = tr.loc[tr.prior_low]
    assert len(low) > 0
    if low.rise_event.nunique() < 2:
        probability = np.full(len(ev), float(low.rise_event.mean()))
        return {"constant_probability": float(low.rise_event.mean())}, probability, True
    estimator = classifier(family, params)
    estimator.fit(low[features], low.rise_event.astype(int))
    probability = estimator.predict_proba(ev[features])[:, 1]
    return estimator, probability, False


def probability(estimator, ev, features):
    if isinstance(estimator, dict):
        return np.full(len(ev), estimator["constant_probability"])
    return estimator.predict_proba(ev[features])[:, 1]


def alarm_stats(frame, probability_values, threshold):
    low = frame.prior_low.to_numpy()
    y = frame.rise_event.to_numpy()[low]
    p = np.asarray(probability_values)[low]
    alarm = p >= threshold
    tp, fp, fn, tn = [int(x.sum()) for x in [alarm & y, alarm & ~y, ~alarm & y, ~alarm & ~y]]
    return {"prior_low_hours": len(y), "positive_hours": int(y.sum()), "prevalence": float(y.mean()),
        "AP": float(average_precision_score(y, p)), "AUROC": float(roc_auc_score(y, p)),
        "Brier": float(brier_score_loss(y, p)), "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": tp / (tp + fp) if tp + fp else 0., "recall": tp / (tp + fn),
        "false_positive_fraction": fp / (fp + tn), "threshold": threshold}


def gated(base, expert, p, low, threshold, blend):
    trigger = np.asarray(low) & (np.asarray(p) >= threshold)
    return np.asarray(base) + trigger * blend * (np.asarray(expert) - np.asarray(base)), trigger


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    assert not (OUT / "run.json").exists(), "Preserve previous run; create a versioned comparison."
    OUT.mkdir(parents=True, exist_ok=True)
    MODELS.mkdir(parents=True, exist_ok=True)
    c = json.loads(CONTRACT.read_text(encoding="utf-8"))
    parent = read_contract()
    old_path = ROOT / "Modeling/tables/m03/c/predictions.csv"
    assert sha(old_path) == c["parent_predictions_sha256"]
    old_verify = json.loads((ROOT / "Modeling/tables/m03/c/independent_verification.json").read_text(encoding="utf-8"))
    assert old_verify["status"] == "passed" and old_verify["run_sha256"] == sha(ROOT / "Modeling/tables/m03/c/run.json")
    old = read_csv(old_path, ["timestamp", "date"])
    old = old.loc[old.target.eq(c["target"]) & old.group.eq("B")].copy()
    old_models = read_csv(ROOT / "Modeling/tables/m03/ab/model_manifest.csv")
    f, base_features, features = build_frame(c)
    save(f[["timestamp", "prior_low", "rise_event", "power_transition"] + features], "feature_frame.csv")
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    started = time.perf_counter()
    run = {"status": "running", "started": now(), "contract_sha256": sha(CONTRACT),
        "inputs_sha256": {"Modeling/tables/m01/hourly_frame.csv": c["frame_sha256"], parent["source"]: c["source_sha256"],
                          "Modeling/tables/m03/c/predictions.csv": c["parent_predictions_sha256"]},
        "features": {"B": base_features, "dynamic": features}, "no_july_august_evaluation": True,
        "runtime": {"executable": sys.executable, "version": sys.version}, "script_sha256": sha(Path(__file__))}
    (OUT / "run.json").write_text(json.dumps(run, indent=2), encoding="utf-8")
    trials, audits, manifests, configs, predictions, inner_predictions, classification, counts = [], [], [], [], [], [], [], []
    fit_counter = {"inner_regression": 0, "inner_classifier": 0, "outer": 0}
    warning_messages = []
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        for outer in parent["splits"]:
            split = outer["name"]
            if split not in c["outer_splits"]:
                continue
            tr = f.loc[f.timestamp.le(outer["train_end"])].copy()
            ev = f.loc[f.timestamp.between(outer["eval_start"], outer["eval_end"])].copy()
            template = old.loc[old.split.eq(split)].sort_values("timestamp").copy()
            assert template.timestamp.tolist() == ev.timestamp.tolist() and np.array_equal(template.actual, ev.target_maximum)
            for role, part in [("train", tr), ("eval", ev)]:
                event = part.loc[part.rise_event]
                counts.append({"split": split, "role": role, "hours": len(part), "prior_low_hours": int(part.prior_low.sum()),
                    "rise_hours": len(event), "rise_dates": event.date.nunique()})
            folds, inner_parts, inner_b, inner_d = [], [], [], []
            for month in c["inner_months"][split]:
                start = pd.Timestamp(2021, month, 1)
                t = tr.loc[tr.timestamp.lt(start)]
                v = tr.loc[tr.timestamp.between(start, start + pd.offsets.MonthBegin(1) - pd.Timedelta(hours=1))]
                assert t.timestamp.max() < v.timestamp.min() <= v.timestamp.max() < ev.timestamp.min()
                folds.append((t, v))
                audits.append({"split": split, "inner_month": month, "train_hours": len(t), "validation_hours": len(v),
                    "train_last": str(t.timestamp.max()), "validation_first": str(v.timestamp.min()), "validation_last": str(v.timestamp.max()),
                    "outer_first": str(ev.timestamp.min()), "train_low": int(t.prior_low.sum()), "train_rises": int(t.rise_event.sum()),
                    "validation_low": int(v.prior_low.sum()), "validation_rises": int(v.rise_event.sum())})
                for cols, container in [(base_features, inner_b), (features, inner_d)]:
                    estimator = HistGradientBoostingRegressor(**c["hgb_parameters"])
                    estimator.fit(t[cols], t.target_maximum)
                    container.append(estimator.predict(v[cols]))
                    fit_counter["inner_regression"] += 1
                inner_parts.append(v)
            iv = pd.concat(inner_parts)
            ib, idynamic = np.concatenate(inner_b), np.concatenate(inner_d)
            reference = stats(iv, ib)
            selected = {"B": ib, "D_dynamic": idynamic}
            def study(direction="minimize"):
                return optuna.create_study(direction=direction, sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=10),
                                          pruner=optuna.pruners.NopPruner())
            wcache = {}
            def weighted_objective(trial):
                weight = trial.suggest_float("event_weight", *c["weighted"]["weight_range"], log=True)
                parts = []
                for t, v in folds:
                    estimator = HistGradientBoostingRegressor(**c["hgb_parameters"])
                    estimator.fit(t[features], t.target_maximum, sample_weight=np.where(t.rise_event, weight, 1.))
                    parts.append(estimator.predict(v[features]))
                    fit_counter["inner_regression"] += 1
                result = np.concatenate(parts)
                values = stats(iv, result)
                trial.set_user_attr("metrics", values)
                wcache[trial.number] = result
                return score(values, reference, c["development_selection"])
            ws = study()
            for weight in c["weighted"]["enqueue"]:
                ws.enqueue_trial({"event_weight": weight})
            ws.optimize(weighted_objective, n_trials=c["weighted"]["trials"])
            wt = min(ws.trials, key=lambda trial: (trial.value, trial.number))
            selected["W_rise_weighted"] = wcache[wt.number]
            split_config = {"split": split, "event_weight": wt.params["event_weight"], "weighted_best_trial": wt.number,
                            "weighted_inner_metrics": wt.user_attrs["metrics"], "classifiers": {}}
            def log_trials(study_object, family):
                for trial in study_object.trials:
                    trials.append({"split": split, "family": family, "trial": trial.number, "state": trial.state.name,
                        "objective": trial.value, "parameters": json.dumps(trial.params),
                        "metrics": json.dumps(trial.user_attrs), "seconds": trial.duration.total_seconds()})
            log_trials(ws, "weighted")
            parent_row = old_models.loc[old_models.split.eq(split) & old_models.target.eq(c["target"]) & old_models.group.eq("B")].iloc[0]
            parent_path = ROOT / parent_row.file
            assert sha(parent_path) == parent_row.sha256
            bmodel = joblib.load(parent_path)
            assert np.allclose(bmodel.predict(ev[base_features]), template.prediction, atol=1e-9, rtol=0)
            outer_predictions = {"B": template.prediction.to_numpy()}
            for name, weight in [("D_dynamic", 1.), ("W_rise_weighted", wt.params["event_weight"])]:
                estimator = HistGradientBoostingRegressor(**c["hgb_parameters"])
                estimator.fit(tr[features], tr.target_maximum, sample_weight=None if weight == 1 else np.where(tr.rise_event, weight, 1.))
                predicted = estimator.predict(ev[features])
                path = MODELS / f"{split}_{name}.joblib"
                joblib.dump(estimator, path, compress=3)
                assert np.allclose(joblib.load(path).predict(ev[features]), predicted, atol=1e-9, rtol=0)
                manifests.append({"split": split, "name": name, "file": path.relative_to(ROOT).as_posix(), "sha256": sha(path), "features": 27})
                outer_predictions[name] = predicted
                fit_counter["outer"] += 1
            for family in ["Logistic", "HGB"]:
                pcache = {}
                def classifier_objective(trial):
                    if family == "Logistic":
                        params = {"C": trial.suggest_float("C", .01, 100., log=True)}
                    else:
                        params = {"max_leaf_nodes": trial.suggest_categorical("max_leaf_nodes", [3,7,15]),
                            "min_samples_leaf": trial.suggest_categorical("min_samples_leaf", [10,20,40]),
                            "l2_regularization": trial.suggest_float("l2_regularization", .1, 10., log=True)}
                    parts, constants = [], 0
                    for t, v in folds:
                        estimator, prob, constant = fit_probability(family, params, t, v, features)
                        parts.append(prob)
                        constants += int(constant)
                        fit_counter["inner_classifier"] += int(not constant)
                    prob = np.concatenate(parts)
                    values = alarm_stats(iv, prob, .5)
                    trial.set_user_attr("classification", values)
                    trial.set_user_attr("constant_folds", constants)
                    pcache[trial.number] = prob
                    return values["AP"]
                cs = study("maximize")
                cs.optimize(classifier_objective, n_trials=c["classifier"]["trials_per_family"])
                ct = min(cs.trials, key=lambda trial: (-trial.value, trial.user_attrs["classification"]["Brier"], trial.number))
                ip = pcache[ct.number]
                def gate_objective(trial):
                    threshold = trial.suggest_float("threshold", *c["gate"]["threshold"])
                    blend = trial.suggest_float("blend", *c["gate"]["blend"])
                    pred, trigger = gated(ib, selected["W_rise_weighted"], ip, iv.prior_low, threshold, blend)
                    values = stats(iv, pred)
                    alarm = alarm_stats(iv, ip, threshold)
                    trial.set_user_attr("metrics", values)
                    trial.set_user_attr("classification", alarm)
                    return score(values, reference, c["development_selection"], alarm["false_positive_fraction"])
                gs = study()
                for params in c["gate"]["enqueue"]:
                    gs.enqueue_trial(params)
                gs.optimize(gate_objective, n_trials=c["gate"]["trials_per_family"])
                gt = min(gs.trials, key=lambda trial: (trial.value, trial.number))
                name = "G_" + family
                selected[name], _ = gated(ib, selected["W_rise_weighted"], ip, iv.prior_low, **gt.params)
                estimator, op, constant = fit_probability(family, ct.params, tr, ev, features)
                path = MODELS / f"{split}_{family}_classifier.joblib"
                joblib.dump(estimator, path, compress=3)
                assert np.allclose(probability(joblib.load(path), ev, features), op, atol=1e-9, rtol=0)
                manifests.append({"split": split, "name": family + "_classifier", "file": path.relative_to(ROOT).as_posix(), "sha256": sha(path), "features": 27})
                fit_counter["outer"] += int(not constant)
                outer_predictions[name], trigger = gated(outer_predictions["B"], outer_predictions["W_rise_weighted"], op, ev.prior_low, **gt.params)
                split_config["classifiers"][family] = {"parameters": ct.params, "classifier_best_trial": ct.number,
                    "classifier_inner": ct.user_attrs["classification"], "gate": gt.params, "gate_best_trial": gt.number,
                    "gate_inner": gt.user_attrs, "constant_outer": constant}
                log_trials(cs, "classifier_" + family)
                log_trials(gs, "gate_" + family)
                for role, part, prob in [("inner", iv, ip), ("outer", ev, op)]:
                    classification.append({"split": split, "role": role, "model": name,
                                           **alarm_stats(part, prob, gt.params["threshold"])})
                selected[name + "_probability"] = ip
                outer_predictions[name + "_probability"] = op
                outer_predictions[name + "_trigger"] = trigger
                print(f"{split} {name}: inner AP={ct.value:.3f}, gate={gt.params}", flush=True)
            for name in c["methods"]:
                record = template.copy()
                record["group"] = name
                record["prediction"] = outer_predictions[name]
                record["prior_low"] = ev.prior_low.to_numpy()
                record["rise_event"] = ev.rise_event.to_numpy()
                record["prior_state"] = ev.prior_state.to_numpy()
                record["probability"] = outer_predictions.get(name + "_probability", np.full(len(ev), np.nan))
                record["trigger"] = outer_predictions.get(name + "_trigger", np.zeros(len(ev), dtype=bool))
                predictions.append(record)
                record_inner = iv[["timestamp", "date", "prior_low", "rise_event", "power_transition"]].copy()
                record_inner["split"], record_inner["group"] = split, name
                record_inner["actual"] = iv.target_maximum.to_numpy()
                record_inner["prediction"] = selected[name]
                record_inner["daily_maximum_weight"] = peak_weights(iv)
                record_inner["probability"] = selected.get(name + "_probability", np.full(len(iv), np.nan))
                inner_predictions.append(record_inner)
            configs.append(split_config)
            save(pd.DataFrame(trials), "trials.csv")
            save(pd.DataFrame(audits), "fold_audit.csv")
            (OUT / "selected_configurations.json").write_text(json.dumps(configs, indent=2), encoding="utf-8")
            print(f"completed {split}: {len(trials)} cumulative trials", flush=True)
        warning_messages = sorted(set(str(w.message) for w in caught))
    p = pd.concat(predictions, ignore_index=True)
    p["signed_error"] = p.prediction - p.actual
    p["absolute_error"] = p.signed_error.abs()
    p["under_amount"] = (-p.signed_error).clip(lower=0)
    p["over_amount"] = p.signed_error.clip(lower=0)
    assert p.timestamp.max() < pd.Timestamp("2021-07-01") and np.isfinite(p.prediction).all()
    save(p, "predictions.csv")
    save(pd.concat(inner_predictions, ignore_index=True), "inner_predictions.csv")
    save(pd.DataFrame(manifests), "model_manifest.csv")
    save(pd.DataFrame(classification), "classification.csv")
    save(pd.DataFrame(counts), "sample_counts.csv")
    metrics, selection_rows, paired, daily = [], [], [], []
    reference_pred = p.loc[p.group.eq("B")]
    reference_frame = f.loc[f.timestamp.isin(reference_pred.timestamp)].sort_values("timestamp")
    reference_stats = stats(reference_frame, reference_pred.sort_values("timestamp").prediction)
    for name, part in p.groupby("group"):
        for kind, value, subset, weights in subsets(part):
            metrics.append({"group": name, "kind": kind, "value": value, **metric(subset, weights)})
        values = stats(reference_frame, part.sort_values("timestamp").prediction)
        row = {"group": name, **values, "balanced_score": max(values[k] / reference_stats[k] for k in ["overall_MAE", "daily_maximum_MAE", "rise_MAE"])}
        requirements = []
        for key, setting in [("overall_MAE", "overall_MAE_ratio_max"), ("daily_maximum_MAE", "daily_maximum_MAE_ratio_max"),
            ("low_stay_MAE", "low_stay_MAE_ratio_max"), ("low_stay_over", "low_stay_mean_over_ratio_max"),
            ("rise_MAE", "rise_MAE_ratio_max"), ("rise_under", "rise_mean_under_ratio_max")]:
            row[key + "_ratio"] = values[key] / reference_stats[key]
            row[key + "_passed"] = bool(row[key + "_ratio"] <= c["development_selection"][setting] + 1e-12)
            requirements.append(row[key + "_passed"])
        month_ok = True
        for split, month_part in part.groupby("split"):
            target_rise = month_part.loc[month_part.rise_event]
            old_rise = reference_pred.loc[reference_pred.split.eq(split) & reference_pred.rise_event]
            month_ok &= target_rise.absolute_error.mean() <= old_rise.absolute_error.mean() * c["development_selection"]["rise_monthly_MAE_ratio_max"] + 1e-9
        row["rise_monthly_passed"] = bool(month_ok)
        requirements.append(bool(month_ok))
        if name.startswith("G_"):
            alarm = alarm_stats(reference_frame, part.sort_values("timestamp").probability, .5)
            # Threshold differs by split; use actual stored trigger for pooled confusion counts.
            low = part.loc[part.prior_low]
            fp = int((low.trigger & ~low.rise_event).sum())
            tn = int((~low.trigger & ~low.rise_event).sum())
            row["gate_false_positive_fraction"] = fp / (fp + tn)
            row["gate_false_positive_passed"] = bool(row["gate_false_positive_fraction"] <= .05 + 1e-12)
            requirements.append(row["gate_false_positive_passed"])
        row["eligible"] = all(requirements) and name != "B"
        selection_rows.append(row)
        q = part[["timestamp", "date", "split", "power_transition", "prediction", "absolute_error", "under_amount", "over_amount"]].merge(
            reference_pred[["timestamp", "prediction", "absolute_error", "under_amount", "over_amount"]], on="timestamp", suffixes=("", "_B"), validate="one_to_one")
        q["group"] = name
        for key, label in [("absolute_error", "abs"), ("under_amount", "under"), ("over_amount", "over")]:
            q["delta_" + label] = q[key] - q[key + "_B"]
        paired.append(q)
        d = q.groupby(["split", "date"], as_index=False).agg(hours=("timestamp", "size"), delta_MAE=("delta_abs", "mean"), delta_under=("delta_under", "mean"), delta_over=("delta_over", "mean"))
        d["group"] = name
        daily.append(d)
    table = pd.DataFrame(selection_rows)
    eligible = table.loc[table.eligible].copy()
    if len(eligible):
        near = eligible.loc[eligible.balanced_score.le(eligible.balanced_score.min() * 1.01)].copy()
        near["preference"] = near.group.map({name: i for i, name in enumerate(c["methods"])})
        chosen = near.sort_values(["preference", "balanced_score"]).iloc[0].group
    else:
        chosen = "B"
    selection = {"selected": chosen, "eligible_methods": eligible.group.tolist(), "development_only": True,
                 "rule": c["development_selection"], "no_july_august_evaluation": True}
    save(pd.DataFrame(metrics), "metrics.csv")
    save(table, "selection_metrics.csv")
    save(pd.concat(paired, ignore_index=True), "paired_predictions.csv")
    save(pd.concat(daily, ignore_index=True), "daily_errors.csv")
    (OUT / "selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
    run.update({"status": "completed", "finished": now(), "seconds": time.perf_counter() - started,
        "fits": fit_counter, "trials": len(trials), "prediction_rows": len(p), "metric_rows": len(metrics),
        "warnings": warning_messages, "outputs_sha256": {path.name: sha(path) for path in OUT.iterdir() if path.name != "run.json"},
        "helper_sha256": {name: sha(ROOT / "Modeling/scripts" / name) for name in ["m01_prepare.py", "m02_compare.py", "m02_rerun.py", "m04_compare.py"]}})
    (OUT / "run.json").write_text(json.dumps(run, indent=2), encoding="utf-8")
    print(table[["group", "overall_MAE", "daily_maximum_MAE", "rise_MAE", "rise_under", "low_stay_MAE", "low_stay_over", "eligible"]].to_string(index=False), flush=True)
    print(json.dumps(selection), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m04_rise_compare.py`

### `Modeling/scripts/m04_rise_documents.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Write or verify the detailed M04 rise manuscript from independently checked results.

주요 함수: `comparison()`, `monthly()`, `section()`, `write()`, `verify()`, `main()`.

In [49]:
%%submission_source Modeling/scripts/m04_rise_documents.py
"""Write or verify the detailed M04 rise manuscript from independently checked results."""
import argparse
import json
import re
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

from m03_verify import rows, sha

ROOT = Path(__file__).resolve().parents[2]
MD = ROOT / "Modeling/04_Modeling_원고.md"
LOG = ROOT / "Modeling/10.05_M04_상승예측결합_실행기록.md"
PARENT = ROOT / "Modeling/tables/m04_rise"
OUT = ROOT / "Modeling/tables/m04_rise_gate"
TITLE = "## 4.7 M04 추가 개발: 상승 예측 결합의 개선과 채택 한계"
ORDER = ["B", "D_dynamic", "W_rise_weighted", "G_Logistic", "G_HGB", "G_Logistic_rise_focus", "G_HGB_rise_focus"]
LABELS = {"B": "기존 B", "D_dynamic": "변화 입력 HGB", "W_rise_weighted": "상승 가중 HGB",
    "G_Logistic": "Logistic 분류 결합", "G_HGB": "HGB 분류 결합",
    "G_Logistic_rise_focus": "Logistic 결합·상승 중심 선택", "G_HGB_rise_focus": "HGB 결합·상승 중심 선택"}


def comparison():
    indexed = {r["group"]: r for r in rows(OUT / "selection_metrics.csv")}
    text = "| 방법 | 전체 MAE | 일별 최대 시간 MAE | 상승 MAE | 상승 평균 과소 | 낮은 유지 MAE | 낮은 유지 평균 과대 |\n|---|---:|---:|---:|---:|---:|---:|\n"
    for name in ORDER:
        r = indexed[name]
        text += "| " + LABELS[name] + " | " + " | ".join(f"{float(r[k]):.3f}" for k in ["overall_MAE", "daily_maximum_MAE", "rise_MAE", "rise_under", "low_stay_MAE", "low_stay_over"]) + " |\n"
    return text


def monthly():
    metrics = rows(PARENT / "metrics.csv")
    alarms = rows(PARENT / "classification.csv")
    configs = json.loads((PARENT / "selected_configurations.json").read_text(encoding="utf-8"))
    text = "| 개발 월 | 상승 수 | B 상승 MAE | HGB 결합 상승 MAE | AP | TP / FP / FN / TN | 임계값 / 결합 비율 |\n|---|---:|---:|---:|---:|---|---|\n"
    for split, label in [("dev_apr", "4월"), ("dev_may", "5월"), ("dev_jun", "6월")]:
        alarm = next(r for r in alarms if r["split"] == split and r["role"] == "outer" and r["model"] == "G_HGB")
        errors = []
        for name in ["B", "G_HGB"]:
            pool = [r for r in rows(PARENT / "predictions.csv") if r["split"] == split and r["group"] == name and r["rise_event"] == "True"]
            errors.append(sum(float(r["absolute_error"]) for r in pool) / len(pool))
        gate = next(r for r in configs if r["split"] == split)["classifiers"]["HGB"]["gate"]
        confusion = " / ".join(alarm[k] for k in ["TP", "FP", "FN", "TN"])
        text += f"| {label} | {alarm['positive_hours']} | {errors[0]:.3f} | {errors[1]:.3f} | {float(alarm['AP']):.3f} | {confusion} | {gate['threshold']:.4f} / {gate['blend']:.4f} |\n"
    return text


def section():
    table = {r["group"]: r for r in rows(OUT / "selection_metrics.csv")}
    rise_gain = 100 * (1 - float(table["G_HGB"]["rise_MAE_ratio"]))
    under_gain = 100 * (1 - float(table["G_HGB"]["rise_under_ratio"]))
    low_cost = 100 * (float(table["G_HGB"]["low_stay_MAE_ratio"]) - 1)
    return f"""{TITLE}

M03에서 마지막 값 추가는 전체·일별 최대 시간 오차를 줄였지만, 낮은 상태에서 26초과 상태로 바뀐 20시간을 모두 과소예측했다. 4.6의 직전 값 대체도 그 과소예측을 48.772에서 75.400으로 늘려 기각했다. 이 결과에서 '과거 변화 정보나 상승 사례의 학습 비중을 늘리면 상승 오차를 줄일 수 있는가, 그리고 상승 확률로 적용 대상을 고르면 낮은 유지의 오경보 부담을 줄일 수 있는가'를 후속 질문으로 정했다. [추가 실행 기록](10.05_M04_상승예측결합_실행기록.md)과 [첫 비교 조건](config/m04_rise_contract.json)에 질문·비교 범위·판단 기준을 기록했다.

### 상승의 정의와 시간순 비교

이번 상승은 기존 low→above26 전환이다. 직전 최대값이 양수이고 직전 정확한 평균을 `floor(mean+0.5)`로 반올림한 값이 20~26이며, 다음 시간의 제공 평균값이 26을 넘는 경우를 뜻한다. 모든 큰 증가나 0 이후 재가동을 포괄하는 정의가 아니다. 그 범위의 성과는 별도로 확인해야 한다. 목표는 다음 시간 네 15분 값의 최댓값이며 단위가 확인되지 않은 값에 임의로 kW를 붙이지 않는다.

외부 개발 비교는 1~3월 학습→4월, 1~4월→5월, 1~5월→6월의 같은 2,184시간·91일이다. 학습 상승 사례는 각각 31·39·47시간, 평가 상승은 8·8·4시간이다. 내부 검증은 1월→2월부터 과거 월만 순차 확장해 외부 평가 월 이전에 끝낸다. 직전 상태는 예측 시점에 관측할 수 있고, 다음 상태·일별 최대 시간·다음 생산량은 입력에 넣지 않는다. 7~8월은 이번 튜닝과 비교에 사용하지 않았다. 다만 후반기 자료의 과거 관찰 이력은 있으므로 완전히 처음 보는 독립 시험으로 표현하지 않는다.

전체 MAE와 일별 최대 시간 MAE, 낮은 유지 MAE는 B 대비 악화를 1% 이내로 제한하고, 낮은 유지 평균 과대예측은 5% 이내로 제한했다. 상승 MAE와 평균 과소예측은 각각 10% 이상 줄고 각 월의 상승 MAE도 악화되지 않아야 한다. 분류 결합의 오경보율은 직전 낮은 상태의 비상승 사례 중 5% 이내로 정했다. 이는 실행 전에 기록한 개발 허용 범위이며 현장 비용에서 추정한 최적값이 아니다. 외부 개발 결과로 채택 여부를 확인하되 그 결과에 맞춰 임계값을 직접 조정하지 않았다.

### 변화 입력·상승 가중 학습·분류 결합의 구분

B의 18개 입력에 과거 변화 입력 9개를 추가해 총 27개를 비교했다. 추가 입력은 직전 네 구간의 마지막값-첫값, 범위, 선형 기울기, 직전과 그 이전 시간의 마지막값·평균·최대·생산량 차이, 최대 6시간의 낮은 상태 연속 길이와 그 조회 중 시간 누락 여부다. 모든 값은 t-1 이전 관측으로 계산하고 누락 시각을 건너뛰어 연결하지 않는다. 기존 날씨 후보 D와 혼동하지 않도록 이번 `D_dynamic`은 변화 입력 HGB로 표기한다.

변화 입력 HGB는 기존 B의 설정을 고정한다. 상승 가중 HGB는 같은 변화 입력에서 학습 자료의 상승 정답에만 가중치 1~12를 부여하고 Optuna로 선택한다. 미래 상승 정답은 학습 가중치에 쓰며 추론 때 주어지는 정보로 사용하지 않는다. HGB 설정은 잎 15, 반복 150, 학습률 0.05, 최소 잎 표본 20, L2 규제 1, 조기 종료 사용 안 함, 난수 시드 42다. 변화 입력만 추가한 효과와 상승 학습 비중의 효과를 구분할 수 있도록 순차 비교했다.

분류기는 직전 상태가 낮은 학습 행만 이용해 다음 시간 상승 확률을 추정한다. Logistic은 학습 구간에서만 표준화하며 C를 0.01~100에서 찾고, HGB 분류기는 잎 3·7·15, 최소 잎 표본 10·20·40, L2 0.1~10을 비교했다. 희소 상승을 상위로 구분하는 능력은 내부 AP로 선택하고 Brier를 보조 확인했다. 여기서 AP는 비보간 평균 정밀도이며 사다리꼴 PR 면적과 같은 수치로 표기하지 않는다. AUROC·정밀도·재현율·오경보도 보존했다. 분류 지표는 상승 여부, 회귀 MAE는 전력 크기를 평가하므로 서로 대체하지 않는다.

결합은 `B + I(직전 낮은 상태이고 상승확률≥임계값) × 결합비율 × (상승 가중 HGB-B)`다. 조건 밖에서는 B를 그대로 사용한다. 확률 임계값 0.01~0.8과 결합 비율 0~1은 과거 내부 검증의 예측만으로 선택했다. 첫 회차는 전체·최대 시간·상승의 B 대비 상대 MAE 중 최댓값과 허용 범위 초과 벌점을 최소화했다. 가중치·두 분류기·두 결합을 외부 월마다 각 30회 탐색해 총 Optuna 450회, 학습 840회를 실행했다. 이 중 외부 최종 학습은 12회이며 저장 모델 12개를 재로딩해 예측을 대조했다.

### 같은 개발 시간에서의 결과

{comparison()}
상승 가중 HGB는 상승 MAE를 35.682로 줄였지만 전체·최대 시간·낮은 유지 오차를 모두 키웠다. 변화 입력만 추가한 모형도 기존 B보다 좋아지지 않았다. Logistic 결합의 첫 내부 선택은 결합 비율을 0으로 정해 회귀 예측이 B와 같았다. 분류기를 붙였다는 사실만으로 상승 예측이 개선되는 것은 아니었다.

HGB 분류 결합은 상승 MAE를 {rise_gain:.1f}%, 평균 과소예측을 {under_gain:.1f}% 줄였고 전체 MAE도 6.191에서 6.146으로 약 0.7% 줄였다. 그러나 낮은 유지 MAE는 4.110에서 4.228로 {low_cost:.1f}% 늘어 사전 허용 범위 1%를 넘었다. 다른 채택 요건은 통과했지만 이 요건을 충족하지 못해 최종 교체 후보로 채택하지 않았다. 일별 최대 시간 MAE가 같은 이유는 이 결합이 바꾼 예측이 해당 최대 시간과 겹치지 않았기 때문이다. 이번 상승 개선을 일별 최대 시간 개선으로 표현하지 않는다.

### 어느 월에서 상승을 찾았고 어디서 오경보가 남았는가

다음 표는 첫 HGB 결합의 월별 결과다. AP의 분모는 직전 낮은 상태의 200·262·188시간이고 상승은 8·8·4시간이다. TP는 상승에서 결합 조건을 충족한 수, FP는 비상승에서 그 조건을 충족한 수다. 각 월의 확률·비율은 해당 월 이전 내부 검증으로 정했다.

{monthly()}
세 달을 합치면 상승 20시간 중 9시간에서 조건을 충족해 재현율 45.0%였고, 총 15번 중 9번이 상승이므로 정밀도는 60.0%였다. 오경보 6시간은 비상승 630시간의 약 0.95%다. 비상승에는 낮은 유지 625시간과 낮은 상태에서 20미만으로 내려간 5시간이 포함된다. 오경보율이 작아도 예측값을 크게 높인 오경보 한 번이 회귀 오차를 늘릴 수 있다. 이 수치는 현장 경보를 실제 운영한 성과가 아닌 개발 자료의 결합 조건 평가다.

예를 들어 6월 6일 08시는 실제 최대값 22를 90.295로, 6월 27일 07시는 실제 23을 87.341로 예측했다. 이처럼 낮은 유지에서 상승 전문가가 과도하게 반영된 사례가 남았다. 4월에는 상승 8시간을 모두 놓쳤고 결합 비율도 약 0.0013이어서 수정 효과가 거의 없었다. 반면 5·6월은 일부 상승을 찾아 그 크기 오차를 줄였다. 월별 AP의 0.188·0.796·1.000 차이와 6월 상승 4시간의 작은 분모를 함께 제시해야 하며 후반기에도 같은 성능이 나온다고 단정하지 않는다.

### 결과에 따른 한 번의 선택 점수 수정

첫 내부 점수는 최대 시간 오차가 B와 같으면 최댓값 항이 1에 머물러, 다른 조건이 허용 범위 안에 있는 상승 개선을 충분히 구분하지 못할 수 있었다. 이 문제만 확인하기 위해 [후속 조건](config/m04_rise_gate_contract.json)을 기록하고 최댓값 항을 상승 상대 MAE로 바꿨다. 나머지 벌점·채택 허용 범위·분류기·회귀모델·입력·분할은 고정했다. 저장된 과거 내부 예측으로 결합 임계값과 비율만 두 계열·세 외부 월에 각 50회, 총 300회 탐색했으며 새 모델 학습은 없었다.

HGB의 상승 중심 선택은 상승 MAE 40.471, 낮은 유지 MAE 4.226으로 첫 결합보다 충분한 개선을 보이지 않았고 낮은 유지 요건도 계속 실패했다. Logistic의 상승 중심 선택은 낮은 유지 오차를 줄였으나 상승 MAE 감소가 약 0.5%에 그쳐 10% 요건을 충족하지 못했다. 합계 750회의 개발 탐색 이후 모든 요건을 통과한 추가 후보는 없었고 기존 B를 유지했다. 이번에 정한 비교와 한 번의 점수 수정은 여기서 종료하며, 외부 결과에 맞춰 허용 범위를 늘리거나 선택 경계를 계속 바꾸지 않는다.

### 보고서 연결과 다음 판단

확인된 사실은 마지막 값 추가의 기존 효과와 별개로, 상승 분류와 가중 회귀의 결합이 일부 상승의 과소예측을 줄일 수 있었다는 점이다. 유지할 판단은 기존 B를 현재 모델 선택의 개발 후보로 유지한다는 점이며, 수정할 주장은 '급상승을 전반적으로 잘 예측한다'는 표현이다. 낮은 유지의 과대예측과 4월 상승 미탐지가 남았고, 운영 원인과 실제 절감 효과는 현 자료에서 식별하거나 실측하지 못했다.

제2장에는 동일 조건의 추가 비교와 B 유지 이유, 제3장에는 상승 개선·오경보·월별 실패를 연결한다. 제5장의 주된 차별성은 M03에서 검증한 마지막 값의 기여로 작성한다. 이번 17.5% 상승 오차 감소는 허용 요건을 통과하지 못한 보완 실험의 부분 성과로 설명하며 최종 모델 성과에 합산하지 않는다. 짧은 최종 보고서에서는 전체 탐색표를 재현 자료로 두고 핵심 상충만 남길 수 있다. 다음 질문은 고정 B의 A 대비 개선과 이 실패조건이 후반기에도 유지되는가다. M05 전에 방법·판단 기준을 고정한 뒤 1~6월 학습→7~8월 확인으로 연결하며, 이번에는 M05를 실행하지 않았다.

### 전체 코드와 독립 검증

[학습·분류·가중 회귀·결합 코드](scripts/m04_rise_compare.py), [원본 입력·모델 재로딩·독립 검증 코드](scripts/m04_rise_verify.py), [저장 내부 예측의 선택 점수 수정 코드](scripts/m04_rise_gate.py), [300회 선택·결합 독립 검증 코드](scripts/m04_rise_gate_verify.py), [원고 생성·검증 코드](scripts/m04_rise_documents.py)를 4.5의 전체 코드 보존 목록에 포함했다. 공통 모듈도 해당 목록과 함께 보존한다.

[첫 전체 지표](tables/m04_rise/metrics.csv), [첫 월별 설정](tables/m04_rise/selected_configurations.json), [분류 지표](tables/m04_rise/classification.csv), [개별 예측](tables/m04_rise/predictions.csv), [첫 독립 검증](tables/m04_rise/independent_verification.json), [최종 비교표](tables/m04_rise_gate/selection_metrics.csv), [최종 선택](tables/m04_rise_gate/selection.json), [후속 독립 검증](tables/m04_rise_gate/independent_verification.json)에 근거를 남겼다. 첫 회차는 과거 입력 112,752값·10,920예측행·260지표행·450시도와 12개 모델을, 후속은 15,288예측행·364지표행·300시도의 점수·AP·혼동행렬과 변경 없는 채택 요건을 대조했다. 두 검증은 정규 CPython 3.13에서 종료 코드 0으로 통과했다. 원자료와 기존 M01~M04 결과를 유지했고 압축본·PDF·새 그림·파일 삭제는 수행하지 않았다.
"""


def write():
    text = MD.read_text(encoding="utf-8")
    assert TITLE not in text, "Preserve existing section; use --verify after writing."
    text = text.replace("사용한 20개 파일이다.", "사용한 25개 파일이다. 상승 추가 개발의 학습·검증·문서 코드도 포함한다.")
    added = "\n".join([
        "| [m04_rise_compare.py](scripts/m04_rise_compare.py) | 4.7 변화 입력·상승 가중·두 분류 결합의 시간순 튜닝과 예측 | `build_frame()`, `stats()`, `score()`, `main()` |",
        "| [m04_rise_verify.py](scripts/m04_rise_verify.py) | 4.7 원본 27입력·AP·모든 지표·튜닝 선택·저장 모델 검증 | `classification()`, `stats()`, `main()` |",
        "| [m04_rise_gate.py](scripts/m04_rise_gate.py) | 4.7 과거 내부 예측에서 상승 중심 결합 점수만 변경 | `main()`; 새 학습 없음 |",
        "| [m04_rise_gate_verify.py](scripts/m04_rise_gate_verify.py) | 4.7 결합식·300회 점수·지표·변경 없는 채택 요건 독립 검증 | `gate_rows()`, `objective()`, `main()` |",
        "| [m04_rise_documents.py](scripts/m04_rise_documents.py) | 4.7 검증된 표의 상세 원고·기록·README 통합 및 수치·링크 검증 | `--write` 최초 작성, 이후 `--verify` |",
    ])
    text = text.replace("| [modeling_submission_audit.py]", added + "\n| [modeling_submission_audit.py]", 1)
    text = text.replace("[M04](config/m04_contract.json).", "[M04](config/m04_contract.json), [상승 비교](config/m04_rise_contract.json), [결합 점수 수정](config/m04_rise_gate_contract.json).")
    text = text.replace("`m04/`, `observed_timeline/`", "`m04/`, `m04_rise/`, `m04_rise_gate/`, `observed_timeline/`")
    text = text.replace("`m02_rerun/`의 파일. 모델 목록", "`m02_rerun/`, `m04_rise/`의 파일. 모델 목록")
    old = "| 9 | `Modeling/scripts/m02_rerun_document_verify.py`, `Modeling/scripts/m04_verify.py --documents` | 원고 수치·링크·기존 M03 문서 및 M04 문서 검증 |\n| 10 |"
    new = "| 9 | `Modeling/scripts/m04_rise_compare.py` → `Modeling/scripts/m04_rise_verify.py` | 변화 입력·상승 가중·분류 결합의 추가 학습과 독립 검증 |\n| 10 | `Modeling/scripts/m04_rise_gate.py` → `Modeling/scripts/m04_rise_gate_verify.py` | 검증된 과거 내부 예측의 결합 점수 수정·독립 검증; 새 학습 없음 |\n| 11 | `Modeling/scripts/m02_rerun_document_verify.py`, `Modeling/scripts/m04_verify.py --documents`, `Modeling/scripts/m04_rise_documents.py --verify` | 기존·추가 원고 수치와 링크 검증; 문서 작성 코드는 학습 재현과 별도로 사용 |\n| 12 |"
    assert old in text
    text = text.replace(old, new)
    MD.write_text(text.rstrip() + "\n\n" + section(), encoding="utf-8")
    LOG.write_text("# M04 상승 예측 결합 실행 기록\n\n" + section().replace(TITLE, "## 질문·방법·결과와 판단") +
        "\n## 재현 실행 기록\n\n첫 실행은 `Modeling/scripts/m04_rise_compare.py`, 검증은 `m04_rise_verify.py`로 수행했다. 첫 run.json의 학습 수는 내부 회귀288·내부 분류540·외부12이며 Optuna450회다. 후속 `m04_rise_gate.py`는 검증된 첫 실행의 모델과 내부 예측을 재사용해 Optuna300회만 수행했고 `m04_rise_gate_verify.py`로 검증했다. 완료 폴더를 덮어쓰지 않는 보호 조건을 유지한다.\n\n[첫 run.json](tables/m04_rise/run.json), [첫 시간순 분할](tables/m04_rise/fold_audit.csv), [첫 모든 시도](tables/m04_rise/trials.csv), [첫 모델 목록](tables/m04_rise/model_manifest.csv), [후속 run.json](tables/m04_rise_gate/run.json), [후속 모든 시도](tables/m04_rise_gate/trials.csv), [후속 설정](tables/m04_rise_gate/selected_configurations.json)을 함께 보존한다. 문서 작성과 검증은 `m04_rise_documents.py --write`·`--verify`이며 원자료를 수정하지 않는다.\n", encoding="utf-8")
    root_readme = ROOT / "README.md"
    text = root_readme.read_text(encoding="utf-8")
    text = text.replace("M04 조건별 오류·낮은 상태 규칙 비교까지 완료했다.", "M04 조건별 오류·낮은 상태 규칙·상승 결합 비교까지 완료했다.")
    anchor = "같은 수의 관리시간으로 높은 최대전력 시간을 더 잘 찾는 성과는 별도 검증이 필요하다."
    summary = "[M04 상승 추가 개발](Modeling/10.05_M04_상승예측결합_실행기록.md): 변화 입력·상승 가중 HGB·Logistic/HGB 분류 결합과 한 번의 결합 점수 수정을 시간순으로 비교했다. HGB 결합은 상승 MAE48.772→40.217(17.5% 감소), 전체6.191→6.146이지만 낮은 유지 MAE4.110→4.228(2.9% 증가)로 사전1% 허용을 넘었다. 상승20시간 중9시간을 찾아 재현율45%·정밀도60%, 비상승630시간 중오경보6시간이었다. Optuna750회·새학습840회와 독립 검증을 마쳤고 모든 교체 요건을 통과한 후보가 없어 B를 유지했다. 원고4.7·전체25코드·제출 보존 목록에 연결했으며 M05는 미실행이다.\n\n"
    assert anchor in text
    text = text.replace(anchor, summary + anchor, 1)
    root_readme.write_text(text, encoding="utf-8")
    readme = ROOT / "Modeling/README.md"
    text = readme.read_text(encoding="utf-8")
    text = text.replace("M04 조건별 오류·낮은 상태 규칙 비교 완료", "M04 조건별 오류·낮은 상태 규칙·상승 결합 비교 완료", 1)
    proposal = next(line for line in text.splitlines() if line.startswith("직전 low 일괄 대체 질문은 기각으로 종료한다."))
    replacement = "직전 low 일괄 대체 질문은 기각으로 종료했다. 사용자 요청으로 이어간 [상승 추가 개발](10.05_M04_상승예측결합_실행기록.md)은 변화 입력·상승 가중 HGB·Logistic/HGB 분류 결합을 비교했다. 첫 HGB 결합은 상승 MAE48.772→40.217(17.5% 감소), 전체6.191→6.146이나 낮은 유지4.110→4.228(2.9% 악화)로 사전1% 허용을 넘었다. 재현율45%·정밀도60%·오경보6/630과 4월 미탐지를 확인했다. 이 결과에서 결합 내부 선택 점수만 상승 중심으로 바꿔 한 번 더 비교했지만 추가 후보도 교체 요건을 통과하지 못했다. [첫 조건](config/m04_rise_contract.json), [후속 조건](config/m04_rise_gate_contract.json), [최종 비교표](tables/m04_rise_gate/selection_metrics.csv), [독립 검증](tables/m04_rise_gate/independent_verification.json)에 근거를 보존한다. 총 Optuna750회·학습840회 후 선언한 비교는 종료하고 B를 유지한다. 원고4.7·4.5의 전체25코드와 재현 파일을 연결했다. 다음 질문은 고정 A/B의 효과와 실패조건이 7~8월에서도 유지되는가이며 M05는 아직 실행하지 않았다."
    text = text.replace(proposal, replacement)
    text = text.replace("4.4는 Optuna·시간순 검증·확장 모델과 HGB 유지 판단을 설명한다.", "4.4는 Optuna·시간순 검증·확장 모델과 HGB 유지 판단, 4.5는 전체25코드 보존, 4.6은 낮은 상태 규칙 기각, 4.7은 상승 결합의 부분 개선·오경보·B 유지 판단을 설명한다.")
    readme.write_text(text, encoding="utf-8")


def verify():
    for directory, verifier in [(PARENT, "m04_rise_verify.py"), (OUT, "m04_rise_gate_verify.py")]:
        result = json.loads((directory / "independent_verification.json").read_text(encoding="utf-8"))
        assert result["status"] == "passed" and result["run_sha256"] == sha(directory / "run.json")
        assert result["script_sha256"] == sha(ROOT / "Modeling/scripts" / verifier)
    manuscript = MD.read_text(encoding="utf-8")
    assert manuscript.count(TITLE) == 1
    assert manuscript.split(TITLE)[1].split("\n## 4.8")[0].rstrip() == section().split(TITLE)[1].rstrip()
    first_predictions = rows(PARENT / "predictions.csv")
    baseline = {r["timestamp"]: r for r in first_predictions if r["group"] == "B"}
    hgb = [r for r in first_predictions if r["group"] == "G_HGB"]
    assert not any(float(r["daily_maximum_weight"]) > 0 and
                   abs(float(r["prediction"]) - float(baseline[r["timestamp"]]["prediction"])) > 1e-9 for r in hgb)
    assert sum(r["trigger"] == "True" and r["rise_event"] == "True" for r in hgb) == 9
    assert sum(r["trigger"] == "True" and r["rise_event"] == "False" for r in hgb) == 6
    for timestamp, actual, prediction in [("2021-06-06 08:00:00", 22, 90.295), ("2021-06-27 07:00:00", 23, 87.341)]:
        case = next(r for r in hgb if r["timestamp"] == timestamp)
        assert float(case["actual"]) == actual and f"{float(case['prediction']):.3f}" == f"{prediction:.3f}"
    first_run = json.loads((PARENT / "run.json").read_text(encoding="utf-8"))
    follow_run = json.loads((OUT / "run.json").read_text(encoding="utf-8"))
    assert sum(first_run["fits"].values()) == 840 and first_run["trials"] + follow_run["trials"] == 750
    linked = 0
    for path in [MD, LOG]:
        text = path.read_text(encoding="utf-8")
        for target in re.findall(r"!?\[[^\]]*\]\(([^)]+)\)", text):
            if target.startswith(("http:", "https:", "#")):
                continue
            assert (path.parent / target.split("#")[0].strip("<>")).resolve().exists(), target
            linked += 1
        assert comparison() in text and monthly() in text
    result = {"status": "passed", "checked_at": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
        "manuscript_sha256": sha(MD), "execution_record_sha256": sha(LOG), "script_sha256": sha(Path(__file__)),
        "comparison_rows": 7, "monthly_rows": 3, "local_links_checked": linked,
        "selection": "B", "no_training_or_deletion": True}
    (OUT / "document_verification.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps(result), flush=True)


def main():
    parser = argparse.ArgumentParser()
    mode = parser.add_mutually_exclusive_group(required=True)
    mode.add_argument("--write", action="store_true")
    mode.add_argument("--verify", action="store_true")
    args = parser.parse_args()
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    if args.write:
        write()
    verify()


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m04_rise_documents.py`

### `Modeling/scripts/m04_rise_gate.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: One objective change using verified past-only cached predictions; no new fitting.

주요 함수: `save()`, `main()`.

In [50]:
%%submission_source Modeling/scripts/m04_rise_gate.py
"""One objective change using verified past-only cached predictions; no new fitting."""
import json
import sys
from pathlib import Path

import numpy as np
import optuna
import pandas as pd

from m01_prepare import ROOT, sha
from m02_compare import metric
from m04_compare import subsets
from m04_rise_compare import gated, stats, score, alarm_stats, now, read_csv

OUT = ROOT / "Modeling/tables/m04_rise_gate"
CONTRACT = ROOT / "Modeling/config/m04_rise_gate_contract.json"


def save(frame, name):
    frame.to_csv(OUT / name, index=False, encoding="utf-8-sig")


def main():
    assert sys.version_info[:2] == (3,13) and sys._is_gil_enabled()
    assert not (OUT / "run.json").exists(), "Preserve completed run."
    follow = json.loads(CONTRACT.read_text(encoding="utf-8"))
    parent_dir = ROOT / "Modeling/tables/m04_rise"
    parent_run = json.loads((parent_dir / "run.json").read_text(encoding="utf-8"))
    verified = json.loads((parent_dir / "independent_verification.json").read_text(encoding="utf-8"))
    assert verified["status"] == "passed" and verified["run_sha256"] == sha(parent_dir / "run.json") == follow["parent_run_sha256"]
    for name in ["predictions.csv", "inner_predictions.csv"]:
        assert sha(parent_dir / name) == parent_run["outputs_sha256"][name]
    c = json.loads((ROOT / "Modeling/config/m04_rise_contract.json").read_text(encoding="utf-8"))
    p = read_csv(parent_dir / "predictions.csv", ["timestamp","date"])
    inner = read_csv(parent_dir / "inner_predictions.csv", ["timestamp","date"])
    OUT.mkdir(parents=True, exist_ok=True)
    records, configs, trials, classifications, selected_inner = [p], [], [], [], []
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    for split in c["outer_splits"]:
        iv = inner.loc[inner.split.eq(split) & inner.group.eq("B")].copy()
        iv["target_maximum"] = iv.actual
        ib = iv.prediction.to_numpy()
        iw = inner.loc[inner.split.eq(split) & inner.group.eq("W_rise_weighted")].prediction.to_numpy()
        ev = p.loc[p.split.eq(split) & p.group.eq("B")].copy()
        ev["target_maximum"] = ev.actual
        expert = p.loc[p.split.eq(split) & p.group.eq("W_rise_weighted")].prediction.to_numpy()
        reference = stats(iv, ib)
        for family in follow["families"]:
            ip = inner.loc[inner.split.eq(split) & inner.group.eq("G_"+family)].probability.to_numpy()
            op = p.loc[p.split.eq(split) & p.group.eq("G_"+family)].probability.to_numpy()
            def objective(trial):
                threshold = trial.suggest_float("threshold", *c["gate"]["threshold"])
                blend = trial.suggest_float("blend", *c["gate"]["blend"])
                prediction, trigger = gated(ib, iw, ip, iv.prior_low, threshold, blend)
                values = stats(iv, prediction)
                alarm = alarm_stats(iv, ip, threshold)
                maximum_term = max(values[k] / reference[k] for k in ["overall_MAE","daily_maximum_MAE","rise_MAE"])
                new_score = score(values, reference, c["development_selection"], alarm["false_positive_fraction"]) - maximum_term + values["rise_MAE"] / reference["rise_MAE"]
                trial.set_user_attr("metrics", values)
                trial.set_user_attr("classification", alarm)
                return new_score
            study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42,n_startup_trials=10), pruner=optuna.pruners.NopPruner())
            for params in c["gate"]["enqueue"]:
                study.enqueue_trial(params)
            study.optimize(objective,n_trials=50)
            best = min(study.trials,key=lambda trial:(trial.value,trial.number))
            name = "G_" + family + "_rise_focus"
            rec = ev.copy()
            rec["group"] = name
            rec["prediction"], rec["trigger"] = gated(ev.prediction,expert,op,ev.prior_low,**best.params)
            rec["probability"] = op
            rec = rec.drop(columns="target_maximum")
            records.append(rec)
            ir = iv.copy()
            ir["group"] = name
            ir["prediction"], ir["trigger"] = gated(ib,iw,ip,iv.prior_low,**best.params)
            ir["probability"] = ip
            selected_inner.append(ir.drop(columns="target_maximum"))
            configs.append({"split":split,"group":name,"best_trial":best.number,"parameters":best.params,"inner_metrics":best.user_attrs})
            for trial in study.trials:
                trials.append({"split":split,"group":name,"trial":trial.number,"objective":trial.value,"parameters":json.dumps(trial.params),"metrics":json.dumps(trial.user_attrs)})
            for role,part,prob in [("inner",iv,ip),("outer",ev,op)]:
                classifications.append({"split":split,"role":role,"group":name,**alarm_stats(part,prob,best.params["threshold"])})
            print(f"{split} {name}: {best.params}",flush=True)
    p = pd.concat(records,ignore_index=True)
    p["signed_error"] = p.prediction-p.actual
    p["absolute_error"] = p.signed_error.abs()
    p["under_amount"] = (-p.signed_error).clip(lower=0)
    p["over_amount"] = p.signed_error.clip(lower=0)
    save(p,"predictions.csv")
    save(pd.concat(selected_inner,ignore_index=True),"inner_predictions.csv")
    save(pd.DataFrame(trials),"trials.csv")
    save(pd.DataFrame(classifications),"classification.csv")
    (OUT/"selected_configurations.json").write_text(json.dumps(configs,indent=2),encoding="utf-8")
    metrics,table = [],[]
    baseline = p.loc[p.group.eq("B")].sort_values("timestamp").copy()
    baseline["target_maximum"] = baseline.actual
    reference = stats(baseline,baseline.prediction)
    for name,part in p.groupby("group"):
        for kind,value,subset,weights in subsets(part):
            metrics.append({"group":name,"kind":kind,"value":value,**metric(subset,weights)})
        values = stats(baseline,part.sort_values("timestamp").prediction)
        row = {"group":name,**values,"balanced_score":max(values[k]/reference[k] for k in ["overall_MAE","daily_maximum_MAE","rise_MAE"])}
        conditions = []
        for key,setting in [("overall_MAE","overall_MAE_ratio_max"),("daily_maximum_MAE","daily_maximum_MAE_ratio_max"),
            ("low_stay_MAE","low_stay_MAE_ratio_max"),("low_stay_over","low_stay_mean_over_ratio_max"),
            ("rise_MAE","rise_MAE_ratio_max"),("rise_under","rise_mean_under_ratio_max")]:
            row[key+"_ratio"] = values[key]/reference[key]
            row[key+"_passed"] = bool(row[key+"_ratio"] <= c["development_selection"][setting]+1e-12)
            conditions.append(row[key+"_passed"])
        row["rise_monthly_passed"] = bool(all(part.loc[part.split.eq(split)&part.rise_event,"absolute_error"].mean() <=
            baseline.loc[baseline.split.eq(split)&baseline.rise_event,"absolute_error"].mean()+1e-9 for split in c["outer_splits"]))
        conditions.append(row["rise_monthly_passed"])
        if name.startswith("G_"):
            low = part.loc[part.prior_low]
            fp = int((low.trigger&~low.rise_event).sum())
            negatives = int((~low.rise_event).sum())
            row["gate_false_positive_fraction"] = fp/negatives
            row["gate_false_positive_passed"] = bool(fp/negatives <= .05+1e-12)
            conditions.append(row["gate_false_positive_passed"])
        row["eligible"] = name!="B" and all(conditions)
        table.append(row)
    table = pd.DataFrame(table)
    eligible = table.loc[table.eligible].copy()
    order = c["methods"]+["G_Logistic_rise_focus","G_HGB_rise_focus"]
    if len(eligible):
        near = eligible.loc[eligible.balanced_score.le(eligible.balanced_score.min()*1.01)].copy()
        near["preference"] = near.group.map({name:i for i,name in enumerate(order)})
        chosen = near.sort_values(["preference","balanced_score"]).iloc[0].group
    else:
        chosen = "B"
    selection = {"selected":chosen,"eligible_methods":eligible.group.tolist(),"acceptance_criteria_unchanged":True,"development_only":True,"no_july_august_evaluation":True}
    save(pd.DataFrame(metrics),"metrics.csv")
    save(table,"selection_metrics.csv")
    (OUT/"selection.json").write_text(json.dumps(selection,indent=2),encoding="utf-8")
    run = {"status":"completed","finished":now(),"contract_sha256":sha(CONTRACT),"parent_run_sha256":sha(parent_dir/"run.json"),
        "script_sha256":sha(Path(__file__)),"no_new_fitting":True,"no_july_august_evaluation":True,"trials":len(trials),
        "outputs_sha256":{path.name:sha(path) for path in OUT.iterdir() if path.name!="run.json"},"prediction_rows":len(p),"metric_rows":len(metrics)}
    (OUT/"run.json").write_text(json.dumps(run,indent=2),encoding="utf-8")
    print(table[["group","overall_MAE","daily_maximum_MAE","rise_MAE","rise_under","low_stay_MAE","low_stay_over","eligible"]].to_string(index=False),flush=True)
    print(json.dumps(selection),flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m04_rise_gate.py`

### `Modeling/scripts/m04_rise_gate_verify.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Independently verify cached gate trials, predictions and unchanged acceptance gates.

주요 함수: `grouped()`, `gate_rows()`, `objective()`, `main()`.

In [51]:
%%submission_source Modeling/scripts/m04_rise_gate_verify.py
"""Independently verify cached gate trials, predictions and unchanged acceptance gates."""
import json
import math
import sys
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path

from m03_verify import close, rows, sha, verify_metric
from m04_rise_verify import boolean, classification, stats

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/m04_rise_gate"
PARENT = ROOT / "Modeling/tables/m04_rise"


def grouped(pool):
    result = defaultdict(list)
    for r in pool:
        result[r["split"], r["group"]].append(r)
    return result


def gate_rows(base, expert, probability, threshold, blend):
    result = []
    for b, w, p in zip(base, expert, probability):
        assert b["timestamp"] == w["timestamp"] == p["timestamp"]
        trigger = boolean(b["prior_low"]) and float(p["probability"]) >= threshold
        r = dict(b)
        r["prediction"] = float(b["prediction"]) + trigger * blend * (float(w["prediction"]) - float(b["prediction"]))
        r["probability"] = p["probability"]
        r["trigger"] = str(trigger)
        result.append(r)
    return result


def objective(values, reference, limits, fpr):
    penalty = math.fsum(max(values[k] / reference[k] / limits[setting] - 1, 0) for k, setting in [
        ("overall_MAE", "overall_MAE_ratio_max"), ("daily_maximum_MAE", "daily_maximum_MAE_ratio_max"),
        ("low_stay_MAE", "low_stay_MAE_ratio_max"), ("low_stay_over", "low_stay_mean_over_ratio_max")])
    penalty += max(fpr / limits["gate_false_positive_fraction_max"] - 1, 0)
    return values["rise_MAE"] / reference["rise_MAE"] + 10 * penalty


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    run = json.loads((OUT / "run.json").read_text(encoding="utf-8"))
    follow_path = ROOT / "Modeling/config/m04_rise_gate_contract.json"
    follow = json.loads(follow_path.read_text(encoding="utf-8"))
    contract = json.loads((ROOT / "Modeling/config/m04_rise_contract.json").read_text(encoding="utf-8"))
    parent = json.loads((PARENT / "run.json").read_text(encoding="utf-8"))
    independent = json.loads((PARENT / "independent_verification.json").read_text(encoding="utf-8"))
    assert run["status"] == "completed" and run["no_new_fitting"] and run["no_july_august_evaluation"]
    assert sha(follow_path) == run["contract_sha256"]
    assert sha(ROOT / "Modeling/scripts/m04_rise_gate.py") == run["script_sha256"]
    assert independent["status"] == "passed"
    assert sha(PARENT / "run.json") == run["parent_run_sha256"] == follow["parent_run_sha256"] == independent["run_sha256"]
    for name, expected in parent["outputs_sha256"].items():
        assert sha(PARENT / name) == expected, name
    for name, expected in run["outputs_sha256"].items():
        assert sha(OUT / name) == expected, name
    pred, inn = rows(OUT / "predictions.csv"), rows(OUT / "inner_predictions.csv")
    original, cached = rows(PARENT / "predictions.csv"), rows(PARENT / "inner_predictions.csv")
    index = {(r["split"], r["group"], r["timestamp"]): r for r in pred}
    assert len(index) == len(pred) == run["prediction_rows"] == 15288
    for r in original:
        current = index[r["split"], r["group"], r["timestamp"]]
        for field in r:
            if field in ["prediction", "signed_error", "absolute_error", "under_amount", "over_amount"]:
                close(current[field], r[field])
            else:
                assert current[field] == r[field], field
    parent_groups, cached_groups = grouped(original), grouped(cached)
    inner_groups, outer_groups = grouped(inn), grouped(pred)
    configs = json.loads((OUT / "selected_configurations.json").read_text(encoding="utf-8"))
    trial_rows, alarms = rows(OUT / "trials.csv"), rows(OUT / "classification.csv")
    assert len(configs) == 6 and len(trial_rows) == run["trials"] == 300 and len(alarms) == 12
    trial_checks = 0
    for config in configs:
        split, group = config["split"], config["group"]
        family = group.removesuffix("_rise_focus")
        part = [r for r in trial_rows if r["split"] == split and r["group"] == group]
        assert len(part) == 50 and {int(r["trial"]) for r in part} == set(range(50))
        best = min(part, key=lambda r: (float(r["objective"]), int(r["trial"])))
        assert int(best["trial"]) == config["best_trial"] and json.loads(best["parameters"]) == config["parameters"]
        start = datetime(2021, {"dev_apr": 4, "dev_may": 5, "dev_jun": 6}[split], 1)
        assert max(datetime.fromisoformat(r["timestamp"]) for r in cached_groups[split, "B"]) < start
        reference = stats(cached_groups[split, "B"])
        for trial in part:
            parameters = json.loads(trial["parameters"])
            assert .01 <= parameters["threshold"] <= .8 and 0 <= parameters["blend"] <= 1
            generated = gate_rows(cached_groups[split, "B"], cached_groups[split, "W_rise_weighted"],
                                  cached_groups[split, family], **parameters)
            values = stats(generated)
            alarm = classification(generated, parameters["threshold"])
            close(trial["objective"], objective(values, reference, contract["development_selection"], alarm["false_positive_fraction"]))
            attrs = json.loads(trial["metrics"])
            for key, value in values.items():
                close(attrs["metrics"][key], value)
            for key, value in alarm.items():
                close(attrs["classification"][key], value)
            trial_checks += 1
        for role, base_groups, actual_groups in [("inner", cached_groups, inner_groups), ("outer", parent_groups, outer_groups)]:
            generated = gate_rows(base_groups[split, "B"], base_groups[split, "W_rise_weighted"],
                                  base_groups[split, family], **config["parameters"])
            actual = actual_groups[split, group]
            assert len(actual) == len(generated)
            for r, expected in zip(actual, generated):
                assert r["timestamp"] == expected["timestamp"] and r["trigger"] == expected["trigger"]
                close(r["prediction"], expected["prediction"])
                close(r["probability"], expected["probability"])
            alarm = classification(actual, config["parameters"]["threshold"])
            saved = next(r for r in alarms if (r["split"], r["role"], r["group"]) == (split, role, group))
            for key, value in alarm.items():
                close(saved[key], value)
            if role == "inner":
                for key, value in stats(actual).items():
                    close(config["inner_metrics"]["metrics"][key], value)
    groups = defaultdict(list)
    for r in pred:
        groups[r["group"]].append(r)
        assert datetime.fromisoformat(r["timestamp"]).month in [4, 5, 6]
        error = float(r["prediction"]) - float(r["actual"])
        for key, value in [("signed_error", error), ("absolute_error", abs(error)), ("under_amount", max(-error, 0)), ("over_amount", max(error, 0))]:
            close(r[key], value)
    expected_times = {r["timestamp"] for r in groups["B"]}
    assert len(groups) == 7 and len(expected_times) == 2184
    assert all(len(g) == 2184 and {r["timestamp"] for r in g} == expected_times for g in groups.values())
    metrics = rows(OUT / "metrics.csv")
    assert len(metrics) == run["metric_rows"] == 364
    for r in metrics:
        g, kind, value = groups[r["group"]], r["kind"], r["value"]
        weight = "all"
        if kind in ["daily_maximum", "profile_reweighted"]:
            weight = kind
            part = [x for x in g if float(x["daily_maximum_weight"]) > 0] if kind == "daily_maximum" else g
        elif kind.startswith("peak_"):
            weight = "daily_maximum"
            part = [x for x in g if float(x["daily_maximum_weight"]) > 0 and x[kind[5:]] == value]
        else:
            part = g if kind == "all" else [x for x in g if x[kind] == value]
        verify_metric(r, part, weight)
    table = rows(OUT / "selection_metrics.csv")
    reference = stats(groups["B"])
    for r in table:
        values = stats(groups[r["group"]])
        for key, value in values.items():
            close(r[key], value)
        passed = []
        for key, setting in [("overall_MAE", "overall_MAE_ratio_max"), ("daily_maximum_MAE", "daily_maximum_MAE_ratio_max"),
            ("low_stay_MAE", "low_stay_MAE_ratio_max"), ("low_stay_over", "low_stay_mean_over_ratio_max"),
            ("rise_MAE", "rise_MAE_ratio_max"), ("rise_under", "rise_mean_under_ratio_max")]:
            ratio = values[key] / reference[key]
            flag = ratio <= contract["development_selection"][setting] + 1e-12
            close(r[key + "_ratio"], ratio)
            assert boolean(r[key + "_passed"]) == flag
            passed.append(flag)
        monthly = all(stats(outer_groups[split, r["group"]])["rise_MAE"] <= stats(outer_groups[split, "B"])["rise_MAE"] + 1e-9 for split in contract["outer_splits"])
        assert boolean(r["rise_monthly_passed"]) == monthly
        passed.append(monthly)
        if r["group"].startswith("G_"):
            low = [x for x in groups[r["group"]] if boolean(x["prior_low"])]
            fp = sum(boolean(x["trigger"]) and not boolean(x["rise_event"]) for x in low)
            negatives = sum(not boolean(x["rise_event"]) for x in low)
            close(r["gate_false_positive_fraction"], fp / negatives)
            flag = fp / negatives <= contract["development_selection"]["gate_false_positive_fraction_max"] + 1e-12
            assert boolean(r["gate_false_positive_passed"]) == flag
            passed.append(flag)
        assert boolean(r["eligible"]) == (r["group"] != "B" and all(passed))
        close(r["balanced_score"], max(values[k] / reference[k] for k in ["overall_MAE", "daily_maximum_MAE", "rise_MAE"]))
    possible = [r for r in table if boolean(r["eligible"])]
    order = contract["methods"] + ["G_Logistic_rise_focus", "G_HGB_rise_focus"]
    if possible:
        minimum = min(float(r["balanced_score"]) for r in possible)
        near = [r for r in possible if float(r["balanced_score"]) <= minimum * 1.01]
        chosen = min(near, key=lambda r: (order.index(r["group"]), float(r["balanced_score"]))) ["group"]
    else:
        chosen = "B"
    selection = json.loads((OUT / "selection.json").read_text(encoding="utf-8"))
    assert selection["selected"] == chosen and selection["eligible_methods"] == [r["group"] for r in possible]
    assert selection["acceptance_criteria_unchanged"] and selection["no_july_august_evaluation"]
    result = {"status": "passed", "checked_at": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
        "run_sha256": sha(OUT / "run.json"), "script_sha256": sha(Path(__file__)), "prediction_rows": len(pred),
        "metric_rows": len(metrics), "trials_independently_recomputed": trial_checks,
        "prior_five_methods_preserved": True, "AP_and_confusion_independently_recomputed": True,
        "acceptance_criteria_unchanged": True, "no_new_fitting": True, "no_july_august_evaluation": True, "selected": chosen}
    (OUT / "independent_verification.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps(result), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m04_rise_gate_verify.py`

### `Modeling/scripts/m04_rise_verify.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Independent raw-time, AP, chronological selection and model checks for M04 rise.

주요 함수: `boolean()`, `raw_state()`, `ap()`, `classification()`, `stats()`, `main()`.

In [52]:
%%submission_source Modeling/scripts/m04_rise_verify.py
"""Independent raw-time, AP, chronological selection and model checks for M04 rise."""
import os
os.environ["OMP_NUM_THREADS"] = "1"
import json
import math
import sys
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from m03_verify import close, rows, sha, verify_metric

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/m04_rise"


def boolean(value):
    return value == "True"


def raw_state(record, rounded=None):
    maximum = record["maximum"]
    rounded = record["rounded"] if rounded is None else rounded
    return "zero" if maximum == 0 else "below20" if rounded < 20 else "low" if rounded <= 26 else "above26"


def ap(labels, scores):
    ordered = sorted(zip(scores, labels), reverse=True)
    tp, count, total = 0, 0, sum(labels)
    assert total > 0
    result = 0.
    grouped = defaultdict(list)
    for score, label in ordered:
        grouped[score].append(label)
    for score in sorted(grouped, reverse=True):
        group = grouped[score]
        positives = sum(group)
        tp += positives
        count += len(group)
        result += positives / total * tp / count
    return result


def classification(g, threshold):
    low = [r for r in g if boolean(r["prior_low"])]
    labels = [boolean(r["rise_event"]) for r in low]
    scores = [float(r["probability"]) for r in low]
    alarm = [p >= threshold for p in scores]
    tp = sum(a and y for a, y in zip(alarm, labels))
    fp = sum(a and not y for a, y in zip(alarm, labels))
    fn = sum(not a and y for a, y in zip(alarm, labels))
    tn = sum(not a and not y for a, y in zip(alarm, labels))
    pos = [p for p, y in zip(scores, labels) if y]
    neg = [p for p, y in zip(scores, labels) if not y]
    return {"prior_low_hours": len(low), "positive_hours": sum(labels), "prevalence": sum(labels) / len(low),
        "AP": ap(labels, scores), "AUROC": sum(float(p > n) + .5 * (p == n) for p in pos for n in neg) / (len(pos) * len(neg)),
        "Brier": math.fsum((p - y) ** 2 for p, y in zip(scores, labels)) / len(low),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn, "precision": tp / (tp + fp) if tp + fp else 0.,
        "recall": tp / (tp + fn), "false_positive_fraction": fp / (fp + tn), "threshold": threshold}


def stats(g):
    errors = [float(r["prediction"]) - float(r["actual"]) for r in g]
    rise = [e for e, r in zip(errors, g) if boolean(r["rise_event"])]
    stay = [e for e, r in zip(errors, g) if r["power_transition"] == "low->low"]
    weights = [float(r["daily_maximum_weight"]) for r in g]
    return {"overall_MAE": math.fsum(abs(e) for e in errors) / len(errors),
        "daily_maximum_MAE": math.fsum(abs(e) * w for e, w in zip(errors, weights)) / math.fsum(weights),
        "rise_MAE": math.fsum(abs(e) for e in rise) / len(rise), "rise_under": math.fsum(max(-e, 0) for e in rise) / len(rise),
        "low_stay_MAE": math.fsum(abs(e) for e in stay) / len(stay), "low_stay_over": math.fsum(max(e, 0) for e in stay) / len(stay),
        "prior_low_hours": sum(boolean(r["prior_low"]) for r in g), "rise_hours": len(rise)}


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    cpath = ROOT / "Modeling/config/m04_rise_contract.json"
    c = json.loads(cpath.read_text(encoding="utf-8"))
    run = json.loads((OUT / "run.json").read_text(encoding="utf-8"))
    assert run["status"] == "completed" and run["no_july_august_evaluation"]
    assert sha(cpath) == run["contract_sha256"] and sha(ROOT / "Modeling/scripts/m04_rise_compare.py") == run["script_sha256"]
    for key in ["inputs_sha256", "outputs_sha256", "helper_sha256"]:
        for name, expected in run[key].items():
            path = ROOT / name if key == "inputs_sha256" else OUT / name if key == "outputs_sha256" else ROOT / "Modeling/scripts" / name
            assert sha(path) == expected, (key, name)
    parent = json.loads((ROOT / "Modeling/config/m01_contract.json").read_text(encoding="utf-8"))
    raw = {}
    for r in rows(ROOT / parent["source"]):
        if not 20210101 <= int(r["날짜"]) <= 20210630 or not 0 <= int(r["시간"]) <= 23:
            continue
        t = datetime.strptime(r["날짜"], "%Y%m%d") + timedelta(hours=int(r["시간"]))
        values = [float(r[name]) for name in ["15분", "30분", "45분", "60분"]]
        assert t not in raw
        raw[t] = {"slots": values, "mean": sum(values) / 4, "maximum": max(values), "last": values[-1],
                  "production": float(r["생산량"]), "rounded": float(r["평균"])}
    eligible = {t for t in raw if all(t - timedelta(hours=lag) in raw for lag in [1,2,24,168])}
    feature_rows = rows(OUT / "feature_frame.csv")
    frame = {datetime.fromisoformat(r["timestamp"]): r for r in feature_rows}
    assert set(frame) == eligible and len(frame) == len(feature_rows)
    feature_checks = 0
    for t, r in frame.items():
        previous, previous2 = raw[t - timedelta(hours=1)], raw[t - timedelta(hours=2)]
        prior = raw_state(previous, math.floor(previous["mean"] + .5))
        event = prior == "low" and raw_state(raw[t]) == "above26"
        assert boolean(r["prior_low"]) == (prior == "low") and boolean(r["rise_event"]) == event
        assert r["power_transition"] == prior + "->" + raw_state(raw[t])
        expected = {"month": t.month, "weekend": int(t.weekday() >= 5), "hour_sin": math.sin(2 * math.pi * t.hour / 24),
            "hour_cos": math.cos(2 * math.pi * t.hour / 24), "lag1_production": previous["production"],
            "lag1_production_zero": int(previous["production"] == 0)}
        expected.update({f"dow_{day}": int(t.weekday() == day) for day in range(7)})
        for lag, record in [(1, previous), (2, previous2)]:
            for field in ["mean", "maximum"]:
                expected[f"lag{lag}_{field}"] = record[field]
        expected["lag1_last"] = previous["last"]
        slots = previous["slots"]
        expected.update({"prev_slot_rise": slots[-1] - slots[0], "prev_slot_range": max(slots) - min(slots),
            "prev_slot_slope": math.fsum(v * w for v, w in zip(slots, [-1.5,-.5,.5,1.5])) / 5,
            "last_delta": previous["last"] - previous2["last"], "mean_delta": previous["mean"] - previous2["mean"],
            "maximum_delta": previous["maximum"] - previous2["maximum"], "production_delta": previous["production"] - previous2["production"]})
        duration, gap = 0, 0
        for lag in range(1, 7):
            earlier = raw.get(t - timedelta(hours=lag))
            if earlier is None:
                gap = 1
                break
            if raw_state(earlier) != "low":
                break
            duration += 1
        expected["low_run_length6"], expected["low_history_gap6"] = duration, gap
        assert set(expected) == set(run["features"]["dynamic"])
        for name, value in expected.items():
            close(r[name], value)
            feature_checks += 1
    outer_starts = {"dev_apr": datetime(2021,4,1), "dev_may": datetime(2021,5,1), "dev_jun": datetime(2021,6,1)}
    for r in rows(OUT / "fold_audit.csv"):
        start = datetime(2021, int(r["inner_month"]), 1)
        train = [t for t in eligible if t < start]
        val = [t for t in eligible if t.month == start.month]
        assert int(r["train_hours"]) == len(train) and int(r["validation_hours"]) == len(val)
        assert datetime.fromisoformat(r["train_last"]) == max(train) < min(val)
        assert datetime.fromisoformat(r["validation_first"]) == min(val)
        assert datetime.fromisoformat(r["validation_last"]) == max(val) < outer_starts[r["split"]]
        for name, part in [("train", train), ("validation", val)]:
            assert int(r[name + "_low"]) == sum(boolean(frame[t]["prior_low"]) for t in part)
            assert int(r[name + "_rises"]) == sum(boolean(frame[t]["rise_event"]) for t in part)
    pred, inner = rows(OUT / "predictions.csv"), rows(OUT / "inner_predictions.csv")
    index = {(r["split"], r["group"], r["timestamp"]): r for r in pred}
    assert len(index) == len(pred) == 10920
    groups = {name: [r for r in pred if r["group"] == name] for name in c["methods"]}
    expected_times = {t for t in eligible if 4 <= t.month <= 6}
    assert len(expected_times) == 2184
    parent_b = {r["timestamp"]: r for r in rows(ROOT / "Modeling/tables/m03/c/predictions.csv") if r["target"] == c["target"] and r["group"] == "B"}
    for name, g in groups.items():
        assert {datetime.fromisoformat(r["timestamp"]) for r in g} == expected_times
        for r in g:
            t = datetime.fromisoformat(r["timestamp"])
            close(r["actual"], raw[t]["maximum"])
            for field in ["prior_low", "rise_event", "power_transition"]:
                assert r[field] == frame[t][field]
            for field in ["daily_maximum_weight", "profile_weight"]:
                close(r[field], parent_b[r["timestamp"]][field])
            if name == "B":
                close(r["prediction"], parent_b[r["timestamp"]]["prediction"])
            error = float(r["prediction"]) - float(r["actual"])
            for field, value in [("signed_error", error), ("absolute_error", abs(error)), ("under_amount", max(-error,0)), ("over_amount", max(error,0))]:
                close(r[field], value)
    configs = json.loads((OUT / "selected_configurations.json").read_text(encoding="utf-8"))
    trials = rows(OUT / "trials.csv")
    assert len(trials) == 450 and all(r["state"] == "COMPLETE" for r in trials)
    for config in configs:
        split = config["split"]
        selected_trials = [("weighted", config["weighted_best_trial"], {"event_weight": config["event_weight"]})]
        for family, details in config["classifiers"].items():
            selected_trials.extend([("classifier_" + family, details["classifier_best_trial"], details["parameters"]),
                                    ("gate_" + family, details["gate_best_trial"], details["gate"])])
            for role, pool in [("outer", pred), ("inner", inner)]:
                g = [r for r in pool if r["split"] == split and r["group"] == "G_" + family]
                result = classification(g, details["gate"]["threshold"])
                saved = next(r for r in rows(OUT / "classification.csv") if (r["split"],r["role"],r["model"]) == (split,role,"G_"+family))
                for key, value in result.items():
                    close(saved[key], value)
                if role == "inner":
                    for key, value in stats(g).items():
                        close(details["gate_inner"]["metrics"][key], value)
                for r in g:
                    reference = next(x for x in pool if x["split"] == split and x["group"] == "B" and x["timestamp"] == r["timestamp"])
                    expert = next(x for x in pool if x["split"] == split and x["group"] == "W_rise_weighted" and x["timestamp"] == r["timestamp"])
                    trigger = boolean(r["prior_low"]) and float(r["probability"]) >= details["gate"]["threshold"]
                    predicted = float(reference["prediction"]) + trigger * details["gate"]["blend"] * (float(expert["prediction"]) - float(reference["prediction"]))
                    close(r["prediction"], predicted)
                    if role == "outer":
                        assert boolean(r["trigger"]) == trigger
        for family, number, parameters in selected_trials:
            part = [r for r in trials if r["split"] == split and r["family"] == family]
            assert len(part) == 30
            if family.startswith("classifier_"):
                best = min(part, key=lambda r: (-float(r["objective"]), json.loads(r["metrics"])["classification"]["Brier"], int(r["trial"])))
            else:
                best = min(part, key=lambda r: (float(r["objective"]), int(r["trial"])))
            assert int(best["trial"]) == number and json.loads(best["parameters"]) == parameters
    for r in rows(OUT / "metrics.csv"):
        g = groups[r["group"]]
        kind, value = r["kind"], r["value"]
        weight_kind = "all"
        if kind in ["daily_maximum", "profile_reweighted"]:
            weight_kind = kind
            part = [x for x in g if float(x["daily_maximum_weight"]) > 0] if kind == "daily_maximum" else g
        elif kind.startswith("peak_"):
            weight_kind = "daily_maximum"
            part = [x for x in g if float(x["daily_maximum_weight"]) > 0 and x[kind[5:]] == value]
        else:
            part = g if kind == "all" else [x for x in g if x[kind] == value]
        verify_metric(r, part, weight_kind)
    table = rows(OUT / "selection_metrics.csv")
    reference_stats = stats(groups["B"])
    for r in table:
        values = stats(groups[r["group"]])
        for key, value in values.items():
            close(r[key], value)
        requirements = []
        for key, setting in [("overall_MAE", "overall_MAE_ratio_max"), ("daily_maximum_MAE", "daily_maximum_MAE_ratio_max"),
            ("low_stay_MAE", "low_stay_MAE_ratio_max"), ("low_stay_over", "low_stay_mean_over_ratio_max"),
            ("rise_MAE", "rise_MAE_ratio_max"), ("rise_under", "rise_mean_under_ratio_max")]:
            ratio = values[key] / reference_stats[key]
            passed = ratio <= c["development_selection"][setting] + 1e-12
            close(r[key + "_ratio"], ratio)
            assert boolean(r[key + "_passed"]) == passed
            requirements.append(passed)
        monthly = all(stats([x for x in groups[r["group"]] if x["split"] == split])["rise_MAE"] <=
            stats([x for x in groups["B"] if x["split"] == split])["rise_MAE"] + 1e-9 for split in outer_starts)
        assert boolean(r["rise_monthly_passed"]) == monthly
        requirements.append(monthly)
        if r["group"].startswith("G_"):
            low = [x for x in groups[r["group"]] if boolean(x["prior_low"])]
            fp = sum(boolean(x["trigger"]) and not boolean(x["rise_event"]) for x in low)
            negatives = sum(not boolean(x["rise_event"]) for x in low)
            close(r["gate_false_positive_fraction"], fp / negatives)
            passed = fp / negatives <= .05 + 1e-12
            assert boolean(r["gate_false_positive_passed"]) == passed
            requirements.append(passed)
        assert boolean(r["eligible"]) == (all(requirements) and r["group"] != "B")
        close(r["balanced_score"], max(values[k] / reference_stats[k] for k in ["overall_MAE", "daily_maximum_MAE", "rise_MAE"]))
    selection = json.loads((OUT / "selection.json").read_text(encoding="utf-8"))
    possible = [r for r in table if boolean(r["eligible"])]
    if possible:
        minimum = min(float(r["balanced_score"]) for r in possible)
        near = [r for r in possible if float(r["balanced_score"]) <= minimum * 1.01]
        chosen = min(near, key=lambda r: (c["methods"].index(r["group"]), float(r["balanced_score"])))["group"]
    else:
        chosen = "B"
    assert selection["selected"] == chosen and selection["eligible_methods"] == [r["group"] for r in possible]
    # Validate paired/daily arithmetic, not merely their stored hash.
    for r in rows(OUT / "paired_predictions.csv"):
        x, b = index[r["split"],r["group"],r["timestamp"]], index[r["split"],"B",r["timestamp"]]
        for field, label in [("absolute_error","abs"), ("under_amount","under"), ("over_amount","over")]:
            close(r["delta_" + label], float(x[field]) - float(b[field]))
    for r in rows(OUT / "daily_errors.csv"):
        g = [x for x in groups[r["group"]] if x["date"] == r["date"]]
        b = [x for x in groups["B"] if x["date"] == r["date"]]
        assert len(g) == len(b) == int(r["hours"]) == 24
        for field, label in [("absolute_error","MAE"), ("under_amount","under"), ("over_amount","over")]:
            close(r["delta_" + label], (math.fsum(float(x[field]) for x in g) - math.fsum(float(x[field]) for x in b)) / 24)
    df = pd.read_csv(OUT / "feature_frame.csv", encoding="utf-8-sig", float_precision="round_trip").set_index("timestamp")
    reloads = 0
    for r in rows(OUT / "model_manifest.csv"):
        path = ROOT / r["file"]
        assert sha(path) == r["sha256"]
        estimator = joblib.load(path)
        part = [x for x in groups["B"] if x["split"] == r["split"]]
        inputs = df.loc[[x["timestamp"] for x in part], run["features"]["dynamic"]]
        if r["name"].endswith("_classifier"):
            family = r["name"].split("_")[0]
            expected = [float(index[r["split"],"G_" + family,x["timestamp"]]["probability"]) for x in part]
            actual = np.full(len(part), estimator["constant_probability"]) if isinstance(estimator, dict) else estimator.predict_proba(inputs)[:,1]
            if family == "Logistic" and not isinstance(estimator, dict):
                training = [t for t in frame if t < outer_starts[r["split"]] and boolean(frame[t]["prior_low"])]
                matrix = df.loc[[str(t) for t in training], run["features"]["dynamic"]]
                assert np.allclose(estimator[0].mean_, matrix.mean().to_numpy(), atol=1e-9, rtol=1e-10)
        else:
            expected = [float(index[r["split"],r["name"],x["timestamp"]]["prediction"]) for x in part]
            actual = estimator.predict(inputs)
        assert np.allclose(actual, expected, atol=1e-9, rtol=0)
        reloads += 1
    result = {"status": "passed", "checked_at": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
        "run_sha256": sha(OUT / "run.json"), "script_sha256": sha(Path(__file__)), "raw_feature_values_checked": feature_checks,
        "prediction_rows": len(pred), "metric_rows": len(rows(OUT / "metrics.csv")), "trials": len(trials),
        "model_reload_checks": reloads, "selected": chosen, "no_july_august_evaluation": True,
        "AP_and_confusion_independently_recomputed": True, "training_only_scaler_checked": True}
    (OUT / "independent_verification.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps(result), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m04_rise_verify.py`

### `Modeling/scripts/m04_surge_calibration.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: One cached negative-tail alarm calibration, preserving models and selection requirements.

주요 함수: `negative_cutoff()`, `main()`.

In [53]:
%%submission_source Modeling/scripts/m04_surge_calibration.py
"""One cached negative-tail alarm calibration, preserving models and selection requirements."""
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from m01_prepare import ROOT,sha
from m04_rise_compare import now,read_csv
from m04_surge_diagnose import alarm_metrics,confusion

OUT=ROOT/"Modeling/tables/m04_surge_calibration"
PARENT=ROOT/"Modeling/tables/m04_surge_risk"
CONTRACT=ROOT/"Modeling/config/m04_surge_calibration_contract.json"


def negative_cutoff(labels,scores,cap):
    negative=np.asarray(scores)[~np.asarray(labels,dtype=bool)]
    ordered=np.sort(negative)[::-1]
    k=int(np.floor(cap*len(ordered)))
    threshold=np.nextafter(ordered[k],np.inf) if k<len(ordered) else np.nextafter(ordered[-1],-np.inf)
    return float(threshold),len(ordered),k


def main():
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    assert not (OUT/"run.json").exists(),"Preserve completed comparison."
    c=json.loads(CONTRACT.read_text(encoding="utf-8"))
    parent_run=json.loads((PARENT/"run.json").read_text(encoding="utf-8"))
    verified=json.loads((PARENT/"independent_verification.json").read_text(encoding="utf-8"))
    assert verified["status"]=="passed" and verified["run_sha256"]==sha(PARENT/"run.json")==c["parent_run_sha256"]
    for name,digest in parent_run["outputs_sha256"].items():
        assert sha(PARENT/name)==digest,name
    p=read_csv(PARENT/"predictions.csv",["timestamp","date"])
    inner=read_csv(PARENT/"inner_predictions.csv",["timestamp","date"])
    risk_contract=json.loads((ROOT/"Modeling/config/m04_surge_risk_contract.json").read_text(encoding="utf-8"))
    policy_rows,alarm_rows=[] ,[]
    for (split,name),pool in p.groupby(["split","group"]):
        past=inner.loc[inner.split.eq(split)&inner.group.eq(name)]
        assert past.timestamp.max()<pool.timestamp.min()
        for cap in [.01,.05]:
            threshold,negative_count,k=negative_cutoff(past.increase.ge(93),past.score,cap)
            inside=confusion(past.increase.ge(93),past.score,threshold)
            assert inside["FP"]<=k and inside["false_positive_fraction"]<=cap+1e-12
            policy_rows.append({"split":split,"group":name,"cap":cap,"negative_count":negative_count,"budget_count":k,
                "inner_last":str(past.timestamp.max()),"outer_first":str(pool.timestamp.min()),**inside})
            p.loc[pool.index,"alarm_"+str(cap)]=pool.score.ge(threshold).to_numpy()
            for label,boundary in [("q95",93.),("q90",risk_contract["boundary_sensitivity"])]:
                alarm_rows.append({"split":split,"group":name,"cap":cap,"boundary_name":label,
                    **alarm_metrics(pool.increase.ge(boundary),pool.score,threshold)})
    alarm_frame=pd.DataFrame(alarm_rows)
    original=read_csv(PARENT/"selection_metrics.csv")
    table=[]
    for name,g in p.groupby("group"):
        row=original.loc[original.group.eq(name)].iloc[0].to_dict()
        labels=g.increase.ge(93).to_numpy()
        selected=g["alarm_0.01"].to_numpy(dtype=bool)
        tp=int((selected&labels).sum());fp=int((selected&~labels).sum())
        row.update({"TP":tp,"FP":fp,"recall":tp/int(labels.sum()),"precision":tp/(tp+fp) if tp+fp else 0.,"false_positive_fraction":fp/int((~labels).sum())})
        table.append(row)
    table=pd.DataFrame(table)
    reference=table.loc[table.group.eq("B_score")].iloc[0]
    for i,r in table.iterrows():
        monthly=alarm_frame.loc[alarm_frame.group.eq(r.group)&alarm_frame.cap.eq(.01)&alarm_frame.boundary_name.eq("q95")].set_index("split")
        baseline=alarm_frame.loc[alarm_frame.group.eq("B_score")&alarm_frame.cap.eq(.01)&alarm_frame.boundary_name.eq("q95")].set_index("split")
        flags={"rank_passed":bool(r.weighted_monthly_AP>=1.05*reference.weighted_monthly_AP),"recall_passed":bool(r.TP>=reference.TP),
            "monthly_recall_passed":bool((monthly.TP>=baseline.TP).all()),"false_positive_passed":bool(r.false_positive_fraction<=.01+1e-12),
            "strict_gain_passed":bool(r.TP>reference.TP or r.FP<reference.FP)}
        for name,value in flags.items():
            table.loc[i,name]=value
        table.loc[i,"eligible"]=r.group in ["Logistic_all","HGB_all","Q90_upper"] and all(flags.values())
    eligible=table.loc[table.eligible.astype(bool)]
    if len(eligible):
        near=eligible.loc[eligible.weighted_monthly_AP.ge(eligible.weighted_monthly_AP.max()/1.01)].copy()
        near["preference"]=near.group.map({name:i for i,name in enumerate(["Logistic_all","HGB_all","Q90_upper"])})
        chosen=near.sort_values(["preference","weighted_monthly_AP"],ascending=[True,False]).iloc[0].group
    else:
        chosen="B_score"
    selection={"selected_risk_channel":chosen,"normal_forecast":"B","eligible_methods":eligible.group.tolist(),
        "selection_gates_unchanged":True,"no_new_fitting":True,"development_only":True,"no_july_august_evaluation":True}
    OUT.mkdir(parents=True,exist_ok=True)
    for frame,name in [(p,"predictions.csv"),(pd.DataFrame(policy_rows),"alarm_policies.csv"),(alarm_frame,"alarm_metrics.csv"),(table,"selection_metrics.csv")]:
        frame.to_csv(OUT/name,index=False,encoding="utf-8-sig")
    (OUT/"selection.json").write_text(json.dumps(selection,indent=2),encoding="utf-8")
    run={"status":"completed","finished":now(),"script_sha256":sha(Path(__file__)),"contract_sha256":sha(CONTRACT),
        "inputs_sha256":{path.relative_to(ROOT).as_posix():sha(path) for path in [PARENT/"run.json",PARENT/"independent_verification.json",PARENT/"predictions.csv",PARENT/"inner_predictions.csv",PARENT/"selection_metrics.csv",ROOT/"Modeling/config/m04_surge_risk_contract.json"]},
        "outputs_sha256":{path.name:sha(path) for path in OUT.iterdir()},"prediction_rows":len(p),"normal_forecast_unchanged":True,
        "selection_gates_unchanged":True,"no_new_fitting":True,"no_july_august_evaluation":True}
    (OUT/"run.json").write_text(json.dumps(run,indent=2),encoding="utf-8")
    print(table[["group","AP","weighted_monthly_AP","TP","FP","recall","precision","false_positive_fraction","eligible"]].to_string(index=False),flush=True)
    print(json.dumps(selection),flush=True)


if __name__=="__main__":
    main()


코드 등록 완료: `Modeling/scripts/m04_surge_calibration.py`

### `Modeling/scripts/m04_surge_calibration_verify.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Independent order-statistic cutoff and frozen-score/selection verification.

주요 함수: `main()`.

In [54]:
%%submission_source Modeling/scripts/m04_surge_calibration_verify.py
"""Independent order-statistic cutoff and frozen-score/selection verification."""
import json
import math
import sys
from collections import defaultdict
from datetime import datetime,timedelta,timezone
from pathlib import Path

from m03_verify import rows,sha,close
from m04_surge_verify import alarm,check_alarm

ROOT=Path(__file__).resolve().parents[2]
OUT=ROOT/"Modeling/tables/m04_surge_calibration"
PARENT=ROOT/"Modeling/tables/m04_surge_risk"


def main():
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    run=json.loads((OUT/"run.json").read_text(encoding="utf-8"))
    cpath=ROOT/"Modeling/config/m04_surge_calibration_contract.json"
    c=json.loads(cpath.read_text(encoding="utf-8"))
    risk=json.loads((ROOT/"Modeling/config/m04_surge_risk_contract.json").read_text(encoding="utf-8"))
    verified=json.loads((PARENT/"independent_verification.json").read_text(encoding="utf-8"))
    assert verified["status"]=="passed" and verified["run_sha256"]==sha(PARENT/"run.json")==c["parent_run_sha256"]
    assert run["status"]=="completed" and run["no_new_fitting"] and run["selection_gates_unchanged"] and run["normal_forecast_unchanged"]
    assert sha(cpath)==run["contract_sha256"] and sha(ROOT/"Modeling/scripts/m04_surge_calibration.py")==run["script_sha256"]
    for field in ["inputs_sha256","outputs_sha256"]:
        for name,digest in run[field].items():
            assert sha(ROOT/name if field=="inputs_sha256" else OUT/name)==digest
    parent={(r["split"],r["group"],r["timestamp"]):r for r in rows(PARENT/"predictions.csv")}
    pred=rows(OUT/"predictions.csv")
    inner=rows(PARENT/"inner_predictions.csv")
    parts,inside,groups=defaultdict(list),defaultdict(list),defaultdict(list)
    for r in pred:
        original=parent[r["split"],r["group"],r["timestamp"]]
        for field,value in original.items():
            if field.startswith("alarm_"):
                continue
            assert r[field]==value,(field,r["timestamp"])
        parts[r["split"],r["group"]].append(r)
        groups[r["group"]].append(r)
    assert len(pred)==run["prediction_rows"]==10920
    for r in inner:
        inside[r["split"],r["group"]].append(r)
    policies=rows(OUT/"alarm_policies.csv")
    for r in policies:
        pool=inside[r["split"],r["group"]]
        negatives=sorted([float(x["score"]) for x in pool if float(x["increase"])<93],reverse=True)
        k=math.floor(float(r["cap"])*len(negatives))
        threshold=math.nextafter(negatives[k],math.inf) if k<len(negatives) else math.nextafter(negatives[-1],-math.inf)
        close(r["threshold"],threshold)
        assert len(negatives)==int(r["negative_count"]) and k==int(r["budget_count"])
        assert max(x["timestamp"] for x in pool)==r["inner_last"]<r["outer_first"]
        values=alarm([float(x["increase"])>=93 for x in pool],[float(x["score"]) for x in pool],threshold)
        assert values["FP"]<=k and values["false_positive_fraction"]<=float(r["cap"])+1e-12
        for key,value in values.items():
            close(r[key],value)
        for x in parts[r["split"],r["group"]]:
            assert (x["alarm_"+r["cap"]]=="True")== (float(x["score"])>=threshold)
    alarms=rows(OUT/"alarm_metrics.csv")
    for r in alarms:
        pool=parts[r["split"],r["group"]]
        threshold=next(p["threshold"] for p in policies if p["split"]==r["split"] and p["group"]==r["group"] and p["cap"]==r["cap"])
        close(r["threshold"],threshold)
        check_alarm(r,pool,[float(x["score"]) for x in pool],93 if r["boundary_name"]=="q95" else risk["boundary_sensitivity"])
    table=rows(OUT/"selection_metrics.csv")
    original_table={r["group"]:r for r in rows(PARENT/"selection_metrics.csv")}
    reference=next(r for r in table if r["group"]=="B_score")
    for r in table:
        for field in ["AP","AUROC","events","hours","weighted_monthly_AP"]:
            close(r[field],original_table[r["group"]][field])
        pool=groups[r["group"]]
        tp=sum(x["alarm_0.01"]=="True" and float(x["increase"])>=93 for x in pool)
        fp=sum(x["alarm_0.01"]=="True" and float(x["increase"])<93 for x in pool)
        close(r["TP"],tp);close(r["FP"],fp)
        close(r["recall"],tp/8);close(r["precision"],tp/(tp+fp) if tp+fp else 0)
        close(r["false_positive_fraction"],fp/2176)
        monthly=[x for x in alarms if x["group"]==r["group"] and x["cap"]=="0.01" and x["boundary_name"]=="q95"]
        monthly_ok=all(int(x["TP"])>=int(next(b for b in alarms if b["group"]=="B_score" and b["split"]==x["split"] and b["cap"]=="0.01" and b["boundary_name"]=="q95")["TP"]) for x in monthly)
        flags={"rank_passed":float(r["weighted_monthly_AP"])>=1.05*float(reference["weighted_monthly_AP"]),"recall_passed":tp>=int(reference["TP"]),
            "monthly_recall_passed":monthly_ok,"false_positive_passed":fp/2176<=.01+1e-12,"strict_gain_passed":tp>int(reference["TP"]) or fp<int(reference["FP"])}
        for key,value in flags.items():
            assert (r[key]=="True")==value
        assert (r["eligible"]=="True")== (r["group"] in ["Logistic_all","HGB_all","Q90_upper"] and all(flags.values()))
    eligible=[r for r in table if r["eligible"]=="True"]
    if eligible:
        best=max(float(r["weighted_monthly_AP"]) for r in eligible)
        near=[r for r in eligible if float(r["weighted_monthly_AP"])>=best/1.01]
        chosen=min(near,key=lambda r:(["Logistic_all","HGB_all","Q90_upper"].index(r["group"]),-float(r["weighted_monthly_AP"]))) ["group"]
    else:
        chosen="B_score"
    selection=json.loads((OUT/"selection.json").read_text(encoding="utf-8"))
    assert selection["selected_risk_channel"]==chosen and selection["normal_forecast"]=="B" and selection["selection_gates_unchanged"]
    assert len(policies)==30 and len(alarms)==60
    result={"status":"passed","checked_at":datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
        "run_sha256":sha(OUT/"run.json"),"script_sha256":sha(Path(__file__)),"prediction_rows":len(pred),"policy_rows":len(policies),
        "alarm_rows":len(alarms),"scores_and_models_unchanged":True,"normal_forecast_unchanged":True,"selection_gates_unchanged":True,
        "selected_risk_channel":chosen,"no_new_fitting":True,"no_july_august_evaluation":True}
    (OUT/"independent_verification.json").write_text(json.dumps(result,indent=2),encoding="utf-8")
    print(json.dumps(result),flush=True)


if __name__=="__main__":
    main()


코드 등록 완료: `Modeling/scripts/m04_surge_calibration_verify.py`

### `Modeling/scripts/m04_surge_diagnose.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Frozen January surge boundary, existing errors, alarm policies and April gate diagnosis.

주요 함수: `confusion()`, `threshold_at_cap()`, `alarm_metrics()`, `error_metrics()`, `main()`.

In [55]:
%%submission_source Modeling/scripts/m04_surge_diagnose.py
"""Frozen January surge boundary, existing errors, alarm policies and April gate diagnosis."""
import os
os.environ["OMP_NUM_THREADS"] = "1"
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from m01_prepare import ROOT, sha
from m04_rise_compare import now, read_csv

OUT = ROOT / "Modeling/tables/m04_surge"
CONTRACT = ROOT / "Modeling/config/m04_surge_contract.json"


def confusion(y, score, threshold):
    y, score = np.asarray(y, dtype=bool), np.asarray(score, dtype=float)
    alarm = score >= threshold
    tp, fp, fn, tn = [int(v.sum()) for v in [alarm & y, alarm & ~y, ~alarm & y, ~alarm & ~y]]
    return {"hours": len(y), "events": int(y.sum()), "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "recall": tp / (tp + fn) if tp + fn else 0., "precision": tp / (tp + fp) if tp + fp else 0.,
        "false_positive_fraction": fp / (fp + tn) if fp + tn else 0., "threshold": float(threshold)}


def threshold_at_cap(y, score, cap):
    y, score = np.asarray(y, dtype=bool), np.asarray(score, dtype=float)
    choices = [np.nextafter(score.max(), np.inf)] + sorted(set(score), reverse=True)
    values = [confusion(y, score, threshold) for threshold in choices]
    eligible = [r for r in values if r["false_positive_fraction"] <= cap + 1e-12]
    return min(eligible, key=lambda r: (-r["TP"], r["FP"], -r["threshold"]))


def alarm_metrics(y, score, threshold):
    return {**confusion(y, score, threshold), "AP": float(average_precision_score(y, score)),
            "AUROC": float(roc_auc_score(y, score)), "prevalence": float(np.mean(y))}


def error_metrics(part):
    error = part.prediction.to_numpy() - part.actual.to_numpy()
    return {"hours": len(part), "dates": part.date.nunique(), "MAE": float(np.abs(error).mean()),
        "RMSE": float(np.sqrt(np.square(error).mean())), "mean_under": float(np.maximum(-error, 0).mean()),
        "mean_over": float(np.maximum(error, 0).mean()), "under_hours": int((error < 0).sum()),
        "mean_actual": float(part.actual.mean()), "mean_increase": float(part.increase.mean())}


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    assert not (OUT / "run.json").exists(), "Preserve completed diagnosis."
    c = json.loads(CONTRACT.read_text(encoding="utf-8"))
    first = ROOT / "Modeling/tables/m04_rise"
    parent = ROOT / "Modeling/tables/m04_rise_gate"
    for directory, expected in [(first, c["first_run_sha256"]), (parent, c["parent_run_sha256"])]:
        verified = json.loads((directory / "independent_verification.json").read_text(encoding="utf-8"))
        assert verified["status"] == "passed" and verified["run_sha256"] == sha(directory / "run.json") == expected
        run = json.loads((directory / "run.json").read_text(encoding="utf-8"))
        for name, digest in run["outputs_sha256"].items():
            assert sha(directory / name) == digest, name
    frame_path = ROOT / "Modeling/tables/m01/hourly_frame.csv"
    assert sha(frame_path) == c["frame_sha256"] and sha(ROOT / "data/origin/okm_augumented_2021.csv") == c["source_sha256"]
    f = read_csv(frame_path, ["timestamp", "date"])
    f = f.loc[f.eligible_common & f.timestamp.lt("2021-07-01")].copy()
    f["increase"] = f.target_maximum - f.lag1_maximum
    calibration = f.loc[f.timestamp.lt("2021-02-01")]
    positive = calibration.loc[calibration.increase.gt(0), "increase"]
    boundaries = {"q95": float(positive.quantile(.95, interpolation="linear")), "q90": float(positive.quantile(.90, interpolation="linear"))}
    assert boundaries["q95"] >= boundaries["q90"] > 0
    p = read_csv(parent / "predictions.csv", ["timestamp", "date"])
    p = p.merge(f[["timestamp", "lag1_maximum", "increase"]], on="timestamp", validate="many_to_one")
    p["surge"] = p.increase.ge(boundaries["q95"])
    p["predicted_increase"] = p.prediction - p.lag1_maximum
    counts, errors, alarms, policies = [], [], [], []
    for boundary_name, boundary in boundaries.items():
        for role, part in [("calibration_jan", calibration), ("train_jan_mar", f.loc[f.timestamp.lt("2021-04-01")]),
            ("dev_apr", f.loc[f.timestamp.dt.month.eq(4)]), ("dev_may", f.loc[f.timestamp.dt.month.eq(5)]),
            ("dev_jun", f.loc[f.timestamp.dt.month.eq(6)])]:
            event = part.increase.ge(boundary)
            old = (np.floor(part.lag1_mean + .5).between(20,26) & part.lag1_maximum.gt(0) & part.diag_target_rounded_mean.gt(26))
            counts.append({"boundary_name": boundary_name, "boundary": boundary, "role": role, "hours": len(part),
                "positive_increases": int(part.increase.gt(0).sum()), "surge_hours": int(event.sum()),
                "surge_dates": part.loc[event, "date"].nunique(), "old_rise_hours": int(old.sum()),
                "overlap_hours": int((old & event).sum()), "outside_prior_low_surges": int((event & ~np.floor(part.lag1_mean + .5).between(20,26)).sum()),
                "previous_zero_hours": int(part.lag1_maximum.eq(0).sum())})
        for name, g in p.groupby("group"):
            for split, part in [("pooled_development", g)] + list(g.groupby("split")):
                for condition, mask in [("all", np.ones(len(part), dtype=bool)), ("surge", part.increase.ge(boundary)),
                    ("non_surge", part.increase.lt(boundary)), ("legacy_rise", part.rise_event),
                    ("low_stay", part.power_transition.eq("low->low"))]:
                    selected = part.loc[mask]
                    if len(selected):
                        errors.append({"boundary_name": boundary_name, "boundary": boundary, "group": name,
                            "split": split, "condition": condition, **error_metrics(selected)})
                alarms.append({"boundary_name": boundary_name, "group": name, "split": split,
                    "score": "predicted_increase", "policy": "frozen_magnitude_boundary", "cap": np.nan,
                    **alarm_metrics(part.increase.ge(boundary), part.predicted_increase, boundary)})
    old_inner = read_csv(first / "inner_predictions.csv", ["timestamp", "date"])
    old_inner = old_inner.merge(f[["timestamp", "lag1_maximum", "increase"]], on="timestamp", validate="many_to_one")
    primary = boundaries["q95"]
    for score_name, group in [("B_predicted_increase", "B"), ("legacy_HGB_probability", "G_HGB")]:
        for split in ["dev_apr", "dev_may", "dev_jun"]:
            iv = old_inner.loc[old_inner.group.eq(group) & old_inner.split.eq(split)].copy()
            ev = p.loc[p.group.eq(group) & p.split.eq(split)].copy()
            inner_score = iv.prediction - iv.lag1_maximum if group == "B" else np.where(iv.prior_low, iv.probability, 0.)
            outer_score = ev.predicted_increase if group == "B" else np.where(ev.prior_low, ev.probability, 0.)
            assert iv.timestamp.max() < ev.timestamp.min()
            for cap in c["alarm"]["fixed_false_positive_caps"]:
                policy = threshold_at_cap(iv.increase.ge(primary), inner_score, cap)
                policies.append({"split": split, "score": score_name, "cap": cap, "inner_first": str(iv.timestamp.min()),
                    "inner_last": str(iv.timestamp.max()), "outer_first": str(ev.timestamp.min()), **policy})
                alarms.append({"boundary_name": "q95", "group": group, "split": split, "score": score_name,
                    "policy": "past_inner_selected", "cap": cap, **alarm_metrics(ev.increase.ge(primary), outer_score, policy["threshold"])})
                frontier = threshold_at_cap(ev.increase.ge(primary), outer_score, cap)
                alarms.append({"boundary_name": "q95", "group": group, "split": split, "score": score_name,
                    "policy": "descriptive_outer_frontier", "cap": cap, **alarm_metrics(ev.increase.ge(primary), outer_score, frontier["threshold"])})
    configs = json.loads((first / "selected_configurations.json").read_text(encoding="utf-8"))
    gate = next(r for r in configs if r["split"] == "dev_apr")["classifiers"]["HGB"]["gate"]
    april = p.loc[p.group.eq("G_HGB") & p.split.eq("dev_apr") & p.prior_low].copy()
    pool_scores = april.probability.to_numpy()
    indexed = p.set_index(["group", "timestamp"])
    diagnostic = []
    for _, r in april.loc[april.rise_event].iterrows():
        b = float(indexed.loc[("B", r.timestamp), "prediction"])
        w = float(indexed.loc[("W_rise_weighted", r.timestamp), "prediction"])
        diagnostic.append({"timestamp": r.timestamp, "actual": r.actual, "previous_maximum": r.lag1_maximum,
            "increase": r.increase, "surge": bool(r.surge), "probability": r.probability,
            "rank_min": 1 + int((pool_scores > r.probability).sum()), "rank_max": int((pool_scores >= r.probability).sum()),
            "ranking_pool_hours": len(april), "threshold": gate["threshold"], "blend": gate["blend"], "trigger": bool(r.trigger),
            "prediction_B": b, "prediction_expert": w, "prediction_combined": r.prediction,
            "expert_minus_B": w - b, "counterfactual_blend_one": w if r.trigger else b,
            "B_under": max(r.actual-b,0), "expert_under": max(r.actual-w,0)})
    assert len(diagnostic) == 8
    OUT.mkdir(parents=True, exist_ok=True)
    for data, filename in [(counts,"event_counts.csv"), (errors,"errors.csv"), (alarms,"alarm_metrics.csv"),
        (policies,"alarm_policies.csv"), (diagnostic,"april_legacy_cases.csv")]:
        pd.DataFrame(data).to_csv(OUT / filename, index=False, encoding="utf-8-sig")
    p.to_csv(OUT / "predictions.csv", index=False, encoding="utf-8-sig")
    summary = {"boundary_primary": primary, "boundary_sensitivity": boundaries["q90"],
        "calibration_hours": len(calibration), "calibration_positive_hours": len(positive),
        "calibration_first": str(calibration.timestamp.min()), "calibration_last": str(calibration.timestamp.max()),
        "development_surges": int(p.loc[p.group.eq("B"), "surge"].sum()),
        "april_old_cases": len(diagnostic), "april_old_cases_above_gate": sum(r["trigger"] for r in diagnostic),
        "april_old_rank_min": min(r["rank_min"] for r in diagnostic), "april_old_rank_max": max(r["rank_max"] for r in diagnostic),
        "no_fitting": True, "no_july_august_evaluation": True}
    (OUT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    run = {"status": "completed", "finished": now(), "contract_sha256": sha(CONTRACT), "script_sha256": sha(Path(__file__)),
        "inputs_sha256": {str(path.relative_to(ROOT).as_posix()): sha(path) for path in [frame_path, first/"run.json", first/"inner_predictions.csv", first/"selected_configurations.json", parent/"run.json", parent/"predictions.csv"]},
        "outputs_sha256": {path.name: sha(path) for path in OUT.iterdir()}, "prediction_rows": len(p),
        "no_fitting": True, "no_july_august_evaluation": True}
    (OUT / "run.json").write_text(json.dumps(run, indent=2), encoding="utf-8")
    print(json.dumps(summary), flush=True)
    print(pd.DataFrame(errors).query("boundary_name=='q95' and split=='pooled_development' and condition=='surge'")[["group","hours","MAE","mean_under","mean_over"]].to_string(index=False), flush=True)
    print(pd.DataFrame(diagnostic)[["timestamp","increase","probability","rank_min","rank_max","trigger","prediction_B","prediction_expert"]].to_string(index=False), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m04_surge_diagnose.py`

### `Modeling/scripts/m04_surge_documents.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Document verified surge diagnosis, risk comparison and one calibration follow-up.

주요 함수: `verified_tables()`, `error_table()`, `risk_table()`, `month_table()`, `upper_summary()`, `section()`, `write()`, `verify()`, `main()`.

In [56]:
%%submission_source Modeling/scripts/m04_surge_documents.py
"""Document verified surge diagnosis, risk comparison and one calibration follow-up."""
import argparse
import json
import re
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

from m03_verify import rows, sha

ROOT = Path(__file__).resolve().parents[2]
MD = ROOT / 'Modeling/04_Modeling_원고.md'
LOG = ROOT / 'Modeling/10.05_M04_증가량급등과위험출력_실행기록.md'
TITLE = '## 4.8 M04 후속: 일반 급등의 재평가와 별도 위험 출력'
NAMES = ['B_score', 'G_HGB_score', 'HGB_all', 'Logistic_all', 'Q90_upper']
LABELS = dict(zip(NAMES, ['B 증가 점수', '기존 HGB 결합 증가 점수', '전체 상태 HGB 분류', '전체 상태 Logistic 분류', 'Q90 상단 증가 점수']))
NEW = {
    'm04_surge_diagnose.py': '증가량 경계 고정·기존 7방법 재평가·4월 순위/임계값/전문가 진단',
    'm04_surge_verify.py': '원본 정확한 시차·분위수·급등 오차·AP/혼동행렬 독립 검증',
    'm04_surge_risk.py': '전체 상태 분류의 120회 Optuna·Q90 보조 학습·위험 출력 비교',
    'm04_surge_risk_verify.py': '120회 내부 OOF·시간 경계·모델 재로딩·모든 선택 요건 검증',
    'm04_surge_calibration.py': '고정된 내부 음성 꼬리 경계로 한 번의 경보 보정; 새 학습 없음',
    'm04_surge_calibration_verify.py': '순서통계량 경계·복사 예측·경보·변경 없는 선택 요건 독립 검증',
    'm04_surge_documents.py': '4.8 상세 원고·기록·README 작성과 표/링크 검증; --write 또는 --verify',
}


def verified_tables():
    for folder in ['m04_surge', 'm04_surge_risk', 'm04_surge_calibration']:
        directory = ROOT / 'Modeling/tables' / folder
        run = json.loads((directory / 'run.json').read_text(encoding='utf-8'))
        independent = json.loads((directory / 'independent_verification.json').read_text(encoding='utf-8'))
        assert run['status'] == 'completed' and independent['status'] == 'passed'
        assert independent['run_sha256'] == sha(directory / 'run.json')
        for name, expected in run['outputs_sha256'].items():
            assert sha(directory / name) == expected, (folder, name)


def error_table():
    records = rows(ROOT / 'Modeling/tables/m04_surge/errors.csv')
    table = '| 기존 방법 | 증가량≥93의 MAE | 증가량≥61.6의 MAE |\n|---|---:|---:|\n'
    for name in ['B', 'D_dynamic', 'W_rise_weighted', 'G_Logistic', 'G_HGB', 'G_Logistic_rise_focus', 'G_HGB_rise_focus']:
        values = [next(r for r in records if r['boundary_name'] == boundary and r['group'] == name
                       and r['split'] == 'pooled_development' and r['condition'] == 'surge') for boundary in ['q95', 'q90']]
        table += f"| {name} | {float(values[0]['MAE']):.3f} | {float(values[1]['MAE']):.3f} |\n"
    return table


def risk_table():
    before = {r['group']: r for r in rows(ROOT / 'Modeling/tables/m04_surge_risk/selection_metrics.csv')}
    after = {r['group']: r for r in rows(ROOT / 'Modeling/tables/m04_surge_calibration/selection_metrics.csv')}
    table = '| 위험 출력 | 월별 AP의 사건 수 가중 평균 | 첫 경계 TP/FP | 음성 꼬리 경계 TP/FP | 재현율 | 정밀도 | 실제 오경보율 |\n|---|---:|---:|---:|---:|---:|---:|\n'
    for name in NAMES:
        a, b = before[name], after[name]
        table += f"| {LABELS[name]} | {float(b['weighted_monthly_AP']):.3f} | {a['TP']}/{a['FP']} | {b['TP']}/{b['FP']} | {100*float(b['recall']):.1f}% | {100*float(b['precision']):.1f}% | {100*float(b['false_positive_fraction']):.3f}% |\n"
    return table


def month_table():
    records = rows(ROOT / 'Modeling/tables/m04_surge_calibration/alarm_metrics.csv')
    table = '| 월 | 급등 수 | B 점수 TP/FP | 기존 HGB 결합 점수 TP/FP | 전체 상태 HGB TP/FP |\n|---|---:|---:|---:|---:|\n'
    for split, label in [('dev_apr', '4월'), ('dev_may', '5월'), ('dev_jun', '6월')]:
        values = [next(r for r in records if r['split'] == split and r['group'] == name
                       and r['cap'] == '0.01' and r['boundary_name'] == 'q95') for name in ['B_score', 'G_HGB_score', 'HGB_all']]
        table += f"| {label} | {values[0]['events']} | " + ' | '.join(f"{r['TP']}/{r['FP']}" for r in values) + ' |\n'
    return table


def upper_summary():
    records = [r for r in rows(ROOT / 'Modeling/tables/m04_surge_risk/regression_metrics.csv') if r['group'] == 'Q90_upper']
    def pooled(condition, key):
        rs = [r for r in records if r['condition'] == condition]
        return sum(int(r['hours'])*float(r[key]) for r in rs)/sum(int(r['hours']) for r in rs)
    coverage = [next(r for r in records if r['split'] == split and r['condition'] == 'all')['coverage']
                for split in ['dev_apr', 'dev_may', 'dev_jun']]
    return (pooled('surge_q95', 'MAE'), pooled('surge_q95', 'mean_under'), pooled('all', 'MAE'),
            pooled('all', 'coverage'), pooled('surge_q95', 'coverage'), [100*float(x) for x in coverage])


def section():
    u_mae, u_under, all_mae, coverage, event_coverage, monthly_cov = upper_summary()
    return f'''{TITLE}

4.7의 결합은 낮은 상태에서 26초과로 바뀐 20시간의 오차를 일부 줄였지만, 그 정의는 시간당 큰 전력 증가 자체와 달랐다.
이번 질문은 '직전 상태가 낮지 않아도 발생하는 큰 증가에서 개선이 남는가, 4월 미탐지는 어디에서 생기는가, 전력 크기 예측을 유지하면서 별도 위험 출력으로 더 잘 찾을 수 있는가'다. [실행 기록](10.05_M04_증가량급등과위험출력_실행기록.md)에 진단→최소 재학습→한 번의 경보 보정 순서를 남겼다. 앞선 실패를 삭제하거나 허용 범위를 늘려 통과시킨 작업은 아니다.

### 급등을 학습 자료에서 먼저 정의

급등은 `다음 시간 최대값 - 정확히 직전 시간 최대값 ≥ 93`으로 정했다. 1월 공통 비교 576시간에서 양의 증가 233건의 95분위수가 93이었다. 같은 학습 자료의 90분위수 61.6은 민감도 점검에만 사용했다. 1월 공통 표본은 시차 168시간 때문에 1월8일부터 시작한다. 시간 누락을 건너뛰어 차이를 만들지 않으며 임의 단위를 붙이지 않는다. [진단 조건](config/m04_surge_contract.json)은 계산 전에 기록했다.

같은 개발 2,184시간·91일에서 주 급등은 8시간·8일(4월2·5월4·6월2)이었고, 민감도 급등은 38시간(11·13·14)이었다. 8시간 중 기존 low→above26 전환과 겹치는 것은 3시간뿐이며 5시간은 직전 낮은 상태가 아니다. 38시간에서는 17시간이 겹치고 21시간은 직전 낮은 상태 밖이다. 따라서 prior-low에서만 학습한 분류기는 범위상 모든 급등을 다루기 어렵다. 1~3월 학습의 주 급등도 15시간·7일에 불과하다. 1~6월 공통 표본에는 직전 최대값이 0인 사례가 없어 이번 결과는 0 이후 재가동의 성능을 입증하지 않는다. [사건 수](tables/m04_surge/event_counts.csv)를 함께 보존한다.

### 기존 예측을 새 정의에서 재평가

이번 표는 모델을 다시 학습하지 않고 기존 7방법의 같은 예측을 사용한 결과다. 주 급등 8건과 민감도 급등 38건의 분모를 섞지 않는다.

{error_table()}
B에서 주 급등 8건은 모두 과소예측되었고 MAE와 평균 과소예측은 38.586이었다. 기존 HGB 분류 결합은 둘 다 31.125로 19.3% 줄였으며, 민감도 급등 MAE도 30.444→25.941로 14.8% 줄였다. 이는 급등 크기의 오차 개선이다. 예측 증가가 93 이상일 때만 경보를 내는 직접 규칙에서는 B와 기존 결합 모두 TP2·FP2·FN6으로 같았다. 크기를 덜 작게 예상해도 경계를 넘지 않으면 탐지 수는 늘지 않는다. 또한 기존 결합은 4.7의 낮은 유지 MAE 허용 요건에 실패했으므로 이 결과만으로 정상 전력 예측을 교체하지 않는다. [전체 오류표](tables/m04_surge/errors.csv)와 [직접/보정 경보표](tables/m04_surge/alarm_metrics.csv)를 연결한다.

### 4월 미탐지를 순위·경계·전문가로 구분

4월의 기존 상승 8시간은 prior-low 200시간 중 확률 순위 9~34위였다. 확률은 0.00377~0.06329로 선택 경계 0.73860보다 모두 낮았고, 결합 비율도 약 0.00129였다. 이 8시간 중 새 주 급등 정의를 만족한 것은 4월9일07시 한 건이었다. 해당 시간은 실제 증가 104인데 B 최대 예측은 32.474, 상승 전문가 예측은 34.951이었다. 경보 경계만 낮춰도 전문가가 실제 최대값을 충분히 높게 예측하지 못하는 문제가 남는다. [8건의 순위·확률·전문가 상세](tables/m04_surge/april_legacy_cases.csv)를 보존했다.

이 진단으로 '기존 낮은 상태 분류기의 경계를 계속 조절'하는 대신, 전체 상태에서 급등 여부를 학습하고 정상 예측과 위험 출력을 분리하는 비교를 선택했다. 4월의 모든 상승을 새 급등으로 표현하지 않으며 관측상 원인을 설비 기동 등으로 단정하지 않는다.

### 전체 상태 분류와 Q90 보조 출력의 시간순 비교

[추가 학습 조건](config/m04_surge_risk_contract.json)은 외부 1~3월→4월, 1~4월→5월, 1~5월→6월과 내부 과거 월 확장 검증을 유지했다. 전체 상태 Logistic/HGB 분류기는 검증된 과거 변화 포함 27입력으로 증가량≥93 여부를 학습했다. Logistic은 학습 구간에서만 표준화하고 C 0.001~100을 탐색했다. HGB는 잎3·7·15, 최소 잎 표본10·20·40, L2 0.1~10, 반복150·학습률0.05·조기종료 안 함으로 비교했다. 외부 월마다 각20회 Optuna, 총120회다. 내부 AP를 최대화하고 동률은 Brier·시도 순서로 구분했다.

AP는 비보간 평균 정밀도다. 분류 순위의 주 비교는 각 월 AP를 그 월 사건 수로 가중한 평균이며, 월별 점수 척도가 달라질 수 있어 합친 AP는 보조값으로 보존했다. AUROC도 남겼지만 전체 정상 시간이 대부분인 자료에서 그 수치만으로 희소 급등 탐지를 평가하지 않았다. 전력 크기는 여전히 MAE·과소/과대로 따로 평가한다.

Q90은 B의 18입력·기존 HGB 설정에 분위수 손실 0.9만 사용한 보조 상단 예측이다. 분위수를 새로 탐색하지 않았다. 분류 내부360회·분위수 내부9회·외부9회, 총378회 학습하고 저장 모델9개를 재로딩했다. 기존 B와 기존 HGB 결합의 위험 점수는 저장 예측에서 `예측 최대값-직전 최대값`으로 재사용했다. 모든 방법에서 정상 전력 예측은 B로 유지해 위험 출력 때문에 낮은 유지의 정상 예측이 악화되지 않게 했다. 이 구조는 현장 경보 비용이 사라졌다는 뜻은 아니다.

### 경보 경계의 한 번의 보정과 결과

첫 경보 선택은 과거 내부 예측에서 오경보 허용률 1%(보조5%) 아래 TP를 최대화하고 동률이면 FP가 적은 경계를 골랐다. 초기 내부 자료에 주 급등이 매우 적어 허용 경계에서 TP가 모두 0이면 일부 방법은 경보를 전혀 내지 않는 선택으로 귀결됐다. 이를 확인한 뒤 [후속 조건](config/m04_surge_calibration_contract.json)에 음성 꼬리 경계를 한 번만 적용하기로 기록했다. 새 모델·입력·급등 정의·AP·채택 기준은 바꾸지 않았고 새 학습도 없었다.

보정 경계는 과거 내부 비급등 점수만 내림차순 정렬해 `k=floor(허용률×음성 수)`로 정하고, 인덱스 k의 점수 바로 위 부동소수점 값을 사용한다. 같은 점수의 동률은 보수적으로 제외한다. 미래 평가 월의 점수나 정답으로 경계를 고르지 않는다. 아래는 주 허용률1% 결과다. 허용률은 내부 선택 조건이며 외부에서 같은 오경보율을 보장하는 값이 아니다.

{risk_table()}
{month_table()}
기존 결합 증가 점수는 B 점수의 5/8건 탐지에서 6/8건으로 늘었다. 대신 오경보가 8→10건으로 늘어 정밀도는 38.5%→37.5%였다. 이는 '동일 오경보 수' 비교가 아니라 '같은 내부 허용률에서 정한 경계'의 비교다. 실제 오경보율은 각각 8/2,176=0.368%, 10/2,176=0.460%다. 전체 상태 HGB는 순위 AP가 더 높아도 고정된 경계에서 3건만 찾았고 4월 주 급등 2건을 모두 놓쳤다. AP의 개선과 실제 운용 경계의 탐지 개선을 구분해야 한다. [최종 경보표](tables/m04_surge_calibration/alarm_metrics.csv), [전체 비교표](tables/m04_surge_calibration/selection_metrics.csv)에 FN/TN·5% 조건·민감도 결과도 보존했다.

### 상단 예측의 성과와 한계

Q90의 주 급등 MAE는 {u_mae:.3f}, 평균 과소예측은 {u_under:.3f}로 B보다 작았다. 그러나 상단을 정상 예측으로 대체하면 전체 MAE가 6.191에서 {all_mae:.3f}로 커진다. 전체 관측이 상단 이하인 비율은 4~6월 각각 {monthly_cov[0]:.2f}%·{monthly_cov[1]:.2f}%·{monthly_cov[2]:.2f}%, 합계 {100*coverage:.2f}%였다. 주 급등에서의 포함률은 {100*event_coverage:.1f}%로 8건 중1건뿐이다. 따라서 이를 급등의 90% 보장 상단이나 신뢰구간이라고 표현할 수 없다. [상단 오류·포함률](tables/m04_surge_risk/regression_metrics.csv)은 보조 분석으로 남기고 최종 정상 예측이나 위험 채널로 채택하지 않았다.

### 채택 판단과 다음 회차의 변경 근거

사전 추가 후보 목록은 전체 상태 Logistic/HGB와 Q90이었다. 채택하려면 B 대비 가중 AP가 5% 이상 높고, 합계 TP와 각 월 TP가 작지 않으며, 실제 합계 오경보율1% 이내이고 TP 증가 또는 FP 감소가 있어야 했다. 이 새 후보들은 모든 요건을 함께 충족하지 못해 [자동 선택](tables/m04_surge_calibration/selection.json)은 B 점수를 유지했다. 기존 HGB 결합 점수는 수치 요건을 모두 통과했지만 사전에 '기존 결과 재평가용'으로 제외한 방법이다. 결과를 본 뒤 후보 목록을 바꿔 자동 채택했다고 쓰지 않는다.

다만 '기존 결합을 정상 예측 교체가 아닌 위험 점수로 사용'하는 방향은 이번 결과에서 얻은 후속 후보다. 정상 예측 B를 유지하면서 6/8건을 찾은 성과와 추가 오경보2건을 함께 보여줄 수 있다. 이것은 같은 개발 자료에서 찾은 탐색적 후보이며 8건의 작은 분모를 가진다. 보고서 제3장에는 급등 크기 오차19.3% 감소와 위험 출력의 이득·오경보 부담, 4월 한계·미평가 0재가동을 연결한다. 제5장의 주요 근거는 M03 마지막 값의 기여를 유지하고, 이번 결과는 그 뒤에 확인한 보완 가능성으로 제시한다. '모든 급등을 미리 정확히 맞춘다'는 주장은 하지 않는다.

선언한 새 모델 비교와 한 번의 경보 보정은 여기서 종료한다. 다음 M05에서는 정상 예측 A/B와 별도 위험 출력 B 점수/기존 HGB 결합 점수의 비교 조건을 먼저 고정한다. M01에서 정한 대로 1~6월 자료로 각 모델을 한 번 학습하고, 같은 모델을 고정해 7~8월 전체를 평가한다. 증가 경계93과 음성 꼬리 보정식을 유지하며 모델 설정·결합 규칙·경보 경계는 6월 말까지의 자료로 확정한다. 7월 정답을 추가해 모델을 다시 학습하거나 8월용 경보 경계를 새로 정하지 않는다. 평가 중에는 관측을 마친 직전 시간의 전력·생산량 등을 입력에 반영해 다음 한 시간을 예측하며, 이 입력 갱신은 모델 재학습과 구분한다. 7~8월 결과를 보고 재튜닝하지 않으며, 후반기 자료의 과거 관찰 이력이 있으므로 완전히 처음 보는 시험이라고 쓰지 않는다. 이번 회차는 M05·현장 경보 운영·전력 절감 실측·압축본·파일 삭제를 실행하지 않았다.

### 코드와 재현 근거

[진단 실행](scripts/m04_surge_diagnose.py)→[진단 독립 검증](scripts/m04_surge_verify.py)→[위험 출력 학습](scripts/m04_surge_risk.py)→[학습 독립 검증](scripts/m04_surge_risk_verify.py)→[경보 보정](scripts/m04_surge_calibration.py)→[보정 독립 검증](scripts/m04_surge_calibration_verify.py)→[문서 작성·검증](scripts/m04_surge_documents.py)의 순서다. 전체 코드는 4.5의 32개 파일 목록에 포함한다.

추가 조건3개, `tables/m04_surge/`, `tables/m04_surge_risk/`, `tables/m04_surge_calibration/`, `models/m04_surge_risk/`의 모델9개를 함께 보존한다. `trial_oof_predictions.npz`는 120회 선택을 독립 검증하는 배열이므로 본문에 직접 인용하지 않더라도 필요하다. 진단 15,288행·추가 위험 출력 10,920행·내부 예측32,160행·후속 경보10,920행의 검증을 마쳤다. 새 그림이나 별도 압축 보고서는 만들지 않고 상세 원고와 두 README의 기존 흐름에 연결했다.
'''


def write():
    source = MD.read_text(encoding='utf-8')
    assert TITLE not in source, 'Preserve the existing section; do not overwrite.'
    source = source.replace('사용한 25개 파일이다.', '사용한 32개 파일이다.')
    insert = '\n'.join(f'| [{name}](scripts/{name}) | 4.8 {role} | 전체 파일 보존; `main()` |' for name, role in NEW.items()) + '\n'
    marker = '| [modeling_submission_audit.py]'
    source = source.replace(marker, insert + marker, 1)
    source = source.replace('[결합 점수 수정](config/m04_rise_gate_contract.json).', '[결합 점수 수정](config/m04_rise_gate_contract.json), [일반 급등 진단](config/m04_surge_contract.json), [위험 출력](config/m04_surge_risk_contract.json), [경보 보정](config/m04_surge_calibration_contract.json).')
    source = source.replace('`m04_rise_gate/`, `observed_timeline/`', '`m04_rise_gate/`, `m04_surge/`, `m04_surge_risk/`, `m04_surge_calibration/`, `observed_timeline/`')
    source = source.replace('`m02_rerun/`, `m04_rise/`의 파일.', '`m02_rerun/`, `m04_rise/`, `m04_surge_risk/`의 파일.')
    start = source.index('| 11 | `Modeling/scripts/m02_rerun_document_verify.py`')
    end = source.index('\n\n정리 전에는', start)
    source = source[:start] + '''| 11 | `Modeling/scripts/m04_surge_diagnose.py` → `Modeling/scripts/m04_surge_verify.py` | 증가량 정의·기존 예측 재평가·4월 진단 |
| 12 | `Modeling/scripts/m04_surge_risk.py` → `Modeling/scripts/m04_surge_risk_verify.py` | 전체 상태 분류·Q90 보조 학습과 검증 |
| 13 | `Modeling/scripts/m04_surge_calibration.py` → `Modeling/scripts/m04_surge_calibration_verify.py` | 저장 내부 예측의 음성 꼬리 경보 보정; 새 학습 없음 |
| 14 | `Modeling/scripts/m02_rerun_document_verify.py`, `Modeling/scripts/m04_verify.py --documents`, `Modeling/scripts/m04_rise_documents.py --verify`, `Modeling/scripts/m04_surge_documents.py --verify` | 기존·추가 원고 수치와 링크 검증 |
| 15 | `Modeling/scripts/modeling_submission_audit.py --record` → `Modeling/scripts/modeling_submission_audit.py --verify` | 최신 상태를 보존 목록에 기록하고 의존성 확인 |''' + source[end:]
    MD.write_text(source.rstrip() + '\n\n' + section().rstrip() + '\n', encoding='utf-8')
    LOG.write_text('# M04 증가량 급등과 별도 위험 출력 실행 기록\n\n' + section() + '\n', encoding='utf-8')
    paragraph = '''일반 급등을 학습 1월 양의 증가 95분위수인 증가량93 이상으로 먼저 정하고 기존 예측을 재평가했다. 개발8건에서 기존 HGB 결합의 MAE는38.586→31.125(19.3%감소)였다. 전체 상태 분류·Q90을120회 Optuna·378회 학습으로 비교한 뒤, 저장 내부 예측의 음성 꼬리 경계를 한 번 보정했다. 정상 예측은 B로 유지한다. 별도 위험 점수로 기존 결합을 쓰면 B의5/8건 탐지·오경보8건에서6/8건·오경보10건으로 바뀌었다. 새 후보는 모든 채택 요건을 충족하지 못했고 자동 선택은 B 점수다. 기존 결합 점수는 사전 후보 목록 밖의 탐색적 후속 후보이므로 M05에서 고정 조건으로 확인할 대상으로 연결한다. 3단계 독립 검증을 마쳤고 M05는 미실행이다.'''
    for path, target in [(ROOT / 'README.md', 'Modeling/10.05_M04_증가량급등과위험출력_실행기록.md'), (ROOT / 'Modeling/README.md', '10.05_M04_증가량급등과위험출력_실행기록.md')]:
        content = path.read_text(encoding='utf-8')
        assert target not in content
        position = content.index('\n\n', content.index('# '))
        content = content[:position] + f'\n\n[일반 급등·위험 출력 실행 기록]({target}): {paragraph}' + content[position:]
        if path.parent.name == 'Modeling':
            content = re.sub(r'사용한 Python 코드 \d+개', '사용한 Python 코드 32개', content)
        path.write_text(content, encoding='utf-8')


def verify():
    source = MD.read_text(encoding='utf-8')
    assert source.count(TITLE) == 1
    assert source.split(TITLE, 1)[1].split('\n## 4.9', 1)[0].rstrip() == section().split(TITLE, 1)[1].rstrip()
    assert LOG.read_text(encoding='utf-8').split(TITLE, 1)[1].rstrip() == section().split(TITLE, 1)[1].rstrip()
    links = 0
    for target in re.findall(r'!?\[[^\]]*\]\(([^)]+)\)', source):
        if target.startswith(('http:', 'https:', '#', 'app:')):
            continue
        assert (MD.parent / target.split('#')[0].strip('<>')).resolve().exists(), target
        links += 1
    for name in NEW:
        assert f'(scripts/{name})' in source
    result = {'status': 'passed', 'checked_at': datetime.now(timezone(timedelta(hours=9))).isoformat(timespec='minutes'),
              'script_sha256': sha(Path(__file__)), 'manuscript_sha256': sha(MD), 'execution_record_sha256': sha(LOG),
              'error_rows': 7, 'risk_rows': 5, 'monthly_rows': 3, 'local_links_checked': links,
              'normal_forecast': 'B', 'automatic_risk_selection': 'B_score',
              'provisional_followup_risk_candidate': 'G_HGB_score', 'no_training_or_deletion': True}
    (ROOT / 'Modeling/tables/m04_surge_calibration/document_verification.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    print(json.dumps(result), flush=True)


def main():
    parser = argparse.ArgumentParser()
    group = parser.add_mutually_exclusive_group(required=True)
    group.add_argument('--write', action='store_true')
    group.add_argument('--verify', action='store_true')
    args = parser.parse_args()
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    verified_tables()
    if args.write:
        write()
    verify()


if __name__ == '__main__':
    main()


코드 등록 완료: `Modeling/scripts/m04_surge_documents.py`

### `Modeling/scripts/m04_surge_risk.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: All-state surge classification and a separate upper quantile; chronological development.

주요 함수: `probability()`, `fit_classifier()`, `rank()`, `regression()`, `main()`.

In [57]:
%%submission_source Modeling/scripts/m04_surge_risk.py
"""All-state surge classification and a separate upper quantile; chronological development."""
import os
os.environ["OMP_NUM_THREADS"] = "1"
import json
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score, mean_pinball_loss

from m01_prepare import ROOT, sha, read_contract
from m04_rise_compare import classifier, now, read_csv
from m04_surge_diagnose import alarm_metrics, threshold_at_cap

OUT = ROOT / "Modeling/tables/m04_surge_risk"
MODELS = ROOT / "Modeling/models/m04_surge_risk"
CONTRACT = ROOT / "Modeling/config/m04_surge_risk_contract.json"


def probability(model, inputs):
    return np.full(len(inputs),model["constant_probability"]) if isinstance(model,dict) else model.predict_proba(inputs)[:,1]


def fit_classifier(name, parameters, tr, ev, features):
    labels = tr.increase.ge(93).astype(int)
    if labels.nunique()<2:
        model = {"constant_probability":float(labels.mean())}
        return model, probability(model,ev[features]), True
    model = classifier(name.removesuffix("_all"),parameters)
    model.fit(tr[features],labels)
    return model, probability(model,ev[features]), False


def rank(y, score):
    return {"AP":float(average_precision_score(y,score)), "AUROC":float(roc_auc_score(y,score)), "events":int(np.sum(y)), "hours":len(y)}


def regression(frame, prediction):
    error = np.asarray(prediction)-frame.actual.to_numpy()
    weight = frame.daily_maximum_weight.to_numpy()
    return {"hours":len(frame), "MAE":float(np.abs(error).mean()), "mean_under":float(np.maximum(-error,0).mean()),
        "mean_over":float(np.maximum(error,0).mean()), "coverage":float(np.mean(error>=0)),
        "pinball90":float(mean_pinball_loss(frame.actual,prediction,alpha=.9)),
        "daily_maximum_MAE":float(np.average(np.abs(error),weights=weight)) if weight.sum() else np.nan}


def main():
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    assert not (OUT/"run.json").exists(), "Preserve completed comparison."
    c = json.loads(CONTRACT.read_text(encoding="utf-8"))
    diagnosis = ROOT/"Modeling/tables/m04_surge"
    verified = json.loads((diagnosis/"independent_verification.json").read_text(encoding="utf-8"))
    assert verified["status"]=="passed" and verified["run_sha256"]==sha(diagnosis/"run.json")==c["diagnosis_run_sha256"]
    assert verified["boundary"]["q95"]==c["boundary_primary"]==93
    parent = ROOT/"Modeling/tables/m04_rise"
    parent_run = json.loads((parent/"run.json").read_text(encoding="utf-8"))
    for filename in ["feature_frame.csv","inner_predictions.csv","predictions.csv"]:
        assert sha(parent/filename)==parent_run["outputs_sha256"][filename]
    original_contract = json.loads((ROOT/"Modeling/config/m04_rise_contract.json").read_text(encoding="utf-8"))
    m1 = read_contract()
    f = read_csv(ROOT/"Modeling/tables/m01/hourly_frame.csv",["timestamp","date"])
    f = f.loc[f.eligible_common & f.timestamp.lt("2021-07-01")].copy()
    f = f.merge(read_csv(parent/"feature_frame.csv",["timestamp"])[["timestamp"]+original_contract["features"]["add"]],on="timestamp",validate="one_to_one")
    f["increase"] = f.target_maximum-f.lag1_maximum
    features = parent_run["features"]["dynamic"]
    base_features = parent_run["features"]["B"]
    cached = read_csv(parent/"inner_predictions.csv",["timestamp","date"])
    old = read_csv(parent/"predictions.csv",["timestamp","date"])
    OUT.mkdir(parents=True,exist_ok=True)
    MODELS.mkdir(parents=True,exist_ok=True)
    predictions,inners,trials,configs,manifests,fold_audit,policies,alarms,ranks,regressions = [],[],[],[],[],[],[],[],[],[]
    trial_arrays = {}
    fit_counts = {"inner_classifier":0,"inner_quantile":0,"outer_classifier":0,"outer_quantile":0}
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    started = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        for outer in m1["splits"]:
            split = outer["name"]
            if split not in c["time"]["outer"]:
                continue
            tr = f.loc[f.timestamp.le(outer["train_end"])]
            ev = f.loc[f.timestamp.between(outer["eval_start"],outer["eval_end"])].copy()
            template = old.loc[old.split.eq(split)&old.group.eq("B")].sort_values("timestamp").copy()
            assert template.timestamp.tolist()==ev.timestamp.tolist()
            template["increase"] = ev.increase.to_numpy()
            template["previous_maximum"] = ev.lag1_maximum.to_numpy()
            template["normal_prediction"] = template.prediction
            folds = []
            for month in c["time"]["inner_months"][split]:
                start = pd.Timestamp(2021,month,1)
                t = tr.loc[tr.timestamp.lt(start)]
                v = tr.loc[tr.timestamp.dt.month.eq(month)]
                assert t.timestamp.max()<v.timestamp.min()<=v.timestamp.max()<ev.timestamp.min()
                folds.append((t,v))
                fold_audit.append({"split":split,"month":month,"train_hours":len(t),"validation_hours":len(v),
                    "train_events":int(t.increase.ge(93).sum()),"validation_events":int(v.increase.ge(93).sum()),
                    "train_last":str(t.timestamp.max()),"validation_first":str(v.timestamp.min()),"validation_last":str(v.timestamp.max()),"outer_first":str(ev.timestamp.min())})
            iv = pd.concat([v for t,v in folds],ignore_index=True)
            inner_score,outer_score = {},{}
            for name,source_group in [("B_score","B"),("G_HGB_score","G_HGB")]:
                record = cached.loc[cached.split.eq(split)&cached.group.eq(source_group)].sort_values("timestamp")
                assert record.timestamp.tolist()==iv.timestamp.tolist()
                inner_score[name] = record.prediction.to_numpy()-iv.lag1_maximum.to_numpy()
                outer_record = old.loc[old.split.eq(split)&old.group.eq(source_group)].sort_values("timestamp")
                outer_score[name] = outer_record.prediction.to_numpy()-ev.lag1_maximum.to_numpy()
            for name in c["classifiers"]["methods"]:
                predictions_by_trial = {}
                def objective(trial):
                    if name=="Logistic_all":
                        parameters = {"C":trial.suggest_float("C",.001,100,log=True)}
                    else:
                        parameters = {"max_leaf_nodes":trial.suggest_categorical("max_leaf_nodes",[3,7,15]),
                            "min_samples_leaf":trial.suggest_categorical("min_samples_leaf",[10,20,40]),
                            "l2_regularization":trial.suggest_float("l2_regularization",.1,10,log=True)}
                    pieces,constants = [],0
                    for t,v in folds:
                        model,values,constant = fit_classifier(name,parameters,t,v,features)
                        fit_counts["inner_classifier"]+=int(not constant)
                        constants+=int(constant)
                        pieces.append(values)
                    values = np.concatenate(pieces)
                    result = rank(iv.increase.ge(93),values)
                    result["Brier"] = float(brier_score_loss(iv.increase.ge(93),values))
                    trial.set_user_attr("metrics",result)
                    trial.set_user_attr("constant_folds",constants)
                    predictions_by_trial[trial.number]=values
                    return result["AP"]
                study = optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler(seed=42,n_startup_trials=10),pruner=optuna.pruners.NopPruner())
                study.optimize(objective,n_trials=20)
                selected = min(study.trials,key=lambda t:(-t.value,t.user_attrs["metrics"]["Brier"],t.number))
                inner_score[name]=predictions_by_trial[selected.number]
                trial_arrays[split+"_"+name]=np.stack([predictions_by_trial[t.number] for t in study.trials])
                model,values,constant = fit_classifier(name,selected.params,tr,ev,features)
                outer_score[name]=values
                fit_counts["outer_classifier"]+=int(not constant)
                path = MODELS/f"{split}_{name}.joblib"
                joblib.dump(model,path,compress=3)
                assert np.allclose(probability(joblib.load(path),ev[features]),values,atol=1e-9,rtol=0)
                manifests.append({"split":split,"name":name,"file":path.relative_to(ROOT).as_posix(),"sha256":sha(path),"features":len(features)})
                configs.append({"split":split,"name":name,"best_trial":selected.number,"parameters":selected.params,"inner_metrics":selected.user_attrs["metrics"],"constant_outer":constant})
                for trial in study.trials:
                    trials.append({"split":split,"name":name,"trial":trial.number,"state":trial.state.name,"objective":trial.value,
                        "parameters":json.dumps(trial.params),"metrics":json.dumps(trial.user_attrs)})
                print(f"{split} {name} inner AP={selected.value:.4f}",flush=True)
            quantile_pieces=[]
            for t,v in folds:
                model=HistGradientBoostingRegressor(**original_contract["hgb_parameters"],loss="quantile",quantile=.9)
                model.fit(t[base_features],t.target_maximum)
                quantile_pieces.append(model.predict(v[base_features]))
                fit_counts["inner_quantile"]+=1
            inner_upper=np.concatenate(quantile_pieces)
            model=HistGradientBoostingRegressor(**original_contract["hgb_parameters"],loss="quantile",quantile=.9)
            model.fit(tr[base_features],tr.target_maximum)
            outer_upper=model.predict(ev[base_features])
            fit_counts["outer_quantile"]+=1
            path=MODELS/f"{split}_Q90_upper.joblib"
            joblib.dump(model,path,compress=3)
            assert np.allclose(joblib.load(path).predict(ev[base_features]),outer_upper,atol=1e-9,rtol=0)
            manifests.append({"split":split,"name":"Q90_upper","file":path.relative_to(ROOT).as_posix(),"sha256":sha(path),"features":18})
            inner_score["Q90_upper"]=inner_upper-iv.lag1_maximum.to_numpy()
            outer_score["Q90_upper"]=outer_upper-ev.lag1_maximum.to_numpy()
            for name in c["comparators"]:
                record=template.copy()
                record["group"]=name
                record["score"]=outer_score[name]
                record["score_type"]="probability" if name.endswith("_all") else "predicted_increase"
                record["upper_prediction"]=outer_upper if name=="Q90_upper" else np.nan
                inner_record=iv[["timestamp","date","increase","lag1_maximum","target_maximum"]].copy()
                inner_record["split"],inner_record["group"],inner_record["score"]=split,name,inner_score[name]
                inner_record["upper_prediction"]=inner_upper if name=="Q90_upper" else np.nan
                inners.append(inner_record)
                for cap in [c["alarms"]["primary_false_positive_cap"],c["alarms"]["secondary_false_positive_cap"]]:
                    chosen=threshold_at_cap(iv.increase.ge(93),inner_score[name],cap)
                    policies.append({"split":split,"group":name,"cap":cap,**chosen})
                    alarm=outer_score[name]>=chosen["threshold"]
                    record["alarm_"+str(cap)]=alarm
                    for key,boundary in [("q95",93.),("q90",c["boundary_sensitivity"])]:
                        values=alarm_metrics(ev.increase.ge(boundary),outer_score[name],chosen["threshold"])
                        if name.endswith("_all") and key=="q95":
                            values["Brier"]=float(brier_score_loss(ev.increase.ge(boundary),outer_score[name]))
                        alarms.append({"split":split,"group":name,"cap":cap,"boundary_name":key,**values})
                predictions.append(record)
            for name,prediction in [("B",template.normal_prediction.to_numpy()),("G_HGB",old.loc[old.split.eq(split)&old.group.eq("G_HGB")].sort_values("timestamp").prediction.to_numpy()),("Q90_upper",outer_upper)]:
                for condition,mask in [("all",np.ones(len(ev),dtype=bool)),("surge_q95",ev.increase.ge(93)),("surge_q90",ev.increase.ge(c["boundary_sensitivity"])),("low_stay",template.power_transition.eq("low->low").to_numpy())]:
                    part=template.loc[np.asarray(mask)]
                    if len(part):
                        regressions.append({"split":split,"group":name,"condition":condition,**regression(part,np.asarray(prediction)[np.asarray(mask)])})
        p=pd.concat(predictions,ignore_index=True)
        # All new methods retain exactly B as their normal point forecast.
        assert np.array_equal(p.prediction.to_numpy(),p.normal_prediction.to_numpy())
        alarm_frame=pd.DataFrame(alarms)
        selection=[]
        for name,g in p.groupby("group"):
            y=g.increase.ge(93).to_numpy()
            values=rank(y,g.score)
            monthly=alarm_frame.loc[alarm_frame.group.eq(name)&alarm_frame.cap.eq(.01)&alarm_frame.boundary_name.eq("q95")]
            values["weighted_monthly_AP"]=float(np.average(monthly.AP,weights=monthly.events))
            chosen=g["alarm_0.01"].to_numpy()
            tp=int((chosen&y).sum());fp=int((chosen&~y).sum())
            values.update({"group":name,"TP":tp,"FP":fp,"recall":tp/int(y.sum()),"precision":tp/(tp+fp) if tp+fp else 0.,"false_positive_fraction":fp/int((~y).sum())})
            selection.append(values)
        table=pd.DataFrame(selection)
        ref=table.loc[table.group.eq("B_score")].iloc[0]
        for i,r in table.iterrows():
            monthly=alarm_frame.loc[alarm_frame.group.eq(r.group)&alarm_frame.cap.eq(.01)&alarm_frame.boundary_name.eq("q95")].set_index("split")
            baseline=alarm_frame.loc[alarm_frame.group.eq("B_score")&alarm_frame.cap.eq(.01)&alarm_frame.boundary_name.eq("q95")].set_index("split")
            flags={"rank_passed":bool(r.weighted_monthly_AP>=1.05*ref.weighted_monthly_AP),"recall_passed":bool(r.TP>=ref.TP),
                "monthly_recall_passed":bool((monthly.TP>=baseline.TP).all()),"false_positive_passed":bool(r.false_positive_fraction<=.01+1e-12),
                "strict_gain_passed":bool(r.TP>ref.TP or r.FP<ref.FP)}
            for key,value in flags.items():
                table.loc[i,key]=value
            table.loc[i,"eligible"]=r.group in ["Logistic_all","HGB_all","Q90_upper"] and all(flags.values())
        eligible=table.loc[table.eligible.astype(bool)]
        if len(eligible):
            near=eligible.loc[eligible.weighted_monthly_AP.ge(eligible.weighted_monthly_AP.max()/1.01)].copy()
            preference={name:i for i,name in enumerate(["Logistic_all","HGB_all","Q90_upper"])}
            near["preference"]=near.group.map(preference)
            chosen=near.sort_values(["preference","weighted_monthly_AP"],ascending=[True,False]).iloc[0].group
        else:
            chosen="B_score"
        selection_result={"selected_risk_channel":chosen,"normal_forecast":"B","eligible_methods":eligible.group.tolist(),"development_only":True,"no_july_august_evaluation":True}
        for data,filename in [(p,"predictions.csv"),(pd.concat(inners,ignore_index=True),"inner_predictions.csv"),(pd.DataFrame(trials),"trials.csv"),
            (pd.DataFrame(manifests),"model_manifest.csv"),(pd.DataFrame(fold_audit),"fold_audit.csv"),(pd.DataFrame(policies),"alarm_policies.csv"),
            (alarm_frame,"alarm_metrics.csv"),(pd.DataFrame(regressions),"regression_metrics.csv"),(table,"selection_metrics.csv")]:
            data.to_csv(OUT/filename,index=False,encoding="utf-8-sig")
        np.savez_compressed(OUT/"trial_oof_predictions.npz",**trial_arrays)
        (OUT/"selected_configurations.json").write_text(json.dumps(configs,indent=2),encoding="utf-8")
        (OUT/"selection.json").write_text(json.dumps(selection_result,indent=2),encoding="utf-8")
        warning_messages=[str(w.message) for w in caught]
    run={"status":"completed","finished":now(),"seconds":time.perf_counter()-started,"script_sha256":sha(Path(__file__)),"contract_sha256":sha(CONTRACT),
        "inputs_sha256":{path.relative_to(ROOT).as_posix():sha(path) for path in [diagnosis/"run.json",diagnosis/"independent_verification.json",parent/"run.json",parent/"feature_frame.csv",parent/"inner_predictions.csv",parent/"predictions.csv",ROOT/"Modeling/tables/m01/hourly_frame.csv",ROOT/"Modeling/config/m04_rise_contract.json"]},
        "helper_sha256":{name:sha(ROOT/"Modeling/scripts"/name) for name in ["m01_prepare.py","m04_rise_compare.py","m04_surge_diagnose.py"]},
        "features":{"classifier":features,"quantile":base_features},"fits":fit_counts,"trials":len(trials),"warnings":warning_messages,
        "normal_forecast_unchanged":True,"no_july_august_evaluation":True,"runtime":{"executable":sys.executable,"version":sys.version},
        "outputs_sha256":{path.name:sha(path) for path in OUT.iterdir()},"prediction_rows":len(p)}
    (OUT/"run.json").write_text(json.dumps(run,indent=2),encoding="utf-8")
    print(table.to_string(index=False),flush=True)
    print(json.dumps(selection_result),flush=True)
    print(json.dumps({"fits":fit_counts,"trials":len(trials),"warnings":warning_messages}),flush=True)


if __name__=="__main__":
    main()


코드 등록 완료: `Modeling/scripts/m04_surge_risk.py`

### `Modeling/scripts/m04_surge_risk_verify.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Independent trial arrays, chronological gates, saved models and separate forecast checks.

주요 함수: `main()`.

In [58]:
%%submission_source Modeling/scripts/m04_surge_risk_verify.py
"""Independent trial arrays, chronological gates, saved models and separate forecast checks."""
import os
os.environ["OMP_NUM_THREADS"] = "1"
import json
import math
import sys
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from m03_verify import rows, sha, close
from m04_rise_verify import ap
from m04_surge_verify import frontier, check_alarm

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/m04_surge_risk"


def main():
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    run=json.loads((OUT/"run.json").read_text(encoding="utf-8"))
    cpath=ROOT/"Modeling/config/m04_surge_risk_contract.json"
    c=json.loads(cpath.read_text(encoding="utf-8"))
    assert run["status"]=="completed" and run["normal_forecast_unchanged"] and run["no_july_august_evaluation"]
    assert sha(cpath)==run["contract_sha256"] and sha(ROOT/"Modeling/scripts/m04_surge_risk.py")==run["script_sha256"]
    for field in ["inputs_sha256","outputs_sha256","helper_sha256"]:
        for name,digest in run[field].items():
            path=ROOT/name if field=="inputs_sha256" else OUT/name if field=="outputs_sha256" else ROOT/"Modeling/scripts"/name
            assert sha(path)==digest,(field,name)
    diagnosis=json.loads((ROOT/"Modeling/tables/m04_surge/independent_verification.json").read_text(encoding="utf-8"))
    assert diagnosis["status"]=="passed" and diagnosis["run_sha256"]==c["diagnosis_run_sha256"]
    original=rows(ROOT/"Modeling/tables/m04_surge/predictions.csv")
    inherited={(r["group"],r["timestamp"]):r for r in original}
    frame={r["timestamp"]:r for r in rows(ROOT/"Modeling/tables/m01/hourly_frame.csv") if r["eligible_common"]=="True" and r["timestamp"]<"2021-07-01"}
    features=rows(ROOT/"Modeling/tables/m04_rise/feature_frame.csv")
    feature_map={r["timestamp"]:r for r in features}
    pred,inner=rows(OUT/"predictions.csv"),rows(OUT/"inner_predictions.csv")
    groups,parts,inside=defaultdict(list),defaultdict(list),defaultdict(list)
    for r in pred:
        baseline=inherited["B",r["timestamp"]]
        for field in ["actual","normal_prediction","prediction"]:
            close(r[field],baseline["actual"] if field=="actual" else baseline["prediction"])
        close(r["increase"],baseline["increase"])
        close(r["previous_maximum"],baseline["lag1_maximum"])
        assert datetime.fromisoformat(r["timestamp"]).month in [4,5,6]
        if r["group"] in ["B_score","G_HGB_score"]:
            source="B" if r["group"]=="B_score" else "G_HGB"
            close(r["score"],float(inherited[source,r["timestamp"]]["prediction"])-float(baseline["lag1_maximum"]))
        if r["group"]=="Q90_upper":
            close(r["score"],float(r["upper_prediction"])-float(r["previous_maximum"]))
        groups[r["group"]].append(r)
        parts[r["split"],r["group"]].append(r)
    assert len(pred)==run["prediction_rows"]==10920 and len(groups)==5 and all(len(g)==2184 for g in groups.values())
    for r in inner:
        f=frame[r["timestamp"]]
        close(r["increase"],float(f["target_maximum"])-float(f["lag1_maximum"]))
        close(r["target_maximum"],f["target_maximum"])
        close(r["lag1_maximum"],f["lag1_maximum"])
        inside[r["split"],r["group"]].append(r)
    assert len(inner)==32160
    for r in rows(OUT/"fold_audit.csv"):
        month=int(r["month"])
        start=datetime(2021,month,1)
        train=[v for t,v in frame.items() if datetime.fromisoformat(t)<start]
        validation=[v for t,v in frame.items() if datetime.fromisoformat(t).month==month]
        assert len(train)==int(r["train_hours"]) and len(validation)==int(r["validation_hours"])
        assert r["train_last"]<r["validation_first"]<=r["validation_last"]<r["outer_first"]
        for label,pool in [("train",train),("validation",validation)]:
            assert int(r[label+"_events"])==sum(float(v["target_maximum"])-float(v["lag1_maximum"])>=93 for v in pool)
    trial_rows=rows(OUT/"trials.csv")
    configs=json.loads((OUT/"selected_configurations.json").read_text(encoding="utf-8"))
    trial_arrays=np.load(OUT/"trial_oof_predictions.npz",allow_pickle=False)
    checked_trials=0
    for config in configs:
        split,name=config["split"],config["name"]
        part=[r for r in trial_rows if r["split"]==split and r["name"]==name]
        assert len(part)==20 and all(r["state"]=="COMPLETE" for r in part)
        pool=inside[split,name]
        labels=[float(r["increase"])>=93 for r in pool]
        assert max(r["timestamp"] for r in pool)<min(r["timestamp"] for r in parts[split,name])
        array=trial_arrays[split+"_"+name]
        assert array.shape==(20,len(pool))
        for r in part:
            scores=array[int(r["trial"])].tolist()
            values=json.loads(r["metrics"])["metrics"]
            close(r["objective"],ap(labels,scores))
            close(values["AP"],ap(labels,scores))
            close(values["Brier"],math.fsum((s-y)**2 for y,s in zip(labels,scores))/len(pool))
            checked_trials+=1
        selected=min(part,key=lambda r:(-float(r["objective"]),json.loads(r["metrics"])["metrics"]["Brier"],int(r["trial"])))
        assert int(selected["trial"])==config["best_trial"] and json.loads(selected["parameters"])==config["parameters"]
        assert np.allclose(array[config["best_trial"]],[float(r["score"]) for r in pool],atol=1e-12,rtol=0)
    assert checked_trials==run["trials"]==120
    policies=rows(OUT/"alarm_policies.csv")
    for r in policies:
        pool=inside[r["split"],r["group"]]
        expected=frontier([float(x["increase"])>=93 for x in pool],[float(x["score"]) for x in pool],float(r["cap"]))
        for field,value in expected.items():
            close(r[field],value)
        for x in parts[r["split"],r["group"]]:
            assert (x["alarm_"+r["cap"]]=="True")== (float(x["score"])>=float(r["threshold"]))
    alarm_rows=rows(OUT/"alarm_metrics.csv")
    assert len(alarm_rows)==60
    for r in alarm_rows:
        pool=parts[r["split"],r["group"]]
        boundary=93 if r["boundary_name"]=="q95" else c["boundary_sensitivity"]
        check_alarm(r,pool,[float(x["score"]) for x in pool],boundary)
        policy=next(p for p in policies if p["split"]==r["split"] and p["group"]==r["group"] and p["cap"]==r["cap"])
        close(r["threshold"],policy["threshold"])
        if r["Brier"]:
            close(r["Brier"],math.fsum((float(x["score"])-(float(x["increase"])>=93))**2 for x in pool)/len(pool))
    regression_rows=rows(OUT/"regression_metrics.csv")
    for r in regression_rows:
        pool=parts[r["split"],"Q90_upper" if r["group"]=="Q90_upper" else "B_score"]
        condition=r["condition"]
        if condition.startswith("surge_"):
            boundary=93 if condition.endswith("q95") else c["boundary_sensitivity"]
            pool=[x for x in pool if float(x["increase"])>=boundary]
        elif condition=="low_stay":
            pool=[x for x in pool if x["power_transition"]=="low->low"]
        values=[]
        for x in pool:
            prediction=float(x["upper_prediction"]) if r["group"]=="Q90_upper" else float(inherited[r["group"],x["timestamp"]]["prediction"])
            values.append(prediction-float(x["actual"]))
        weights=[float(x["daily_maximum_weight"]) for x in pool]
        expected={"hours":len(pool),"MAE":math.fsum(abs(e) for e in values)/len(pool),"mean_under":math.fsum(max(-e,0) for e in values)/len(pool),
            "mean_over":math.fsum(max(e,0) for e in values)/len(pool),"coverage":sum(e>=0 for e in values)/len(pool),
            "pinball90":math.fsum(.9*max(-e,0)+.1*max(e,0) for e in values)/len(pool)}
        if math.fsum(weights):
            expected["daily_maximum_MAE"]=math.fsum(abs(e)*w for e,w in zip(values,weights))/math.fsum(weights)
        for field,value in expected.items():
            close(r[field],value)
    table=rows(OUT/"selection_metrics.csv")
    baseline=next(r for r in table if r["group"]=="B_score")
    for r in table:
        pool=groups[r["group"]]
        y=[float(x["increase"])>=93 for x in pool]
        scores=[float(x["score"]) for x in pool]
        close(r["AP"],ap(y,scores))
        monthly=[x for x in alarm_rows if x["group"]==r["group"] and x["cap"]=="0.01" and x["boundary_name"]=="q95"]
        close(r["weighted_monthly_AP"],math.fsum(float(x["AP"])*int(x["events"]) for x in monthly)/8)
        tp=sum(x["alarm_0.01"]=="True" and label for x,label in zip(pool,y))
        fp=sum(x["alarm_0.01"]=="True" and not label for x,label in zip(pool,y))
        close(r["TP"],tp);close(r["FP"],fp)
        close(r["recall"],tp/8);close(r["precision"],tp/(tp+fp) if tp+fp else 0)
        close(r["false_positive_fraction"],fp/2176)
        monthly_ok=all(int(x["TP"])>=int(next(b for b in alarm_rows if b["group"]=="B_score" and b["split"]==x["split"] and b["cap"]=="0.01" and b["boundary_name"]=="q95")["TP"]) for x in monthly)
        flags={"rank_passed":float(r["weighted_monthly_AP"])>=1.05*float(baseline["weighted_monthly_AP"]),
            "recall_passed":tp>=int(baseline["TP"]),"monthly_recall_passed":monthly_ok,"false_positive_passed":fp/2176<=.01+1e-12,
            "strict_gain_passed":tp>int(baseline["TP"]) or fp<int(baseline["FP"])}
        for field,value in flags.items():
            assert (r[field]=="True")==value
        assert (r["eligible"]=="True")== (r["group"] in ["Logistic_all","HGB_all","Q90_upper"] and all(flags.values()))
    possible=[r for r in table if r["eligible"]=="True"]
    order=["Logistic_all","HGB_all","Q90_upper"]
    if possible:
        best=max(float(r["weighted_monthly_AP"]) for r in possible)
        chosen=min([r for r in possible if float(r["weighted_monthly_AP"])>=best/1.01],key=lambda r:(order.index(r["group"]),-float(r["weighted_monthly_AP"]))) ["group"]
    else:
        chosen="B_score"
    selection=json.loads((OUT/"selection.json").read_text(encoding="utf-8"))
    assert selection["selected_risk_channel"]==chosen and selection["normal_forecast"]=="B"
    df=pd.read_csv(ROOT/"Modeling/tables/m04_rise/feature_frame.csv",encoding="utf-8-sig",float_precision="round_trip").set_index("timestamp")
    reloads=0
    for r in rows(OUT/"model_manifest.csv"):
        path=ROOT/r["file"]
        assert sha(path)==r["sha256"]
        model=joblib.load(path)
        pool=parts[r["split"],r["name"]]
        columns=run["features"]["quantile" if r["name"]=="Q90_upper" else "classifier"]
        inputs=df.loc[[x["timestamp"] for x in pool],columns]
        if r["name"]=="Q90_upper":
            actual=model.predict(inputs)
            expected=[float(x["upper_prediction"]) for x in pool]
            assert model.loss=="quantile" and model.quantile==.9
        else:
            actual=np.full(len(pool),model["constant_probability"]) if isinstance(model,dict) else model.predict_proba(inputs)[:,1]
            expected=[float(x["score"]) for x in pool]
            if r["name"]=="Logistic_all" and not isinstance(model,dict):
                start=min(x["timestamp"] for x in pool)
                training=df.loc[df.index<start,columns]
                assert np.allclose(model[0].mean_,training.mean().to_numpy(),atol=1e-9,rtol=1e-10)
        assert np.allclose(actual,expected,atol=1e-9,rtol=0)
        reloads+=1
    assert reloads==9 and sum(run["fits"].values())==378
    result={"status":"passed","checked_at":datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
        "run_sha256":sha(OUT/"run.json"),"script_sha256":sha(Path(__file__)),"prediction_rows":len(pred),"inner_rows":len(inner),
        "trials_independently_recomputed":checked_trials,"model_reload_checks":reloads,"policies_checked":len(policies),
        "alarm_rows":len(alarm_rows),"regression_rows":len(regression_rows),"normal_forecast_unchanged":True,
        "selected_risk_channel":chosen,"no_july_august_evaluation":True}
    (OUT/"independent_verification.json").write_text(json.dumps(result,indent=2),encoding="utf-8")
    print(json.dumps(result),flush=True)


if __name__=="__main__":
    main()


코드 등록 완료: `Modeling/scripts/m04_surge_risk_verify.py`

### `Modeling/scripts/m04_surge_verify.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Independent raw maximum, January quantile, errors and alarm-policy checks.

주요 함수: `quantile()`, `alarm()`, `frontier()`, `check_alarm()`, `main()`.

In [59]:
%%submission_source Modeling/scripts/m04_surge_verify.py
"""Independent raw maximum, January quantile, errors and alarm-policy checks."""
import json
import math
import sys
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path

from m03_verify import rows, sha, close
from m04_rise_verify import ap

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/m04_surge"


def quantile(values, fraction):
    ordered = sorted(values)
    position = (len(ordered) - 1) * fraction
    lower = math.floor(position)
    upper = math.ceil(position)
    return ordered[lower] + (position-lower) * (ordered[upper]-ordered[lower])


def alarm(y, scores, threshold):
    predicted = [value >= threshold for value in scores]
    tp = sum(a and b for a, b in zip(predicted, y))
    fp = sum(a and not b for a, b in zip(predicted, y))
    positives = sum(y)
    negatives = len(y)-positives
    return {"hours": len(y), "events": positives, "TP": tp, "FP": fp, "FN": positives-tp, "TN": negatives-fp,
        "recall": tp/positives if positives else 0., "precision": tp/(tp+fp) if tp+fp else 0.,
        "false_positive_fraction": fp/negatives if negatives else 0., "threshold": threshold}


def frontier(y, scores, cap):
    candidates = [math.nextafter(max(scores), math.inf)] + sorted(set(scores), reverse=True)
    acceptable = [alarm(y, scores, t) for t in candidates]
    acceptable = [r for r in acceptable if r["false_positive_fraction"] <= cap+1e-12]
    return min(acceptable, key=lambda r: (-r["TP"], r["FP"], -r["threshold"]))


def check_alarm(saved, pool, scores, boundary):
    labels = [float(r["increase"]) >= boundary for r in pool]
    expected = alarm(labels, scores, float(saved["threshold"]))
    for key, value in expected.items():
        close(saved[key], value)
    close(saved["AP"], ap(labels, scores))
    positive_scores = [s for y, s in zip(labels, scores) if y]
    negative_scores = [s for y, s in zip(labels, scores) if not y]
    auc = math.fsum(float(p>n)+.5*(p==n) for p in positive_scores for n in negative_scores)/(len(positive_scores)*len(negative_scores))
    close(saved["AUROC"], auc)
    close(saved["prevalence"], sum(labels)/len(labels))


def main():
    assert sys.version_info[:2] == (3,13) and sys._is_gil_enabled()
    run = json.loads((OUT / "run.json").read_text(encoding="utf-8"))
    cpath = ROOT / "Modeling/config/m04_surge_contract.json"
    c = json.loads(cpath.read_text(encoding="utf-8"))
    assert sha(cpath) == run["contract_sha256"] and sha(ROOT / "Modeling/scripts/m04_surge_diagnose.py") == run["script_sha256"]
    assert run["status"] == "completed" and run["no_fitting"] and run["no_july_august_evaluation"]
    for section in ["inputs_sha256", "outputs_sha256"]:
        for name, expected in run[section].items():
            assert sha(ROOT / name if section=="inputs_sha256" else OUT/name) == expected
    source = ROOT / "data/origin/okm_augumented_2021.csv"
    assert sha(source) == c["source_sha256"]
    raw = {}
    for r in rows(source):
        if not 20210101 <= int(r["날짜"]) <= 20210630 or not 0<=int(r["시간"])<=23:
            continue
        timestamp = datetime.strptime(r["날짜"], "%Y%m%d") + timedelta(hours=int(r["시간"]))
        assert timestamp not in raw
        slots = [float(r[name]) for name in ["15분","30분","45분","60분"]]
        raw[timestamp] = {"maximum":max(slots), "mean":math.fsum(slots)/4, "rounded":float(r["평균"])}
    eligible = {t for t in raw if all(t-timedelta(hours=lag) in raw for lag in [1,2,24,168])}
    frame = {}
    for t in eligible:
        before = raw[t-timedelta(hours=1)]
        frame[t] = {"timestamp":str(t), "date":t.date().isoformat(), "increase":raw[t]["maximum"]-before["maximum"],
            "maximum":raw[t]["maximum"], "previous_maximum":before["maximum"],
            "prior_low":before["maximum"]>0 and 20<=math.floor(before["mean"]+.5)<=26,
            "legacy_rise":before["maximum"]>0 and 20<=math.floor(before["mean"]+.5)<=26 and raw[t]["rounded"]>26}
    january = [r for t,r in frame.items() if t.month==1]
    positive = [r["increase"] for r in january if r["increase"]>0]
    boundaries = {"q95":quantile(positive,.95), "q90":quantile(positive,.90)}
    summary = json.loads((OUT / "summary.json").read_text(encoding="utf-8"))
    close(summary["boundary_primary"], boundaries["q95"])
    close(summary["boundary_sensitivity"], boundaries["q90"])
    assert len(january)==summary["calibration_hours"]==576 and len(positive)==summary["calibration_positive_hours"]==233
    for r in rows(OUT/"event_counts.csv"):
        role = r["role"]
        selected = january if role=="calibration_jan" else [x for t,x in frame.items() if t<datetime(2021,4,1)] if role=="train_jan_mar" else [x for t,x in frame.items() if t.month=={"dev_apr":4,"dev_may":5,"dev_jun":6}[role]]
        boundary = boundaries[r["boundary_name"]]
        events = [x for x in selected if x["increase"]>=boundary]
        expected = {"hours":len(selected), "positive_increases":sum(x["increase"]>0 for x in selected),
            "surge_hours":len(events), "surge_dates":len({x["date"] for x in events}),
            "old_rise_hours":sum(x["legacy_rise"] for x in selected), "overlap_hours":sum(x["legacy_rise"] for x in events),
            "outside_prior_low_surges":sum(not x["prior_low"] for x in events), "previous_zero_hours":sum(x["previous_maximum"]==0 for x in selected)}
        for key,value in expected.items():
            close(r[key],value)
    pred = rows(OUT/"predictions.csv")
    original = {(r["group"],r["timestamp"]):r for r in rows(ROOT/"Modeling/tables/m04_rise_gate/predictions.csv")}
    groups = defaultdict(list)
    for r in pred:
        t = datetime.fromisoformat(r["timestamp"])
        assert t.month in [4,5,6] and t in eligible
        previous = frame[t]
        close(r["actual"],previous["maximum"])
        close(r["lag1_maximum"],previous["previous_maximum"])
        close(r["increase"],previous["increase"])
        close(r["prediction"],original[r["group"],r["timestamp"]]["prediction"])
        close(r["predicted_increase"],float(r["prediction"])-previous["previous_maximum"])
        assert (r["surge"]=="True")== (previous["increase"]>=boundaries["q95"])
        groups[r["group"]].append(r)
    assert len(pred)==run["prediction_rows"]==15288 and all(len(g)==2184 for g in groups.values())
    for r in rows(OUT/"errors.csv"):
        part = groups[r["group"]]
        if r["split"]!="pooled_development":
            part = [x for x in part if x["split"]==r["split"]]
        boundary = boundaries[r["boundary_name"]]
        condition = r["condition"]
        if condition=="surge":
            part = [x for x in part if float(x["increase"])>=boundary]
        elif condition=="non_surge":
            part = [x for x in part if float(x["increase"])<boundary]
        elif condition=="legacy_rise":
            part = [x for x in part if x["rise_event"]=="True"]
        elif condition=="low_stay":
            part = [x for x in part if x["power_transition"]=="low->low"]
        errors = [float(x["prediction"])-float(x["actual"]) for x in part]
        expected = {"hours":len(part), "dates":len({x["date"] for x in part}), "MAE":math.fsum(abs(e) for e in errors)/len(part),
            "RMSE":math.sqrt(math.fsum(e*e for e in errors)/len(part)), "mean_under":math.fsum(max(-e,0) for e in errors)/len(part),
            "mean_over":math.fsum(max(e,0) for e in errors)/len(part), "under_hours":sum(e<0 for e in errors),
            "mean_actual":math.fsum(float(x["actual"]) for x in part)/len(part), "mean_increase":math.fsum(float(x["increase"]) for x in part)/len(part)}
        for key,value in expected.items():
            close(r[key],value)
    cached = rows(ROOT/"Modeling/tables/m04_rise/inner_predictions.csv")
    policies = rows(OUT/"alarm_policies.csv")
    for policy in policies:
        group = "B" if policy["score"]=="B_predicted_increase" else "G_HGB"
        pool = [r for r in cached if r["split"]==policy["split"] and r["group"]==group]
        y = [frame[datetime.fromisoformat(r["timestamp"])]["increase"]>=boundaries["q95"] for r in pool]
        scores = [float(r["prediction"])-frame[datetime.fromisoformat(r["timestamp"])]["previous_maximum"] if group=="B" else float(r["probability"]) if r["prior_low"]=="True" else 0. for r in pool]
        chosen = frontier(y,scores,float(policy["cap"]))
        for key,value in chosen.items():
            close(policy[key],value)
        assert max(r["timestamp"] for r in pool)==policy["inner_last"]<policy["outer_first"]
    alarms = rows(OUT/"alarm_metrics.csv")
    for r in alarms:
        pool = groups[r["group"]]
        if r["split"]!="pooled_development":
            pool = [x for x in pool if x["split"]==r["split"]]
        boundary = boundaries[r["boundary_name"]]
        scores = [float(x["probability"]) if x["prior_low"]=="True" else 0. for x in pool] if r["score"]=="legacy_HGB_probability" else [float(x["predicted_increase"]) for x in pool]
        check_alarm(r,pool,scores,boundary)
        if r["policy"]=="past_inner_selected":
            policy = next(x for x in policies if x["split"]==r["split"] and x["score"]==r["score"] and x["cap"]==r["cap"])
            close(r["threshold"],policy["threshold"])
        elif r["policy"]=="descriptive_outer_frontier":
            chosen = frontier([float(x["increase"])>=boundary for x in pool],scores,float(r["cap"]))
            for key,value in chosen.items():
                close(r[key],value)
    april = [r for r in groups["G_HGB"] if r["split"]=="dev_apr" and r["prior_low"]=="True"]
    indexed = {(r["group"],r["timestamp"]):r for r in pred}
    for case in rows(OUT/"april_legacy_cases.csv"):
        actual = indexed["G_HGB",case["timestamp"]]
        assert actual["rise_event"]=="True" and case["trigger"]==actual["trigger"]
        p = float(case["probability"])
        assert int(case["rank_min"])==1+sum(float(r["probability"])>p for r in april)
        assert int(case["rank_max"])==sum(float(r["probability"])>=p for r in april)
        assert (case["surge"]=="True")== (float(case["increase"])>=boundaries["q95"])
        for name,label in [("B","prediction_B"),("W_rise_weighted","prediction_expert"),("G_HGB","prediction_combined")]:
            close(case[label],indexed[name,case["timestamp"]]["prediction"])
    result = {"status":"passed", "checked_at":datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
        "run_sha256":sha(OUT/"run.json"), "script_sha256":sha(Path(__file__)), "boundary":boundaries,
        "prediction_rows":len(pred), "error_rows":len(rows(OUT/"errors.csv")), "alarm_rows":len(alarms),
        "policies_checked":len(policies), "april_cases_checked":8, "raw_common_hours_checked":len(frame),
        "no_fitting":True, "no_july_august_evaluation":True}
    (OUT/"independent_verification.json").write_text(json.dumps(result,indent=2),encoding="utf-8")
    print(json.dumps(result),flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/m04_surge_verify.py`

### `Modeling/scripts/m04_verify.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Independent CSV/raw-time/metric/selection checks for M04; no model fitting.

주요 함수: `sha()`, `state()`, `main()`, `documents()`.

In [60]:
%%submission_source Modeling/scripts/m04_verify.py
"""Independent CSV/raw-time/metric/selection checks for M04; no model fitting."""
import argparse
import csv
import hashlib
import json
import math
import re
import sys
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path

from m03_verify import close, rows, verify_metric

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/m04"


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def state(maximum, mean):
    if maximum == 0:
        return "zero"
    rounded = math.floor(mean + .5)
    return "below20" if rounded < 20 else "low" if rounded <= 26 else "above26"


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    cpath = ROOT / "Modeling/config/m04_contract.json"
    c = json.loads(cpath.read_text(encoding="utf-8"))
    run = json.loads((OUT / "run.json").read_text(encoding="utf-8"))
    assert run["status"] == "completed" and run["no_fitting"] and run["no_july_august_evaluation"]
    assert sha(cpath) == run["contract_sha256"]
    assert sha(ROOT / "Modeling/scripts/m04_compare.py") == run["script_sha256"]
    for path, expected in run["inputs_sha256"].items():
        assert sha(ROOT / path) == expected
    for name, expected in run["outputs_sha256"].items():
        assert sha(OUT / name) == expected
    m1 = json.loads((ROOT / "Modeling/config/m01_contract.json").read_text(encoding="utf-8"))
    source = ROOT / m1["source"]
    assert sha(source) == m1["source_sha256"]
    raw = {}
    for r in rows(source):
        if not m1["period"][0] <= int(r["날짜"]) <= m1["period"][1] or not 0 <= int(r["시간"]) <= 23:
            continue
        t = datetime.strptime(r["날짜"], "%Y%m%d") + timedelta(hours=int(r["시간"]))
        assert t not in raw
        values = [float(r[name]) for name in ["15분", "30분", "45분", "60분"]]
        raw[t] = {"mean": sum(values) / 4, "maximum": max(values), "last": values[-1], "production": float(r["생산량"])}
    parent = {(r["group"], r["timestamp"]): r for r in rows(ROOT / c["parent_predictions"])
              if r["target"] == c["target"] and r["group"] in ["A", "B"]}
    pred = rows(OUT / "predictions.csv")
    groups = defaultdict(list)
    index = {}
    rule_name = c["rule"]["name"]
    changed = 0
    origin_values_checked = 0
    for r in pred:
        group, timestamp = r["group"], r["timestamp"]
        assert (group, timestamp) not in index
        index[group, timestamp] = r
        groups[group].append(r)
        t = datetime.fromisoformat(timestamp)
        assert t.month in [4, 5, 6] and r["split"] == {4: "dev_apr", 5: "dev_may", 6: "dev_jun"}[t.month]
        current, previous, previous2 = raw[t], raw[t - timedelta(hours=1)], raw[t - timedelta(hours=2)]
        close(r["actual"], current["maximum"])
        assert r["lag1_timestamp"] == str(t - timedelta(hours=1))
        for lag, record in [(1, previous), (2, previous2)]:
            for name in ["mean", "maximum", "last"]:
                close(r[f"lag{lag}_{name}"], record[name])
                origin_values_checked += 1
        close(r["lag1_production"], previous["production"])
        close(r["lag1_last_minus_mean"], previous["last"] - previous["mean"])
        close(r["lag1_last_minus_lag2_last"], previous["last"] - previous2["last"])
        prior = state(previous["maximum"], previous["mean"])
        assert r["prior_state"] == prior
        # Target state's historical diagnostic definition uses the supplied rounded mean.
        target_raw = math.floor(current["mean"] + .5)
        assert r["power_transition"] == prior + "->" + state(current["maximum"], target_raw)
        parent_group = group if group in ["A", "B"] else "B"
        original = parent[parent_group, timestamp]
        for field in ["date", "split", "power_transition", "diag_production_transition", "train_profile_overlap", "diag_target_profile"]:
            assert r[field] == original[field]
        for field in ["daily_maximum_weight", "profile_weight"]:
            close(r[field], original[field])
        applied = group == rule_name and prior == "low"
        assert (r["rule_applied"] == "True") == applied
        prediction = previous["maximum"] if group == "lag1" or applied else float(original["prediction"])
        close(r["prediction"], prediction)
        if applied:
            changed += 1
        error = prediction - current["maximum"]
        for field, value in [("signed_error", error), ("absolute_error", abs(error)),
                             ("under_amount", max(-error, 0)), ("over_amount", max(error, 0))]:
            close(r[field], value)
    assert set(groups) == {"A", "B", "lag1", rule_name}
    times = {r["timestamp"] for r in groups["B"]}
    assert len(times) == 2184 and all({r["timestamp"] for r in g} == times for g in groups.values())
    dates = defaultdict(list)
    for r in groups["B"]:
        dates[r["date"]].append(r)
    assert len(dates) == 91
    peak_hours = 0
    for date, g in dates.items():
        assert len(g) == 24
        maximum = max(float(r["actual"]) for r in g)
        ties = sum(float(r["actual"]) == maximum for r in g)
        for r in g:
            expected = 1 / ties if float(r["actual"]) == maximum else 0
            close(r["daily_maximum_weight"], expected)
            peak_hours += expected > 0
    assert peak_hours == 137 and changed == 650
    metric_rows = rows(OUT / "metrics.csv")
    for r in metric_rows:
        g = groups[r["group"]]
        kind, value = r["kind"], r["value"]
        weight_kind = "all"
        if kind in ["daily_maximum", "profile_reweighted"]:
            weight_kind = kind
            subset = [x for x in g if float(x["daily_maximum_weight"]) > 0] if kind == "daily_maximum" else g
        elif kind.startswith("peak_"):
            weight_kind = "daily_maximum"
            subset = [x for x in g if float(x["daily_maximum_weight"]) > 0 and x[kind[5:]] == value]
        else:
            subset = g if kind == "all" else [x for x in g if x[kind] == value]
        verify_metric(r, subset, weight_kind)
        close(r["absolute_error_share"], math.fsum(float(x["absolute_error"]) for x in subset) /
              math.fsum(float(x["absolute_error"]) for x in g))
    pairs = rows(OUT / "paired_predictions.csv")
    assert len(pairs) == 4368
    for r in pairs:
        before, after = ("A", "B") if r["comparison"] == "A_to_B" else ("B", rule_name)
        for side, group in [("before", before), ("after", after)]:
            for field in ["actual", "prediction", "absolute_error", "under_amount", "over_amount"]:
                close(r[field + "_" + side], index[group, r["timestamp"]][field])
        for field, short in [("absolute_error", "abs"), ("under_amount", "under"), ("over_amount", "over")]:
            close(r["delta_" + short], float(r[field + "_after"]) - float(r[field + "_before"]))
    daily = rows(OUT / "daily_errors.csv")
    assert len(daily) == 91
    for r in daily:
        subset = [x for x in pairs if x["comparison"] == "B_to_rule" and x["date"] == r["date"]]
        assert len(subset) == int(r["hours"]) == 24
        for field, source_field in [("MAE_B", "absolute_error_before"), ("MAE_rule", "absolute_error_after"),
                ("delta_MAE", "delta_abs"), ("delta_under", "delta_under"), ("delta_over", "delta_over")]:
            close(r[field], math.fsum(float(x[source_field]) for x in subset) / 24)
    selection = json.loads((OUT / "selection.json").read_text(encoding="utf-8"))
    tol = c["selection"]["numeric_tolerance"]
    def cell(group, kind, value):
        result = [r for r in metric_rows if r["group"] == group and r["kind"] == kind and r["value"] == value]
        assert len(result) == 1
        return result[0]
    expected_gates = [("all", "all", "MAE", True), ("daily_maximum", "all", "MAE", False),
                      ("daily_maximum", "all", "mean_under", False), ("power_transition", "low->above26", "mean_under", False)]
    assert len(selection["gates"]) == len(expected_gates)
    for gate, expected in zip(selection["gates"], expected_gates):
        kind, value, name, strict = expected
        assert (gate["kind"], gate["value"], gate["metric"], gate["strict_decrease"]) == expected
        reference, candidate = float(cell("B", kind, value)[name]), float(cell(rule_name, kind, value)[name])
        close(gate["B"], reference)
        close(gate["rule"], candidate)
        close(gate["delta"], candidate - reference)
        assert gate["passed"] == (candidate - reference < -tol if strict else candidate - reference <= tol)
    accepted = all(g["passed"] for g in selection["gates"])
    assert selection["status"] == ("accepted" if accepted else "rejected")
    assert selection["selected"] == (rule_name if accepted else "B")
    assert selection["rule_applied_hours"] == changed
    low = rows(OUT / "prior_low_cases.csv")
    assert len(low) == 650 and len({r["timestamp"] for r in low}) == 650
    for r in low:
        assert index["B", r["timestamp"]]["prior_state"] == "low"
        for group, suffix in [("B", "B"), (rule_name, "rule")]:
            for field in ["prediction", "absolute_error", "under_amount", "over_amount"]:
                close(r[field + "_" + suffix], index[group, r["timestamp"]][field])
    for r in rows(OUT / "origin_feature_summary.csv"):
        values = sorted(float(x[r["feature"]]) for x in low if x["power_transition"] == r["power_transition"])
        assert len(values) == int(r["hours"])
        close(r["mean"], math.fsum(values) / len(values))
        close(r["min"], values[0])
        close(r["max"], values[-1])
        for field, fraction in [("q25", .25), ("median", .5), ("q75", .75)]:
            pos = (len(values) - 1) * fraction
            lo, hi = math.floor(pos), math.ceil(pos)
            close(r[field], values[lo] + (values[hi] - values[lo]) * (pos - lo))
    for r in rows(OUT / "origin_hour_counts.csv"):
        assert int(r["size"]) == sum(x["split"] == r["split"] and x["hour"] == r["hour"] and
                                    x["power_transition"] == r["power_transition"] for x in low)
    signatures = defaultdict(list)
    for r in low:
        t = datetime.fromisoformat(r["timestamp"])
        key = (t.month, t.weekday(), t.hour) + tuple(float(r[x]) for x in [
            "lag1_production", "lag1_mean", "lag1_maximum", "lag2_mean", "lag2_maximum", "lag1_last"])
        signatures[key].append(r)
    conflicting = [g for g in signatures.values() if len({x["power_transition"] for x in g}) > 1]
    collision_rows = rows(OUT / "origin_signature_collisions.csv")
    assert len(conflicting) == len(collision_rows) == selection["origin_signature_collisions"]
    actual_collisions = sorted((len(g), len({x["date"] for x in g}), min(float(x["actual"]) for x in g),
        max(float(x["actual"]) for x in g), len({x["power_transition"] for x in g})) for g in conflicting)
    saved_collisions = sorted((int(r["hours"]), int(r["dates"]), float(r["actual_min"]),
        float(r["actual_max"]), int(r["transitions"])) for r in collision_rows)
    assert actual_collisions == saved_collisions
    assert selection["days_improved"] == sum(float(r["delta_MAE"]) < -tol for r in daily)
    assert selection["days_worsened"] == sum(float(r["delta_MAE"]) > tol for r in daily)
    assert selection["days_unchanged"] == sum(abs(float(r["delta_MAE"])) <= tol for r in daily)
    output = {"status": "passed", "checked_at": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
        "run_sha256": sha(OUT / "run.json"), "script_sha256": sha(Path(__file__)),
        "raw_origin_values_checked": origin_values_checked, "prediction_rows": len(pred),
        "metric_rows": len(metric_rows), "paired_rows": len(pairs), "daily_rows": len(daily),
        "rule_applied_hours": changed, "identical_evaluation_hours": len(times),
        "daily_maximum_hours": peak_hours, "daily_maximum_dates": len(dates),
        "origin_signature_collisions_checked": len(conflicting),
        "selection_checked": True, "no_fitting": True, "no_july_august_evaluation": True}
    (OUT / "independent_verification.json").write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(output, ensure_ascii=False), flush=True)


def documents():
    verified = json.loads((OUT / "independent_verification.json").read_text(encoding="utf-8"))
    assert verified["status"] == "passed" and verified["run_sha256"] == sha(OUT / "run.json")
    assert verified["script_sha256"] == sha(Path(__file__))
    manuscript_path = ROOT / "Modeling/04_Modeling_원고.md"
    manuscript = manuscript_path.read_text(encoding="utf-8")
    section = manuscript.split("## 4.6 M04:")[1]
    metrics = rows(OUT / "metrics.csv")
    checks = [("전체 MAE", "all", "all", "MAE"), ("전체 RMSE", "all", "all", "RMSE"),
        ("실제 일별 최대 시간 MAE", "daily_maximum", "all", "MAE"),
        ("실제 일별 최대 시간 평균 과소예측", "daily_maximum", "all", "mean_under"),
        ("낮은 구간 유지 MAE", "power_transition", "low->low", "MAE"),
        ("상승 전환 평균 과소예측", "power_transition", "low->above26", "mean_under"),
        ("낮은 유지 중 일별 최대 시간 MAE", "peak_power_transition", "low->low", "MAE"),
        ("전체 평균 과소예측", "all", "all", "mean_under")]
    for label, kind, value, field in checks:
        numbers = []
        for group in ["B", "B_prior_low_persistence"]:
            found = [r for r in metrics if (r["group"], r["kind"], r["value"]) == (group, kind, value)]
            assert len(found) == 1
            numbers.append(f"{float(found[0][field]):.3f}")
        assert f"| {label} | {numbers[0]} | {numbers[1]} |" in section, label
    contribution_checks = [("26초과 상태 유지", "above26->above26", 1),
        ("낮은 구간 유지", "low->low", 1), ("낮은 구간→26초과", "low->above26", 1),
        ("26초과→낮은 구간", "above26->low", 1), ("20미만→낮은 구간", "below20->low", 2),
        ("낮은 구간→20미만", "low->below20", 2)]
    for label, value, places in contribution_checks:
        r = next(r for r in metrics if (r["group"], r["kind"], r["value"]) == ("B", "power_transition", value))
        expected = f'| {label} | {int(r["hours"]):,} | {float(r["MAE"]):.3f} | {100 * float(r["absolute_error_share"]):.{places}f}% |'
        assert expected in section, expected
    docs = [manuscript_path, ROOT / "Modeling/README.md", ROOT / "README.md",
            ROOT / "Modeling/10.05_M04_조건별오류와낮은전력규칙_실행기록.md"]
    links = 0
    for path in docs:
        text = path.read_text(encoding="utf-8")
        if path == ROOT / "README.md":
            text = text.split("## 이전 jsw 작업 단계 기록")[0]
        for target in re.findall(r'!?\[[^\]]*\]\(([^)]+)\)', text):
            if target.startswith(("http:", "https:", "#", "app:")):
                continue
            linked = (path.parent / target.split("#")[0].strip("<>")).resolve()
            assert linked.exists(), (path, target)
            links += 1
    result = {"status": "passed", "checked_at": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
        "numeric_rows": len(checks) + len(contribution_checks), "local_links": links, "manuscript_sha256": sha(manuscript_path),
        "script_sha256": sha(Path(__file__))}
    (OUT / "document_verification.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result), flush=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--documents", action="store_true")
    args = parser.parse_args()
    documents() if args.documents else main()


코드 등록 완료: `Modeling/scripts/m04_verify.py`

### `Modeling/scripts/m05_documents.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Write and verify the M05 detailed manuscript from independently checked results.

주요 함수: `metric()`, `comparison()`, `months()`, `alarms()`, `section()`, `write()`, `verify()`, `main()`.

In [61]:
%%submission_source Modeling/scripts/m05_documents.py
"""Write and verify the M05 detailed manuscript from independently checked results."""
import argparse
import json
import re
from datetime import datetime, timedelta, timezone
from pathlib import Path
from m03_verify import rows, sha

ROOT=Path(__file__).resolve().parents[2]
OUT=ROOT/'Modeling/tables/m05'
MD=ROOT/'Modeling/04_Modeling_원고.md'
LOG=ROOT/'Modeling/10.05_M05_고정후반기평가_실행기록.md'
TITLE='## 4.9 M05: 고정 후반기 평가와 성과 범위의 갱신'


def metric(group, condition='all', period='pooled', pool='common'):
    return next(r for r in rows(OUT/'metrics.csv') if r['pool']==pool and r['group']==group and r['period']==period and r['condition']==condition and r['value']=='all')


def comparison():
    text='| 방법 | 전체 MAE | 실제 일별 최대 시간 MAE | 최대 시간 평균 과소 | 최대 시간 평균 과대 |\n|---|---:|---:|---:|---:|\n'
    for name in ['lag1','lag24','lag168','A','B','G_aux']:
        overall,peak=metric(name),metric(name,'daily_maximum')
        text+=f"| {name} | {float(overall['MAE']):.3f} | {float(peak['MAE']):.3f} | {float(peak['mean_under']):.3f} | {float(peak['mean_over']):.3f} |\n"
    return text


def months():
    text='| 月 | A 전체 MAE | B 전체 MAE | A 최대 시간 MAE | B 최대 시간 MAE |\n|---|---:|---:|---:|---:|\n'.replace('月','월')
    for month in ['7','8']:
        values=[metric(name,cond,month) for cond in ['all','daily_maximum'] for name in ['A','B']]
        text+=f'| {month}월 | '+' | '.join(f"{float(r['MAE']):.3f}" for r in values)+' |\n'
    return text


def alarms():
    records=rows(OUT/'risk_metrics.csv')
    text='| 기간 | 위험 점수 | 급등 수 | TP / FP / FN / TN | 재현율 | 정밀도 | 실제 오경보율 | AP |\n|---|---|---:|---|---:|---:|---:|---:|\n'
    for period in ['pooled','7','8']:
        for group in ['B_score','G_HGB_score']:
            r=next(r for r in records if r['group']==group and r['period']==period and r['cap']=='0.01' and r['boundary']=='93.0')
            text+=f"| {'7~8월' if period=='pooled' else period+'월'} | {group} | {r['events']} | "+' / '.join(r[key] for key in ['TP','FP','FN','TN'])+' | '+' | '.join(f'{100*float(r[key]):.1f}%' for key in ['recall','precision'])+f" | {100*float(r['false_positive_fraction']):.3f}% | {float(r['AP']):.3f} |\n"
    return text


def section():
    diagnostics=json.loads((OUT/'confirmed_diagnostics.json').read_text(encoding='utf-8'))
    c=json.loads((ROOT/'Modeling/config/m05_contract.json').read_text(encoding='utf-8'))
    base_a,base_b=metric('A'),metric('B')
    core_a,core_b=metric('A',pool='core'),metric('B',pool='core')
    core_peak_a,core_peak_b=metric('A','daily_maximum',pool='core'),metric('B','daily_maximum',pool='core')
    limits={r['group']:r['threshold'] for r in c['alarms']['thresholds'] if r['cap']==.01}
    gain=100*(1-float(metric('G_aux','surge_q95')['MAE'])/float(metric('B','surge_q95')['MAE']))
    return f'''{TITLE}

M03에서 마지막 값 추가 B는 개발 전체 MAE와 실제 일별 최대 시간 과소예측을 줄였지만, M04에서는 상승 전환·낮은 유지의 상충이 남았다. 일반 급등에서 기존 결합의 크기 오차가 줄고 위험 점수도 일부 탐지를 늘려 후속 후보로 연결했다. M05는 이 두 효과가 후반기에도 유지되는지 확인하기 위해 실행했다. [실행 조건](config/m05_contract.json)은 {c['recorded_at']}에 기록했으며, 그 뒤 모델을 학습하고 후반기 예측을 생성했다.

### 학습·평가와 고정한 정보

1~6월 공통 4,176시간으로 A·B HGB, 상승 가중 전문가, 직전 낮은 상태 HGB 분류기를 각각 한 번 학습했다. 분류기의 실제 학습 대상은 직전 낮은 상태 1,305시간이며 가장 늦은 대상은 6월28일07시다. 모델4개를 저장·재로딩했다. 평가 중에는 매시간 t-1 관측을 입력에 반영하되 모델·결합 규칙·경보 경계를 고정했다. 7월 정답을 추가한 8월 재학습이나 후반기 Optuna 탐색은 없었다.

주 비교는 단순 기준3개와 A/B를 같은 공통 1,344시간·56일에서 평가했다. 7월600시간·8월744시간이며 실제 일별 최대 시간은77행·날짜 가중치 합56이다. 기본 시차만 필요한1,436시간은 같은 A/B 모델의 추론 범위를 확장한 보조 평가다. 보조 표본에서 다시 학습하지 않았고 두 표본 점수를 섞지 않았다. 후반기는 이전 EDA 관찰 이력이 있어 완전히 처음 보는 독립 시험으로 표현하지 않는다.

A는17입력, B는 직전 마지막 값을 더한18입력이고 기존 HGB 잎15·반복150·학습률0.05·최소 잎 표본20·L2 1·조기종료 안 함·시드42를 고정했다. 위험 후보는 마지막 개발월6월의 설정을 선택한 뒤 1~6월로 재적합했다. 상승 학습 가중치1.730905, 분류기 잎15·최소 잎 표본20·L2 0.155280, 결합 확률 경계0.373766·비율0.822355를 사용했다. 이는 후반기 점수가 가장 좋은 설정을 고른 결과가 아니다.

급등 정답 경계는1월 양의 증가95분위수93을 유지했다. 경보 수치 경계는 기존4~6월의 시간순 외부 예측에서 비급등 점수의 음성 꼬리1%로 산정했다. B 위험 점수 경계는 {limits['B_score']:.6f}, 기존 결합 위험 점수는 {limits['G_HGB_score']:.6f}이며 두 평가 월에 같은 값을 썼다. 급등 정답 경계93, 결합 확률 경계, 경보 점수 경계는 각각 다른 역할이다. 학습 자료에 재대입한 예측이나7~8월 정답으로 경계를 고르지 않았다. [경보 산정표](tables/m05/alarm_calibration.csv)를 보존한다.

### 전체 및 실제 최대 시간의 비교

{comparison()}
G_aux는 위험 점수를 만들기 위한 보조 결합값이며 정상 전력 출력은 B다. 해당 행의 회귀 오차가 조금 낮더라도 이를 최종 전력 모델 교체로 해석하지 않는다.

B의 전체 MAE는 직전 기준12.678보다 {diagnostics['baseline_reduction_percent']:.1f}% 작았다. 실제 최대 시간 MAE도19.830→13.803으로 감소했다. A 역시 단순 기준보다 낮은 오차를 보였다. 따라서 과거 정보와 달력을 사용하는 HGB의 기본 예측 성과는 후반기에서도 확인됐다. 이는 마지막 값 한 변수의 추가 효과와 구분되는 모델 성과다.

A→B 전체 MAE는6.683→6.621로 {diagnostics['overall_reduction_percent']:.2f}% 감소했으나, 실제 최대 시간 MAE는13.376→13.803으로 {diagnostics['peak_MAE_change_percent']:.2f}% 증가했다. 최대 시간 평균 과소도12.936→13.341로 증가했다. 개발에서 나타난 전체8.1%·최대 시간 과소37.9% 감소가 후반기에서 같은 방향과 크기로 유지된 것은 아니다.

실행 전 확인 기준은 A 대비 전체·최대 시간 MAE 각각1% 이상 감소, 최대 시간 평균 과소 감소, 직전 기준보다 낮은 전체·최대 시간 MAE, 각 월의 A 대비 MAE 악화1% 이내, 배열 재가중 전체 MAE 비악화였다. 1%는 기존 수치상 동률 규칙을 이어받은 것이며 현장 비용의 최적 허용값이 아니다. [판단 파일](tables/m05/conclusion.json)에서 전체·최대 시간·과소·월별 요건이 실패했다. 단순 기준 개선과 배열 재가중 요건은 통과했지만, 마지막 값의 광범위한 효과는 확인되지 않았다.

### 월별·전력 상태·적용 범위의 차이

{months()}
7월은 전체·최대 시간 모두 B가 A보다 나빠졌고, 8월은 전체는 개선했지만 최대 시간 MAE가 소폭 증가했다. 후반기 공통 표본의 학습과 동일한 하루 배열은0일이었다. 전체 점수 하나만으로 개발의 효과가 새로운 배열에서도 안정적으로 유지됐다고 주장하지 않는다.

26초과 유지795시간의 MAE는 A8.041→B7.852, 26초과에서 낮은 상태로 내려간10시간은14.012→9.125로 감소했다. 반면 낮은 유지508시간은3.418→3.610, 상승 전환10시간은28.400→29.482로 증가했다. 전력 크기나 변화 방향에 따라 입력 추가의 이득이 달랐다. [조건별 결과](tables/m05/metrics.csv)와 [같은 시간의 A/B 오차 차이](tables/m05/paired_predictions.csv)를 남겼다.

보조1,436시간에서도 전체 MAE는 {float(core_a['MAE']):.3f}→{float(core_b['MAE']):.3f}로 줄었지만 최대 시간은 {float(core_peak_a['MAE']):.3f}→{float(core_peak_b['MAE']):.3f}로 늘었다. 주 표본에서 제외된 시간으로 범위를 넓혀도 최대 시간의 상충은 남았다. 좋은 점수의 표본만 선택해 결론을 바꾸지 않았다.

8월28일18시~29일10시의0값17시간에서 B는 평균24.047을 과대예측했다. 그 뒤 첫 양수 시간의 실제 최대값41은23.559로 예측해17.441만큼 작게 예상했다. 이 사례의 증가41은 주 급등 경계93에 미달하므로 급등 미탐지6건과 같은 사건으로 세지 않는다. 학습에는 최대값0 기록이 없었고, 실제0을 입력으로 받았을 때의 성능은 별도 한계로 확인됐다. 원인이나 설비 휴무 여부는 현 자료에서 식별하지 못했다. [0값 전후 사례](tables/m05/zero_cases.csv)를 보존한다.

### 급등 위험 출력의 탐지와 오경보

후반기 주 급등은6시간·6일이며7·8월 각각3건이다. 이 중4건은 직전 낮은 상태,2건은26초과 상태였다. 경보1% 조건의 결과는 다음과 같다.

{alarms()}
월별 AP의 사건 수 가중 평균은 B0.328→기존 결합0.488로 증가했다. 급등6건 중 탐지는3→4건, 오경보는16→14건으로 개선됐으나 실제 오경보율은1.196%→1.046%로 후보도 사전1% 요건을 넘었다. 정밀도는15.8%→22.2%로, 후보 경보18건 중 실제 급등은4건이다. 재현율66.7%만 제시하면 경보 부담을 판단할 수 없다.

7월에는 B1건→후보3건을 찾았지만8월에는 B2건→후보1건으로 감소했다. 8월9일08시는 두 점수가 모두57.887이었다. 직전 상태가26초과여서 결합값은 B와 같았지만, B 경계56.267은 넘고 후보 경계58.080은 넘지 못했다. 따라서 이 한 건의 추가 미탐지는 분류기의 순위 악화가 아니라 방법별 고정 경보 경계 차이에서 생겼다. 8월16일07시는 두 점수가 모두 경계 아래여서 두 방법 모두 놓쳤다. [급등6건](tables/m05/surge_cases.csv)·[검증된 점수/경보 사례](tables/m05/confirmed_diagnostics.json)를 함께 확인했다. 이 사례를 본 뒤 경계를 다시 낮추지는 않았다.

위험 후보는 가중 AP5% 이상 개선·합계 탐지 비악화·각 월 탐지 비악화·실제 오경보율1% 이내·탐지 증가 또는 오경보 감소를 함께 만족해야 했다. 합계 순위·탐지·오경보 수는 개선했지만 월별 탐지와 실제 오경보율이 실패해 최종 위험 채널의 안정적 우위를 확인하지 못했다. 주 급등 크기 MAE 자체는 B38.417→보조 결합32.100으로 {gain:.1f}% 감소했다. 이는 보조 결합값의 부분 성과이며 최종 채택을 의미하지 않는다.

증가61.6 이상은25시간·17일이었다. 보조 경계에서는 순위·정밀도 개선도 있었으나 사전에 민감도 점검으로 정한 범위다. 또한5% 경보 허용에서는 B가6건 모두 찾는 대신 오경보47건을 냈다. 이 조건의 좋은 탐지율로 주1% 평가를 대체하거나 '급등을 모두 잘 찾았다'고 주장하지 않는다. [전체 위험 지표](tables/m05/risk_metrics.csv)에 모든 조건을 보존했다.

### 보고서와 다음 작업의 판단 갱신

확인된 성과는 후반기에서도 HGB가 직전·전날·전주 기준보다 전체·실제 최대 시간 오차를 줄였다는 점이다. 유지할 근거는 M02의 동일 조건 모델 비교와 M03의 개발 입력 기여 결과이며, 수정할 주장은 마지막 값과 급등 결합이 후반기까지 안정적으로 최대 시간의 과소예측을 줄였다는 표현이다. 현재 결과에서는 그 강한 주장을 입증하지 못했다.

최종 보고서 제2장은 기본/확장 모델 선택과 후반기 단순 기준 대비 HGB 성과를 중심으로 구성할 수 있다. 마지막 값의 개발 개선은 후반기 결과와 함께 설명해야 한다. 사용자의 성과 중심 선별 방향에 따라 미채택 급등 분류·결합의 상세 실험표는 최종 성과 본문에서 제외할 수 있으나, 제3장에는 선택 후보의 상승·0구간·최대 시간 오류를 짧게 남긴다. 제5장에서 '급등 탐지 필살기를 최종 입증했다'고 쓰지 않는다. 상세 원고·코드·모델·검증 근거는 보존한다.

이번 M05는 선언한 고정 확인 평가로 종료한다. B의 개발 후보와 보조 위험 후보를 후반기 결과로 재튜닝하지 않았고, B의 최종 채택이나 위험 후보의 운영 적용을 확정하지 않았다. 다음 판단은 기존 A/B의 상충을 반영해 보고서의 주 성과와 모델 선택 근거를 정리할지, 현재 자료에서 상승·0구간에 대한 구체적인 새 가설을 세워 별도 개발을 시작할지다. 새 개발을 선택하면7~8월도 선택에 이용한 기간임을 기록하며 재평가를 새로운 독립 확인으로 표현하지 않는다. 현장 활용 제안은 확인된 범위의 성능과 실패조건에 맞춰야 한다.

### 실행 코드와 검증

[m05_evaluate.py](scripts/m05_evaluate.py)의 `--stage freeze`→`--stage run`으로 조건·경보 경계를 먼저 기록하고 학습·추론·결과를 생성했다. [m05_verify.py](scripts/m05_verify.py)는 원본에서149,040입력값,10,936예측행,1,000지표행,24경보행을 대조하고4모델을 재로딩했다. [m05_documents.py](scripts/m05_documents.py)는 검증된 결과를 상세 원고와 연결 기록에 반영하고 표·문서 링크를 검증한다. 새 Optuna 탐색은0회, 학습은4회이며 원자료·기존M01~M04 결과를 보존했다.

[실행 기록](tables/m05/run.json)·[독립 검증](tables/m05/independent_verification.json)·[저장 모델 목록](tables/m05/model_manifest.csv)·[학습 범위](tables/m05/training_audit.csv)·[전체 예측](tables/m05/predictions.csv)·[별도 위험 출력](tables/m05/risk_predictions.csv)·[일별 오류](tables/m05/daily_errors.csv)·[큰 오류](tables/m05/largest_errors.csv)를 함께 보존한다. 원고4.5의 전체 Python35개 목록과 제출 보존 목록도 갱신한다. 압축본·최종 제출·파일 삭제·실제 절감 실측은 이번 실행에 포함되지 않았다.
'''


def write():
    source=MD.read_text(encoding='utf-8')
    assert TITLE not in source
    source=source.replace('사용한 32개 파일이다.','사용한 35개 파일이다.')
    marker='| [modeling_submission_audit.py]'
    code_rows='''| [m05_evaluate.py](scripts/m05_evaluate.py) | 4.9 개발 설정·경보 경계 고정 후 후반기 학습·추론·평가 | `--stage freeze` → `--stage run` |
| [m05_verify.py](scripts/m05_verify.py) | 4.9 원본 입력·시간 경계·모델 재로딩·전체 지표와 판단 독립 검증 | `main()`; 새 학습 없음 |
| [m05_documents.py](scripts/m05_documents.py) | 4.9 검증 결과의 상세 원고·기록·README 반영과 문서 검증 | `--write` 또는 `--verify` |
'''
    source=source.replace(marker,code_rows+marker,1)
    source=source.replace('[경보 보정](config/m04_surge_calibration_contract.json).','[경보 보정](config/m04_surge_calibration_contract.json), [M05](config/m05_contract.json).')
    source=source.replace('`m04_surge_calibration/`, `observed_timeline/`','`m04_surge_calibration/`, `m05/`, `observed_timeline/`')
    source=source.replace('`m04_surge_risk/`의 파일.','`m04_surge_risk/`, `m05/`의 파일.')
    marker='| 14 | `Modeling/scripts/m02_rerun_document_verify.py`'
    source=source.replace(marker,'| 14 | `Modeling/scripts/m05_evaluate.py --stage freeze` → `--stage run` → `Modeling/scripts/m05_verify.py` | 후반기 조건 고정·학습·추론·독립 검증 |\n| 15 | `Modeling/scripts/m02_rerun_document_verify.py`')
    source=source.replace('`Modeling/scripts/m04_surge_documents.py --verify` | 기존','`Modeling/scripts/m04_surge_documents.py --verify`, `Modeling/scripts/m05_documents.py --verify` | 기존')
    source=source.replace('| 15 | `Modeling/scripts/modeling_submission_audit.py','| 16 | `Modeling/scripts/modeling_submission_audit.py')
    MD.write_text(source.rstrip()+'\n\n'+section().rstrip()+'\n',encoding='utf-8')
    LOG.write_text('# M05 고정 후반기 평가 실행 기록\n\n'+section(),encoding='utf-8')
    paragraph='''M05는1~6월 공통4,176시간으로4개 모델을 한 번씩 학습하고 고정해7~8월 공통1,344시간을 평가했다. 정상B의 전체MAE6.621은 직전12.678보다47.8%작지만 A6.683 대비 개선은0.92%였고 최대시간MAE13.376→13.803은3.19%악화했다. 급등6건의 위험 후보는TP3→4·FP16→14였으나8월TP2→1과 실제오경보율1.046%로 사전 요건을 충족하지 못했다. 기본HGB 성과는 유지되지만 마지막 값·급등 위험의 강한 후반기 효과는 미확인이다. 원본149,040입력·10,936예측·4모델 재로딩 독립검증을 완료했고 후반기 결과로 재튜닝하지 않았다.'''
    for path,target in [(ROOT/'README.md','Modeling/10.05_M05_고정후반기평가_실행기록.md'),(ROOT/'Modeling/README.md','10.05_M05_고정후반기평가_실행기록.md')]:
        text=path.read_text(encoding='utf-8')
        assert target not in text
        position=text.index('\n\n',text.index('# '))
        text=text[:position]+f'\n\n[M05 현재 결과]({target}): {paragraph}'+text[position:]
        if path.parent.name=='Modeling':
            text=text.replace('4.5는 전체32코드 보존','4.5는 전체35코드 보존').replace('사용한 Python 코드 32개','사용한 Python 코드 35개')
            text=text.replace('4.8은 일반 급등 재평가와 별도 위험 출력의 탐색적 후보를 설명한다.','4.8은 일반 급등 재평가와 별도 위험 출력의 탐색적 후보, 4.9는 고정 후반기 성과와 주장 범위의 갱신을 설명한다.')
            text=text.replace('날씨 D는 보류이며 후반기 M05는 아직 실행하지 않았다.','날씨 D는 보류이며 후반기 M05의 최신 결과와 판단은 문서 상단 및 원고4.9를 따른다.')
        else:
            text=text.replace('마지막 값의 개발 개선은 확인했고 M05 후반기 안정성과 최종 필살기 채택은 아직 미확인이다.','마지막 값의 개발 개선은 확인했지만 M05에서는 전체 개선0.92%와 최대시간 악화가 나타나 안정적 기여를 확정하지 못했다. 최신 판단은 문서 상단 및 원고4.9를 따른다.')
        path.write_text(text,encoding='utf-8')


def verify():
    source=MD.read_text(encoding='utf-8')
    assert source.count(TITLE)==1 and source.split(TITLE,1)[1].split('\n## 4.10',1)[0].rstrip()==section().split(TITLE,1)[1].rstrip()
    assert LOG.read_text(encoding='utf-8').split(TITLE,1)[1].rstrip()==section().split(TITLE,1)[1].rstrip()
    for target in re.findall(r'!?\[[^\]]*\]\(([^)]+)\)',source):
        if not target.startswith(('http:','https:','#','app:')):
            assert (MD.parent/target.split('#')[0].strip('<>')).resolve().exists(),target
    result={'status':'passed','checked_at':datetime.now(timezone(timedelta(hours=9))).isoformat(timespec='minutes'),
        'script_sha256':sha(Path(__file__)),'manuscript_sha256':sha(MD),'execution_record_sha256':sha(LOG),
        'comparison_rows':6,'monthly_rows':2,'alarm_rows':6,'no_fitting':True}
    (OUT/'document_verification.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
    print(json.dumps(result),flush=True)


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--write',action='store_true')
    parser.add_argument('--verify',action='store_true')
    args=parser.parse_args()
    assert args.write!=args.verify
    run=json.loads((OUT/'run.json').read_text(encoding='utf-8'))
    independent=json.loads((OUT/'independent_verification.json').read_text(encoding='utf-8'))
    assert independent['status']=='passed' and independent['run_sha256']==sha(OUT/'run.json')
    for name,digest in run['outputs_sha256'].items():
        assert sha(OUT/name)==digest
    if args.write:
        write()
    verify()


if __name__=='__main__':
    main()


코드 등록 완료: `Modeling/scripts/m05_documents.py`

### `Modeling/scripts/m05_evaluate.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Freeze development choices, then evaluate fixed January-June models in July-August.

주요 함수: `save()`, `freeze()`, `construct_frame()`, `alarm_metrics()`, `subsets()`, `assess()`, `run()`, `main()`.

In [62]:
%%submission_source Modeling/scripts/m05_evaluate.py
"""Freeze development choices, then evaluate fixed January-June models in July-August."""
import argparse
import json
import math
import os
os.environ['OMP_NUM_THREADS'] = '1'
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import average_precision_score, roc_auc_score
from m01_prepare import ROOT, SLOTS, feature_columns, read_contract, sha
from m02_compare import metric, state
from m04_rise_compare import now, classifier, gated, read_csv

OUT = ROOT / 'Modeling/tables/m05'
MODELS = ROOT / 'Modeling/models/m05'
CONTRACT = ROOT / 'Modeling/config/m05_contract.json'


def save(frame, name):
    frame.to_csv(OUT / name, index=False, encoding='utf-8-sig')


def freeze():
    assert not CONTRACT.exists() and not (OUT / 'run.json').exists(), 'Preserve completed conditions.'
    OUT.mkdir(parents=True, exist_ok=True)
    parent = read_contract()
    assert sha(ROOT / parent['source']) == parent['source_sha256']
    dependencies = [parent['source'], 'Modeling/config/m01_contract.json', 'Modeling/config/m03_ab_contract.json',
        'Modeling/config/m04_rise_contract.json', 'Modeling/tables/m01/hourly_frame.csv',
        'Modeling/tables/m01/evaluation_diagnostics.csv', 'Modeling/tables/m01/split_counts.csv',
        'Modeling/tables/m04_rise/selected_configurations.json', 'Modeling/tables/m04_rise/feature_frame.csv',
        'Modeling/tables/m04_surge_risk/predictions.csv']
    for folder in ['m01', 'm04_rise', 'm04_surge_risk', 'm04_surge_calibration']:
        directory = ROOT / 'Modeling/tables' / folder
        verification = json.loads((directory / 'independent_verification.json').read_text(encoding='utf-8'))
        assert verification['status'] == 'passed'
        dependencies.append(f'Modeling/tables/{folder}/independent_verification.json')
        if folder != 'm01':
            run = json.loads((directory / 'run.json').read_text(encoding='utf-8'))
            assert run['status'] == 'completed' and verification['run_sha256'] == sha(directory / 'run.json')
            for name, digest in run['outputs_sha256'].items():
                assert sha(directory / name) == digest, (folder, name)
            dependencies.append(f'Modeling/tables/{folder}/run.json')
    configurations = json.loads((ROOT / 'Modeling/tables/m04_rise/selected_configurations.json').read_text(encoding='utf-8'))
    june = next(r for r in configurations if r['split'] == 'dev_jun')
    development = read_csv(ROOT / 'Modeling/tables/m04_surge_risk/predictions.csv', ['timestamp', 'date'])
    calibration = []
    for group in ['B_score', 'G_HGB_score']:
        part = development.loc[development.group.eq(group)].sort_values('timestamp')
        assert len(part) == 2184 and part.timestamp.min() >= pd.Timestamp('2021-04-01') and part.timestamp.max() < pd.Timestamp('2021-07-01')
        negatives = sorted(part.loc[part.increase.lt(93), 'score'].tolist(), reverse=True)
        for cap in [.01, .05]:
            k = math.floor(cap * len(negatives))
            threshold = math.nextafter(negatives[k], math.inf)
            calibration.append({'group': group, 'cap': cap, 'threshold': threshold,
                'hours': len(part), 'events': int(part.increase.ge(93).sum()), 'negatives': len(negatives),
                'budget': k, 'calibration_FP': sum(value >= threshold for value in negatives),
                'calibration_start': str(part.timestamp.min()), 'calibration_end': str(part.timestamp.max())})
    save(pd.DataFrame(calibration), 'alarm_calibration.csv')
    c = {'version': 'm05-v1', 'recorded_at': now(),
        'reason': 'Confirm M03 last-slot contribution and the post-M04 provisional G risk candidate; preserve fixed later split.',
        'source': parent['source'], 'source_sha256': parent['source_sha256'],
        'inputs_sha256': {name: sha(ROOT / name) for name in dependencies},
        'training': {'end': '2021-06-30 23:00:00', 'pool': 'common', 'hours': 4176, 'fits': 'A, B, weighted expert, prior-low HGB classifier each fitted once'},
        'evaluation': {'start': '2021-07-01 00:00:00', 'end': '2021-08-31 23:00:00', 'common_hours': 1344, 'core_hours': 1436,
            'information_time': 'Observe t-1, predict t; update hourly observations but no model fitting or threshold updates in July-August.',
            'core': 'Inference-only A/B coverage extension with the same common-trained models; separate from main comparison.',
            'history': 'Period already observed in EDA; confirmation on a later internal period, not untouched independent test.'},
        'target': 'target_maximum', 'normal_groups': ['lag1', 'lag24', 'lag168', 'A', 'B'],
        'hgb_parameters': json.loads((ROOT / 'Modeling/config/m03_ab_contract.json').read_text(encoding='utf-8'))['hgb_parameters'],
        'dynamic_features': json.loads((ROOT / 'Modeling/config/m04_rise_contract.json').read_text(encoding='utf-8'))['features']['add'],
        'G': {'source_split': 'dev_jun', 'reason': 'Latest chronological development settings, not best later score.',
            'event_weight': june['event_weight'], 'classifier_parameters': june['classifiers']['HGB']['parameters'],
            'gate': june['classifiers']['HGB']['gate'], 'expert_label': 'legacy low->above26',
            'normal_output': 'B; G is an auxiliary risk score, not a replacement point forecast.'},
        'surge_boundary': 93., 'sensitivity_boundary': 61.60000000000002,
        'alarms': {'calibration': 'Existing rolling outer April-June scores; negative-tail order statistic; no training resubstitution.',
            'primary_cap': .01, 'secondary_cap': .05, 'thresholds': calibration,
            'calibration_sha256': sha(OUT / 'alarm_calibration.csv'), 'same_threshold_for_both_later_months': True},
        'normal_confirmation': {'overall_ratio_max': .99, 'peak_ratio_max': .99,
            'peak_under': 'strictly less than A', 'lag1': 'B overall and peak MAE strictly better than lag1',
            'monthly_ratio_max': 1.01, 'profile_reweighted': 'B overall MAE no worse than A',
            'meaning': 'All gates confirm broad last-slot contribution; 1% is numerical near-tie convention, not economic optimum. Failed gates limit claims and do not trigger automatic retuning.'},
        'risk_confirmation': {'rank_ratio_min': 1.05, 'metric': 'event-count-weighted monthly non-interpolated AP',
            'recall': 'Pooled TP >= B and each nonempty month TP >= B', 'actual_false_positive_max': .01,
            'strict_gain': 'Positive TP and either additional TP or fewer FP',
            'missing_events': 'No events => not assessable; small/repeated samples restrict strength of claims.',
            'secondary': 'q90 and 5% are descriptive only, never select candidate'},
        'postprocessing': 'none; no clipping or deletion of hard cases', 'new_optuna_trials': 0,
        'stop': 'Evaluate once, independently verify, report all outcomes and retain failure conditions; no later-driven optimization.'}
    CONTRACT.write_text(json.dumps(c, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps({'status': 'frozen', 'recorded_at': c['recorded_at'], 'thresholds': calibration, 'no_later_scoring': True}), flush=True)


def construct_frame(c):
    f = read_csv(ROOT / 'Modeling/tables/m01/hourly_frame.csv', ['timestamp', 'date'])
    raw = read_csv(ROOT / c['source'])
    raw = raw.loc[raw['날짜'].between(20210101, 20210831) & raw['시간'].between(0, 23)].copy()
    raw['timestamp'] = pd.to_datetime(raw['날짜'].astype(str), format='%Y%m%d') + pd.to_timedelta(raw['시간'], unit='h')
    raw = raw.set_index('timestamp')
    slots = raw[SLOTS].reindex(pd.DatetimeIndex(f.timestamp-pd.Timedelta(hours=1))).to_numpy()
    f['prev_slot_rise'] = slots[:, 3]-slots[:, 0]
    f['prev_slot_range'] = np.max(slots, axis=1)-np.min(slots, axis=1)
    f['prev_slot_slope'] = slots @ np.array([-1.5, -.5, .5, 1.5])/5
    for stem in ['last', 'mean', 'maximum', 'production']:
        f[stem+'_delta'] = f['lag1_'+stem]-f['lag2_'+stem]
    f['prior_state'] = state(f.lag1_maximum, np.floor(f.lag1_mean+.5))
    f['power_state'] = state(f.target_maximum, f.diag_target_rounded_mean)
    f['power_transition'] = f.prior_state+'->'+f.power_state
    f['prior_low'] = f.prior_state.eq('low')
    f['rise_event'] = f.power_transition.eq('low->above26')
    history = dict(zip(f.timestamp, f.power_state))
    lengths, gaps = [], []
    for ts in f.timestamp:
        length, gap = 0, 0
        for lag in range(1, 7):
            previous = history.get(ts-pd.Timedelta(hours=lag))
            if previous is None:
                gap = 1
                break
            if previous != 'low':
                break
            length += 1
        lengths.append(length)
        gaps.append(gap)
    f['low_run_length6'], f['low_history_gap6'] = lengths, gaps
    f['hour'] = f.timestamp.dt.hour
    f['increase'] = f.target_maximum-f.lag1_maximum
    return f


def alarm_metrics(part, threshold, boundary):
    y = part.increase.ge(boundary).to_numpy()
    score = part.score.to_numpy()
    alarms = score >= threshold
    tp, fp = int((alarms & y).sum()), int((alarms & ~y).sum())
    n, events = len(y), int(y.sum())
    return {'hours': n, 'events': events, 'event_dates': part.loc[y, 'date'].nunique(),
        'TP': tp, 'FP': fp, 'FN': events-tp, 'TN': n-events-fp,
        'recall': tp/events if events else None, 'precision': tp/(tp+fp) if tp+fp else 0.,
        'false_positive_fraction': fp/(n-events) if n>events else None,
        'AP': float(average_precision_score(y, score)) if events else None,
        'AUROC': float(roc_auc_score(y, score)) if 0<events<n else None,
        'threshold': threshold, 'prevalence': events/n}


def subsets(g):
    yield 'all', 'all', g, None
    yield 'profile_reweighted', 'all', g, g.profile_weight
    peak = g.loc[g.daily_maximum_weight.gt(0)]
    if len(peak):
        yield 'daily_maximum', 'all', peak, peak.daily_maximum_weight
    for col in ['month', 'power_transition', 'hour', 'train_profile_overlap']:
        for value, part in g.groupby(col):
            yield col, str(value), part, None
    for label, mask in [('surge_q95', g.increase.ge(93)), ('surge_q90', g.increase.ge(61.60000000000002)),
        ('legacy_rise', g.rise_event), ('actual_zero', g.actual.eq(0)), ('prior_zero', g.lag1_maximum.eq(0)),
        ('restart_after_zero', g.lag1_maximum.eq(0)&g.actual.gt(0)), ('sharp_drop', g.increase.le(-93))]:
        if mask.any():
            yield label, 'all', g.loc[mask], None


def assess(p, metrics, risk, c):
    main = metrics.loc[metrics.pool.eq('common')]
    def value(group, period='pooled', condition='all', key='MAE'):
        return float(main.loc[main.group.eq(group)&main.period.eq(period)&main.condition.eq(condition)&main.value.eq('all'), key].iloc[0])
    normal = {
        'overall': value('B') <= c['normal_confirmation']['overall_ratio_max']*value('A'),
        'daily_maximum': value('B', condition='daily_maximum') <= c['normal_confirmation']['peak_ratio_max']*value('A', condition='daily_maximum'),
        'peak_under': value('B', condition='daily_maximum', key='mean_under') < value('A', condition='daily_maximum', key='mean_under'),
        'lag1_overall': value('B') < value('lag1'),
        'lag1_peak': value('B', condition='daily_maximum') < value('lag1', condition='daily_maximum'),
        'monthly': all(value('B', str(m)) <= 1.01*value('A', str(m)) and value('B', str(m), 'daily_maximum') <= 1.01*value('A', str(m), 'daily_maximum') for m in [7, 8]),
        'profile_reweighted': value('B', condition='profile_reweighted') <= value('A', condition='profile_reweighted')}
    table = []
    for group in ['B_score', 'G_HGB_score']:
        part = risk.loc[risk.group.eq(group)&risk.cap.eq(.01)&risk.boundary.eq(93)]
        pooled = part.loc[part.period.eq('pooled')].iloc[0]
        months = part.loc[part.period.ne('pooled')]
        count = int(months.events.sum())
        weighted_ap = float((months.AP.fillna(0)*months.events).sum()/count) if count else None
        table.append({'group': group, 'weighted_monthly_AP': weighted_ap,
            **{key: int(pooled[key]) for key in ['events', 'TP', 'FP', 'FN', 'TN']},
            **{key: None if pd.isna(pooled[key]) else float(pooled[key]) for key in ['recall', 'precision', 'false_positive_fraction']}})
    b, g = table
    if b['events']:
        base_months = risk.loc[risk.group.eq('B_score')&risk.cap.eq(.01)&risk.boundary.eq(93)&risk.period.ne('pooled')].set_index('period')
        candidate_months = risk.loc[risk.group.eq('G_HGB_score')&risk.cap.eq(.01)&risk.boundary.eq(93)&risk.period.ne('pooled')].set_index('period')
        gates = {'rank': g['weighted_monthly_AP'] >= 1.05*b['weighted_monthly_AP'], 'pooled_recall': g['TP']>=b['TP'],
            'monthly_recall': bool((candidate_months.TP>=base_months.TP).all()),
            'false_positive': g['false_positive_fraction']<=.01,
            'strict_gain': g['TP']>0 and (g['TP']>b['TP'] or g['FP']<b['FP'])}
    else:
        gates = {'assessable': False}
    save(pd.DataFrame(table), 'risk_confirmation.csv')
    return {'normal_forecast_development_candidate': 'B', 'normal_confirmation_gates': normal,
        'broad_last_slot_effect_confirmed': all(normal.values()), 'risk_confirmation_gates': gates,
        'G_risk_effect_confirmed': all(gates.values()), 'risk_summary': table,
        'no_automatic_retuning': True, 'later_period_already_observed': True,
        'normal_forecast_not_replaced_by_G': True}


def run():
    assert not (OUT / 'run.json').exists(), 'Preserve completed run.'
    c = json.loads(CONTRACT.read_text(encoding='utf-8'))
    for name, digest in c['inputs_sha256'].items():
        assert sha(ROOT / name) == digest, name
    assert sha(OUT / 'alarm_calibration.csv') == c['alarms']['calibration_sha256']
    begin = time.perf_counter()
    record = {'status': 'running', 'started': now(), 'contract_sha256': sha(CONTRACT),
        'script_sha256': sha(Path(__file__)), 'inputs_sha256': c['inputs_sha256'],
        'runtime': {'executable': sys.executable, 'version': sys.version}, 'new_optuna_trials': 0}
    (OUT / 'run.json').write_text(json.dumps(record, indent=2), encoding='utf-8')
    f = construct_frame(c)
    tr = f.loc[f.eligible_common&f.timestamp.le(c['training']['end'])].copy()
    ev = f.loc[f.eligible_common&f.timestamp.between(c['evaluation']['start'], c['evaluation']['end'])].copy()
    core = f.loc[f.eligible_core&f.timestamp.between(c['evaluation']['start'], c['evaluation']['end'])].copy()
    assert (len(tr), len(ev), len(core)) == (4176, 1344, 1436)
    assert tr.timestamp.max() < ev.timestamp.min()
    features = {name: feature_columns(name) for name in ['A', 'B']}
    features['dynamic'] = features['B']+c['dynamic_features']
    assert all(not col.startswith(('target_', 'diag_')) for cols in features.values() for col in cols)
    previous = read_csv(ROOT / 'Modeling/tables/m04_rise/feature_frame.csv', ['timestamp'])
    assert previous.timestamp.tolist() == tr.timestamp.tolist()
    assert np.allclose(previous[features['dynamic']], tr[features['dynamic']], rtol=0, atol=1e-10)
    save(pd.concat([tr.assign(role='train'), ev.assign(role='evaluation')])[['timestamp', 'role']+features['dynamic']], 'feature_frame.csv')
    estimators, manifest, audits = {}, [], []
    MODELS.mkdir(parents=True, exist_ok=True)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        for name in ['A', 'B', 'expert', 'classifier']:
            cols = features[name] if name in ['A', 'B'] else features['dynamic']
            sample = tr.loc[tr.prior_low] if name == 'classifier' else tr
            model = classifier('HGB', c['G']['classifier_parameters']) if name == 'classifier' else HistGradientBoostingRegressor(**c['hgb_parameters'])
            target = sample.rise_event.astype(int) if name == 'classifier' else sample.target_maximum
            weights = np.where(sample.rise_event, c['G']['event_weight'], 1.) if name == 'expert' else None
            start = time.perf_counter()
            if weights is None:
                model.fit(sample[cols], target)
            else:
                model.fit(sample[cols], target, sample_weight=weights)
            path = MODELS / (name+'.joblib')
            joblib.dump(model, path, compress=3)
            loaded = joblib.load(path)
            before = model.predict_proba(ev[cols]) if name == 'classifier' else model.predict(ev[cols])
            after = loaded.predict_proba(ev[cols]) if name == 'classifier' else loaded.predict(ev[cols])
            assert np.allclose(before, after, rtol=0, atol=1e-10)
            estimators[name] = model
            manifest.append({'name': name, 'file': path.relative_to(ROOT).as_posix(), 'sha256': sha(path), 'reload_checked': True})
            audits.append({'name': name, 'train_hours': len(sample), 'training_start': str(sample.timestamp.min()),
                'training_end': str(sample.timestamp.max()), 'feature_count': len(cols), 'fit_count': 1, 'seconds': time.perf_counter()-start})
        warning_messages = [str(w.message) for w in caught]
    diag = read_csv(ROOT / 'Modeling/tables/m01/evaluation_diagnostics.csv', ['timestamp', 'date'])
    predictions = []
    for pool, part in [('common', ev), ('core', core)]:
        d = diag.loc[diag.split.eq('later_jul_aug')&diag.pool.eq(pool)]
        base = part[['timestamp', 'date', 'month', 'hour', 'power_transition', 'prior_low', 'rise_event', 'lag1_maximum', 'increase', 'diag_target_profile']].merge(
            d[['timestamp', 'train_profile_overlap', 'profile_weight', 'daily_maximum_weight']], on='timestamp', validate='one_to_one')
        assert base.timestamp.tolist() == part.timestamp.tolist()
        outputs = {name: estimators[name].predict(part[features[name]]) for name in ['A', 'B']}
        if pool == 'common':
            outputs.update({lag: part[lag+'_maximum'].to_numpy() for lag in ['lag1', 'lag24', 'lag168']})
            probability = estimators['classifier'].predict_proba(part[features['dynamic']])[:, 1]
            expert = estimators['expert'].predict(part[features['dynamic']])
            outputs['G_aux'], trigger = gated(outputs['B'], expert, probability, part.prior_low, **c['G']['gate'])
            detail = base.copy()
            detail['B'], detail['expert'], detail['probability'], detail['trigger'], detail['G_aux'] = outputs['B'], expert, probability, trigger, outputs['G_aux']
            save(detail, 'G_components.csv')
        for group, values in outputs.items():
            r = base.copy()
            r['pool'], r['group'], r['actual'], r['prediction'] = pool, group, part.target_maximum.to_numpy(), values
            r['signed_error'] = r.prediction-r.actual
            r['absolute_error'], r['under_amount'], r['over_amount'] = r.signed_error.abs(), (-r.signed_error).clip(lower=0), r.signed_error.clip(lower=0)
            predictions.append(r)
    p = pd.concat(predictions, ignore_index=True)
    assert np.isfinite(p.prediction).all()
    save(p, 'predictions.csv')
    save(pd.DataFrame(manifest), 'model_manifest.csv')
    save(pd.DataFrame(audits), 'training_audit.csv')
    metrics, daily, pairs, risk_rows, risk_predictions = [], [], [], [], []
    for (pool, group), g in p.groupby(['pool', 'group']):
        for period, part in [('pooled', g)]+[(str(m), x) for m, x in g.groupby('month')]:
            for condition, value, subset, weights in subsets(part):
                metrics.append({'pool': pool, 'group': group, 'period': period, 'condition': condition, 'value': value, **metric(subset, weights)})
        for date, part in g.groupby('date'):
            daily.append({'pool': pool, 'group': group, 'date': date, **metric(part)})
    for pool in ['common', 'core']:
        a = p.loc[p.pool.eq(pool)&p.group.eq('A')]
        b = p.loc[p.pool.eq(pool)&p.group.eq('B')]
        q = a[['timestamp', 'date', 'prediction', 'absolute_error', 'under_amount', 'over_amount']].merge(
            b[['timestamp', 'prediction', 'absolute_error', 'under_amount', 'over_amount']], on='timestamp', suffixes=('_A', '_B'), validate='one_to_one')
        q['pool'] = pool
        for key in ['absolute_error', 'under_amount', 'over_amount']:
            q['delta_'+key] = q[key+'_B']-q[key+'_A']
        pairs.append(q)
    for name, source_group in [('B_score', 'B'), ('G_HGB_score', 'G_aux')]:
        part = p.loc[p.pool.eq('common')&p.group.eq(source_group)].copy()
        part['group'], part['score'] = name, part.prediction-part.lag1_maximum
        for cap in [.01, .05]:
            threshold = next(r['threshold'] for r in c['alarms']['thresholds'] if r['group'] == name and r['cap'] == cap)
            part['alarm_'+str(cap)] = part.score.ge(threshold)
            for boundary in [93., c['sensitivity_boundary']]:
                for period, subset in [('pooled', part)]+[(str(m), x) for m, x in part.groupby('month')]:
                    risk_rows.append({'group': name, 'cap': cap, 'boundary': boundary, 'period': period, **alarm_metrics(subset, threshold, boundary)})
        part['normal_prediction'] = p.loc[p.pool.eq('common')&p.group.eq('B'), 'prediction'].to_numpy()
        risk_predictions.append(part)
    metric_frame, risk_frame = pd.DataFrame(metrics), pd.DataFrame(risk_rows)
    save(metric_frame, 'metrics.csv')
    save(pd.DataFrame(daily), 'daily_errors.csv')
    save(pd.concat(pairs), 'paired_predictions.csv')
    save(pd.concat(risk_predictions), 'risk_predictions.csv')
    save(risk_frame, 'risk_metrics.csv')
    save(p.loc[p.pool.eq('common')].sort_values('absolute_error', ascending=False).groupby('group').head(20), 'largest_errors.csv')
    save(p.loc[p.pool.eq('common')&p.group.eq('B')&p.increase.ge(93)], 'surge_cases.csv')
    save(p.loc[p.group.isin(['A', 'B'])&(p.actual.eq(0)|p.lag1_maximum.eq(0))], 'zero_cases.csv')
    conclusion = assess(p, metric_frame, risk_frame, c)
    (OUT / 'conclusion.json').write_text(json.dumps(conclusion, ensure_ascii=False, indent=2), encoding='utf-8')
    record.update({'status': 'completed', 'finished': now(), 'seconds': time.perf_counter()-begin, 'fits': 4,
        'prediction_rows': len(p), 'metrics_rows': len(metric_frame), 'features': features,
        'warnings': warning_messages, 'helper_sha256': {name: sha(ROOT / 'Modeling/scripts' / name) for name in ['m01_prepare.py', 'm02_compare.py', 'm04_rise_compare.py']},
        'outputs_sha256': {path.name: sha(path) for path in OUT.iterdir() if path.name != 'run.json'}})
    (OUT / 'run.json').write_text(json.dumps(record, indent=2), encoding='utf-8')
    print(metric_frame.loc[metric_frame.pool.eq('common')&metric_frame.period.eq('pooled')&metric_frame.condition.isin(['all', 'daily_maximum']), ['group','condition','MAE','mean_under','mean_over']].to_string(index=False), flush=True)
    print(json.dumps(conclusion), flush=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--stage', choices=['freeze', 'run'], required=True)
    args = parser.parse_args()
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    freeze() if args.stage == 'freeze' else run()


if __name__ == '__main__':
    main()


코드 등록 완료: `Modeling/scripts/m05_evaluate.py`

### `Modeling/scripts/m05_verify.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Independent raw input, fixed split, saved model, metrics and decision verification.

주요 함수: `state()`, `subset()`, `main()`.

In [63]:
%%submission_source Modeling/scripts/m05_verify.py
"""Independent raw input, fixed split, saved model, metrics and decision verification."""
import os
os.environ['OMP_NUM_THREADS'] = '1'
import json
import math
import sys
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from m03_verify import rows, sha, close, verify_metric
from m04_rise_verify import ap

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / 'Modeling/tables/m05'


def state(maximum, rounded):
    return 'zero' if maximum == 0 else 'below20' if rounded < 20 else 'low' if rounded <= 26 else 'above26'


def subset(records, condition, value):
    if condition in ['all', 'profile_reweighted']:
        return records
    if condition == 'daily_maximum':
        return [r for r in records if float(r['daily_maximum_weight'])>0]
    if condition in ['month', 'power_transition', 'hour', 'train_profile_overlap']:
        return [r for r in records if r[condition] == value]
    rules = {
        'surge_q95': lambda r: float(r['increase'])>=93,
        'surge_q90': lambda r: float(r['increase'])>=61.60000000000002,
        'legacy_rise': lambda r: r['rise_event']=='True',
        'actual_zero': lambda r: float(r['actual'])==0,
        'prior_zero': lambda r: float(r['lag1_maximum'])==0,
        'restart_after_zero': lambda r: float(r['lag1_maximum'])==0 and float(r['actual'])>0,
        'sharp_drop': lambda r: float(r['increase'])<=-93,
    }
    return [r for r in records if rules[condition](r)]


def main():
    assert sys.version_info[:2] == (3,13) and sys._is_gil_enabled()
    run = json.loads((OUT / 'run.json').read_text(encoding='utf-8'))
    cpath = ROOT / 'Modeling/config/m05_contract.json'
    c = json.loads(cpath.read_text(encoding='utf-8'))
    assert run['status']=='completed' and run['fits']==4 and run['new_optuna_trials']==0
    assert sha(cpath)==run['contract_sha256'] and sha(ROOT / 'Modeling/scripts/m05_evaluate.py')==run['script_sha256']
    for name, digest in run['outputs_sha256'].items():
        assert sha(OUT / name)==digest, name
    for name, digest in run['inputs_sha256'].items():
        assert sha(ROOT / name)==digest, name
    for name, digest in run['helper_sha256'].items():
        assert sha(ROOT / 'Modeling/scripts' / name)==digest, name
    raw = {}
    for r in rows(ROOT / c['source']):
        date, hour = int(r['날짜']), float(r['시간'])
        if 20210101<=date<=20210831 and hour.is_integer() and 0<=hour<=23:
            ts = datetime.strptime(str(date),'%Y%m%d')+timedelta(hours=int(hour))
            slots = [float(r[k]) for k in ['15분','30분','45분','60분']]
            assert ts not in raw
            raw[ts] = {'slots': slots, 'maximum': max(slots), 'mean': math.fsum(slots)/4,
                'last': slots[-1], 'production': float(r['생산량']), 'rounded': float(r['평균'])}
    assert len(raw)==5784
    dates = defaultdict(list)
    for ts in sorted(raw):
        dates[ts.date()].extend(raw[ts]['slots'])
    import hashlib
    profiles = {day: hashlib.sha256(','.join(format(x,'.17g') for x in slots).encode()).hexdigest() for day,slots in dates.items()}
    eligible = {pool: [ts for ts in sorted(raw) if datetime(2021,7,1)<=ts<datetime(2021,9,1)
        and all(ts-timedelta(hours=lag) in raw for lag in lags)] for pool,lags in [('common',[1,2,24,168]),('core',[1,2])]}
    training = [ts for ts in sorted(raw) if ts<datetime(2021,7,1) and all(ts-timedelta(hours=lag) in raw for lag in [1,2,24,168])]
    assert (len(training),len(eligible['common']),len(eligible['core']))==(4176,1344,1436)
    expected_inputs = {}
    for r in rows(OUT / 'feature_frame.csv'):
        ts = datetime.fromisoformat(r['timestamp'])
        assert ts in training if r['role']=='train' else ts in eligible['common']
        own, prev, prev2 = raw[ts], raw[ts-timedelta(hours=1)], raw[ts-timedelta(hours=2)]
        vals = {'hour_sin': math.sin(2*math.pi*ts.hour/24), 'hour_cos': math.cos(2*math.pi*ts.hour/24),
            'month': ts.month, 'weekend': int(ts.weekday()>=5), 'lag1_production_zero': int(prev['production']==0),
            'prev_slot_rise': prev['slots'][-1]-prev['slots'][0], 'prev_slot_range': max(prev['slots'])-min(prev['slots']),
            'prev_slot_slope': math.fsum(x*w for x,w in zip(prev['slots'],[-1.5,-.5,.5,1.5]))/5}
        for day in range(7):
            vals[f'dow_{day}']=int(ts.weekday()==day)
        for stem in ['last','mean','maximum','production']:
            vals['lag1_'+stem]=prev[stem]
            vals['lag2_'+stem]=prev2[stem]
            vals[stem+'_delta']=prev[stem]-prev2[stem]
        length,gap=0,0
        for lag in range(1,7):
            old=raw.get(ts-timedelta(hours=lag))
            if old is None:
                gap=1
                break
            if state(old['maximum'],old['rounded'])!='low':
                break
            length+=1
        vals['low_run_length6'],vals['low_history_gap6']=length,gap
        for col in run['features']['dynamic']:
            close(r[col],vals[col])
        expected_inputs[ts]=vals
    assert len(expected_inputs)==5520
    predictions=rows(OUT / 'predictions.csv')
    groups=defaultdict(list)
    keyed={}
    train_profiles={profiles[ts.date()] for ts in training}
    for r in predictions:
        ts=datetime.fromisoformat(r['timestamp'])
        pool,group=r['pool'],r['group']
        assert ts in eligible[pool]
        own,prev=raw[ts],raw[ts-timedelta(hours=1)]
        close(r['actual'],own['maximum'])
        close(r['lag1_maximum'],prev['maximum'])
        close(r['increase'],own['maximum']-prev['maximum'])
        transition=state(prev['maximum'],math.floor(prev['mean']+.5))+'->'+state(own['maximum'],own['rounded'])
        assert r['power_transition']==transition
        assert (r['prior_low']=='True')==transition.startswith('low->')
        assert (r['rise_event']=='True')==(transition=='low->above26')
        assert r['diag_target_profile']==profiles[ts.date()]
        assert (r['train_profile_overlap']=='True')==(profiles[ts.date()] in train_profiles)
        pool_dates={x.date() for x in eligible[pool]}
        count=sum(profiles[day]==profiles[ts.date()] for day in pool_dates)
        close(r['profile_weight'],1/count)
        day_ts=[x for x in eligible[pool] if x.date()==ts.date()]
        maximum=max(raw[x]['maximum'] for x in day_ts)
        ties=[x for x in day_ts if raw[x]['maximum']==maximum]
        close(r['daily_maximum_weight'],1/len(ties) if len(day_ts)==24 and ts in ties else 0)
        error=float(r['prediction'])-own['maximum']
        for col,value in [('signed_error',error),('absolute_error',abs(error)),('under_amount',max(-error,0)),('over_amount',max(error,0))]:
            close(r[col],value)
        if group.startswith('lag'):
            close(r['prediction'],raw[ts-timedelta(hours=int(group[3:]))]['maximum'])
        key=(pool,group,ts)
        assert key not in keyed
        keyed[key]=r
        groups[pool,group].append(r)
    assert len(predictions)==6*1344+2*1436
    for (pool,group),part in groups.items():
        assert [datetime.fromisoformat(r['timestamp']) for r in part]==eligible[pool]
    for r in rows(OUT / 'metrics.csv'):
        part=groups[r['pool'],r['group']]
        if r['period']!='pooled':
            part=[x for x in part if x['month']==r['period']]
        chosen=subset(part,r['condition'],r['value'])
        verify_metric(r,chosen,r['condition'] if r['condition'] in ['daily_maximum','profile_reweighted'] else 'all')
    for r in rows(OUT / 'daily_errors.csv'):
        verify_metric(r,[x for x in groups[r['pool'],r['group']] if x['date']==r['date']])
    for r in rows(OUT / 'paired_predictions.csv'):
        ts=datetime.fromisoformat(r['timestamp'])
        a,b=keyed[r['pool'],'A',ts],keyed[r['pool'],'B',ts]
        for col in ['prediction','absolute_error','under_amount','over_amount']:
            close(r[col+'_A'],a[col]);close(r[col+'_B'],b[col])
            if col!='prediction':
                close(r['delta_'+col],float(b[col])-float(a[col]))
    calibration_source=rows(ROOT / 'Modeling/tables/m04_surge_risk/predictions.csv')
    for r in rows(OUT / 'alarm_calibration.csv'):
        past=[x for x in calibration_source if x['group']==r['group']]
        assert len(past)==2184 and max(x['timestamp'] for x in past)<'2021-07-01'
        negative=sorted([float(x['score']) for x in past if float(x['increase'])<93],reverse=True)
        budget=math.floor(float(r['cap'])*len(negative))
        close(r['threshold'],math.nextafter(negative[budget],math.inf))
        close(r['budget'],budget);close(r['calibration_FP'],sum(x>=float(r['threshold']) for x in negative))
    risk=rows(OUT / 'risk_predictions.csv')
    for r in risk:
        ts=datetime.fromisoformat(r['timestamp'])
        source=keyed['common','B' if r['group']=='B_score' else 'G_aux',ts]
        close(r['score'],float(source['prediction'])-float(source['lag1_maximum']))
        close(r['normal_prediction'],keyed['common','B',ts]['prediction'])
        for cap in [.01,.05]:
            threshold=next(x['threshold'] for x in c['alarms']['thresholds'] if x['group']==r['group'] and x['cap']==cap)
            assert (r['alarm_'+str(cap)]=='True')==(float(r['score'])>=threshold)
    for r in rows(OUT / 'risk_metrics.csv'):
        part=[x for x in risk if x['group']==r['group'] and (r['period']=='pooled' or x['month']==r['period'])]
        boundary,threshold=float(r['boundary']),float(r['threshold'])
        y=[float(x['increase'])>=boundary for x in part]
        scores=[float(x['score']) for x in part]
        alarms=[x>=threshold for x in scores]
        events=sum(y);tp=sum(a and b for a,b in zip(alarms,y));fp=sum(a and not b for a,b in zip(alarms,y))
        expected={'hours':len(part),'events':events,'TP':tp,'FP':fp,'FN':events-tp,'TN':len(part)-events-fp,
            'precision':tp/(tp+fp) if tp+fp else 0,'false_positive_fraction':fp/(len(part)-events),
            'prevalence':events/len(part),'event_dates':len({x['date'] for x in part if float(x['increase'])>=boundary})}
        if events:
            expected.update(recall=tp/events,AP=ap(y,scores))
        else:
            assert r['AP']==r['recall']==''
        if 0<events<len(y):
            positive=[score for label,score in zip(y,scores) if label]
            negative=[score for label,score in zip(y,scores) if not label]
            expected['AUROC']=sum((a>b)+.5*(a==b) for a in positive for b in negative)/(len(positive)*len(negative))
        for col,value in expected.items():
            close(r[col],value)
    models={}
    m1=pd.read_csv(ROOT / 'Modeling/tables/m01/hourly_frame.csv',encoding='utf-8-sig',float_precision='round_trip',parse_dates=['timestamp']).set_index('timestamp')
    features=pd.read_csv(OUT / 'feature_frame.csv',encoding='utf-8-sig',float_precision='round_trip',parse_dates=['timestamp']).set_index('timestamp')
    for r in rows(OUT / 'model_manifest.csv'):
        path=ROOT / r['file']
        assert sha(path)==r['sha256']
        model=joblib.load(path);name=r['name'];models[name]=model
        cols=run['features'][name] if name in ['A','B'] else run['features']['dynamic']
        assert list(model.feature_names_in_)==cols
        parameters=c['hgb_parameters'] if name!='classifier' else {**c['G']['classifier_parameters'],'max_iter':150,'learning_rate':.05,'early_stopping':False,'random_state':42}
        assert all(model.get_params()[key]==value for key,value in parameters.items())
        if name in ['A','B']:
            for pool in ['common','core']:
                actual=model.predict(m1.loc[eligible[pool],cols])
                assert np.allclose(actual,[float(x['prediction']) for x in groups[pool,name]],rtol=0,atol=1e-10)
    components=rows(OUT / 'G_components.csv')
    expert=models['expert'].predict(features.loc[eligible['common'],run['features']['dynamic']])
    probabilities=models['classifier'].predict_proba(features.loc[eligible['common'],run['features']['dynamic']])[:,1]
    for r,w,prob in zip(components,expert,probabilities):
        ts=datetime.fromisoformat(r['timestamp']);b=float(keyed['common','B',ts]['prediction'])
        trigger=r['prior_low']=='True' and prob>=c['G']['gate']['threshold']
        g=b+(c['G']['gate']['blend']*(w-b) if trigger else 0)
        close(r['B'],b);close(r['expert'],w);close(r['probability'],prob);close(r['G_aux'],g)
        assert (r['trigger']=='True')==trigger
        close(keyed['common','G_aux',ts]['prediction'],g)
    audits=rows(OUT / 'training_audit.csv')
    assert len(audits)==4 and sum(int(x['fit_count']) for x in audits)==4
    for r in audits:
        sample=[ts for ts in training if state(raw[ts-timedelta(hours=1)]['maximum'],math.floor(raw[ts-timedelta(hours=1)]['mean']+.5))=='low'] if r['name']=='classifier' else training
        assert r['training_start']==str(min(sample)) and r['training_end']==str(max(sample))
        assert max(sample)<datetime(2021,7,1)
        close(r['train_hours'],len(sample))
    # Rebuild the decision flags without importing the execution evaluator.
    metric_records=rows(OUT / 'metrics.csv')
    def value(group,period='pooled',condition='all',key='MAE'):
        return float(next(r[key] for r in metric_records if r['pool']=='common' and r['group']==group and r['period']==period and r['condition']==condition and r['value']=='all'))
    normal={'overall':value('B')<=.99*value('A'),'daily_maximum':value('B',condition='daily_maximum')<=.99*value('A',condition='daily_maximum'),
        'peak_under':value('B',condition='daily_maximum',key='mean_under')<value('A',condition='daily_maximum',key='mean_under'),
        'lag1_overall':value('B')<value('lag1'),'lag1_peak':value('B',condition='daily_maximum')<value('lag1',condition='daily_maximum'),
        'monthly':all(value('B',str(m))<=1.01*value('A',str(m)) and value('B',str(m),'daily_maximum')<=1.01*value('A',str(m),'daily_maximum') for m in [7,8]),
        'profile_reweighted':value('B',condition='profile_reweighted')<=value('A',condition='profile_reweighted')}
    rmetrics=rows(OUT / 'risk_metrics.csv')
    weighted={}
    for group in ['B_score','G_HGB_score']:
        months=[r for r in rmetrics if r['group']==group and r['cap']=='0.01' and r['boundary']=='93.0' and r['period']!='pooled']
        events=sum(int(r['events']) for r in months)
        weighted[group]=sum(float(r['AP'] or 0)*int(r['events']) for r in months)/events if events else None
    pooled={r['group']:r for r in rmetrics if r['cap']=='0.01' and r['boundary']=='93.0' and r['period']=='pooled'}
    b,g=pooled['B_score'],pooled['G_HGB_score']
    def month_tp(group,m):
        return int(next(r['TP'] for r in rmetrics if r['group']==group and r['cap']=='0.01' and r['boundary']=='93.0' and r['period']==str(m)))
    gates={'rank':weighted['G_HGB_score']>=1.05*weighted['B_score'],'pooled_recall':int(g['TP'])>=int(b['TP']),
        'monthly_recall':all(month_tp('G_HGB_score',m)>=month_tp('B_score',m) for m in [7,8]),
        'false_positive':float(g['false_positive_fraction'])<=.01,
        'strict_gain':int(g['TP'])>0 and (int(g['TP'])>int(b['TP']) or int(g['FP'])<int(b['FP']))} if int(b['events']) else {'assessable':False}
    conclusion=json.loads((OUT / 'conclusion.json').read_text(encoding='utf-8'))
    assert conclusion['normal_confirmation_gates']==normal and conclusion['broad_last_slot_effect_confirmed']==all(normal.values())
    assert conclusion['risk_confirmation_gates']==gates and conclusion['G_risk_effect_confirmed']==all(gates.values())
    for r in rows(OUT / 'risk_confirmation.csv'):
        if weighted[r['group']] is not None:
            close(r['weighted_monthly_AP'],weighted[r['group']])
    events=[]
    for r in risk:
        if r['group']=='B_score' and float(r['increase'])>=93:
            other=next(x for x in risk if x['timestamp']==r['timestamp'] and x['group']=='G_HGB_score')
            events.append({'timestamp':r['timestamp'],'actual':float(r['actual']),'increase':float(r['increase']),
                'B_score':float(r['score']),'G_score':float(other['score']),
                'B_alarm':r['alarm_0.01']=='True','G_alarm':other['alarm_0.01']=='True',
                'prior_low':r['prior_low']=='True'})
    diagnostic={'events':events,'overall_reduction_percent':100*(1-value('B')/value('A')),
        'peak_MAE_change_percent':100*(value('B',condition='daily_maximum')/value('A',condition='daily_maximum')-1),
        'baseline_reduction_percent':100*(1-value('B')/value('lag1')),
        'no_test_based_change':True}
    (OUT / 'confirmed_diagnostics.json').write_text(json.dumps(diagnostic,indent=2),encoding='utf-8')
    result={'status':'passed','checked_at':datetime.now(timezone(timedelta(hours=9))).isoformat(timespec='minutes'),
        'run_sha256':sha(OUT / 'run.json'),'script_sha256':sha(Path(__file__)),
        'raw_feature_cells_checked':5520*27,'prediction_rows_checked':len(predictions),'models_reloaded':4,
        'metrics_rows_checked':len(metric_records),'alarm_rows_checked':len(rmetrics),
        'diagnostics_sha256':sha(OUT / 'confirmed_diagnostics.json'),'confirmation_flags_verified':True,'no_fitting':True}
    (OUT / 'independent_verification.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
    print(json.dumps(result),flush=True)


if __name__=='__main__':
    main()


코드 등록 완료: `Modeling/scripts/m05_verify.py`

### `Modeling/scripts/modeling_close_analysis.py`

원고 연결: Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_원고.md.

코드 설명: Close modeling: exact Logistic contributions and raw-to-result reproduction.

No new candidate selection or hyperparameter tuning. Keeps frozen results intact.

주요 함수: `readj()`, `freeze()`, `run()`.

In [64]:
%%submission_source Modeling/scripts/modeling_close_analysis.py
"""Close modeling: exact Logistic contributions and raw-to-result reproduction.

No new candidate selection or hyperparameter tuning. Keeps frozen results intact.
"""
import os
os.environ.setdefault('OMP_NUM_THREADS', '4')
import json
import io
import sys
from datetime import datetime, timezone, timedelta
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from m01_prepare import load_frame
from regime_followup import extended
from regime_forecast import model, threshold, sha, save_json

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / 'Modeling/tables/modeling_close'
AGE = ROOT / 'Modeling/tables/regime_age_ablation'


def readj(p):
    return json.loads(Path(p).read_text(encoding='utf-8'))


def freeze():
    OUT.mkdir(parents=True, exist_ok=True)
    assert not (OUT / 'contract.json').exists()
    names = ['Modeling/tables/regime_age_ablation/contract.json', 'Modeling/tables/regime_age_ablation/run.json',
             'Modeling/tables/regime_age_ablation/independent_verification.json',
             'Modeling/tables/regime_followup/contract.json', 'Modeling/tables/regime_followup/run.json',
             'Modeling/scripts/m01_prepare.py', 'Modeling/scripts/regime_followup.py',
             'Modeling/scripts/regime_diagnose.py', 'Modeling/scripts/regime_forecast.py',
             'data/origin/okm_augumented_2021.csv']
    c = {'created_at': datetime.now(timezone(timedelta(hours=9))).isoformat(timespec='minutes'),
         'script_sha256': sha(__file__), 'inputs_sha256': {n: sha(ROOT / n) for n in names},
         'question': 'Calendar vs prior observed state contribution to fixed no-age Logistic; observed explanation only, no causal claims or new model choice.',
         'method': 'Exact standardized-feature times coefficient additive logit decomposition, grouped calendar/power/production/state; intercept retained. Not SHAP or probability percentage.',
         'case_rule': 'Among true detected onsets with positive absolute error gain choose middle sorted gain (tie timestamp); among missed onsets choose middle sorted actual maximum (tie timestamp). Both require complete t-4..t+5 common forecast windows. State selection counts explicitly.',
         'reproduction': 'Rebuild Jan-Aug exact-time features and causal labels from source; recreate train-only availability flags; refit April/May/June classifiers for OOF threshold, fixed June classifier and B0/B1; match stored predictions and gates. No retuning and no claim of clean environment.',
         'stop': 'Freeze current method as reporting candidate with limited exploratory scope; close model search after explanation, figures, deployment proposal and reproduction checks.',
         'root_readme_sha256': sha(ROOT / 'README.md')}
    save_json(OUT / 'contract.json', c)
    print('Closing analysis contract frozen.', flush=True)


def run():
    c = readj(OUT / 'contract.json'); assert not (OUT / 'run.json').exists()
    assert sha(__file__) == c['script_sha256']
    for p, h in c['inputs_sha256'].items():
        assert sha(ROOT / p) == h
    ac = readj(AGE / 'contract.json'); fc = readj(ROOT / 'Modeling/tables/regime_followup/contract.json')
    sc = readj(ROOT / 'Modeling/tables/regime_forecast/contract.json')
    ar = readj(AGE / 'run.json')
    for n, h in ar['outputs_sha256'].items():
        assert sha(AGE / n) == h
    _, _, _, m = load_frame()
    # Reproduce the original feature artifact's CSV serialization boundary.
    m = pd.read_csv(io.StringIO(m.to_csv(index=False)), parse_dates=['timestamp'])
    z, events = extended(fc)
    ct = z.loc[z.eligible_onset & z.persistence_known & (z.label_confirmed_at < '2021-07-01')].copy()
    assert len(ct) == 4336
    features = ac['features']
    oo = []
    for month in [4, 5, 6]:
        boundary = pd.Timestamp(2021, month, 1)
        tr = ct.loc[ct.label_confirmed_at < boundary]
        te = ct.loc[ct.month == month]
        fit = model('logistic', sc); fit.fit(tr[features], tr.sustained_onset.eq(1).astype(int))
        pp = te[['timestamp','label_confirmed_at','sustained_onset']].copy()
        pp['score'] = fit.predict_proba(te[features])[:,1]; pp['event'] = te.sustained_onset.eq(1).astype(int)
        oo.append(pp)
    oof = pd.concat(oo)
    boundary, _, _, _ = threshold(oof.event.to_numpy(int), oof.score.to_numpy(float))
    close_cal = pd.read_csv(AGE / 'calibration.csv', encoding='utf-8-sig')
    assert np.isclose(boundary, float(close_cal.loc[close_cal.period == 'Jul-Aug', 'threshold'].iloc[0]), atol=1e-10)
    clf = model('logistic', sc); clf.fit(ct[features], ct.sustained_onset.eq(1).astype(int))
    late = z.loc[z.eligible_onset & z.month.ge(7)].copy()
    score = clf.predict_proba(late[features])[:,1]
    saved = pd.read_csv(AGE / 'followup_classification.csv', encoding='utf-8-sig', parse_dates=['timestamp'])
    saved = saved.loc[saved.method == 'noage'].set_index('timestamp').loc[late.timestamp]
    np.testing.assert_allclose(score, saved.score, rtol=1e-9, atol=1e-8)
    np.testing.assert_array_equal(score >= boundary, saved.alarm)
    base = fc['regression_features']['B0']; expanded = fc['regression_features']['B1']
    extras = [x for x in expanded if x not in base and not x.endswith('classifier_available')]
    zh = pd.read_csv(io.StringIO(z.loc[z.month.le(6)].to_csv(index=False)), parse_dates=['timestamp','label_confirmed_at'])
    zz = pd.concat([zh,z.loc[z.month.ge(7)]], ignore_index=True)
    common = zz.loc[zz.eligible_onset, ['timestamp','label_confirmed_at','month','profile','sustained_onset','persistence_known'] + extras]
    common = common.drop(columns='month').merge(m.loc[m.eligible_common, ['timestamp','target_maximum']+base], on='timestamp', validate='one_to_one')
    # Availability at the origin uses mature earlier labels, not target values.
    for direction, sign in [('up',1),('down',-1)]:
        availability = {}
        for month in range(2,9):
            boundary_month = pd.Timestamp(2021, min(month,7), 1)
            tr = ct.loc[ct.label_confirmed_at < boundary_month]
            positive = tr.loc[tr.sustained_onset == sign]
            availability[month] = int(len(positive) >= 8 and positive.date.nunique() >= 5)
        common[f'{direction}_classifier_available'] = common.month.map(availability)
    rt = common.loc[common.month.between(2,6) & common.persistence_known & (common.label_confirmed_at < '2021-07-01')]
    rt = pd.read_csv(io.StringIO(rt.to_csv(index=False)), parse_dates=['timestamp','label_confirmed_at'])
    test = common.loc[common.month.ge(7)].copy()
    assert len(rt) == 3598 and len(test) == 1344
    parent_train = pd.read_csv(ROOT / 'Modeling/tables/regime_integration/meta_features.csv', encoding='utf-8-sig', parse_dates=['timestamp']).set_index('timestamp').loc[rt.timestamp]
    np.testing.assert_allclose(rt[expanded], parent_train[expanded], atol=1e-9)
    differences = {n:float(np.max(np.abs(rt[n].to_numpy()-parent_train[n].to_numpy()))) for n in expanded if not np.array_equal(rt[n].to_numpy(),parent_train[n].to_numpy())}
    print(json.dumps({'remaining_training_feature_differences':differences}),flush=True)
    predictions = {}
    for variant, cols in [('B0',base),('B1_all',expanded)]:
        reg = HistGradientBoostingRegressor(**fc['hgb_parameters']); reg.fit(rt[cols], rt.target_maximum)
        predictions[variant] = reg.predict(test[cols])
    scores_index = pd.Series(score, index=late.timestamp)
    alarm = scores_index.loc[test.timestamp].to_numpy() >= boundary
    predictions['G1_noage'] = np.where(alarm, predictions['B1_all'], predictions['B0'])
    parent = pd.read_csv(AGE / 'followup_predictions.csv', encoding='utf-8-sig', parse_dates=['timestamp'])
    output = []
    for name, pred in predictions.items():
        old = parent.loc[parent.variant == name].set_index('timestamp').loc[test.timestamp]
        np.testing.assert_allclose(old.actual, test.target_maximum, atol=1e-9)
        np.testing.assert_allclose(old.prediction, pred, atol=1e-8, rtol=1e-9)
        part = test[['timestamp','sustained_onset']].copy()
        part['variant'] = name; part['actual'] = test.target_maximum; part['prediction'] = pred
        output.append(part)
    replay = pd.concat(output)
    # Direct linear decomposition in log-odds, checked against both decision and probability.
    scaler, linear = clf[0], clf[1]
    contrib = scaler.transform(late[features]) * linear.coef_[0]
    logit = float(linear.intercept_[0]) + contrib.sum(axis=1)
    np.testing.assert_allclose(logit, clf.decision_function(late[features]), atol=1e-10)
    np.testing.assert_allclose(1/(1+np.exp(-logit)), score, atol=1e-12)
    groups = {'calendar':[n for n in features if n in sc['feature_names'][13:]],
              'production':['prior_production','prior_production_delta'],
              'state':['prior_state_left_censored','prior_state_direction']}
    groups['power'] = [n for n in features if n not in sum(groups.values(), [])]
    assert set(sum(groups.values(), [])) == set(features) and sum(map(len, groups.values())) == len(features)
    explained = late[['timestamp','month','profile','prior_mean','prior_production','sustained_onset','persistence_known']].copy()
    explained['score'] = score; explained['alarm'] = (score >= boundary).astype(int)
    explained['intercept'] = float(linear.intercept_[0]); explained['logit'] = logit
    explained['common'] = explained.timestamp.isin(test.timestamp)
    for group, names in groups.items():
        explained[group+'_contribution'] = contrib[:,[features.index(n) for n in names]].sum(axis=1)
    coefficients = pd.DataFrame({'feature': features, 'standardized_coefficient': linear.coef_[0],
                                 'training_mean':scaler.mean_, 'training_scale':scaler.scale_,
                                 'mean_abs_late_contribution':np.abs(contrib).mean(axis=0)})
    e = explained.loc[explained.common & explained.persistence_known]
    masks = {'detected_up':e.sustained_onset.eq(1)&e.alarm.eq(1),
             'missed_up':e.sustained_onset.eq(1)&e.alarm.eq(0),
             'Monday08_non_event':e.timestamp.dt.dayofweek.eq(0)&e.timestamp.dt.hour.eq(8)&e.sustained_onset.ne(1)}
    summary=[]
    for name, mask in masks.items():
        gg=e.loc[mask]
        summary.append({'condition':name,'hours':len(gg),'mean_score':float(gg.score.mean()),
                        **{n:float(gg[n].mean()) for n in ['intercept','logit']+[k+'_contribution' for k in groups]}})
    events_table = e.loc[e.sustained_onset.eq(1)].copy()
    b0=parent.loc[parent.variant=='B0'].set_index('timestamp'); b1=parent.loc[parent.variant=='G1_noage'].set_index('timestamp')
    events_table['actual_maximum'] = b0.loc[events_table.timestamp,'actual'].to_numpy()
    events_table['base_prediction'] = b0.loc[events_table.timestamp,'prediction'].to_numpy()
    events_table['supplement_prediction'] = b1.loc[events_table.timestamp,'prediction'].to_numpy()
    events_table['absolute_error_gain'] = abs(events_table.actual_maximum-events_table.base_prediction)-abs(events_table.actual_maximum-events_table.supplement_prediction)
    cases=[]; selection=[]
    for category, pool, sortcol in [('detected_improved',events_table.loc[events_table.alarm.eq(1)&events_table.absolute_error_gain.gt(0)],'absolute_error_gain'),
                                    ('missed',events_table.loc[events_table.alarm.eq(0)],'actual_maximum')]:
        eligible=pool.loc[pool.timestamp.apply(lambda t: pd.date_range(t-pd.Timedelta(hours=4), t+pd.Timedelta(hours=5), freq='h').isin(b0.index).all())].sort_values([sortcol,'timestamp'])
        assert len(eligible)
        chosen=eligible.iloc[len(eligible)//2]; t=chosen.timestamp
        times=pd.date_range(t-pd.Timedelta(hours=4),t+pd.Timedelta(hours=5),freq='h')
        window=parent.loc[parent.timestamp.isin(times)&parent.variant.isin(['B0','G1_noage'])].copy()
        window['case']=category;window['onset_time']=t
        window['observed_mean']=z.set_index('timestamp').loc[window.timestamp,'level'].to_numpy()
        window['alarm']=explained.set_index('timestamp').loc[window.timestamp,'alarm'].to_numpy()
        cases.append(window)
        selection.append({'case':category,'timestamp':str(t),'candidate_events':len(pool),'complete_windows':len(eligible),'rank_zero_based':len(eligible)//2,'sort_column':sortcol,'actual':float(chosen.actual_maximum),'gain':float(chosen.absolute_error_gain)})
    outputs={'reproduced_predictions.csv':replay,'logit_contributions.csv':explained,'coefficient_table.csv':coefficients,
             'contribution_summary.csv':pd.DataFrame(summary),'event_explanations.csv':events_table,'case_windows.csv':pd.concat(cases),
             'case_selection.csv':pd.DataFrame(selection)}
    for name, frame in outputs.items():
        frame.to_csv(OUT/name,index=False,encoding='utf-8-sig')
    result={'status':'passed','script_sha256':sha(__file__),'contract_sha256':sha(OUT/'contract.json'),
            'inputs_sha256':c['inputs_sha256'],'outputs_sha256':{n:sha(OUT/n) for n in outputs},
            'raw_valid_hours':len(m),'class_train':len(ct),'regression_train':len(rt),'classification_predictions':len(late),
            'regression_predictions_checked':len(replay),'refits':6,'new_candidates':0,
            'oof_threshold':float(boundary),'exact_logit_reconstruction':True,'clean_environment':False,
            'runtime':sys.version,'scope':'Current fixed-method raw-to-features/training/OOF-threshold/inference replay; historical Optuna search not rerun.'}
    assert sha(ROOT/'README.md')==c['root_readme_sha256']
    save_json(OUT/'run.json',result)
    print(json.dumps({'reproduction':result,'contributions':summary,'cases':selection}),flush=True)


if __name__=='__main__':
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    {'freeze':freeze,'run':run}[sys.argv[1]]()


코드 등록 완료: `Modeling/scripts/modeling_close_analysis.py`

### `Modeling/scripts/modeling_close_documents.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Generate detailed and structurally condensed Markdown manuscripts only.

주요 함수: `compact()`, `update()`, `detailed()`.

In [65]:
%%submission_source Modeling/scripts/modeling_close_documents.py
"""Generate detailed and structurally condensed Markdown manuscripts only."""
import argparse
import json
from pathlib import Path
from regime_forecast import sha, save_json

ROOT=Path(__file__).resolve().parents[2]
OUT=ROOT/'Modeling/tables/modeling_close'
MAN=ROOT/'Modeling/04_Modeling_원고.md'
COMPACT=ROOT/'Modeling/04_Modeling_3페이지용_압축본.md'

def compact():
    return """# 4. Modeling

## 4.1 시간순 평가와 기본 모델 선택

Analysis에서 확인한 시간 내부 전력 차이를 반영해 **다음 한 시간의 네 15분 값 중 최대값**을 예측했다. 직전 관측까지의 전력·생산량과 달력을 사용하고, 시간 오류·공백은 정확한 시차 연결로 처리했다. 무작위 분할 대신 1~3월→4월, 1~4월→5월, 1~5월→6월로 평가했으며 Optuna 튜닝도 각 평가 월 이전의 내부 시간순 검증에서 수행했다.

![모델 개발과 매시간 적용 흐름](figures/modeling_close/pipeline.png)

*그림 1. 위는 개발, 아래는 관측 시점에 맞춘 적용 흐름이다. OOF는 해당 평가 월 이전 자료로 학습한 모델의 예측이다.*

Ridge·Elastic Net·SVR, HGB·XGBoost·LightGBM·CatBoost·ExtraTrees와 앙상블을 비교했다. 같은 기본 입력·2,184시간에서 대표 결과는 다음과 같다.

| 모델 | 전체 MAE | 일별 최대 시간 MAE |
|---|---:|---:|
| 직전 값 기준 | 13.793 | 36.589 |
| HGB | **6.738** | 11.751 |
| ExtraTrees | 6.825 | **11.540** |
| XGBoost | 7.131 | 12.864 |

**전체·최대 시간 오차를 함께 보는 사전 선택 규칙에 따라 HGB를 유지했다.** 마지막 15분 값 추가는 개발에서 유리했지만 후반기 최대 시간까지 개선이 유지되지는 않았다. 이에 전력 수준이 크게 올라 유지되는 시작을 따로 예측하고, 그때만 HGB를 보완했다.

관련 Python: [시차 구성](scripts/m01_prepare.py) · [시간순 모델 비교·튜닝](scripts/m02_rerun.py) · [입력 효과](scripts/m03_ablation.py). 그림 1 생성: [modeling_close_figures.py](scripts/modeling_close_figures.py)의 `pipeline()`.

## 4.2 지속 상승 예측과 선택적 HGB 보완

시간 산술평균이 직전 값과 과거 6시간 중앙값보다 각각 59.5 이상 높고, 중앙값 대비 상승폭이 과거 IQR의 1.5배 이상인 시작을 후보로 삼았다. 이후 3시간 동안 기존 기준선보다 29.75 이상 높은 상태가 이어지면 **지속 상승**으로 정의했다. 1월 자료에서 정한 분석 기준이며, 미래 3시간은 정답 확인에만 사용했다.

Logistic이 과거 정보로 상승을 경고할 때만 상태 입력을 보완한 HGB B1을 적용하고 나머지는 기본 HGB B0를 유지했다. 초기 오경보 38건은 모두 상태 경과시간이 학습 최대 64시간을 넘어선 180~227시간에서 발생했다. **분류기의 경과시간 입력 하나를 제거**하고 과거 OOF에서 경계 0.4555를 정하자 개발 적중 12건·오경보 0건을 유지했다.

![성능·오경보와 개선·미탐지 사례 통합](figures/modeling_close/compact_evidence.png)

*그림 2. 7~8월 탐색적 재평가. 위: 동일 1,344시간의 오차와 정답이 확인된 1,340시간의 경고 수. 일별 최대는 완전한 56일을 같은 가중치로 평가했다. 아래: 개선·미탐지 사례; 음영은 지속 정답 3시간, 점선은 직전 관측 완료 시점이다.*

상승 시작 MAE는 **20.037→18.205(9.1% 감소)**, 평균 과소예측은 **19.419→16.342(15.8% 감소)**, 일별 최대 시간 MAE는 13.179→12.574(4.6% 감소)였다. 전체 MAE는 6.335→6.317로 0.28% 줄었다. 보완은 7시간에만 적용했고 적중 7건을 유지하면서 오경보를 38→0건으로 줄였다.

분류는 1~6월 4,336시간, 회귀는 2~6월 3,598시간으로 한 번 학습해 7~8월에 고정했다. 매시간 입력만 갱신했으며 7월 정답으로 재학습하지 않았다. 이미 확인한 후반기 결과를 바탕으로 수정했으므로 독립 최종시험이 아닌 탐색적 결과다.

관련 Python: [사건 정의](scripts/regime_diagnose.py) · [분류](scripts/regime_forecast.py) · [보완 HGB](scripts/regime_integrate.py) · [선택 결합](scripts/regime_route.py) · [고정 평가](scripts/regime_followup.py) · [경과시간 제거](scripts/regime_age_ablation.py). 그림 2 생성: [modeling_close_figures.py](scripts/modeling_close_figures.py)의 `compact_evidence()`.

## 4.3 오류 해석·현장 활용과 재현성

공통 상승 13건 중 7건을 탐지해 **정밀도 100%(7/7), 재현율 53.8%(7/13)**였다. 경고 7건 중 값의 오차는 4건에서 줄고 3건에서 늘었다. 그림 2의 7/12는 실제 최대 198에 대해 181.13→192.52로 개선됐지만, 8/13은 경고가 없어 실제 207을 182.40으로 낮게 예측했다. 각 사례는 개선폭·실제 최대값의 중앙 순위로 선정했다.

표준화 입력×계수로 Logistic의 로짓을 분해하면 적중·미탐지의 과거 전력 평균 기여는 13.86·4.16이었다. 과거 전력도 판단에 기여했지만 경고는 모두 월요일 08시에 집중됐다. 사후 단순 주간 규칙은 8건 적중·오경보 1건이고 일별 최대 오차는 같아 추가 이득은 작았다. 성과는 **지속 상승 일부에서 오경보를 억제하고 과소예측을 줄인 선택적 보완**이며, 양방향 전환 전체의 탐지로 확대하지 않는다.

현장에서는 직전 관측 완료 후 최대 전력·상승 경고를 확인하고, 담당자가 예정된 가동·동시 사용 조건과 함께 보완 예측을 검토한다. 무경고를 안전 판정으로 쓰지 않으며 필수 시차가 없으면 보완을 보류한다. 가동 조정과 전력·비용 절감은 실측 성과가 아닌 활용 제안이다.

원자료부터 현재 고정 방법을 재실행해 분류 1,428개·회귀 4,032개 예측의 일치를 확인했다. 관련 Python: [원자료 재현·변수 기여·사례 선정](scripts/modeling_close_analysis.py) · [주간 규칙 대조](scripts/regime_age_calendar_control.py) · [한 번에 재현](scripts/modeling_reproduce_selected.py) · [수치·그림·링크 검증](scripts/modeling_close_verify.py). 정규 CPython 3.13에서 재현 스크립트에 `--output-dir Modeling/reproduction_new`로 새 폴더를 지정한다.

전체 실행 명령·단계별 검증 코드·근거표는 [상세 원고 4.5·4.12·4.13](04_Modeling_원고.md), 전체 파일 목록은 [제출 보존 목록](submission_manifest.json), 환경은 [requirements.txt](requirements.txt)에 연결했다. 새 환경 설치와 과거 Optuna 전체 재탐색을 검증한 것은 아니다.
"""


def update():
    for path,name in [(MAN,'manuscript_before_markdown_correction.md'),
                      (ROOT/'Modeling/04_Modeling_4페이지용_압축본.md','compact_before_3page.md')]:
        backup=OUT/name
        if path.exists() and not backup.exists():backup.write_bytes(path.read_bytes())
    text=MAN.read_text(encoding='utf-8')
    assert text.count('## 4.13 ')==1
    MAN.write_text(text.split('## 4.13 ')[0].rstrip()+'\n\n'+detailed().strip()+'\n',encoding='utf-8')
    COMPACT.write_text(compact(),encoding='utf-8')
    readme=ROOT/'Modeling/README.md'
    text=readme.read_text(encoding='utf-8')
    start=text.index('사용자 승인으로 추가 성능 탐색을 마치고')
    end=text.index('\n\n',start)
    note='사용자 승인으로 추가 성능 탐색을 마치고 [상세 원고 4.13](04_Modeling_원고.md#413-보고서-시각화영향요인-해석과-모델링-종료)에 사례·변수 기여·활용·재현을 연결했다. [3페이지용 압축본](04_Modeling_3페이지용_압축본.md)은 평가·모델 선택, 지속 상승 보완, 오류·활용·재현성의 세 절로 재구성했다. 흐름도와 성능·경고·사례 통합 그림 두 개를 사용하고, 절마다 Python 전체 소스 링크를 연결한다. 마크다운만 작성하며 3페이지는 편집 목표다. 전체 실험·그림·코드와 근거는 상세 원고에 보존한다.'
    readme.write_text(text[:start]+note+text[end:],encoding='utf-8')
    record={'status':'written','script_sha256':sha(__file__),'format':'markdown_only',
            'target_pages':3,'pagination_verified':False,'compact_sections':3,'compact_figures':2,
            'structural_merges':['evaluation_and_model_selection','state_forecast_and_performance','errors_use_and_reproduction'],
            'figure_merge':'metrics_alarms_improved_and_missed_cases',
            'outputs_sha256':{p.relative_to(ROOT).as_posix():sha(p) for p in [MAN,readme,COMPACT]},
            'original_manuscript_sha256':sha(OUT/'manuscript_before.md')}
    save_json(OUT/'documents.json',record)
    print(json.dumps({'status':'written','format':'markdown_only','target_pages':3,'sections':3,'figures':2}))


def detailed():
    return '''## 4.13 보고서 시각화·영향요인 해석과 모델링 종료

최종 설명의 중심은 4.12의 지속 상승 경고와 선택적 HGB 보완이다. 추가 모델 탐색은 종료하고, 보고서용 그림·해석·활용과 현재 고정 방법의 재현 점검을 마쳤다. 전체 분석 흐름도는 4.1 앞에 두고, 성능·경고 비교는 4.12에 연결했다. 아래 그림은 제3장의 사례·영향요인 근거다.

![지속 상승의 개선·미탐지 사례](figures/modeling_close/transition_cases.png)

개선 사례는 공통 상승 중 경고가 있고 절대오차가 줄어든 4건을 개선폭 순으로 정렬한 중앙 순위의 7월 12일 08시다. 실제 최대값 198에 대해 기본 예측 181.129가 보완 후 192.522로 가까워졌다. 미탐지 사례는 놓친 6건의 실제 최대값 중앙 순위인 8월 13일 08시다. 실제 207에 비해 두 예측 모두 182.404로 낮았다. 시작 전 4시간부터 이후 5시간까지 공통 예측이 완전한 사례만 후보로 삼았고, 두 후보군 모두 해당 조건을 충족했다. [선정 규칙·결과](tables/modeling_close/case_selection.csv)와 [전체 사례 창](tables/modeling_close/case_windows.csv)을 보존했다. 경고 7건 중 개별 절대오차가 줄어든 것은 4건이고 3건은 늘었다. 사례 그림의 개선폭을 전체 평균 성과로 일반화하지 않는다.

![Logistic의 변수군별 기여](figures/modeling_close/logit_contributions.png)

추가 해석의 질문은 “경고가 월요일 08시에 집중되는 이유가 달력뿐인가, 과거 관측 상태도 판단에 기여하는가”였다. 현재 분류기는 표준화 뒤 선형 Logistic이므로, 입력별 표준화값에 계수를 곱한 항과 절편을 직접 합산해 모델의 로짓을 정확하게 복원했다. SHAP을 추가 실행하지 않았으며 위 값은 확률 백분율이나 인과효과가 아니다. 변수 간 상관 때문에 각 항을 독립적인 공정 영향력으로 해석하지 않는다.

| 공통 평가 조건 | 건수 | 과거 전력 평균 기여 | 달력 평균 기여 | 평균 로짓 |
|---|---:|---:|---:|---:|
| 상승 적중 | 7 | 13.857 | 0.300 | 4.008 |
| 상승 미탐지 | 6 | 4.159 | -1.220 | -7.339 |
| 월요일 08시 비사건 | 1 | 2.719 | 0.150 | -6.813 |

공통 절편은 -10.104이며 생산량·추적 상태 항도 합산에 포함했다. 과거 전력 입력도 점수에 기여했지만, 이 분해가 달력보다 독립적인 예측 가치가 크다는 실험은 아니다. 경고는 여전히 월요일 08시에 한정되고 단순 주간 규칙과의 성능 차이는 작다. 한 건뿐인 월요일 비사건의 해석도 해당 사례 범위로 제한한다. [전체 입력 계수](tables/modeling_close/coefficient_table.csv), [1,428시간 기여](tables/modeling_close/logit_contributions.csv), [조건별 요약](tables/modeling_close/contribution_summary.csv), [상승 13건](tables/modeling_close/event_explanations.csv)을 보존한다.

### 현장 활용과 종료 범위

제4장의 제안은 직전 관측 완료 → 다음 최대 전력·지속 상승 경고 확인 → 담당자가 계획된 가동·동시 사용 조건과 함께 보완 예측을 검토하는 절차다. 무경고를 안전으로 판단하지 않으며, 필수 시차가 없는 경우 보완 예측을 보류하고 자료 상태를 표시한다. 가동 조정·자동 제어·절감량은 실측하지 않았다. 제5장에는 전력 수준의 지속 전환을 별도 정의하고, 오류 진단에서 찾은 입력 문제를 수정해 불필요한 보완 적용을 줄인 기여를 연결한다.

현재 방법은 제한된 탐색 성과를 보고하는 후보로 고정한다. 원자료 진단·시간순 모델 비교·입력 효과·상승 보완·오류 해석·시각화·현재 환경 재현 점검이 끝났으므로 모델링의 추가 성능 탐색을 종료한다. 남은 작업은 최종 한글 양식에 각 장을 배치하고 EDA·Analysis와 분량을 조정하는 제출 편집이다. 전체 급등·급락 탐지가 완성되었다는 의미로 종료를 표현하지 않는다.

### 원자료부터 현재 선택 방법까지 재현

[modeling_close_analysis.py](scripts/modeling_close_analysis.py)는 기존 조건을 읽어 원자료에서 M01 시차와 지속 사건을 다시 만들고, 분류 학습 4,336시간·회귀 학습 3,598시간을 재구성했다. 4~6월 OOF 분류 3개와 6월 말 고정 분류기·B0·B1을 다시 학습한 뒤, 과거 경계 0.4554612321·분류 1,428개·회귀 4,032개 예측을 대조했다. 모두 일치했다. 새 후보나 튜닝을 추가한 재학습은 아니다.

최초 재현에서는 원자료에서 만든 실수값과 기존 CSV를 읽은 값 사이의 미세한 수치 차이로 HGB 예측이 일치하지 않았다. 기존 입력 파일과 합친 학습표의 CSV 저장·읽기 단계를 동일하게 적용하자 학습 입력이 정확히 일치했고 예측 대조도 통과했다. 실행 조건과 결과를 바꾸지 않고 수치 저장 경계를 코드에 명시했으며, 수정 전 코드·계약도 보존했다. 이 점은 다른 파서·환경에서 재실행할 때 확인해야 할 재현 조건이다.

평가자용 [modeling_reproduce_selected.py](scripts/modeling_reproduce_selected.py)는 새 출력 폴더를 지정해 원자료부터 고정 방법의 학습·추론·기존 결과 대조까지 한 번에 수행한다. 실제로 `Modeling/tables/modeling_replay`에서 실행했고 [실행 기록](tables/modeling_replay/run.json)의 분류·회귀 대조가 통과했다. 이미 사용한 폴더는 덮어쓰지 않으므로 재실행 때는 새로운 폴더명을 지정한다. 현재 방법의 재현에 필요한 기존 정의·설정·참조 결과를 함께 제출한다.

다음 명령을 프로젝트 루트의 정규 CPython 3.13·승격 Start-Process 경로에서 실행한다. 첫 줄은 평가자용 재현 경로이며, 뒤의 명령은 최초 분석·그림·문서 생성 이력이다. 현재 자료의 무결성은 verify 명령으로 확인한다.

~~~text
Modeling/scripts/modeling_reproduce_selected.py --output-dir Modeling/reproduction_new
Modeling/scripts/modeling_close_analysis.py freeze
Modeling/scripts/modeling_close_analysis.py run
Modeling/scripts/modeling_close_figures.py
Modeling/scripts/modeling_close_documents.py --update
Modeling/scripts/modeling_close_verify.py
Modeling/scripts/modeling_submission_audit.py --verify
~~~

[modeling_close_figures.py](scripts/modeling_close_figures.py)는 저장 결과에서 PNG·SVG 그림을 만든다. [modeling_close_documents.py](scripts/modeling_close_documents.py)는 상세 원고를 보존하고 [3페이지용 압축본](04_Modeling_3페이지용_압축본.md)을 마크다운으로 생성한다. 압축본은 평가·모델 선택, 지속 상승 보완, 오류·활용·재현성의 세 절로 재구성했다. 성능·경고 수·개선 및 미탐지 사례를 [통합 그림](figures/modeling_close/compact_evidence.png) 하나로 묶어 흐름도와 함께 두 그림만 싣는다. 변수 기여의 전체 그림·표와 과거 실험별 결과·전체 코드 연결은 이 상세 원고에 남겼다. [modeling_close_verify.py](scripts/modeling_close_verify.py)는 기여 합·예측 오차·그림 근거·문서 보존·링크를 검증한다. 3페이지는 편집 목표이며 마크다운 자체의 실제 페이지 수를 검증했다는 뜻은 아니다.

원본 상세 원고는 [시각화 추가 전 사본](tables/modeling_close/manuscript_before.md)에 보존했다. [실행 기록](tables/modeling_close/run.json)과 [원자료부터 재현한 예측](tables/modeling_close/reproduced_predictions.csv)을 제출 보존 목록에 추가한다. 현재 환경에서 선택 방법을 재실행한 결과이며, 깨끗한 새 환경의 설치 검증이나 기존 Optuna 전체 탐색을 다시 수행한 것은 아니다.
'''



if __name__=='__main__':
    parser=argparse.ArgumentParser()
    parser.add_argument('--update',action='store_true',required=True)
    parser.parse_args()
    update()


코드 등록 완료: `Modeling/scripts/modeling_close_documents.py`

### `Modeling/scripts/modeling_close_figures.py`

원고 연결: Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_원고.md; Report\03_05_보고서_원고.md.

코드 설명: Publication figures from frozen results; no fitting or rescaling to hide errors.

주요 함수: `keep()`, `pipeline()`, `performance()`, `cases()`, `compact_evidence()`, `contribution()`, `main()`.

In [66]:
%%submission_source Modeling/scripts/modeling_close_figures.py
"""Publication figures from frozen results; no fitting or rescaling to hide errors."""
import json
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np
import pandas as pd
from PIL import Image
from regime_forecast import sha, save_json

ROOT=Path(__file__).resolve().parents[2]
OUT=ROOT/'Modeling/tables/modeling_close'
FIG=ROOT/'Modeling/figures/modeling_close'
AGE=ROOT/'Modeling/tables/regime_age_ablation'
BLUE='#275D8C'; ORANGE='#C66B31'; GREY='#8995A0'; DARK='#253746'


def keep(fig,name):
    fig.savefig(FIG/(name+'.png'),dpi=200,facecolor='white',bbox_inches='tight')
    fig.savefig(FIG/(name+'.svg'),facecolor='white',bbox_inches='tight')
    plt.close(fig)


def pipeline():
    fig,ax=plt.subplots(figsize=(10.6,3.6));ax.set_xlim(0,10);ax.set_ylim(-.2,3.7);ax.axis('off')
    def box(x,y,w,h,label,color='#F0F4F7'):
        ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.05',facecolor=color,edgecolor='#9CACB8',lw=1))
        ax.text(x+w/2,y+h/2,label,ha='center',va='center',fontsize=11,color=DARK)
    def arrow(a,b): ax.annotate('',xy=b,xytext=a,arrowprops={'arrowstyle':'->','color':GREY,'lw':1.4})
    box(.1,2.35,2.15,1,'자료 진단·시차 구성\n정확한 과거 시각 연결')
    box(2.65,2.35,2.5,1,'시간순 모델·입력 비교\n전체·일별 최대 오차 평가')
    box(5.55,2.35,2.05,1,'기본 HGB 선택\n상승 오류 진단')
    box(8,2.35,1.8,1,'지속 상승 정의\n과거 정보로 분류')
    for a,b in [((2.25,2.85),(2.65,2.85)),((5.15,2.85),(5.55,2.85)),((7.6,2.85),(8,2.85))]:arrow(a,b)
    box(.1,.35,2.15,1,'매시간 t-1 관측 완료\n다음 t 시간 예측')
    box(2.65,.35,2.5,1,'지속 상승 경고 판정\n과거 OOF로 정한 경계')
    box(5.7,1,1.8,.65,'경고 있음: 보완 B1','#E5EFF8')
    box(5.7,0,1.8,.65,'경고 없음: 기본 B0')
    box(8,.35,1.8,1,'최대 전력 예측\n담당자 확인')
    arrow((2.25,.85),(2.65,.85));arrow((5.15,.9),(5.7,1.3));arrow((5.15,.8),(5.7,.3))
    arrow((7.5,1.3),(8,1));arrow((7.5,.3),(8,.65))
    ax.plot([8.9,8.9,3.9],[2.35,1.88,1.88],color=GREY,lw=1.4)
    arrow((3.9,1.88),(3.9,1.35))
    ax.text(.1,3.55,'개발 흐름',fontsize=11,weight='bold');ax.text(.1,1.62,'적용 흐름',fontsize=11,weight='bold')
    keep(fig,'pipeline')


def performance(axes=None):
    m=pd.read_csv(AGE/'followup_metrics.csv',encoding='utf-8-sig')
    standalone=axes is None
    if standalone:fig,axes=plt.subplots(1,2,figsize=(10.6,3.3),gridspec_kw={'width_ratios':[1.6,1]})
    ax,bx=axes
    cols=['all','up_start','daily_peak_day_equal']; y=np.arange(3); height=.33
    for name,dy,color,label in [('B0',-height/2,GREY,'기본 HGB'),('G1_noage',height/2,BLUE,'선택적 보완')]:
        vals=[float(m.loc[(m.variant==name)&(m.period=='Jul-Aug')&(m.condition==x),'mae'].iloc[0]) for x in cols]
        ax.barh(y+dy,vals,height,color=color,label=label)
        for yy,v in zip(y+dy,vals):ax.text(v+.25,yy,f'{v:.2f}',va='center',fontsize=10)
    ax.set_yticks(y,['전체','상승 시작','일별 최대 시간']);ax.invert_yaxis();ax.set_xlim(0,24)
    ax.set_xlabel('MAE (원자료 전력 척도)');ax.set_title('7~8월 동일 1,344시간 비교',fontsize=12)
    ax.legend(frameon=False,fontsize=10,loc='lower right');ax.grid(axis='x',alpha=.15);ax.set_axisbelow(True)
    bx.barh([0,1],[7,7],color=BLUE,label='적중');bx.barh([0,1],[38,0],left=[7,7],color=ORANGE,label='오경보')
    bx.set_yticks([0,1],['수정 전','경과시간 제거']);bx.invert_yaxis();bx.set_xlim(0,50)
    bx.text(3.5,0,'7',ha='center',va='center',color='white');bx.text(26,0,'38',ha='center',va='center',color='white')
    bx.text(3.5,1,'7',ha='center',va='center',color='white');bx.text(8,1,'오경보 0',va='center',fontsize=10)
    bx.set_title('적중 유지·오경보 억제',fontsize=12);bx.set_xlabel('경고 수 (공통 정답 1,340시간)')
    bx.legend(frameon=False,fontsize=10,loc='lower right')
    if standalone:
        fig.tight_layout(w_pad=2);keep(fig,'performance_and_alarms')


def cases(axes=None):
    p=pd.read_csv(OUT/'case_windows.csv',encoding='utf-8-sig',parse_dates=['timestamp','onset_time'])
    standalone=axes is None
    if standalone:fig,axes=plt.subplots(1,2,figsize=(10.6,3.65),sharey=True)
    for ax,name,label in zip(axes,['detected_improved','missed'],['탐지·오차 개선 사례','미탐지 사례']):
        g=p.loc[(p.case==name)&(p.variant=='B0')].sort_values('timestamp')
        other=p.loc[(p.case==name)&(p.variant=='G1_noage')].set_index('timestamp').loc[g.timestamp]
        t=g.onset_time.iloc[0];x=(g.timestamp-t).dt.total_seconds()/3600
        ax.axvspan(0,2,color='#EAF1F6');ax.axvline(-1,color=GREY,lw=.9,ls=':')
        ax.plot(x,g.actual,'o-',color=DARK,label='실제 최대',ms=3)
        ax.plot(x,g.observed_mean,':',color=GREY,label='실제 평균')
        ax.plot(x,g.prediction,'--',color=ORANGE,label='기본 HGB',lw=1.6)
        ax.plot(x,other.prediction,'-',color=BLUE,label='선택적 보완',lw=1.4,marker='.',ms=4)
        ax.set_title(f'{label} | {t:%m/%d %H시}',fontsize=12)
        ax.set_xticks([-4,-2,0,2,4]);ax.set_xlabel('상승 시작 기준 시간 (0 = 시작)')
        ax.set_ylim(0,240);ax.grid(alpha=.15);ax.text(.03,.91,'음영: 지속 정답의 3시간',transform=ax.transAxes,fontsize=9)
    axes[0].set_ylabel('전력 (원자료 척도)')
    if standalone:
        handles,labels=axes[0].get_legend_handles_labels();fig.legend(handles,labels,ncol=4,loc='lower center',frameon=False,fontsize=10)
        fig.tight_layout(rect=[0,.1,1,1]);keep(fig,'transition_cases')


def compact_evidence():
    """Draw metrics, alarms and contrasting cases together from the source tables."""
    fig,axes=plt.subplots(2,2,figsize=(10.6,6.7))
    performance(axes[0]);cases(axes[1])
    handles,labels=axes[1,0].get_legend_handles_labels()
    fig.legend(handles,labels,ncol=4,loc='lower center',frameon=False,fontsize=10)
    fig.tight_layout(rect=[0,.055,1,1],h_pad=2,w_pad=2)
    keep(fig,'compact_evidence')


def contribution():
    f=pd.read_csv(OUT/'contribution_summary.csv',encoding='utf-8-sig')
    fig,ax=plt.subplots(figsize=(8,3.6));x=np.arange(3)
    for i,(col,label,color) in enumerate([('calendar_contribution','달력',GREY),('power_contribution','과거 전력',BLUE),('production_contribution','생산량',ORANGE),('state_contribution','추적 상태','#6A8B72')]):
        ax.bar(x+(i-1.5)*.18,f[col],width=.18,color=color,label=label)
    ax.set_xticks(x,['상승 적중 (7건)','상승 미탐지 (6건)','월요일 08시 비사건 (1건)'])
    ax.axhline(0,color=DARK,lw=.7);ax.set_ylabel('평균 로짓 기여 (확률·인과효과 아님)')
    ax.legend(frameon=False,ncol=4,loc='upper right',fontsize=10)
    ax.set_title('고정 Logistic의 실제 계산 분해 | 공통 절편 -10.104',fontsize=12)
    ax.grid(axis='y',alpha=.15);ax.set_axisbelow(True);fig.tight_layout();keep(fig,'logit_contributions')


def main():
    r=json.loads((OUT/'run.json').read_text(encoding='utf-8'))
    assert r['status']=='passed'
    for n,h in r['outputs_sha256'].items():assert sha(OUT/n)==h
    FIG.mkdir(parents=True,exist_ok=True)
    plt.rcParams.update({'font.family':'Malgun Gothic','axes.unicode_minus':False,'font.size':11,'svg.fonttype':'path'})
    pipeline();performance();cases();contribution();compact_evidence()
    preview=ROOT/'tmp/modeling_visual_review/closing';preview.mkdir(parents=True,exist_ok=True)
    for p in FIG.glob('*.png'):
        with Image.open(p) as im:im.resize((im.width//2,im.height//2),Image.Resampling.LANCZOS).save(preview/p.name)
    save_json(OUT/'figures.json',{'status':'generated','script_sha256':sha(__file__),'run_sha256':sha(OUT/'run.json'),
                               'figures_sha256':{p.relative_to(ROOT).as_posix():sha(p) for p in FIG.iterdir() if p.suffix in ['.png','.svg']},
                               'visual_review':'pending','new_fits':0})
    print(json.dumps({'status':'generated','figures':5,'preview_directory':str(preview)}))


if __name__=='__main__':main()


코드 등록 완료: `Modeling/scripts/modeling_close_figures.py`

### `Modeling/scripts/modeling_close_verify.py`

원고 연결: Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_원고.md.

코드 설명: Verify closing analysis, selected forecasts, explanation identities and report scope.

주요 함수: `main()`.

In [67]:
%%submission_source Modeling/scripts/modeling_close_verify.py
"""Verify closing analysis, selected forecasts, explanation identities and report scope."""
import argparse
import json
import math
import re
from pathlib import Path
import numpy as np
import pandas as pd
from regime_forecast import sha, save_json

ROOT=Path(__file__).resolve().parents[2]
OUT=ROOT/'Modeling/tables/modeling_close'


def main(visual=False):
    run=json.loads((OUT/'run.json').read_text(encoding='utf-8'))
    c=json.loads((OUT/'contract.json').read_text(encoding='utf-8'))
    assert run['status']=='passed' and run['script_sha256']==c['script_sha256']==sha(ROOT/'Modeling/scripts/modeling_close_analysis.py')
    assert run['contract_sha256']==sha(OUT/'contract.json')
    for p,h in run['inputs_sha256'].items():assert sha(ROOT/p)==h,p
    for p,h in run['outputs_sha256'].items():assert sha(OUT/p)==h,p
    x=pd.read_csv(OUT/'logit_contributions.csv',encoding='utf-8-sig',parse_dates=['timestamp'])
    all_terms=['calendar_contribution','power_contribution','production_contribution','state_contribution']
    for r in x.itertuples():
        summed=math.fsum([r.intercept]+[getattr(r,n) for n in all_terms])
        assert abs(summed-r.logit)<1e-9
        assert abs(1/(1+math.exp(-summed))-r.score)<1e-9
        assert r.alarm==int(r.score>=run['oof_threshold'])
    p=pd.read_csv(OUT/'reproduced_predictions.csv',encoding='utf-8-sig',parse_dates=['timestamp'])
    old=pd.read_csv(ROOT/'Modeling/tables/regime_age_ablation/followup_predictions.csv',encoding='utf-8-sig',parse_dates=['timestamp'])
    joined=p.merge(old,on=['timestamp','variant'],suffixes=('_new','_old'),validate='one_to_one')
    assert len(joined)==4032
    np.testing.assert_allclose(joined.prediction_new,joined.prediction_old,atol=1e-8,rtol=1e-9)
    metrics=[]
    for (variant),g in p.groupby('variant'):
        for cond,gg in [('all',g),('up_start',g.loc[g.sustained_onset==1])]:
            errors=[r.prediction-r.actual for r in gg.itertuples()]
            metrics.append({'variant':variant,'condition':cond,'mae':math.fsum(abs(e) for e in errors)/len(errors)})
    refs=pd.read_csv(ROOT/'Modeling/tables/regime_age_ablation/followup_metrics.csv',encoding='utf-8-sig')
    for row in metrics:
        expected=float(refs.loc[(refs.variant==row['variant'])&(refs.condition==row['condition'])&(refs.period=='Jul-Aug'),'mae'].iloc[0])
        assert math.isclose(row['mae'],expected,abs_tol=1e-8)
    e=pd.read_csv(OUT/'event_explanations.csv',encoding='utf-8-sig')
    assert len(e)==13 and e.alarm.sum()==7
    assert ((e.alarm==1)&(e.absolute_error_gain>0)).sum()==4
    assert ((e.alarm==1)&(e.absolute_error_gain<0)).sum()==3
    fig=json.loads((OUT/'figures.json').read_text(encoding='utf-8'))
    assert fig['script_sha256']==sha(ROOT/'Modeling/scripts/modeling_close_figures.py')
    for n,h in fig['figures_sha256'].items():assert sha(ROOT/n)==h,n
    docs=json.loads((OUT/'documents.json').read_text(encoding='utf-8'))
    assert docs['script_sha256']==sha(ROOT/'Modeling/scripts/modeling_close_documents.py')
    for n,h in docs['outputs_sha256'].items():assert sha(ROOT/n)==h,n
    assert docs['original_manuscript_sha256']==sha(OUT/'manuscript_before.md')
    man=(ROOT/'Modeling/04_Modeling_원고.md').read_text(encoding='utf-8')
    before=(OUT/'manuscript_before.md').read_text(encoding='utf-8')
    clean=re.sub(r'<!-- MODELING_CLOSE_BEGIN -->.*?<!-- MODELING_CLOSE_END -->\s*','',man,flags=re.S).split('## 4.13 ')[0].strip()
    assert clean==before.strip()
    links=0
    for path in [ROOT/'Modeling/04_Modeling_원고.md',ROOT/'Modeling/04_Modeling_3페이지용_압축본.md',ROOT/'Modeling/README.md']:
        text=path.read_text(encoding='utf-8')
        if path.name=='README.md':text=text.split('## S06 실행 결과:')[0]
        for target in re.findall(r'\]\(([^)]+)\)',text):
            if target.startswith(('http:','https:','#')):continue
            linked=(path.parent/target.split('#')[0].strip('<>')).resolve()
            if linked != (OUT/'verification.json').resolve():
                assert linked.exists(),target
            links+=1
    compact=(ROOT/'Modeling/04_Modeling_3페이지용_압축본.md').read_text(encoding='utf-8')
    assert len(re.findall(r'!\[',compact))==2
    assert len(re.findall(r'^## ',compact,flags=re.M))==3
    assert docs['format']=='markdown_only' and docs['target_pages']==3
    assert not docs['pagination_verified']
    for path in [ROOT/'Modeling/04_Modeling_원고.md',ROOT/'Modeling/04_Modeling_3페이지용_압축본.md']:
        assert '.pdf)' not in path.read_text(encoding='utf-8')
    assert sha(ROOT/'README.md')==c['root_readme_sha256']
    result={'status':'passed','script_sha256':sha(__file__),'run_sha256':sha(OUT/'run.json'),
            'documents_sha256':sha(OUT/'documents.json'),'figures_manifest_sha256':sha(OUT/'figures.json'),
            'classification_logit_identities':len(x),'regression_predictions_checked':len(joined),
            'scalar_regression_metrics_checked':len(metrics),'historical_manuscript_preserved':True,
            'compact_target_pages':3,'pagination_verified':False,'compact_sections':3,
            'compact_figures':2,'links_checked':links,
            'visual_review':'passed' if visual else 'pending','no_new_models_selected':True}
    save_json(OUT/'verification.json',result)
    print(json.dumps(result),flush=True)


if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--visual-reviewed',action='store_true');main(p.parse_args().visual_reviewed)


코드 등록 완료: `Modeling/scripts/modeling_close_verify.py`

### `Modeling/scripts/modeling_reproduce_selected.py`

원고 연결: Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_원고.md.

코드 설명: Replay selected modeling from raw data into a new output folder.

Example: python Modeling/scripts/modeling_reproduce_selected.py --output-dir Modeling/tables/modeling_replay
Run with regular CPython3.13. Existing outputs are never deleted or overwritten.
Frozen source definitions/model settings and reference results must be retained.

주요 함수: `main()`.

In [68]:
%%submission_source Modeling/scripts/modeling_reproduce_selected.py
"""Replay selected modeling from raw data into a new output folder.

Example: python Modeling/scripts/modeling_reproduce_selected.py --output-dir Modeling/tables/modeling_replay
Run with regular CPython3.13. Existing outputs are never deleted or overwritten.
Frozen source definitions/model settings and reference results must be retained.
"""
import argparse
import sys
from pathlib import Path
import modeling_close_analysis as pipeline


def main():
    p=argparse.ArgumentParser()
    p.add_argument('--output-dir',required=True,help='New directory under project Modeling/')
    args=p.parse_args()
    root=pipeline.ROOT.resolve()
    destination=(root/args.output_dir).resolve()
    if not destination.is_relative_to(root/'Modeling') or destination==root/'Modeling':
        raise ValueError('Choose a new subdirectory inside project Modeling/.')
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError('Output directory is not empty; choose a new path. Nothing was overwritten.')
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    pipeline.OUT=destination
    pipeline.freeze()
    pipeline.run()


if __name__=='__main__':main()


코드 등록 완료: `Modeling/scripts/modeling_reproduce_selected.py`

### `Modeling/scripts/modeling_submission_audit.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Record or verify the complete preservation inventory for the current Modeling manuscript.

No training, prediction, deletion, package installation, or original-data modification.
Run --record after an intentional documented update, then use --verify before cleanup.

주요 함수: `digest()`, `local_imports()`, `source_and_result_checks()`, `main()`.

In [69]:
%%submission_source Modeling/scripts/modeling_submission_audit.py
"""Record or verify the complete preservation inventory for the current Modeling manuscript.

No training, prediction, deletion, package installation, or original-data modification.
Run --record after an intentional documented update, then use --verify before cleanup.
"""
import argparse
import ast
import hashlib
import importlib.metadata
import json
import re
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
SCRIPTS = ROOT / 'Modeling/scripts'
MANIFEST = ROOT / 'Modeling/submission_manifest.json'
PACKAGES = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'joblib', 'optuna',
            'xgboost', 'lightgbm', 'catboost', 'matplotlib', 'Pillow']
MANUSCRIPT_SCRIPT_NAMES = [
    'm01_prepare.py', 'm01_verify.py', 'm02_compare.py', 'm02_verify.py',
    'm03_ablation.py', 'm03_verify.py', 'm03_error_followup.py',
    'plot_observed_power_timeline.py', 'plot_m02_manuscript.py', 'plot_m03_manuscript.py',
    'm022_environment.py', 'm022_compare.py', 'm02_rerun.py', 'm02_rerun_verify.py',
    'm02_rerun_documents.py', 'verify_m03_documents.py', 'm02_rerun_document_verify.py',
    'm04_compare.py', 'm04_verify.py', 'modeling_submission_audit.py',
    'm04_rise_compare.py', 'm04_rise_verify.py', 'm04_rise_gate.py',
    'm04_rise_gate_verify.py', 'm04_rise_documents.py',
    'm04_surge_diagnose.py', 'm04_surge_verify.py', 'm04_surge_risk.py',
    'm04_surge_risk_verify.py', 'm04_surge_calibration.py',
    'm04_surge_calibration_verify.py', 'm04_surge_documents.py',
    'm05_evaluate.py', 'm05_verify.py', 'm05_documents.py',
    'redevelopment_r01.py', 'redevelopment_compare.py', 'redevelopment_verify.py', 'redevelopment_documents.py',
    'redevelopment_peak.py', 'redevelopment_peak_verify.py',
]
RESEARCH_SCRIPT_NAMES = ['profile_history.py', 'profile_history_verify.py', 'training_design_diagnose.py', 'training_design_compare.py', 'training_design_recent.py', 'training_design_verify.py', 'error_warning.py', 'error_warning_fit.py', 'error_warning_verify.py', 'priority_review.py', 'priority_review_verify.py', 'priority_review_documents.py', 'regime_diagnose.py', 'regime_diagnose_verify.py', 'regime_forecast.py', 'regime_forecast_verify.py', 'regime_integrate.py', 'regime_integrate_verify.py', 'regime_route.py', 'regime_route_verify.py', 'regime_followup.py', 'regime_followup_verify.py']
RESEARCH_SCRIPT_NAMES += ['regime_age_ablation.py', 'regime_age_ablation_verify.py', 'regime_age_calendar_control.py']
MANUSCRIPT_SCRIPT_NAMES += ['regime_manuscript_documents.py']
MANUSCRIPT_SCRIPT_NAMES += ['modeling_close_analysis.py', 'modeling_close_figures.py', 'modeling_close_documents.py', 'modeling_close_verify.py', 'modeling_reproduce_selected.py']
SCRIPT_NAMES = MANUSCRIPT_SCRIPT_NAMES + RESEARCH_SCRIPT_NAMES
TABLE_DIRS = ['m01', 'm02', 'm03/ab', 'm03/c', 'm03/diagnostics', 'm02_rerun', 'm04', 'm04_rise', 'm04_rise_gate', 'm04_surge', 'm04_surge_risk', 'm04_surge_calibration', 'm05', 'observed_timeline', 'redevelopment', 'profile_history', 'training_design', 'error_warning', 'priority_review', 'regime_diagnosis', 'regime_forecast', 'regime_integration', 'regime_routing', 'regime_followup']
MODEL_DIRS = ['m02', 'm03/ab', 'm03/c', 'm02_rerun', 'm04_rise', 'm04_surge_risk', 'm05', 'redevelopment', 'profile_history', 'training_design', 'error_warning', 'regime_forecast', 'regime_integration', 'regime_followup']
TABLE_DIRS += ['regime_age_ablation']
TABLE_DIRS += ['regime_manuscript']
TABLE_DIRS += ['modeling_close', 'modeling_replay']
MODEL_DIRS += ['regime_age_ablation']
CONFIG_NAMES = ['m01_contract.json', 'm02_contract.json', 'm03_ab_contract.json',
                'm03_c_contract.json', 'm02_rerun_contract.json', 'm022_contract.json', 'm04_contract.json',
                'm04_rise_contract.json', 'm04_rise_gate_contract.json', 'm04_surge_contract.json',
                'm04_surge_risk_contract.json', 'm04_surge_calibration_contract.json', 'm05_contract.json', 'redevelopment_r01_contract.json', 'redevelopment_r02_contract.json', 'redevelopment_peak_contract.json', 'profile_history_contract.json']
DOC_NAMES = ['EDA/README.md', 'Analysis/README.md', 'README.md', 'AGENT_MEMORY.md', 'AGENTS.md', 'Modeling/README.md',
             'Modeling/04_Modeling_원고.md', 'Modeling/10.04_M01_자료시차평가조건_검증기록.md',
             'Modeling/10.04_M02_기본모형비교_실행기록.md',
             'Modeling/10.04_M03_마지막값추가와시차입력_실행기록.md',
             'Modeling/10.04_M02_확장모델비교_실행기록.md',
             'Modeling/10.05_M04_조건별오류와낮은전력규칙_실행기록.md',
             'Modeling/10.05_M04_상승예측결합_실행기록.md',
             'Modeling/10.05_M04_증가량급등과위험출력_실행기록.md',
             'Modeling/10.05_M05_고정후반기평가_실행기록.md',
             'Modeling/10.05_R01_R04_차별성재개발_실행기록.md']


def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def local_imports(path, local_modules):
    tree = ast.parse(path.read_text(encoding='utf-8-sig'))
    found = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom) and node.module:
            name = node.module.split('.')[0]
            if name in local_modules:
                found.add(name)
        elif isinstance(node, ast.Import):
            found.update(n.name.split('.')[0] for n in node.names if n.name.split('.')[0] in local_modules)
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == '__import__':
            if node.args and isinstance(node.args[0], ast.Constant) and isinstance(node.args[0].value, str):
                name = node.args[0].value.split('.')[0]
                if name in local_modules:
                    found.add(name)
    return sorted(found)


def source_and_result_checks():
    contract = json.loads((ROOT / 'Modeling/config/m01_contract.json').read_text(encoding='utf-8'))
    assert digest(ROOT / contract['source']) == contract['source_sha256']
    m1 = ROOT / 'Modeling/tables/m01'
    verify = json.loads((m1 / 'verification.json').read_text(encoding='utf-8'))
    assert verify['status'] == 'passed'
    for name, expected in verify['outputs_sha256'].items():
        assert digest(m1 / name) == expected, name
    for folder in ['m02', 'm03/ab', 'm03/c', 'm02_rerun']:
        directory = ROOT / 'Modeling/tables' / folder
        run = json.loads((directory / 'run.json').read_text(encoding='utf-8'))
        independent = json.loads((directory / 'independent_verification.json').read_text(encoding='utf-8'))
        assert run['status'] == 'completed' and independent['status'] == 'passed'
        if 'run_sha256' in independent:
            assert independent['run_sha256'] == digest(directory / 'run.json')
        for name, expected in run['outputs_sha256'].items():
            assert digest(directory / name) == expected, (folder, name)
        if 'configuration_sha256' in run:
            assert digest(directory / 'selected_configurations.json') == run['configuration_sha256']
        if 'selection_sha256' in run:
            assert digest(directory / 'selection.json') == run['selection_sha256']
        if 'script_sha256' in run:
            script = {'m02': 'm02_compare.py', 'm03/ab': 'm03_ablation.py',
                      'm03/c': 'm03_ablation.py', 'm02_rerun': 'm02_rerun.py'}[folder]
            assert digest(SCRIPTS / script) == run['script_sha256']
        import csv
        with (directory / 'model_manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
            for record in csv.DictReader(stream):
                assert digest(ROOT / record['file']) == record['sha256'], record['file']
    m4 = ROOT / 'Modeling/tables/m04'
    run = json.loads((m4 / 'run.json').read_text(encoding='utf-8'))
    independent = json.loads((m4 / 'independent_verification.json').read_text(encoding='utf-8'))
    assert run['status'] == 'completed' and independent['status'] == 'passed'
    assert independent['run_sha256'] == digest(m4 / 'run.json')
    assert run['script_sha256'] == digest(SCRIPTS / 'm04_compare.py')
    assert independent['script_sha256'] == digest(SCRIPTS / 'm04_verify.py')
    assert run['contract_sha256'] == digest(ROOT / 'Modeling/config/m04_contract.json')
    for name, expected in run['outputs_sha256'].items():
        assert digest(m4 / name) == expected, ('m04', name)
    for name, expected in run['inputs_sha256'].items():
        assert digest(ROOT / name) == expected, ('m04', name)
    for folder, script, verifier, configuration in [
        ('m04_rise', 'm04_rise_compare.py', 'm04_rise_verify.py', 'm04_rise_contract.json'),
        ('m04_rise_gate', 'm04_rise_gate.py', 'm04_rise_gate_verify.py', 'm04_rise_gate_contract.json'),
        ('m04_surge', 'm04_surge_diagnose.py', 'm04_surge_verify.py', 'm04_surge_contract.json'),
        ('m04_surge_risk', 'm04_surge_risk.py', 'm04_surge_risk_verify.py', 'm04_surge_risk_contract.json'),
        ('m04_surge_calibration', 'm04_surge_calibration.py', 'm04_surge_calibration_verify.py', 'm04_surge_calibration_contract.json')]:
        directory = ROOT / 'Modeling/tables' / folder
        run = json.loads((directory / 'run.json').read_text(encoding='utf-8'))
        independent = json.loads((directory / 'independent_verification.json').read_text(encoding='utf-8'))
        assert run['status'] == 'completed' and independent['status'] == 'passed'
        assert independent['run_sha256'] == digest(directory / 'run.json')
        assert run['script_sha256'] == digest(SCRIPTS / script)
        assert independent['script_sha256'] == digest(SCRIPTS / verifier)
        assert run['contract_sha256'] == digest(ROOT / 'Modeling/config' / configuration)
        for name, expected in run['outputs_sha256'].items():
            assert digest(directory / name) == expected, (folder, name)
        for name, expected in run.get('inputs_sha256', {}).items():
            assert digest(ROOT / name) == expected, (folder, name)
        for name, expected in run.get('helper_sha256', {}).items():
            assert digest(SCRIPTS / name) == expected, (folder, name)
        if folder in ['m04_rise', 'm04_surge_risk']:
            import csv
            with (directory / 'model_manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
                for record in csv.DictReader(stream):
                    assert digest(ROOT / record['file']) == record['sha256'], record['file']
        elif folder == 'm04_rise_gate':
            assert run['parent_run_sha256'] == digest(ROOT / 'Modeling/tables/m04_rise/run.json')
    m5 = ROOT / 'Modeling/tables/m05'
    run = json.loads((m5 / 'run.json').read_text(encoding='utf-8'))
    independent = json.loads((m5 / 'independent_verification.json').read_text(encoding='utf-8'))
    assert run['status'] == 'completed' and independent['status'] == 'passed'
    assert independent['run_sha256'] == digest(m5 / 'run.json')
    assert run['script_sha256'] == digest(SCRIPTS / 'm05_evaluate.py')
    assert independent['script_sha256'] == digest(SCRIPTS / 'm05_verify.py')
    assert run['contract_sha256'] == digest(ROOT / 'Modeling/config/m05_contract.json')
    assert independent['diagnostics_sha256'] == digest(m5 / 'confirmed_diagnostics.json')
    for name, expected in run['outputs_sha256'].items():
        assert digest(m5 / name) == expected, ('m05', name)
    for name, expected in run['inputs_sha256'].items():
        assert digest(ROOT / name) == expected, ('m05', name)
    for name, expected in run['helper_sha256'].items():
        assert digest(SCRIPTS / name) == expected, ('m05', name)
    import csv
    with (m5 / 'model_manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
        for record in csv.DictReader(stream):
            assert digest(ROOT / record['file']) == record['sha256'], record['file']
    for stage in ['r01','r02','r03','r03b','r04']:
        directory = ROOT / 'Modeling/tables/redevelopment' / stage
        run = json.loads((directory / 'run.json').read_text(encoding='utf-8'))
        verified = json.loads((directory / 'independent_verification.json').read_text(encoding='utf-8'))
        assert run['status'] in ['completed','skipped'] and verified['status'] == 'passed'
        assert verified['run_sha256'] == digest(directory / 'run.json')
        assert verified['script_sha256'] == digest(SCRIPTS / ('redevelopment_peak_verify.py' if stage in ['r03b','r04'] else 'redevelopment_verify.py'))
        for name, expected in run.get('outputs_sha256', {}).items():
            assert digest(directory / name) == expected, (stage,name)
        for name, expected in run.get('inputs_sha256', {}).items():
            assert digest(ROOT / name) == expected, (stage,name)
        if 'entrypoint_sha256' in run:
            assert run['entrypoint_sha256'] == digest(SCRIPTS / 'redevelopment_peak.py')
        if run['status'] == 'completed' and stage != 'r01':
            assert run['script_sha256'] == digest(SCRIPTS / 'redevelopment_compare.py')
            with (directory / 'model_manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
                for record in csv.DictReader(stream):
                    assert digest(ROOT / record['file']) == record['sha256'], record['file']
    directory = ROOT / 'Modeling/tables/profile_history'
    run = json.loads((directory / 'run.json').read_text(encoding='utf-8'))
    verified = json.loads((directory / 'independent_verification.json').read_text(encoding='utf-8'))
    assert run['status'] == 'completed' and verified['status'] == 'passed'
    assert verified['run_sha256'] == digest(directory / 'run.json')
    assert verified['verifier_sha256'] == digest(SCRIPTS / 'profile_history_verify.py')
    cpath = ROOT / 'Modeling/config/profile_history_contract.json'
    assert run['contract_sha256'] == digest(cpath)
    c = json.loads(cpath.read_text(encoding='utf-8'))
    for path, expected in {**c['inputs_sha256'], **run['outputs_sha256'], **verified['analysis_outputs_sha256']}.items():
        assert digest(ROOT / path) == expected, path
    with (directory / 'model_manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
        for record in csv.DictReader(stream):
            assert digest(ROOT / record['file']) == record['sha256'], record['file']
    p1 = ROOT / 'EDA/tables/p01_peak_shapes'
    c1 = json.loads((p1 / 'contract.json').read_text(encoding='utf-8'))
    r1 = json.loads((p1 / 'run.json').read_text(encoding='utf-8'))
    assert r1['contract_sha256'] == digest(p1 / 'contract.json')
    assert c1['script_sha256'] == digest(ROOT / 'EDA/scripts/p01_peak_shapes.py')
    assert c1['source_sha256'] == digest(ROOT / contract['source'])
    for name, expected in r1['outputs_sha256'].items():
        assert digest(p1 / name) == expected, name
    td = ROOT / 'Modeling/tables/training_design'
    for stage in ['diagnosis', 'weights', 'recent']:
        directory = td / stage
        run = json.loads((directory / 'run.json').read_text(encoding='utf-8'))
        verified = json.loads((directory / 'independent_verification.json').read_text(encoding='utf-8'))
        c = json.loads((directory / 'contract.json').read_text(encoding='utf-8'))
        assert run['status'] == 'completed' and verified['status'] == 'passed'
        assert verified['run_sha256'] == digest(directory / 'run.json')
        assert verified['verifier_sha256'] == digest(SCRIPTS / 'training_design_verify.py')
        assert run['contract_sha256'] == digest(directory / 'contract.json')
        for path, expected in c['inputs_sha256'].items():
            assert digest(ROOT / path) == expected, path
        for path, expected in run['outputs_sha256'].items():
            assert digest(directory / path) == expected, path
        if stage != 'diagnosis':
            with (directory / 'model_manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
                for record in csv.DictReader(stream):
                    assert digest(ROOT / record['file']) == record['sha256'], record['file']
    directory = ROOT / 'Modeling/tables/error_warning'
    run = json.loads((directory / 'run.json').read_text(encoding='utf-8'))
    verified = json.loads((directory / 'independent_verification.json').read_text(encoding='utf-8'))
    c = json.loads((directory / 'contract.json').read_text(encoding='utf-8'))
    diagnosis = json.loads((directory / 'diagnosis.json').read_text(encoding='utf-8'))
    dc = json.loads((directory / 'diagnosis_contract.json').read_text(encoding='utf-8'))
    assert run['status'] == diagnosis['status'] == 'completed' and verified['status'] == 'passed'
    assert verified['run_sha256'] == digest(directory / 'run.json')
    assert verified['diagnosis_sha256'] == c['diagnosis_sha256'] == digest(directory / 'diagnosis.json')
    assert verified['verifier_sha256'] == digest(SCRIPTS / 'error_warning_verify.py')
    assert run['contract_sha256'] == digest(directory / 'contract.json')
    assert diagnosis['contract_sha256'] == digest(directory / 'diagnosis_contract.json')
    assert c['entrypoint_sha256'] == digest(SCRIPTS / 'error_warning_fit.py')
    assert c['frame_sha256'] == digest(directory / 'risk_frame.csv')
    for path, expected in dc['inputs_sha256'].items():
        assert digest(ROOT / path) == expected, path
    for record in [diagnosis, run]:
        for path, expected in record['outputs_sha256'].items():
            assert digest(directory / path) == expected, path
    with (directory / 'model_manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
        for record in csv.DictReader(stream):
            assert digest(ROOT / record['file']) == record['sha256'], record['file']
    directory = ROOT / 'Modeling/tables/priority_review'
    run = json.loads((directory / 'run.json').read_text(encoding='utf-8'))
    verified = json.loads((directory / 'independent_verification.json').read_text(encoding='utf-8'))
    c = json.loads((directory / 'contract.json').read_text(encoding='utf-8'))
    docs = json.loads((directory / 'document_verification.json').read_text(encoding='utf-8'))
    assert run['status'] == 'completed' and verified['status'] == docs['status'] == 'passed'
    assert run['contract_sha256'] == digest(directory / 'contract.json')
    assert verified['run_sha256'] == docs['run_sha256'] == digest(directory / 'run.json')
    assert verified['verifier_sha256'] == digest(SCRIPTS / 'priority_review_verify.py')
    assert c['entrypoint_sha256'] == digest(SCRIPTS / 'priority_review.py')
    assert docs['script_sha256'] == digest(SCRIPTS / 'priority_review_documents.py')
    assert docs['verifier_sha256'] == digest(directory / 'independent_verification.json')
    assert docs['original_manuscript_sha256'] == digest(directory / 'manuscript_before.md')
    for path, expected in {**c['inputs_sha256'], **docs['documents_sha256']}.items():
        assert digest(ROOT / path) == expected, path
    for path, expected in run['outputs_sha256'].items():
        assert digest(directory / path) == expected, path
    directory = ROOT / 'Modeling/tables/regime_diagnosis'
    run = json.loads((directory / 'run.json').read_text(encoding='utf-8'))
    c = json.loads((directory / 'contract.json').read_text(encoding='utf-8'))
    verified = json.loads((directory / 'independent_verification.json').read_text(encoding='utf-8'))
    assert run['status'] == 'completed' and verified['status'] == 'passed'
    assert run['contract_sha256'] == digest(directory / 'contract.json')
    assert verified['run_sha256'] == digest(directory / 'run.json')
    assert verified['verifier_sha256'] == digest(SCRIPTS / 'regime_diagnose_verify.py')
    assert c['entrypoint_sha256'] == digest(SCRIPTS / 'regime_diagnose.py')
    assert c['source_sha256'] == digest(ROOT / contract['source'])
    assert c['probe_sha256'] == digest(directory / 'probe.json')
    for path, expected in {**run['outputs_sha256'], **verified['extra_outputs_sha256']}.items():
        assert digest(directory / path) == expected, path
    directory = ROOT / 'Modeling/tables/regime_forecast'
    run = json.loads((directory / 'run.json').read_text(encoding='utf-8'))
    c = json.loads((directory / 'contract.json').read_text(encoding='utf-8'))
    verified = json.loads((directory / 'independent_verification.json').read_text(encoding='utf-8'))
    assert run['status'] == 'completed' and verified['status'] == 'passed'
    assert run['contract_sha256'] == digest(directory / 'contract.json')
    assert verified['run_sha256'] == digest(directory / 'run.json')
    assert verified['verifier_sha256'] == digest(SCRIPTS / 'regime_forecast_verify.py')
    assert c['entrypoint_sha256'] == run['script_sha256'] == digest(SCRIPTS / 'regime_forecast.py')
    for path, expected in {**c['inputs_sha256'], **run['inputs_sha256']}.items():
        assert digest(ROOT / path) == expected, path
    for path, expected in {**run['outputs_sha256'], **verified['extra_outputs_sha256']}.items():
        assert digest(directory / path) == expected, path
    import csv
    with (directory / 'model_manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
        records = list(csv.DictReader(stream))
    assert len(records) == run['new_fits'] == verified['models_reloaded'] == 16
    for record in records:
        assert digest(ROOT / record['file']) == record['sha256'], record['file']
    for folder, script, verifier in [('regime_integration', 'regime_integrate.py', 'regime_integrate_verify.py'),
                                      ('regime_routing', 'regime_route.py', 'regime_route_verify.py')]:
        directory = ROOT / 'Modeling/tables' / folder
        run = json.loads((directory / 'run.json').read_text(encoding='utf-8'))
        c = json.loads((directory / 'contract.json').read_text(encoding='utf-8'))
        verified = json.loads((directory / 'independent_verification.json').read_text(encoding='utf-8'))
        assert run['status'] == 'completed' and verified['status'] == 'passed'
        assert run['contract_sha256'] == digest(directory / 'contract.json')
        assert verified['run_sha256'] == digest(directory / 'run.json')
        assert verified['verifier_sha256'] == digest(SCRIPTS / verifier)
        assert c['entrypoint_sha256'] == digest(SCRIPTS / script)
        for path, expected in c['inputs_sha256'].items():
            assert digest(ROOT / path) == expected, path
        for path, expected in {**run['outputs_sha256'], **verified.get('extra_outputs_sha256', {})}.items():
            assert digest(directory / path) == expected, path
        if folder == 'regime_integration':
            with (directory / 'model_manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
                records = list(csv.DictReader(stream))
            assert len(records) == run['new_regression_fits'] == verified['models_reloaded'] == 15
            for record in records:
                assert digest(ROOT / record['file']) == record['sha256'], record['file']
        else:
            assert run['new_fits'] == verified['new_fits'] == 0
    directory = ROOT / 'Modeling/tables/regime_followup'
    run = json.loads((directory / 'run.json').read_text(encoding='utf-8'))
    c = json.loads((directory / 'contract.json').read_text(encoding='utf-8'))
    verified = json.loads((directory / 'independent_verification.json').read_text(encoding='utf-8'))
    assert run['status'] == 'completed' and verified['status'] == 'passed'
    assert run['contract_sha256'] == digest(directory / 'contract.json')
    assert c['entrypoint_sha256'] == run['script_sha256'] == digest(SCRIPTS / 'regime_followup.py')
    assert verified['run_sha256'] == digest(directory / 'run.json')
    assert verified['verifier_sha256'] == digest(SCRIPTS / 'regime_followup_verify.py')
    for path, expected in c['inputs_sha256'].items():
        assert digest(ROOT / path) == expected, path
    for path, expected in {**run['outputs_sha256'], **verified['extra_outputs_sha256']}.items():
        assert digest(directory / path) == expected, path
    with (directory / 'model_manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
        records = list(csv.DictReader(stream))
    assert len(records) == run['new_fits'] == verified['models_reloaded'] == 3
    for record in records:
        assert digest(ROOT / record['file']) == record['sha256'], record['file']
    directory = ROOT / 'Modeling/tables/regime_age_ablation'
    c = json.loads((directory / 'contract.json').read_text(encoding='utf-8'))
    run = json.loads((directory / 'run.json').read_text(encoding='utf-8'))
    verified = json.loads((directory / 'independent_verification.json').read_text(encoding='utf-8'))
    assert run['status'] == 'completed' and verified['status'] == 'passed'
    assert c['entrypoint_sha256'] == run['script_sha256'] == digest(SCRIPTS / 'regime_age_ablation.py')
    assert run['contract_sha256'] == digest(directory / 'contract.json')
    assert verified['run_sha256'] == digest(directory / 'run.json')
    assert verified['verifier_sha256'] == digest(SCRIPTS / 'regime_age_ablation_verify.py')
    for path, expected in c['inputs_sha256'].items():
        assert digest(ROOT / path) == expected, path
    for path, expected in run['outputs_sha256'].items():
        assert digest(directory / path) == expected, path
    with (directory / 'model_manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
        records = list(csv.DictReader(stream))
    assert len(records) == run['new_fits'] == verified['models_reloaded'] == verified['independent_classifier_refits']
    assert run['new_regression_fits'] == verified['new_regression_fits'] == 0
    for record in records:
        assert digest(ROOT / record['file']) == record['sha256'], record['file']
    if (directory / 'calendar_control_run.json').exists():
        diagnostic = json.loads((directory / 'calendar_control_run.json').read_text(encoding='utf-8'))
        dc = json.loads((directory / 'calendar_control_contract.json').read_text(encoding='utf-8'))
        assert diagnostic['status'] == 'completed' and diagnostic['scalar_routing_check'] == 'passed'
        assert dc['entrypoint_sha256'] == diagnostic['script_sha256'] == digest(SCRIPTS / 'regime_age_calendar_control.py')
        assert diagnostic['contract_sha256'] == digest(directory / 'calendar_control_contract.json')
        for path, expected in {**dc['inputs_sha256'], **diagnostic['outputs_sha256']}.items():
            assert digest(directory / path) == expected, path
    directory = ROOT / 'Modeling/tables/regime_manuscript'
    docs = json.loads((directory / 'document_verification.json').read_text(encoding='utf-8'))
    assert docs['status'] == 'passed' and docs['historical_tables_preserved']
    assert docs['script_sha256'] == digest(SCRIPTS / 'regime_manuscript_documents.py')
    assert docs['original_manuscript_sha256'] == digest(directory / 'manuscript_before.md')
    for path, expected in {**docs['source_sha256'], **docs['documents_sha256']}.items():
        assert digest(ROOT / path) == expected, path
    directory = ROOT / 'Modeling/tables/modeling_close'
    closed = json.loads((directory / 'verification.json').read_text(encoding='utf-8'))
    assert closed['status'] == 'passed' and closed['visual_review'] == 'passed'
    assert closed['script_sha256'] == digest(SCRIPTS / 'modeling_close_verify.py')
    assert closed['run_sha256'] == digest(directory / 'run.json')
    assert closed['documents_sha256'] == digest(directory / 'documents.json')
    assert closed['figures_manifest_sha256'] == digest(directory / 'figures.json')
    for folder in ['modeling_close', 'modeling_replay']:
        d = ROOT / 'Modeling/tables' / folder
        r = json.loads((d / 'run.json').read_text(encoding='utf-8'))
        assert r['status'] == 'passed' and r['script_sha256'] == digest(SCRIPTS / 'modeling_close_analysis.py')
        assert r['contract_sha256'] == digest(d / 'contract.json')
        for path, expected in r['inputs_sha256'].items():
            assert digest(ROOT / path) == expected, path
        for path, expected in r['outputs_sha256'].items():
            assert digest(d / path) == expected, path
    for file, field in [('documents.json', 'outputs_sha256'), ('figures.json', 'figures_sha256')]:
        record = json.loads((directory / file).read_text(encoding='utf-8'))
        for path, expected in record[field].items():
            assert digest(ROOT / path) == expected, path
    return contract['source']


def main():
    parser = argparse.ArgumentParser()
    group = parser.add_mutually_exclusive_group(required=True)
    group.add_argument('--record', action='store_true')
    group.add_argument('--verify', action='store_true')
    args = parser.parse_args()
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    modules = {Path(name).stem for name in SCRIPT_NAMES}
    manuscript_path = ROOT / 'Modeling/04_Modeling_원고.md'
    manuscript = manuscript_path.read_text(encoding='utf-8')
    assert manuscript.count('## 4.5 제출 코드와 재현 파일의 보존') == 1
    for name in SCRIPT_NAMES:
        path = SCRIPTS / name
        assert path.is_file(), path
        reference_text = manuscript if name in MANUSCRIPT_SCRIPT_NAMES else (ROOT / 'Modeling/README.md').read_text(encoding='utf-8')
        assert f'(scripts/{name})' in reference_text, f'Code missing from manuscript or research README: {name}'
    all_local = {p.stem for p in SCRIPTS.glob('*.py')}
    dependencies = {name: local_imports(SCRIPTS / name, all_local) for name in SCRIPT_NAMES}
    for name, imports in dependencies.items():
        assert set(imports) <= modules, (name, imports)
    linked_files = set()
    for target in re.findall(r'!?\[[^\]]*\]\(([^)]+)\)', manuscript):
        if target.startswith(('http:', 'https:', '#', 'app:')):
            continue
        path = (manuscript_path.parent / target.split('#')[0].strip('<>')).resolve()
        if args.record and path in {MANIFEST.resolve(), (ROOT / 'Modeling/requirements.txt').resolve()}:
            continue
        assert path.exists(), target
        if path.is_file():
            linked_files.add(path)
    if args.verify:
        current = json.loads(MANIFEST.read_text(encoding='utf-8'))
        assert current['script_names'] == SCRIPT_NAMES
        assert current['local_import_dependencies'] == dependencies
        for record in current['files']:
            path = ROOT / record['path']
            assert path.is_file(), f'Missing preserved file: {record["path"]}'
            assert path.stat().st_size == record['bytes'] and digest(path) == record['sha256'], record['path']
        source_and_result_checks()
        print(json.dumps({'status': 'passed', 'mode': 'verify', 'scripts': len(SCRIPT_NAMES),
                          'preserved_files': len(current['files']), 'no_training_or_deletion': True}), flush=True)
        return
    source = source_and_result_checks()
    packages = {name: importlib.metadata.version(name) for name in PACKAGES}
    requirements = ROOT / 'Modeling/requirements.txt'
    requirements.write_text('# Direct libraries for regular CPython 3.13; verified installed versions.\n' +
                            '\n'.join(f'{name}=={version}' for name, version in packages.items()) + '\n', encoding='utf-8')
    keep = set(linked_files)
    keep.add(ROOT / 'EDA/scripts/p01_peak_shapes.py')
    for extra in ['EDA/tables/p01_peak_shapes', 'Analysis/tables/p02_profile_history']:
        keep.update(p for p in (ROOT / extra).rglob('*') if p.is_file())
    keep.update(SCRIPTS / name for name in SCRIPT_NAMES)
    keep.update(ROOT / 'Modeling/config' / name for name in CONFIG_NAMES)
    keep.update(ROOT / name for name in DOC_NAMES)
    keep.update([ROOT / source, requirements, ROOT / 'Modeling/tables/m022/environment.json'])
    for folder in TABLE_DIRS:
        directory = ROOT / 'Modeling/tables' / folder
        assert directory.is_dir(), folder
        keep.update(p for p in directory.rglob('*') if p.is_file())
    for folder in MODEL_DIRS:
        directory = ROOT / 'Modeling/models' / folder
        assert directory.is_dir(), folder
        keep.update(p for p in directory.rglob('*') if p.is_file())
    keep.update(p for p in (ROOT / 'Modeling/figures').rglob('*') if p.is_file())
    keep.discard(MANIFEST)
    records = []
    for path in sorted(keep):
        assert path.is_file(), path
        path = path.resolve()
        relative = path.relative_to(ROOT.resolve()).as_posix()
        records.append({'path': relative, 'bytes': path.stat().st_size, 'sha256': digest(path)})
    manifest = {'scope': 'Current Modeling manuscript M01/M02/M03/expanded M02/M04 and its direct references; not a whole-project deletion authorization',
                'recorded_at': datetime.now(timezone(timedelta(hours=9))).isoformat(timespec='minutes'),
                'self': 'Modeling/submission_manifest.json', 'script_names': SCRIPT_NAMES,
                'manuscript_script_names': MANUSCRIPT_SCRIPT_NAMES, 'research_script_names': RESEARCH_SCRIPT_NAMES,
                'external_research_scripts': ['EDA/scripts/p01_peak_shapes.py'],
                'local_import_dependencies': dependencies, 'runtime': {'executable': sys.executable, 'version': sys.version, 'packages': packages},
                'requirements': 'Modeling/requirements.txt', 'files': records,
                'not_final_submission': True, 'clean_environment_reproduction_executed': False,
                'no_training_or_deletion': True}
    MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps({'status': 'recorded', 'scripts': len(SCRIPT_NAMES), 'preserved_files': len(records),
                      'direct_packages': len(packages), 'no_training_or_deletion': True}), flush=True)


if __name__ == '__main__':
    main()


코드 등록 완료: `Modeling/scripts/modeling_submission_audit.py`

### `Modeling/scripts/plot_m02_manuscript.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Manuscript figures from verified M02 tables; no fitting or new selection.

주요 함수: `load_verified()`, `plot_comparison()`, `plot_conditions()`, `main()`.

In [70]:
%%submission_source Modeling/scripts/plot_m02_manuscript.py
"""Manuscript figures from verified M02 tables; no fitting or new selection."""
import hashlib
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path(__file__).resolve().parents[2]
TABLES = ROOT / "Modeling/tables/m02"
OUT = ROOT / "Modeling/figures/m02"
LABELS = {"previous_hour": "직전 시간", "previous_day": "전날", "previous_week": "전주",
          "ridge_00": "Ridge", "elasticnet_04": "Elastic Net", "svr_03": "SVR", "hgb_01": "HGB"}


def load_verified():
    run = json.loads((TABLES / "run.json").read_text(encoding="utf-8"))
    check = json.loads((TABLES / "independent_verification.json").read_text(encoding="utf-8"))
    assert run["status"] == "completed" and check["status"] == "passed"
    for name in ["comparison.csv", "condition_errors.csv"]:
        assert hashlib.sha256((TABLES / name).read_bytes()).hexdigest() == run["outputs_sha256"][name]
    comparison = pd.read_csv(TABLES / "comparison.csv", encoding="utf-8-sig")
    conditions = pd.read_csv(TABLES / "condition_errors.csv", encoding="utf-8-sig")
    return comparison.loc[comparison.target.eq("target_maximum")], conditions.loc[conditions.target.eq("target_maximum")]


def plot_comparison(comparison):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    overall = comparison.loc[comparison.split.eq("pooled_development")].set_index("model").loc[list(LABELS)]
    colors = ["#a7b2ba"] * 6 + ["#2463a0"]
    bars = axes[0].barh(list(LABELS.values()), overall.MAE, color=colors)
    axes[0].invert_yaxis()
    axes[0].set_xlim(0, 49)
    axes[0].set_xlabel("최대전력 예측 MAE")
    for bar, value in zip(bars, overall.MAE):
        axes[0].text(value + .5, bar.get_y() + bar.get_height()/2, f"{value:.2f}", va="center", fontsize=9)
    for model, color, marker in [("previous_hour", "#767f86", "o"), ("previous_week", "#b55432", "s"), ("hgb_01", "#2463a0", "o")]:
        values = [float(comparison.loc[comparison.model.eq(model) & comparison.split.eq(split), "MAE"].iloc[0]) for split in ["dev_apr", "dev_may", "dev_jun"]]
        axes[1].plot([4, 5, 6], values, marker=marker, color=color, label=LABELS[model])
        for month, value in zip([4, 5, 6], values):
            axes[1].annotate(f"{value:.2f}", (month, value), xytext=(0, 7), textcoords="offset points", ha="center", fontsize=9, color=color)
    axes[1].set_xticks([4, 5, 6], ["4월", "5월", "6월"])
    axes[1].set_ylim(0, 31)
    axes[1].set_ylabel("최대전력 예측 MAE")
    axes[1].legend(frameon=False, loc="upper right")
    for ax in axes:
        ax.grid(axis="x" if ax is axes[0] else "y", alpha=.2)
        ax.set_axisbelow(True)
    fig.tight_layout()
    path = OUT / "model_comparison.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


def plot_conditions(conditions):
    wanted = [("low->above26", "낮은 구간→26 초과", 20), ("above26->low", "26 초과→낮은 구간", 19),
              ("low->low", "낮은 구간 유지", 625), ("above26->above26", "26 초과 유지", 1510)]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})
    y = np.arange(4)
    for model, shift, color in [("previous_hour", -.18, "#a7b2ba"), ("hgb_01", .18, "#2463a0")]:
        values = [float(conditions.loc[conditions.model.eq(model) & conditions.condition.eq("power_transition") & conditions.value.eq(state), "MAE"].iloc[0]) for state, _, _ in wanted]
        bars = axes[0].barh(y + shift, values, height=.34, label=LABELS[model], color=color)
        for bar, value in zip(bars, values):
            axes[0].text(value+.8, bar.get_y()+bar.get_height()/2, f"{value:.2f}", va="center", fontsize=9)
    axes[0].set_yticks(y, [f"{label} ({n}시간)" for _, label, n in wanted])
    axes[0].invert_yaxis()
    axes[0].set_xlim(0, 89)
    axes[0].set_xlabel("최대전력 예측 MAE")
    axes[0].legend(frameon=False, loc="lower right")
    peak = conditions.loc[conditions.condition.eq("daily_maximum")].set_index("model")
    models = ["previous_hour", "previous_week", "hgb_01"]
    under = peak.loc[models, "mean_under"].to_numpy()
    over = peak.loc[models, "mean_over"].to_numpy()
    axes[1].bar(np.arange(3), under, label="평균 과소예측량", color="#2463a0")
    axes[1].bar(np.arange(3), over, bottom=under, label="평균 과대예측량", color="#b55432")
    for i, (u, o) in enumerate(zip(under, over)):
        axes[1].text(i, u+o+.8, f"{u+o:.2f}", ha="center", fontsize=9)
    axes[1].set_xticks(np.arange(3), [LABELS[m] for m in models])
    axes[1].set_ylabel("실제 일별 최대 시간의 예측오차")
    axes[1].set_ylim(0, 45)
    axes[1].legend(frameon=False, loc="upper right", fontsize=9)
    for ax in axes:
        ax.grid(axis="x" if ax is axes[0] else "y", alpha=.2)
        ax.set_axisbelow(True)
    fig.tight_layout()
    path = OUT / "condition_errors.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    plt.rcParams.update({"font.family": "Malgun Gothic", "axes.unicode_minus": False, "font.size": 10})
    comparison, conditions = load_verified()
    paths = [plot_comparison(comparison), plot_conditions(conditions)]
    qa = ROOT / "tmp/modeling_visual_review"
    qa.mkdir(parents=True, exist_ok=True)
    previews = []
    for path in paths:
        with Image.open(path) as im:
            previews.append(im.convert("RGB").resize((im.width//2, im.height//2), Image.Resampling.LANCZOS))
    sheet = Image.new("RGB", (max(im.width for im in previews), sum(im.height for im in previews)), "white")
    y = 0
    for im in previews:
        sheet.paste(im, (0, y))
        y += im.height
    sheet.save(qa / "m02_contact_50.png")
    result = {"status": "generated", "visual_review": "pending", "input_sha256": {name: hashlib.sha256((TABLES/name).read_bytes()).hexdigest() for name in ["comparison.csv", "condition_errors.csv"]},
              "figures": {str(path.relative_to(ROOT)): hashlib.sha256(path.read_bytes()).hexdigest() for path in paths}, "no_fitting": True}
    (TABLES / "manuscript_figures.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result, ensure_ascii=False))


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/plot_m02_manuscript.py`

### `Modeling/scripts/plot_m03_manuscript.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: M03 manuscript figure from independently verified C results, without fitting.

주요 함수: `sha()`, `load_verified()`, `plot_ablation()`, `main()`.

In [71]:
%%submission_source Modeling/scripts/plot_m03_manuscript.py
"""M03 manuscript figure from independently verified C results, without fitting."""
import hashlib
import json
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path(__file__).resolve().parents[2]
TABLES = ROOT / "Modeling/tables/m03/c"
OUT = ROOT / "Modeling/figures/m03"
COLORS = {"A": "#919ba4", "B": "#2463a0", "C_A": "#c69b73", "C_B": "#398a70"}


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def load_verified():
    run = json.loads((TABLES / "run.json").read_text(encoding="utf-8"))
    verification = json.loads((TABLES / "independent_verification.json").read_text(encoding="utf-8"))
    assert run["status"] == "completed" and verification["status"] == "passed"
    assert verification["run_sha256"] == sha(TABLES / "run.json")
    for name in ["comparison.csv", "condition_errors.csv"]:
        assert sha(TABLES / name) == run["outputs_sha256"][name]
    return [pd.read_csv(TABLES / name, encoding="utf-8-sig").query("target == 'target_maximum'") for name in ["comparison.csv", "condition_errors.csv"]]


def plot_ablation(comparison, conditions):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for group, color in COLORS.items():
        part = comparison.loc[comparison.group.eq(group)].set_index("split")
        values = [float(part.loc[s, "MAE"]) for s in ["dev_apr", "dev_may", "dev_jun"]]
        axes[0].plot([4, 5, 6], values, marker="o", color=color, label=group,
                     linestyle="--" if group.startswith("C_") else "-")
    axes[0].set_xticks([4, 5, 6], ["4월", "5월", "6월"])
    axes[0].set_ylim(0, 11)
    axes[0].set_ylabel("최대전력 예측 MAE")
    axes[0].legend(frameon=False, ncol=2)
    peak = conditions.loc[conditions.condition.eq("daily_maximum")].set_index("group").loc[list(COLORS)]
    y = np.arange(4)
    axes[1].bar(y, peak.mean_under, color="#2463a0", label="평균 과소예측량")
    axes[1].bar(y, peak.mean_over, bottom=peak.mean_under, color="#b55432", label="평균 과대예측량")
    for i, (u, o) in enumerate(zip(peak.mean_under, peak.mean_over)):
        axes[1].text(i, u/2, f"{u:.2f}", ha="center", va="center", color="white", fontsize=10)
        axes[1].text(i, u+o+.25, f"{u+o:.2f}", ha="center", fontsize=9)
    axes[1].set_xticks(y, list(COLORS))
    axes[1].set_ylim(0, 15)
    axes[1].set_ylabel("실제 일별 최대 시간의 예측오차")
    axes[1].legend(frameon=False, loc="upper right", fontsize=9)
    for ax in axes:
        ax.grid(axis="y", alpha=.2)
        ax.set_axisbelow(True)
    fig.tight_layout()
    path = OUT / "last_slot_ablation.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    plt.rcParams.update({"font.family": "Malgun Gothic", "axes.unicode_minus": False, "font.size": 10})
    comparison, conditions = load_verified()
    path = plot_ablation(comparison, conditions)
    preview = ROOT / "tmp/modeling_visual_review/m03_contact_50.png"
    preview.parent.mkdir(parents=True, exist_ok=True)
    with Image.open(path) as im:
        im.resize((im.width//2, im.height//2), Image.Resampling.LANCZOS).save(preview)
    result = {"status": "generated", "visual_review": "pending", "no_fitting": True,
              "input_sha256": {name: sha(TABLES/name) for name in ["comparison.csv", "condition_errors.csv"]},
              "figures": {str(path.relative_to(ROOT)): sha(path)}, "script_sha256": sha(Path(__file__))}
    (TABLES / "manuscript_figures.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result, ensure_ascii=False), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/plot_m03_manuscript.py`

### `Modeling/scripts/plot_observed_power_timeline.py`

원고 연결: EDA\02_EDA_3페이지용_압축본.md; EDA\02_EDA_원고.md; EDA\02_EDA_원고.md; EDA\02_EDA_원고.md; Analysis\03_Analysis_원고.md; Analysis\03_Analysis_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Observed timeline from verified M01 rows; no forecasts or event thresholds.

주요 함수: `main()`.

In [72]:
%%submission_source Modeling/scripts/plot_observed_power_timeline.py
"""Observed timeline from verified M01 rows; no forecasts or event thresholds."""
import hashlib
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

ROOT = Path(__file__).resolve().parents[2]
SOURCE = ROOT / "Modeling/tables/m01/hourly_frame.csv"
OUT = ROOT / "Modeling/figures"
TABLES = ROOT / "Modeling/tables/observed_timeline"


def main():
    expected = json.loads((SOURCE.parent / "verification.json").read_text(encoding="utf-8"))
    digest = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
    assert digest == expected["outputs_sha256"][SOURCE.name]
    data = pd.read_csv(SOURCE, encoding="utf-8-sig", parse_dates=["timestamp"], float_precision="round_trip")
    assert len(data) == 5784
    data["maximum_delta"] = data.target_maximum - data.lag1_maximum
    data["mean_delta"] = data.target_mean - data.lag1_mean
    zero = data.loc[data.target_maximum == 0]
    assert len(zero) == 17
    assert (zero.timestamp.diff().dropna() == pd.Timedelta(hours=1)).all()
    first, last = zero.timestamp.min(), zero.timestamp.max()
    assert data.maximum_delta.notna().sum() == 5781
    OUT.mkdir(parents=True, exist_ok=True)
    TABLES.mkdir(parents=True, exist_ok=True)
    plt.rcParams.update({"font.family": "Malgun Gothic", "axes.unicode_minus": False, "font.size": 10})
    # Reindex gaps to NaN: excluded dates never become a fabricated connecting line.
    indexed = data.set_index("timestamp")
    timeline = indexed.reindex(pd.date_range(indexed.index.min(), indexed.index.max(), freq="h"))
    figures = []
    for kind, cols, ylabel, filename in (
        ("level", ["target_mean", "target_maximum"], "전력값", "observed_power_monthly_2021_01_08.png"),
        ("delta", ["mean_delta", "maximum_delta"], "직전 시간 대비 차이", "observed_power_delta_monthly_2021_01_08.png"),
    ):
        fig, axes = plt.subplots(4, 2, figsize=(16, 11), sharey=True)
        for month, ax in enumerate(axes.flat, 1):
            part = timeline.loc[timeline.index.month == month]
            for col, color, label in zip(cols, ["#2463a0", "#b55432"],
                                         ["시간 평균", "시간 최대"] if kind == "level" else ["평균 변화", "최대값 변화"]):
                ax.plot(part.index, part[col], linewidth=0.75, color=color, alpha=0.85, label=label)
            if kind == "delta":
                ax.axhline(0, color="#555555", linewidth=0.5)
                ax.set_ylim(-230, 230)
            else:
                ax.set_ylim(-5, 235)
            ax.set_title(f"2021년 {month}월", loc="left", fontsize=11)
            ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
            ax.grid(axis="y", alpha=0.2)
            if month % 2 == 1:
                ax.set_ylabel(ylabel)
        axes[0, 0].legend(frameon=False, ncol=2, fontsize=9, loc="upper right")
        fig.tight_layout()
        path = OUT / filename
        fig.savefig(path, dpi=150)
        plt.close(fig)
        figures.append(path)
    zoom = timeline.loc[first - pd.Timedelta(hours=24):last + pd.Timedelta(hours=24)]
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    for ax, cols, ylabel in zip(axes, [["target_mean", "target_maximum"], ["mean_delta", "maximum_delta"]],
                                ["전력값", "직전 시간 대비 차이"]):
        for col, color, label in zip(cols, ["#2463a0", "#b55432"], ["평균", "최대"]):
            ax.plot(zoom.index, zoom[col], linewidth=1.5, marker=".", markersize=3, color=color, label=label)
        ax.axvspan(first, last + pd.Timedelta(hours=1), color="#999999", alpha=0.15, label="전력 0인 17시간")
        ax.grid(axis="y", alpha=0.2)
        ax.set_ylabel(ylabel)
        ax.legend(frameon=False, ncol=3, fontsize=9, loc="upper left")
    axes[0].set_ylim(-5, 240)
    axes[1].axhline(0, color="#555555", linewidth=0.6)
    axes[1].xaxis.set_major_locator(mdates.HourLocator(interval=12))
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H시"))
    axes[1].set_xlabel("관측 시각")
    fig.tight_layout()
    path = OUT / "observed_power_zero_interval.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    figures.append(path)
    cols = ["timestamp", "target_mean", "target_maximum", "lag1_mean", "lag1_maximum", "mean_delta", "maximum_delta"]
    pd.concat([data.nsmallest(5, "maximum_delta").assign(direction="largest_drop"),
               data.nlargest(5, "maximum_delta").assign(direction="largest_rise")])[cols + ["direction"]].to_csv(
                   TABLES / "largest_changes.csv", index=False, encoding="utf-8-sig")
    boundary = data.loc[data.timestamp.between(first - pd.Timedelta(hours=1), last + pd.Timedelta(hours=1)), cols]
    boundary.to_csv(TABLES / "zero_interval.csv", index=False, encoding="utf-8-sig")
    summary = {"source_frame_sha256": digest, "rows": len(data), "exact_hour_changes": 5781,
               "zero_hours": len(zero), "zero_start": str(first), "zero_end": str(last),
               "before_zero": boundary.iloc[0].astype(str).to_dict(), "after_zero": boundary.iloc[-1].astype(str).to_dict(),
               "maximum_drop": float(data.maximum_delta.min()), "maximum_rise": float(data.maximum_delta.max()),
               "figures": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in figures},
               "gaps_not_connected": True, "no_smoothing": True, "no_model_fitting": True,
               "visual_review": "pending"}
    (TABLES / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    # Authorized visual review: contact sheet at exactly 50 percent, at most four images.
    previews = []
    for p in figures:
        with Image.open(p) as img:
            previews.append(img.resize((img.width // 2, img.height // 2)))
    sheet = Image.new("RGB", (max(im.width for im in previews), sum(im.height for im in previews)), "white")
    y = 0
    for im in previews:
        sheet.paste(im, (0, y))
        y += im.height
    sheet.save(TABLES / "contact_sheet_50.png")
    print(json.dumps({key: summary[key] for key in ("zero_hours", "zero_start", "zero_end", "before_zero", "after_zero", "maximum_drop", "maximum_rise")}, ensure_ascii=False))


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/plot_observed_power_timeline.py`

### `Modeling/scripts/priority_review.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Replay a fixed daily review budget using current/past scores only.
This is exploratory development evidence, not measured plant intervention.

주요 함수: `sha()`, `dump()`, `read()`, `freeze()`, `replay()`, `summary()`, `sensitivity()`, `run()`.

In [73]:
%%submission_source Modeling/scripts/priority_review.py
"""Replay a fixed daily review budget using current/past scores only.
This is exploratory development evidence, not measured plant intervention.
"""
import argparse
import hashlib
import json
import sys
from datetime import datetime, timezone, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
PARENT = ROOT / "Modeling/tables/error_warning"
OUT = ROOT / "Modeling/tables/priority_review"
METHODS = ["high_prediction", "prior_maximum", "neighbor_std", "selected_simple", "selected_ML"]

def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def dump(x, p):
    Path(p).write_text(json.dumps(x, ensure_ascii=False, indent=2), encoding="utf-8")

def read(p):
    return pd.read_csv(p, encoding="utf-8-sig", float_precision="round_trip",
                       parse_dates=["timestamp", "date"])

def freeze():
    assert not (OUT / "contract.json").exists()
    verified = json.loads((PARENT / "independent_verification.json").read_text(encoding="utf-8"))
    assert verified["status"] == "passed"
    inputs = [PARENT / n for n in ["predictions.csv", "calibration_predictions.csv",
              "selection.json", "contract.json", "run.json", "independent_verification.json"]]
    OUT.mkdir(parents=True, exist_ok=True)
    contract = {
        "recorded_at": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
        "entrypoint_sha256": sha(__file__),
        "inputs_sha256": {p.relative_to(ROOT).as_posix(): sha(p) for p in inputs},
        "scope": "April-June 2021 reused development, not independent final test; no later-month evaluation",
        "target": "Existing February-derived large-underprediction event >=9.73851536905958",
        "daily_budget": 2,
        "schedule": "24 hourly origins 00..23 on each date; planned availability, not future score/outcome",
        "policy": "At hour h, b reviews remain and r=24-h origins remain. If b=0 skip. If b=r select to use quota. Otherwise select iff current score > previous-calendar-month score quantile 1-b/r (linear interpolation). Reset b=2 each date. No within-evaluation-month outcome or score update.",
        "calibration": "Use original prior-month calibration scores from same frozen model; model selection remains frozen from parent AP, not reselected for new policy",
        "methods": METHODS,
        "primary_reference": "selected_simple frozen in parent study",
        "strong_baseline_check": "compare pooled TP against max of all three fixed simple rules and selected_simple; this is a conservative hurdle, not deployed future selection",
        "reportworthy": "selected_ML pooled TP >= strongest simple TP+5 and >=1.20*strongest simple TP, and TP strictly exceeds selected_simple in at least 2 of 3 months. No all-month-win requirement. Report all monthly and severity results even if mixed.",
        "secondary": "precision, recall, non-event reviews, missed events, captured underprediction amount, forced-end-of-day reviews, hourly allocation",
        "uncertainty": "paired calendar-week cluster bootstrap, 2000 resamples, seed42, descriptive sensitivity only given reused development/repeated arrays; not selection gate",
        "fits": 0,
        "no_policy_search": True,
        "interpretation": "Online replay of choosing which forecast to review, not correction, automatic plant action, cost saving, or forecasting the event after its outcome",
        "report_scope": "If reportworthy preserve original then add bounded detailed manuscript result; otherwise keep research README only. No compressed manuscript edits."
    }
    dump(contract, OUT / "contract.json")
    print("Priority review contract frozen before replay.", flush=True)

def replay(scores, calibration, timestamps, budget=2):
    """Only scalars already available at each origin enter the decision."""
    remaining = budget
    previous_date = None
    rows = []
    for t, score in zip(timestamps, scores):
        date = t.date()
        if date != previous_date:
            if previous_date is not None:
                assert remaining == 0
            remaining = budget
            previous_date = date
        r = 24 - t.hour
        b = remaining
        if not b:
            threshold = np.nan
            chosen, forced = False, False
        elif b == r:
            threshold = np.nan
            chosen, forced = True, True
        else:
            threshold = float(np.quantile(calibration, 1 - b / r, method="linear"))
            chosen, forced = bool(score > threshold), False
        if chosen:
            remaining -= 1
        rows.append({"remaining_before": b, "hours_remaining": r, "boundary": threshold,
                     "review": chosen, "forced": forced})
    assert remaining == 0
    return pd.DataFrame(rows)

def summary(g):
    reviewed = g.review.to_numpy(bool)
    event = g.event.to_numpy(bool)
    under = g.under_amount.to_numpy()
    tp = int((reviewed & event).sum())
    n = int(event.sum())
    reviews = int(reviewed.sum())
    return {"hours": len(g), "days": g.date.nunique(), "events": n,
            "reviews": reviews, "TP": tp, "non_event_reviews": reviews-tp, "FN": n-tp,
            "precision": tp/reviews, "recall": tp/n if n else 0.,
            "review_fraction": reviews/len(g),
            "captured_under": float(under[reviewed].sum()),
            "missed_under": float(under[~reviewed].sum()),
            "captured_under_share": float(under[reviewed].sum()/under.sum()),
            "forced_reviews": int(g.forced.sum()),
            "forced_TP": int((g.forced & g.event).sum())}

def sensitivity(daily):
    """Preserve within-week pairing; do not claim independent-date inference."""
    d = daily.copy()
    d["week"] = pd.to_datetime(d.date).dt.to_period("W-SUN").astype(str)
    rows = []
    for ref in METHODS[:-1]:
        target = d[d.method.eq("selected_ML")].set_index("date")
        base = d[d.method.eq(ref)].set_index("date")
        diff = pd.DataFrame({"week": target.week,
                             "TP_difference": target.TP-base.TP,
                             "under_difference": target.captured_under-base.captured_under})
        clusters = diff.groupby("week")[["TP_difference","under_difference"]].sum().to_numpy()
        rng = np.random.default_rng(42)
        samples = clusters[rng.integers(0,len(clusters),size=(2000,len(clusters)))].sum(axis=1)
        rows.append({"reference": ref, "weeks": len(clusters),
            "TP_difference": int(diff.TP_difference.sum()),
            "TP_low": float(np.quantile(samples[:,0],.025)),
            "TP_high": float(np.quantile(samples[:,0],.975)),
            "under_difference": float(diff.under_difference.sum()),
            "under_low": float(np.quantile(samples[:,1],.025)),
            "under_high": float(np.quantile(samples[:,1],.975))})
    return pd.DataFrame(rows)

def run():
    assert not (OUT / "run.json").exists()
    c = json.loads((OUT / "contract.json").read_text(encoding="utf-8"))
    assert c["entrypoint_sha256"] == sha(__file__)
    for p, h in c["inputs_sha256"].items():
        assert sha(ROOT / p) == h
    source = read(PARENT / "predictions.csv")
    calibration = read(PARENT / "calibration_predictions.csv")
    selections = {x["month"]: x for x in json.loads((PARENT / "selection.json").read_text(encoding="utf-8"))}
    outputs = []
    for method in METHODS:
        for month in [4,5,6]:
            actual_method = selections[month]["ML" if method == "selected_ML" else "simple"] if method.startswith("selected_") else method
            ev = source[source.method.eq(method) & source.month.eq(month)].sort_values("timestamp").reset_index(drop=True)
            cal = calibration[calibration.method.eq(actual_method) & calibration.outer_month.eq(month)].sort_values("timestamp")
            assert cal.timestamp.max() < ev.timestamp.min()
            assert cal.month.eq(month-1).all()
            assert ev.date.nunique() == {4:30,5:31,6:30}[month]
            for _, day in ev.groupby("date"):
                assert day.timestamp.dt.hour.tolist() == list(range(24))
            result = replay(ev.score.to_numpy(), cal.score.to_numpy(), ev.timestamp, c["daily_budget"])
            z = pd.concat([ev[["timestamp","date","month","method","score","event","under_amount","actual","prediction"]],result],axis=1)
            z["calibration_end"] = cal.timestamp.max()
            z["score_method"] = actual_method
            outputs.append(z)
    predictions = pd.concat(outputs,ignore_index=True)
    daily = pd.DataFrame([{"method":m,"date":str(date.date()),"month":int(g.month.iloc[0]),**summary(g)}
                          for (m,date),g in predictions.groupby(["method","date"])])
    assert daily.reviews.eq(c["daily_budget"]).all()
    metrics = []
    for (m,month),g in predictions.groupby(["method","month"]):
        metrics.append({"method":m,"period":str(month),**summary(g)})
    for m,g in predictions.groupby("method"):
        metrics.append({"method":m,"period":"pooled",**summary(g)})
    metrics = pd.DataFrame(metrics)
    index = metrics.set_index(["method","period"])
    a = index.loc[("selected_ML","pooled")]
    strongest = max(int(index.loc[(m,"pooled"),"TP"]) for m in METHODS[:-1])
    flags = {"at_least_five_more": bool(a.TP >= strongest+5),
             "at_least_twenty_percent_more": bool(a.TP >= strongest*1.20),
             "two_month_wins": sum(int(index.loc[("selected_ML",str(m)),"TP"] >
                                       index.loc[("selected_simple",str(m)),"TP"]) for m in [4,5,6]) >= 2}
    decision = {"reportworthy_development_result": all(flags.values()),"criteria":flags,
                "strongest_simple_TP":strongest,
                "not_automatic_warning_acceptance": True,
                "not_independent_test": True}
    hourly = predictions.groupby(["method",predictions.timestamp.dt.hour.rename("hour")]).agg(
        reviews=("review","sum"),forced=("forced","sum")).reset_index()
    for name,table in [("predictions",predictions),("daily",daily),("metrics",metrics),
                       ("sensitivity",sensitivity(daily)),("hourly_allocation",hourly)]:
        table.to_csv(OUT/(name+".csv"),index=False,encoding="utf-8-sig")
    dump(decision,OUT/"decision.json")
    result = {"status":"completed","contract_sha256":sha(OUT/"contract.json"),"new_fits":0,
              "hours_per_method":2184,"dates":91,"reviews_per_method":182,"events":184,
              "decision":decision,"later_evaluated":False,
              "outputs_sha256":{p.name:sha(p) for p in [OUT/(n+".csv") for n in
                 ["predictions","daily","metrics","sensitivity","hourly_allocation"]]+[OUT/"decision.json"]}}
    dump(result, OUT/"run.json")
    print(metrics[metrics.period.eq("pooled")].to_string(index=False),flush=True)
    print(metrics[metrics.method.isin(["selected_ML","selected_simple"])].to_string(index=False),flush=True)
    print(json.dumps(decision),flush=True)

if __name__ == "__main__":
    assert sys.version_info[:2] == (3,13) and sys._is_gil_enabled()
    parser = argparse.ArgumentParser()
    parser.add_argument("stage", choices=["freeze","run"])
    globals()[parser.parse_args().stage]()

코드 등록 완료: `Modeling/scripts/priority_review.py`

### `Modeling/scripts/priority_review_documents.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Preserve and extend the detailed manuscript with bounded positive evidence.

주요 함수: `sha()`, `section()`, `write()`, `verify()`.

In [74]:
%%submission_source Modeling/scripts/priority_review_documents.py
"""Preserve and extend the detailed manuscript with bounded positive evidence."""
import argparse
import hashlib
import json
import re
from pathlib import Path
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/priority_review"
MANUSCRIPT = ROOT / "Modeling/04_Modeling_원고.md"
HEADING = "## 4.11 하루 2회 검토를 위한 과소예측 우선순위"
README_HEADING = "## 하루 2회 검토 우선순위: 개발 성과 반영"

def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def section():
    metrics = pd.read_csv(OUT/"metrics.csv",encoding="utf-8-sig").set_index(["method","period"])
    sensitivity = pd.read_csv(OUT/"sensitivity.csv",encoding="utf-8-sig").set_index("reference")
    labels = {"high_prediction":"예측값이 높은 시간", "prior_maximum":"직전 최대값이 높은 시간",
              "neighbor_std":"과거 유사 상태의 변동이 큰 시간", "selected_simple":"이전 월에서 선택한 단순 점수",
              "selected_ML":"과소예측 보조 모델"}
    table = ["| 검토 우선순위 점수 | 확인 시간 | 큰 과소예측 선별 | 확인 중 적중 비율 | 전체 사건 재현율 |",
             "|---|---:|---:|---:|---:|"]
    for method in labels:
        r=metrics.loc[(method,"pooled")]
        table.append(f"| {labels[method]} | {int(r.reviews)} | {int(r.TP)} | {r.precision*100:.1f}% | {r.recall*100:.1f}% |")
    monthly = ["| 월 | 높은 예측값 기준 | 유사 상태 변동 기준 | 이전 월 선택 기준 | 보조 모델 |",
               "|---|---:|---:|---:|---:|"]
    for month in [4,5,6]:
        monthly.append("| "+str(month)+"월 | "+" | ".join(str(int(metrics.loc[(m,str(month)),"TP"])) for m in
                    ["high_prediction","neighbor_std","selected_simple","selected_ML"])+" |")
    return HEADING+"""

### 예측오류 분석에서 검토 우선순위로 연결

기존 HGB B는 다음 시간의 최대값을 예측한다. 이후 오류분석에서는 이 예측이 실제보다 크게 낮아질 가능성을 보조 점수로 계산했다. 고정 경고 경계의 성과는 월별로 달랐지만, 사후에 같은 수의 시간을 골랐을 때 보조 점수의 선별력이 높아지는 단서가 있었다. 이에 실제 확인 횟수를 제한하고 미래 점수를 보지 않아도 개선이 남는지 비교했다.

이번 비교의 목표는 **하루 2회라는 동일한 확인량에서 큰 과소예측을 더 많이 선별하는 것**이다. 점 예측값 자체를 수정하거나 급등의 발생 여부를 다시 정의하지 않았다. 정상 예측과 검토 우선순위를 함께 제시하는 보완 방식으로 설계했다.

### 사건 정의와 시간순 보조 점수

큰 과소예측은 실제 다음 시간 최대값이 B 예측보다 9.738515 이상 큰 경우다. 이 경계는 2월의 학습 밖 시간순 예측 오차에서 정한 과소예측량 90분위수로 고정했다. 현장의 위험 한계나 비용 기준은 아니며, 이전의 시간당 증가량 93 이상인 급등과 구분한다. 4 ~ 6월 2,184시간 중 해당 사건은 184건이었다.

보조 모델의 입력은 기존 B의 18개 변수, B 예측값, 과거 유사 상태의 다음 값 표준편차·범위·20번째 이웃 거리·이웃 평균과 B 예측의 차이로 총 23개다. 유사 상태는 해당 평가월보다 앞선 자료에서만 표준화하고 가까운 20개를 선택했다. 보조 모델의 정답도 점 예측 모델의 학습 밖 오차를 사용했다.

4월 평가는 2월 오차로 보조 모델을 학습하고 3월 점수로 모델과 기준을 정했다. 5월은 2 ~ 3월 학습·4월 선택, 6월은 2 ~ 4월 학습·5월 선택으로 확장했다. Logistic과 HGB 중 이전 월 AP가 높은 모델을 선택한 결과 세 달 모두 HGB였다. 높은 B 예측값·직전 최대값·유사 상태 변동의 세 단순 점수와, 이들 중 이전 월 AP로 고른 기준도 함께 비교했다. 이번 선택 규칙 평가에서는 기존 점수와 모델을 그대로 사용했고 신규 학습·튜닝은 하지 않았다.

### 미래 점수를 보지 않는 하루 2회 선택

매시간 다음 시간의 점수가 생성되는 시점에 바로 검토 여부를 정했다. 하루의 24개 예측 시점을 예정된 확인 기회로 두고, 남은 확인 횟수를 b, 현재를 포함한 남은 시간을 r로 표시했다. 이전 달 점수 분포의 1−b/r 분위수를 현재 점수가 넘으면 한 번을 배정했다. 이미 두 번 확인했으면 더 선택하지 않았고, 남은 시간과 확인 횟수가 같으면 남은 시간을 선택했다. 이 규칙으로 모든 방법이 매일 정확히 2회, 91일간 182시간을 선택했다.

예측 시점에는 현재까지 관측한 입력·이전 달 점수·그날 남은 확인 횟수만 사용했다. 아직 오지 않은 시간의 점수나 실제 전력은 사용하지 않았다. 하루 2회와 24시간 확인 가능 조건은 비교를 위한 운영 가정이며, 실제 사업장의 인력·응답 시간으로 검증한 값은 아니다.

결과를 보기 전에 보고서에 남길 개발 성과의 기준을 정했다. 전체 선별 건수가 가장 좋은 단순 기준보다 최소 5건·20% 이상 많고, 이전 월에서 선택한 단순 기준보다 세 달 중 두 달 이상에서 많으면 제한된 기여를 인정하기로 했다. 모든 달에서 이겨야 한다는 조건은 두지 않았으며, 월별 차이와 과소예측량은 함께 제시한다. 이는 자동 경고 후보의 이전 채택 기준을 변경한 것이 아니라 동일 확인량이라는 별도 활용 목적의 비교다.

### 동일 확인량에서 확인한 개선

"""+"\n".join(table)+"""

가장 좋은 단순 기준인 유사 상태 변동 점수는 20건, 보조 모델은 27건을 선별했다. 같은 182시간에서 **7건, 35.0% 증가**했으며 확인 중 적중 비율은 11.0%에서 14.8%로 높아졌다. 높은 예측값만 보는 기준의 12건과 이전 월에서 고른 단순 기준의 14건보다도 많았다. 단순히 예상 전력이 큰 시간을 확인하는 것에 더해, 예측이 크게 낮아질 가능성을 별도 점수로 제공한 기여다. 개별 이웃 변수의 기여를 분리한 실험은 아니므로 특정 입력 하나의 효과로 해석하지 않는다.

월별 확인량은 4월 60회, 5월 62회, 6월 60회로 각 방법에 동일했다.

"""+"\n".join(monthly)+"""

보조 모델은 이전 월 선택 기준보다 세 달 모두 많이 선별했다. 유사 상태 변동 기준과 비교하면 4월 6→7건, 5월 8→6건, 6월 6→14건으로 월별 차이가 남았다. 선택된 시간의 과소예측량 합은 유사 상태 변동 기준 586.723에서 보조 모델 667.693으로 13.8% 증가했다. 이는 확인 대상으로 포함한 예측오차의 합이며 실제로 보정하거나 절감한 전력량이 아니다.

### 성과의 적용 범위

이 결과는 **동일한 하루 2회 확인 조건에서 선별 효율을 높인 탐색적 개발 성과**다. 전체 큰 과소예측 184건 중 157건은 선택하지 않았으므로 자동 경고나 전체 오류 방지 성능으로 확대하지 않는다. 보조 모델의 선택 182회 중 33회는 남은 확인 횟수를 소진하는 규칙에 따라 배정됐고 이 중 큰 과소예측은 없었다. 이 비효율도 동일 확인량 규칙의 적용 결과로 남긴다.

날짜별 결과를 달력 주 단위로 묶어 동일한 주를 함께 재표집한 2,000회 민감도 분석에서, 유사 상태 변동 기준 대비 선별 증가 7건의 95% 백분위 범위는 −6 ~ 19건이었다. 표본 구성에 따라 우위가 바뀔 여지가 있다. 4 ~ 6월은 여러 개발 실험에서 사용한 기간이고 반복 배열도 있어 이 범위를 독립 시험의 통계적 입증으로 해석하지 않는다. 이번 35%는 해당 개발 조건에서 관측한 개선률이다.

현장 활용안은 다음 시간의 최대값 예측과 함께 검토 필요 표시를 제시하고, 선택된 시간에 담당자가 이미 확인 가능한 가동 계획과 관측 상태를 추가 점검하는 것이다. 실제 조치의 효과는 측정하지 않았다. 결과보고서에서는 제3장의 오류분석에 선별 성과를, 제4장에 하루 확인량을 제한한 활용 절차를 연결하고, 제5장에는 예측값과 과소예측 검토 우선순위를 함께 제공한 기여를 기술할 수 있다.

### 전체 Python 코드와 재현 근거

| 코드 | 역할 |
|---|---|
| [error_warning.py](scripts/error_warning.py) | diagnose: B·6시간 입력의 동일 상태 진단, 과거 이웃과 보조 입력 구성 |
| [error_warning_fit.py](scripts/error_warning_fit.py) | freeze·run: 사건 경계·시간순 분할 고정, 보조 모델 6개 학습, 월별 점수·이전 월 선택 저장 |
| [error_warning_verify.py](scripts/error_warning_verify.py) | 기존 입력·이웃·모델·점수·사건 정의와 HGB 독립 재학습 검증 |
| [priority_review.py](scripts/priority_review.py) | freeze·run: 하루 2회 순차 선택, 동일 확인량 비교, 일별·월별 지표와 주별 민감도 |
| [priority_review_verify.py](scripts/priority_review_verify.py) | 별도 순차 계산으로 선택·분위수·지표 검증, 미래 점수 변조로 과거 선택 불변 확인 |
| [priority_review_documents.py](scripts/priority_review_documents.py) | 기존 원고 보존, 결과표 기반 상세 절 생성, 새 절·README·코드 링크 검증 |

부모 입력의 공통 구성은 [profile_history.py](scripts/profile_history.py)와 그 [검증 코드](scripts/profile_history_verify.py)를 따른다. 계약과 점수의 해시로 기존 [보조 모델 결과](tables/error_warning/run.json), [모델 목록](tables/error_warning/model_manifest.csv), [부모 검증](tables/error_warning/independent_verification.json)을 연결했다. 첫 진단 파일의 초기 freeze·run 초안은 학습 진입점으로 사용하지 않으며 실제 학습은 별도 fit 파일을 사용한다.

이번 [고정 계약](tables/priority_review/contract.json), [전체 비교표](tables/priority_review/metrics.csv), [매시간 선택](tables/priority_review/predictions.csv), [날짜별 결과](tables/priority_review/daily.csv), [주별 민감도](tables/priority_review/sensitivity.csv), [판정](tables/priority_review/decision.json), [실행 기록](tables/priority_review/run.json), [독립 검증](tables/priority_review/independent_verification.json)을 함께 보존한다.

최초 실행 순서는 프로젝트 루트에서 정규 CPython 3.13으로 다음과 같다. Windows에서는 루트 AGENTS.md의 Start-Process 경로를 사용한다. 부모 보조 점수는 위 error_warning 코드로 먼저 재현한다.

~~~text
Modeling/scripts/priority_review.py freeze
Modeling/scripts/priority_review.py run
Modeling/scripts/priority_review_verify.py
Modeling/scripts/priority_review_documents.py --write
Modeling/scripts/priority_review_documents.py --verify
~~~

완료된 계약·결과의 덮어쓰기는 막으며 현재 저장 결과는 두 검증 명령으로 확인한다. 독립 검증은 10,920개 시간별 선택, 455개 방법·날짜별 할당, 지표 20행과 2,000회 주별 재표집을 대조했다. 미래 점수만 변경한 45개 시험에서 앞선 선택이 유지됐고, 기존 모델 6개와 부모 결과의 해시도 확인했다. 이번 비교의 신규 학습은 0회다. 상세 원고만 추가하며 압축본과 기존 M05 결과는 변경하지 않았다.
"""

def write():
    decision=json.loads((OUT/"decision.json").read_text(encoding="utf-8"))
    verified=json.loads((OUT/"independent_verification.json").read_text(encoding="utf-8"))
    assert decision["reportworthy_development_result"] and verified["status"]=="passed"
    original=MANUSCRIPT.read_text(encoding="utf-8")
    assert HEADING not in original and not (OUT/"manuscript_before.md").exists()
    backup=OUT/"manuscript_before.md"
    backup.write_bytes(MANUSCRIPT.read_bytes())
    assert sha(backup)==sha(MANUSCRIPT)
    intro="이번 원고는 M01의 자료·평가 설계에서 M02 모델 비교, M03 입력 비교, M04 오류분석, M05 고정 후반기 평가와 후속 개발까지 이어진다. 마지막 값의 개발 개선과 후반기 한계를 구분하고, 4.11에서는 하루 2회 확인 조건의 과소예측 검토 우선순위 성과를 다룬다. 각 절의 당시 판단을 보존하며 현재 성과 범위는 후속 평가에 따라 갱신한다. 수치는 원자료의 전력 척도로 제시하고 실제 전력·비용 절감량으로 환산하지 않는다."
    updated=re.sub(r"(?m)^이번 원고는 .*?$",lambda _:intro,original,count=1)
    MANUSCRIPT.write_text(updated.rstrip()+"\n\n"+section().strip()+"\n",encoding="utf-8")
    readme=ROOT/"Modeling/README.md"
    text=readme.read_text(encoding="utf-8")
    assert README_HEADING not in text
    addition=README_HEADING+"""

사용자가 완벽한 월별 성과보다 보고서에 쓸 수 있는 실용적 기여를 검토하도록 요청해, 자동 경고와 별도로 하루 2회 검토 우선순위를 평가했다. 매시간 이전 월 점수·남은 횟수·시간만 사용하는 규칙을 먼저 고정했다. 4 ~ 6월 91일간 각 방법이 똑같이 182시간을 선택했으며 보조 모델은 큰 과소예측 27건, 가장 좋은 단순 기준은 20건을 선별했다. 7건·35.0% 증가로 사전의 최소 5건·20% 개선과 2개월 이상 개선 조건을 통과했다.

제한된 개발 성과로 [상세 원고 4.11](04_Modeling_원고.md#411-하루-2회-검토를-위한-과소예측-우선순위)에 반영했다. 보조 모델의 월별 적중은 7·6·14건이며 가장 좋은 단순 점수는 6·8·6건이다. 5월 열세, 전체 184건 중 157건 미선택, 주별 재표집의 증가 범위 −6 ~ 19건도 함께 남겼다. 35%는 이 개발 조건의 관측 개선률이고 새 독립 시험이나 자동 경고 채택을 뜻하지 않는다. 이 절이 아래 과거 실험의 다음 작업 안내에 우선한다.

| 실제 사용 코드 | 결과·재현 |
|---|---|
| [priority_review.py](scripts/priority_review.py) | freeze·run, [계약](tables/priority_review/contract.json), [전체 지표](tables/priority_review/metrics.csv), [매시간 선택](tables/priority_review/predictions.csv), [판정](tables/priority_review/decision.json) |
| [priority_review_verify.py](scripts/priority_review_verify.py) | [독립 검증](tables/priority_review/independent_verification.json): 10,920개 선택·455개 일별 할당·45개 미래 점수 변경 시험 통과 |
| [priority_review_documents.py](scripts/priority_review_documents.py) | --write·--verify, 원고 원본 보존·추가·결과표와 링크 대조 |

부모 error_warning 코드·모델·점수는 아래 절과 원고 코드표에 모두 연결했다. 새 학습·튜닝은 0회이며 반복 실행으로 유리한 규칙을 고르지 않았다. 기존 자동 경고 후보의 미채택 판정과 M05를 유지하고 압축본은 변경하지 않았다. 보고서에서는 예측오류 점수와 제한된 확인 횟수를 연결한 보조 활용으로 기술하며 일반적인 성능 우위나 실제 절감 효과를 주장하지 않는다. 이 비교는 종료하고, 동일 개발 기간에서 수치를 더 높이기 위한 규칙 탐색은 확대하지 않는다.

"""
    readme.write_text(text.replace("## 과소예측 위험 경고 비교 결과",addition+"## 과소예측 위험 경고 비교 결과",1),encoding="utf-8")
    root=ROOT/"README.md"
    text=root.read_text(encoding="utf-8")
    notice="**현재 상태 — 2026-10-05 하루 2회 검토 우선순위 개발 성과 반영:** 미래 점수를 보지 않는 순차 선택으로 4 ~ 6월 182시간씩 확인했을 때, 큰 과소예측 선별은 가장 좋은 단순 기준 20건에서 보조 모델 27건으로 35.0% 늘었다. 월별 열세와 낮은 전체 재현율·표본 불확실성은 함께 명시해 [상세 원고 4.11](Modeling/04_Modeling_원고.md#411-하루-2회-검토를-위한-과소예측-우선순위)에 제한된 개발 성과로 반영했다. [규칙·코드·검증](Modeling/README.md#하루-2회-검토-우선순위-개발-성과-반영)을 보존했다. 이번 기여는 같은 확인량에서의 선별 효율이며 자동 경고 채택·새 독립 시험·실제 절감 입증은 아니다."
    text=re.sub(r"(?m)^[*][*]현재 상태[^\r\n]*",lambda _:notice,text,count=1)
    assert notice in text
    root.write_text(text,encoding="utf-8")
    analysis=ROOT/"Analysis/README.md"
    text=analysis.read_text(encoding="utf-8")
    note="""**후속 활용 비교:** 기존 위험 점수를 이용한 하루 2회 순차 검토에서 동일 182시간의 큰 과소예측 선별 20→27건을 확인했다. [Modeling 상세 원고 4.11](../Modeling/04_Modeling_원고.md#411-하루-2회-검토를-위한-과소예측-우선순위)에 개발 성과와 월별 한계를 함께 기록했다. EDA·Analysis 본문과 압축본은 변경하지 않았다.

"""
    analysis.write_text(text.replace("**동일·유사 입력과 과소예측 위험 진단 완료:**",note+"**동일·유사 입력과 과소예측 위험 진단 완료:**",1),encoding="utf-8")
    print("Preserved original and added detailed section 4.11.",flush=True)

def verify():
    manuscript=MANUSCRIPT.read_text(encoding="utf-8")
    assert manuscript.count(HEADING)==1
    # Later manuscript sections are outside the generated 4.11 scope.
    actual_section = re.split(r"(?m)^## ", manuscript.split(HEADING)[1], maxsplit=1)[0]
    assert actual_section.strip()==section().split(HEADING)[1].strip()
    assert (OUT/"manuscript_before.md").exists()
    paths=[MANUSCRIPT,ROOT/"Modeling/README.md",ROOT/"README.md",ROOT/"Analysis/README.md"]
    count=0
    for path in paths:
        text=path.read_text(encoding="utf-8")
        if path==ROOT/"README.md":
            text="\n".join(text.splitlines()[:4])
        elif path==ROOT/"Modeling/README.md":
            text=text.split("## 과소예측 위험 경고 비교 결과")[0]
        elif path==ROOT/"Analysis/README.md":
            text=text.split("**동일·유사 입력과 과소예측 위험 진단 완료:**")[0]
        for target in re.findall(r"!?\[[^\]]*\]\(([^)]+)\)",text):
            if target.startswith(("http:","https:","#","app:")):
                continue
            assert (path.parent/target.split("#")[0].strip("<>")).exists(),target
            count+=1
    # Use the simple literal delimiter pattern to check all links as well.
    for target in re.findall(r"\]\(([^)]+)\)",manuscript):
        assert (MANUSCRIPT.parent/target.split("#")[0]).exists(),target
    record={"status":"passed","documents_sha256":{p.relative_to(ROOT).as_posix():sha(p) for p in paths},
            "original_manuscript_sha256":sha(OUT/"manuscript_before.md"),
            "run_sha256":sha(OUT/"run.json"),"verifier_sha256":sha(OUT/"independent_verification.json"),
            "script_sha256":sha(__file__),"links_checked":count,"generated_section_matches":True}
    (OUT/"document_verification.json").write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding="utf-8")
    print(json.dumps(record,ensure_ascii=False),flush=True)

if __name__=="__main__":
    parser=argparse.ArgumentParser()
    g=parser.add_mutually_exclusive_group(required=True)
    g.add_argument("--write",action="store_true")
    g.add_argument("--verify",action="store_true")
    args=parser.parse_args()
    if args.write:
        write()
    else:
        verify()


코드 등록 완료: `Modeling/scripts/priority_review_documents.py`

### `Modeling/scripts/priority_review_verify.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Independent streaming replay, metric and paired-week sensitivity checks.

주요 함수: `sha()`, `records()`, `truth()`, `quantile()`, `close()`, `summarize()`, `main()`.

In [75]:
%%submission_source Modeling/scripts/priority_review_verify.py
"""Independent streaming replay, metric and paired-week sensitivity checks."""
import csv
import hashlib
import json
import math
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/priority_review"
PARENT = ROOT / "Modeling/tables/error_warning"

def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def records(p):
    with Path(p).open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

def truth(x):
    assert x in ("True","False")
    return x == "True"

def quantile(sorted_values, q):
    position = (len(sorted_values)-1)*q
    low, high = math.floor(position), math.ceil(position)
    return sorted_values[low] + (sorted_values[high]-sorted_values[low])*(position-low)

def close(a,b):
    assert math.isclose(float(a),float(b),abs_tol=1e-8,rel_tol=1e-10),(a,b)

def summarize(rows):
    event = [truth(x["event"]) for x in rows]
    selected = [truth(x["review"]) for x in rows]
    forced = [truth(x["forced"]) for x in rows]
    under = [float(x["under_amount"]) for x in rows]
    count = sum(event)
    tp = sum(a and b for a,b in zip(event,selected))
    num = sum(selected)
    captured = math.fsum(u for u,s in zip(under,selected) if s)
    total = math.fsum(under)
    return {"hours":len(rows),"days":len({x["date"] for x in rows}),"events":count,
            "reviews":num,"TP":tp,"non_event_reviews":num-tp,"FN":count-tp,
            "precision":tp/num,"recall":tp/count if count else 0.,
            "review_fraction":num/len(rows),"captured_under":captured,
            "missed_under":total-captured,"captured_under_share":captured/total,
            "forced_reviews":sum(forced),
            "forced_TP":sum(a and b for a,b in zip(forced,event))}

def main():
    assert sys.version_info[:2] == (3,13) and sys._is_gil_enabled()
    c = json.loads((OUT/"contract.json").read_text(encoding="utf-8"))
    run = json.loads((OUT/"run.json").read_text(encoding="utf-8"))
    assert run["status"] == "completed" and run["contract_sha256"] == sha(OUT/"contract.json")
    assert c["entrypoint_sha256"] == sha(ROOT/"Modeling/scripts/priority_review.py")
    for path,digest in c["inputs_sha256"].items():
        assert sha(ROOT/path) == digest,path
    for name,digest in run["outputs_sha256"].items():
        assert sha(OUT/name) == digest,name
    parent_verified = json.loads((PARENT/"independent_verification.json").read_text(encoding="utf-8"))
    assert parent_verified["status"] == "passed"
    assert parent_verified["run_sha256"] == sha(PARENT/"run.json")
    assert parent_verified["verifier_sha256"] == sha(ROOT/"Modeling/scripts/error_warning_verify.py")
    parent_run = json.loads((PARENT/"run.json").read_text(encoding="utf-8"))
    for name,digest in parent_run["outputs_sha256"].items():
        assert sha(PARENT/name) == digest
    for m in records(PARENT/"model_manifest.csv"):
        assert sha(ROOT/m["file"]) == m["sha256"]
    parent = {(x["method"],x["timestamp"]):x for x in records(PARENT/"predictions.csv")}
    selection = {x["month"]:x for x in json.loads((PARENT/"selection.json").read_text(encoding="utf-8"))}
    cal = defaultdict(list)
    cal_end = {}
    for x in records(PARENT/"calibration_predictions.csv"):
        key = (x["method"],int(x["outer_month"]))
        assert int(x["month"]) == key[1]-1
        cal[key].append(float(x["score"]))
        cal_end[key] = max(cal_end.get(key,""),x["timestamp"])
    for values in cal.values():
        values.sort()
    result = records(OUT/"predictions.csv")
    groups = defaultdict(list)
    for x in result:
        groups[(x["method"],x["date"])].append(x)
    assert len(result) == 10920 and len(groups) == 455
    for (method,date),rows in groups.items():
        rows.sort(key=lambda x:x["timestamp"])
        assert [datetime.fromisoformat(x["timestamp"]).hour for x in rows] == list(range(24))
        used = 0
        for hour,row in enumerate(rows):
            month = int(row["month"])
            src = parent[(method,row["timestamp"])]
            for field in ["event","under_amount","actual","prediction","score","month","date"]:
                if field in ("under_amount","actual","prediction","score"):
                    close(row[field],src[field])
                else:
                    assert row[field] == src[field],field
            model = selection[month]["ML" if method == "selected_ML" else "simple"] if method.startswith("selected_") else method
            key = (model,month)
            assert row["score_method"] == model and row["calibration_end"] == cal_end[key]
            assert cal_end[key] < row["timestamp"]
            remaining = 2-used
            assert int(row["remaining_before"]) == remaining
            assert int(row["hours_remaining"]) == 24-hour
            forced = remaining > 0 and remaining == 24-hour
            threshold = None
            if remaining == 0:
                chosen = False
            elif forced:
                chosen = True
            else:
                threshold = quantile(cal[key],1-remaining/(24-hour))
                chosen = float(row["score"]) > threshold
            assert truth(row["review"]) == chosen and truth(row["forced"]) == forced
            if threshold is None:
                assert row["boundary"] == ""
            else:
                close(row["boundary"],threshold)
            used += int(chosen)
        assert used == 2
    metric_rows = records(OUT/"metrics.csv")
    for row in metric_rows:
        rs = [x for x in result if x["method"] == row["method"] and
              (row["period"] == "pooled" or x["month"] == row["period"])]
        for field,value in summarize(rs).items():
            close(row[field],value)
    daily = records(OUT/"daily.csv")
    for row in daily:
        for field,value in summarize(groups[(row["method"],row["date"])]).items():
            close(row[field],value)
    for row in records(OUT/"hourly_allocation.csv"):
        selected = [x for x in result if x["method"] == row["method"] and
                    datetime.fromisoformat(x["timestamp"]).hour == int(row["hour"])]
        assert sum(truth(x["review"]) for x in selected) == int(row["reviews"])
        assert sum(truth(x["forced"]) for x in selected) == int(row["forced"])
    by_day = {(x["method"],x["date"]):x for x in daily}
    dates = sorted({x["date"] for x in daily})
    for row in records(OUT/"sensitivity.csv"):
        weeks = defaultdict(lambda:[0.,0.])
        for date in dates:
            target = by_day[("selected_ML",date)]
            base = by_day[(row["reference"],date)]
            t = pd.Timestamp(date)
            monday = (t-pd.Timedelta(days=t.dayofweek)).strftime("%Y-%m-%d")
            weeks[monday][0] += int(target["TP"])-int(base["TP"])
            weeks[monday][1] += float(target["captured_under"])-float(base["captured_under"])
        values = [weeks[w] for w in sorted(weeks)]
        assert int(row["weeks"]) == len(values)
        indices = np.random.default_rng(42).integers(0,len(values),size=(2000,len(values)))
        samples = [[math.fsum(values[i][j] for i in sample) for j in [0,1]] for sample in indices]
        for j,prefix in enumerate(["TP","under"]):
            close(row[prefix+"_difference"],math.fsum(v[j] for v in values))
            sorted_samples = sorted(s[j] for s in samples)
            close(row[prefix+"_low"],quantile(sorted_samples,.025))
            close(row[prefix+"_high"],quantile(sorted_samples,.975))
    index = {(x["method"],x["period"]):x for x in metric_rows}
    a = int(index[("selected_ML","pooled")]["TP"])
    strongest = max(int(index[(m,"pooled")]["TP"]) for m in c["methods"] if m != "selected_ML")
    flags = {"at_least_five_more":a>=strongest+5,"at_least_twenty_percent_more":a>=strongest*1.2,
             "two_month_wins":sum(int(index[("selected_ML",str(m))]["TP"]) >
                                 int(index[("selected_simple",str(m))]["TP"]) for m in [4,5,6])>=2}
    assert flags == run["decision"]["criteria"]
    assert all(flags.values()) == run["decision"]["reportworthy_development_result"]
    # Future score perturbation must leave all preceding decisions unchanged.
    from priority_review import replay
    prefix_cases = 0
    for method in c["methods"]:
        for month in [4,5,6]:
            first_day = sorted((v for (m,d),v in groups.items() if m==method and int(v[0]["month"])==month),
                               key=lambda r:r[0]["timestamp"])[0]
            t = pd.to_datetime([x["timestamp"] for x in first_day])
            scores = np.array([float(x["score"]) for x in first_day])
            model = first_day[0]["score_method"]
            original = replay(scores,np.array(cal[(model,month)]),t)
            for cut in [1,12,23]:
                changed = scores.copy()
                changed[cut:] = 1e9
                altered = replay(changed,np.array(cal[(model,month)]),t)
                pd.testing.assert_frame_equal(original.iloc[:cut],altered.iloc[:cut])
                prefix_cases += 1
    verification = {"status":"passed","verifier_sha256":sha(__file__),
                    "run_sha256":sha(OUT/"run.json"),"parent_models_hash_checked":6,
                    "decision_rows_checked":len(result),"daily_quotas_checked":len(groups),
                    "metrics_checked":len(metric_rows),"future_score_perturbation_cases":prefix_cases,
                    "bootstrap_resamples_per_reference":2000,"new_model_fits":0,
                    "decision":run["decision"],"limitations":"reused development, not plant effectiveness"}
    (OUT/"independent_verification.json").write_text(json.dumps(verification,indent=2),encoding="utf-8")
    print(json.dumps(verification),flush=True)

if __name__ == "__main__":
    main()

코드 등록 완료: `Modeling/scripts/priority_review_verify.py`

### `Modeling/scripts/profile_history.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: P02/P03: bounded past-profile comparison and four-slot output ablation.
Freeze before fitting. Nested chronological selection; no later-period tuning.

주요 함수: `csvout()`, `load()`, `freeze()`, `base()`, `execute()`.

In [76]:
%%submission_source Modeling/scripts/profile_history.py
"""P02/P03: bounded past-profile comparison and four-slot output ablation.
Freeze before fitting. Nested chronological selection; no later-period tuning.
"""
import argparse, json, os, sys
from pathlib import Path
for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']: os.environ.setdefault(k,'4')
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import HistGradientBoostingRegressor
from m01_prepare import ROOT, feature_columns, read_contract, sha
from redevelopment_r01 import dump, now
from redevelopment_compare import pw, assess, record
OUT=ROOT/'Modeling/tables/profile_history'
MODELS=ROOT/'Modeling/models/profile_history'
ANALYSIS=ROOT/'Analysis/tables/p02_profile_history'
CONTRACT=ROOT/'Modeling/config/profile_history_contract.json'
SLOTS=['15분','30분','45분','60분']
PARAM={'max_leaf_nodes':15,'max_iter':150,'learning_rate':.05,'min_samples_leaf':20,'l2_regularization':1.,'early_stopping':False,'random_state':42}
M1=read_contract();A=feature_columns('A',M1);B=feature_columns('B',M1)
FEATURES={'A':A,'B':B}
for n in [2,6,24]:
    # lag1 slot4 is exactly the existing lag1_last; avoid duplicate columns.
    FEATURES['H'+str(n)]=B+[f'past_{lag}_slot_{s}' for lag in range(1,n+1) for s in range(1,5) if (lag,s)!=(1,4)]+(['prior_state_age','prior_state_censored'] if n>2 else [])
def csvout(d,name):d.to_csv(OUT/(name+'.csv'),index=False,encoding='utf-8-sig')
def load():
    f=pd.read_csv(ROOT/'Modeling/tables/m01/hourly_frame.csv',encoding='utf-8-sig',float_precision='round_trip',parse_dates=['timestamp','date'])
    raw=pd.read_csv(ROOT/M1['source'],encoding='utf-8-sig')
    raw=raw.loc[raw['날짜'].between(20210101,20210831)&raw['시간'].between(0,23)].copy()
    raw['timestamp']=pd.to_datetime(raw['날짜'].astype(str),format='%Y%m%d')+pd.to_timedelta(raw['시간'],unit='h')
    raw=raw.sort_values('timestamp').set_index('timestamp')
    assert raw.index.tolist()==f.timestamp.tolist()
    features={}
    for lag in range(1,25):
        prev=raw.reindex(pd.DatetimeIndex(f.timestamp-pd.Timedelta(hours=lag)))
        for s,col in enumerate(SLOTS,1):features[f'past_{lag}_slot_{s}']=prev[col].to_numpy(float)
    states=np.select([raw[SLOTS].max(axis=1).eq(0),raw['평균'].lt(20),raw['평균'].between(20,26)],['zero','below20','low'],default='above26')
    age=[];censored=[];run=0;left=True
    for i,ts in enumerate(raw.index):
        contiguous=i>0 and ts-raw.index[i-1]==pd.Timedelta(hours=1)
        if not contiguous or states[i]!=states[i-1]:run=1;left=not contiguous
        else:run+=1
        age.append(run);censored.append(int(left))
    stateframe=pd.DataFrame({'state':states,'age':age,'censored':censored},index=raw.index)
    prev=stateframe.reindex(pd.DatetimeIndex(f.timestamp-pd.Timedelta(hours=1)))
    features['prior_state_age']=prev.age.to_numpy();features['prior_state_censored']=prev.censored.to_numpy()
    f=pd.concat([f,pd.DataFrame(features)],axis=1)
    f['prior_state']=prev.state.to_numpy()
    for s,col in enumerate(SLOTS,1): f['target_slot_'+str(s)]=raw[col].to_numpy(float)
    f['history_complete']=f[[f'past_{lag}_slot_1' for lag in range(1,25)]].notna().all(axis=1)
    f['eligible_history']=f.eligible_common & f.history_complete
    return f

def freeze():
    assert not CONTRACT.exists()
    proof=json.loads((ROOT/'EDA/tables/p01_peak_shapes/run.json').read_text(encoding='utf-8'))
    assert proof['verified']
    paths=['Modeling/scripts/profile_history.py','Modeling/scripts/redevelopment_compare.py','Modeling/scripts/redevelopment_r01.py','Modeling/scripts/m01_prepare.py',
           'Modeling/config/m01_contract.json','Modeling/tables/m01/hourly_frame.csv','Modeling/tables/m03/ab/model_manifest.csv','Modeling/tables/m03/ab/run.json',
           'EDA/tables/p01_peak_shapes/run.json','EDA/tables/p01_peak_shapes/verification.json',M1['source']]
    c={'version':'profile-history-v1','recorded_at':now(),'inputs_sha256':{p:sha(ROOT/p) for p in paths},'features':FEATURES,'parameters':PARAM,
        'reason':'P01 high360h: one high slot136 vs all four28; low duration median24h max223h. Test past shapes/duration before further tuning.',
        'eligibility':'M01 common plus all preceding 24 exact hours for every method; no imputation',
        'inner_months':{'4':[2,3],'5':[2,3,4],'6':[2,3,4,5]},
        'selection':'Pool inner predictions; minimize max(overall MAE/lag1 MAE, daily-maximum MAE/lag1 daily-maximum MAE); tie B,H2,H6,H24. B fallback allowed.',
        'history_for_slot_output':'Same inner direct-maximum selection used for both outputs to isolate output effect',
        'outer_months':[4,5,6],'no_optuna':True,'no_postprocessing':True,'maximum_new_fits':60,
        'models':['fixed_HGB_A','fixed_HGB_B','history_direct','B_four_slots','history_four_slots','lag1'],
        'slot_metrics':'Slot MAE; location credit=intersection of predicted/actual argmax sets divided by predicted ties; no operational timing claim',
        'gates':json.loads((ROOT/'Modeling/config/redevelopment_r02_contract.json').read_text(encoding='utf-8'))['gates'],
        'later':'Only eligible development policy; Jan-Jun fit once, July-Aug fixed, history from June outer selection; already observed, not independent',
        'no_manuscript_edit':True}
    dump(c,CONTRACT);print('Frozen bounded P02/P03 contract',flush=True)

def base(ev,tr,split):
    q=ev[['timestamp','date','month','target_maximum','lag1_maximum','lag1_last','diag_target_profile','prior_state','prior_state_age']].rename(columns={'target_maximum':'actual'}).copy()
    q['split']=split;q['daily_maximum_weight']=pw(ev)
    q['train_profile_overlap']=q.diag_target_profile.isin(tr.diag_target_profile)
    freq=q[['date','diag_target_profile']].drop_duplicates().diag_target_profile.value_counts()
    w=1/q.diag_target_profile.map(freq);q['profile_weight']=w/w.mean()
    return q

def execute():
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    c=json.loads(CONTRACT.read_text(encoding='utf-8'))
    for p,h in c['inputs_sha256'].items(): assert sha(ROOT/p)==h,p
    for d in [OUT,MODELS,ANALYSIS]:d.mkdir(parents=True,exist_ok=True)
    assert not (OUT/'run.json').exists()
    allf=load();f=allf.loc[allf.eligible_history].copy()
    csvout(allf[['timestamp','eligible_common','history_complete','eligible_history']],'coverage')
    csvout(f[['timestamp']+list(dict.fromkeys(A+FEATURES['H24']+['target_maximum','target_slot_1','target_slot_2','target_slot_3','target_slot_4']))],'feature_audit')
    manifests=[];fits=0;cache={};inner_records=[];selection=[];predictions=[];slot_records=[]
    parent=pd.read_csv(ROOT/'Modeling/tables/m03/ab/model_manifest.csv',encoding='utf-8-sig')
    def train_predict(tr,ev,group,target,tag,split=None):
        nonlocal fits
        key=(int(ev.month.iloc[0]),group,target)
        if key in cache:return cache[key]
        columns=FEATURES[group];assert tr.timestamp.max()<ev.timestamp.min()
        assert tr[columns].notna().all().all() and ev[columns].notna().all().all()
        reused=False
        oldtr=allf.loc[allf.eligible_common & allf.timestamp.lt(ev.timestamp.min())]
        if group in ['A','B'] and target=='target_maximum' and split and oldtr.timestamp.tolist()==tr.timestamp.tolist():
            row=parent.loc[parent.split.eq(split)&parent.group.eq(group)&parent.target.eq(target)].iloc[0]
            path=ROOT/row.file;assert sha(path)==row.sha256;m=joblib.load(path);reused=True
        else:
            m=HistGradientBoostingRegressor(**PARAM);m.fit(tr[columns],tr[target]);fits+=1
            path=MODELS/f'{tag}_{group}_{target}.joblib';joblib.dump(m,path,compress=3)
        p=m.predict(ev[columns]);assert np.isfinite(p).all()
        assert list(m.feature_names_in_)==columns
        assert all(m.get_params()[k]==v for k,v in PARAM.items())
        assert np.allclose(joblib.load(path).predict(ev[columns]),p,rtol=0,atol=1e-10)
        manifests.append({'tag':tag,'group':group,'target':target,'file':str(path.relative_to(ROOT)),'sha256':sha(path),'train_hours':len(tr),'eval_hours':len(ev),
            'train_end':str(tr.timestamp.max()),'eval_month':int(ev.month.iloc[0]),'reused':reused})
        cache[key]=p;return p
    # Each inner forecast is generated from strictly earlier months and cached.
    for month in range(2,6):
        tr=f.loc[f.month.lt(month)];ev=f.loc[f.month.eq(month)];q=base(ev,tr,f'inner_{month}')
        for group in ['B','H2','H6','H24']:
            p=train_predict(tr,ev,group,'target_maximum',f'inner_{month}',{4:'dev_apr',5:'dev_may'}.get(month))
            z=q.copy();z['group']=group;z['prediction']=p;inner_records.append(z)
        print(f'Inner month {month}: train={len(tr)} eval={len(ev)} fits={fits}',flush=True)
    inn=pd.concat(inner_records,ignore_index=True);csvout(inn,'inner_predictions')
    def choose(month):
        z=inn.loc[inn.month.lt(month)];lag=z.loc[z.group.eq('B')]
        den=[np.mean(abs(lag.actual-lag.lag1_maximum)),np.average(abs(lag.actual-lag.lag1_maximum),weights=lag.daily_maximum_weight)]
        rows=[]
        for i,group in enumerate(['B','H2','H6','H24']):
            p=z.loc[z.group.eq(group)];mae=float(np.mean(abs(p.actual-p.prediction)));peak=float(np.average(abs(p.actual-p.prediction),weights=p.daily_maximum_weight))
            rows.append({'outer_month':month,'group':group,'inner_months':','.join(str(v) for v in sorted(p.month.unique())),'MAE':mae,'peak_MAE':peak,'score':max(mae/den[0],peak/den[1]),'order':i})
        winner=min(rows,key=lambda x:(x['score'],x['order']))['group']
        for x in rows:x['selected']=x['group']==winner
        selection.extend(rows);return winner
    for month,split in [(4,'dev_apr'),(5,'dev_may'),(6,'dev_jun')]:
        tr=f.loc[f.month.lt(month)];ev=f.loc[f.month.eq(month)];q=base(ev,tr,split);history=choose(month)
        choices=[('fixed_HGB_A','A'),('fixed_HGB_B','B'),('history_direct',history)]
        for name,group in choices:
            p=train_predict(tr,ev,group,'target_maximum',split,split)
            predictions.append(record(q,name,'HGB',group,'direct',p))
        for name,group in [('B_four_slots','B'),('history_four_slots',history)]:
            ps=np.column_stack([train_predict(tr,ev,group,'target_slot_'+str(s),split,split) for s in range(1,5)])
            predictions.append(record(q,name,'HGB',group,'four_slots',ps.max(axis=1)))
            for s in range(1,5):
                z=ev[['timestamp','month']].copy();z['method']=name;z['slot']=s;z['actual']=ev['target_slot_'+str(s)].to_numpy();z['prediction']=ps[:,s-1];slot_records.append(z)
        predictions.append(record(q,'lag1','baseline','B','lag1',ev.lag1_maximum.to_numpy()))
        print(f'Outer {month}: history={history} train={len(tr)} eval={len(ev)} fits={fits}',flush=True)
    p=pd.concat(predictions,ignore_index=True);sp=pd.concat(slot_records,ignore_index=True)
    csvout(p,'predictions');csvout(sp,'slot_predictions');csvout(pd.DataFrame(selection),'inner_selection');csvout(pd.DataFrame(manifests),'model_manifest')
    tables,comparison,decision,leave=assess(p,c)
    csvout(tables,'metrics');csvout(comparison,'comparison');csvout(leave,'leave_one_date_out');dump(decision,OUT/'selection.json')
    # Conditional tables are explanatory only, never inputs or a new selection gate.
    conditions=[]
    for name,g in p.groupby('method'):
        g=g.copy();g['age_bin']=pd.cut(g.prior_state_age,[0,2,6,24,float('inf')],labels=['1-2','3-6','7-24','25+'])
        for (state,age),part in g.groupby(['prior_state','age_bin'],observed=True):
            conditions.append({'method':name,'state':state,'observed_age':str(age),'hours':len(part),'dates':part.date.nunique(),
                'actual_mean':part.actual.mean(),'next_low_fraction':float(part.actual.le(26).mean()),'MAE':float(abs(part.actual-part.prediction).mean()),'under':float((part.actual-part.prediction).clip(lower=0).mean())})
    pd.DataFrame(conditions).to_csv(ANALYSIS/'conditional_errors.csv',index=False,encoding='utf-8-sig')
    slot_metrics=[]
    for (name,month),g in sp.groupby(['method','month']):
        actual=g.pivot(index='timestamp',columns='slot',values='actual').to_numpy();pred=g.pivot(index='timestamp',columns='slot',values='prediction').to_numpy()
        ap=actual==actual.max(1)[:,None];pp=pred==pred.max(1)[:,None]
        slot_metrics.append({'method':name,'month':int(month),'hours':len(actual),'slot_MAE':float(abs(actual-pred).mean()),'location_credit':float(((ap&pp).sum(1)/pp.sum(1)).mean())})
    csvout(pd.DataFrame(slot_metrics),'slot_metrics')
    # Reused parent outputs are diagnostic references, not a later-period run.
    assert fits<=c['maximum_new_fits'] and p.timestamp.max()<pd.Timestamp('2021-07-01')
    result={'status':'completed','finished':now(),'contract_sha256':sha(CONTRACT),'new_fits':fits,'models':len(manifests),
        'full_common':int(allf.eligible_common.sum()),'history_common':len(f),'development_hours':int(p.loc[p.method.eq('fixed_HGB_B')].shape[0]),
        'inner_selection':[{k:v for k,v in r.items() if k in ['outer_month','group','score']} for r in selection if r['selected']],
        'selected':decision['selected'],'later_evaluated':False,'manuscript_edited':False,
        'outputs_sha256':{str(t.relative_to(ROOT)):sha(t) for t in list(OUT.glob('*.csv'))+[OUT/'selection.json',ANALYSIS/'conditional_errors.csv']}}
    dump(result,OUT/'run.json');print(comparison.to_string(index=False),flush=True);print(json.dumps(result,ensure_ascii=False),flush=True)
if __name__=='__main__':
    parser=argparse.ArgumentParser();parser.add_argument('--freeze',action='store_true');parser.add_argument('--run',action='store_true');args=parser.parse_args()
    if args.freeze:freeze()
    if args.run:execute()

코드 등록 완료: `Modeling/scripts/profile_history.py`

### `Modeling/scripts/profile_history_verify.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Independent raw-feature, model reload, selection and metric checks for P01-P04.
Also record fixed descriptive condition diagnostics. Does not fit models or edit manuscripts.

주요 함수: `sha()`, `read()`, `close()`, `dump()`, `summary()`, `main()`.

In [77]:
%%submission_source Modeling/scripts/profile_history_verify.py
"""Independent raw-feature, model reload, selection and metric checks for P01-P04.
Also record fixed descriptive condition diagnostics. Does not fit models or edit manuscripts.
"""
import csv,json,hashlib,os,math
from collections import Counter,defaultdict
from datetime import datetime,timedelta
from pathlib import Path
for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']:os.environ.setdefault(k,'4')
import numpy as np
import pandas as pd
import joblib
ROOT=Path(__file__).resolve().parents[2];OUT=ROOT/'Modeling/tables/profile_history';AO=ROOT/'Analysis/tables/p02_profile_history'
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def read(n,dates=None):return pd.read_csv(OUT/(n+'.csv'),encoding='utf-8-sig',float_precision='round_trip',parse_dates=dates)
def close(a,b):assert np.allclose(a,b,atol=1e-9,rtol=1e-11),(a,b)
def dump(x,p):p.write_text(json.dumps(x,ensure_ascii=False,indent=2),encoding='utf-8')
def summary(q):
    e=q.prediction.to_numpy()-q.actual.to_numpy();w=q.daily_maximum_weight.to_numpy()
    return [np.mean(abs(e)),np.average(abs(e),weights=w),np.average(np.maximum(-e,0),weights=w),np.average(np.maximum(e,0),weights=w)]
def main():
    run=json.loads((OUT/'run.json').read_text(encoding='utf-8'));c=json.loads((ROOT/'Modeling/config/profile_history_contract.json').read_text(encoding='utf-8'))
    assert run['contract_sha256']==sha(ROOT/'Modeling/config/profile_history_contract.json')
    for path,h in {**c['inputs_sha256'],**run['outputs_sha256']}.items():assert sha(ROOT/path)==h,path
    raw={}
    with (ROOT/'data/origin/okm_augumented_2021.csv').open(encoding='utf-8-sig',newline='') as f:
        for r in csv.DictReader(f):
            date=int(r['날짜']);h=float(r['시간'])
            if not (20210101<=date<=20210831 and 0<=h<=23):continue
            ts=datetime.strptime(str(date),'%Y%m%d')+timedelta(hours=h)
            v=[float(r[x]) for x in ['15분','30분','45분','60분']];mean=sum(v)/4
            state='zero' if max(v)==0 else 'below20' if float(r['평균'])<20 else 'low' if float(r['평균'])<=26 else 'above26'
            raw[ts]={'values':v,'max':max(v),'mean':mean,'production':float(r['생산량']),'state':state}
    for ts,r in sorted(raw.items()):
        prev=raw.get(ts-timedelta(hours=1));r['age']=prev['age']+1 if prev and prev['state']==r['state'] else 1
        r['censored']=prev['censored'] if prev and prev['state']==r['state'] else int(prev is None)
    f=read('feature_audit',['timestamp']);checked=0
    for q in f.to_dict('records'):
        ts=q['timestamp'].to_pydatetime();r=raw[ts]
        exp={'month':ts.month,'weekend':int(ts.weekday()>=5),'hour_sin':math.sin(2*math.pi*ts.hour/24),'hour_cos':math.cos(2*math.pi*ts.hour/24),'target_maximum':r['max']}
        exp.update({f'dow_{i}':int(ts.weekday()==i) for i in range(7)})
        for lag in range(1,25):
            p=raw[ts-timedelta(hours=lag)]
            for s in range(1,5):exp[f'past_{lag}_slot_{s}']=p['values'][s-1]
            if lag<=2:exp[f'lag{lag}_mean']=p['mean'];exp[f'lag{lag}_maximum']=p['max']
            if lag==1:exp.update(lag1_last=p['values'][3],lag1_production=p['production'],lag1_production_zero=int(p['production']==0),prior_state_age=p['age'],prior_state_censored=p['censored'])
        exp.update({f'target_slot_{s}':r['values'][s-1] for s in range(1,5)})
        for key,value in q.items():
            if key!='timestamp':assert math.isclose(value,exp[key],abs_tol=1e-12), (ts,key,value,exp[key]);checked+=1
    p=read('predictions',['timestamp','date']);inner=read('inner_predictions',['timestamp','date']);sp=read('slot_predictions',['timestamp'])
    # Independent peak-date weights using raw values and complete evaluated dates.
    for table,methodcol in [(p,'method'),(inner,'group')]:
        for (_,day),g in table.groupby([methodcol,'date']):
            actual=[raw[t.to_pydatetime()]['max'] for t in g.timestamp];assert actual==g.actual.tolist()
            peak=max(actual);ties=actual.count(peak)
            expected=[1/ties if v==peak and len(g)==24 else 0 for v in actual];close(g.daily_maximum_weight,expected)
    manifest=read('model_manifest');models_checked=0;reload_values=0
    for q in manifest.to_dict('records'):
        path=ROOT/q['file'];assert sha(path)==q['sha256'];m=joblib.load(path)
        ev=f.loc[f.timestamp.dt.month.eq(q['eval_month'])];cols=c['features'][q['group']]
        assert list(m.feature_names_in_)==cols and all(m.get_params()[k]==v for k,v in c['parameters'].items())
        assert pd.Timestamp(q['train_end'])<ev.timestamp.min();assert q['train_hours']==int(f.timestamp.dt.month.lt(q['eval_month']).sum())
        pred=m.predict(ev[cols]);checks=[]
        if q['target']=='target_maximum':
            z=inner.loc[inner.month.eq(q['eval_month'])&inner.group.eq(q['group'])]
            if len(z):checks.append(z)
            z=p.loc[p.month.eq(q['eval_month'])&p.input.eq(q['group'])&p.variant.eq('direct')]
            checks.extend(g for _,g in z.groupby('method'))
        else:
            s=int(q['target'][-1]);methods=p.loc[p.month.eq(q['eval_month'])&p.input.eq(q['group'])&p.variant.eq('four_slots'),'method'].unique()
            checks.extend(sp.loc[sp.month.eq(q['eval_month'])&sp.method.eq(name)&sp.slot.eq(s)] for name in methods)
        assert checks
        for z in checks:
            assert z.timestamp.tolist()==ev.timestamp.tolist();close(pred,z.prediction);reload_values+=len(z)
        models_checked+=1
    for (name,month),g in sp.groupby(['method','month']):
        matrix=g.pivot(index='timestamp',columns='slot',values='prediction')
        q=p.loc[p.method.eq(name)&p.month.eq(month)].set_index('timestamp').loc[matrix.index]
        close(matrix.max(axis=1),q.prediction)
    sel=read('inner_selection');selected={}
    for month in [4,5,6]:
        z=inner.loc[inner.month.lt(month)];ref=z.loc[z.group.eq('B')];den=[np.mean(abs(ref.actual-ref.lag1_maximum)),np.average(abs(ref.actual-ref.lag1_maximum),weights=ref.daily_maximum_weight)]
        scores=[]
        for order,group in enumerate(['B','H2','H6','H24']):
            q=z.loc[z.group.eq(group)];s=summary(q);score=max(s[0]/den[0],s[1]/den[1]);scores.append((score,order,group))
            line=sel.loc[sel.outer_month.eq(month)&sel.group.eq(group)].iloc[0];close([line.MAE,line.peak_MAE,line.score],[s[0],s[1],score])
            assert set(q.month)==set(c['inner_months'][str(month)])
        selected[month]=min(scores)[2]
        assert sel.loc[sel.outer_month.eq(month)&sel.selected,'group'].tolist()==[selected[month]]
        assert set(p.loc[p.month.eq(month)&p.method.str.startswith('history_'),'input'])=={selected[month]}
    comp=read('comparison');sums={name:summary(q) for name,q in p.groupby('method')}
    decision=json.loads((OUT/'selection.json').read_text(encoding='utf-8'))
    for row in decision['candidate_gates']:
        name=row['method'];q=p.loc[p.method.eq(name)];s=sums[name];flags={}
        close(s,comp.loc[comp.method.eq(name),['overall_MAE','peak_MAE','peak_under','peak_over']].to_numpy()[0])
        for ref in ['fixed_HGB_A','fixed_HGB_B']:
            r=sums[ref];flags.update({ref+'_overall':s[0]<=r[0]*1.01,ref+'_peak':s[1]<=r[1]*.95,ref+'_under':s[2]<=r[2]*.95,ref+'_over':s[3]-r[3]<=r[1]*.05})
            days=sorted(q.date.unique());rp=p.loc[p.method.eq(ref)];dif=[]
            for day in days:dif.append(summary(q.loc[q.date.ne(day)])[1]-summary(rp.loc[rp.date.ne(day)])[1])
            flags[ref+'_leave_one_date']=max(dif)<0
        flags['monthly']=all(summary(q.loc[q.month.eq(m)])[i]<=summary(p.loc[p.method.eq('fixed_HGB_B')&p.month.eq(m)])[i]*1.05 for m in [4,5,6] for i in [0,1])
        assert flags==row['gates'] and all(flags.values())==row['eligible']
    # P02 same hour, same rounded preceding mean, observed duration contrast.
    bh=p.loc[p.method.eq('fixed_HGB_B')].set_index('timestamp');hp=p.loc[p.method.eq('history_direct')].set_index('timestamp')
    a=pd.DataFrame({'timestamp':bh.index,'hour':bh.index.hour,'month':bh.month.to_numpy(),
        'prior_rounded_mean':[math.floor(raw[t.to_pydatetime()-timedelta(hours=1)]['mean']+.5) for t in bh.index],
        'prior_state':bh.prior_state.to_numpy(),'age':bh.prior_state_age.to_numpy(),'actual':bh.actual.to_numpy(),
        'error_B':abs(bh.prediction-bh.actual).to_numpy(),'error_history':abs(hp.prediction-hp.actual).to_numpy(),'daily_weight':bh.daily_maximum_weight.to_numpy()})
    a['age_bin']=pd.cut(a.age,[0,2,6,24,float('inf')],labels=['1-2','3-6','7-24','25+'])
    t=a.groupby(['hour','prior_rounded_mean','prior_state','age_bin'],observed=True).agg(hours=('actual','size'),actual_mean=('actual','mean'),actual_min=('actual','min'),actual_max=('actual','max'),MAE_B=('error_B','mean'),MAE_history=('error_history','mean')).reset_index()
    t.to_csv(AO/'matched_state_history.csv',index=False,encoding='utf-8-sig')
    paired=[]
    for label,mask in [('all',a.index==a.index),('non_daily_peak',a.daily_weight.eq(0)),('daily_peak_rows',a.daily_weight.gt(0))]:
        z=a.loc[mask];paired.append({'scope':label,'hours':len(z),'MAE_B':z.error_B.mean(),'MAE_history':z.error_history.mean()})
    pd.DataFrame(paired).to_csv(AO/'improvement_location.csv',index=False,encoding='utf-8-sig')
    assert not decision['selected'] # This run's observed outcome; no later evaluation should be dispatched.
    assert all(x<7 for x in p.month.unique()) and run['new_fits']==sum(~manifest.reused)
    v={'status':'passed','run_sha256':sha(OUT/'run.json'),'verifier_sha256':sha(Path(__file__)),
       'raw_feature_values_checked':checked,'model_reloads':models_checked,'prediction_values_rechecked':reload_values,
       'outer_prediction_rows':len(p),'slot_prediction_rows':len(sp),'inner_selections_checked':3,'gates_checked':len(decision['candidate_gates']),
       'analysis_outputs_sha256':{str(x.relative_to(ROOT)):sha(x) for x in AO.glob('*.csv')},'later_evaluation':'skipped: no eligible method','manuscripts_edited':False}
    dump(v,OUT/'independent_verification.json');print(json.dumps(v),flush=True)
    print(pd.DataFrame(paired).to_string(index=False),flush=True)
if __name__=='__main__':main()

코드 등록 완료: `Modeling/scripts/profile_history_verify.py`

### `Modeling/scripts/redevelopment_compare.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: R02-R04: bounded model/input comparison and conditional delta target.

주요 함수: `save()`, `pw()`, `model()`, `fit()`, `predict()`, `obj()`, `score()`, `suggest()`, `current_inputs()`, `freeze()`, `data()`, `splits()`, `evmeta()`, `record()`, `met()`, `metrics()`, `assess()`, `manifest_row()`, `check_inputs()`, `compare()`, `delta_stage()`, `later_stage()`, `main()`.

In [78]:
%%submission_source Modeling/scripts/redevelopment_compare.py
"""R02-R04: bounded model/input comparison and conditional delta target."""
import argparse, json, os, sys, time, importlib.metadata
from pathlib import Path
for k in ["OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS"]:
    os.environ.setdefault(k,"4")
import numpy as np
import pandas as pd
import optuna, joblib
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from m01_prepare import ROOT, feature_columns, read_contract, sha
from redevelopment_r01 import read, dump, now

BASE=ROOT/"Modeling/tables/redevelopment"
CFILE=ROOT/"Modeling/config/redevelopment_r02_contract.json"
MODEL=ROOT/"Modeling/models/redevelopment"
M1=read_contract()
TARGET="target_maximum"
HGB={"max_leaf_nodes":15,"max_iter":150,"learning_rate":.05,"min_samples_leaf":20,"l2_regularization":1.}
FITS=0
def save(df,d,name):
    df.to_csv(d/name,index=False,encoding="utf-8-sig")
def pw(g):
    dates=g.timestamp.dt.normalize()
    full=g.groupby(dates).timestamp.transform("count").eq(24)
    flag=g[TARGET].eq(g.groupby(dates)[TARGET].transform("max")) & full
    ties=flag.groupby(dates).transform("sum").clip(lower=1)
    return np.where(flag,1/ties,0).astype(float)
def model(family,params):
    if family=="HGB":
        return HistGradientBoostingRegressor(**params,early_stopping=False,random_state=42)
    return ExtraTreesRegressor(**params,random_state=42,n_jobs=4,bootstrap=False)
def fit(family,params,tr,cols,delta=False):
    global FITS
    m=model(family,params)
    y=tr[TARGET]-(tr.lag1_last if delta else 0)
    m.fit(tr[cols],y);FITS+=1
    return m
def predict(m,ev,cols,delta=False):
    return m.predict(ev[cols])+(ev.lag1_last.to_numpy() if delta else 0)
def obj(y,p,w):
    e=abs(np.asarray(p)-np.asarray(y))
    return float(e.mean()),float(np.average(e,weights=w))
def score(v,d):
    return max(v[0]/d[0],v[1]/d[1])
def suggest(t,family,spaces):
    params={}
    for name,spec in spaces[family].items():
        if name in ["max_leaf_nodes","max_depth"]:params[name]=t.suggest_categorical(name,spec)
        elif name in ["max_iter","n_estimators","min_samples_leaf"]:
            params[name]=t.suggest_int(name,spec[0],spec[1],step=spec[2] if len(spec)==3 else 1)
        else:params[name]=t.suggest_float(name,spec[0],spec[1],log=len(spec)==3 and spec[2]=="log")
    return params
def current_inputs():
    paths=["Modeling/config/m01_contract.json","Modeling/tables/m01/hourly_frame.csv",
           "Modeling/tables/m01/evaluation_diagnostics.csv","Modeling/config/m02_rerun_contract.json",
           "Modeling/tables/m02_rerun/selected_configurations.json","Modeling/tables/m02_rerun/model_manifest.csv",
           "Modeling/tables/m03/ab/model_manifest.csv","Modeling/tables/redevelopment/r01/run.json",
           "Modeling/tables/redevelopment/r01/conditions.csv","Modeling/tables/redevelopment/r01/independent_verification.json",
           "Modeling/scripts/redevelopment_compare.py"]
    return {p:sha(ROOT/p) for p in paths}
def freeze():
    assert not CFILE.exists()
    ver=json.loads((BASE/"r01/independent_verification.json").read_text(encoding="utf-8"))
    assert ver["status"]=="passed"
    parent=json.loads((ROOT/"Modeling/config/m02_rerun_contract.json").read_text(encoding="utf-8"))
    diag=read(BASE/"r01/paired.csv")
    bias=diag.groupby(["prior_state",diag.month.le(6).map({True:"dev",False:"later"})])[["bias_A","bias_B"]].mean()
    c={"version":"redevelopment-r02-v1","recorded_at":now(),"reason":"R01 high-prior and upward-slot peak effects reversed in later period; compare model/input before any conditional gate.",
       "inputs_sha256":current_inputs(),"features":{g:feature_columns(g,M1) for g in ["A","B"]},
       "families":["HGB","ExtraTrees"],"inputs":["A","B"],"outer":["dev_apr","dev_may","dev_jun"],
       "inner_months":{"dev_apr":[2,3],"dev_may":[2,3,4],"dev_jun":[2,3,4,5]},
       "fixed_HGB":HGB,"fixed_ExtraTrees":"existing per-outer A-selected parameters; same settings applied to A and B",
       "search_spaces":{f:parent["search_spaces"][f] for f in ["HGB","ExtraTrees"]},
       "trials_per_study":30,"max_trials":360,"sampler_seed":42,"startup":10,"extension":False,
       "objective":"two MAEs, choose minimum max(normalized by inner lag1 baseline); tie trial number",
       "postprocessing":"none for direct and delta; negative outputs retained and diagnosed",
       "gates":{"peak_ratio_max":.95,"peak_under_ratio_max":.95,"overall_ratio_max":1.01,
                "peak_over_increase_as_reference_peak_MAE":.05,"month_overall_ratio_max":1.05,"month_peak_ratio_max":1.05,
                "comparison":["fixed_HGB_A","fixed_HGB_B"],"jackknife_peak_improvement":True},
       "R03":"only if no R02 candidate eligible and R01 level-specific bias remains; choose best B-input family/variant by development balanced score; compare delta with identical B direct model and parameters per outer",
       "R03_level_bias_present":bool(bias.loc[("above26","later"),"bias_B"]<0 and bias.loc[("low20_26","later"),"bias_B"]>0),
       "R04":"only an eligible development method; latest chronological outer configuration, no July refit; no test-based reselection",
       "scope":"development redesign informed by already observed July-August; no new independent test"}
    dump(c,CFILE);print("R02 contract recorded; 360-trial cap",flush=True)
def data():
    f=read(ROOT/"Modeling/tables/m01/hourly_frame.csv")
    return f.loc[f.eligible_common].copy()
def splits():
    return {s["name"]:s for s in M1["splits"]}
def evmeta(ev,split):
    diag=read(ROOT/"Modeling/tables/m01/evaluation_diagnostics.csv")
    d=diag.loc[diag.split.eq(split)&diag.pool.eq("common")]
    p=ev[["timestamp","date","month",TARGET,"lag1_maximum","lag1_last","diag_target_profile"]].rename(columns={TARGET:"actual"})
    p=p.merge(d[["timestamp","profile_weight","train_profile_overlap","daily_maximum_weight"]],on="timestamp",validate="one_to_one")
    assert len(p)==len(ev) and np.allclose(p.daily_maximum_weight,pw(ev),rtol=0,atol=1e-12)
    p["split"]=split
    return p
def record(base,name,family,group,variant,p):
    r=base.copy();r["method"]=name;r["family"]=family;r["input"]=group;r["variant"]=variant;r["prediction"]=p
    e=r.prediction-r.actual
    r["absolute_error"]=abs(e);r["under_amount"]=(-e).clip(lower=0);r["over_amount"]=e.clip(lower=0);r["bias"]=e
    return r
def met(g,w=None):
    w=np.ones(len(g)) if w is None else np.asarray(w)
    return {"hours":len(g),"weight_sum":float(w.sum()),"MAE":float(np.average(abs(g.prediction-g.actual),weights=w)),
            "mean_under":float(np.average(np.maximum(g.actual-g.prediction,0),weights=w)),
            "mean_over":float(np.average(np.maximum(g.prediction-g.actual,0),weights=w)),
            "bias":float(np.average(g.prediction-g.actual,weights=w)),
            "mean_actual":float(np.average(g.actual,weights=w)),"negative_predictions":int((g.prediction<0).sum())}
def metrics(p):
    rows=[]
    for method,g in p.groupby("method"):
        for period,part in [("pooled",g)]+[(s,h) for s,h in g.groupby("split")]:
            for scope in ["all","peak","profile_reweighted"]:
                weights=part.daily_maximum_weight if scope=="peak" else part.profile_weight if scope=="profile_reweighted" else None
                if weights is not None and weights.sum()==0:continue
                rows.append({"method":method,"period":period,"scope":scope,**met(part,weights)})
        for field in ["month","train_profile_overlap"]:
            for value,part in g.groupby(field):
                rows.append({"method":method,"period":f"{field}:{value}","scope":"all",**met(part)})
                if field=="month" and part.daily_maximum_weight.sum()>0:
                    rows.append({"method":method,"period":f"{field}:{value}","scope":"peak",**met(part,part.daily_maximum_weight)})
        delta=g.actual-g.lag1_maximum
        for label,mask in [("surge93",delta.ge(93)),("zero_actual",g.actual.eq(0)),("prior_zero",g.lag1_maximum.eq(0))]:
            if mask.any():rows.append({"method":method,"period":label,"scope":"all",**met(g.loc[mask])})
    return pd.DataFrame(rows)
def assess(p,c):
    tables=metrics(p)
    idx=tables.set_index(["method","period","scope"])
    lag=idx.loc[("lag1","pooled","all"),"MAE"],idx.loc[("lag1","pooled","peak"),"MAE"]
    rows=[];leave=[]
    for method,g in p.groupby("method"):
        overall=idx.loc[(method,"pooled","all")];peak=idx.loc[(method,"pooled","peak")]
        flags={}
        for ref in c["gates"]["comparison"]:
            ra=idx.loc[(ref,"pooled","all")];rp=idx.loc[(ref,"pooled","peak")]
            flags[ref+"_overall"]=bool(overall.MAE<=ra.MAE*c["gates"]["overall_ratio_max"])
            flags[ref+"_peak"]=bool(peak.MAE<=rp.MAE*c["gates"]["peak_ratio_max"])
            flags[ref+"_under"]=bool(peak.mean_under<=rp.mean_under*c["gates"]["peak_under_ratio_max"])
            flags[ref+"_over"]=bool(peak.mean_over-rp.mean_over<=rp.MAE*c["gates"]["peak_over_increase_as_reference_peak_MAE"])
            paired=g[["timestamp","date","absolute_error","daily_maximum_weight"]].merge(
                p.loc[p.method.eq(ref),["timestamp","absolute_error"]],on="timestamp",suffixes=("_candidate","_reference"),validate="one_to_one")
            differences=[]
            for day in sorted(paired.date.unique()):
                rem=paired.loc[paired.date.ne(day)]
                value=float(np.average(rem.absolute_error_candidate-rem.absolute_error_reference,weights=rem.daily_maximum_weight))
                leave.append({"method":method,"reference":ref,"excluded_date":day,"peak_delta_MAE":value});differences.append(value)
            flags[ref+"_leave_one_date"]=bool(max(differences)<0)
        mb=[]
        for s in [f"month:{x}" for x in sorted(g.month.unique())]:
            for scope in ["all","peak"]:
                ratio=idx.loc[(method,s,scope),"MAE"]/idx.loc[("fixed_HGB_B",s,scope),"MAE"]
                mb.append(ratio<=c["gates"]["month_overall_ratio_max" if scope=="all" else "month_peak_ratio_max"])
        flags["monthly"]=bool(all(mb))
        rows.append({"method":method,"overall_MAE":float(overall.MAE),"peak_MAE":float(peak.MAE),
                     "peak_under":float(peak.mean_under),"peak_over":float(peak.mean_over),
                     "balanced_score":score((overall.MAE,peak.MAE),lag),"eligible":all(flags.values()),
                     "gates":flags})
    table=pd.DataFrame([{k:v for k,v in r.items() if k!="gates"} for r in rows])
    eligible=table.loc[table.eligible]
    preferred=lambda df:df.sort_values(["balanced_score","method"]).iloc[0].method
    chosen=preferred(eligible) if len(eligible) else None
    return tables,table,{"selected":chosen,"candidate_gates":rows,"no_eligible_candidate":chosen is None},pd.DataFrame(leave)
def manifest_row(m,path,split,name,family,group,variant,params,tr,source):
    return {"split":split,"method":name,"family":family,"input":group,"variant":variant,"file":str(path.relative_to(ROOT)),
            "sha256":sha(path),"parameters":json.dumps(params),"train_start":str(tr.timestamp.min()),"train_end":str(tr.timestamp.max()),
            "train_hours":len(tr),"source":source,"reload_checked":True}
def check_inputs(c):
    for p,h in c["inputs_sha256"].items():assert sha(ROOT/p)==h,p
def compare():
    c=json.loads(CFILE.read_text(encoding="utf-8"));check_inputs(c)
    d=BASE/"r02";d.mkdir(parents=True,exist_ok=True);MODEL.mkdir(parents=True,exist_ok=True)
    assert not (d/"run.json").exists()
    dump({"status":"running","started":now(),"contract_sha256":sha(CFILE),"script_sha256":sha(Path(__file__))},d/"run.json")
    f=data();oldconfigs=json.loads((ROOT/"Modeling/tables/m02_rerun/selected_configurations.json").read_text(encoding="utf-8"))
    oldet=pd.read_csv(ROOT/"Modeling/tables/m02_rerun/model_manifest.csv",encoding="utf-8-sig")
    oldh=pd.read_csv(ROOT/"Modeling/tables/m03/ab/model_manifest.csv",encoding="utf-8-sig")
    predictions=[];man=[];configs=[];trials=[];foldscores=[];innerrecords=[];audit=[]
    start=time.perf_counter()
    for split in c["outer"]:
        s=splits()[split];tr=f.loc[f.timestamp.le(s["train_end"])];ev=f.loc[f.timestamp.between(s["eval_start"],s["eval_end"])]
        base=evmeta(ev,split)
        predictions.append(record(base,"lag1","baseline","none","baseline",ev.lag1_maximum.to_numpy()))
        folds=[]
        for month in c["inner_months"][split]:
            bound=pd.Timestamp(2021,month,1);a=tr.loc[tr.timestamp.lt(bound)];b=tr.loc[tr.timestamp.ge(bound)&tr.timestamp.lt(bound+pd.offsets.MonthBegin(1))]
            assert a.timestamp.max()<b.timestamp.min()<ev.timestamp.min()
            folds.append((month,a,b,pw(b)))
            audit.append({"split":split,"inner_month":month,"train_start":str(a.timestamp.min()),"train_end":str(a.timestamp.max()),
                          "validation_start":str(b.timestamp.min()),"validation_end":str(b.timestamp.max()),"train_hours":len(a),"validation_hours":len(b)})
        ys=np.concatenate([b[TARGET] for _,_,b,_ in folds]);ws=np.concatenate([w for _,_,_,w in folds])
        denomin=obj(ys,np.concatenate([b.lag1_maximum for _,_,b,_ in folds]),ws)
        for family in c["families"]:
            frozen=HGB if family=="HGB" else next(x["parameters"] for x in oldconfigs if x["split"]==split and x["model"]=="ExtraTrees")
            for group in c["inputs"]:
                cols=c["features"][group];name=f"fixed_{family}_{group}"
                if family=="HGB":
                    mr=oldh.loc[oldh.split.eq(split)&oldh.target.eq(TARGET)&oldh.group.eq(group)].iloc[0]
                elif group=="A":
                    mr=oldet.loc[oldet.split.eq(split)&oldet.model.eq("ExtraTrees")].iloc[0]
                else:mr=None
                if mr is not None:
                    path=ROOT/mr.file;assert sha(path)==mr.sha256;m=joblib.load(path);source="parent_reload"
                    for k,v in frozen.items():assert m.get_params()[k]==v
                else:
                    m=fit(family,frozen,tr,cols);path=MODEL/f"{split}_{name}.joblib";joblib.dump(m,path,compress=3);source="new_fit"
                pred=predict(m,ev,cols);assert np.allclose(pred,predict(joblib.load(path),ev,cols),rtol=0,atol=1e-9)
                predictions.append(record(base,name,family,group,"fixed",pred));man.append(manifest_row(m,path,split,name,family,group,"fixed",frozen,tr,source))
                configs.append({"split":split,"method":name,"family":family,"input":group,"variant":"fixed","parameters":frozen})
                cache={}
                def objective(t):
                    params=suggest(t,family,c["search_spaces"]);parts=[]
                    for month,a,b,w in folds:
                        mm=fit(family,params,a,cols);pp=predict(mm,b,cols);o=obj(b[TARGET],pp,w)
                        foldscores.append({"split":split,"family":family,"input":group,"trial":t.number,"month":month,
                                           "hours":len(b),"peak_weight_sum":float(w.sum()),"overall_MAE":o[0],"peak_MAE":o[1]})
                        parts.append(pp)
                    cache[t.number]=np.concatenate(parts)
                    return obj(ys,cache[t.number],ws)
                study=optuna.create_study(directions=["minimize","minimize"],sampler=optuna.samplers.TPESampler(seed=42,n_startup_trials=10),
                                         pruner=optuna.pruners.NopPruner())
                if family=="HGB":study.enqueue_trial(HGB)
                study.optimize(objective,n_trials=30,n_jobs=1)
                selected=min(study.best_trials,key=lambda t:(score(t.values,denomin),t.number))
                params=selected.params;tname=f"tuned_{family}_{group}"
                mm=fit(family,params,tr,cols);path=MODEL/f"{split}_{tname}.joblib";joblib.dump(mm,path,compress=3)
                pred=predict(mm,ev,cols);assert np.allclose(pred,predict(joblib.load(path),ev,cols),rtol=0,atol=1e-9)
                predictions.append(record(base,tname,family,group,"tuned",pred));man.append(manifest_row(mm,path,split,tname,family,group,"tuned",params,tr,"new_fit"))
                configs.append({"split":split,"method":tname,"family":family,"input":group,"variant":"tuned","parameters":params,
                                "selected_trial":selected.number,"inner_objectives":list(selected.values),"denominator":list(denomin),"trials":30})
                times=np.concatenate([b.timestamp.to_numpy() for _,_,b,_ in folds])
                innerrecords.append(pd.DataFrame({"split":split,"method":tname,"timestamp":times,"actual":ys,"prediction":cache[selected.number],
                                                  "daily_maximum_weight":ws,"lag1_prediction":np.concatenate([b.lag1_maximum for _,_,b,_ in folds])}))
                for t in study.trials:
                    trials.append({"split":split,"family":family,"input":group,"number":t.number,"state":t.state.name,
                                   "overall_MAE":t.values[0],"peak_MAE":t.values[1],"baseline_overall":denomin[0],"baseline_peak":denomin[1],
                                   "balanced_score":score(t.values,denomin),"parameters":json.dumps(t.params),"selected":t.number==selected.number})
                save(pd.DataFrame(trials),d,"trials.csv");save(pd.DataFrame(foldscores),d,"trial_fold_scores.csv");dump(configs,d/"configurations.json")
                print(f"{split} {tname}:30 trials, trial {selected.number}, MAE/peak {obj(ev[TARGET],pred,pw(ev))}; fits={FITS}",flush=True)
    p=pd.concat(predictions,ignore_index=True)
    save(p,d,"predictions.csv");save(pd.DataFrame(man),d,"model_manifest.csv");save(pd.DataFrame(audit),d,"time_audit.csv");save(pd.concat(innerrecords),d,"selected_inner_predictions.csv")
    tables,table,selection,leave=assess(p,c)
    save(tables,d,"metrics.csv");save(table,d,"comparison.csv");save(leave,d,"leave_one_date_out.csv");dump(selection,d/"selection.json")
    dump({"status":"completed","completed_at":now(),"seconds":time.perf_counter()-start,"new_fits":FITS,"new_trials":len(trials),
          "contract_sha256":sha(CFILE),"script_sha256":sha(Path(__file__)),"inputs_sha256":c["inputs_sha256"],
          "outputs_sha256":{p.name:sha(p) for p in d.iterdir() if p.suffix in [".csv",".json"] and p.name!="run.json"}},d/"run.json")
    print(table.to_string(index=False),flush=True)
def delta_stage():
    c=json.loads(CFILE.read_text(encoding="utf-8"));check_inputs(c);d=BASE/"r03";d.mkdir(parents=True,exist_ok=True)
    assert not (d/"run.json").exists()
    r2=BASE/"r02";sel=json.loads((r2/"selection.json").read_text(encoding="utf-8"))
    assert json.loads((r2/"independent_verification.json").read_text(encoding="utf-8"))["status"]=="passed"
    if sel["selected"] is not None or not c["R03_level_bias_present"]:
        dump({"status":"skipped","reason":"R02 has eligible candidate or level-bias hypothesis unsupported","recorded_at":now()},d/"run.json");print("R03 skipped",flush=True);return
    table=pd.read_csv(r2/"comparison.csv",encoding="utf-8-sig")
    b=table.loc[table.method.str.endswith("_B")].sort_values(["balanced_score","method"]).iloc[0].method
    conf=json.loads((r2/"configurations.json").read_text(encoding="utf-8"))
    dump({"recorded_at":now(),"direct_method":b,"hypothesis":"origin-anchored change may reduce level bias; same direct B parameters, no Optuna",
          "r02_run_sha256":sha(r2/"run.json"),"later_not_used_in_selection":True},d/"contract.json")
    f=data();records=[];man=[]; configs=[]
    p=read(r2/"predictions.csv")
    baseline=p.loc[p.method.isin(["fixed_HGB_A","fixed_HGB_B","lag1",b])].drop_duplicates(["timestamp","method"])
    records.append(baseline)
    for split in c["outer"]:
        s=splits()[split];tr=f.loc[f.timestamp.le(s["train_end"])];ev=f.loc[f.timestamp.between(s["eval_start"],s["eval_end"])]
        cf=next(x for x in conf if x["split"]==split and x["method"]==b)
        family,params=cf["family"],cf["parameters"];cols=c["features"]["B"];name="delta_"+b
        mm=fit(family,params,tr,cols,True);path=MODEL/f"{split}_{name}.joblib";joblib.dump(mm,path,compress=3);pp=predict(mm,ev,cols,True)
        assert np.allclose(pp,predict(joblib.load(path),ev,cols,True),rtol=0,atol=1e-9)
        records.append(record(evmeta(ev,split),name,family,"B","delta",pp))
        man.append(manifest_row(mm,path,split,name,family,"B","delta",params,tr,"new_fit"))
        configs.append({"split":split,"method":name,"family":family,"input":"B","variant":"delta","parameters":params})
    pred=pd.concat(records,ignore_index=True)
    save(pred,d,"predictions.csv");save(pd.DataFrame(man),d,"model_manifest.csv");dump(configs,d/"configurations.json")
    tables,table,selection,leave=assess(pred,c)
    save(tables,d,"metrics.csv");save(table,d,"comparison.csv");save(leave,d,"leave_one_date_out.csv");dump(selection,d/"selection.json")
    dump({"status":"completed","completed_at":now(),"new_fits":FITS,"new_trials":0,"script_sha256":sha(Path(__file__)),"parent_run_sha256":sha(r2/"run.json"),
          "outputs_sha256":{p.name:sha(p) for p in d.iterdir() if p.suffix in [".csv",".json"] and p.name!="run.json"}},d/"run.json")
    print("Direct B comparison selected for R03:",b,flush=True);print(table.to_string(index=False),flush=True)
def later_stage():
    c=json.loads(CFILE.read_text(encoding="utf-8"));check_inputs(c)
    d=BASE/"r04";d.mkdir(parents=True,exist_ok=True);assert not (d/"run.json").exists()
    r2=BASE/"r02";r3=BASE/"r03"
    s2=json.loads((r2/"selection.json").read_text(encoding="utf-8"));s3run=json.loads((r3/"run.json").read_text(encoding="utf-8"))
    selected=s2["selected"];source=r2
    if selected is None and s3run["status"]=="completed":
        selected=json.loads((r3/"selection.json").read_text(encoding="utf-8"))["selected"];source=r3
    if selected is None:
        dump({"status":"skipped","reason":"No candidate met fixed development criteria; do not search later data for a winner",
              "recorded_at":now(),"differentiation_goal":"unresolved"},d/"run.json");print("R04 skipped: no eligible development candidate",flush=True);return
    v=json.loads((source/"independent_verification.json").read_text(encoding="utf-8"));assert v["status"]=="passed"
    configs=json.loads((source/"configurations.json").read_text(encoding="utf-8"))
    cf=next(x for x in configs if x["method"]==selected and x["split"]=="dev_jun")
    # Freeze latest chronological configuration before any new later prediction.
    dump({"recorded_at":now(),"method":selected,"configuration":cf,"training_end":"2021-06-30 23:00:00",
          "refit_july":False,"scope":"observed later-period redevelopment evaluation","parent_run_sha256":sha(source/"run.json")},d/"contract.json")
    f=data();tr=f.loc[f.timestamp.lt("2021-07-01")];ev=f.loc[f.timestamp.ge("2021-07-01")]
    assert len(tr)==4176 and len(ev)==1344
    cols=c["features"][cf["input"]];delta=cf["variant"]=="delta"
    mm=fit(cf["family"],cf["parameters"],tr,cols,delta);path=MODEL/f"later_{selected}.joblib";joblib.dump(mm,path,compress=3)
    pred=predict(mm,ev,cols,delta);assert np.allclose(pred,predict(joblib.load(path),ev,cols,delta),rtol=0,atol=1e-9)
    base=evmeta(ev,"later_jul_aug")
    records=[record(base,selected,cf["family"],cf["input"],cf["variant"],pred)]
    old=read(ROOT/"Modeling/tables/m05/predictions.csv")
    for group,name in [("A","fixed_HGB_A"),("B","fixed_HGB_B"),("lag1","lag1")]:
        part=old.loc[old.pool.eq("common")&old.group.eq(group)]
        assert len(part)==len(ev) and part.timestamp.tolist()==ev.timestamp.tolist()
        records.append(record(base,name,"HGB" if group!="lag1" else "baseline",group,"fixed",part.prediction.to_numpy()))
    p=pd.concat(records,ignore_index=True);save(p,d,"predictions.csv")
    save(pd.DataFrame([manifest_row(mm,path,"later_jul_aug",selected,cf["family"],cf["input"],cf["variant"],cf["parameters"],tr,"new_fit")]),d,"model_manifest.csv")
    tables,table,selection,leave=assess(p,c);save(tables,d,"metrics.csv");save(table,d,"comparison.csv");save(leave,d,"leave_one_date_out.csv");dump(selection,d/"selection.json")
    dump([cf],d/"configurations.json")
    dump({"status":"completed","completed_at":now(),"new_fits":FITS,"new_trials":0,"script_sha256":sha(Path(__file__)),
          "outputs_sha256":{p.name:sha(p) for p in d.iterdir() if p.suffix in [".csv",".json"] and p.name!="run.json"}},d/"run.json")
    print(table.to_string(index=False),flush=True)
def main():
    ap=argparse.ArgumentParser();ap.add_argument("--stage",choices=["freeze","r02","r03","r04"],required=True);a=ap.parse_args()
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    {"freeze":freeze,"r02":compare,"r03":delta_stage,"r04":later_stage}[a.stage]()
if __name__=="__main__":main()


코드 등록 완료: `Modeling/scripts/redevelopment_compare.py`

### `Modeling/scripts/redevelopment_documents.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Generate/verify redevelopment manuscript and execution record from checked results.

주요 함수: `j()`, `checked()`, `comparison()`, `section()`, `write()`, `verify()`, `main()`.

In [79]:
%%submission_source Modeling/scripts/redevelopment_documents.py
"""Generate/verify redevelopment manuscript and execution record from checked results."""
import argparse,json,re
from pathlib import Path
import pandas as pd
from m01_prepare import ROOT,sha
from redevelopment_r01 import now
BASE=ROOT/"Modeling/tables/redevelopment"
MD=ROOT/"Modeling/04_Modeling_원고.md"
LOG=ROOT/"Modeling/10.05_R01_R04_차별성재개발_실행기록.md"
TITLE="## 4.10 후반기 오류를 반영한 예측 설계 재개발"
CODES={"redevelopment_r01.py":"R01 기존 A/B의 조건별 오차와 학습 자료 기준 구간 경계 계산",
       "redevelopment_compare.py":"R02 모델×입력·Optuna 비교, 조건부 R03 변화량 목표, 적격 후보 R04 후반기 평가",
       "redevelopment_verify.py":"원자료 시차·지표·튜닝 선택·모델 추론·채택 요건 독립 검증",
       "redevelopment_documents.py":"상세 원고·실행 기록 생성, 사용 코드와 문서 링크 검증",
       "redevelopment_peak.py":"R02/R03 실패 근거를 반영한 일별 최대 시간 학습 가중 비교·조건부 후반기 평가",
       "redevelopment_peak_verify.py":"학습 가중·내부 선택·가중 모델 3개 독립 재학습 및 평가 검증"}
def j(p):return json.loads(p.read_text(encoding="utf-8"))
def checked(stage):
    d=BASE/stage;r=j(d/"run.json");v=j(d/"independent_verification.json")
    assert v["status"]=="passed" and v["run_sha256"]==sha(d/"run.json")
    for name,h in r.get("outputs_sha256",{}).items():assert sha(d/name)==h
    return r
def comparison(stage):
    df=pd.read_csv(BASE/stage/"comparison.csv",encoding="utf-8-sig")
    lines=["| 방법 | 전체 MAE | 최대 시간 MAE | 최대 시간 과소 | 최대 시간 과대 | 요건 |","|---|---:|---:|---:|---:|---|"]
    for r in df.itertuples():lines.append(f"| {r.method} | {r.overall_MAE:.3f} | {r.peak_MAE:.3f} | {r.peak_under:.3f} | {r.peak_over:.3f} | {'충족' if r.eligible else '미충족'} |")
    return "\n".join(lines)
def section():
    runs={stage:checked(stage) for stage in ["r01","r02","r03","r03b","r04"]}
    s=["### 이전 결과와 새 질문",
       "M03에서 마지막 값 추가로 전체 MAE가 8.1%, 실제 일별 최대 시간 MAE가 35.4% 감소했지만 M05에서는 전체 0.92% 개선·최대 시간 3.19% 악화로 달라졌다. M02는 A 입력으로 모델을 선택했으므로 B에서도 HGB가 가장 적합한지는 미확인이었다. 사용자의 차별성 입증 요구에 따라 기존 오류 진단 → 모델×입력 비교 → 조건부 변화량 목표 → 적격 후보 후반기 재평가를 실행했다. 원고 4.1~4.9와 기존 결과는 보존했다.",
       "### R01: 마지막 값의 이득과 손해",
       "학습 없이 개발 2,184시간·후반기 1,344시간의 같은 시각 A/B 예측 3,528쌍을 비교했다. 직전 전력 수준, 직전 네 전력값의 변동 폭·상승/하강, 마지막 값과 최대값 차이로 구분하고 새 경계는 각 평가 시점 이전 학습 자료에서 정했다. 실제 하루 최대 여부는 평가용 가중으로만 사용했다.",
       "| 기간 | 직전 관측 조건 | 최대 시간 행 수 | 날짜 가중 합 | A MAE | B MAE | B−A |","|---|---|---:|---:|---:|---:|---:|"]
    cond=pd.read_csv(BASE/"r01/conditions.csv",encoding="utf-8-sig")
    part=cond.loc[cond.scope.eq("peak")&((cond.condition.eq("prior_state")&cond.value.eq("above26"))|(cond.condition.eq("prior_trend")&cond.value.eq("up")))]
    for r in part.itertuples():s.append(f"| {r.period} | {r.condition}:{r.value} | {r.hours} | {r.weight_sum:.3f} | {r.abs_A:.3f} | {r.abs_B:.3f} | {r.delta_MAE:+.3f} |")
    s += ["가중 합은 완전한 날짜의 실제 최대 동률을 나눈 합이며 독립 사례 수가 아니다. 개발에서 개선되던 직전 조건이 후반기에서 악화해 이 조건만으로 B를 선택하는 규칙을 만들 근거가 부족했다. 전력 수준별 편향도 남아 있어 변화량 목표의 제한 비교 근거로 삼았다. 관측 조건이 오류와 관련된다는 사실만으로 운영 원인을 확정하지 않았다.",
          "근거: [R01 계약](config/redevelopment_r01_contract.json), [예측쌍](tables/redevelopment/r01/paired.csv), [조건별 결과](tables/redevelopment/r01/conditions.csv), [학습 구간 경계](tables/redevelopment/r01/thresholds.csv), [날짜별 오류](tables/redevelopment/r01/daily_errors.csv), [검증](tables/redevelopment/r01/independent_verification.json).",
          "### R02: 모델과 입력의 교차 비교",
          "HGB·ExtraTrees × A·B를 비교했다. A는 17입력이고 B는 마지막 값을 더한 18입력이다. 고정 비교에는 HGB의 기존 잎 15 설정, ExtraTrees의 기존 외부 월별 A 선택 설정을 A/B에 동일하게 적용했다. 재튜닝 비교는 모델×입력×외부 월마다 Optuna 30회·seed 42로 동일 예산을 적용했다. 기존 모델은 해시와 입력·설정을 확인해 재로딩했다.",
          "외부 비교는 1~3월→4월, 1~4월→5월, 1~5월→6월의 모든 적격 2,184시간이다. 내부는 각 외부 학습 구간 안에서 1월→2월부터 월 단위 확장 검증을 한다. 외부 평가 월에는 모델을 고정하고 매시간 입력만 갱신한다. 전체·날짜 균등 최대 시간 MAE 두 목표를 직전 기준으로 정규화한 상대오차 최댓값으로 설정을 선택했다. 고정 A/B로 입력의 기여를 확인하며 재튜닝의 기여와 구분한다.",
          f"Optuna {runs['r02']['new_trials']}회·새 학습 {runs['r02']['new_fits']}회를 수행했다. trial 수와 fit 수는 서로 다르다. 실행 전 import 누락을 수정했으며 최초 계약은 [보존본](tables/redevelopment/r02/preflight_contract.json)에 남겼다. 데이터·검색 공간·평가 기준은 바뀌지 않았다.",
          comparison("r02"),
          "새 후보 채택에는 기존 HGB A/B 각각보다 최대 시간 MAE·평균 과소가 5% 이상 작고, 전체 MAE는 각각 1% 이내 악화여야 한다. 최대 시간 과대 증가량은 각 기준 최대 시간 MAE의 5% 이내, 각 월의 전체·최대 MAE는 기존 B보다 5% 이내 악화여야 한다. 날짜 하나씩 제외해도 최대 시간 개선 방향이 유지돼야 한다. 1%·5%는 사전 수치 비교 목표이며 통계적 유의성이나 경제적 손익 기준이 아니다.",
          f"개발 선택: {j(BASE/'r02/selection.json')['selected'] or '새 적격 후보 없음'}.",
          "근거: [R02 계약·검색 범위](config/redevelopment_r02_contract.json), [360회 trial](tables/redevelopment/r02/trials.csv), [내부 월별 점수](tables/redevelopment/r02/trial_fold_scores.csv), [시간 경계](tables/redevelopment/r02/time_audit.csv), [선택 설정](tables/redevelopment/r02/configurations.json), [예측](tables/redevelopment/r02/predictions.csv), [지표](tables/redevelopment/r02/metrics.csv), [판정](tables/redevelopment/r02/selection.json), [검증](tables/redevelopment/r02/independent_verification.json).",
          "### R03: 변화량 목표의 조건부 비교"]
    if runs["r03"]["status"]=="completed":
        ct=j(BASE/"r03/contract.json")
        s += [f"R02에 적격 후보가 없고 전력 수준별 편향이 남아 {ct['direct_method']}를 동일 설정의 B 직접 예측 비교 상대로 선택했다. 모델 종류·입력·외부 월별 설정을 유지하고 학습 목표만 '다음 최대값−직전 마지막 값'으로 바꿨다. 추론 시 관측 마지막 값에 예측 변화량을 더하며 실제 미래 전력을 사용하지 않는다. 추가 Optuna 없이 새 학습 3회다. 출력은 직접 예측처럼 사후 절단하지 않았다.",comparison("r03"),
              "근거: [R03 조건](tables/redevelopment/r03/contract.json), [설정](tables/redevelopment/r03/configurations.json), [예측](tables/redevelopment/r03/predictions.csv), [판정](tables/redevelopment/r03/selection.json)."]
    else:s.append("사전에 정한 실행 조건이 충족되지 않아 R03을 생략했다.")
    s += ["R03 [실행 또는 생략 기록](tables/redevelopment/r03/run.json), [검증](tables/redevelopment/r03/independent_verification.json).",
          "### R04: 후반기 재개발 평가와 판단 갱신"]
    if runs["r04"]["status"]=="completed":
        ct=j(BASE/"r04/contract.json");sel=j(BASE/"r04/selection.json")
        s += [f"개발 선택 방법 {ct['method']}의 6월 외부 설정을 고정하고 1~6월 4,176시간으로 한 번 학습해 7~8월 공통 1,344시간을 평가했다. 7월 정답으로 재학습하지 않았다. 아래 기준의 후반기 충족 여부는 {'충족' if sel['selected'] else '미충족'}이다.",comparison("r04"),
              "후반기 결과를 보고 후보·경계·설정을 다시 선택하지 않았다. 차별성은 개선과 안정성이 확인된 범위에서만 주장한다."]
    else:s.append("R02·R03·R03b에 새 적격 후보가 없어 R04를 생략했다. 이번 모델×입력 및 변화량 비교에서는 강한 차별성을 입증할 새 방법을 채택하지 못했다. 선언한 비교는 종료하고 차별성 목표는 미해결로 남긴다. 같은 탐색을 늘리지 않고 새 근거가 있을 때 후속 가설을 검토한다.")
    s += ["R04 [실행 또는 생략 기록](tables/redevelopment/r04/run.json), [검증](tables/redevelopment/r04/independent_verification.json).",
          "4~6월은 반복 선택에 사용한 개발 기간, 7~8월은 이번 가설 형성에 사용한 후반기 기간이다. 시간순 검증을 적용해도 선택·열람 이력이 사라지지 않으므로 새로운 독립 시험으로 표현하지 않는다. 기존 M05 결과는 유지한다. 급등 탐지 주장은 별도 경보 규칙과 AP·TP/FP/FN·실제 오경보율 검증이 필요하며 연속값 오차로 대신하지 않는다. 실제 전력·비용 절감, 현장 적용, 최종 제출은 하지 않았다.",
          "### 사용 코드와 재현 순서"]
    pos=s.index("### R04: 후반기 재개발 평가와 판단 갱신")
    s[pos:pos]=["### R03b: 실제 최대 시간의 학습 비중",
                "모델×입력과 변화량 목표 비교가 모두 새 후보를 만들지 못해, 평가의 핵심인 최대 시간과 학습의 초점이 일치하는지를 추가로 확인했다. 이전 M04의 급상승 전환 가중과 달리, 이번에는 과거 학습 자료의 실제 일별 최대 시간에 비중을 줬다. HGB B의 입력·트리 설정은 고정하고 한 가지 학습 가중만 바꿨다. 새 이유·범위·7개 가중과 내부 선택 규칙을 실행 전에 별도 계약으로 기록했다.",
                "학습 가중은 1+(lambda−1)×날짜 균등 최대 가중이다. 완전한 학습 날짜의 실제 최대 동률에는 1/동률 수를 적용하고 다른 시간에는 추가 가중을 주지 않는다. 평가 날짜의 미래 최대 여부는 학습·추론에 사용하지 않는다. lambda는 1, 1.25, 1.5, 2, 3, 4, 8로 제한했다. 각 외부 학습 구간의 과거 내부 예측에서 전체·과대·월별 악화를 제한하고 최대 시간 MAE와 과소의 상대오차 최댓값으로 선택했다. 새 Optuna 없이 내부 21개 설정 비교·새 학습 75회·독립 검증용 재학습 3회를 수행했다.",
                comparison("r03b"),
                "4월·5월에는 lambda=1, 6월에는 1.5가 선택됐다. 최대 시간 MAE 7.589→7.551의 약 0.5% 개선과 평균 과소 6.987→6.966의 약 0.3% 개선은 새 채택 기준의 5%에 미달했다. 기존 기준은 완화하지 않았고 이 방법도 새 최종 후보로 채택하지 않았다.",
                "근거: [가중 계약](config/redevelopment_peak_contract.json), [내부 예측](tables/redevelopment/r03b/inner_predictions.csv), [21개 비교](tables/redevelopment/r03b/weight_trials.csv), [학습 가중 기록](tables/redevelopment/r03b/training_weight_audit.csv), [선택 설정](tables/redevelopment/r03b/configurations.json), [외부 예측](tables/redevelopment/r03b/predictions.csv), [판정](tables/redevelopment/r03b/selection.json), [독립 검증](tables/redevelopment/r03b/independent_verification.json)."]
    s += [f"- [{name}](scripts/{name}): {role}." for name,role in CODES.items()]
    s += ["공통 입력 코드 [m01_prepare.py](scripts/m01_prepare.py)를 함께 보존한다. R01은 paired()/summary(), 비교는 compare()/assess()/delta_stage()/later_stage(), 검증은 r01()/stages(), 문서는 section()/verify()를 사용한다.",
          "아래는 최초 작성·실행 때의 단계 순서다. 정규 CPython 3.13 및 루트 AGENTS.md의 승격 Start-Process 경로를 사용한다. 기존 run.json이 있으면 덮어쓰기를 거부하므로 새로운 실행은 별도 버전 경로를 사용한다.",
          "~~~text\nredevelopment_r01.py --stage freeze\nredevelopment_r01.py --stage run\nredevelopment_verify.py --stage r01\nredevelopment_compare.py --stage freeze\nredevelopment_compare.py --stage r02\nredevelopment_verify.py --stage r02\nredevelopment_compare.py --stage r03\nredevelopment_verify.py --stage r03\nredevelopment_peak.py --stage freeze\nredevelopment_peak.py --stage run\nredevelopment_peak_verify.py --stage r03b\nredevelopment_peak.py --stage later\nredevelopment_peak_verify.py --stage r04\nredevelopment_documents.py --write\nredevelopment_documents.py --verify\n~~~",
          "모든 코드·부모 해시·모델은 [재개발 결과](tables/redevelopment)와 [재개발 모델](models/redevelopment)에 보존한다. 원고 압축·파일 삭제는 하지 않았다."]
    return TITLE+"\n\n"+"\n\n".join(s)+"\n"
def write():
    text=MD.read_text(encoding="utf-8");assert TITLE not in text
    pos=text.index("## 4.6")
    added="### 재개발에 추가한 코드\n\n| 파일 | 역할 |\n|---|---|\n"+"\n".join(f"| [{name}](scripts/{name}) | {role} |" for name,role in CODES.items())+"\n\n"
    text=text[:pos]+added+text[pos:];text=text.replace("Python 코드 35개","Python 코드 41개").replace("전체 35개","전체 41개").replace("사용한 35개 파일이다","사용한 41개 파일이다")
    MD.write_text(text.rstrip()+"\n\n"+section(),encoding="utf-8")
    LOG.write_text("# 차별성 재개발 실행 기록\n\n"+section(),encoding="utf-8")
def verify():
    expected=section()
    for path in [MD,LOG]:
        text=path.read_text(encoding="utf-8");assert text.count(TITLE)==1
        assert text.split(TITLE,1)[1].rstrip()==expected.split(TITLE,1)[1].rstrip()
    text=MD.read_text(encoding="utf-8");links=re.findall(r'!?\[[^\]]*\]\(([^)]+)\)',text)
    for t in links:
        if not t.startswith(("http:","https:","#","app:")):assert (MD.parent/t.split("#")[0].strip("<>")).resolve().exists(),t
    result={"status":"passed","checked_at":now(),"script_sha256":sha(Path(__file__)),"manuscript_sha256":sha(MD),"record_sha256":sha(LOG),"links":len(links),"new_fits":0}
    (BASE/"document_verification.json").write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding="utf-8")
    print(json.dumps(result),flush=True)
def main():
    ap=argparse.ArgumentParser();g=ap.add_mutually_exclusive_group(required=True);g.add_argument("--write",action="store_true");g.add_argument("--verify",action="store_true");a=ap.parse_args()
    if a.write:write()
    verify()
if __name__=="__main__":main()


코드 등록 완료: `Modeling/scripts/redevelopment_documents.py`

### `Modeling/scripts/redevelopment_peak.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: R03b: historical daily-maximum weighting, same HGB B and no tree retuning.

주요 함수: `freeze()`, `fitted()`, `run()`, `later()`, `main()`.

In [80]:
%%submission_source Modeling/scripts/redevelopment_peak.py
"""R03b: historical daily-maximum weighting, same HGB B and no tree retuning."""
import argparse,json,sys,time
from pathlib import Path
import numpy as np,pandas as pd,joblib
import redevelopment_compare as common
from redevelopment_r01 import now,dump,read
from m01_prepare import ROOT,sha
BASE=common.BASE;CF=ROOT/"Modeling/config/redevelopment_peak_contract.json"
WEIGHTS=[1.,1.25,1.5,2.,3.,4.,8.]
def freeze():
    assert not CF.exists()
    for stage in ["r02","r03"]:
        v=json.loads((BASE/stage/"independent_verification.json").read_text(encoding="utf-8"));assert v["status"]=="passed"
    c={"version":"redevelopment-peak-v1","recorded_at":now(),
       "reason":"R02 has no eligible model/input candidate; same-input delta target has MAE6.181 but peak8.953 versus B7.589. Align training emphasis with daily-max evaluation rather than repeat model tuning.",
       "source_r02":sha(BASE/"r02/run.json"),"source_r03":sha(BASE/"r03/run.json"),
       "helper_sha256":sha(ROOT/"Modeling/scripts/redevelopment_compare.py"),"entrypoint_sha256":sha(Path(__file__)),
       "weight_grid":WEIGHTS,"parameters":common.HGB,"features":"B18; unchanged",
       "training_weight":"1+(lambda-1)*w; w is 1/ties at actual maximum of each complete eligible training date, zero otherwise",
       "selection":"within each outer training interval: overall <=1.01 inner B, over increase <=0.05 inner B peakMAE, monthly overall/peak <=1.05 B; minimize max(peakMAE/B,peakUnder/B), tie smaller lambda. Lambda1 baseline always feasible.",
       "outer_gates":"unchanged redevelopment_r02_contract.json criteria against both original A/B",
       "no_new_optuna":True,"no_later_selection":True,"max_new_fits":75}
    dump(c,CF);print("R03b fixed-weight grid recorded",flush=True)
def fitted(tr,cols,lam):
    m=common.model("HGB",common.HGB)
    w=1+(lam-1)*common.pw(tr)
    m.fit(tr[cols],tr[common.TARGET],sample_weight=w)
    return m,w
def run():
    c=json.loads(CF.read_text(encoding="utf-8"));assert c["entrypoint_sha256"]==sha(Path(__file__))
    assert c["helper_sha256"]==sha(ROOT/"Modeling/scripts/redevelopment_compare.py")
    d=BASE/"r03b";d.mkdir(parents=True,exist_ok=True);assert not (d/"run.json").exists()
    contract=json.loads(common.CFILE.read_text(encoding="utf-8"));common.check_inputs(contract)
    f=common.data();cols=contract["features"]["B"];selected=[];all_inner=[];trials=[];man=[];preds=[];training=[]
    old=read(BASE/"r02/predictions.csv")
    preds.append(old.loc[old.method.isin(["fixed_HGB_A","fixed_HGB_B","lag1"])])
    fits=0;start=time.perf_counter()
    for split in contract["outer"]:
        s=common.splits()[split];tr=f.loc[f.timestamp.le(s["train_end"])];ev=f.loc[f.timestamp.between(s["eval_start"],s["eval_end"])]
        baseline=[];candidates={lam:[] for lam in WEIGHTS}
        for month in contract["inner_months"][split]:
            t=pd.Timestamp(2021,month,1);a=tr.loc[tr.timestamp.lt(t)];b=tr.loc[tr.timestamp.ge(t)&tr.timestamp.lt(t+pd.offsets.MonthBegin(1))]
            base=b[["timestamp","date","month",common.TARGET,"lag1_maximum","lag1_last"]].rename(columns={common.TARGET:"actual"}).copy()
            base["split"]=f"inner_{month}";base["daily_maximum_weight"]=common.pw(b);base["profile_weight"]=1.;base["train_profile_overlap"]=False
            ma=common.fit("HGB",common.HGB,a,contract["features"]["A"]);fits+=1
            baseline.append(common.record(base,"fixed_HGB_A","HGB","A","fixed",ma.predict(b[contract["features"]["A"]])))
            baseline.append(common.record(base,"lag1","baseline","none","baseline",b.lag1_maximum.to_numpy()))
            for lam in WEIGHTS:
                mm,w=fitted(a,cols,lam);fits+=1;name="fixed_HGB_B" if lam==1 else f"weight_{lam:g}"
                rec=common.record(base,name,"HGB","B","weighted",mm.predict(b[cols]))
                candidates[lam].append(rec)
                training.append({"outer":split,"inner_month":month,"lambda":lam,"train_hours":len(a),"weight_sum":float(w.sum()),
                                 "complete_training_dates_weight":float(common.pw(a).sum()),"train_end":str(a.timestamp.max()),"validation_start":str(b.timestamp.min())})
        inn=pd.concat(baseline+[r for parts in candidates.values() for r in parts],ignore_index=True)
        inn["outer"]=split;all_inner.append(inn)
        mtab=common.metrics(inn).set_index(["method","period","scope"]);base_all=mtab.loc[("fixed_HGB_B","pooled","all")];base_peak=mtab.loc[("fixed_HGB_B","pooled","peak")]
        available=[]
        for lam in WEIGHTS:
            name="fixed_HGB_B" if lam==1 else f"weight_{lam:g}"
            allm=mtab.loc[(name,"pooled","all")];pk=mtab.loc[(name,"pooled","peak")]
            monthly=all(mtab.loc[(name,f"month:{month}",scope),"MAE"]<=mtab.loc[("fixed_HGB_B",f"month:{month}",scope),"MAE"]*1.05
                        for month in contract["inner_months"][split] for scope in ["all","peak"])
            feasible=bool(lam==1 or (allm.MAE<=base_all.MAE*1.01 and pk.mean_over-base_peak.mean_over<=base_peak.MAE*.05 and monthly))
            objective=max(pk.MAE/base_peak.MAE,pk.mean_under/base_peak.mean_under)
            trials.append({"outer":split,"lambda":lam,"overall_MAE":float(allm.MAE),"peak_MAE":float(pk.MAE),"peak_under":float(pk.mean_under),
                           "peak_over":float(pk.mean_over),"base_overall":float(base_all.MAE),"base_peak":float(base_peak.MAE),"base_under":float(base_peak.mean_under),
                           "base_over":float(base_peak.mean_over),"monthly_feasible":monthly,"feasible":feasible,"objective":float(objective)})
            if feasible:available.append((objective,lam))
        lam=min(available)[1]
        name="peak_weighted_HGB_B";mm,w=fitted(tr,cols,lam);fits+=1;path=common.MODEL/f"{split}_{name}.joblib";joblib.dump(mm,path,compress=3)
        pp=mm.predict(ev[cols]);assert np.allclose(pp,joblib.load(path).predict(ev[cols]),rtol=0,atol=1e-9)
        preds.append(common.record(common.evmeta(ev,split),name,"HGB","B","peak_weighted",pp))
        mr=common.manifest_row(mm,path,split,name,"HGB","B","peak_weighted",common.HGB,tr,"new_weighted_fit");mr["training_lambda"]=lam;mr["training_weight_sum"]=float(w.sum());man.append(mr)
        selected.append({"split":split,"method":name,"family":"HGB","input":"B","variant":"peak_weighted","parameters":common.HGB,"training_lambda":lam})
        print(f"{split} historical peak lambda={lam}; MAE/peak={common.obj(ev[common.TARGET],pp,common.pw(ev))}",flush=True)
    p=pd.concat(preds,ignore_index=True);tables,tab,sel,loo=common.assess(p,contract)
    for df,name in [(p,"predictions.csv"),(tables,"metrics.csv"),(tab,"comparison.csv"),(loo,"leave_one_date_out.csv"),
                    (pd.DataFrame(man),"model_manifest.csv"),(pd.concat(all_inner),"inner_predictions.csv"),
                    (pd.DataFrame(trials),"weight_trials.csv"),(pd.DataFrame(training),"training_weight_audit.csv")]:common.save(df,d,name)
    dump(selected,d/"configurations.json");dump(sel,d/"selection.json")
    dump({"status":"completed","completed_at":now(),"new_fits":fits,"new_optuna_trials":0,"weight_configurations":21,"seconds":time.perf_counter()-start,
          "script_sha256":sha(ROOT/"Modeling/scripts/redevelopment_compare.py"),"entrypoint_sha256":sha(Path(__file__)),"contract_sha256":sha(CF),
          "outputs_sha256":{p.name:sha(p) for p in d.iterdir() if p.suffix in [".csv",".json"] and p.name!="run.json"}},d/"run.json")
    print(tab.to_string(index=False),flush=True)
def later():
    contract=json.loads(common.CFILE.read_text(encoding="utf-8"));d=BASE/"r04";d.mkdir(parents=True,exist_ok=True);assert not (d/"run.json").exists()
    src=BASE/"r03b";sel=json.loads((src/"selection.json").read_text(encoding="utf-8"));name=sel["selected"]
    if name is None:
        dump({"status":"skipped","recorded_at":now(),"reason":"No eligible candidate in R02, R03 or R03b","differentiation_goal":"unresolved"},d/"run.json")
        print("R04 skipped: no eligible candidate",flush=True);return
    assert json.loads((src/"independent_verification.json").read_text(encoding="utf-8"))["status"]=="passed"
    cf=next(x for x in json.loads((src/"configurations.json").read_text(encoding="utf-8")) if x["split"]=="dev_jun" and x["method"]==name)
    dump({"recorded_at":now(),"method":name,"configuration":cf,"training_end":"2021-06-30 23:00:00","refit_july":False,
          "scope":"observed later-period redevelopment evaluation","parent_run_sha256":sha(src/"run.json")},d/"contract.json")
    f=common.data();tr=f.loc[f.timestamp.lt("2021-07-01")];ev=f.loc[f.timestamp.ge("2021-07-01")];cols=contract["features"]["B"]
    mm,w=fitted(tr,cols,cf["training_lambda"]);path=common.MODEL/f"later_{name}.joblib";joblib.dump(mm,path,compress=3)
    pp=mm.predict(ev[cols]);base=common.evmeta(ev,"later_jul_aug")
    preds=[common.record(base,name,"HGB","B","peak_weighted",pp)]
    old=read(ROOT/"Modeling/tables/m05/predictions.csv")
    for group,method in [("A","fixed_HGB_A"),("B","fixed_HGB_B"),("lag1","lag1")]:
        part=old.loc[old.pool.eq("common")&old.group.eq(group)];assert len(part)==len(ev)
        preds.append(common.record(base,method,"HGB" if group!="lag1" else "baseline",group,"fixed",part.prediction.to_numpy()))
    p=pd.concat(preds,ignore_index=True);tables,tab,sel,loo=common.assess(p,contract)
    mr=common.manifest_row(mm,path,"later_jul_aug",name,"HGB","B","peak_weighted",common.HGB,tr,"new_weighted_fit")
    mr["training_lambda"]=cf["training_lambda"];mr["training_weight_sum"]=float(w.sum())
    for df,n in [(p,"predictions.csv"),(tables,"metrics.csv"),(tab,"comparison.csv"),(loo,"leave_one_date_out.csv"),(pd.DataFrame([mr]),"model_manifest.csv")]:common.save(df,d,n)
    dump([cf],d/"configurations.json");dump(sel,d/"selection.json")
    dump({"status":"completed","completed_at":now(),"new_fits":1,"new_trials":0,"script_sha256":sha(ROOT/"Modeling/scripts/redevelopment_compare.py"),
          "entrypoint_sha256":sha(Path(__file__)),"outputs_sha256":{p.name:sha(p) for p in d.iterdir() if p.suffix in [".csv",".json"] and p.name!="run.json"}},d/"run.json")
    print(tab.to_string(index=False),flush=True)
def main():
    ap=argparse.ArgumentParser();ap.add_argument("--stage",choices=["freeze","run","later"],required=True);a=ap.parse_args()
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    {"freeze":freeze,"run":run,"later":later}[a.stage]()
if __name__=="__main__":main()


코드 등록 완료: `Modeling/scripts/redevelopment_peak.py`

### `Modeling/scripts/redevelopment_peak_verify.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Independent peak-weight arithmetic, selection and weighted-fit reproduction.

주요 함수: `independent_weight()`, `metric()`, `main()`.

In [81]:
%%submission_source Modeling/scripts/redevelopment_peak_verify.py
"""Independent peak-weight arithmetic, selection and weighted-fit reproduction."""
import argparse,json,math,sys
from pathlib import Path
from collections import defaultdict
import numpy as np,pandas as pd,joblib
from sklearn.ensemble import HistGradientBoostingRegressor
from m01_prepare import ROOT,sha
import redevelopment_verify as v
from redevelopment_r01 import now
BASE=ROOT/"Modeling/tables/redevelopment"
CF=ROOT/"Modeling/config/redevelopment_peak_contract.json"
def independent_weight(frame,raw,lam):
    dates=defaultdict(list)
    for ts in frame.timestamp:
        t=ts.to_pydatetime();dates[t.date()].append(t)
    weights={}
    for day,times in dates.items():
        hi=max(raw[t]["maximum"] for t in times);ties=sum(raw[t]["maximum"]==hi for t in times)
        for t in times:weights[t]=1+(lam-1)/ties if len(times)==24 and raw[t]["maximum"]==hi else 1.
    return np.array([weights[ts.to_pydatetime()] for ts in frame.timestamp])
def metric(g):
    e=[float(r["prediction"])-float(r["actual"]) for r in g];w=[float(r["daily_maximum_weight"]) for r in g]
    return {"all":math.fsum(abs(x) for x in e)/len(e),"peak":v.avg([abs(x) for x in e],w),
            "under":v.avg([max(-x,0) for x in e],w),"over":v.avg([max(x,0) for x in e],w)}
def main():
    ap=argparse.ArgumentParser();ap.add_argument("--stage",choices=["r03b","r04"],required=True);stage=ap.parse_args().stage
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    d=BASE/stage;run=v.readj(d/"run.json");c=v.readj(CF)
    assert c["entrypoint_sha256"]==sha(ROOT/"Modeling/scripts/redevelopment_peak.py")
    assert c["helper_sha256"]==sha(ROOT/"Modeling/scripts/redevelopment_compare.py")
    assert c["source_r02"]==sha(BASE/"r02/run.json") and c["source_r03"]==sha(BASE/"r03/run.json")
    if run["status"]=="skipped":
        assert v.readj(BASE/"r03b/selection.json")["selected"] is None
        stats={"scope":"verified no eligible candidate; no later evaluation"}
    else:
        assert run["entrypoint_sha256"]==sha(ROOT/"Modeling/scripts/redevelopment_peak.py")
        if stage=="r03b":assert run["contract_sha256"]==sha(CF)
        raw=v.rawdict();stats=v.stages(stage,raw);f,checks=v.validframe(raw)
        if stage=="r03b":
            conf=v.readj(d/"configurations.json");inner=v.rows(d/"inner_predictions.csv");trials=v.rows(d/"weight_trials.csv")
            assert len(trials)==21
            for split in ["dev_apr","dev_may","dev_jun"]:
                by=defaultdict(list)
                for r in inner:
                    if r["outer"]==split:
                        ts=pd.Timestamp(r["timestamp"]).to_pydatetime()
                        v.close(r["actual"],raw[ts]["maximum"])
                        by[r["method"]].append(r)
                bm=metric(by["fixed_HGB_B"]);eligible=[]
                for tr in [r for r in trials if r["outer"]==split]:
                    lam=float(tr["lambda"]);name="fixed_HGB_B" if lam==1 else f"weight_{lam:g}"
                    cm=metric(by[name]);months={r["month"] for r in by[name]}
                    monthly=all(metric([r for r in by[name] if r["month"]==month])[scope]<=metric([r for r in by["fixed_HGB_B"] if r["month"]==month])[scope]*1.05
                                for month in months for scope in ["all","peak"])
                    feasible=lam==1 or (cm["all"]<=bm["all"]*1.01 and cm["over"]-bm["over"]<=bm["peak"]*.05 and monthly)
                    objective=max(cm["peak"]/bm["peak"],cm["under"]/bm["under"])
                    for col,expected in [("overall_MAE",cm["all"]),("peak_MAE",cm["peak"]),("peak_under",cm["under"]),("peak_over",cm["over"]),
                                         ("base_overall",bm["all"]),("base_peak",bm["peak"]),("base_under",bm["under"]),("base_over",bm["over"]),("objective",objective)]:
                        v.close(tr[col],expected)
                    assert (tr["monthly_feasible"]=="True")==monthly and (tr["feasible"]=="True")==feasible
                    if feasible:eligible.append((objective,lam))
                best=min(eligible)[1];cf=next(x for x in conf if x["split"]==split)
                v.close(cf["training_lambda"],best)
            for a in v.rows(d/"training_weight_audit.csv"):
                train=f.loc[f.timestamp.le(a["train_end"])]
                assert len(train)==int(a["train_hours"]) and pd.Timestamp(a["train_end"])<pd.Timestamp(a["validation_start"])
                w=independent_weight(train,raw,float(a["lambda"]))
                v.close(a["weight_sum"],sum(w))
                if float(a["lambda"])!=1:v.close(a["complete_training_dates_weight"],(sum(w)-len(w))/(float(a["lambda"])-1))
        # Refit each chosen outer model with independently reconstructed weights.
        predictions=v.rows(d/"predictions.csv");c2=v.readj(ROOT/"Modeling/config/redevelopment_r02_contract.json")
        for mr in v.rows(d/"model_manifest.csv"):
            train=f.loc[f.timestamp.le(mr["train_end"])]
            lam=float(mr["training_lambda"]);w=independent_weight(train,raw,lam);v.close(mr["training_weight_sum"],sum(w))
            cols=c2["features"]["B"];params=json.loads(mr["parameters"])
            m=HistGradientBoostingRegressor(**params,early_stopping=False,random_state=42)
            m.fit(train[cols],train.target_maximum,sample_weight=w)
            rec=[r for r in predictions if r["method"]==mr["method"] and r["split"]==mr["split"]]
            ev=f.set_index("timestamp").loc[pd.to_datetime([r["timestamp"] for r in rec])]
            got=m.predict(ev[cols]);assert np.allclose(got,[float(r["prediction"]) for r in rec],rtol=0,atol=1e-8)
        stats["independent_weighted_refits"]=len(v.rows(d/"model_manifest.csv"))
    out={"status":"passed","verified_at":now(),"run_sha256":sha(d/"run.json"),"script_sha256":sha(Path(__file__)),**stats}
    (d/"independent_verification.json").write_text(json.dumps(out,ensure_ascii=False,indent=2),encoding="utf-8")
    print(json.dumps(out),flush=True)
if __name__=="__main__":main()


코드 등록 완료: `Modeling/scripts/redevelopment_peak_verify.py`

### `Modeling/scripts/redevelopment_r01.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: R01: paired A/B errors by forecast-origin information; no model fitting.

주요 함수: `now()`, `read()`, `dump()`, `save()`, `parents()`, `freeze()`, `paired()`, `summary()`, `main()`.

In [82]:
%%submission_source Modeling/scripts/redevelopment_r01.py
"""R01: paired A/B errors by forecast-origin information; no model fitting."""
import argparse, json, sys
from pathlib import Path
from datetime import datetime, timezone, timedelta
import numpy as np
import pandas as pd
from m01_prepare import ROOT, read_contract, sha, load_frame, independent_check

OUT = ROOT / "Modeling/tables/redevelopment/r01"
CFILE = ROOT / "Modeling/config/redevelopment_r01_contract.json"
def now():
    return datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes")
def read(path):
    return pd.read_csv(path, encoding="utf-8-sig", float_precision="round_trip", parse_dates=["timestamp"])
def dump(value,path):
    path.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding="utf-8")
def save(df,name):
    df.to_csv(OUT / name,index=False,encoding="utf-8-sig")
def parents():
    paths = ["Modeling/config/m01_contract.json","Modeling/tables/m01/hourly_frame.csv",
             "Modeling/tables/m03/ab/predictions.csv","Modeling/tables/m03/ab/run.json",
             "Modeling/tables/m03/ab/independent_verification.json",
             "Modeling/tables/m05/predictions.csv","Modeling/tables/m05/run.json",
             "Modeling/tables/m05/independent_verification.json",
             "Modeling/scripts/redevelopment_r01.py"]
    c=read_contract()
    paths.append(c["source"])
    for folder in ["m03/ab","m05"]:
        dr=ROOT/"Modeling/tables"/folder
        run=json.loads((dr/"run.json").read_text(encoding="utf-8"))
        ver=json.loads((dr/"independent_verification.json").read_text(encoding="utf-8"))
        assert run["status"]=="completed" and ver["status"]=="passed"
        assert ver["run_sha256"]==sha(dr/"run.json")
        assert run["outputs_sha256"]["predictions.csv"]==sha(dr/"predictions.csv")
    return {p:sha(ROOT/p) for p in paths}
def freeze():
    assert not CFILE.exists()
    dump({"version":"redevelopment-r01-v1","recorded_at":now(),
          "reason":"M05 A/B peak loss contradicts M03 development gain; diagnose using existing predictions before new fits.",
          "inputs_sha256":parents(),"new_fits":0,
          "splits":["dev_apr","dev_may","dev_jun","later_jul_aug"],
          "bins":"q1/3,q2/3 of positive prior level and range in each outer training interval only; include zero separately",
          "conditions":["prior_state","level_bin","range_bin","prior_trend","last_gap_bin","train_profile_overlap","month"],
          "last_gap":"prior maximum minus prior last; flat <=0, else prior training positive-gap median",
          "weight":"existing complete-date tied-maximum weights; equal dates",
          "interpretation":"exploratory diagnosis, no conditional model selection, later period already observed"},CFILE)
def paired():
    c=read_contract()
    raw,invalid,valid,f=load_frame(c)
    independent_check(c,raw,invalid,valid,f)
    raw_index=valid.set_index("timestamp")
    previous=raw_index.reindex(pd.DatetimeIndex(f.timestamp-pd.Timedelta(hours=1)))
    f["prior_range"]=previous[["15분","30분","45분","60분"]].max(axis=1).to_numpy()-previous[["15분","30분","45분","60분"]].min(axis=1).to_numpy()
    f["prior_slot_change"]=previous["60분"].to_numpy()-previous["15분"].to_numpy()
    f["last_gap"]=f.lag1_maximum-f.lag1_last
    f["prior_state"]=np.select([f.lag1_maximum.eq(0),np.floor(f.lag1_mean+.5).between(20,26),f.lag1_mean.lt(20)],
                                 ["zero","low20_26","positive_below20"],default="above26")
    f["prior_trend"]=np.select([f.prior_slot_change.gt(0),f.prior_slot_change.lt(0)],["up","down"],default="flat")
    dev=read(ROOT/"Modeling/tables/m03/ab/predictions.csv")
    dev=dev.loc[dev.target.eq("target_maximum") & dev.group.isin(["A","B"])].copy()
    later=read(ROOT/"Modeling/tables/m05/predictions.csv")
    later=later.loc[later.pool.eq("common") & later.group.isin(["A","B"])].copy()
    later["split"]="later_jul_aug"
    p=pd.concat([dev,later],ignore_index=True)
    keys=["timestamp","split"]
    metadata=["actual","daily_maximum_weight","profile_weight","train_profile_overlap"]
    a=p.loc[p.group.eq("A"),keys+metadata+["prediction"]].rename(columns={"prediction":"prediction_A"})
    b=p.loc[p.group.eq("B"),keys+["prediction"]].rename(columns={"prediction":"prediction_B"})
    pair=a.merge(b,on=keys,validate="one_to_one").merge(f,on="timestamp",validate="one_to_one")
    assert len(pair)==3528 and np.array_equal(pair.actual,pair.target_maximum)
    thresholds=[]
    splits={s["name"]:s for s in c["splits"]}
    for split,part in pair.groupby("split"):
        train_end = "2021-06-30 23:00:00" if split=="later_jul_aug" else splits[split]["train_end"]
        tr=f.loc[f.eligible_common & f.timestamp.le(train_end)]
        assert tr.timestamp.max()<part.timestamp.min()
        qs={}
        for field,name in [("lag1_mean","level"),("prior_range","range")]:
            values=tr.loc[tr[field].gt(0),field]
            q=values.quantile([1/3,2/3]).tolist()
            qs[name]=q
            pair.loc[part.index,f"{name}_bin"]=np.select([part[field].eq(0),part[field].le(q[0]),part[field].le(q[1])],["zero","lower","middle"],default="upper")
        positive=tr.loc[tr.last_gap.gt(0),"last_gap"]
        median=float(positive.median())
        pair.loc[part.index,"last_gap_bin"]=np.select([part.last_gap.le(0),part.last_gap.le(median)],["at_maximum","small_gap"],default="large_gap")
        thresholds.append({"split":split,"train_end":train_end,"train_hours":len(tr),"level_q33":qs["level"][0],"level_q67":qs["level"][1],
                           "range_q33":qs["range"][0],"range_q67":qs["range"][1],"gap_median":median})
    for group in ["A","B"]:
        err=pair[f"prediction_{group}"]-pair.actual
        pair[f"abs_{group}"]=abs(err)
        pair[f"under_{group}"]=(-err).clip(lower=0)
        pair[f"over_{group}"]=err.clip(lower=0)
        pair[f"bias_{group}"]=err
    pair["delta_absolute_error"]=pair.abs_B-pair.abs_A
    pair["date"]=pair.timestamp.dt.strftime("%Y-%m-%d")
    return pair,pd.DataFrame(thresholds)
def summary(pair):
    rows=[]
    for period, part in [("development",pair.loc[pair.month.le(6)]),("later",pair.loc[pair.month.ge(7)])]:
        for col in ["all","prior_state","level_bin","range_bin","prior_trend","last_gap_bin","train_profile_overlap","month"]:
            sections=[("all",part)] if col=="all" else list(part.groupby(col))
            for value,g in sections:
                for scope in ["all","peak"]:
                    w=np.ones(len(g)) if scope=="all" else g.daily_maximum_weight.to_numpy()
                    if w.sum()==0: continue
                    row={"period":period,"condition":col,"value":str(value),"scope":scope,"hours":len(g) if scope=="all" else int((w>0).sum()),
                         "days":g.loc[w>0,"date"].nunique(),"weight_sum":float(w.sum())}
                    for label in ["A","B"]:
                        for metric in ["abs","under","over","bias"]:
                            row[f"{metric}_{label}"]=float(np.average(g[f"{metric}_{label}"],weights=w))
                    row["delta_MAE"]=row["abs_B"]-row["abs_A"]
                    row["unweighted_delta_sum"]=float(g.delta_absolute_error.sum())
                    rows.append(row)
    return pd.DataFrame(rows)
def main():
    parser=argparse.ArgumentParser();parser.add_argument("--stage",choices=["freeze","run"],required=True)
    args=parser.parse_args()
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    if args.stage=="freeze": freeze();print("R01 contract recorded",flush=True);return
    OUT.mkdir(parents=True,exist_ok=True);assert not (OUT/"run.json").exists()
    c=json.loads(CFILE.read_text(encoding="utf-8")); assert c["inputs_sha256"]==parents()
    pair,thresholds=paired()
    save(pair,"paired.csv");save(thresholds,"thresholds.csv");save(summary(pair),"conditions.csv")
    daily=pair.groupby(["split","date","month"]).agg(hours=("timestamp","size"),abs_A=("abs_A","mean"),abs_B=("abs_B","mean"),
                                                   under_A=("under_A","mean"),under_B=("under_B","mean"),delta_sum=("delta_absolute_error","sum")).reset_index()
    save(daily,"daily_errors.csv")
    loo=[]
    for period,g in pair.groupby(pair.month.le(6).map({True:"development",False:"later"})):
        for day in sorted(g.date.unique()):
            rem=g.loc[g.date.ne(day)]
            for scope in ["all","peak"]:
                w=np.ones(len(rem)) if scope=="all" else rem.daily_maximum_weight
                loo.append({"period":period,"excluded_date":day,"scope":scope,"delta_MAE":float(np.average(rem.delta_absolute_error,weights=w))})
    save(pd.DataFrame(loo),"leave_one_date_out.csv")
    run={"status":"completed","completed_at":now(),"new_fits":0,"pairs":len(pair),"contract_sha256":sha(CFILE),"inputs_sha256":c["inputs_sha256"],
         "outputs_sha256":{p.name:sha(p) for p in OUT.glob("*.csv")}}
    dump(run,OUT/"run.json")
    print(summary(pair).loc[lambda d:d.condition.eq("all")].to_string(index=False),flush=True)
if __name__=="__main__":main()


코드 등록 완료: `Modeling/scripts/redevelopment_r01.py`

### `Modeling/scripts/redevelopment_verify.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Independent redevelopment checks: raw timestamps, weighted errors, trials and saved model predictions.

주요 함수: `rows()`, `close()`, `avg()`, `readj()`, `dump()`, `rawdict()`, `validframe()`, `hashes()`, `r01()`, `stages()`, `main()`.

In [83]:
%%submission_source Modeling/scripts/redevelopment_verify.py
"""Independent redevelopment checks: raw timestamps, weighted errors, trials and saved model predictions."""
import argparse,csv,json,math,sys
from collections import defaultdict
from datetime import datetime,timedelta
from pathlib import Path
import numpy as np,pandas as pd,joblib
from m01_prepare import ROOT,sha
from redevelopment_r01 import now
BASE=ROOT/"Modeling/tables/redevelopment"
def rows(path):
    with path.open(encoding="utf-8-sig",newline="") as h:return list(csv.DictReader(h))
def close(a,b):
    assert math.isclose(float(a),float(b),abs_tol=1e-8,rel_tol=1e-10),(a,b)
def avg(v,w):
    return math.fsum(x*y for x,y in zip(v,w))/math.fsum(w)
def readj(path):return json.loads(path.read_text(encoding="utf-8"))
def dump(v,path):path.write_text(json.dumps(v,ensure_ascii=False,indent=2),encoding="utf-8")
def rawdict():
    c=readj(ROOT/"Modeling/config/m01_contract.json")
    assert sha(ROOT/c["source"])==c["source_sha256"]
    out={}
    for r in rows(ROOT/c["source"]):
        d,h=int(r["날짜"]),int(r["시간"])
        if 20210101<=d<=20210831 and 0<=h<=23:
            t=datetime.strptime(str(d),"%Y%m%d")+timedelta(hours=h)
            vals=[float(r[k]) for k in ["15분","30분","45분","60분"]]
            out[t]={"slots":vals,"mean":sum(vals)/4,"maximum":max(vals),"last":vals[-1],"production":float(r["생산량"])}
    return out
def validframe(raw):
    f=pd.read_csv(ROOT/"Modeling/tables/m01/hourly_frame.csv",encoding="utf-8-sig",float_precision="round_trip",parse_dates=["timestamp"])
    f=f.loc[f.eligible_common]
    checked=0
    for r in f.itertuples():
        ts=r.timestamp.to_pydatetime();own=raw[ts]
        close(r.target_maximum,own["maximum"])
        for lag in [1,2,24,168]:
            prior=raw[ts-timedelta(hours=lag)]
            for col in ["mean","maximum","last","production"]:
                close(getattr(r,f"lag{lag}_{col}"),prior[col]);checked+=1
        close(r.hour_sin,math.sin(2*math.pi*ts.hour/24));close(r.hour_cos,math.cos(2*math.pi*ts.hour/24))
        close(r.weekend,int(ts.weekday()>=5))
        close(r.lag1_production_zero,int(raw[ts-timedelta(hours=1)]["production"]==0))
        for i in range(7):close(getattr(r,f"dow_{i}"),int(ts.weekday()==i))
    return f,checked
def hashes(d):
    run=readj(d/"run.json")
    assert run["status"]=="completed"
    for name,h in run["outputs_sha256"].items():assert sha(d/name)==h,(d,name)
    for name,h in run.get("inputs_sha256",{}).items():assert sha(ROOT/name)==h,name
    return run
def r01(raw):
    d=BASE/"r01";run=hashes(d);c=readj(ROOT/"Modeling/config/redevelopment_r01_contract.json")
    assert sha(ROOT/"Modeling/config/redevelopment_r01_contract.json")==run["contract_sha256"]
    for p,h in c["inputs_sha256"].items():assert sha(ROOT/p)==h
    f,checks=validframe(raw)
    pairs=rows(d/"paired.csv");assert len(pairs)==3528
    old={}
    for path in ["Modeling/tables/m03/ab/predictions.csv","Modeling/tables/m05/predictions.csv"]:
        for r in rows(ROOT/path):
            if r["group"] in ["A","B"] and r.get("target","target_maximum")=="target_maximum" and r.get("pool","common")=="common":
                old[(r["timestamp"],r["group"])]=r
    for r in pairs:
        ts=datetime.fromisoformat(r["timestamp"]);prior=raw[ts-timedelta(hours=1)];slots=prior["slots"]
        close(r["actual"],raw[ts]["maximum"]);close(r["prior_range"],max(slots)-min(slots));close(r["prior_slot_change"],slots[-1]-slots[0])
        close(r["last_gap"],max(slots)-slots[-1])
        state="zero" if max(slots)==0 else "low20_26" if 20<=math.floor(prior["mean"]+.5)<=26 else "positive_below20" if prior["mean"]<20 else "above26"
        assert r["prior_state"]==state
        assert r["prior_trend"]==("up" if slots[-1]>slots[0] else "down" if slots[-1]<slots[0] else "flat")
        for group in ["A","B"]:
            close(r[f"prediction_{group}"],old[(r["timestamp"],group)]["prediction"])
            e=float(r[f"prediction_{group}"])-float(r["actual"])
            for name,value in [("abs",abs(e)),("under",max(-e,0)),("over",max(e,0)),("bias",e)]:close(r[f"{name}_{group}"],value)
        close(r["delta_absolute_error"],float(r["abs_B"])-float(r["abs_A"]))
    thresholds={r["split"]:r for r in rows(d/"thresholds.csv")}
    for split,t in thresholds.items():
        train=f.loc[f.timestamp.le(t["train_end"])]
        assert len(train)==int(t["train_hours"])
        for field,prefix in [("lag1_mean","level"),("prior_range","range")]:
            vals=[]
            for ts in train.timestamp:
                prior=raw[ts.to_pydatetime()-timedelta(hours=1)]
                v=prior["mean"] if field=="lag1_mean" else max(prior["slots"])-min(prior["slots"])
                if v>0:vals.append(v)
            qs=np.quantile(vals,[1/3,2/3]);close(t[prefix+"_q33"],qs[0]);close(t[prefix+"_q67"],qs[1])
        gaps=[raw[ts.to_pydatetime()-timedelta(hours=1)]["maximum"]-raw[ts.to_pydatetime()-timedelta(hours=1)]["last"] for ts in train.timestamp]
        close(t["gap_median"],np.median([x for x in gaps if x>0]))
    for r in pairs:
        t=thresholds[r["split"]]
        for field,prefix in [("lag1_mean","level"),("prior_range","range")]:
            v=float(r[field]);label="zero" if v==0 else "lower" if v<=float(t[prefix+"_q33"]) else "middle" if v<=float(t[prefix+"_q67"]) else "upper"
            assert r[prefix+"_bin"]==label
        gap=float(r["last_gap"]);assert r["last_gap_bin"]==("at_maximum" if gap<=0 else "small_gap" if gap<=float(t["gap_median"]) else "large_gap")
    for m in rows(d/"conditions.csv"):
        selected=[r for r in pairs if (int(r["month"])<=6)==(m["period"]=="development") and (m["condition"]=="all" or r[m["condition"]]==m["value"])]
        w=[float(r["daily_maximum_weight"]) if m["scope"]=="peak" else 1. for r in selected]
        close(m["weight_sum"],sum(w))
        assert int(m["hours"])==sum(x>0 for x in w)
        for group in ["A","B"]:
            for metric in ["abs","under","over","bias"]:close(m[f"{metric}_{group}"],avg([float(r[f"{metric}_{group}"]) for r in selected],w))
        close(m["delta_MAE"],float(m["abs_B"])-float(m["abs_A"]))
    for m in rows(d/"leave_one_date_out.csv"):
        selected=[r for r in pairs if (int(r["month"])<=6)==(m["period"]=="development") and r["date"]!=m["excluded_date"]]
        w=[float(r["daily_maximum_weight"]) if m["scope"]=="peak" else 1. for r in selected]
        close(m["delta_MAE"],avg([float(r["delta_absolute_error"]) for r in selected],w))
    return {"pairs":len(pairs),"raw_lag_values":checks,"conditions":len(rows(d/"conditions.csv"))}
def stages(stage,raw):
    d=BASE/stage;run=hashes(d);c=readj(ROOT/"Modeling/config/redevelopment_r02_contract.json")
    for p,h in c["inputs_sha256"].items():assert sha(ROOT/p)==h,p
    assert run["script_sha256"]==sha(ROOT/"Modeling/scripts/redevelopment_compare.py")
    if stage=="r02":assert run["contract_sha256"]==sha(ROOT/"Modeling/config/redevelopment_r02_contract.json")
    f,checks=validframe(raw);idx=f.set_index("timestamp")
    p=rows(d/"predictions.csv");groups=defaultdict(list)
    for r in p:
        ts=datetime.fromisoformat(r["timestamp"]);close(r["actual"],raw[ts]["maximum"])
        close(r["lag1_last"],raw[ts-timedelta(hours=1)]["last"])
        close(r["lag1_maximum"],raw[ts-timedelta(hours=1)]["maximum"])
        e=float(r["prediction"])-float(r["actual"])
        for name,val in [("absolute_error",abs(e)),("under_amount",max(-e,0)),("over_amount",max(e,0)),("bias",e)]:close(r[name],val)
        groups[r["method"]].append(r)
    for name,g in groups.items():
        assert len({r["timestamp"] for r in g})==len(g)
        dates=defaultdict(list)
        for r in g:dates[r["date"]].append(r)
        for day,part in dates.items():
            hi=max(float(r["actual"]) for r in part);ties=sum(float(r["actual"])==hi for r in part)
            for r in part:close(r["daily_maximum_weight"],1/ties if len(part)==24 and float(r["actual"])==hi else 0)
    for m in rows(d/"metrics.csv"):
        g=groups[m["method"]];period=m["period"];scope=m["scope"]
        if period!="pooled":
            if period.startswith("month:"):g=[r for r in g if r["month"]==period.split(":")[1]]
            elif period.startswith("train_profile_overlap:"):g=[r for r in g if r["train_profile_overlap"]==period.split(":")[1]]
            elif period=="surge93":g=[r for r in g if float(r["actual"])-float(r["lag1_maximum"])>=93]
            elif period=="zero_actual":g=[r for r in g if float(r["actual"])==0]
            elif period=="prior_zero":g=[r for r in g if float(r["lag1_maximum"])==0]
            else:g=[r for r in g if r["split"]==period]
        assert len(g)==int(m["hours"])
        w=[float(r["daily_maximum_weight"]) if scope=="peak" else float(r["profile_weight"]) if scope=="profile_reweighted" else 1. for r in g]
        close(m["weight_sum"],sum(w))
        err=[float(r["prediction"])-float(r["actual"]) for r in g]
        for name,v in [("MAE",[abs(x) for x in err]),("mean_under",[max(-x,0) for x in err]),("mean_over",[max(x,0) for x in err]),("bias",err),
                       ("mean_actual",[float(r["actual"]) for r in g])]:close(m[name],avg(v,w))
    conf=readj(d/"configurations.json")
    for mr in rows(d/"model_manifest.csv"):
        path=ROOT/mr["file"];assert sha(path)==mr["sha256"]
        mm=joblib.load(path);params=json.loads(mr["parameters"])
        for k,v in params.items():assert mm.get_params()[k]==v
        cols=c["features"][mr["input"]];assert list(mm.feature_names_in_)==cols
        records=[r for r in groups[mr["method"]] if r["split"]==mr["split"]]
        ev=idx.loc[pd.to_datetime([r["timestamp"] for r in records])]
        got=mm.predict(ev[cols])+(ev.lag1_last.to_numpy() if mr["variant"]=="delta" else 0)
        assert np.allclose(got,[float(r["prediction"]) for r in records],rtol=0,atol=1e-8)
        eligible=f.loc[f.timestamp.le(mr["train_end"])]
        assert len(eligible)==int(mr["train_hours"]) and eligible.timestamp.max()<ev.index.min()
    if stage=="r02":
        trials=rows(d/"trials.csv");folds=rows(d/"trial_fold_scores.csv");assert len(trials)==360
        studies=defaultdict(list)
        for t in trials:
            key=(t["split"],t["family"],t["input"]);studies[key].append(t)
            selected=[r for r in folds if (r["split"],r["family"],r["input"],r["trial"])==(*key,t["number"])]
            close(t["overall_MAE"],avg([float(r["overall_MAE"]) for r in selected],[int(r["hours"]) for r in selected]))
            close(t["peak_MAE"],avg([float(r["peak_MAE"]) for r in selected],[float(r["peak_weight_sum"]) for r in selected]))
            close(t["balanced_score"],max(float(t["overall_MAE"])/float(t["baseline_overall"]),float(t["peak_MAE"])/float(t["baseline_peak"])))
        for key,triallist in studies.items():
            assert len(triallist)==30 and {int(t["number"]) for t in triallist}==set(range(30))
            best=min(triallist,key=lambda t:(float(t["balanced_score"]),int(t["number"])))
            assert best["selected"]=="True"
        for a in rows(d/"time_audit.csv"):
            assert a["train_end"]<a["validation_start"]<=a["validation_end"]
            parent=readj(ROOT/"Modeling/config/m01_contract.json")
            outer=next(s for s in parent["splits"] if s["name"]==a["split"])
            assert pd.Timestamp(a["validation_end"])<pd.Timestamp(outer["eval_start"])
        inner=rows(d/"selected_inner_predictions.csv")
        by=defaultdict(list)
        for r in inner:by[(r["split"],r["method"])].append(r)
        for cf in conf:
            if cf["variant"]!="tuned":continue
            g=by[(cf["split"],cf["method"])];err=[abs(float(r["prediction"])-float(r["actual"])) for r in g]
            close(cf["inner_objectives"][0],math.fsum(err)/len(err));close(cf["inner_objectives"][1],avg(err,[float(r["daily_maximum_weight"]) for r in g]))
    # Independently rederive declaration gates from verified metric arithmetic and date omissions.
    ms={(r["method"],r["period"],r["scope"]):r for r in rows(d/"metrics.csv")}
    selection=readj(d/"selection.json")
    leave=rows(d/"leave_one_date_out.csv")
    for candidate in selection["candidate_gates"]:
        name=candidate["method"];flags={};a=ms[(name,"pooled","all")];pk=ms[(name,"pooled","peak")]
        for ref in c["gates"]["comparison"]:
            ra=ms[(ref,"pooled","all")];rp=ms[(ref,"pooled","peak")]
            flags[ref+"_overall"]=float(a["MAE"])<=float(ra["MAE"])*1.01
            flags[ref+"_peak"]=float(pk["MAE"])<=float(rp["MAE"])*.95
            flags[ref+"_under"]=float(pk["mean_under"])<=float(rp["mean_under"])*.95
            flags[ref+"_over"]=float(pk["mean_over"])-float(rp["mean_over"])<=float(rp["MAE"])*.05
            lhs={r["timestamp"]:r for r in groups[name]};rhs={r["timestamp"]:r for r in groups[ref]}
            diffs=[]
            for day in {r["date"] for r in groups[name]}:
                g=[r for r in groups[name] if r["date"]!=day];w=[float(r["daily_maximum_weight"]) for r in g]
                value=avg([float(r["absolute_error"])-float(rhs[r["timestamp"]]["absolute_error"]) for r in g],w)
                diffs.append(value)
                stored=next(r for r in leave if r["method"]==name and r["reference"]==ref and r["excluded_date"]==day)
                close(stored["peak_delta_MAE"],value)
            flags[ref+"_leave_one_date"]=max(diffs)<0
        flags["monthly"]=all(float(ms[(name,s,sc)]["MAE"])<=float(ms[("fixed_HGB_B",s,sc)]["MAE"])*1.05 for s in {f"month:{r[chr(109)+chr(111)+chr(110)+chr(116)+chr(104)]}" for r in groups[name]} for sc in ["all","peak"])
        assert flags==candidate["gates"] and candidate["eligible"]==all(flags.values())
        close(candidate["balanced_score"],max(float(a["MAE"])/float(ms[("lag1","pooled","all")]["MAE"]),float(pk["MAE"])/float(ms[("lag1","pooled","peak")]["MAE"])))
    eligible=[r for r in selection["candidate_gates"] if r["eligible"]]
    selected=min(eligible,key=lambda r:(r["balanced_score"],r["method"]))["method"] if eligible else None
    assert selection["selected"]==selected
    return {"prediction_rows":len(p),"raw_lag_values":checks,"models_reloaded":len(rows(d/"model_manifest.csv")),
            "metrics":len(rows(d/"metrics.csv")),"trials":360 if stage=="r02" else 0,"selection":selected}
def main():
    ap=argparse.ArgumentParser();ap.add_argument("--stage",choices=["r01","r02","r03","r04"],required=True);stage=ap.parse_args().stage
    assert sys.version_info[:2]==(3,13) and sys._is_gil_enabled()
    raw=rawdict();d=BASE/stage;run=readj(d/"run.json")
    if run["status"]=="skipped":
        dump({"status":"passed","scope":"recorded skip; no prediction result","run_sha256":sha(d/"run.json"),"script_sha256":sha(Path(__file__))},d/"independent_verification.json")
        print("Verified skip",stage);return
    stats=r01(raw) if stage=="r01" else stages(stage,raw)
    dump({"status":"passed","verified_at":now(),"run_sha256":sha(d/"run.json"),"script_sha256":sha(Path(__file__)),**stats},d/"independent_verification.json")
    print(json.dumps({"status":"passed","stage":stage,**stats}),flush=True)
if __name__=="__main__":main()


코드 등록 완료: `Modeling/scripts/redevelopment_verify.py`

### `Modeling/scripts/regime_age_ablation.py`

원고 연결: Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_원고.md.

코드 설명: Single predeclared classifier age removal; reuse frozen HGB forecasts.

Development first. Follow-up is conditional and explicitly exploratory.
No completed parent artifact is edited, no hyperparameter or threshold search.

주요 함수: `inputs()`, `freeze()`, `classify_frame()`, `class_summary()`, `regress_frame()`, `regress_summary()`, `gates()`, `run()`.

In [84]:
%%submission_source Modeling/scripts/regime_age_ablation.py
"""Single predeclared classifier age removal; reuse frozen HGB forecasts.

Development first. Follow-up is conditional and explicitly exploratory.
No completed parent artifact is edited, no hyperparameter or threshold search.
"""
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import json
import sys
from datetime import datetime, timezone, timedelta
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from regime_forecast import FEATURES, load_frame, model, threshold, sha, save_json
from regime_followup import detection, metrics

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/regime_age_ablation"
MODELS = ROOT / "Modeling/models/regime_age_ablation"
FEATURES_NOAGE = [x for x in FEATURES if x != "prior_state_age"]


def inputs():
    names = ["Modeling/scripts/regime_forecast.py", "Modeling/scripts/regime_followup.py",
             "Modeling/tables/regime_diagnosis/hourly_labels.csv",
             "Modeling/tables/regime_forecast/contract.json", "Modeling/tables/regime_forecast/predictions.csv",
             "Modeling/tables/regime_routing/predictions.csv", "Modeling/tables/regime_routing/independent_verification.json",
             "Modeling/tables/regime_followup/contract.json", "Modeling/tables/regime_followup/run.json",
             "Modeling/tables/regime_followup/independent_verification.json",
             "Modeling/tables/regime_followup/classification_predictions.csv",
             "Modeling/tables/regime_followup/predictions.csv",
             "Modeling/tables/regime_followup/verified_alarm_conditions.csv"]
    return {n: sha(ROOT / n) for n in names}


def freeze():
    OUT.mkdir(parents=True, exist_ok=True)
    assert not (OUT / "contract.json").exists()
    c = {
        "stage": "S06 single age ablation", "created_at": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
        "entrypoint_sha256": sha(__file__), "inputs_sha256": inputs(),
        "reason": "S05 all38 August FP had age180..227 vs training max64, age largest positive logit. Test one classifier feature removal, not proof of a physical cause.",
        "change": "Remove prior_state_age only from upward Logistic classifier; retain original24 vs candidate23 inputs. HGB B0/B1 and their age input remain unchanged to isolate routing selection.",
        "features": FEATURES_NOAGE,
        "logistic": json.loads((ROOT / "Modeling/tables/regime_forecast/contract.json").read_text(encoding="utf-8"))["logistic"],
        "definition": "Frozen S01 three-hour persistent mean-level upward onset, predicted at t-1; target max only for HGB error evaluation.",
        "development": "Apr-Jun cumulative preceding-month training with label_confirmed_at < boundary; Apr ranking, May-Jun automatic alarms. Min8 events/5 dates. Threshold from earlier matured OOF only, maximizeTP/minFP/maxthreshold under1%calibrationFPR. No method selection/tuning.",
        "development_gate": "On May-Jun min5 events, recall>=.8, precision>=.8, FPR<=.01; then G1 upMAE and under reduction>=5%vsB0, overall and daily peak worsening<=1%. Monthly results diagnostic, no all-month victory demand.",
        "conditional_followup": "Only if BOTH development gates pass: one Jan-Jun mature-label fit; threshold computed once from candidate Apr-Jun OOF scores; apply fixed model/threshold Jul-Aug without July refit. Reuse S05 HGB predictions. Same gates; already-observed exploratory reuse, no independent confirmation claim.",
        "scope": "Hypothesis selected after observing S05. Development and follow-up are exploratory; no new independent test.",
        "controls": ["original Logistic", "B0", "B1_all", "original G1", "candidate G1", "hour08"],
        "coverage": "Use exactly parent forecast rows, retain unknown future labels for regression but exclude them from classification; no record deletion, age clipping or label changes.",
        "stop": "Exactly one candidate. Preserve failed results and stop this comparison if gate fails; do not try age transforms or tune boundaries in same run.",
        "root_readme_sha256": sha(ROOT / "README.md"),
        "manuscript_sha256": sha(ROOT / "Modeling/04_Modeling_원고.md"),
    }
    save_json(OUT / "contract.json", c)
    print(json.dumps({"status": "frozen", "contract_sha256": sha(OUT / "contract.json")}), flush=True)


def classify_frame(test, score, th, old):
    z = test[["timestamp", "label_confirmed_at", "month", "profile", "sustained_onset"]].copy()
    z["event"] = (z.sustained_onset == 1).astype(int)
    z["score"] = score; z["threshold"] = th
    z["alarm"] = (score >= th).astype(int) if th is not None else np.nan
    z["method"] = "noage"
    baseline = old.set_index("timestamp").loc[z.timestamp]
    b = z.copy()
    for name in ["score", "threshold", "alarm"]:
        b[name] = baseline[name].to_numpy()
    b["method"] = "original"
    return pd.concat([z, b], ignore_index=True)


def class_summary(p, common, period):
    rows = []
    for scope, part in [("six_history", p), ("common_regression", p.loc[p.timestamp.isin(common)])]:
        part = part.loc[part.event.notna() & part.alarm.notna()]
        for subperiod, gg in [(period, part)] + [(str(m), g) for m, g in part.groupby("month")]:
            for method, g in gg.groupby("method"):
                rows.append({"scope": scope, "period": subperiod, "method": method, **detection(g)})
    return pd.DataFrame(rows)


def regress_frame(parent, cp, late=False):
    b = parent.loc[parent.variant == "B0"].copy().sort_values("timestamp")
    other = parent.loc[parent.variant == "B1_all"].set_index("timestamp").loc[b.timestamp]
    old = parent.loc[parent.variant == "G1"].set_index("timestamp").loc[b.timestamp]
    hour = parent.loc[parent.variant == "hour08"].set_index("timestamp").loc[b.timestamp]
    assert np.allclose(b.actual, other.actual)
    alarm = cp.loc[cp.method == "noage"].set_index("timestamp").loc[b.timestamp, "alarm"].to_numpy(int)
    options = {"B0": b.prediction.to_numpy(), "B1_all": other.prediction.to_numpy(),
               "G1_old": old.prediction.to_numpy(), "G1_noage": np.where(alarm, other.prediction, b.prediction),
               "hour08": hour.prediction.to_numpy()}
    masks = {"B0": np.zeros(len(b), int), "B1_all": np.ones(len(b), int),
             "G1_old": old.routed.to_numpy(int), "G1_noage": alarm,
             "hour08": hour.routed.to_numpy(int)}
    parts = []
    cols = ["timestamp", "month", "date", "profile", "sustained_onset", "actual", "seen_regression_profile"]
    for variant, values in options.items():
        g = b[cols].copy(); g["variant"] = variant; g["prediction"] = values; g["routed"] = masks[variant]
        parts.append(g)
    return pd.concat(parts, ignore_index=True)


def regress_summary(p, period):
    rows = []; daily = []; windows = []; burden = []
    for variant, g in p.groupby("variant"):
        for date, day in g.groupby("date"):
            if len(day) == 24:
                peak = day.loc[day.actual == day.actual.max()]
                daily.append({"variant": variant, "date": date, "month": int(day.month.iloc[0]), "ties": len(peak), **metrics(peak)})
        for r in g.loc[g.sustained_onset == 1].itertuples():
            w = g.loc[g.timestamp.isin(pd.date_range(r.timestamp, periods=3, freq="h"))]
            if len(w) == 3:
                windows.append({"variant": variant, "timestamp": r.timestamp, "month": r.month, **metrics(w)})
    dd = pd.DataFrame(daily); ww = pd.DataFrame(windows)
    for variant, g in p.groupby("variant"):
        for subperiod, part in [(period, g)] + [(str(m), gg) for m, gg in g.groupby("month")]:
            for condition, gg in [("all", part), ("up_start", part.loc[part.sustained_onset == 1]),
                                  ("down_start", part.loc[part.sustained_onset == -1])]:
                rows.append({"variant": variant, "period": subperiod, "condition": condition, **metrics(gg)})
            for condition, source in [("daily_peak_day_equal", dd), ("up_window_event_equal", ww)]:
                a = source.loc[source.variant == variant]
                if subperiod != period:
                    a = a.loc[a.month == int(subperiod)]
                stat = {n: float(a[n].mean()) if len(a) else None for n in ["mae", "under", "over", "bias"]}
                stat["rmse"] = float(np.sqrt(np.mean(a.rmse ** 2))) if len(a) else None
                rows.append({"variant": variant, "period": subperiod, "condition": condition,
                             "hours": int(a.ties.sum()) if condition.startswith("daily") else len(a) * 3, "units": len(a), **stat})
            burden.append({"variant": variant, "period": subperiod, "hours": len(part), "routed": int(part.routed.sum()),
                           "routed_up": int(((part.routed == 1) & (part.sustained_onset == 1)).sum()),
                           "routed_non_up_known": int(((part.routed == 1) & part.sustained_onset.notna() & (part.sustained_onset != 1)).sum()),
                           "routed_unknown": int(((part.routed == 1) & part.sustained_onset.isna()).sum())})
    return pd.DataFrame(rows), dd, ww, pd.DataFrame(burden)


def gates(det, reg, period):
    a = det.loc[(det.scope == "common_regression") & (det.period == period) & (det.method == "noage")].iloc[0]
    def get(v, cond):
        return reg.loc[(reg.variant == v) & (reg.period == period) & (reg.condition == cond)].iloc[0]
    start_gain = 1 - get("G1_noage", "up_start").mae / get("B0", "up_start").mae
    under_gain = 1 - get("G1_noage", "up_start").under / get("B0", "up_start").under
    overall = get("G1_noage", "all").mae / get("B0", "all").mae - 1
    peak = get("G1_noage", "daily_peak_day_equal").mae / get("B0", "daily_peak_day_equal").mae - 1
    return {"classification_passed": bool(a.events >= 5 and a.recall >= .8 and pd.notna(a.precision) and a.precision >= .8 and a.fpr <= .01),
            "routing_passed": bool(a.events >= 5 and start_gain >= .05 and under_gain >= .05 and overall <= .01 and peak <= .01),
            "start_mae_reduction": float(start_gain), "under_reduction": float(under_gain),
            "overall_mae_change": float(overall), "daily_peak_mae_change": float(peak)}


def run():
    assert not (OUT / "run.json").exists()
    c = json.loads((OUT / "contract.json").read_text(encoding="utf-8"))
    assert c["entrypoint_sha256"] == sha(__file__) and c["inputs_sha256"] == inputs()
    f = load_frame(); base_contract = json.loads((ROOT / "Modeling/tables/regime_forecast/contract.json").read_text(encoding="utf-8"))
    old = pd.read_csv(ROOT / "Modeling/tables/regime_forecast/predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at"])
    old = old.loc[(old.direction == "up") & (old.method == "logistic") & (old.month >= 4)]
    MODELS.mkdir(parents=True, exist_ok=True)
    manifest = []; calrows = []; devparts = []; history = []
    def fit(train, test, name, boundary):
        y = (train.sustained_onset == 1).astype(int)
        assert y.sum() >= 8 and train.loc[y == 1, "date"].nunique() >= 5
        assert (train.label_confirmed_at < boundary).all()
        fitted = model("logistic", base_contract); fitted.fit(train[c["features"]], y)
        path = MODELS / (name + ".joblib"); assert not path.exists(); joblib.dump(fitted, path)
        manifest.append({"name": name, "file": path.relative_to(ROOT).as_posix(), "sha256": sha(path),
                         "boundary": str(boundary), "train_hours": len(train), "train_events": int(y.sum()),
                         "train_latest_confirmed": str(train.label_confirmed_at.max()), "features": "|".join(c["features"])})
        return fitted.predict_proba(test[c["features"]])[:, 1]
    def calibrate(cal, period):
        th, tp, fp, cap = threshold(cal.event.to_numpy(int), cal.score.to_numpy(float))
        calrows.append({"period": period, "threshold": th, "tp": tp, "fp": fp, "fp_cap": cap,
                        "hours": len(cal), "events": int(cal.event.sum()), "latest_confirmed": str(cal.label_confirmed_at.max())})
        return th
    for month in [4, 5, 6]:
        boundary = pd.Timestamp(2021, month, 1)
        train = f.loc[(f.timestamp < boundary) & (f.label_confirmed_at < boundary)]
        test = f.loc[f.month == month]
        score = fit(train, test, f"noage_{month:02d}", boundary)
        cal = pd.concat(history, ignore_index=True) if history else None
        th = calibrate(cal.loc[cal.label_confirmed_at < boundary], str(month)) if cal is not None else None
        cp = classify_frame(test, score, th, old)
        devparts.append(cp); history.append(cp.loc[cp.method == "noage"])
    dev = pd.concat(devparts, ignore_index=True)
    rp = pd.read_csv(ROOT / "Modeling/tables/regime_routing/predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    pred = regress_frame(rp, dev)
    primary = dev.loc[dev.month >= 5]
    det = class_summary(primary, pred.timestamp.unique(), "May-Jun")
    reg, daily, windows, burden = regress_summary(pred, "May-Jun")
    dg = gates(det, reg, "May-Jun")
    outputs = {"development_classification.csv": dev, "development_detection.csv": det,
               "development_predictions.csv": pred, "development_metrics.csv": reg,
               "development_daily.csv": daily, "development_windows.csv": windows, "development_burden.csv": burden}
    late_executed = dg["classification_passed"] and dg["routing_passed"]
    decision = {"development": dg, "followup_executed": late_executed, "scope": c["scope"]}
    print(json.dumps({"development": dg, "followup_executed": late_executed}), flush=True)
    if late_executed:
        # This branch is fixed before any candidate result. Late labels never select it.
        full = pd.read_csv(ROOT / "Modeling/tables/regime_followup/classification_predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at"])
        train = f.loc[f.label_confirmed_at < "2021-07-01"]
        score = fit(train, full, "noage_fixed_june", pd.Timestamp("2021-07-01"))
        cal = dev.loc[(dev.method == "noage") & (dev.label_confirmed_at < "2021-07-01")]
        th = calibrate(cal, "Jul-Aug")
        cp = full[["timestamp", "label_confirmed_at", "month", "profile", "sustained_onset", "event"]].copy()
        cp["score"] = score; cp["threshold"] = th; cp["alarm"] = (score >= th).astype(int); cp["method"] = "noage"
        original = cp.copy(); original["score"] = full.score_logistic; original["alarm"] = full.alarm_logistic
        original["method"] = "original"
        original["threshold"] = json.loads((ROOT / "Modeling/tables/regime_followup/contract.json").read_text(encoding="utf-8"))["classification_thresholds"]["logistic"]["threshold"]
        cp = pd.concat([cp, original], ignore_index=True)
        rp = pd.read_csv(ROOT / "Modeling/tables/regime_followup/predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
        pp = regress_frame(rp, cp, late=True)
        dd = class_summary(cp, pp.timestamp.unique(), "Jul-Aug")
        rr, daily, windows, burden = regress_summary(pp, "Jul-Aug")
        decision["followup"] = gates(dd, rr, "Jul-Aug")
        outputs.update({"followup_classification.csv": cp, "followup_detection.csv": dd,
                        "followup_predictions.csv": pp, "followup_metrics.csv": rr,
                        "followup_daily.csv": daily, "followup_windows.csv": windows, "followup_burden.csv": burden})
    outputs.update({"model_manifest.csv": pd.DataFrame(manifest), "calibration.csv": pd.DataFrame(calrows)})
    for name, frame in outputs.items():
        frame.to_csv(OUT / name, index=False, encoding="utf-8-sig")
    save_json(OUT / "decision.json", decision)
    save_json(OUT / "run.json", {"status": "completed", "script_sha256": sha(__file__), "contract_sha256": sha(OUT / "contract.json"),
                                "inputs_sha256": c["inputs_sha256"], "new_fits": len(manifest), "new_regression_fits": 0,
                                "outputs_sha256": {n: sha(OUT / n) for n in list(outputs) + ["decision.json"]}, "runtime": sys.version})
    assert c["root_readme_sha256"] == sha(ROOT / "README.md")
    assert c["manuscript_sha256"] == sha(ROOT / "Modeling/04_Modeling_원고.md")
    print(json.dumps({"status": "completed", "new_fits": len(manifest), "decision": decision}), flush=True)


if __name__ == "__main__":
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    {"freeze": freeze, "run": run}[sys.argv[1]]()


코드 등록 완료: `Modeling/scripts/regime_age_ablation.py`

### `Modeling/scripts/regime_age_ablation_verify.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Independent checks for the single-feature ablation and reused HGB routing.

주요 함수: `sha()`, `close()`, `calendar()`, `errstats()`, `main()`.

In [85]:
%%submission_source Modeling/scripts/regime_age_ablation_verify.py
"""Independent checks for the single-feature ablation and reused HGB routing."""
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import csv
import hashlib
import json
import math
import sys
from datetime import datetime, timedelta
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from regime_forecast_verify import average_precision, independent_threshold

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/regime_age_ablation"


def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def close(a, b):
    if b is None or pd.isna(b):
        assert pd.isna(a), (a, b)
    else:
        assert math.isclose(float(a), float(b), rel_tol=1e-9, abs_tol=1e-8), (a, b)


def calendar(f):
    f = f.copy()
    f["hour_sin"] = np.sin(2 * np.pi * f.timestamp.dt.hour / 24)
    f["hour_cos"] = np.cos(2 * np.pi * f.timestamp.dt.hour / 24)
    for i in range(7):
        f[f"dow_{i}"] = (f.timestamp.dt.dayofweek == i).astype(int)
    return f


def errstats(g):
    errors = [float(r.prediction) - float(r.actual) for r in g.itertuples()]
    if not errors:
        return {n: None for n in ["mae", "rmse", "under", "over", "bias"]}
    n = len(errors)
    return {"mae": math.fsum(abs(e) for e in errors) / n,
            "rmse": math.sqrt(math.fsum(e * e for e in errors) / n),
            "under": math.fsum(max(-e, 0) for e in errors) / n,
            "over": math.fsum(max(e, 0) for e in errors) / n,
            "bias": math.fsum(errors) / n}


def main():
    c = json.loads((OUT / "contract.json").read_text(encoding="utf-8"))
    run = json.loads((OUT / "run.json").read_text(encoding="utf-8"))
    decision = json.loads((OUT / "decision.json").read_text(encoding="utf-8"))
    assert c["entrypoint_sha256"] == run["script_sha256"] == sha(ROOT / "Modeling/scripts/regime_age_ablation.py")
    assert run["contract_sha256"] == sha(OUT / "contract.json") and run["new_regression_fits"] == 0
    for name, expected in c["inputs_sha256"].items():
        assert sha(ROOT / name) == expected
    for name, expected in run["outputs_sha256"].items():
        assert sha(OUT / name) == expected
    original = json.loads((ROOT / "Modeling/tables/regime_forecast/contract.json").read_text(encoding="utf-8"))
    assert c["features"] == [x for x in original["feature_names"] if x != "prior_state_age"]
    assert len(c["features"]) == 23 and c["logistic"] == original["logistic"]
    assert not set(c["features"]) & set(original["forbidden_inputs"])
    f = pd.read_csv(ROOT / "Modeling/tables/regime_diagnosis/hourly_labels.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at"])
    f = calendar(f.loc[f.eligible_onset & f.persistence_known].copy())
    dev = pd.read_csv(OUT / "development_classification.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at"])
    late = pd.read_csv(OUT / "followup_classification.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at"]) if decision["followup_executed"] else None
    frames = {"development": dev}
    if late is not None:
        frames["followup"] = late
    full = pd.read_csv(ROOT / "Modeling/tables/regime_followup/classification_predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at"])
    manifest = pd.read_csv(OUT / "model_manifest.csv", encoding="utf-8-sig")
    assert len(manifest) == run["new_fits"]
    refits = 0
    for r in manifest.itertuples():
        boundary = pd.Timestamp(r.boundary)
        train = f.loc[(f.timestamp < boundary) & (f.label_confirmed_at < boundary)]
        if r.name == "noage_fixed_june":
            test = full
            saved = late.loc[late.method == "noage"]
        else:
            m = int(r.name[-2:]); test = f.loc[f.month == m]
            saved = dev.loc[(dev.method == "noage") & (dev.month == m)]
        assert saved.timestamp.tolist() == test.timestamp.tolist()
        assert len(train) == r.train_hours and (train.sustained_onset == 1).sum() == r.train_events
        assert pd.Timestamp(r.train_latest_confirmed) == train.label_confirmed_at.max() < boundary
        assert sha(ROOT / r.file) == r.sha256 and r.features.split("|") == c["features"]
        loaded = joblib.load(ROOT / r.file)
        fresh = make_pipeline(StandardScaler(), LogisticRegression(C=1., max_iter=3000, random_state=42, class_weight=None))
        fresh.fit(train[c["features"]], (train.sustained_onset == 1).astype(int))
        for fitted in [loaded, fresh]:
            assert fitted.feature_names_in_.tolist() == c["features"]
            np.testing.assert_allclose(saved.score, fitted.predict_proba(test[c["features"]])[:, 1], rtol=1e-9, atol=1e-8)
            np.testing.assert_allclose(fitted[0].mean_, train[c["features"]].mean(), rtol=1e-9, atol=1e-8)
        refits += 1
    cal = pd.read_csv(OUT / "calibration.csv", encoding="utf-8-sig")
    for r in cal.itertuples():
        boundary = pd.Timestamp("2021-07-01") if r.period == "Jul-Aug" else pd.Timestamp(2021, int(r.period), 1)
        previous = dev.loc[(dev.method == "noage") & (dev.timestamp < boundary) & (dev.label_confirmed_at < boundary)]
        th, tp, fp, cap = independent_threshold(previous)
        close(r.threshold, th)
        assert (r.tp, r.fp, r.fp_cap, r.hours, r.events) == (tp, fp, cap, len(previous), previous.event.sum())
        assert previous.label_confirmed_at.max() == pd.Timestamp(r.latest_confirmed) < boundary
        target = late if r.period == "Jul-Aug" else dev.loc[dev.month == int(r.period)]
        target = target.loc[target.method == "noage"]
        assert target.threshold.eq(th).all()
        np.testing.assert_array_equal(target.alarm, (target.score >= th).astype(int))
    raw = {}
    with (ROOT / "data/origin/okm_augumented_2021.csv").open(encoding="utf-8-sig", newline="") as stream:
        for row in csv.DictReader(stream):
            date, hour = int(row["날짜"]), int(row["시간"])
            if 20210101 <= date <= 20210831 and 0 <= hour < 24:
                t = datetime.strptime(str(date), "%Y%m%d") + timedelta(hours=hour)
                raw[t] = max(float(row[n]) for n in ["15분", "30분", "45분", "60분"])
    class_checks = regression_checks = 0
    for stage, cp in frames.items():
        period = "May-Jun" if stage == "development" else "Jul-Aug"
        pp = pd.read_csv(OUT / f"{stage}_predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
        parentdir = "regime_routing" if stage == "development" else "regime_followup"
        parent = pd.read_csv(ROOT / f"Modeling/tables/{parentdir}/predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
        candidate = cp.loc[cp.method == "noage"].set_index("timestamp")
        b0 = parent.loc[parent.variant == "B0"].set_index("timestamp")
        b1 = parent.loc[parent.variant == "B1_all"].set_index("timestamp")
        for variant, g in pp.groupby("variant"):
            assert g.timestamp.tolist() == b0.index.tolist()
            for r in g.itertuples():
                close(r.actual, raw[r.timestamp.to_pydatetime()])
                if variant == "G1_noage":
                    assert r.routed == int(candidate.loc[r.timestamp, "alarm"])
                    close(r.prediction, (b1 if r.routed else b0).loc[r.timestamp, "prediction"])
                else:
                    name = "G1" if variant == "G1_old" else variant
                    saved = parent.loc[(parent.variant == name) & (parent.timestamp == r.timestamp)].iloc[0]
                    close(r.prediction, saved.prediction); assert r.routed == saved.routed
        dt = pd.read_csv(OUT / f"{stage}_detection.csv", encoding="utf-8-sig")
        for r in dt.itertuples():
            g = cp.loc[(cp.method == r.method) & cp.event.notna() & cp.alarm.notna()]
            if stage == "development":
                g = g.loc[g.month >= 5]
            if r.scope == "common_regression":
                g = g.loc[g.timestamp.isin(b0.index)]
            if r.period != period:
                g = g.loc[g.month == int(r.period)]
            tp = fp = fn = tn = 0
            for rr in g.itertuples():
                if rr.event == 1:
                    tp += int(rr.alarm == 1); fn += int(rr.alarm == 0)
                else:
                    fp += int(rr.alarm == 1); tn += int(rr.alarm == 0)
            assert (r.tp, r.fp, r.fn, r.tn, r.hours, r.events) == (tp, fp, fn, tn, len(g), tp + fn)
            for name, value in {"recall": tp / (tp + fn), "precision": tp / (tp + fp) if tp + fp else None,
                                "fpr": fp / (fp + tn), "ap": average_precision(g)}.items():
                close(getattr(r, name), value)
            class_checks += 1
        mt = pd.read_csv(OUT / f"{stage}_metrics.csv", encoding="utf-8-sig")
        daily = pd.read_csv(OUT / f"{stage}_daily.csv", encoding="utf-8-sig")
        windows = pd.read_csv(OUT / f"{stage}_windows.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
        for r in daily.itertuples():
            g = pp.loc[(pp.variant == r.variant) & (pp.date == r.date)]
            assert len(g) == 24
            peak = g.loc[g.actual == g.actual.max()]
            assert r.ties == len(peak)
            for n, v in errstats(peak).items():
                close(getattr(r, n), v)
        for r in windows.itertuples():
            g = pp.loc[(pp.variant == r.variant) & pp.timestamp.isin(pd.date_range(r.timestamp, periods=3, freq="h"))]
            assert len(g) == 3
            for n, v in errstats(g).items():
                close(getattr(r, n), v)
        for r in mt.itertuples():
            g = pp.loc[pp.variant == r.variant]
            if r.period != period:
                g = g.loc[g.month == int(r.period)]
            if r.condition in ["all", "up_start", "down_start"]:
                if r.condition != "all":
                    g = g.loc[g.sustained_onset == (1 if r.condition == "up_start" else -1)]
                assert len(g) == r.hours
                stat = errstats(g)
            else:
                source = daily if r.condition == "daily_peak_day_equal" else windows
                z = source.loc[source.variant == r.variant]
                if r.period != period:
                    z = z.loc[z.month == int(r.period)]
                stat = {n: math.fsum(z[n]) / len(z) if len(z) else None for n in ["mae", "under", "over", "bias"]}
                stat["rmse"] = math.sqrt(math.fsum(z.rmse ** 2) / len(z)) if len(z) else None
                assert r.units == len(z)
            for name, value in stat.items():
                close(getattr(r, name), value)
            regression_checks += 1
        def get(variant, condition):
            return mt.loc[(mt.variant == variant) & (mt.period == period) & (mt.condition == condition)].iloc[0]
        gain = 1 - get("G1_noage", "up_start").mae / get("B0", "up_start").mae
        under = 1 - get("G1_noage", "up_start").under / get("B0", "up_start").under
        overall = get("G1_noage", "all").mae / get("B0", "all").mae - 1
        peak = get("G1_noage", "daily_peak_day_equal").mae / get("B0", "daily_peak_day_equal").mae - 1
        dr = dt.loc[(dt.scope == "common_regression") & (dt.period == period) & (dt.method == "noage")].iloc[0]
        expected = {"classification_passed": bool(dr.events >= 5 and dr.recall >= .8 and dr.precision >= .8 and dr.fpr <= .01),
                    "routing_passed": bool(dr.events >= 5 and gain >= .05 and under >= .05 and overall <= .01 and peak <= .01),
                    "start_mae_reduction": gain, "under_reduction": under,
                    "overall_mae_change": overall, "daily_peak_mae_change": peak}
        for name, value in expected.items():
            close(decision[stage][name], value)
    assert decision["followup_executed"] == (decision["development"]["classification_passed"] and decision["development"]["routing_passed"])
    assert c["root_readme_sha256"] == sha(ROOT / "README.md") and c["manuscript_sha256"] == sha(ROOT / "Modeling/04_Modeling_원고.md")
    result = {"status": "passed", "run_sha256": sha(OUT / "run.json"), "verifier_sha256": sha(__file__),
              "models_reloaded": len(manifest), "independent_classifier_refits": refits,
              "classifier_metrics_checked": class_checks, "regression_metrics_checked": regression_checks,
              "new_regression_fits": 0, "raw_valid_hours": len(raw), "extra_outputs_sha256": {}}
    (OUT / "independent_verification.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps(result), flush=True)


if __name__ == "__main__":
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    main()


코드 등록 완료: `Modeling/scripts/regime_age_ablation_verify.py`

### `Modeling/scripts/regime_age_calendar_control.py`

원고 연결: Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_원고.md.

코드 설명: Posthoc Monday08 control after S06: diagnosis only, never adopted or tuned.

주요 함수: `main()`.

In [86]:
%%submission_source Modeling/scripts/regime_age_calendar_control.py
"""Posthoc Monday08 control after S06: diagnosis only, never adopted or tuned."""
import json
from datetime import datetime, timezone, timedelta
import numpy as np
import pandas as pd
from regime_age_ablation_verify import ROOT, OUT, sha, errstats


def main():
    assert not (OUT / "calendar_control_contract.json").exists()
    names = ["followup_classification.csv", "followup_predictions.csv", "run.json", "independent_verification.json"]
    c = {"created_at": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
         "entrypoint_sha256": sha(__file__), "inputs_sha256": {n: sha(OUT / n) for n in names},
         "reason": "All7 candidate alarms at Monday08; check whether a weekday/hour rule explains the selection and value gain.",
         "rule": "Monday hour08, no other weekday/hour trials or threshold changes",
         "scope": "Posthoc control from already-seen July-August; not independent validation or a new adopted candidate",
         "fits": 0}
    (OUT / "calendar_control_contract.json").write_text(json.dumps(c, indent=2), encoding="utf-8")
    cp = pd.read_csv(OUT / "followup_classification.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    cp = cp.loc[cp.method == "noage"]
    p = pd.read_csv(OUT / "followup_predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    base = p.loc[p.variant == "B0"].copy()
    b1 = p.loc[p.variant == "B1_all"].set_index("timestamp").loc[base.timestamp]
    mask = (base.timestamp.dt.dayofweek == 0) & (base.timestamp.dt.hour == 8)
    g = base.copy(); g["prediction"] = np.where(mask, b1.prediction, base.prediction)
    g["variant"] = "Monday08_posthoc"; g["routed"] = mask.astype(int)
    det = []; rows = []
    common = base.timestamp
    for period in ["Jul-Aug", "7", "8"]:
        subset = cp.loc[cp.event.notna() & cp.timestamp.isin(common)]
        part = g
        if period != "Jul-Aug":
            subset = subset.loc[subset.month == int(period)]; part = part.loc[part.month == int(period)]
        a = (subset.timestamp.dt.dayofweek == 0) & (subset.timestamp.dt.hour == 8)
        tp = int((a & subset.event.eq(1)).sum()); fp = int((a & subset.event.eq(0)).sum())
        fn = int(subset.event.sum()) - tp; tn = len(subset) - tp - fp - fn
        det.append({"period": period, "hours": len(subset), "tp": tp, "fp": fp, "fn": fn, "tn": tn,
                    "recall": tp / (tp + fn), "precision": tp / (tp + fp), "fpr": fp / (fp + tn)})
        for condition, pp in [("all", part), ("up_start", part.loc[part.sustained_onset == 1])]:
            rows.append({"period": period, "condition": condition, "hours": len(pp), **errstats(pp)})
        days = []
        for date, pp in part.groupby("date"):
            if len(pp) == 24:
                peak = pp.loc[pp.actual == pp.actual.max()]
                days.append(errstats(peak))
        rows.append({"period": period, "condition": "daily_peak_day_equal", "units": len(days),
                     **{n: float(np.mean([d[n] for d in days])) for n in ["mae", "under", "over", "bias"]}})
    # Validate mask against Python datetime and routed values against scalar selection.
    for r in g.itertuples():
        selected = r.timestamp.weekday() == 0 and r.timestamp.hour == 8
        assert r.routed == int(selected)
        expected = b1.loc[r.timestamp, "prediction"] if selected else base.set_index("timestamp").loc[r.timestamp, "prediction"]
        assert abs(r.prediction - expected) < 1e-9
    artifacts = {"calendar_control_detection.csv": pd.DataFrame(det), "calendar_control_metrics.csv": pd.DataFrame(rows),
                 "calendar_control_predictions.csv": g}
    for name, frame in artifacts.items():
        frame.to_csv(OUT / name, index=False, encoding="utf-8-sig")
    result = {"status": "completed", "contract_sha256": sha(OUT / "calendar_control_contract.json"),
              "script_sha256": sha(__file__), "new_fits": 0, "scalar_routing_check": "passed",
              "outputs_sha256": {n: sha(OUT / n) for n in artifacts}, "scope": c["scope"]}
    (OUT / "calendar_control_run.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps({"detection": det, "metrics": [r for r in rows if r["period"] == "Jul-Aug"]}), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/regime_age_calendar_control.py`

### `Modeling/scripts/regime_diagnose.py`

원고 연결: Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_원고.md.

코드 설명: S01/S02: causal onset labels, future persistence, past-only precursor diagnosis.
No classifier, regression, July/August evaluation, or manuscript generation.

주요 함수: `sha()`, `dump()`, `save()`, `load()`, `thresholds()`, `label()`, `probe()`, `freeze()`, `precursor()`, `run()`.

In [87]:
%%submission_source Modeling/scripts/regime_diagnose.py
"""S01/S02: causal onset labels, future persistence, past-only precursor diagnosis.
No classifier, regression, July/August evaluation, or manuscript generation.
"""
import argparse
import hashlib
import json
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/regime_diagnosis"
SLOTS = ["15분", "30분", "45분", "60분"]
SOURCE = ROOT / "data/origin/okm_augumented_2021.csv"
SOURCE_SHA = "8f7af2e49366c93e1d6f5fdef4b5e350066c1792ac463c2c2886e370f4674830"
FEATURES = ["prior_mean", "past_median6", "past_iqr6", "past_range6",
            "past_slope6", "prior_delta", "prior_last_minus_mean",
            "prior_slot_range", "prior_production", "prior_production_delta",
            "prior_state_age", "prior_state_left_censored"]


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def dump(obj, path):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def save(frame, name):
    frame.to_csv(OUT / (name + ".csv"), index=False, encoding="utf-8-sig")


def load():
    assert sha(SOURCE) == SOURCE_SHA
    raw = pd.read_csv(SOURCE, encoding="utf-8-sig")
    raw = raw.loc[raw["날짜"].between(20210101, 20210630) & raw["시간"].between(0, 23)].copy()
    raw["timestamp"] = pd.to_datetime(raw["날짜"].astype(str), format="%Y%m%d") + pd.to_timedelta(raw["시간"], unit="h")
    raw = raw.sort_values("timestamp").set_index("timestamp")
    assert raw.index.is_unique
    d = raw.reindex(pd.date_range("2021-01-01", "2021-06-30 23:00", freq="h"))
    d.index.name = "timestamp"
    d["valid"] = d[SLOTS].notna().all(axis=1)
    d["level"] = d[SLOTS].mean(axis=1).where(d.valid)
    d["maximum"] = d[SLOTS].max(axis=1).where(d.valid)
    d["date"] = d.index.normalize()
    d["month"] = d.index.month
    d["hour"] = d.index.hour
    d["weekend"] = (d.index.dayofweek >= 5).astype(int)
    profiles = {date: hashlib.sha256(g[SLOTS].to_numpy(dtype=np.int64).tobytes()).hexdigest()
                for date, g in raw.groupby(raw.index.normalize())}
    d["profile"] = d.date.map(profiles)
    history = pd.concat([d.level.shift(lag).rename(str(lag)) for lag in range(1, 7)], axis=1)
    d["eligible_onset"] = d.valid & history.notna().all(axis=1)
    d["prior_mean"] = history["1"]
    d["past_median6"] = history.median(axis=1).where(d.eligible_onset)
    d["past_iqr6"] = (history.quantile(.75, axis=1)-history.quantile(.25, axis=1)).where(d.eligible_onset)
    d["past_range6"] = (history.max(axis=1)-history.min(axis=1)).where(d.eligible_onset)
    d["past_slope6"] = ((history["1"]-history["6"])/5).where(d.eligible_onset)
    d["prior_delta"] = d.level.shift(1)-d.level.shift(2)
    d["prior_last_minus_mean"] = d["60분"].shift(1)-d.prior_mean
    d["prior_slot_range"] = (d[SLOTS].max(axis=1)-d[SLOTS].min(axis=1)).shift(1).where(d.valid.shift(1, fill_value=False))
    d["prior_production"] = d["생산량"].shift(1)
    d["prior_production_delta"] = d["생산량"].shift(1)-d["생산량"].shift(2)
    d["delta"] = d.level-d.prior_mean
    d["deviation"] = d.level-d.past_median6
    return d


def thresholds(d, quantile=.90):
    jan = d.loc[d.month.eq(1) & d.eligible_onset]
    return {"up": float(jan.loc[jan.delta.gt(0), "delta"].quantile(quantile)),
            "down": float((-jan.loc[jan.delta.lt(0), "delta"]).quantile(quantile))}


def label(d, th, multiplier=1.5):
    """Onset identity uses only current/past values; future is used only afterwards."""
    z = d.copy()
    up = z.eligible_onset & z.delta.ge(th["up"]) & z.deviation.ge(th["up"]) & z.deviation.ge(multiplier*z.past_iqr6)
    down = z.eligible_onset & (-z.delta).ge(th["down"]) & (-z.deviation).ge(th["down"]) & (-z.deviation).ge(multiplier*z.past_iqr6)
    candidate = np.select([up, down], [1, -1], default=0)
    starts = np.zeros(len(z), dtype=int)
    ages, censored, directions, available = [], [], [], []
    active = None
    last_gap = None
    event_rows = []
    for i, (ts, row) in enumerate(z.iterrows()):
        # State at t-1, before current t measurement.
        ages.append(i-active["i"] if active else 0)
        directions.append(active["sign"] if active else 0)
        censored.append(int(active is None and (last_gap is None or i-last_gap <= 6)))
        available.append(ts-pd.Timedelta(hours=1))
        if not row.valid:
            active = None
            last_gap = i
            continue
        sign = int(candidate[i])
        if active is not None:
            if active["sign"]*(row.level-active["baseline"]) < .5*th[active["direction"]]:
                active = None
            elif sign and sign != active["sign"]:
                active = None
        if sign and active is None:
            direction = "up" if sign == 1 else "down"
            active = {"i": i, "baseline": row.past_median6, "sign": sign, "direction": direction}
            starts[i] = sign
            event_rows.append({"event_id": len(event_rows)+1, "position": i,
                              "timestamp": ts, "date": row.date, "month": row.month,
                              "hour": row.hour, "weekend": row.weekend,
                              "direction": direction, "baseline": row.past_median6,
                              "first_mean": row.level, "first_maximum": row.maximum,
                              "first_delta": row.delta, "first_deviation": row.deviation,
                              "past_iqr6": row.past_iqr6,
                              "profile": row.profile,
                              "previous_mean": row.prior_mean,
                              "production_before": row.prior_production,
                              "production_at_start": row["생산량"]})
    z["candidate"] = candidate
    z["onset"] = starts
    z["prior_state_age"] = ages
    z["prior_state_direction"] = directions
    z["prior_state_left_censored"] = censored
    z["latest_input_timestamp"] = available
    z["label_confirmed_at"] = z.index + pd.Timedelta(hours=2)
    z["persistence_known"] = z.valid & z.valid.shift(-1, fill_value=False) & z.valid.shift(-2, fill_value=False)
    z["sustained_onset"] = 0.
    z.loc[~z.persistence_known, "sustained_onset"] = np.nan
    for event in event_rows:
        i, direction = event["position"], event["direction"]
        sign = 1 if direction == "up" else -1
        boundary = .5*th[direction]
        n = 0
        while i+n < len(z) and bool(z.valid.iloc[i+n]) and sign*(z.level.iloc[i+n]-event["baseline"]) >= boundary:
            n += 1
        end = i+n
        reason = "scope_end" if end == len(z) else ("gap" if not bool(z.valid.iloc[end]) else "return")
        event["reference_departure_hours"] = n
        event["right_censored"] = reason != "return"
        event["end_reason"] = reason
        event["last_departure_timestamp"] = z.index[i+n-1]
        event["duration_confirmed_at"] = z.index[end] if end < len(z) else pd.NaT
        for horizon in [1, 3, 6]:
            window = z.iloc[i:i+horizon]
            known = len(window) == horizon and bool(window.valid.all())
            sustained = bool((sign*(window.level-event["baseline"]) >= boundary).all()) if known else None
            event[f"sustained{horizon}"] = sustained
        event["label_confirmed_at"] = z.index[i]+pd.Timedelta(hours=2) if event["sustained3"] is not None else pd.NaT
        first3 = z.iloc[i:i+3]
        event["next3_mean"] = float(first3.level.mean()) if event["sustained3"] is not None else np.nan
        event["next3_min_mean"] = float(first3.level.min()) if event["sustained3"] is not None else np.nan
        event["next3_max_mean"] = float(first3.level.max()) if event["sustained3"] is not None else np.nan
        z.iloc[i, z.columns.get_loc("sustained_onset")] = sign if event["sustained3"] else (0. if event["sustained3"] is not None else np.nan)
    events = pd.DataFrame(event_rows)
    return z, events


def probe():
    assert not (OUT/"probe.json").exists()
    OUT.mkdir(parents=True, exist_ok=True)
    jan = load().loc[lambda x: x.month.eq(1)].copy()
    result = {"source_sha256": sha(SOURCE), "script_sha256": sha(__file__), "scope": "January only, no model fit",
              "threshold_quantiles": {}, "primary_preference": "q90 predefined large hourly changes plus median departure and 1.5IQR filter; q95 sensitivity only"}
    for q in [.90,.95]:
        th = thresholds(jan,q)
        _, ev = label(jan,th)
        result["threshold_quantiles"][str(q)] = {"thresholds": th,
            "onsets": ev.groupby("direction").size().to_dict(),
            "sustained3": ev[ev.sustained3.eq(True)].groupby("direction").size().to_dict()}
    result["ranges"] = {"mean_min":float(jan.level.min()),"mean_max":float(jan.level.max()),
                        "maximum":float(jan.maximum.max())}
    dump(result,OUT/"probe.json")
    save(jan.loc[jan.eligible_onset, ["level","prior_mean","past_median6","past_iqr6","delta","deviation"]].reset_index(),"january_probe")
    print(json.dumps(result,ensure_ascii=False),flush=True)


def freeze():
    assert not (OUT/"contract.json").exists()
    p = json.loads((OUT/"probe.json").read_text(encoding="utf-8"))
    assert p["script_sha256"] == sha(__file__)
    th = p["threshold_quantiles"]["0.9"]["thresholds"]
    c = {"recorded_at":datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
         "entrypoint_sha256":sha(__file__),"source_sha256":SOURCE_SHA,
         "probe_sha256":sha(OUT/"probe.json"),
         "scope":"January-June only. S02 condition diagnosis January-March; April-June onset counts only, no performance selection.",
         "thresholds":th,"quantile":.90,"calibration":"January valid rows with six exact past observations; separate positive/negative one-hour mean deltas",
         "onset":"delta and departure from preceding6 median each >=direction threshold; departure>=1.5 preceding6 IQR",
         "episode":"freeze baseline at onset; suppress same-direction starts while departure>=half threshold; allow opposite qualifying start; reset on return or gap. State features prior-only. Large same-direction steps saved as suppressed candidates, not new starts.",
         "persistence":"at t,t+1,t+2 all observed levels in original departure direction at least half threshold. Unknown if gap/end; confirmed t+2.",
         "duration":"consecutive hours on one side of original frozen baseline until return/gap/end; opposite-event reference durations can overlap; not a disjoint state segmentation or constant box guarantee",
         "sensitivity":"q95 and horizon1/6 descriptive; no model-dependent definition choice",
         "features":FEATURES+["prior_state_direction"],
         "conditions":"January-March only month/hour/previous-level-bin; January prior mean quintiles; min2event and min5control records; weights=min(event,control); one-event cells sensitivity",
         "controls":"eligible onset=0 rows with the same complete persistence observation window; controls include ongoing states, not all stable",
         "no_future_inputs":"target level/delta/deviation, event outcomes, full-day profile and future duration excluded from features",
         "splits":"for validation months2..6 use timestamp before month and confirmed_at before month. No class prediction fitted.",
         "no_manuscript_edit":True,"fits":0,"later_evaluated":False}
    dump(c,OUT/"contract.json")
    print("S01/S02 contract frozen; thresholds="+json.dumps(th),flush=True)


def precursor(z, events):
    eligible = z.eligible_onset & z.persistence_known & z.month.le(3)
    early = z.loc[eligible].copy()
    edges = sorted(set(early.loc[early.month.eq(1),"prior_mean"].quantile([.2,.4,.6,.8]).tolist()))
    early["level_bin"] = np.searchsorted(edges,early.prior_mean,side="right")
    rows, cells, coverage = [], [], []
    targets = {"up":early.onset.eq(1),"down":early.onset.eq(-1),
               "sustained_up":early.sustained_onset.eq(1),"sustained_down":early.sustained_onset.eq(-1)}
    for name, mask in targets.items():
        a = early.loc[mask]
        b = early.loc[early.onset.eq(0)]
        for feature in FEATURES:
            rows.append({"target":name,"feature":feature,"events":len(a),"controls":len(b),
                         "event_mean":float(a[feature].mean()),"control_mean":float(b[feature].mean()),
                         "event_median":float(a[feature].median()),"control_median":float(b[feature].median())})
        keys = ["month","hour","level_bin"]
        event_groups = {k:g for k,g in a.groupby(keys)}
        control_groups = {k:g for k,g in b.groupby(keys)}
        for minimum in [1,2]:
            shared = [k for k in event_groups if k in control_groups and len(event_groups[k])>=minimum and len(control_groups[k])>=5]
            count_a = sum(len(event_groups[k]) for k in shared)
            count_b = sum(len(control_groups[k]) for k in shared)
            coverage.append({"target":name,"minimum_events_per_cell":minimum,"matched_cells":len(shared),
                             "events_total":len(a),"events_covered":count_a,
                             "event_coverage":count_a/len(a) if len(a) else 0.,
                             "controls_covered":count_b})
            for feature in FEATURES:
                diffs, weights = [], []
                for key in shared:
                    ag,bg = event_groups[key],control_groups[key]
                    w = min(len(ag),len(bg))
                    av,bv = float(ag[feature].mean()),float(bg[feature].mean())
                    cells.append({"target":name,"minimum_events_per_cell":minimum,
                        "month":key[0],"hour":key[1],"level_bin":key[2],"feature":feature,
                        "event_n":len(ag),"control_n":len(bg),"event_mean":av,"control_mean":bv,"weight":w})
                    diffs.append(av-bv);weights.append(w)
                rows.append({"target":name,"feature":feature,"weighting":"matched",
                    "minimum_events_per_cell":minimum,"events":count_a,"controls":count_b,
                    "matched_cells":len(shared),"matched_difference":float(np.average(diffs,weights=weights)) if diffs else np.nan})
    calendar = early.groupby(["month","weekend","hour"]).agg(hours=("onset","size"),
        up=("onset",lambda x:int(x.eq(1).sum())),down=("onset",lambda x:int(x.eq(-1).sum())),
        sustained_up=("sustained_onset",lambda x:int(x.eq(1).sum())),
        sustained_down=("sustained_onset",lambda x:int(x.eq(-1).sum()))).reset_index()
    for field in ["up","down","sustained_up","sustained_down"]:
        calendar[field+"_rate"] = calendar[field]/calendar.hours
    splits = []
    for month in range(2,7):
        boundary = pd.Timestamp(2021,month,1)
        train = z.loc[z.eligible_onset & z.persistence_known & (z.index < boundary) & z.label_confirmed_at.lt(boundary)]
        valid = z.loc[z.eligible_onset & z.persistence_known & z.month.eq(month)]
        for group, data in [("train",train),("validation_count_only",valid)]:
            for sign,direction in [(1,"up"),(-1,"down")]:
                part = data.loc[data.onset.eq(sign)]
                splits.append({"month":month,"partition":group,"direction":direction,"hours":len(data),
                    "onsets":len(part),"sustained3":int(part.sustained_onset.eq(sign).sum()),
                    "event_dates":part.date.nunique(),"event_profiles":part.profile.nunique(),
                    "last_label_confirmation":str(data.label_confirmed_at.max())})
    save(pd.DataFrame(rows),"precursor_summary")
    save(pd.DataFrame(cells),"matched_cells")
    save(pd.DataFrame(coverage),"matched_coverage")
    save(calendar,"calendar_conditions")
    save(pd.DataFrame(splits),"split_counts")
    dump({"prior_level_edges":edges,"scope":"Jan-Mar","comparison":"association, not forecasting or causal evidence"},OUT/"precursor_contract.json")


def run():
    assert not (OUT/"run.json").exists()
    c = json.loads((OUT/"contract.json").read_text(encoding="utf-8"))
    assert c["entrypoint_sha256"] == sha(__file__) and c["source_sha256"] == sha(SOURCE)
    assert c["probe_sha256"] == sha(OUT/"probe.json")
    raw = load()
    z, events = label(raw,c["thresholds"])
    rows = []
    for (month,direction),g in events.groupby(["month","direction"]):
        complete = g.loc[~g.right_censored]
        rows.append({"month":month,"direction":direction,"events":len(g),"dates":g.date.nunique(),
            "profiles":g.profile.nunique(),"sustained3":int(g.sustained3.eq(True).sum()),
            "short3":int(g.sustained3.eq(False).sum()),"unknown3":int(g.sustained3.isna().sum()),
            "sustained6":int(g.sustained6.eq(True).sum()),"censored_duration":int(g.right_censored.sum()),
            "complete_duration_median":float(complete.reference_departure_hours.median()) if len(complete) else None})
    save(pd.DataFrame(rows),"event_summary")
    save(events,"events")
    keep = ["valid","eligible_onset","level","maximum","date","month","hour","weekend","profile",
            "candidate","onset","sustained_onset","persistence_known","label_confirmed_at",
            "latest_input_timestamp","delta","deviation"]+FEATURES+["prior_state_direction"]
    save(z[keep].reset_index(),"hourly_labels")
    sens = []
    for q in [.90,.95]:
        th = thresholds(raw,q)
        _,ev = label(raw,th)
        for (month,direction),g in ev.groupby(["month","direction"]):
            sens.append({"quantile":q,"month":month,"direction":direction,
                         "threshold":th[direction],"events":len(g),
                         "sustained3":int(g.sustained3.eq(True).sum()),
                         "sustained6":int(g.sustained6.eq(True).sum())})
    save(pd.DataFrame(sens),"definition_sensitivity")
    precursor(z,events)
    # Deterministic representative windows, using Jan-Mar only.
    selected = events.loc[events.month.le(3)].sort_values("first_deviation")
    selected = pd.concat([selected.head(2),selected.tail(2),
                          selected.loc[selected.sustained3.eq(False)].head(2)]).drop_duplicates("event_id")
    windows = []
    for _,ev in selected.iterrows():
        part = z.loc[ev.timestamp-pd.Timedelta(hours=6):ev.timestamp+pd.Timedelta(hours=8),
                     ["level","maximum","valid"]].reset_index()
        part["event_id"] = ev.event_id
        part["direction"] = ev.direction
        part["baseline"] = ev.baseline
        windows.append(part)
    save(pd.concat(windows,ignore_index=True),"representative_windows")
    result = {"status":"completed","contract_sha256":sha(OUT/"contract.json"),
              "hours":int(z.valid.sum()),"eligible_onset_hours":int(z.eligible_onset.sum()),
              "events":len(events),"sustained3":int(events.sustained3.eq(True).sum()),
              "thresholds":c["thresholds"],"new_fits":0,"later_evaluated":False,
              "recorded_ranges":{"mean_max":float(z.level.max()),"maximum":float(z.maximum.max())},
              "outputs_sha256":{p.name:sha(p) for p in OUT.glob("*.csv")} }
    result["outputs_sha256"]["precursor_contract.json"] = sha(OUT/"precursor_contract.json")
    dump(result,OUT/"run.json")
    print(pd.DataFrame(rows).to_string(index=False),flush=True)
    print(json.dumps(result,ensure_ascii=False),flush=True)


if __name__ == "__main__":
    assert sys.version_info[:2] == (3,13) and sys._is_gil_enabled()
    parser = argparse.ArgumentParser()
    parser.add_argument("stage",choices=["probe","freeze","run"])
    globals()[parser.parse_args().stage]()


코드 등록 완료: `Modeling/scripts/regime_diagnose.py`

### `Modeling/scripts/regime_diagnose_verify.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Independent CSV/scalar S01/S02 verification and short/persistent comparisons.

주요 함수: `sha()`, `read()`, `truth()`, `close()`, `q()`, `reconstruct()`, `main()`.

In [88]:
%%submission_source Modeling/scripts/regime_diagnose_verify.py
"""Independent CSV/scalar S01/S02 verification and short/persistent comparisons."""
import csv
import hashlib
import json
import math
import sys
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path
from statistics import median

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/regime_diagnosis"
SOURCE = ROOT / "data/origin/okm_augumented_2021.csv"
SLOTS = ["15분", "30분", "45분", "60분"]
FEATURES = ["prior_mean","past_median6","past_iqr6","past_range6","past_slope6",
            "prior_delta","prior_last_minus_mean","prior_slot_range","prior_production",
            "prior_production_delta","prior_state_age","prior_state_left_censored"]


def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def read(p):
    with Path(p).open(encoding="utf-8-sig",newline="") as stream:
        return list(csv.DictReader(stream))


def truth(x):
    assert x in ("True","False"),x
    return x == "True"


def close(a,b):
    assert math.isclose(float(a),float(b),rel_tol=1e-10,abs_tol=1e-8),(a,b)


def q(values,p):
    a = sorted(values)
    loc = (len(a)-1)*p
    lo,hi = math.floor(loc),math.ceil(loc)
    return a[lo]+(a[hi]-a[lo])*(loc-lo)


def reconstruct(raw,times,thresholds):
    active=None
    gap=None
    results={}
    events=[]
    for i,t in enumerate(times):
        previous=[raw.get(t-timedelta(hours=k)) for k in range(1,7)]
        own=raw.get(t)
        eligible=own is not None and all(x is not None for x in previous)
        state={"age":i-active["index"] if active else 0,
               "direction":active["sign"] if active else 0,
               "censored":int(active is None and (gap is None or i-gap<=6))}
        if own is None:
            active=None
            gap=i
            results[t]={"eligible":False,"onset":0,"state":state,"candidate":0}
            continue
        candidate=0
        features={}
        if eligible:
            levels=[x["mean"] for x in previous]
            med=median(levels)
            iqr=q(levels,.75)-q(levels,.25)
            delta=own["mean"]-levels[0]
            dev=own["mean"]-med
            if delta>=thresholds["up"] and dev>=thresholds["up"] and dev>=1.5*iqr:
                candidate=1
            if -delta>=thresholds["down"] and -dev>=thresholds["down"] and -dev>=1.5*iqr:
                candidate=-1
            features={"prior_mean":levels[0],"past_median6":med,"past_iqr6":iqr,
                "past_range6":max(levels)-min(levels),"past_slope6":(levels[0]-levels[-1])/5,
                "prior_delta":levels[0]-levels[1],
                "prior_last_minus_mean":previous[0]["slots"][-1]-levels[0],
                "prior_slot_range":max(previous[0]["slots"])-min(previous[0]["slots"]),
                "prior_production":previous[0]["production"],
                "prior_production_delta":previous[0]["production"]-previous[1]["production"],
                "prior_state_age":state["age"],"prior_state_left_censored":state["censored"]}
        onset=0
        if active:
            th=thresholds["up" if active["sign"]==1 else "down"]
            if active["sign"]*(own["mean"]-active["baseline"]) < th/2 or (candidate and candidate!=active["sign"]):
                active=None
        if candidate and active is None:
            onset=candidate
            active={"index":i,"sign":candidate,"baseline":features["past_median6"]}
            events.append((t,candidate,features["past_median6"]))
        results[t]={"eligible":eligible,"onset":onset,"candidate":candidate,
                    "state":state,"features":features}
    return results,events


def main():
    assert sys.version_info[:2] == (3,13) and sys._is_gil_enabled()
    c=json.loads((OUT/"contract.json").read_text(encoding="utf-8"))
    run=json.loads((OUT/"run.json").read_text(encoding="utf-8"))
    assert run["status"]=="completed" and run["contract_sha256"]==sha(OUT/"contract.json")
    assert sha(SOURCE)==c["source_sha256"]
    assert sha(ROOT/"Modeling/scripts/regime_diagnose.py")==c["entrypoint_sha256"]
    assert sha(OUT/"probe.json")==c["probe_sha256"]
    for name,digest in run["outputs_sha256"].items():
        assert sha(OUT/name)==digest,name
    raw={}
    for row in read(SOURCE):
        date,hour=int(row["날짜"]),int(row["시간"])
        if not (20210101<=date<=20210630 and 0<=hour<24):
            continue
        t=datetime.strptime(str(date),"%Y%m%d")+timedelta(hours=hour)
        assert t not in raw
        slots=[float(row[s]) for s in SLOTS]
        raw[t]={"slots":slots,"mean":math.fsum(slots)/4,
                "maximum":max(slots),"production":float(row["생산량"])}
    times=[datetime(2021,1,1)+timedelta(hours=i) for i in range(181*24)]
    changes=[]
    for t in times:
        if t.month==1 and t in raw and all(t-timedelta(hours=k) in raw for k in range(1,7)):
            changes.append(raw[t]["mean"]-raw[t-timedelta(hours=1)]["mean"])
    for key,values in [("up",[x for x in changes if x>0]),("down",[-x for x in changes if x<0])]:
        close(c["thresholds"][key],q(values,.9))
    independent,event_starts=reconstruct(raw,times,c["thresholds"])
    hourly=read(OUT/"hourly_labels.csv")
    assert len(hourly)==len(times)
    parent={datetime.fromisoformat(r["timestamp"]):r for r in read(ROOT/"Modeling/tables/m01/hourly_frame.csv")
            if r["timestamp"]<"2021-07"}
    assert set(parent)==set(raw)
    for row,t in zip(hourly,times):
        assert datetime.fromisoformat(row["timestamp"])==t
        expected=independent[t]
        assert truth(row["valid"])==(t in raw)
        assert truth(row["eligible_onset"])==expected["eligible"]
        assert int(row["onset"])==expected["onset"] and int(row["candidate"])==expected["candidate"]
        assert int(row["prior_state_age"])==expected["state"]["age"]
        assert int(row["prior_state_direction"])==expected["state"]["direction"]
        assert int(row["prior_state_left_censored"])==expected["state"]["censored"]
        assert datetime.fromisoformat(row["latest_input_timestamp"])==t-timedelta(hours=1)
        if t in raw:
            for field,value in [("level",raw[t]["mean"]),("maximum",raw[t]["maximum"])]:
                close(row[field],value)
            close(parent[t]["target_mean"],raw[t]["mean"])
            close(parent[t]["target_maximum"],raw[t]["maximum"])
        if expected["eligible"]:
            for feature,value in expected["features"].items():
                close(row[feature],value)
    events=read(OUT/"events.csv")
    assert len(events)==len(event_starts)==run["events"]
    for row,(t,sign,baseline) in zip(events,event_starts):
        assert datetime.fromisoformat(row["timestamp"])==t
        assert row["direction"]==("up" if sign==1 else "down")
        close(row["baseline"],baseline)
        boundary=c["thresholds"][row["direction"]]/2
        n=0
        while t+timedelta(hours=n) in raw and sign*(raw[t+timedelta(hours=n)]["mean"]-baseline)>=boundary:
            n+=1
        assert int(row["reference_departure_hours"])==n
        end=t+timedelta(hours=n)
        reason="scope_end" if end>times[-1] else ("gap" if end not in raw else "return")
        assert row["end_reason"]==reason and truth(row["right_censored"])==(reason!="return")
        for horizon in [1,3,6]:
            ts=[t+timedelta(hours=k) for k in range(horizon)]
            known=all(s in raw for s in ts)
            actual=row[f"sustained{horizon}"]
            if known:
                assert truth(actual)==all(sign*(raw[s]["mean"]-baseline)>=boundary for s in ts)
            else:
                assert actual==""
        if row["sustained3"]:
            assert datetime.fromisoformat(row["label_confirmed_at"])==t+timedelta(hours=2)
            close(row["next3_mean"],math.fsum(raw[t+timedelta(hours=k)]["mean"] for k in range(3))/3)
        hr=hourly[times.index(t)]
        if row["sustained3"]:
            close(hr["sustained_onset"],sign if truth(row["sustained3"]) else 0)
    for row in read(OUT/"event_summary.csv"):
        es=[e for e in events if e["month"]==row["month"] and e["direction"]==row["direction"]]
        assert int(row["events"])==len(es)
        assert int(row["sustained3"])==sum(e["sustained3"]=="True" for e in es)
        assert int(row["short3"])==sum(e["sustained3"]=="False" for e in es)
    early=[r for r in hourly if truth(r["eligible_onset"]) and truth(r["persistence_known"]) and int(r["month"])<=3]
    edges=json.loads((OUT/"precursor_contract.json").read_text(encoding="utf-8"))["prior_level_edges"]
    jan=[float(r["prior_mean"]) for r in early if r["month"]=="1"]
    assert len(edges)==len(sorted(set(q(jan,p) for p in [.2,.4,.6,.8])))
    for a,b in zip(edges,sorted(set(q(jan,p) for p in [.2,.4,.6,.8]))):
        close(a,b)
    targets={"up":lambda r:int(r["onset"])==1,"down":lambda r:int(r["onset"])==-1,
             "sustained_up":lambda r:float(r["sustained_onset"])==1,
             "sustained_down":lambda r:float(r["sustained_onset"])==-1}
    indexed=defaultdict(list)
    for r in early:
        level_bin=sum(float(r["prior_mean"])>=e for e in edges)
        indexed[(int(r["month"]),int(r["hour"]),level_bin)].append(r)
    # Rebuild every matched condition from source scalar features.
    cells=read(OUT/"matched_cells.csv")
    for cell in cells:
        key=(int(cell["month"]),int(cell["hour"]),int(cell["level_bin"]))
        a=[r for r in indexed[key] if targets[cell["target"]](r)]
        b=[r for r in indexed[key] if int(r["onset"])==0]
        assert len(a)==int(cell["event_n"]) and len(b)==int(cell["control_n"])
        close(cell["event_mean"],math.fsum(float(r[cell["feature"]]) for r in a)/len(a))
        close(cell["control_mean"],math.fsum(float(r[cell["feature"]]) for r in b)/len(b))
    for coverage in read(OUT/"matched_coverage.csv"):
        fn=targets[coverage["target"]]
        support=[]
        minimum=int(coverage["minimum_events_per_cell"])
        for key,rows in indexed.items():
            a=[r for r in rows if fn(r)]
            b=[r for r in rows if int(r["onset"])==0]
            if len(a)>=minimum and len(b)>=5:
                support.append((key,a,b))
        assert int(coverage["matched_cells"])==len(support)
        assert int(coverage["events_total"])==sum(fn(r) for r in early)
        assert int(coverage["events_covered"])==sum(len(x[1]) for x in support)
    for r in read(OUT/"precursor_summary.csv"):
        if r["weighting"]=="matched":
            cs=[x for x in cells if x["target"]==r["target"] and x["feature"]==r["feature"]
                and int(x["minimum_events_per_cell"])==int(float(r["minimum_events_per_cell"]))]
            if cs:
                close(r["matched_difference"],sum(float(x["weight"])*(float(x["event_mean"])-float(x["control_mean"])) for x in cs)/sum(float(x["weight"]) for x in cs))
    splits=read(OUT/"split_counts.csv")
    for row in splits:
        boundary=datetime(2021,int(row["month"]),1)
        subset=[r for r in hourly if truth(r["eligible_onset"]) and truth(r["persistence_known"])]
        if row["partition"]=="train":
            subset=[r for r in subset if datetime.fromisoformat(r["timestamp"])<boundary and datetime.fromisoformat(r["label_confirmed_at"])<boundary]
        else:
            subset=[r for r in subset if r["month"]==row["month"]]
        sign=1 if row["direction"]=="up" else -1
        assert int(row["hours"])==len(subset)
        assert int(row["onsets"])==sum(int(r["onset"])==sign for r in subset)
        assert int(row["sustained3"])==sum(float(r["sustained_onset"])==sign for r in subset)
    # Description of pre-onset conditions within events: short vs persistent.
    comparisons=[]
    event_map={r["timestamp"]:r for r in events if int(r["month"])<=3}
    for direction in ["up","down"]:
        groups={flag:[r for r in early if r["timestamp"] in event_map and event_map[r["timestamp"]]["direction"]==direction
                     and event_map[r["timestamp"]]["sustained3"]==flag] for flag in ["True","False"]}
        for feature in FEATURES:
            def avg(flag):
                rs=groups[flag]
                return sum(float(r[feature]) for r in rs)/len(rs) if rs else None
            comparisons.append({"direction":direction,"feature":feature,"persistent_n":len(groups["True"]),
                "short_n":len(groups["False"]),"persistent_mean":avg("True"),"short_mean":avg("False"),
                "scope":"Jan-Mar unmatched event-only description"})
    pd.DataFrame(comparisons).to_csv(OUT/"persistence_precursors.csv",index=False,encoding="utf-8-sig")
    # Prefix invariance: change future records and require all earlier onset/state outputs unchanged.
    prefix_tests=0
    for cut in [datetime(2021,1,16,8),datetime(2021,3,7,8),datetime(2021,6,15)]:
        prefix_times=[t for t in times if t<cut]
        trimmed={t:v for t,v in raw.items() if t<cut}
        rebuilt,_=reconstruct(trimmed,prefix_times,c["thresholds"])
        for t in prefix_times:
            assert rebuilt[t]["onset"]==independent[t]["onset"]
            assert rebuilt[t]["state"]==independent[t]["state"]
        prefix_tests+=1
    # Synthetic cases establish opposite direction, repeated suppression, short persistence and gaps.
    from regime_diagnose import label
    synthetic=[]
    for values,expected in [([100.]*6+[200.]*6,[(6,1)]),
                           ([100.]*6+[200.,100.,100.,100.],[(6,1),(7,-1)]),
                           ([100.]*6+[20.]*6,[(6,-1)]),
                           ([100.]*6+[200.,200.,np.nan,200.,200.],[(6,1)])]:
        ts=pd.date_range("2021-01-01",periods=len(values),freq="h")
        f=pd.DataFrame(index=ts)
        f["level"]=values;f["maximum"]=values;f["valid"]=f.level.notna()
        h=pd.concat([f.level.shift(k) for k in range(1,7)],axis=1)
        f["eligible_onset"]=f.valid&h.notna().all(axis=1)
        f["past_median6"]=h.median(axis=1);f["past_iqr6"]=h.quantile(.75,axis=1)-h.quantile(.25,axis=1)
        f["prior_mean"]=f.level.shift(1);f["delta"]=f.level-f.prior_mean;f["deviation"]=f.level-f.past_median6
        f["date"]=ts.normalize();f["month"]=1;f["hour"]=ts.hour;f["weekend"]=0;f["profile"]="synthetic"
        f["prior_production"]=0.;f["생산량"]=0.
        labeled,ev=label(f,{"up":50.,"down":40.})
        got=[(int(r.position),1 if r.direction=="up" else -1) for _,r in ev.iterrows()]
        # A one-hour spike returning to baseline is not a new fall from a stable high regime:
        # the 6h median is still 100, hence no separate qualifying downward onset.
        if values[6:8]==[200.,100.]:
            expected=[(6,1)]
        assert got==expected,(got,expected)
        if len(values)==12:
            assert bool(ev.sustained3.iloc[0])
        if any(pd.isna(v) for v in values):
            assert pd.isna(ev.sustained3.iloc[0])
        synthetic.append({"events":got,"persistence":str(ev.sustained3.iloc[0])})
    result={"status":"passed","run_sha256":sha(OUT/"run.json"),"verifier_sha256":sha(__file__),
            "raw_hours_checked":len(raw),"hourly_labels_checked":len(hourly),"events_checked":len(events),
            "matched_cells_checked":len(cells),"split_records_checked":len(splits),
            "prefix_tests":prefix_tests,"synthetic_cases":synthetic,"new_fits":0,
            "extra_outputs_sha256":{"persistence_precursors.csv":sha(OUT/"persistence_precursors.csv")}}
    (OUT/"independent_verification.json").write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding="utf-8")
    print(json.dumps(result,ensure_ascii=False),flush=True)


if __name__=="__main__":
    main()


코드 등록 완료: `Modeling/scripts/regime_diagnose_verify.py`

### `Modeling/scripts/regime_followup.py`

원고 연결: Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_원고.md.

코드 설명: S05 fixed June-end upward classifier and B0/B1 routing, July-August follow-up.

Freeze using only previous artifacts, then run once. Future labels are diagnostic.

주요 함수: `sha()`, `jsave()`, `inputs()`, `contracts()`, `classification_train()`, `add_calendar()`, `freeze()`, `extended()`, `metrics()`, `detection()`, `run()`.

In [89]:
%%submission_source Modeling/scripts/regime_followup.py
"""S05 fixed June-end upward classifier and B0/B1 routing, July-August follow-up.

Freeze using only previous artifacts, then run once. Future labels are diagnostic.
"""
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
import hashlib
import json
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from regime_diagnose import label
from regime_forecast import model, threshold, calendar_scores

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/regime_followup"
MODELS = ROOT / "Modeling/models/regime_followup"


def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def jsave(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def inputs():
    names = ["data/origin/okm_augumented_2021.csv", "Modeling/tables/m01/hourly_frame.csv", "Modeling/tables/m01/verification.json",
             "Modeling/tables/regime_diagnosis/hourly_labels.csv", "Modeling/tables/regime_diagnosis/contract.json",
             "Modeling/tables/regime_forecast/contract.json", "Modeling/tables/regime_forecast/predictions.csv",
             "Modeling/tables/regime_forecast/independent_verification.json", "Modeling/tables/regime_integration/contract.json",
             "Modeling/tables/regime_integration/meta_features.csv", "Modeling/tables/regime_integration/independent_verification.json",
             "Modeling/tables/regime_routing/contract.json", "Modeling/tables/regime_routing/decision.json",
             "Modeling/tables/regime_routing/independent_verification.json", "Modeling/scripts/regime_diagnose.py",
             "Modeling/scripts/regime_forecast.py"]
    return {n: sha(ROOT / n) for n in names}


def contracts():
    return [json.loads((ROOT / f"Modeling/tables/{folder}/contract.json").read_text(encoding="utf-8"))
            for folder in ["regime_diagnosis", "regime_forecast", "regime_integration"]]


def classification_train():
    f = pd.read_csv(ROOT / "Modeling/tables/regime_diagnosis/hourly_labels.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at"])
    f = f.loc[f.eligible_onset & f.persistence_known & (f.label_confirmed_at < "2021-07-01")].copy()
    return add_calendar(f)


def add_calendar(f):
    f = f.copy()
    f["hour_sin"] = np.sin(2 * np.pi * f.timestamp.dt.hour / 24)
    f["hour_cos"] = np.cos(2 * np.pi * f.timestamp.dt.hour / 24)
    for i in range(7):
        f[f"dow_{i}"] = (f.timestamp.dt.dayofweek == i).astype(int)
    return f


def freeze():
    OUT.mkdir(parents=True, exist_ok=True)
    assert not (OUT / "contract.json").exists()
    for folder in ["regime_forecast", "regime_integration", "regime_routing"]:
        verified = json.loads((ROOT / f"Modeling/tables/{folder}/independent_verification.json").read_text(encoding="utf-8"))
        assert verified["status"] == "passed" and verified["run_sha256"] == sha(ROOT / f"Modeling/tables/{folder}/run.json")
    d, s, r = contracts()
    cp = pd.read_csv(ROOT / "Modeling/tables/regime_forecast/predictions.csv", encoding="utf-8-sig", parse_dates=["label_confirmed_at"])
    cp = cp.loc[(cp.direction == "up") & cp.month.between(4, 6) & (cp.label_confirmed_at < "2021-07-01")]
    calibrated = {}
    for method in ["logistic", "calendar"]:
        g = cp.loc[cp.method == method]
        t, tp, fp, cap = threshold(g.event.to_numpy(int), g.score.to_numpy(float))
        calibrated[method] = {"threshold": float(t) if np.isfinite(t) else "infinity", "tp": int(tp), "fp": int(fp), "fp_cap": int(cap),
                              "hours": len(g), "events": int(g.event.sum()), "latest_label_confirmation": str(g.label_confirmed_at.max())}
    ct = classification_train()
    rt = pd.read_csv(ROOT / "Modeling/tables/regime_integration/meta_features.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at"])
    rt = rt.loc[rt.label_confirmed_at < "2021-07-01"]
    assert len(ct) == 4336 and len(rt) == 3598
    c = {"stage": "S05", "created_at": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
         "entrypoint_sha256": sha(__file__), "inputs_sha256": inputs(), "onset_definition": d,
         "classification_features": s["feature_names"], "logistic": s["logistic"],
         "regression_features": {name: r["features"][name] for name in ["B0", "B1"]}, "hgb_parameters": r["hgb_parameters"],
         "training": "Classification Jan-Jun4336 matured labels; regression Feb-Jun3598 common genuine previous-month probability rows. One fit per model before July; no July/August fitting or parameter changes.",
         "models": ["up_logistic", "B0", "B1"], "classification_thresholds": calibrated,
         "threshold_policy": "Apply existing S03 threshold algorithm once to Apr-Jun previous-month predictions, FPR<=1%, fixed for both July/August. No in-sample calibration.",
         "routing": "G1 selects B1 only when fixed Logistic score>=fixed threshold, else B0. Compare B0,B1_all,G1,hour08,simple_gate.",
         "regression_flags": "Both classifier-availability flags1 at July/August forecast origin; June training had sufficient support. No down classifier prediction is needed by chosen B1/routing.",
         "scope": "Already observed July/August follow-up, not a new independent test. Do not retune from these results.",
         "availability": "Classification requires six exact earlier valid hourly records; regression additionally M01 common lags1/2/24/168. Predict whenever past inputs complete and current target valid; future persistence missing does not remove regression prediction. Classification metrics separately require observed t,t+1,t+2.",
         "classification_gate": {"min_events": 5, "recall_at_least": .8, "precision_at_least": .8, "fpr_at_most": .01},
         "routing_gate": {"min_events": 5, "start_mae_reduction_vs_B0_at_least": .05, "up_under_reduction_vs_B0_at_least": .05,
                          "overall_mae_worsening_vs_B0_at_most": .01, "daily_peak_mae_worsening_vs_B0_at_most": .01,
                          "monthly_results": "Diagnostic, no demand for every-month win; always report."},
         "coverage": "Report normal/invalid/gap hours, six-history vs common counts, known/unknown labels and excluded persistent events. Evaluate both classification full-six-history and primary common-regression scope.",
         "metrics": "AP with zero-event months undefined; TP/FP/FN/TN/precision/recall, onset/window MAE/under/over, full24-hour day-equal peak MAE, month/profile conditions, same-target paired errors and routing counts.",
         "preflight": {"classification_train": len(ct), "regression_train": len(rt), "training_up_events": int((ct.sustained_onset == 1).sum())},
         "no_tuning": True, "no_clipping": True, "no_root_readme_or_manuscript_edit": True}
    jsave(OUT / "contract.json", c)
    print(json.dumps({"status": "frozen", "preflight": c["preflight"], "thresholds": calibrated}), flush=True)


def extended(c):
    raw = pd.read_csv(ROOT / "data/origin/okm_augumented_2021.csv", encoding="utf-8-sig")
    raw = raw.loc[raw["날짜"].between(20210101, 20210831) & raw["시간"].between(0, 23)].copy()
    raw["timestamp"] = pd.to_datetime(raw["날짜"].astype(str), format="%Y%m%d") + pd.to_timedelta(raw["시간"], unit="h")
    raw = raw.sort_values("timestamp").set_index("timestamp")
    assert raw.index.is_unique
    f = raw.reindex(pd.date_range("2021-01-01", "2021-08-31 23:00", freq="h")); f.index.name = "timestamp"
    slots = ["15분", "30분", "45분", "60분"]
    f["valid"] = f[slots].notna().all(axis=1); f["level"] = f[slots].mean(axis=1).where(f.valid)
    f["maximum"] = f[slots].max(axis=1).where(f.valid)
    f["date"] = f.index.normalize(); f["month"] = f.index.month; f["hour"] = f.index.hour
    f["weekend"] = (f.index.dayofweek >= 5).astype(int)
    profiles = {date: hashlib.sha256(g[slots].to_numpy(dtype=np.int64).tobytes()).hexdigest() for date, g in raw.groupby(raw.index.normalize())}
    f["profile"] = f.date.map(profiles)
    h = pd.concat([f.level.shift(k).rename(str(k)) for k in range(1, 7)], axis=1)
    f["eligible_onset"] = f.valid & h.notna().all(axis=1); f["prior_mean"] = h["1"]
    f["past_median6"] = h.median(axis=1).where(f.eligible_onset)
    f["past_iqr6"] = (h.quantile(.75, axis=1) - h.quantile(.25, axis=1)).where(f.eligible_onset)
    f["past_range6"] = (h.max(axis=1) - h.min(axis=1)).where(f.eligible_onset)
    f["past_slope6"] = ((h["1"] - h["6"]) / 5).where(f.eligible_onset)
    f["prior_delta"] = f.level.shift(1) - f.level.shift(2)
    f["prior_last_minus_mean"] = f["60분"].shift(1) - f.prior_mean
    f["prior_slot_range"] = (f[slots].max(axis=1) - f[slots].min(axis=1)).shift(1).where(f.valid.shift(1, fill_value=False))
    f["prior_production"] = f["생산량"].shift(1); f["prior_production_delta"] = f["생산량"].shift(1) - f["생산량"].shift(2)
    f["delta"] = f.level - f.prior_mean; f["deviation"] = f.level - f.past_median6
    z, events = label(f, c["onset_definition"]["thresholds"])
    return add_calendar(z.reset_index()), events


def metrics(g):
    if not len(g):
        return {"hours": 0, **{n: None for n in ["mae", "rmse", "bias", "under", "over"]}}
    err = g.prediction.to_numpy() - g.actual.to_numpy()
    return {"hours": len(g), "mae": float(np.mean(abs(err))), "rmse": float(np.sqrt(np.mean(err ** 2))),
            "bias": float(np.mean(err)), "under": float(np.mean(np.maximum(-err, 0))), "over": float(np.mean(np.maximum(err, 0)))}


def detection(g):
    from sklearn.metrics import average_precision_score
    y, a = g.event.to_numpy(int), g.alarm.to_numpy(int)
    tp = int(((y == 1) & (a == 1)).sum()); fp = int(((y == 0) & (a == 1)).sum())
    fn = int(y.sum()) - tp; tn = len(g) - int(y.sum()) - fp
    return {"hours": len(g), "events": int(y.sum()), "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "recall": tp / (tp + fn) if tp + fn else None, "precision": tp / (tp + fp) if tp + fp else None,
            "fpr": fp / (fp + tn) if fp + tn else None, "ap": float(average_precision_score(y, g.score)) if y.sum() else None}


def run():
    assert not (OUT / "run.json").exists()
    c = json.loads((OUT / "contract.json").read_text(encoding="utf-8"))
    assert c["entrypoint_sha256"] == sha(__file__) and c["inputs_sha256"] == inputs()
    z, events = extended(c)
    ct = classification_train(); rt = pd.read_csv(ROOT / "Modeling/tables/regime_integration/meta_features.csv", encoding="utf-8-sig", parse_dates=["label_confirmed_at"])
    rt = rt.loc[rt.label_confirmed_at < "2021-07-01"]
    s = json.loads((ROOT / "Modeling/tables/regime_forecast/contract.json").read_text(encoding="utf-8"))
    y = (ct.sustained_onset == 1).astype(int).to_numpy(); classifier = model("logistic", s)
    classifier.fit(ct[c["classification_features"]], y)
    MODELS.mkdir(parents=True, exist_ok=True); manifest = []
    def keep(fitted, name, count):
        path = MODELS / (name + ".joblib"); assert not path.exists(); joblib.dump(fitted, path)
        manifest.append({"name": name, "file": path.relative_to(ROOT).as_posix(), "sha256": sha(path), "train_hours": count, "trained_before": "2021-07-01"})
    keep(classifier, "up_logistic", len(ct))
    class_eval = z.loc[z.month >= 7].copy()
    class_eval = class_eval.loc[class_eval.eligible_onset]
    score = classifier.predict_proba(class_eval[c["classification_features"]])[:, 1]
    cal_score = calendar_scores(ct, class_eval, y)
    for method, sc in [("logistic", score), ("calendar", cal_score)]:
        class_eval["score_" + method] = sc
        t = c["classification_thresholds"][method]["threshold"]
        class_eval["alarm_" + method] = (sc >= (np.inf if t == "infinity" else t)).astype(int)
    class_eval["event"] = np.where(class_eval.persistence_known, (class_eval.sustained_onset == 1).astype(int), np.nan)
    class_eval["seen_classification_profile"] = class_eval.profile.isin(ct.profile).astype(int)
    m = pd.read_csv(ROOT / "Modeling/tables/m01/hourly_frame.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    base = m.loc[m.eligible_common & (m.month >= 7), ["timestamp", "target_maximum"] + c["regression_features"]["B0"]]
    extra = [n for n in c["regression_features"]["B1"] if n not in c["regression_features"]["B0"] and not n.endswith("classifier_available")]
    f = class_eval[["timestamp", "label_confirmed_at", "latest_input_timestamp", "profile", "level", "maximum", "onset", "sustained_onset", "persistence_known", "event", "score_logistic", "score_calendar", "alarm_logistic", "alarm_calendar"] + extra].merge(base, on="timestamp", validate="one_to_one")
    f["up_classifier_available"] = 1; f["down_classifier_available"] = 1
    f["seen_regression_profile"] = f.profile.isin(rt.profile).astype(int)
    assert np.allclose(f.maximum, f.target_maximum)
    fitted_reg = {}
    for name in ["B0", "B1"]:
        fitted = HistGradientBoostingRegressor(**c["hgb_parameters"])
        fitted.fit(rt[c["regression_features"][name]], rt.target_maximum); keep(fitted, name, len(rt))
        fitted_reg[name] = fitted.predict(f[c["regression_features"][name]])
    b0, b1 = fitted_reg["B0"], fitted_reg["B1"]
    options = {"B0": b0, "B1_all": b1, "G1": np.where(f.alarm_logistic == 1, b1, b0),
               "hour08": np.where(f.timestamp.dt.hour == 8, b1, b0), "simple_gate": np.where(f.alarm_calendar == 1, b1, b0)}
    masks = {"B0": np.zeros(len(f)), "B1_all": np.ones(len(f)), "G1": f.alarm_logistic,
             "hour08": (f.timestamp.dt.hour == 8).astype(int), "simple_gate": f.alarm_calendar}
    pred = []; metric_rows = []; event_rows = []; window_coverage = []; daily_rows = []; burden = []
    for variant, yhat in options.items():
        g = f.copy(); g["variant"] = variant; g["actual"] = g.target_maximum; g["prediction"] = yhat
        g["routed"] = np.asarray(masks[variant], int); g["date"] = g.timestamp.dt.strftime("%Y-%m-%d"); pred.append(g)
        for period, part in [("Jul-Aug", g), ("7", g.loc[g.month == 7]), ("8", g.loc[g.month == 8])]:
            for condition, gg in [("all", part), ("up_start", part.loc[part.sustained_onset == 1]),
                                  ("down_start", part.loc[part.sustained_onset == -1]), ("other_known_hours", part.loc[part.persistence_known & (part.sustained_onset != 1)])]:
                metric_rows.append({"variant": variant, "period": period, "condition": condition, **metrics(gg)})
            for seen, gg in part.groupby("seen_regression_profile"):
                metric_rows.append({"variant": variant, "period": period, "condition": f"profile_{seen}", **metrics(gg)})
            burden.append({"variant": variant, "period": period, "hours": len(part), "routed": int(part.routed.sum()),
                           "routed_up_starts": int(((part.routed == 1) & (part.sustained_onset == 1)).sum()),
                           "routed_non_up_known": int(((part.routed == 1) & part.persistence_known & (part.sustained_onset != 1)).sum()),
                           "routed_unknown": int(((part.routed == 1) & ~part.persistence_known).sum())})
        for r in g.loc[g.sustained_onset.isin([1, -1])].itertuples():
            times = pd.date_range(r.timestamp, periods=3, freq="h")
            window = g.loc[g.timestamp.isin(times)]; complete = len(window) == 3
            window_coverage.append({"variant": variant, "timestamp": r.timestamp, "direction": "up" if r.sustained_onset == 1 else "down", "complete": complete})
            if complete:
                event_rows.append({"variant": variant, "timestamp": r.timestamp, "month": r.month, "direction": "up" if r.sustained_onset == 1 else "down", **metrics(window)})
        for date, gg in g.groupby("date"):
            if len(gg) == 24:
                peak = gg.loc[gg.actual == gg.actual.max()]
                daily_rows.append({"variant": variant, "date": date, "month": int(gg.month.iloc[0]), "ties": len(peak), **metrics(peak)})
    p = pd.concat(pred, ignore_index=True); daily = pd.DataFrame(daily_rows); ew = pd.DataFrame(event_rows)
    for variant in options:
        for period in ["Jul-Aug", "7", "8"]:
            for condition, source in [("daily_peak_day_equal", daily.loc[daily.variant == variant]),
                                      ("up_window_event_equal", ew.loc[(ew.variant == variant) & (ew.direction == "up")])]:
                gg = source if period == "Jul-Aug" else source.loc[source.month == int(period)]
                result = {"hours": int(gg.ties.sum()) if condition.startswith("daily") else len(gg) * 3, "units": len(gg)}
                result.update({n: float(gg[n].mean()) if len(gg) else None for n in ["mae", "under", "over", "bias"]})
                result["rmse"] = float(np.sqrt(np.mean(gg.rmse ** 2))) if len(gg) else None
                metric_rows.append({"variant": variant, "period": period, "condition": condition, **result})
    det = []
    for scope, part in [("six_history", class_eval), ("common_regression", class_eval.loc[class_eval.timestamp.isin(f.timestamp)])]:
        part = part.loc[part.persistence_known]
        for period, gg in [("Jul-Aug", part), ("7", part.loc[part.month == 7]), ("8", part.loc[part.month == 8])]:
            for method in ["logistic", "calendar"]:
                det.append({"scope": scope, "period": period, "method": method,
                            **detection(gg.assign(score=gg["score_" + method], alarm=gg["alarm_" + method]))})
    table = pd.DataFrame(metric_rows); detection_table = pd.DataFrame(det)
    def get(v, condition):
        return table.loc[(table.variant == v) & (table.period == "Jul-Aug") & (table.condition == condition)].iloc[0]
    a, b = get("G1", "up_start"), get("B0", "up_start")
    gain = 1 - a.mae / b.mae if b.mae else None; under_gain = 1 - a.under / b.under if b.under else None
    all_change = get("G1", "all").mae / get("B0", "all").mae - 1
    peak_change = get("G1", "daily_peak_day_equal").mae / get("B0", "daily_peak_day_equal").mae - 1
    dr = detection_table.loc[(detection_table.scope == "common_regression") & (detection_table.period == "Jul-Aug") & (detection_table.method == "logistic")].iloc[0]
    detection_pass = bool(dr.events >= 5 and dr.recall is not None and dr.recall >= .8 and dr.precision is not None and dr.precision >= .8 and dr.fpr <= .01)
    routing_pass = bool(a.hours >= 5 and gain is not None and gain >= .05 and under_gain is not None and under_gain >= .05 and all_change <= .01 and peak_change <= .01)
    coverage_rows = []
    for month in [7, 8]:
        zz = z.loc[z.month == month]; hh = class_eval.loc[class_eval.month == month]; ff = f.loc[f.month == month]
        ev = events.loc[(events.month == month) & (events.direction == "up") & events.sustained3.eq(True)]
        coverage_rows.append({"month": month, "calendar_hours": len(zz), "valid_hours": int(zz.valid.sum()), "missing_hours": int((~zz.valid).sum()),
                              "six_history_prediction_hours": len(hh), "six_history_known_label_hours": int(hh.persistence_known.sum()),
                              "common_prediction_hours": len(ff), "common_known_label_hours": int(ff.persistence_known.sum()),
                              "all_observed_persistent_up_events": len(ev), "persistent_up_events_in_common": int((ff.sustained_onset == 1).sum()),
                              "excluded_persistent_up_events": int((~ev.timestamp.isin(ff.timestamp)).sum())})
    outputs = {"hourly_extended.csv": z, "events_extended.csv": events, "classification_predictions.csv": class_eval,
               "evaluation_frame.csv": f, "predictions.csv": p, "metrics.csv": table, "detection_metrics.csv": detection_table,
               "coverage.csv": pd.DataFrame(coverage_rows), "routing_burden.csv": pd.DataFrame(burden),
               "event_windows.csv": ew, "event_window_coverage.csv": pd.DataFrame(window_coverage), "daily_peak_errors.csv": daily,
               "model_manifest.csv": pd.DataFrame(manifest)}
    decision = {"classification_gate_passed": detection_pass, "routing_gate_passed": routing_pass,
                "start_mae_reduction": gain, "up_under_reduction": under_gain,
                "overall_mae_change": all_change, "daily_peak_mae_change": peak_change,
                "scope": c["scope"], "no_retuning": True}
    for name, frame in outputs.items():
        frame.to_csv(OUT / name, index=False, encoding="utf-8-sig")
    jsave(OUT / "decision.json", decision)
    jsave(OUT / "run.json", {"status": "completed", "script_sha256": sha(__file__), "contract_sha256": sha(OUT / "contract.json"),
                             "inputs_sha256": c["inputs_sha256"], "new_fits": len(manifest), "outputs_sha256": {n: sha(OUT / n) for n in list(outputs) + ["decision.json"]}, "runtime": sys.version})
    print(json.dumps({"status": "completed", "coverage": coverage_rows, "decision": decision}), flush=True)


if __name__ == "__main__":
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    {"freeze": freeze, "run": run}[sys.argv[1]]()


코드 등록 완료: `Modeling/scripts/regime_followup.py`

### `Modeling/scripts/regime_followup_verify.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Independent raw-source/state/timing/model/error verification of fixed S05.

주요 함수: `sha()`, `close()`, `stats()`, `main()`.

In [90]:
%%submission_source Modeling/scripts/regime_followup_verify.py
"""Independent raw-source/state/timing/model/error verification of fixed S05."""
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import csv
import hashlib
import json
import math
import sys
from datetime import datetime, timedelta
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from regime_diagnose_verify import reconstruct
from regime_forecast_verify import average_precision

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/regime_followup"


def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def close(a, b):
    if pd.isna(a) or b is None:
        assert pd.isna(a) and b is None, (a, b)
    else:
        assert math.isclose(float(a), float(b), rel_tol=1e-9, abs_tol=1e-8), (a, b)


def stats(g):
    err = [float(r.prediction) - float(r.actual) for r in g.itertuples()]
    if not err:
        return {"hours": 0, **{n: None for n in ["mae", "rmse", "bias", "under", "over"]}}
    return {"hours": len(err), "mae": math.fsum(abs(e) for e in err) / len(err),
            "rmse": math.sqrt(math.fsum(e * e for e in err) / len(err)), "bias": math.fsum(err) / len(err),
            "under": math.fsum(max(-e, 0) for e in err) / len(err), "over": math.fsum(max(e, 0) for e in err) / len(err)}


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    c = json.loads((OUT / "contract.json").read_text(encoding="utf-8"))
    run = json.loads((OUT / "run.json").read_text(encoding="utf-8"))
    assert run["status"] == "completed" and run["contract_sha256"] == sha(OUT / "contract.json")
    assert c["entrypoint_sha256"] == run["script_sha256"] == sha(ROOT / "Modeling/scripts/regime_followup.py")
    for name, digest in c["inputs_sha256"].items():
        assert sha(ROOT / name) == digest
    for name, digest in run["outputs_sha256"].items():
        assert sha(OUT / name) == digest
    raw = {}
    with (ROOT / "data/origin/okm_augumented_2021.csv").open(encoding="utf-8-sig", newline="") as stream:
        for row in csv.DictReader(stream):
            date, hour = int(row["날짜"]), int(row["시간"])
            if not (20210101 <= date <= 20210831 and 0 <= hour < 24):
                continue
            t = datetime.strptime(str(date), "%Y%m%d") + timedelta(hours=hour)
            assert t not in raw
            slots = [float(row[n]) for n in ["15분", "30분", "45분", "60분"]]
            raw[t] = {"slots": slots, "mean": math.fsum(slots) / 4, "maximum": max(slots), "production": float(row["생산량"])}
    times = [datetime(2021, 1, 1) + timedelta(hours=i) for i in range(243 * 24)]
    rebuilt, starts = reconstruct(raw, times, c["onset_definition"]["thresholds"])
    z = pd.read_csv(OUT / "hourly_extended.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at", "latest_input_timestamp"])
    assert z.timestamp.tolist() == times
    for r in z.itertuples():
        t = r.timestamp.to_pydatetime(); expect = rebuilt[t]
        assert r.valid == (t in raw) and r.eligible_onset == expect["eligible"]
        assert r.candidate == expect["candidate"] and r.onset == expect["onset"]
        assert r.prior_state_age == expect["state"]["age"] and r.prior_state_direction == expect["state"]["direction"]
        assert r.prior_state_left_censored == expect["state"]["censored"]
        assert r.latest_input_timestamp == t - timedelta(hours=1) and r.label_confirmed_at == t + timedelta(hours=2)
        known = all(t + timedelta(hours=k) in raw for k in [0, 1, 2])
        assert r.persistence_known == known
        target = None if not known else 0
        if known and expect["onset"]:
            sign = expect["onset"]; baseline = expect["features"]["past_median6"]
            boundary = c["onset_definition"]["thresholds"]["up" if sign == 1 else "down"] / 2
            target = sign if all(sign * (raw[t + timedelta(hours=k)]["mean"] - baseline) >= boundary for k in [0, 1, 2]) else 0
        close(r.sustained_onset, target)
        if t in raw:
            close(r.level, raw[t]["mean"]); close(r.maximum, raw[t]["maximum"])
        if expect["eligible"]:
            for feature, value in expect["features"].items():
                close(getattr(r, feature), value)
    old = pd.read_csv(ROOT / "Modeling/tables/regime_diagnosis/hourly_labels.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    joined = old.merge(z, on="timestamp", suffixes=("_old", "_new"))
    for name in ["onset", "candidate", "prior_state_age", "prior_state_direction", "prior_state_left_censored"]:
        assert joined[name + "_old"].tolist() == joined[name + "_new"].tolist()
    cls = pd.read_csv(OUT / "classification_predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    frame = pd.read_csv(OUT / "evaluation_frame.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    ct = z.loc[z.eligible_onset & z.persistence_known & (z.label_confirmed_at < "2021-07-01")]
    assert len(ct) == 4336 and ct.timestamp.max() < pd.Timestamp("2021-07-01")
    assert cls.timestamp.tolist() == z.loc[(z.month >= 7) & z.eligible_onset, "timestamp"].tolist()
    rt = pd.read_csv(ROOT / "Modeling/tables/regime_integration/meta_features.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at"])
    rt = rt.loc[rt.label_confirmed_at < "2021-07-01"]
    m = pd.read_csv(ROOT / "Modeling/tables/m01/hourly_frame.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    expected_times = sorted(set(cls.timestamp) & set(m.loc[(m.month >= 7) & m.eligible_common, "timestamp"]))
    assert frame.timestamp.tolist() == expected_times and len(rt) == 3598
    original = m.set_index("timestamp").loc[frame.timestamp]
    for name in c["regression_features"]["B0"] + ["target_maximum"]:
        np.testing.assert_allclose(frame[name], original[name], rtol=0, atol=1e-8)
    assert (frame.up_classifier_available == 1).all() and (frame.down_classifier_available == 1).all()
    for features in [c["classification_features"], *c["regression_features"].values()]:
        assert not set(features) & {"event", "target_maximum", "maximum", "level", "onset", "sustained_onset", "profile", "delta", "deviation"}
    manifest = pd.read_csv(OUT / "model_manifest.csv", encoding="utf-8-sig")
    models = {}; fresh_predictions = {}; refits = 0
    for r in manifest.itertuples():
        assert sha(ROOT / r.file) == r.sha256 and r.trained_before == "2021-07-01"
        fitted = joblib.load(ROOT / r.file); models[r.name] = fitted
        if r.name == "up_logistic":
            features = c["classification_features"]; y = (ct.sustained_onset == 1).astype(int)
            np.testing.assert_allclose(cls.score_logistic, fitted.predict_proba(cls[features])[:, 1], atol=1e-8, rtol=1e-9)
            fresh = make_pipeline(StandardScaler(), LogisticRegression(C=1., max_iter=3000, random_state=42))
            fresh.fit(ct[features], y)
            np.testing.assert_allclose(cls.score_logistic, fresh.predict_proba(cls[features])[:, 1], atol=1e-8, rtol=1e-9)
            assert r.train_hours == len(ct)
        else:
            features = c["regression_features"][r.name]; fresh = HistGradientBoostingRegressor(**c["hgb_parameters"])
            fresh.fit(rt[features], rt.target_maximum)
            fresh_predictions[r.name] = fresh.predict(frame[features])
            np.testing.assert_allclose(fresh_predictions[r.name], fitted.predict(frame[features]), atol=1e-8, rtol=1e-9)
            assert r.train_hours == len(rt)
        refits += 1
    cal = pd.read_csv(ROOT / "Modeling/tables/regime_forecast/predictions.csv", encoding="utf-8-sig", parse_dates=["label_confirmed_at"])
    cal = cal.loc[(cal.direction == "up") & cal.month.between(4, 6) & (cal.label_confirmed_at < "2021-07-01")]
    for method in ["logistic", "calendar"]:
        mc = cal.loc[cal.method == method]; cap = int((len(mc) - mc.event.sum()) // 100)
        candidates = []
        for threshold in [math.inf] + sorted(mc.score.unique(), reverse=True):
            al = mc.loc[mc.score >= threshold]; tp = int(al.event.sum()); fp = len(al) - tp
            if fp <= cap:
                candidates.append((tp, -fp, threshold))
        tp, negfp, th = max(candidates); rec = c["classification_thresholds"][method]
        actual_th = math.inf if rec["threshold"] == "infinity" else rec["threshold"]
        close(th, actual_th); assert (tp, -negfp, cap, len(mc)) == (rec["tp"], rec["fp"], rec["fp_cap"], rec["hours"])
        assert cls["alarm_" + method].tolist() == (cls["score_" + method] >= actual_th).astype(int).tolist()
    # Calendar probabilities use only the independently rebuilt Jan-Jun labels.
    for r in cls.itertuples():
        group = ct.loc[(ct.hour == r.hour) & (ct.weekend == r.weekend)]
        if len(group) < 20:
            group = ct.loc[ct.hour == r.hour]
        if len(group) < 20:
            group = ct
        close(r.score_calendar, ((group.sustained_onset == 1).sum() + .5) / (len(group) + 1))
    p = pd.read_csv(OUT / "predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    assert len(p) == 5 * len(frame) and not p.duplicated(["timestamp", "variant"]).any()
    for variant, g in p.groupby("variant", sort=False):
        assert g.timestamp.tolist() == frame.timestamp.tolist()
        b0, b1 = fresh_predictions["B0"], fresh_predictions["B1"]
        mask = {"B0": np.zeros(len(frame), bool), "B1_all": np.ones(len(frame), bool), "G1": frame.alarm_logistic == 1,
                "hour08": frame.timestamp.dt.hour == 8, "simple_gate": frame.alarm_calendar == 1}[variant]
        np.testing.assert_allclose(g.prediction, np.where(mask, b1, b0), atol=1e-8, rtol=1e-9)
        assert g.routed.tolist() == np.asarray(mask, int).tolist()
    daily = pd.read_csv(OUT / "daily_peak_errors.csv", encoding="utf-8-sig")
    windows = pd.read_csv(OUT / "event_windows.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    for r in daily.itertuples():
        g = p.loc[(p.variant == r.variant) & (p.date == r.date)]; assert len(g) == 24
        peak = g.loc[g.actual == g.actual.max()]; assert len(peak) == r.ties
        for name, value in stats(peak).items():
            close(getattr(r, name), value)
    for r in windows.itertuples():
        g = p.loc[(p.variant == r.variant) & p.timestamp.isin(pd.date_range(r.timestamp, periods=3, freq="h"))]; assert len(g) == 3
        for name, value in stats(g).items():
            close(getattr(r, name), value)
    table = pd.read_csv(OUT / "metrics.csv", encoding="utf-8-sig", dtype={"period": str})
    for r in table.itertuples():
        g = p.loc[p.variant == r.variant]
        if r.period != "Jul-Aug":
            g = g.loc[g.month == int(r.period)]
        if r.condition == "up_start":
            g = g.loc[g.sustained_onset == 1]
        elif r.condition == "down_start":
            g = g.loc[g.sustained_onset == -1]
        elif r.condition == "other_known_hours":
            g = g.loc[g.persistence_known & (g.sustained_onset != 1)]
        elif r.condition.startswith("profile_"):
            g = g.loc[g.seen_regression_profile == int(r.condition[-1])]
        if r.condition.endswith("_equal"):
            source = daily if r.condition.startswith("daily") else windows.loc[windows.direction == "up"]
            g = source.loc[source.variant == r.variant]
            if r.period != "Jul-Aug":
                g = g.loc[g.month == int(r.period)]
            expected = {n: math.fsum(g[n]) / len(g) if len(g) else None for n in ["mae", "bias", "under", "over"]}
            expected["rmse"] = math.sqrt(math.fsum(v * v for v in g.rmse) / len(g)) if len(g) else None
            expected["hours"] = int(g.ties.sum()) if r.condition.startswith("daily") else 3 * len(g)
            assert r.units == len(g)
        else:
            expected = stats(g)
        for name, value in expected.items():
            close(getattr(r, name), value)
    dt = pd.read_csv(OUT / "detection_metrics.csv", encoding="utf-8-sig", dtype={"period": str})
    for r in dt.itertuples():
        g = cls.loc[cls.persistence_known]
        if r.scope == "common_regression":
            g = g.loc[g.timestamp.isin(frame.timestamp)]
        if r.period != "Jul-Aug":
            g = g.loc[g.month == int(r.period)]
        al = g["alarm_" + r.method]; y = g.event.astype(int)
        tp = int(((y == 1) & (al == 1)).sum()); fp = int(((y == 0) & (al == 1)).sum()); fn = int(y.sum()) - tp; tn = len(g) - int(y.sum()) - fp
        assert (r.hours, r.events, r.tp, r.fp, r.fn, r.tn) == (len(g), y.sum(), tp, fp, fn, tn)
        close(r.recall, tp / (tp + fn) if tp + fn else None); close(r.precision, tp / (tp + fp) if tp + fp else None)
        close(r.fpr, fp / (fp + tn) if fp + tn else None)
        close(r.ap, average_precision(g.assign(score=g["score_" + r.method])))
    for r in pd.read_csv(OUT / "routing_burden.csv", encoding="utf-8-sig", dtype={"period": str}).itertuples():
        g = p.loc[p.variant == r.variant]
        if r.period != "Jul-Aug":
            g = g.loc[g.month == int(r.period)]
        assert r.hours == len(g) and r.routed == int(g.routed.sum())
        assert r.routed_up_starts == int(((g.routed == 1) & (g.sustained_onset == 1)).sum())
        assert r.routed_non_up_known == int(((g.routed == 1) & g.persistence_known & (g.sustained_onset != 1)).sum())
        assert r.routed_unknown == int(((g.routed == 1) & ~g.persistence_known).sum())
    cov = pd.read_csv(OUT / "coverage.csv", encoding="utf-8-sig")
    for r in cov.itertuples():
        full = z.loc[z.month == r.month]; own = frame.loc[frame.month == r.month]; six = cls.loc[cls.month == r.month]
        assert r.calendar_hours == len(full) and r.valid_hours == full.valid.sum() and r.missing_hours == (~full.valid).sum()
        assert r.common_prediction_hours == len(own) and r.common_known_label_hours == own.persistence_known.sum()
        assert r.six_history_prediction_hours == len(six) and r.six_history_known_label_hours == six.persistence_known.sum()
        all_events = full.loc[full.sustained_onset == 1]
        assert r.all_observed_persistent_up_events == len(all_events) and r.persistent_up_events_in_common == (own.sustained_onset == 1).sum()
        assert r.excluded_persistent_up_events == (~all_events.timestamp.isin(own.timestamp)).sum()
    d = json.loads((OUT / "decision.json").read_text(encoding="utf-8"))
    def get(v, condition):
        return table.loc[(table.variant == v) & (table.period == "Jul-Aug") & (table.condition == condition)].iloc[0]
    gain = 1 - get("G1", "up_start").mae / get("B0", "up_start").mae
    under = 1 - get("G1", "up_start").under / get("B0", "up_start").under
    overall = get("G1", "all").mae / get("B0", "all").mae - 1
    peak = get("G1", "daily_peak_day_equal").mae / get("B0", "daily_peak_day_equal").mae - 1
    for name, value in [("start_mae_reduction", gain), ("up_under_reduction", under), ("overall_mae_change", overall), ("daily_peak_mae_change", peak)]:
        close(d[name], value)
    dr = dt.loc[(dt.scope == "common_regression") & (dt.period == "Jul-Aug") & (dt.method == "logistic")].iloc[0]
    assert d["classification_gate_passed"] == bool(dr.events >= 5 and dr.recall >= .8 and dr.precision >= .8 and dr.fpr <= .01)
    assert d["routing_gate_passed"] == bool(get("G1", "up_start").hours >= 5 and gain >= .05 and under >= .05 and overall <= .01 and peak <= .01)
    prefix_tests = 0
    for cut in [datetime(2021, 7, 10), datetime(2021, 8, 10)]:
        trimmed, _ = reconstruct({t: v for t, v in raw.items() if t < cut}, [t for t in times if t < cut], c["onset_definition"]["thresholds"])
        for t, item in trimmed.items():
            assert item["state"] == rebuilt[t]["state"] and item["onset"] == rebuilt[t]["onset"]
        prefix_tests += 1
    # Diagnostic only: linear logit contribution and range of features at each known-label alarm.
    scaler, fitted = models["up_logistic"].steps[0][1], models["up_logistic"].steps[1][1]
    features = c["classification_features"]
    alarm_cases = cls.loc[cls.persistence_known & (cls.alarm_logistic == 1)].copy()
    contrib = scaler.transform(alarm_cases[features]) * fitted.coef_[0]
    age_index = features.index("prior_state_age")
    age_min, age_max = float(ct.prior_state_age.min()), float(ct.prior_state_age.max())
    alarm_cases["age_logit_contribution"] = contrib[:, age_index]
    alarm_cases["largest_positive_logit_feature"] = [features[i] for i in contrib.argmax(axis=1)]
    alarm_cases["age_above_training_max"] = alarm_cases.prior_state_age > age_max
    alarm_cases["classification_training_age_max"] = age_max
    # No counterfactual prediction, re-fit, threshold change, or result cherry-picking.
    alarm_cases.to_csv(OUT / "verified_alarm_cases.csv", index=False, encoding="utf-8-sig")
    contribution_summary = []
    for (month, event), g in alarm_cases.groupby(["month", "event"]):
        contribution_summary.append({"month": int(month), "true_event": int(event), "alarms": len(g),
                                     "prior_state_age_min": float(g.prior_state_age.min()), "prior_state_age_max": float(g.prior_state_age.max()),
                                     "age_above_training_max": int(g.age_above_training_max.sum()),
                                     "mean_age_logit_contribution": float(g.age_logit_contribution.mean()),
                                     "age_largest_positive_feature": int((g.largest_positive_logit_feature == "prior_state_age").sum())})
    pd.DataFrame(contribution_summary).to_csv(OUT / "verified_alarm_conditions.csv", index=False, encoding="utf-8-sig")
    result = {"status": "passed", "run_sha256": sha(OUT / "run.json"), "verifier_sha256": sha(__file__),
              "raw_hours_checked": len(raw), "hourly_state_rows_checked": len(z), "regression_predictions_checked": len(p),
              "classification_prediction_hours": len(cls), "models_reloaded": len(models), "independent_refits": refits,
              "metrics_checked": len(table), "detection_metrics_checked": len(dt), "daily_rows_checked": len(daily),
              "prefix_tests": prefix_tests, "no_july_august_training": True,
              "extra_outputs_sha256": {n: sha(OUT / n) for n in ["verified_alarm_cases.csv", "verified_alarm_conditions.csv"]}}
    (OUT / "independent_verification.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result, ensure_ascii=False), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/regime_followup_verify.py`

### `Modeling/scripts/regime_forecast.py`

원고 연결: Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_원고.md.

코드 설명: S03: frozen, chronological one-hour-ahead persistent transition classification.

No regression integration, held-out July/August reads, tuning, or source edits.
Run freeze before run; completed outputs are never overwritten.

주요 함수: `sha()`, `save_json()`, `frozen_inputs()`, `freeze()`, `load_frame()`, `model()`, `calendar_scores()`, `threshold()`, `ap()`, `measures()`, `past_ap()`, `run()`.

In [91]:
%%submission_source Modeling/scripts/regime_forecast.py
"""S03: frozen, chronological one-hour-ahead persistent transition classification.

No regression integration, held-out July/August reads, tuning, or source edits.
Run freeze before run; completed outputs are never overwritten.
"""
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
import hashlib
import json
import math
import sys
from datetime import datetime, timezone, timedelta
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path(__file__).resolve().parents[2]
PARENT = ROOT / "Modeling/tables/regime_diagnosis"
OUT = ROOT / "Modeling/tables/regime_forecast"
MODELS = ROOT / "Modeling/models/regime_forecast"
DYNAMIC = ["prior_mean", "past_median6", "past_iqr6", "past_range6", "past_slope6",
           "prior_delta", "prior_last_minus_mean", "prior_slot_range", "prior_production",
           "prior_production_delta", "prior_state_age", "prior_state_left_censored",
           "prior_state_direction"]
CALENDAR = ["hour_sin", "hour_cos", "month", "weekend"] + [f"dow_{i}" for i in range(7)]
FEATURES = DYNAMIC + CALENDAR
BASE = ["calendar", "last_delta", "slope6"]
ML = ["logistic", "hgb"]
METHODS = BASE + ML


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def frozen_inputs():
    return {str((PARENT / name).relative_to(ROOT)).replace("\\", "/"): sha(PARENT / name)
            for name in ["hourly_labels.csv", "contract.json", "run.json", "independent_verification.json"]}


def freeze():
    OUT.mkdir(parents=True, exist_ok=True)
    assert not (OUT / "contract.json").exists(), "Contract already exists; verify instead of overwrite."
    parent_verify = json.loads((PARENT / "independent_verification.json").read_text(encoding="utf-8"))
    assert parent_verify["status"] == "passed" and parent_verify["run_sha256"] == sha(PARENT / "run.json")
    contract = {
        "stage": "S03", "created_at": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
        "entrypoint_sha256": sha(__file__), "inputs_sha256": frozen_inputs(),
        "scope": "Jan-Jun only; Feb-Jun chronological predictions; Apr-Jun development ranking",
        "targets": {"up": 1, "down": -1}, "horizon": "t, t+1, t+2 sustained onset; predict at end of t-1",
        "feature_names": FEATURES, "forbidden_inputs": ["level", "maximum", "delta", "deviation", "onset", "sustained_onset", "profile"],
        "training": "All earlier eligible known labels with label_confirmed_at strictly before evaluation-month boundary",
        "minimum_fit_support": {"positive_events": 8, "positive_dates": 5},
        "logistic": {"C": 1.0, "max_iter": 3000, "random_state": 42, "class_weight": None, "standardize": "training only"},
        "hgb": {"max_leaf_nodes": 7, "max_iter": 100, "learning_rate": .05, "min_samples_leaf": 20, "l2_regularization": 1.0, "early_stopping": False, "random_state": 42},
        "calendar_baseline": "Jeffreys-smoothed hour x weekend event rate; cell n<20 falls back to hour; hour n<20 to global rate, using training only",
        "other_baselines": {"last_delta": "direction * prior_delta", "slope6": "direction * past_slope6"},
        "calibration": "Earlier genuinely out-of-month predictions with matured labels, intersect available rows across all five methods, never in-sample rescoring. At least one positive required. No calibration means ranking only.",
        "selection": "Separately choose ML and simple baseline by positive-event-weighted monthly AP on common past calibration rows; tie follows fixed method order. Each direction/month selected before evaluation labels are summarized.",
        "threshold": "score >= threshold; calibration FPR <=1%; maximize TP then minimize FP then maximize threshold; include infinity for no alarms",
        "primary_alarm_scope": "Selected ML vs past-selected simple baseline, Apr-Jun where common past calibration exists. Expected up May-Jun, down Apr-Jun.",
        "development_gate": {"extra_tp_at_least": 2, "relative_extra_tp_at_least": .20, "zero_baseline": "use absolute gain only", "extra_fp_per_100_hours_at_most": .5},
        "metrics": "Monthly AP (undefined with zero events), precision, recall, TP/FN/FP/TN, false alarms per100 evaluated hours, negative FPR; no AUROC-only selection",
        "posthoc": "Short-event false positives, late alarms, seen/new daily arrays are diagnostic only; labels/features/thresholds are not revised from these results",
        "tuning_trials": 0, "regression_integration": False,
    }
    save_json(OUT / "contract.json", contract)
    print(json.dumps({"status": "frozen", "contract_sha256": sha(OUT / "contract.json")}), flush=True)


def load_frame():
    f = pd.read_csv(PARENT / "hourly_labels.csv", encoding="utf-8-sig")
    for name in ["timestamp", "label_confirmed_at", "latest_input_timestamp"]:
        f[name] = pd.to_datetime(f[name])
    f = f.loc[f.eligible_onset & f.persistence_known].copy().reset_index(drop=True)
    f["hour_sin"] = np.sin(2 * np.pi * f.hour / 24)
    f["hour_cos"] = np.cos(2 * np.pi * f.hour / 24)
    for i in range(7):
        f[f"dow_{i}"] = (f.timestamp.dt.dayofweek == i).astype(int)
    assert len(f) == 4336 and f.timestamp.max() < pd.Timestamp("2021-07-01")
    assert (f.latest_input_timestamp == f.timestamp - pd.Timedelta(hours=1)).all()
    assert np.isfinite(f[FEATURES].to_numpy(float)).all()
    return f


def model(method, contract):
    if method == "logistic":
        params = {k: v for k, v in contract[method].items() if k != "standardize"}
        return make_pipeline(StandardScaler(), LogisticRegression(**params))
    return HistGradientBoostingClassifier(**contract[method])


def calendar_scores(train, test, y):
    z = train.assign(y=y)
    cell = z.groupby(["hour", "weekend"]).y.agg(["sum", "count"])
    hour = z.groupby("hour").y.agg(["sum", "count"])
    global_rate = (y.sum() + .5) / (len(y) + 1)
    values = []
    for h, w in zip(test.hour, test.weekend):
        r = cell.loc[(h, w)] if (h, w) in cell.index else None
        if r is None or r["count"] < 20:
            r = hour.loc[h] if h in hour.index else None
        values.append((r["sum"] + .5) / (r["count"] + 1) if r is not None and r["count"] >= 20 else global_rate)
    return np.array(values)


def threshold(y, score):
    # Stable descending order, evaluating all tie groups in a single pass.
    order = np.argsort(-score, kind="stable")
    yy, ss = np.asarray(y)[order], np.asarray(score)[order]
    best = (0, 0, math.inf)
    tp = fp = 0
    cap = math.floor(.01 * int((yy == 0).sum()) + 1e-10)
    for i in range(len(yy)):
        tp += int(yy[i] == 1); fp += int(yy[i] == 0)
        if i + 1 < len(yy) and ss[i + 1] == ss[i]:
            continue
        candidate = (tp, -fp, float(ss[i]))
        if fp <= cap and candidate > best:
            best = candidate
    return best[2], best[0], -best[1], cap


def ap(y, score):
    return float(average_precision_score(y, score)) if np.sum(y) else None


def measures(frame):
    y = frame.event.to_numpy(int)
    scores = frame.score.to_numpy(float)
    result = {"hours": len(frame), "events": int(y.sum()), "ap": ap(y, scores)}
    if frame.alarm.notna().all():
        a = frame.alarm.to_numpy(int)
        tp = int(((a == 1) & (y == 1)).sum()); fp = int(((a == 1) & (y == 0)).sum())
        fn = int(y.sum()) - tp; tn = len(y) - int(y.sum()) - fp
        result.update(tp=tp, fp=fp, fn=fn, tn=tn, recall=tp / (tp + fn) if tp + fn else None,
                      precision=tp / (tp + fp) if tp + fp else None, fp_per100h=100 * fp / len(y),
                      fpr=fp / (fp + tn) if fp + tn else None,
                      short_event_fp=int(((a == 1) & (frame.short_event.to_numpy(int) == 1)).sum()))
    return result


def past_ap(cal, method):
    subset = cal.loc[cal.method == method]
    terms = [(len(g.loc[g.event == 1]), ap(g.event.to_numpy(), g.score.to_numpy())) for _, g in subset.groupby("month")]
    return sum(n * value for n, value in terms if n) / sum(n for n, _ in terms) if sum(n for n, _ in terms) else -math.inf


def run():
    assert not (OUT / "run.json").exists(), "Completed run exists; verify instead of overwrite."
    c = json.loads((OUT / "contract.json").read_text(encoding="utf-8"))
    assert c["entrypoint_sha256"] == sha(__file__) and c["inputs_sha256"] == frozen_inputs()
    f = load_frame()
    MODELS.mkdir(parents=True, exist_ok=True)
    predictions = []; manifests = []; folds = []; choices = []; calibration_rows = []
    for direction, sign in c["targets"].items():
        history = []
        for month in range(2, 7):
            boundary = pd.Timestamp(2021, month, 1)
            train = f.loc[(f.timestamp < boundary) & (f.label_confirmed_at < boundary)]
            test = f.loc[f.month == month]
            y = (train.sustained_onset == sign).astype(int).to_numpy()
            dates = train.loc[y == 1, "date"].nunique()
            available = y.sum() >= 8 and dates >= 5 and (y == 0).any()
            fold = {"direction": direction, "month": month, "train_hours": len(train), "train_events": int(y.sum()),
                    "positive_dates": int(dates), "train_latest_confirmed": str(train.label_confirmed_at.max()),
                    "test_hours": len(test), "test_events": int((test.sustained_onset == sign).sum()), "ml_available": bool(available)}
            folds.append(fold)
            scores = {"calendar": calendar_scores(train, test, y), "last_delta": sign * test.prior_delta.to_numpy(),
                      "slope6": sign * test.past_slope6.to_numpy()}
            if available:
                for method in ML:
                    fitted = model(method, c)
                    fitted.fit(train[FEATURES], y)
                    scores[method] = fitted.predict_proba(test[FEATURES])[:, 1]
                    path = MODELS / f"{direction}_{month:02d}_{method}.joblib"
                    assert not path.exists()
                    joblib.dump(fitted, path)
                    manifests.append({**fold, "method": method, "file": str(path.relative_to(ROOT)).replace("\\", "/"),
                                      "sha256": sha(path), "features": "|".join(FEATURES)})
            # Common prior OOF rows prevent comparing one family using a different calibration sample.
            cal = pd.DataFrame(history)
            if len(cal):
                cal = cal.loc[cal.label_confirmed_at < boundary]
                common = cal.groupby("timestamp").method.nunique()
                cal = cal.loc[cal.timestamp.isin(common.index[common == len(METHODS)])].copy()
            calibrated = bool(available and len(cal) and cal.loc[cal.method == "calendar", "event"].sum() > 0)
            selected = {}
            if calibrated:
                for family, members in [("simple", BASE), ("ml", ML)]:
                    selected[family] = max(members, key=lambda m: (past_ap(cal, m), -members.index(m)))
            choice = {"direction": direction, "month": month, "calibration_available": calibrated,
                      "calibration_hours": len(cal.loc[cal.method == "calendar"]) if len(cal) else 0,
                      "calibration_events": int(cal.loc[cal.method == "calendar", "event"].sum()) if len(cal) else 0,
                      "calibration_latest_confirmed": str(cal.label_confirmed_at.max()) if len(cal) else "",
                      "selected_simple": selected.get("simple", ""), "selected_ml": selected.get("ml", "")}
            choices.append(choice)
            current = []
            for method, score in scores.items():
                th = None
                if calibrated:
                    mc = cal.loc[cal.method == method]
                    th, ctp, cfp, cap = threshold(mc.event.to_numpy(int), mc.score.to_numpy(float))
                    calibration_rows.append({"direction": direction, "month": month, "method": method,
                                             "threshold": th, "tp": ctp, "fp": cfp, "fp_cap": cap,
                                             "hours": len(mc), "events": int(mc.event.sum()), "weighted_monthly_ap": past_ap(cal, method)})
                group = test[["timestamp", "label_confirmed_at", "latest_input_timestamp", "profile", "level", "maximum", "prior_mean"]].copy()
                group["direction"] = direction; group["month"] = month; group["method"] = method
                group["event"] = (test.sustained_onset.to_numpy() == sign).astype(int)
                group["short_event"] = ((test.onset.to_numpy() == sign) & (test.sustained_onset.to_numpy() != sign)).astype(int)
                group["seen_profile"] = test.profile.isin(train.profile).astype(int).to_numpy()
                group["score"] = score; group["threshold"] = th
                group["alarm"] = (score >= th).astype(int) if th is not None else np.nan
                group["selected_family"] = [next((family for family, member in selected.items() if member == method), "")] * len(group)
                current.extend(group.to_dict("records"))
            predictions.extend(current); history.extend(current)
            print(json.dumps({**fold, **choice}), flush=True)
    p = pd.DataFrame(predictions)
    monthly = [{"direction": d, "month": m, "method": method, **measures(g)} for (d, m, method), g in p.groupby(["direction", "month", "method"])]
    selected = p.loc[(p.month >= 4) & (p.selected_family != "")].copy()
    aggregate = []; decisions = []
    for direction, g in selected.groupby("direction"):
        stats = {}
        for family, gg in g.groupby("selected_family"):
            # Cross-month pooled AP is not comparable when probability scales change.
            result = measures(gg); result.pop("ap", None)
            aggregate.append({"direction": direction, "family": family, "months": "|".join(map(str, sorted(gg.month.unique()))), **result})
            stats[family] = result
        a, b = stats["ml"], stats["simple"]
        extra = a["tp"] - b["tp"]; relative = extra / b["tp"] if b["tp"] else None
        fp_extra = a["fp_per100h"] - b["fp_per100h"]
        passed = extra >= 2 and (relative is None or relative >= .20) and fp_extra <= .5 + 1e-12
        decisions.append({"direction": direction, "extra_tp": extra, "relative_extra_tp": relative,
                          "extra_fp_per100h": fp_extra, "development_gate_passed": passed, "scope": "exploratory Jan-Jun development; not independent final testing"})
    events = p.loc[(p.month >= 4) & (p.event == 1)].copy()
    # A subsequent warning is late for this onset and never counted as a true advance detection.
    idx = {(r.direction, r.method, r.timestamp): r for r in p.itertuples()}
    events["alarm_one_hour_late"] = [getattr(idx.get((r.direction, r.method, r.timestamp + pd.Timedelta(hours=1))), "alarm", np.nan) for r in events.itertuples()]
    events["alarm_two_hours_late"] = [getattr(idx.get((r.direction, r.method, r.timestamp + pd.Timedelta(hours=2))), "alarm", np.nan) for r in events.itertuples()]
    condition = []
    for (d, family, seen), gg in selected.groupby(["direction", "selected_family", "seen_profile"]):
        result = measures(gg); result.pop("ap", None)
        condition.append({"direction": d, "family": family, "seen_profile": int(seen), **result})
    outputs = {"predictions.csv": p, "monthly_metrics.csv": pd.DataFrame(monthly), "folds.csv": pd.DataFrame(folds),
               "model_manifest.csv": pd.DataFrame(manifests), "selection.csv": pd.DataFrame(choices),
               "calibration.csv": pd.DataFrame(calibration_rows), "selected_predictions.csv": selected,
               "selected_summary.csv": pd.DataFrame(aggregate), "event_predictions.csv": events,
               "profile_conditions.csv": pd.DataFrame(condition)}
    for name, frame in outputs.items():
        frame.to_csv(OUT / name, index=False, encoding="utf-8-sig")
    save_json(OUT / "decision.json", {"directions": decisions, "no_claim": "No regression improvement, final-test success, physical operating cause, or all-shape transition coverage was evaluated."})
    names = list(outputs) + ["decision.json"]
    result = {"status": "completed", "script_sha256": sha(__file__), "contract_sha256": sha(OUT / "contract.json"),
              "inputs_sha256": c["inputs_sha256"], "new_fits": len(manifests), "eligible_hours": len(f),
              "feature_count": len(FEATURES), "outputs_sha256": {n: sha(OUT / n) for n in names},
              "runtime": sys.version, "packages": {n: __import__(n).__version__ for n in ["numpy", "pandas", "sklearn", "joblib"]}}
    save_json(OUT / "run.json", result)
    print(json.dumps({"status": "completed", "new_fits": len(manifests), "decisions": decisions}), flush=True)


if __name__ == "__main__":
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    {"freeze": freeze, "run": run}[sys.argv[1]]()


코드 등록 완료: `Modeling/scripts/regime_forecast.py`

### `Modeling/scripts/regime_forecast_verify.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Independent S03 chronology, tie-aware AP, thresholds, selection and model checks.

주요 함수: `sha()`, `close()`, `average_precision()`, `metrics()`, `check_stats()`, `independent_threshold()`, `main()`.

In [92]:
%%submission_source Modeling/scripts/regime_forecast_verify.py
"""Independent S03 chronology, tie-aware AP, thresholds, selection and model checks."""
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import hashlib
import json
import math
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/regime_forecast"
PARENT = ROOT / "Modeling/tables/regime_diagnosis"


def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def close(a, b):
    if pd.isna(a) or b is None:
        assert pd.isna(a) and b is None, (a, b)
    else:
        assert math.isclose(float(a), float(b), rel_tol=1e-10, abs_tol=1e-9), (a, b)


def average_precision(rows):
    pairs = sorted(zip(rows.score, rows.event), reverse=True)
    positives = sum(y for _, y in pairs)
    if not positives:
        return None
    tp = total = 0
    numerator = 0.
    for score in sorted(set(s for s, _ in pairs), reverse=True):
        group = [y for s, y in pairs if s == score]
        new_tp = sum(group)
        total += len(group); tp += new_tp
        numerator += new_tp * tp / total
    return numerator / positives


def metrics(g):
    result = {"hours": len(g), "events": int(g.event.sum())}
    if g.alarm.notna().all():
        tp = fp = fn = tn = short_fp = 0
        for r in g.itertuples():
            if r.event:
                tp += int(r.alarm == 1); fn += int(r.alarm == 0)
            else:
                fp += int(r.alarm == 1); tn += int(r.alarm == 0)
                short_fp += int(r.alarm == 1 and r.short_event == 1)
        result.update(tp=tp, fp=fp, fn=fn, tn=tn, recall=tp / (tp + fn) if tp + fn else None,
                      precision=tp / (tp + fp) if tp + fp else None, fp_per100h=100 * fp / len(g),
                      fpr=fp / (fp + tn) if fp + tn else None, short_event_fp=short_fp)
    return result


def check_stats(record, expected):
    for field, value in expected.items():
        close(getattr(record, field), value)


def independent_threshold(g):
    cap = int((len(g) - g.event.sum()) // 100)
    choices = []
    for threshold in [math.inf] + sorted(g.score.unique().tolist(), reverse=True):
        alerted = g.loc[g.score >= threshold]
        tp = int(alerted.event.sum()); fp = len(alerted) - tp
        if fp <= cap:
            choices.append((tp, -fp, threshold))
    best = max(choices)
    return best[2], best[0], -best[1], cap


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    c = json.loads((OUT / "contract.json").read_text(encoding="utf-8"))
    run = json.loads((OUT / "run.json").read_text(encoding="utf-8"))
    assert c["entrypoint_sha256"] == run["script_sha256"] == sha(ROOT / "Modeling/scripts/regime_forecast.py")
    assert run["contract_sha256"] == sha(OUT / "contract.json") and run["status"] == "completed"
    for name, digest in {**c["inputs_sha256"], **run["inputs_sha256"]}.items():
        assert sha(ROOT / name) == digest
    for name, digest in run["outputs_sha256"].items():
        assert sha(OUT / name) == digest
    parent_run = json.loads((PARENT / "run.json").read_text(encoding="utf-8"))
    parent_verify = json.loads((PARENT / "independent_verification.json").read_text(encoding="utf-8"))
    assert parent_verify["status"] == "passed" and parent_verify["run_sha256"] == sha(PARENT / "run.json")
    for name, digest in parent_run["outputs_sha256"].items():
        assert sha(PARENT / name) == digest
    parent_contract = json.loads((PARENT / "contract.json").read_text(encoding="utf-8"))
    assert sha(ROOT / "data/origin/okm_augumented_2021.csv") == parent_contract["source_sha256"]
    full_frame = pd.read_csv(PARENT / "hourly_labels.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at", "latest_input_timestamp"])
    f = full_frame.loc[full_frame.eligible_onset & full_frame.persistence_known].copy()
    features = c["feature_names"]
    assert not set(features) & set(c["forbidden_inputs"])
    f["hour_sin"] = [math.sin(2 * math.pi * int(h) / 24) for h in f.hour]
    f["hour_cos"] = [math.cos(2 * math.pi * int(h) / 24) for h in f.hour]
    for i in range(7):
        f[f"dow_{i}"] = (f.timestamp.dt.weekday == i).astype(int)
    p = pd.read_csv(OUT / "predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at", "latest_input_timestamp"])
    assert not p.duplicated(["direction", "method", "timestamp"]).any()
    assert (p.latest_input_timestamp < p.timestamp).all() and p.timestamp.max() < pd.Timestamp("2021-07-01")
    folds = pd.read_csv(OUT / "folds.csv", encoding="utf-8-sig")
    manifest = pd.read_csv(OUT / "model_manifest.csv", encoding="utf-8-sig")
    model_reloads = 0; refits = 0
    # Every fold/model is reloaded and its probabilities are checked against a separately built frame.
    for fold in folds.itertuples():
        sign = 1 if fold.direction == "up" else -1
        boundary = pd.Timestamp(2021, fold.month, 1)
        tr = f.loc[(f.timestamp < boundary) & (f.label_confirmed_at < boundary)]
        te = f.loc[f.month == fold.month]
        y = (tr.sustained_onset == sign).astype(int).to_numpy()
        assert len(tr) == fold.train_hours and int(y.sum()) == fold.train_events
        assert tr.loc[y == 1, "date"].nunique() == fold.positive_dates
        assert str(tr.label_confirmed_at.max()) == fold.train_latest_confirmed
        assert tr.label_confirmed_at.max() < boundary and len(te) == fold.test_hours
        available = int(y.sum()) >= 8 and fold.positive_dates >= 5 and (y == 0).any()
        assert available == fold.ml_available
        g = p.loc[(p.direction == fold.direction) & (p.month == fold.month)]
        assert g.method.nunique() == (5 if available else 3)
        for method, gg in g.groupby("method"):
            assert gg.timestamp.tolist() == te.timestamp.tolist()
            assert gg.event.tolist() == (te.sustained_onset == sign).astype(int).tolist()
            assert gg.short_event.tolist() == ((te.onset == sign) & (te.sustained_onset != sign)).astype(int).tolist()
            assert gg.label_confirmed_at.tolist() == te.label_confirmed_at.tolist()
            if method == "last_delta":
                np.testing.assert_allclose(gg.score, sign * te.prior_delta, rtol=0, atol=1e-9)
            elif method == "slope6":
                np.testing.assert_allclose(gg.score, sign * te.past_slope6, rtol=0, atol=1e-9)
            elif method == "calendar":
                values = []
                for r in te.itertuples():
                    group = tr.loc[(tr.hour == r.hour) & (tr.weekend == r.weekend)]
                    if len(group) < 20:
                        group = tr.loc[tr.hour == r.hour]
                    if len(group) < 20:
                        group = tr
                    values.append(((group.sustained_onset == sign).sum() + .5) / (len(group) + 1))
                np.testing.assert_allclose(gg.score, values, rtol=0, atol=1e-9)
            else:
                record = manifest.loc[(manifest.direction == fold.direction) & (manifest.month == fold.month) & (manifest.method == method)].iloc[0]
                assert sha(ROOT / record.file) == record.sha256 and record.features == "|".join(features)
                fitted = joblib.load(ROOT / record.file)
                np.testing.assert_allclose(gg.score, fitted.predict_proba(te[features])[:, 1], rtol=1e-8, atol=1e-9)
                model_reloads += 1
                if (fold.direction, fold.month, method) in [("up", 5, "logistic"), ("down", 6, "hgb")]:
                    fresh = (make_pipeline(StandardScaler(), LogisticRegression(C=1., max_iter=3000, random_state=42))
                             if method == "logistic" else HistGradientBoostingClassifier(max_leaf_nodes=7, max_iter=100,
                                 learning_rate=.05, min_samples_leaf=20, l2_regularization=1., early_stopping=False, random_state=42))
                    fresh.fit(tr[features], y)
                    np.testing.assert_allclose(gg.score, fresh.predict_proba(te[features])[:, 1], rtol=1e-8, atol=1e-9)
                    refits += 1
    selection = pd.read_csv(OUT / "selection.csv", encoding="utf-8-sig").fillna({"selected_simple": "", "selected_ml": "", "calibration_latest_confirmed": ""})
    thresholds = pd.read_csv(OUT / "calibration.csv", encoding="utf-8-sig")
    thresholds_checked = 0
    for r in selection.itertuples():
        boundary = pd.Timestamp(2021, r.month, 1)
        prior = p.loc[(p.direction == r.direction) & (p.month < r.month) & (p.label_confirmed_at < boundary)]
        complete_times = prior.groupby("timestamp").method.nunique()
        cal = prior.loc[prior.timestamp.isin(complete_times.index[complete_times == 5])]
        n = len(cal.loc[cal.method == "calendar"])
        positives = int(cal.loc[cal.method == "calendar", "event"].sum())
        assert n == r.calibration_hours and positives == r.calibration_events
        if n:
            assert str(cal.label_confirmed_at.max()) == r.calibration_latest_confirmed
        if r.calibration_available:
            assert positives > 0
            for family, methods in [("simple", ["calendar", "last_delta", "slope6"]), ("ml", ["logistic", "hgb"])]:
                weighted = []
                for method in methods:
                    mc = cal.loc[cal.method == method]
                    terms = [(int(gg.event.sum()), average_precision(gg)) for _, gg in mc.groupby("month")]
                    score = sum(k * value for k, value in terms if k) / sum(k for k, _ in terms)
                    weighted.append(score)
                    threshold, tp, fp, cap = independent_threshold(mc)
                    rec = thresholds.loc[(thresholds.direction == r.direction) & (thresholds.month == r.month) & (thresholds.method == method)].iloc[0]
                    close(rec.threshold, threshold); close(rec.weighted_monthly_ap, score)
                    assert (rec.tp, rec.fp, rec.fp_cap, rec.hours, rec.events) == (tp, fp, cap, n, positives)
                    evaluated = p.loc[(p.direction == r.direction) & (p.month == r.month) & (p.method == method)]
                    assert (evaluated.threshold == threshold).all()
                    assert evaluated.alarm.tolist() == (evaluated.score >= threshold).astype(int).tolist()
                    thresholds_checked += 1
                picked = methods[max(range(len(methods)), key=lambda i: (weighted[i], -i))]
                assert picked == getattr(r, "selected_" + family)
        else:
            g = p.loc[(p.direction == r.direction) & (p.month == r.month)]
            assert g.alarm.isna().all() and g.threshold.isna().all()
    for r in pd.read_csv(OUT / "monthly_metrics.csv", encoding="utf-8-sig").itertuples():
        g = p.loc[(p.direction == r.direction) & (p.method == r.method) & (p.month == r.month)]
        check_stats(r, {**metrics(g), "ap": average_precision(g)})
    selected = p.loc[(p.month >= 4) & p.selected_family.notna()].copy()
    saved_selected = pd.read_csv(OUT / "selected_predictions.csv", encoding="utf-8-sig")
    assert saved_selected.timestamp.tolist() == selected.timestamp.astype(str).tolist()
    summary = pd.read_csv(OUT / "selected_summary.csv", encoding="utf-8-sig")
    for r in summary.itertuples():
        g = selected.loc[(selected.direction == r.direction) & (selected.selected_family == r.family)]
        check_stats(r, metrics(g))
    for r in pd.read_csv(OUT / "profile_conditions.csv", encoding="utf-8-sig").itertuples():
        g = selected.loc[(selected.direction == r.direction) & (selected.selected_family == r.family) & (selected.seen_profile == r.seen_profile)]
        check_stats(r, metrics(g))
    decisions = json.loads((OUT / "decision.json").read_text(encoding="utf-8"))
    for r in decisions["directions"]:
        a = summary.loc[(summary.direction == r["direction"]) & (summary.family == "ml")].iloc[0]
        b = summary.loc[(summary.direction == r["direction"]) & (summary.family == "simple")].iloc[0]
        extra = int(a.tp - b.tp); relative = extra / b.tp if b.tp else None
        close(r["extra_tp"], extra); close(r["relative_extra_tp"], relative)
        close(r["extra_fp_per100h"], a.fp_per100h - b.fp_per100h)
        assert r["development_gate_passed"] == (extra >= 2 and (relative is None or relative >= .20) and a.fp_per100h - b.fp_per100h <= .5)
    # Check full saved event/late-alarm diagnosis separately, and expose case concentration.
    ep = pd.read_csv(OUT / "event_predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    assert len(ep) == len(p.loc[(p.month >= 4) & (p.event == 1)])
    keyed = {(r.direction, r.method, r.timestamp): r for r in p.itertuples()}
    for r in ep.itertuples():
        assert (r.direction, r.method, r.timestamp) in keyed and keyed[(r.direction, r.method, r.timestamp)].event == 1
        for k in [1, 2]:
            other = keyed.get((r.direction, r.method, r.timestamp + pd.Timedelta(hours=k)))
            actual = getattr(r, "alarm_one_hour_late" if k == 1 else "alarm_two_hours_late")
            expected = None if other is None or pd.isna(other.alarm) else other.alarm
            close(actual, expected)
    cases = []
    selected_ml = selected.loc[(selected.selected_family == "ml") & ((selected.event == 1) | (selected.alarm == 1))]
    dynamic = [x for x in features if x.startswith("prior_") or x.startswith("past_")]
    for r in selected_ml.itertuples():
        row = f.loc[f.timestamp == r.timestamp].iloc[0]
        tr = f.loc[(f.timestamp < pd.Timestamp(2021, r.month, 1)) & (f.label_confirmed_at < pd.Timestamp(2021, r.month, 1))]
        identical = np.isclose(tr[dynamic].to_numpy(float), row[dynamic].to_numpy(float), rtol=0, atol=1e-9).all(axis=1)
        future = full_frame.loc[full_frame.timestamp.between(r.timestamp, r.timestamp + pd.Timedelta(hours=2))]
        assert len(future) == 3
        cases.append({"timestamp": r.timestamp, "direction": r.direction, "method": r.method, "event": r.event,
                      "alarm": r.alarm, "score": r.score, "threshold": r.threshold, "seen_daily_profile": r.seen_profile,
                      "identical_dynamic_input_training_rows": int(identical.sum()), "prior_mean": row.prior_mean,
                      "start_mean": row.level, "start_maximum": row.maximum, "observed_next3_mean": float(future.level.mean()),
                      "prior_delta": row.prior_delta, "prior_last_minus_mean": row.prior_last_minus_mean,
                      "past_slope6": row.past_slope6, "prior_state_age": row.prior_state_age, "profile": row.profile})
    cases = pd.DataFrame(cases)
    cases.to_csv(OUT / "verified_cases.csv", index=False, encoding="utf-8-sig")
    support = []
    for direction, gg in cases.loc[cases.event == 1].groupby("direction"):
        support.append({"direction": direction, "events": len(gg), "detected": int(gg.alarm.sum()),
                        "positive_dates": gg.timestamp.dt.date.nunique(), "distinct_daily_profiles": gg.profile.nunique(),
                        "events_with_identical_dynamic_training_input": int((gg.identical_dynamic_input_training_rows > 0).sum()),
                        "new_daily_profile_events": int((gg.seen_daily_profile == 0).sum())})
    pd.DataFrame(support).to_csv(OUT / "verified_support.csv", index=False, encoding="utf-8-sig")
    result = {"status": "passed", "run_sha256": sha(OUT / "run.json"), "verifier_sha256": sha(__file__),
              "prediction_rows_checked": len(p), "folds_checked": len(folds), "thresholds_checked": thresholds_checked,
              "models_reloaded": model_reloads, "independent_refits": refits, "event_rows_checked": len(ep),
              "forbidden_features_absent": True, "mature_labels_only": True,
              "extra_outputs_sha256": {n: sha(OUT / n) for n in ["verified_cases.csv", "verified_support.csv"]}}
    (OUT / "independent_verification.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result, ensure_ascii=False), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/regime_forecast_verify.py`

### `Modeling/scripts/regime_integrate.py`

원고 연결: Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_원고.md.

코드 설명: S04 fixed-HGB controlled integration of genuine chronological S03 probabilities.

freeze -> run. Uses Jan-defined labels and Feb-Jun precomputed OOF scores only.
No July/August scoring, refitting classifiers, tuning, clipping, or source edits.

주요 함수: `sha()`, `write_json()`, `dependencies()`, `definitions()`, `data()`, `freeze()`, `measure()`, `run()`.

In [93]:
%%submission_source Modeling/scripts/regime_integrate.py
"""S04 fixed-HGB controlled integration of genuine chronological S03 probabilities.

freeze -> run. Uses Jan-defined labels and Feb-Jun precomputed OOF scores only.
No July/August scoring, refitting classifiers, tuning, clipping, or source edits.
"""
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
import hashlib
import json
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/regime_integration"
MODELS = ROOT / "Modeling/models/regime_integration"
FORECAST = ROOT / "Modeling/tables/regime_forecast"
DIAG = ROOT / "Modeling/tables/regime_diagnosis"
VARIANTS = ["B0", "B1", "B2_up", "B2_down", "B2_both"]


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def dependencies():
    names = ["Modeling/config/m01_contract.json", "Modeling/config/m03_ab_contract.json",
             "Modeling/tables/m01/hourly_frame.csv", "Modeling/tables/m01/verification.json",
             "Modeling/tables/regime_diagnosis/hourly_labels.csv", "Modeling/tables/regime_diagnosis/contract.json",
             "Modeling/tables/regime_forecast/predictions.csv", "Modeling/tables/regime_forecast/folds.csv",
             "Modeling/tables/regime_forecast/contract.json", "Modeling/tables/regime_forecast/run.json",
             "Modeling/tables/regime_forecast/independent_verification.json"]
    return {n: sha(ROOT / n) for n in names}


def definitions():
    m1 = json.loads((ROOT / "Modeling/config/m01_contract.json").read_text(encoding="utf-8"))
    s3 = json.loads((FORECAST / "contract.json").read_text(encoding="utf-8"))
    base = m1["calendar_features"] + m1["feature_groups"]["A"] + m1["feature_groups"]["B_add"]
    extra = [n for n in s3["feature_names"] if n.startswith(("prior_", "past_")) and n not in ["prior_mean", "prior_production"]]
    # Exact duplicates prior_mean/production are already lag1_mean/production.
    extended = base + extra + ["up_classifier_available", "down_classifier_available"]
    return {"B0": base, "B1": extended, "B2_up": extended + ["p_up"],
            "B2_down": extended + ["p_down"], "B2_both": extended + ["p_up", "p_down"]}


def data():
    m = pd.read_csv(ROOT / "Modeling/tables/m01/hourly_frame.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    h = pd.read_csv(DIAG / "hourly_labels.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at", "latest_input_timestamp"])
    p = pd.read_csv(FORECAST / "predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at"])
    f = h.loc[h.eligible_onset & h.persistence_known & (h.month >= 2)].copy()
    base = definitions()["B0"]
    # Use the same validated M01 common pool; Jan rows are not backfilled with a definition learned at Jan end.
    b = m.loc[m.eligible_common & (m.timestamp < "2021-07-01"), ["timestamp", "target_maximum", "target_mean"] + base]
    dynamic = [n for n in h.columns if n.startswith(("prior_", "past_"))]
    f = f[["timestamp", "label_confirmed_at", "latest_input_timestamp", "month", "profile", "onset", "sustained_onset", "level", "maximum"] + dynamic]
    f = f.drop(columns="month").merge(b, on="timestamp", how="inner", validate="one_to_one")
    assert len(f) == 3598 and np.allclose(f.maximum, f.target_maximum) and np.allclose(f.level, f.target_mean)
    assert np.allclose(f.prior_mean, f.lag1_mean) and np.allclose(f.prior_production, f.lag1_production)
    for direction, method in [("up", "logistic"), ("down", "hgb")]:
        rows = []
        for _, group in p.loc[p.direction == direction].groupby("timestamp", sort=True):
            kind = method if method in set(group.method) else "calendar"
            r = group.loc[group.method == kind].iloc[0]
            rows.append({"timestamp": r.timestamp, f"p_{direction}": r.score,
                         f"{direction}_classifier_available": int(kind == method),
                         f"{direction}_score_method": kind})
        f = f.merge(pd.DataFrame(rows), on="timestamp", validate="one_to_one")
    assert f.timestamp.min() == pd.Timestamp("2021-02-01") and len(f) == 3598
    assert (f.latest_input_timestamp == f.timestamp - pd.Timedelta(hours=1)).all()
    assert f[definitions()["B2_both"]].notna().all().all()
    return f.sort_values("timestamp").reset_index(drop=True)


def freeze():
    OUT.mkdir(parents=True, exist_ok=True)
    assert not (OUT / "contract.json").exists()
    verified = json.loads((FORECAST / "independent_verification.json").read_text(encoding="utf-8"))
    assert verified["status"] == "passed" and verified["run_sha256"] == sha(FORECAST / "run.json")
    frame = data()  # Availability and fixed target counts only; no model or comparative errors.
    c = {"stage": "S04", "created_at": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
         "entrypoint_sha256": sha(__file__), "inputs_sha256": dependencies(), "features": definitions(),
         "reason": "S03 persistent up 12/12 FP0; down 8/15 FP10. Test whether predicted transition probabilities add beyond identical raw causal inputs.",
         "scope": "Exploratory Feb-Jun training, Apr-Jun evaluation; initial definition estimated in Jan. No new independent final test.",
         "hgb_parameters": json.loads((ROOT / "Modeling/config/m03_ab_contract.json").read_text(encoding="utf-8"))["hgb_parameters"],
         "probability_policy": "Fixed S03 Logistic up / HGB down genuine earlier-month predictions; use saved calendar rate when ML unavailable. No in-sample rescoring, no Jan probability backfill. Availability indicators also in B1 control.",
         "training": "Same Feb-to-previous-month rows for all variants; label_confirmed_at < evaluation boundary. Monthly refit, hourly input update. Compare with newly fitted B0, not historical Jan-trained errors.",
         "evaluation": "Same Apr-Jun eligible common rows, sustained label known; last two June hours excluded. April diagnostic for upward integration: no upward ML OOF training yet. Primary up May-Jun; down Apr-Jun.",
         "primary_variant": "B2_up", "secondary_variants": ["B2_down", "B2_both"],
         "development_gate": {"event_start_mae_reduction_vs_both_B0_B1": .05,
                              "directional_error_reduction_vs_both_B0_B1": .05,
                              "overall_mae_worsening_vs_B0_at_most": .01,
                              "daily_peak_mae_worsening_vs_B0_at_most": .01,
                              "minimum_events": 5, "minimum_months_with_start_mae_improvement_vs_B1": 2},
         "directional_errors": "up under=max(actual-pred,0); down over=max(pred-actual,0). Zero baseline cannot show relative reduction; require strict absolute decrease otherwise.",
         "metrics": "MAE/RMSE/bias/under/over, primary event starts, event-equal first-three-hour MAE, monthly directions, full24-hour daily peak tie weights, seen/new daily profiles; no favorable case deletion",
         "event_windows": "t,t+1,t+2 predictions only for completely covered windows. Report exclusion counts and event-equal averages; overlaps across different event baselines possible.",
         "daily_peak": "Only full 24 prediction hours per day; all maximum ties get inverse tie-count weights; June30 excluded from daily comparison.",
         "model_selection": "Fixed ablation variants, primary up candidate predetermined; report all. No best-by-evaluation candidate treated as preselected or validated deployment model.",
         "tuning_trials": 0, "clip_predictions": False,
         "preflight_counts": {"rows": len(frame), "by_month": {str(m): len(g) for m, g in frame.groupby("month")},
                              "features": {v: len(cols) for v, cols in definitions().items()}}}
    write_json(OUT / "contract.json", c)
    print(json.dumps({"status": "frozen", **c["preflight_counts"]}), flush=True)


def measure(g, weights=None):
    err = g.prediction.to_numpy() - g.actual.to_numpy()
    w = np.ones(len(g)) if weights is None else np.asarray(weights)
    return {"hours": len(g), "weight_sum": float(w.sum()), "mae": float(np.average(abs(err), weights=w)),
            "rmse": float(np.sqrt(np.average(err ** 2, weights=w))), "bias": float(np.average(err, weights=w)),
            "under": float(np.average(np.maximum(-err, 0), weights=w)), "over": float(np.average(np.maximum(err, 0), weights=w))}


def run():
    assert not (OUT / "run.json").exists(), "Completed results must not be overwritten."
    c = json.loads((OUT / "contract.json").read_text(encoding="utf-8"))
    assert c["entrypoint_sha256"] == sha(__file__) and c["inputs_sha256"] == dependencies()
    f = data(); MODELS.mkdir(parents=True, exist_ok=True)
    predictions = []; manifest = []; folds = []
    for month in [4, 5, 6]:
        boundary = pd.Timestamp(2021, month, 1)
        tr = f.loc[(f.timestamp < boundary) & (f.label_confirmed_at < boundary)]
        te = f.loc[f.month == month]
        fold = {"month": month, "train_hours": len(tr), "train_start": str(tr.timestamp.min()),
                "train_end": str(tr.timestamp.max()), "last_confirmed": str(tr.label_confirmed_at.max()),
                "eval_hours": len(te), "train_up_ml_probability_hours": int(tr.up_classifier_available.sum()),
                "train_down_ml_probability_hours": int(tr.down_classifier_available.sum())}
        folds.append(fold)
        for variant in VARIANTS:
            cols = c["features"][variant]
            model = HistGradientBoostingRegressor(**c["hgb_parameters"])
            model.fit(tr[cols], tr.target_maximum)
            yhat = model.predict(te[cols])
            path = MODELS / f"{month:02d}_{variant}.joblib"
            assert not path.exists(); joblib.dump(model, path)
            manifest.append({**fold, "variant": variant, "features": "|".join(cols),
                             "file": path.relative_to(ROOT).as_posix(), "sha256": sha(path)})
            g = te[["timestamp", "month", "profile", "onset", "sustained_onset", "prior_mean", "level", "p_up", "p_down", "up_classifier_available", "down_classifier_available"]].copy()
            g["variant"] = variant; g["actual"] = te.target_maximum.to_numpy(); g["prediction"] = yhat
            g["seen_regression_profile"] = te.profile.isin(tr.profile).astype(int).to_numpy()
            predictions.append(g)
        print(json.dumps(fold), flush=True)
    p = pd.concat(predictions, ignore_index=True)
    p["date"] = p.timestamp.dt.date
    metrics = []; event_rows = []; coverage = []; daily_rows = []
    for variant, g in p.groupby("variant"):
        for period, gg in [("Apr-Jun", g), ("May-Jun", g.loc[g.month >= 5])]:
            metrics.append({"variant": variant, "period": period, "condition": "all", **measure(gg)})
            metrics.append({"variant": variant, "period": period, "condition": "non_persistent_start", **measure(gg.loc[gg.sustained_onset == 0])})
            for direction, sign in [("up", 1), ("down", -1)]:
                starts = gg.loc[gg.sustained_onset == sign]
                if len(starts):
                    metrics.append({"variant": variant, "period": period, "condition": f"{direction}_start", **measure(starts)})
            for seen, group in gg.groupby("seen_regression_profile"):
                metrics.append({"variant": variant, "period": period, "condition": f"profile_{int(seen)}", **measure(group)})
        for month, gg in g.groupby("month"):
            metrics.append({"variant": variant, "period": str(month), "condition": "all", **measure(gg)})
            for direction, sign in [("up", 1), ("down", -1)]:
                starts = gg.loc[gg.sustained_onset == sign]
                if len(starts):
                    metrics.append({"variant": variant, "period": str(month), "condition": f"{direction}_start", **measure(starts)})
        indexed = g.set_index("timestamp")
        for r in g.loc[g.sustained_onset != 0].itertuples():
            times = pd.date_range(r.timestamp, periods=3, freq="h")
            complete = all(t in indexed.index for t in times)
            coverage.append({"variant": variant, "timestamp": r.timestamp, "direction": "up" if r.sustained_onset == 1 else "down", "complete": complete})
            if complete:
                window = indexed.loc[times]
                event_rows.append({"variant": variant, "timestamp": r.timestamp, "month": r.month,
                                   "direction": "up" if r.sustained_onset == 1 else "down", "seen_profile": r.seen_regression_profile,
                                   "start_actual": r.actual, "start_prediction": r.prediction,
                                   "p_up": r.p_up, "p_down": r.p_down, **measure(window)})
        for date, gg in g.groupby("date"):
            if len(gg) != 24:
                continue
            peak = gg.loc[gg.actual == gg.actual.max()]
            daily_rows.append({"variant": variant, "date": str(date), "month": int(gg.month.iloc[0]),
                               "peak_actual": float(gg.actual.max()), "ties": len(peak), **measure(peak)})
    events = pd.DataFrame(event_rows); daily = pd.DataFrame(daily_rows)
    for variant, g in events.groupby("variant"):
        for period, gg in [("Apr-Jun", g), ("May-Jun", g.loc[g.month >= 5])]:
            for direction, part in gg.groupby("direction"):
                metrics.append({"variant": variant, "period": period, "condition": direction + "_window_event_equal",
                                "hours": len(part) * 3, "weight_sum": len(part),
                                **{n: float(part[n].mean()) for n in ["mae", "bias", "under", "over"]},
                                "rmse": float(np.sqrt(np.mean(part.rmse ** 2)))})
        for period, gg in [("Apr-Jun", daily.loc[daily.variant == variant]), ("May-Jun", daily.loc[(daily.variant == variant) & (daily.month >= 5)])]:
            # MAE/under/over/bias average by day; RMSE is square root of mean daily MSE.
            metrics.append({"variant": variant, "period": period, "condition": "daily_peak_day_equal", "hours": int(gg.ties.sum()),
                            "weight_sum": len(gg), **{n: float(gg[n].mean()) for n in ["mae", "bias", "under", "over"]},
                            "rmse": float(np.sqrt(np.mean(gg.rmse ** 2)))})
    table = pd.DataFrame(metrics)
    gains = []; decisions = []
    for variant in VARIANTS[2:]:
        for direction, period, directional in [("up", "May-Jun", "under"), ("down", "Apr-Jun", "over")]:
            stats = {}
            for baseline in ["B0", "B1"]:
                a = table.loc[(table.variant == variant) & (table.period == period) & (table.condition == direction + "_start")].iloc[0]
                b = table.loc[(table.variant == baseline) & (table.period == period) & (table.condition == direction + "_start")].iloc[0]
                row = {"variant": variant, "direction": direction, "period": period, "baseline": baseline,
                       "events": int(a.hours), "start_mae_reduction": 1 - a.mae / b.mae,
                       "directional_reduction": 1 - a[directional] / b[directional] if b[directional] else None}
                gains.append(row); stats[baseline] = row
            a_all = table.loc[(table.variant == variant) & (table.period == period) & (table.condition == "all")].iloc[0]
            b_all = table.loc[(table.variant == "B0") & (table.period == period) & (table.condition == "all")].iloc[0]
            a_peak = table.loc[(table.variant == variant) & (table.period == period) & (table.condition == "daily_peak_day_equal")].iloc[0]
            b_peak = table.loc[(table.variant == "B0") & (table.period == period) & (table.condition == "daily_peak_day_equal")].iloc[0]
            monthly_wins = 0
            for m in ([5, 6] if direction == "up" else [4, 5, 6]):
                aa = table.loc[(table.variant == variant) & (table.period == str(m)) & (table.condition == direction + "_start")].iloc[0]
                bb = table.loc[(table.variant == "B1") & (table.period == str(m)) & (table.condition == direction + "_start")].iloc[0]
                monthly_wins += int(aa.mae < bb.mae)
            passed = (all(s["start_mae_reduction"] >= .05 and s["directional_reduction"] is not None and s["directional_reduction"] >= .05 for s in stats.values())
                      and a_all.mae <= b_all.mae * 1.01 and a_peak.mae <= b_peak.mae * 1.01
                      and stats["B0"]["events"] >= 5 and monthly_wins >= 2)
            decisions.append({"variant": variant, "direction": direction, "period": period,
                              "development_gate_passed": bool(passed), "monthly_wins_vs_B1": monthly_wins,
                              "overall_mae_change_vs_B0": a_all.mae / b_all.mae - 1,
                              "daily_peak_mae_change_vs_B0": a_peak.mae / b_peak.mae - 1})
    outputs = {"meta_features.csv": f, "predictions.csv": p, "metrics.csv": table, "folds.csv": pd.DataFrame(folds),
               "model_manifest.csv": pd.DataFrame(manifest), "event_errors.csv": events,
               "event_coverage.csv": pd.DataFrame(coverage), "daily_peak_errors.csv": daily, "paired_gains.csv": pd.DataFrame(gains)}
    for name, frame in outputs.items():
        frame.to_csv(OUT / name, index=False, encoding="utf-8-sig")
    write_json(OUT / "decision.json", {"primary": "B2_up on up May-Jun", "comparisons": decisions,
                                        "scope": "Development ablation using strategy chosen after S03; not independent validation or operating savings."})
    write_json(OUT / "run.json", {"status": "completed", "entrypoint_sha256": sha(__file__), "contract_sha256": sha(OUT / "contract.json"),
                                 "inputs_sha256": c["inputs_sha256"], "new_regression_fits": len(manifest), "new_classifier_fits": 0,
                                 "outputs_sha256": {n: sha(OUT / n) for n in list(outputs) + ["decision.json"]}, "runtime": sys.version})
    print(json.dumps({"status": "completed", "new_fits": len(manifest), "decisions": decisions}), flush=True)


if __name__ == "__main__":
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    {"freeze": freeze, "run": run}[sys.argv[1]]()


코드 등록 완료: `Modeling/scripts/regime_integrate.py`

### `Modeling/scripts/regime_integrate_verify.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Independent S04 joining, OOF chronology, fixed model and scalar error verification.

주요 함수: `sha()`, `close()`, `stats()`, `main()`.

In [94]:
%%submission_source Modeling/scripts/regime_integrate_verify.py
"""Independent S04 joining, OOF chronology, fixed model and scalar error verification."""
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import hashlib
import json
import math
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/regime_integration"


def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def close(a, b):
    assert math.isclose(float(a), float(b), rel_tol=1e-9, abs_tol=1e-8), (a, b)


def stats(g):
    errors = [float(r.prediction) - float(r.actual) for r in g.itertuples()]
    return {"hours": len(errors), "weight_sum": len(errors), "mae": math.fsum(abs(e) for e in errors) / len(errors),
            "rmse": math.sqrt(math.fsum(e * e for e in errors) / len(errors)),
            "bias": math.fsum(errors) / len(errors), "under": math.fsum(max(-e, 0) for e in errors) / len(errors),
            "over": math.fsum(max(e, 0) for e in errors) / len(errors)}


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    c = json.loads((OUT / "contract.json").read_text(encoding="utf-8"))
    run = json.loads((OUT / "run.json").read_text(encoding="utf-8"))
    assert run["status"] == "completed" and run["contract_sha256"] == sha(OUT / "contract.json")
    assert run["entrypoint_sha256"] == c["entrypoint_sha256"] == sha(ROOT / "Modeling/scripts/regime_integrate.py")
    for name, digest in c["inputs_sha256"].items():
        assert sha(ROOT / name) == digest
    for name, digest in run["outputs_sha256"].items():
        assert sha(OUT / name) == digest
    m1c = json.loads((ROOT / "Modeling/config/m01_contract.json").read_text(encoding="utf-8"))
    assert sha(ROOT / m1c["source"]) == m1c["source_sha256"]
    s3verify = json.loads((ROOT / "Modeling/tables/regime_forecast/independent_verification.json").read_text(encoding="utf-8"))
    assert s3verify["status"] == "passed" and s3verify["run_sha256"] == sha(ROOT / "Modeling/tables/regime_forecast/run.json")
    meta = pd.read_csv(OUT / "meta_features.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at", "latest_input_timestamp"])
    h = pd.read_csv(ROOT / "Modeling/tables/regime_diagnosis/hourly_labels.csv", encoding="utf-8-sig", parse_dates=["timestamp", "label_confirmed_at"])
    m = pd.read_csv(ROOT / "Modeling/tables/m01/hourly_frame.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    s = pd.read_csv(ROOT / "Modeling/tables/regime_forecast/predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    folds = pd.read_csv(ROOT / "Modeling/tables/regime_forecast/folds.csv", encoding="utf-8-sig")
    expected_times = sorted(set(h.loc[h.eligible_onset & h.persistence_known & (h.month >= 2), "timestamp"]) & set(m.loc[m.eligible_common & (m.timestamp < "2021-07-01"), "timestamp"]))
    assert meta.timestamp.tolist() == expected_times and len(meta) == 3598
    assert (meta.latest_input_timestamp == meta.timestamp - pd.Timedelta(hours=1)).all()
    for variant, features in c["features"].items():
        assert not set(features) & {"actual", "level", "maximum", "target_maximum", "sustained_onset", "onset", "profile", "delta", "deviation"}
        assert len(features) == len(set(features))
    original = m.set_index("timestamp").loc[meta.timestamp]
    parent = h.set_index("timestamp").loc[meta.timestamp]
    for feature in c["features"]["B0"] + ["target_maximum", "target_mean"]:
        np.testing.assert_allclose(meta[feature], original[feature], rtol=0, atol=1e-9)
    for feature in [n for n in c["features"]["B1"] if n.startswith(("prior_", "past_"))]:
        np.testing.assert_allclose(meta[feature], parent[feature], rtol=0, atol=1e-9)
    assert meta.sustained_onset.tolist() == parent.sustained_onset.tolist()
    for direction, family in [("up", "logistic"), ("down", "hgb")]:
        for month, g in meta.groupby("month"):
            available = bool(folds.loc[(folds.direction == direction) & (folds.month == month), "ml_available"].iloc[0])
            used = family if available else "calendar"
            prior = s.loc[(s.direction == direction) & (s.method == used) & s.timestamp.isin(g.timestamp)].set_index("timestamp").loc[g.timestamp]
            np.testing.assert_allclose(g[f"p_{direction}"], prior.score, rtol=0, atol=1e-9)
            assert (g[f"{direction}_classifier_available"] == int(available)).all()
            assert (g[f"{direction}_score_method"] == used).all()
    p = pd.read_csv(OUT / "predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    manifest = pd.read_csv(OUT / "model_manifest.csv", encoding="utf-8-sig")
    assert len(p) == 5 * 2182 and not p.duplicated(["timestamp", "variant"]).any()
    reloaded = refitted = 0
    for r in manifest.itertuples():
        assert sha(ROOT / r.file) == r.sha256
        boundary = pd.Timestamp(2021, r.month, 1)
        tr = meta.loc[(meta.timestamp < boundary) & (meta.label_confirmed_at < boundary)]
        te = meta.loc[meta.month == r.month]
        g = p.loc[(p.variant == r.variant) & (p.month == r.month)]
        features = c["features"][r.variant]
        assert r.features == "|".join(features) and len(tr) == r.train_hours and len(te) == r.eval_hours
        assert str(tr.timestamp.min()) == r.train_start and str(tr.timestamp.max()) == r.train_end
        assert str(tr.label_confirmed_at.max()) == r.last_confirmed and tr.label_confirmed_at.max() < boundary
        assert int(tr.up_classifier_available.sum()) == r.train_up_ml_probability_hours
        assert int(tr.down_classifier_available.sum()) == r.train_down_ml_probability_hours
        assert g.timestamp.tolist() == te.timestamp.tolist()
        np.testing.assert_allclose(g.actual, te.target_maximum, rtol=0, atol=1e-9)
        assert g.seen_regression_profile.tolist() == te.profile.isin(tr.profile).astype(int).tolist()
        fitted = joblib.load(ROOT / r.file)
        np.testing.assert_allclose(g.prediction, fitted.predict(te[features]), rtol=1e-9, atol=1e-8)
        reloaded += 1
        if (r.month, r.variant) in [(5, "B0"), (6, "B1"), (6, "B2_up")]:
            fresh = HistGradientBoostingRegressor(**c["hgb_parameters"])
            fresh.fit(tr[features], tr.target_maximum)
            np.testing.assert_allclose(g.prediction, fresh.predict(te[features]), rtol=1e-9, atol=1e-8)
            refitted += 1
    event_table = pd.read_csv(OUT / "event_errors.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    coverage = pd.read_csv(OUT / "event_coverage.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    daily_table = pd.read_csv(OUT / "daily_peak_errors.csv", encoding="utf-8-sig")
    assert len(coverage) == 5 * 33
    assert coverage.complete.all() and len(event_table) == len(coverage)
    for r in event_table.itertuples():
        times = pd.date_range(r.timestamp, periods=3, freq="h")
        g = p.loc[(p.variant == r.variant) & p.timestamp.isin(times)]
        assert len(g) == 3
        for name, value in stats(g).items():
            close(getattr(r, name), value)
    for r in daily_table.itertuples():
        g = p.loc[(p.variant == r.variant) & (p.timestamp.dt.strftime("%Y-%m-%d") == r.date)]
        assert len(g) == 24
        peak = g.loc[g.actual == g.actual.max()]
        assert len(peak) == r.ties
        for name, value in stats(peak).items():
            close(getattr(r, name), value)
    table = pd.read_csv(OUT / "metrics.csv", encoding="utf-8-sig", dtype={"period": str})
    for r in table.itertuples():
        gg = p.loc[p.variant == r.variant]
        if r.period == "May-Jun":
            gg = gg.loc[gg.month >= 5]
        elif r.period != "Apr-Jun":
            gg = gg.loc[gg.month == int(r.period)]
        if r.condition == "all":
            expected = stats(gg)
        elif r.condition == "non_persistent_start":
            expected = stats(gg.loc[gg.sustained_onset == 0])
        elif r.condition.startswith("profile_"):
            expected = stats(gg.loc[gg.seen_regression_profile == int(r.condition[-1])])
        elif r.condition.endswith("_start"):
            expected = stats(gg.loc[gg.sustained_onset == (1 if r.condition.startswith("up") else -1)])
        else:
            origin = event_table if "window" in r.condition else daily_table
            rows = origin.loc[origin.variant == r.variant]
            if r.period == "May-Jun":
                rows = rows.loc[rows.month >= 5]
            if "window" in r.condition:
                rows = rows.loc[rows.direction == ("up" if r.condition.startswith("up") else "down")]
            expected = {n: math.fsum(float(v) for v in rows[n]) / len(rows) for n in ["mae", "bias", "under", "over"]}
            expected["rmse"] = math.sqrt(math.fsum(float(v) ** 2 for v in rows.rmse) / len(rows))
            expected["hours"] = 3 * len(rows) if "window" in r.condition else int(rows.ties.sum())
            expected["weight_sum"] = len(rows)
        for name, value in expected.items():
            close(getattr(r, name), value)
    gains = pd.read_csv(OUT / "paired_gains.csv", encoding="utf-8-sig")
    for r in gains.itertuples():
        def get(v):
            return table.loc[(table.variant == v) & (table.period == r.period) & (table.condition == r.direction + "_start")].iloc[0]
        a, b = get(r.variant), get(r.baseline)
        close(r.start_mae_reduction, 1 - a.mae / b.mae)
        name = "under" if r.direction == "up" else "over"
        close(r.directional_reduction, 1 - a[name] / b[name])
    decisions = json.loads((OUT / "decision.json").read_text(encoding="utf-8"))
    for r in decisions["comparisons"]:
        rows = gains.loc[(gains.variant == r["variant"]) & (gains.direction == r["direction"])]
        def get(v, condition):
            return table.loc[(table.variant == v) & (table.period == r["period"]) & (table.condition == condition)].iloc[0]
        overall = get(r["variant"], "all").mae / get("B0", "all").mae - 1
        daily = get(r["variant"], "daily_peak_day_equal").mae / get("B0", "daily_peak_day_equal").mae - 1
        close(r["overall_mae_change_vs_B0"], overall); close(r["daily_peak_mae_change_vs_B0"], daily)
        wins = 0
        for month in ([5, 6] if r["direction"] == "up" else [4, 5, 6]):
            part = table.loc[(table.period == str(month)) & (table.condition == r["direction"] + "_start")].set_index("variant")
            wins += int(part.loc[r["variant"], "mae"] < part.loc["B1", "mae"])
        assert wins == r["monthly_wins_vs_B1"]
        passed = bool((rows.start_mae_reduction >= .05).all() and (rows.directional_reduction >= .05).all()
                      and overall <= .01 and daily <= .01 and rows.events.min() >= 5 and wins >= 2)
        assert passed == r["development_gate_passed"]
    result = {"status": "passed", "run_sha256": sha(OUT / "run.json"), "verifier_sha256": sha(__file__),
              "meta_rows_checked": len(meta), "predictions_checked": len(p), "models_reloaded": reloaded,
              "independent_refits": refitted, "event_windows_checked": len(event_table), "daily_rows_checked": len(daily_table),
              "metrics_checked": len(table), "genuine_prior_month_probabilities": True}
    (OUT / "independent_verification.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result, ensure_ascii=False), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/regime_integrate_verify.py`

### `Modeling/scripts/regime_manuscript_documents.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Preserve the detailed manuscript and append verified S01-S06 section 4.12.

No training, compression, root README changes, or historical-result overwrite.

주요 함수: `sha()`, `rows()`, `revise_history()`, `section()`, `verify()`, `write()`, `refresh()`.

In [95]:
%%submission_source Modeling/scripts/regime_manuscript_documents.py
"""Preserve the detailed manuscript and append verified S01-S06 section 4.12.

No training, compression, root README changes, or historical-result overwrite.
"""
import argparse
import csv
import hashlib
import json
import re
from datetime import datetime, timezone, timedelta
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
MANUSCRIPT = ROOT / "Modeling/04_Modeling_원고.md"
README = ROOT / "Modeling/README.md"
OUT = ROOT / "Modeling/tables/regime_manuscript"
RESULT = ROOT / "Modeling/tables/regime_age_ablation"
HEADING = "## 4.12 지속 상승 예측과 선택적 HGB 보완"
INTRO = "이번 원고는 M01의 자료·평가 설계에서 M02 모델 비교, M03 입력 비교, M04 오류분석, M05 고정 후반기 평가와 후속 개발까지 이어진다. 4.9에는 기존 M05의 고정 후반기 결과를 보존하고, 4.11에는 과소예측 검토 우선순위의 보조 활용을 기록했다. 현재 차별화 성과는 4.12의 지속 상승 예측과 선택적 HGB 보완에 정리한다. 이 절은 개발 성과와 이미 관측한 7~8월에서의 탐색적 개선을 연결하며, 각 과거 절의 당시 판단과 구분한다. 수치는 원자료의 전력 척도로 제시하고 실제 전력·비용 절감량으로 환산하지 않는다."


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def rows(name):
    with (RESULT / name).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def revise_history(text):
    replacements = {
        "이 원고에 사용한 Python 파일의 전체 소스는": "이 원고에 사용한 Python 파일의 전체 소스는 아래 링크의 실제 `.py` 파일에 보존한다. 최종 제출에는 공통 모듈·원본 조회·검증 절차까지 같은 폴더 구조로 포함한다. 아래 표는 M01~M05와 초기 재개발의 코드이며, 후속 분석의 추가 코드는 각 절과 4.12의 코드표에 연결했다. 현재 전체 보존 범위와 의존 관계는 `submission_manifest.json`을 기준으로 확인한다.",
        "M02 시점에는 전날·전주 입력의 필요성과": "M02 시점에는 전날·전주 입력의 필요성과 날씨 비교의 순서를 남은 오류에 따라 갱신하기로 했다. 이어진 4.3에서는 입력 비교를, 4.9에서는 기존 후보의 고정 후반기 평가를 수행했다. 현재의 지속 상승 보완 성과는 4.12에서 별도로 설명한다.",
        "현재 개발 후보는 **B:": "M03에서 선택한 개발 후보는 **B: 기본 입력에 마지막 15분 값만 추가한 HGB**였다. 같은 설정에서 전체·월별·최대 시간의 개선이 확인됐고 C_B보다 입력 수가 적으며 전체와 최대 시간의 오차도 작았다. 계절 시차는 6월에 유리했지만 4월과 학습에 없는 배열에서 B보다 나빠 기본 구성에 추가하지 않았다. 이는 당시 입력·설정에서의 선택이며, 4.9의 후반기 결과와 4.12의 새 비교 조건까지 포함한 최종 우위를 뜻하지 않는다.",
        "필살기의 설명도 결과에 맞춰 좁힌다.": "M03에서 확인한 성과는 **직전 마지막 값을 추가했을 때 다음 최대전력의 전체 오차와 실제 일별 최대 시간의 과소예측이 줄어든 개발 결과**다. 이후 4.9에서는 이 효과가 후반기 최대 시간까지 유지되지 않았음을 확인했다. 따라서 마지막 값 하나의 효과를 현재 보고서의 주된 차별성으로 제시하지 않고, 기본 입력을 선택한 근거와 남은 오류를 설명하는 결과로 사용한다.",
        "따라서 필살기 후보의 핵심 방법과 개발 성과는 M03에 있다.": "M03은 전력 크기의 기본 예측을 구성하는 단계다. M04에서는 상승 전환과 낮은 상태에서 남은 오류를 분석하고 보완 규칙을 비교했으며, M05에서는 당시 후보를 고정해 후반기를 확인했다. 이 과정에서 전력 크기의 오차 감소와 상승 시작을 미리 찾는 능력을 구분할 필요가 드러났다. 그 질문을 이어받은 지속 상승 정의·분류·선택적 HGB 보완은 4.12에 정리한다.",
        "직전 상태와 최대 시간의 겹침 질문은 위 최소 진단으로 답했다.": "직전 상태와 최대 시간의 겹침 질문은 위 최소 진단으로 확인했다. 이어진 M04의 질문은 **직전 낮은 구간에서만 직전 기준을 쓰는 규칙이 유지 오차를 줄이면서 상승 과소예측을 얼마나 늘리는가**였다. 4.6에서 이 규칙을 실제 비교하고 상충 때문에 기각했다. 이후에도 가동 원인을 추정으로 채우지 않고 관측 가능한 전력 전환을 정의하는 방향으로 후속 분석을 진행했다.",
        "제출 보고서에서는 제2장에 동일 조건의 입력 비교와 B 선택 이유를": "제출 보고서 제2장에는 동일 조건의 입력 비교와 당시 B 선택 이유를, 제3장에는 상승·낮은 유지에서 남은 오류와 최대 시간 개선의 조건을 연결한다. 제5장의 현재 차별성은 4.12의 지속 상승 경고와 선택적 보완을 중심으로 설명한다. M03의 개발 개선률을 후속 방법의 개선률과 합산하지 않는다.",
        "제3장은 개선된 조건과 남은 실패·규칙 기각 근거를 연결한다.": "제3장에는 개선 조건과 규칙 기각의 근거를 연결한다. 낮은 상태에서 직전 값을 사용하는 규칙은 상승 과소예측을 악화해 채택하지 않았다. 4.9의 고정 후반기 평가에서도 마지막 값의 최대 시간 개선은 안정적으로 유지되지 않았다. 이 결과가 4.12에서 현재 상태만으로 일괄 보정하지 않고 다음 시간의 지속 상승을 별도로 예측해 보완 여부를 정한 배경이다.",
        "제2장에는 동일 조건의 추가 비교와 B 유지 이유, 제3장에는": "제2장에는 동일 조건의 추가 비교와 당시 B 유지 이유, 제3장에는 상승 개선·오경보·월별 실패를 연결한다. 이번 17.5% 상승 오차 감소는 당시 허용 요건을 통과하지 못한 보완 실험의 부분 성과로 보존한다. 이후 4.9에서 후반기를 확인했고, 현재 보고서의 차별화 결과는 지속 상승을 새로 정의한 4.12에 제시한다. 두 실험의 상승 정의와 평가 조건이 다르므로 개선률을 같은 성과로 합산하지 않는다.",
        "최종 보고서 제2장은 기본/확장 모델 선택과 후반기 단순 기준 대비": "최종 보고서 제2장에는 기본·확장 모델 선택과 이 절의 단순 기준 대비 HGB 성과를 제시하고, 이후 4.12의 동일 조건 지속 상승 보완 비교를 연결한다. 제3장에는 선택 후보의 상승·낮은 구간·최대 시간 오류와 후속 오경보 억제를 설명한다. 제5장의 차별성은 4.12에서 확인한 탐색적 선택 보완 효과를 중심으로 제시한다. 이 M05의 미채택 후보를 그 성과로 바꾸어 설명하지 않는다.",
        "이번 M05는 선언한 고정 확인 평가로 종료한다.": "이 M05는 선언한 고정 확인 평가로 종료했다. 당시 후보의 고정 결과를 보존하고, 이후 별도 개발에서는 이미 확인한 7~8월을 탐색에 이용한 기간으로 명시했다. 4.10의 재개발과 4.11의 검토 우선순위를 거쳐, 4.12에서는 지속 상승 시작을 예측하고 보완 HGB 적용을 제한하는 방법을 평가했다. 후속 성과가 나왔더라도 이 절의 기존 결과와 학습 조건은 변경하지 않는다.",
    }
    for prefix, replacement in replacements.items():
        pattern = r"(?m)^" + re.escape(prefix) + r"[^\n]*"
        text, count = re.subn(pattern, lambda _: replacement, text, count=1)
        assert count == 1, prefix
    return text


def section():
    table = rows("followup_metrics.csv")
    def value(variant, condition, metric="mae"):
        found = [r for r in table if r["variant"] == variant and r["period"] == "Jul-Aug" and r["condition"] == condition]
        assert len(found) == 1
        return float(found[0][metric])
    performance = []
    for variant, label in [("B0", "기본 HGB B0"), ("G1_noage", "지속 상승 경고 때 보완 HGB 적용"),
                           ("hour08", "모든 08시에 보완 HGB 적용")]:
        performance.append(f"| {label} | {value(variant, 'all'):.3f} | {value(variant, 'up_start'):.3f} | {value(variant, 'up_start', 'under'):.3f} | {value(variant, 'daily_peak_day_equal'):.3f} |")
    gains = [(1 - value("G1_noage", cond, metric) / value("B0", cond, metric)) * 100
             for cond, metric in [("up_start", "mae"), ("up_start", "under"), ("daily_peak_day_equal", "mae"), ("all", "mae")]]
    return f'''{HEADING}

### 중요한 상승 구간의 오차를 줄이기 위한 예측 설계

기본 HGB는 전체 전력 예측에서 단순 시차 기준보다 낮은 오차를 보였지만, 큰 변화가 시작되는 시간의 과소예측은 남았다. 따라서 전력 크기 예측과 함께 **평소 수준보다 높은 상태가 여러 시간 이어지는 시작을 한 시간 전에 예측하고, 그 경고가 나온 시간에만 보완 HGB를 적용하는 방법**을 검토했다. 이는 4.11의 예측오류 검토 우선순위와는 별도로 실제 전력 수준의 상승 시작을 정답으로 학습하는 방법이다.

관측 기간의 적용 결과, 수정한 경고는 공통 평가의 지속 상승 13건 중 7건을 선별했고 오경보는 0건이었다. 이 시간에 보완 HGB를 사용하면 기본 HGB 대비 상승 시작 MAE는 **{gains[0]:.1f}%**, 상승 과소예측은 **{gains[1]:.1f}%**, 실제 일별 최대 시간 MAE는 **{gains[2]:.1f}%** 감소했다. 전체 MAE도 **{gains[3]:.2f}%** 줄었다. 아래에서는 이 성과가 나온 사건 정의·평가 조건과 남은 탐지 범위를 설명한다.

이 절은 기존 M05를 새 수치로 바꾼 결과가 아니다. 4.9 M05 이후 전력 수준의 지속 전환을 별도 대상으로 정의한 S01~S06의 후속 연구다. 회귀 학습 구간도 달라졌으므로 4.9의 B 점수를 이 절의 B0 점수와 직접 비교하지 않는다. 특히 7~8월은 이미 확인한 자료이며 경과시간 입력 수정의 가설도 그 오류에서 나왔으므로, **후속 수치는 탐색적 재평가이고 새 독립 시험이 아니다.**

### 한 시간의 높은 값과 지속 상승의 시작을 구분했다

전력 수준은 한 시간 안의 네 15분 전력값을 산술평균해 정의했다. 네 값 중 최대값은 기존 HGB의 예측 목표로 유지했다. 따라서 지속 상승 분류는 평균 수준이 바뀌는 시점을 찾고, 회귀 모델은 그 시간의 최대 전력 크기를 예측한다. 한 구간의 일시적인 높은 값과 여러 시간 유지되는 높은 수준을 같은 사건으로 취급하지 않았다.

| 항목 | 고정한 정의 |
|---|---|
| 예측 시점 | t-1 시간의 관측을 마친 직후 다음 t 시간 예측 |
| 상승 크기 기준 | 1월의 양의 시간 평균 증가량 90분위수인 59.5 |
| 상승 시작 조건 | 직전 시간 대비 증가량과 과거 6시간 중앙값 대비 상승폭이 모두 59.5 이상이고, 중앙값 대비 상승폭이 과거 6시간 IQR의 1.5배 이상 |
| 지속 조건 | t·t+1·t+2의 평균이 시작 직전의 과거 중앙값보다 기준의 절반 이상 높은 상태 유지 |
| 중복 시작 방지 | 시작 당시 기준값을 고정해 같은 방향의 진행 중인 상태에서 시작을 반복 집계하지 않음 |
| 사용 가능한 입력 | 예측 전에 확인된 과거 전력·생산량·상태와 달력 정보 |

지속 여부는 미래 두 시간까지 관측된 뒤 정답으로 확정한다. 이를 과거 학습의 정답으로 사용하되, t 시점의 평균·최대값이나 미래 지속시간은 예측 입력에 넣지 않았다. 월별 학습에서도 다음 월 시작 전에 t+2 정답까지 확인된 기록만 사용했다. 59.5는 자료에서 정한 분석 기준이며 현장의 물리적 위험 한계값은 아니다.

1~6월 진단에서는 상승 시작 34건 중 28건이 3시간 지속 조건을 충족했다. 하락은 시작 112건 중 43건이 해당 방향으로 지속됐다. 이 진단을 바탕으로 상승·하락을 별도로 예측했으며, 아래 결합 성과는 개발 근거가 남은 **지속 상승**에 해당한다. 현재의 지속 정의는 시작 기준선으로부터 높은 수준에 머무는 조건이며, 일정한 폭의 박스권이나 일정한 전력값을 보장하지 않는다. [사건 진단](tables/regime_diagnosis/event_summary.csv), [개별 사건](tables/regime_diagnosis/events.csv), [정답·시점 조건](tables/regime_diagnosis/contract.json)을 보존한다.

### 상승 경고에 따른 두 HGB의 선택

기본 HGB B0는 기존 B 구성의 18개 입력을 사용한다. 보완 HGB B1은 여기에 과거 6시간의 중앙값·변동 폭·변화 방향 등 상태 정보를 더한 31개 입력을 사용한다. 두 모델은 같은 행에서 학습하고 HGB 설정도 같게 유지했다. 잎 수 15, 반복 150, 학습률 0.05, 최소 잎 표본 20, L2 1, 조기 종료 미사용, 난수 시드 42다.

초기 S04에서는 전환 확률을 모든 시간의 HGB 입력에 추가했으나 전체·최대 시간 오차가 악화했다. 이후 보완 HGB가 필요한 시간을 분류 경고로 제한하는 방식을 검토했다. 적용 규칙은 다음과 같다.

~~~text
지속 상승 경고가 있으면: 보완 HGB B1의 다음 시간 최대 전력 예측 사용
지속 상승 경고가 없으면: 기본 HGB B0의 다음 시간 최대 전력 예측 사용
~~~

상승 분류기는 과거 전력·생산량·상태와 달력을 사용하는 Logistic 모델이다. 표준화는 각 학습 자료에서만 계산했고 C=1, 최대 반복 3,000, 클래스 가중치 미사용, 난수 시드 42를 유지했다. 경고 경계는 이전 월의 모델이 낸 예측과 당시 확인된 정답으로 산정했다. 과거 음성 오경보 비율 1% 이내에서 적중 수가 가장 많은 경계를 고르고, 동률이면 오경보가 적고 경계가 높은 경우를 우선했다. 같은 학습 자료에 재대입한 점수로 경계를 정하지 않았다.

### 개발 평가와 고정 후속 평가의 연결

| 구분 | 학습·경계 산정 | 평가와 해석 |
|---|---|---|
| 4~6월 개발 | 각 월 이전의 확인된 정답으로 누적 학습. 경고 경계는 이전 월의 예측에서 산정 | 4월은 순위 평가, 5~6월은 자동 경고와 회귀 결합 평가 |
| 7~8월 후속 | 분류기는 1~6월 정답 확정 4,336시간, 회귀는 2~6월 공통 3,598시간으로 한 번씩 학습. 경계는 4~6월 OOF에서 산정 | 같은 모델·경계를 두 달 동안 유지. 매시간 입력은 갱신하고 7월 정답으로 8월을 다시 학습하지 않음 |

1월 자료 전체로 사건 기준을 정했으므로, 회귀 결합의 이전 월 분류 예측은 2월부터 사용했다. 이 조건에 맞춰 B0·B1도 동일한 2월 이후 행으로 학습했다. 초기 분류 표본이 부족한 시기의 대체와 분류기 가용 표시도 같은 조건으로 관리했다. 기존 M05처럼 회귀를 1월부터 학습한 점수를 이번 비교의 기준으로 재사용하지 않았다.

5~6월 개발 평가에서는 상승 12건을 모두 탐지하고 오경보 0건을 기록했다. 경고 때 B1을 적용한 결과 상승 시작 MAE는 10.046→8.661로 13.8%, 상승 과소예측은 8.002→7.137로 10.8% 줄었다. 경고 밖의 1,450시간은 기본 HGB의 예측을 유지했다. 전체 MAE 감소는 약 0.2%였다. [개발 탐지](tables/regime_age_ablation/development_detection.csv), [개발 회귀](tables/regime_age_ablation/development_metrics.csv)에 원래 분류기와 수정 분류기의 결과를 함께 남겼다.

### 후속 오류 진단에서 경과시간 입력을 수정했다

초기 S05 후속 결과에서는 상승 7건을 찾았지만 8월에 오경보 38건이 발생했다. 경고 때 보완 HGB를 사용하는 조합의 전체 MAE는 기본 HGB 6.335에서 6.680으로 증가했다. 개별 입력의 기여를 확인하니, 오경보 38건 모두 과거 상태의 경과시간이 180~227시간으로 학습 최대값 64시간을 넘었다. 이 변수는 38건 모두에서 상승 확률을 높이는 가장 큰 양의 선형 기여를 보였다.

이 근거로 S06에서는 상승 분류기에서 경과시간 입력 하나만 제거해 24개에서 23개 입력으로 줄였다. HGB B0·B1과 그 예측값은 그대로 사용했으므로, 경고 대상을 바꾼 효과를 분리할 수 있었다. B1의 경과시간 입력까지 제거한 실험은 아니다. 개발 평가에서 12건 적중·오경보 0건과 회귀 개선이 유지된 뒤에만 후속 기간을 재평가했다.

변경된 점수의 경고 경계 0.4554612321은 같은 과거 전용 알고리즘으로 4~6월 OOF 2,182시간에서 정했다. 이 자료의 상승 18건을 찾고 오경보는 0건이었다. 7~8월의 정답에 맞춰 경계를 다시 조정하거나 경과시간이 긴 평가 기록을 삭제하지 않았다. [오경보 조건](tables/regime_followup/verified_alarm_conditions.csv), [단일 입력 제거 조건](tables/regime_age_ablation/contract.json), [과거 경계 산정](tables/regime_age_ablation/calibration.csv)을 함께 보존한다.

### 중요한 시간의 오차 감소와 불필요한 보완 적용의 억제

7~8월 회귀 비교는 동일한 1,344시간을 사용했다. 그중 지속 정답까지 확인할 수 있는 1,340시간에서 분류 성능을 계산했다. 미래 정답이 없는 네 시간도 과거 입력과 현재 최대값이 있으므로 회귀 평가에는 유지했다. 일별 최대 시간 평가는 하루 24시간 공통 예측이 모두 있는 56일을 사용하고, 최대값 동률에는 하루 가중치 합이 1이 되도록 나눴다.

| 방법 | 전체 MAE | 상승 시작 MAE | 상승 평균 과소예측 | 일별 최대 시간 MAE |
|---|---:|---:|---:|---:|
{chr(10).join(performance)}

상승 평균 과소예측은 상승 시작에 속한 모든 평가 시간에서 `max(실제값-예측값, 0)`을 평균한 값이다. 과소예측한 사례만 따로 골라 평균한 수치가 아니다. 실제 상승이나 일별 최대 시간은 평가 조건을 구분하는 데 사용했으며, 예측 대상을 선택하는 입력은 관측 전의 분류 경고였다.

기본 HGB 대비 상승 시작 MAE는 20.037→18.205, 평균 과소예측은 19.419→16.342로 감소했다. 일별 최대 시간 MAE는 13.179→12.574로 감소했다. 전체 MAE는 6.335→6.317로 변화해 전체 평균에 대한 추가 이득은 작았지만, 중요한 구간의 개선을 얻으면서 전체 오차 악화를 피했다. 월별 전체 MAE도 7월 6.528→6.498, 8월 6.180→6.171로 감소했다.

| 지속 상승 탐지 | 경과시간 입력 포함 | 경과시간 입력 제거 |
|---|---:|---:|
| 공통 평가의 실제 상승 | 13 | 13 |
| 적중 / 미탐지 | 7 / 6 | 7 / 6 |
| 오경보 | 38 | 0 |
| 재현율 | 53.8% | 53.8% |
| 정밀도 | 15.6% | 100% (7/7건) |

경과시간 제거는 탐지한 상승 수를 늘리기보다 불필요한 경고를 억제했다. 보완 HGB 적용은 45회에서 7회로 줄었고, 나머지 1,337시간은 기본 HGB 예측과 같다. 오경보 38건을 없애면서 기존의 7개 적중을 유지한 것이 전체 MAE 악화를 해소한 직접적인 변화다. 이 표의 정밀도 100%는 해당 기간 경고 7건의 결과이며, 이후 기간의 무오경보를 보장하지 않는다. [후속 탐지](tables/regime_age_ablation/followup_detection.csv), [동일 시간 회귀 지표](tables/regime_age_ablation/followup_metrics.csv), [보완 적용 횟수](tables/regime_age_ablation/followup_burden.csv)를 근거로 한다.

### 탐지 범위와 단순 주간 규칙에 대한 해석

탐지한 7건은 모두 월요일 08시였다. 결과를 확인한 뒤 월요일 08시에 항상 보완 HGB를 쓰는 규칙도 대조했다. 이 사후 규칙은 상승 8건을 찾는 대신 오경보 1건이 발생했고, 수정 모델은 7건을 찾고 오경보 0건을 기록했다. 단순 규칙의 전체 MAE는 6.320, 상승 시작 MAE는 18.434로 모델의 6.317·18.205보다 조금 컸으며 일별 최대 시간 MAE 12.574는 같았다. 따라서 관측된 개선을 복잡한 분류기만의 큰 우위라고 설명하지 않는다. [주간 규칙의 사후 조건](tables/regime_age_ablation/calendar_control_contract.json), [탐지](tables/regime_age_ablation/calendar_control_detection.csv), [회귀](tables/regime_age_ablation/calendar_control_metrics.csv)를 남겼다.

공통 평가에서는 13건 중 6건을 놓쳤다. 과거 6시간 이력만 요구하는 전체 분류 범위에서는 상승 14건 중 7건을 찾아 재현율 50%였고, 상승 한 건은 회귀 공통 시차 조건을 충족하지 않아 공통 비교에서 제외됐다. 따라서 현재 모델은 모든 상승을 포괄하는 탐지기로 설명하기보다, 관측된 일부 상승에서 오경보를 억제하며 예측값을 보완한 후보로 해석한다.

사전에 정한 분류 기준은 재현율 80%·정밀도 80%·음성 오경보 비율 1% 이하였다. 수정 모델은 재현율 기준에 미달했다. 반면 상승 MAE·과소예측 각각 5% 이상 개선, 전체·일별 최대 MAE 악화 1% 이내라는 회귀 결합 기준은 통과했다. 두 판단을 유지하면서 보고서에서는 확인된 오경보 억제와 오차 감소를 주된 성과로 제시한다. 이는 다른 팀의 결과와 비교한 우위나 실제 운영에서 검증한 절감 효과는 아니다.

### 현장 활용과 보고서의 연결

활용 흐름은 직전 시간의 관측 완료 → 다음 시간의 지속 상승 경고와 최대 전력 예측 생성 → 경고가 있으면 보완 HGB 결과 확인 → 담당자가 예정된 가동과 동시 사용 조건을 점검하는 순서다. 경고가 없을 때도 기본 HGB의 예측은 계속 제공한다. 경고가 없는 시간을 안전하다고 판정하는 체계로 사용하면 현재 미탐지 범위를 설명할 수 없으므로, 담당자의 확인을 보조하는 용도로 제안한다. 실제 가동 조정이나 전력·비용 절감량은 측정하지 않았다.

최종 결과보고서에서는 제1장에 지속 전환의 정의와 데이터 진단 근거를 짧게 연결하고, 제2장에 동일 조건의 HGB 결합과 성능표를 제시한다. 제3장에는 경과시간 외삽에 따른 오경보·입력 제거 효과·미탐지와 주간 규칙 대조를 배치한다. 제4장은 예측값을 확인할 시점과 조치 검토 흐름, 제5장은 **데이터에서 찾은 실패조건을 반영해 필요한 시간에만 예측을 보완한 기여**를 요약한다. 제6장에는 아래 코드·조건·저장 결과의 재현 경로를 제시한다. S01~S06 번호는 내부 작업 이력이며 최종 보고서의 장 번호로 사용할 필요가 없다.

### 사용 코드와 검증·제출 보존

| 단계 | Python 코드와 역할 |
|---|---|
| S01~S02 | [regime_diagnose.py](scripts/regime_diagnose.py): 사건 기준·과거 상태·선행 조건 진단. [regime_diagnose_verify.py](scripts/regime_diagnose_verify.py): 원자료에서 시점·사건·조건 재구성 |
| S03 | [regime_forecast.py](scripts/regime_forecast.py): 방향별 시간순 분류·과거 경계·입력 구성. [regime_forecast_verify.py](scripts/regime_forecast_verify.py): AP·경계·시간순 조건·저장 모델 검증 |
| S04 | [regime_integrate.py](scripts/regime_integrate.py), [regime_integrate_verify.py](scripts/regime_integrate_verify.py): 동일 행 HGB 입력 비교와 독립 검증. [regime_route.py](scripts/regime_route.py), [regime_route_verify.py](scripts/regime_route_verify.py): 경고 시간에만 적용하는 결합·대조 |
| S05 | [regime_followup.py](scripts/regime_followup.py), [regime_followup_verify.py](scripts/regime_followup_verify.py): 6월 말 고정 모델의 후속 평가·원자료와 예측의 대조 |
| S06 | [regime_age_ablation.py](scripts/regime_age_ablation.py): 경과시간 입력 제거·개발 통과 뒤 조건부 후속 평가. [regime_age_ablation_verify.py](scripts/regime_age_ablation_verify.py): 독립 재학습·경계·오차 검증 |
| 사후 주간 대조 | [regime_age_calendar_control.py](scripts/regime_age_calendar_control.py): 월요일 08시 규칙을 새 학습 없이 대조 |
| 입력·문서·보존 | [m01_prepare.py](scripts/m01_prepare.py): 공통 시차와 HGB 입력. [regime_manuscript_documents.py](scripts/regime_manuscript_documents.py): 원고 보존·절 추가·수치와 문서 검증. [modeling_submission_audit.py](scripts/modeling_submission_audit.py): 의존 코드·모델·표·문서 보존 검사 |

이 코드들의 전체 `.py` 소스와 공통 의존 파일을 같은 폴더 구조로 제출 보존한다. 4.5의 기존 코드표에 더해 위 코드들이 이번 4.12의 실행 근거다. [S01~S02 결과](tables/regime_diagnosis), [S03 결과](tables/regime_forecast), [S04 입력 비교](tables/regime_integration), [S04 선택 결합](tables/regime_routing), [S05 결과](tables/regime_followup), [S06 결과](tables/regime_age_ablation)와 각 모델 목록도 함께 유지한다.

S06의 모델 4개를 저장 후 다시 불러오고 별도 코드로 모두 재학습했다. 학습 구간 안의 표준화·정답 확정 시점·과거 OOF 경계, 분류 지표 24개와 회귀 지표 150개를 검증했다. 원자료의 정상 시각 5,784시간에서 최대값을 대조하고, 기존 HGB 예측의 재사용·경고에 따른 선택·일별 최대와 시작 후 3시간 오차를 확인했다. 모든 검증이 통과했으며, S06의 새 회귀 학습은 0회다. [모델 목록](tables/regime_age_ablation/model_manifest.csv), [실행](tables/regime_age_ablation/run.json), [독립 검증](tables/regime_age_ablation/independent_verification.json), [판단](tables/regime_age_ablation/decision.json)을 보존한다.

아래는 최초 분석 실행 순서다. 정규 CPython 3.13에서 루트 AGENTS.md의 승격 Start-Process 경로를 사용한다. 완료된 분석은 기존 실행 파일의 덮어쓰기를 거부하므로 현재 제출 자료의 보존 검사는 마지막 명령으로 수행한다. S06 분석 당시의 원고 상태는 [추가 전 원고](tables/regime_manuscript/manuscript_before.md)에 보존했다. 분석 단계의 검증 기록과 이후 원고 편집의 검증 기록을 구분한다.

~~~text
Modeling/scripts/regime_diagnose.py probe
Modeling/scripts/regime_diagnose.py freeze
Modeling/scripts/regime_diagnose.py run
Modeling/scripts/regime_diagnose_verify.py
Modeling/scripts/regime_forecast.py freeze
Modeling/scripts/regime_forecast.py run
Modeling/scripts/regime_forecast_verify.py
Modeling/scripts/regime_integrate.py freeze
Modeling/scripts/regime_integrate.py run
Modeling/scripts/regime_integrate_verify.py
Modeling/scripts/regime_route.py freeze
Modeling/scripts/regime_route.py run
Modeling/scripts/regime_route_verify.py
Modeling/scripts/regime_followup.py freeze
Modeling/scripts/regime_followup.py run
Modeling/scripts/regime_followup_verify.py
Modeling/scripts/regime_age_ablation.py freeze
Modeling/scripts/regime_age_ablation.py run
Modeling/scripts/regime_age_ablation_verify.py
Modeling/scripts/regime_age_calendar_control.py
Modeling/scripts/regime_manuscript_documents.py --write
Modeling/scripts/regime_manuscript_documents.py --verify
Modeling/scripts/modeling_submission_audit.py --verify
~~~

이번 원고 편집에서는 새로운 학습·예측이나 압축본을 만들지 않았다. M03 이후의 잠정 결론과 후속 연결 문장을 현재 결과에 맞게 수정했고, 기존 M05를 포함한 성능표·수치·실험 조건은 유지했다. 편집 전 원고 전체를 별도로 보존했다. 재현 파일의 무결성 검증과 깨끗한 새 환경에서의 전체 재실행은 구분하며, 최종 제출 완료를 주장하지 않는다.
'''


def verify():
    text = MANUSCRIPT.read_text(encoding="utf-8")
    # Subsequent reporting additions have their own verifier; isolate this section.
    text = re.sub(r'<!-- MODELING_CLOSE_BEGIN -->.*?<!-- MODELING_CLOSE_END -->\s*', '', text, flags=re.S)
    text = text.split('## 4.13 ')[0]
    before = (OUT / "manuscript_before.md").read_text(encoding="utf-8")
    assert text.count(HEADING) == 1 and text.split(HEADING)[1].strip() == section().split(HEADING)[1].strip()
    start = "## 4.1 M01:"
    assert text.split(start, 1)[1].split(HEADING)[0].strip() == revise_history(before).split(start, 1)[1].strip()
    old_tables = [line for line in before.splitlines() if line.startswith("|")]
    historical = text.split(HEADING)[0]
    assert old_tables == [line for line in historical.splitlines() if line.startswith("|")]
    c = json.loads((RESULT / "contract.json").read_text(encoding="utf-8"))
    assert sha(OUT / "manuscript_before.md") == c["manuscript_sha256"]
    assert sha(ROOT / "README.md") == c["root_readme_sha256"]
    sources = {}
    for name in ["followup_metrics.csv", "followup_detection.csv", "development_detection.csv", "decision.json",
                 "run.json", "independent_verification.json", "calendar_control_metrics.csv", "calendar_control_detection.csv"]:
        sources["Modeling/tables/regime_age_ablation/" + name] = sha(RESULT / name)
    run = json.loads((RESULT / "run.json").read_text(encoding="utf-8"))
    verified = json.loads((RESULT / "independent_verification.json").read_text(encoding="utf-8"))
    assert verified["status"] == "passed" and verified["run_sha256"] == sha(RESULT / "run.json")
    for name, expected in run["outputs_sha256"].items():
        assert sha(RESULT / name) == expected
    for name, expected in c["inputs_sha256"].items():
        assert sha(ROOT / name) == expected
    det = [r for r in rows("followup_detection.csv") if r["scope"] == "common_regression" and r["period"] == "Jul-Aug" and r["method"] == "noage"][0]
    assert [int(det[n]) for n in ["hours", "events", "tp", "fp", "fn"]] == [1340, 13, 7, 0, 6]
    links = 0
    for target in re.findall(r"\]\(([^)]+)\)", text):
        if not target.startswith(("http:", "https:", "#")):
            assert (MANUSCRIPT.parent / target.split("#")[0].strip("<>")).exists(), target
            links += 1
    record = {"status": "passed", "script_sha256": sha(__file__), "source_sha256": sources,
              "documents_sha256": {p.relative_to(ROOT).as_posix(): sha(p) for p in [MANUSCRIPT, README, ROOT / "README.md"]},
              "original_manuscript_sha256": sha(OUT / "manuscript_before.md"), "historical_tables_preserved": True, "historical_narrative_updated_as_declared": True,
              "links_checked": links, "new_fits": 0}
    (OUT / "document_verification.json").write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps({"status": "passed", "links_checked": links, "historical_tables_preserved": True, "new_fits": 0}), flush=True)


def write():
    original = MANUSCRIPT.read_text(encoding="utf-8")
    assert HEADING not in original and not (OUT / "manuscript_before.md").exists()
    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / "manuscript_before.md").write_bytes(MANUSCRIPT.read_bytes())
    assert sha(OUT / "manuscript_before.md") == sha(MANUSCRIPT)
    updated, count = re.subn(r"(?m)^이번 원고는 .*?$", lambda _: INTRO, revise_history(original), count=1)
    assert count == 1
    MANUSCRIPT.write_text(updated.rstrip() + "\n\n" + section().strip() + "\n", encoding="utf-8")
    text = README.read_text(encoding="utf-8")
    head = "## S06 실행 결과: 경과시간 입력 제거로 오경보 억제, 상승 탐지 범위의 한계"
    note = "\n\n**상세 원고 반영:** [4.12 지속 상승 예측과 선택적 HGB 보완](04_Modeling_원고.md#412-지속-상승-예측과-선택적-hgb-보완)에 S01~S06의 정의·학습·성능·오경보 개선·활용과 사용 코드를 연결했다. [regime_manuscript_documents.py](scripts/regime_manuscript_documents.py)로 추가 전 원고를 보존하고 기존 4.1~4.11 본문과 수치·링크를 검증했다. 최종 보고서는 방법·성능을 제2장, 오류 개선을 제3장, 차별성을 제5장에 배치한다."
    assert text.count(head) == 1
    README.write_text(text.replace(head, head + note, 1), encoding="utf-8")
    verify()


def refresh():
    # Rebuild only this authorized document from the preserved original and declared edits.
    assert '<!-- MODELING_CLOSE_BEGIN -->' not in MANUSCRIPT.read_text(encoding='utf-8'), 'Later reporting additions exist; use their document workflow.'
    original = (OUT / "manuscript_before.md").read_text(encoding="utf-8")
    updated, count = re.subn(r"(?m)^이번 원고는 .*?$", lambda _: INTRO, revise_history(original), count=1)
    assert count == 1 and MANUSCRIPT.read_text(encoding="utf-8").count(HEADING) == 1
    MANUSCRIPT.write_text(updated.rstrip() + "\n\n" + section().strip() + "\n", encoding="utf-8")
    verify()


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    mode = parser.add_mutually_exclusive_group(required=True)
    mode.add_argument("--write", action="store_true"); mode.add_argument("--verify", action="store_true")
    mode.add_argument("--refresh", action="store_true")
    args = parser.parse_args()
    write() if args.write else refresh() if args.refresh else verify()


코드 등록 완료: `Modeling/scripts/regime_manuscript_documents.py`

### `Modeling/scripts/regime_route.py`

원고 연결: Modeling\04_Modeling_3페이지용_압축본.md; Modeling\04_Modeling_원고.md.

코드 설명: One S04 follow-up: preserve B0 outside causal S03 upward alarms, no new fits.

주요 함수: `sha()`, `write()`, `inputs()`, `freeze()`, `stats()`, `run()`.

In [96]:
%%submission_source Modeling/scripts/regime_route.py
"""One S04 follow-up: preserve B0 outside causal S03 upward alarms, no new fits."""
import hashlib
import json
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
PARENT = ROOT / "Modeling/tables/regime_integration"
S03 = ROOT / "Modeling/tables/regime_forecast"
OUT = ROOT / "Modeling/tables/regime_routing"


def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def write(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def inputs():
    names = ["Modeling/tables/regime_integration/predictions.csv", "Modeling/tables/regime_integration/run.json",
             "Modeling/tables/regime_integration/independent_verification.json", "Modeling/tables/regime_integration/metrics.csv",
             "Modeling/tables/regime_forecast/predictions.csv", "Modeling/tables/regime_forecast/selection.csv",
             "Modeling/tables/regime_forecast/independent_verification.json"]
    return {n: sha(ROOT / n) for n in names}


def freeze():
    OUT.mkdir(parents=True, exist_ok=True)
    assert not (OUT / "contract.json").exists()
    verified = json.loads((PARENT / "independent_verification.json").read_text(encoding="utf-8"))
    assert verified["status"] == "passed" and verified["run_sha256"] == sha(PARENT / "run.json")
    write(OUT / "contract.json", {
        "stage": "S04 one diagnostic follow-up", "created_at": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
        "entrypoint_sha256": sha(__file__), "inputs_sha256": inputs(),
        "reason": "Uniform B2_up reduced up underprediction but worsened global MAE by5.07% and daily peak MAE by4.74%. B1 lowered up-start MAE in both May and June. Restrict supplementary regression use to predicted upward onset, without changing fitted values or alarms.",
        "scope": "May-Jun development only, same1462hours12sustainedup; analysis-selected follow-up, not independent confirmation",
        "variants": {"B0": "unchanged base", "B1_all": "uniform raw-state input model", "G1": "B1 when S03 past-selected up ML alarm, otherwise B0",
                     "G2": "B2_up when identical alarm, otherwise B0", "hour08": "B1 at target hour08, otherwise B0",
                     "simple_gate": "B1 when S03 past-selected simple upward alarm, otherwise B0"},
        "primary": "G1", "no_training": True,
        "alarms": "Frozen S03 selection and 1% calibration FPR thresholds from earlier OOF only. No new boundary, top-k, oracle event mask, threshold tuning or production-origin assertion.",
        "gate": {"up_start_mae_reduction_vs_B0": .05, "up_under_reduction_vs_B0": .05,
                 "overall_mae_worsening_vs_B0_at_most": .01, "daily_peak_mae_worsening_vs_B0_at_most": .01,
                 "monthly_start_improvements_vs_B0_required": 2},
        "control_interpretation": "G1 vs B0 is combined routing/raw-input effect; numeric probability input remains unproven. G1/hour08 have identical values at08sustained event starts, so do not claim event-error improvement beyond calendar routing. Compare all1462hour errors and changed-hour burden too.",
        "stop": "No further same-period routing/model/threshold search after this comparison; retain S03 detection separately from value integration."})
    print(json.dumps({"status": "frozen", "contract_sha256": sha(OUT / "contract.json")}), flush=True)


def stats(g):
    e = g.prediction.to_numpy() - g.actual.to_numpy()
    return {"hours": len(g), "mae": float(np.mean(abs(e))), "rmse": float(np.sqrt(np.mean(e ** 2))),
            "under": float(np.mean(np.maximum(-e, 0))), "over": float(np.mean(np.maximum(e, 0))), "bias": float(np.mean(e))}


def run():
    assert not (OUT / "run.json").exists()
    c = json.loads((OUT / "contract.json").read_text(encoding="utf-8"))
    assert c["inputs_sha256"] == inputs() and c["entrypoint_sha256"] == sha(__file__)
    p = pd.read_csv(PARENT / "predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    p = p.loc[p.month >= 5]
    f = p.loc[p.variant == "B0"].copy().set_index("timestamp")
    assert len(f) == 1462
    f["b0"] = f.prediction
    for variant, name in [("B1", "b1"), ("B2_up", "b2")]:
        other = p.loc[p.variant == variant].set_index("timestamp").loc[f.index]
        assert np.allclose(f.actual, other.actual)
        f[name] = other.prediction
    scores = pd.read_csv(S03 / "predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    selection = pd.read_csv(S03 / "selection.csv", encoding="utf-8-sig")
    f["alarm_ml"] = 0; f["alarm_simple"] = 0
    for month in [5, 6]:
        r = selection.loc[(selection.direction == "up") & (selection.month == month)].iloc[0]
        assert r.calibration_available
        for family in ["ml", "simple"]:
            ss = scores.loc[(scores.direction == "up") & (scores.month == month) & (scores.method == r[f"selected_{family}"])].set_index("timestamp")
            own = f.index[f.month == month]
            assert ss.loc[own, "alarm"].notna().all()
            f.loc[own, "alarm_" + family] = ss.loc[own, "alarm"].to_numpy(int)
    frames = []; metrics = []; burden = []; daily_rows = []
    options = {"B0": f.b0, "B1_all": f.b1,
               "G1": np.where(f.alarm_ml == 1, f.b1, f.b0), "G2": np.where(f.alarm_ml == 1, f.b2, f.b0),
               "hour08": np.where(f.index.hour == 8, f.b1, f.b0),
               "simple_gate": np.where(f.alarm_simple == 1, f.b1, f.b0)}
    masks = {"B0": np.zeros(len(f), bool), "B1_all": np.ones(len(f), bool), "G1": f.alarm_ml == 1,
             "G2": f.alarm_ml == 1, "hour08": f.index.hour == 8, "simple_gate": f.alarm_simple == 1}
    for variant, yhat in options.items():
        g = f.copy(); g["variant"] = variant; g["prediction"] = yhat; g["routed"] = np.asarray(masks[variant], int)
        g = g.reset_index(); frames.append(g)
        for period, gg in [("May-Jun", g), ("5", g.loc[g.month == 5]), ("6", g.loc[g.month == 6])]:
            for name, part in [("all", gg), ("up_start", gg.loc[gg.sustained_onset == 1]),
                               ("other_hours", gg.loc[gg.sustained_onset != 1])]:
                metrics.append({"variant": variant, "period": period, "condition": name, **stats(part)})
        burden.append({"variant": variant, "hours": len(g), "routed_hours": int(g.routed.sum()),
                       "routed_up_starts": int(((g.routed == 1) & (g.sustained_onset == 1)).sum()),
                       "routed_other_hours": int(((g.routed == 1) & (g.sustained_onset != 1)).sum())})
        for date, gg in g.groupby("date"):
            if len(gg) == 24:
                peak = gg.loc[gg.actual == gg.actual.max()]
                daily_rows.append({"variant": variant, "date": date, "ties": len(peak), **stats(peak)})
    predictions = pd.concat(frames, ignore_index=True)
    daily = pd.DataFrame(daily_rows)
    for variant, g in daily.groupby("variant"):
        metrics.append({"variant": variant, "period": "May-Jun", "condition": "daily_peak_day_equal", "hours": int(g.ties.sum()),
                        "days": len(g), **{n: float(g[n].mean()) for n in ["mae", "under", "over", "bias"]},
                        "rmse": float(np.sqrt(np.mean(g.rmse ** 2)))})
    table = pd.DataFrame(metrics)
    def get(v, cond, period="May-Jun"):
        return table.loc[(table.variant == v) & (table.condition == cond) & (table.period == period)].iloc[0]
    decisions = []
    for variant in ["G1", "G2", "hour08"]:
        a, b = get(variant, "up_start"), get("B0", "up_start")
        gain = 1 - a.mae / b.mae; under_gain = 1 - a.under / b.under
        all_change = get(variant, "all").mae / get("B0", "all").mae - 1
        peak_change = get(variant, "daily_peak_day_equal").mae / get("B0", "daily_peak_day_equal").mae - 1
        wins = sum(get(variant, "up_start", m).mae < get("B0", "up_start", m).mae for m in ["5", "6"])
        decisions.append({"variant": variant, "mae_reduction": gain, "under_reduction": under_gain,
                          "overall_mae_change": all_change, "daily_peak_mae_change": peak_change, "monthly_wins": int(wins),
                          "development_gate_passed": bool(gain >= .05 and under_gain >= .05 and all_change <= .01 and peak_change <= .01 and wins >= 2)})
    outputs = {"predictions.csv": predictions, "metrics.csv": table, "routing_burden.csv": pd.DataFrame(burden),
               "daily_peak_errors.csv": daily}
    for name, frame in outputs.items():
        frame.to_csv(OUT / name, index=False, encoding="utf-8-sig")
    write(OUT / "decision.json", {"primary": "G1", "comparisons": decisions, "stop": c["stop"], "numeric_probability_ablation": "Original B2 variants failed. Routing does not overturn that result."})
    write(OUT / "run.json", {"status": "completed", "script_sha256": sha(__file__), "contract_sha256": sha(OUT / "contract.json"),
                             "inputs_sha256": inputs(), "new_fits": 0,
                             "outputs_sha256": {n: sha(OUT / n) for n in list(outputs) + ["decision.json"]}})
    print(json.dumps({"status": "completed", "decisions": decisions}), flush=True)


if __name__ == "__main__":
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    {"freeze": freeze, "run": run}[sys.argv[1]]()


코드 등록 완료: `Modeling/scripts/regime_route.py`

### `Modeling/scripts/regime_route_verify.py`

원고 연결: Modeling\04_Modeling_원고.md.

코드 설명: Independent scalar verification of the single causal S04 routing follow-up.

주요 함수: `sha()`, `close()`, `stats()`, `main()`.

In [97]:
%%submission_source Modeling/scripts/regime_route_verify.py
"""Independent scalar verification of the single causal S04 routing follow-up."""
import hashlib
import json
import math
import sys
from pathlib import Path
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "Modeling/tables/regime_routing"


def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def close(a, b):
    assert math.isclose(float(a), float(b), rel_tol=1e-9, abs_tol=1e-8), (a, b)


def stats(g):
    errors = [float(r.prediction) - float(r.actual) for r in g.itertuples()]
    return {"hours": len(g), "mae": math.fsum(abs(e) for e in errors) / len(g),
            "rmse": math.sqrt(math.fsum(e * e for e in errors) / len(g)),
            "bias": math.fsum(errors) / len(g), "under": math.fsum(max(-e, 0) for e in errors) / len(g),
            "over": math.fsum(max(e, 0) for e in errors) / len(g)}


def main():
    assert sys.version_info[:2] == (3, 13) and sys._is_gil_enabled()
    c = json.loads((OUT / "contract.json").read_text(encoding="utf-8"))
    run = json.loads((OUT / "run.json").read_text(encoding="utf-8"))
    assert run["status"] == "completed" and run["contract_sha256"] == sha(OUT / "contract.json")
    assert c["entrypoint_sha256"] == run["script_sha256"] == sha(ROOT / "Modeling/scripts/regime_route.py")
    for name, digest in c["inputs_sha256"].items():
        assert sha(ROOT / name) == digest
    for name, digest in run["outputs_sha256"].items():
        assert sha(OUT / name) == digest
    parent_verify = json.loads((ROOT / "Modeling/tables/regime_integration/independent_verification.json").read_text(encoding="utf-8"))
    assert parent_verify["status"] == "passed" and parent_verify["run_sha256"] == sha(ROOT / "Modeling/tables/regime_integration/run.json")
    parent = pd.read_csv(ROOT / "Modeling/tables/regime_integration/predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    base = {(r.variant, r.timestamp): r for r in parent.itertuples()}
    scores = pd.read_csv(ROOT / "Modeling/tables/regime_forecast/predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    choices = pd.read_csv(ROOT / "Modeling/tables/regime_forecast/selection.csv", encoding="utf-8-sig")
    alarms = {}
    for month in [5, 6]:
        choice = choices.loc[(choices.direction == "up") & (choices.month == month)].iloc[0]
        assert pd.Timestamp(choice.calibration_latest_confirmed) < pd.Timestamp(2021, month, 1)
        for family in ["ml", "simple"]:
            for r in scores.loc[(scores.direction == "up") & (scores.month == month) & (scores.method == choice["selected_" + family])].itertuples():
                assert not pd.isna(r.alarm) and int(r.alarm) == int(r.score >= r.threshold)
                alarms[(family, r.timestamp)] = int(r.alarm)
    p = pd.read_csv(OUT / "predictions.csv", encoding="utf-8-sig", parse_dates=["timestamp"])
    assert len(p) == 8772 and not p.duplicated(["variant", "timestamp"]).any()
    for r in p.itertuples():
        b0, b1, b2 = [base[(v, r.timestamp)] for v in ["B0", "B1", "B2_up"]]
        a, simple = alarms[("ml", r.timestamp)], alarms[("simple", r.timestamp)]
        expected = {"B0": b0.prediction, "B1_all": b1.prediction, "G1": b1.prediction if a else b0.prediction,
                    "G2": b2.prediction if a else b0.prediction,
                    "hour08": b1.prediction if r.timestamp.hour == 8 else b0.prediction,
                    "simple_gate": b1.prediction if simple else b0.prediction}[r.variant]
        routed = {"B0": 0, "B1_all": 1, "G1": a, "G2": a, "hour08": int(r.timestamp.hour == 8), "simple_gate": simple}[r.variant]
        close(r.prediction, expected); close(r.actual, b0.actual)
        assert r.routed == routed and r.alarm_ml == a and r.alarm_simple == simple
    daily = pd.read_csv(OUT / "daily_peak_errors.csv", encoding="utf-8-sig")
    assert len(daily) == 6 * 60
    for r in daily.itertuples():
        own = p.loc[(p.variant == r.variant) & (p.date == r.date)]
        assert len(own) == 24
        peak = own.loc[own.actual == own.actual.max()]
        assert len(peak) == r.ties
        for name, value in stats(peak).items():
            close(getattr(r, name), value)
    metrics = pd.read_csv(OUT / "metrics.csv", encoding="utf-8-sig", dtype={"period": str})
    for r in metrics.itertuples():
        own = p.loc[p.variant == r.variant]
        if r.period in ["5", "6"]:
            own = own.loc[own.month == int(r.period)]
        if r.condition == "up_start":
            own = own.loc[own.sustained_onset == 1]
        elif r.condition == "other_hours":
            own = own.loc[own.sustained_onset != 1]
        if r.condition == "daily_peak_day_equal":
            own = daily.loc[daily.variant == r.variant]
            expected = {n: math.fsum(own[n]) / len(own) for n in ["mae", "bias", "under", "over"]}
            expected["rmse"] = math.sqrt(math.fsum(v * v for v in own.rmse) / len(own))
            expected["hours"] = int(own.ties.sum())
            assert len(own) == r.days
        else:
            expected = stats(own)
        for name, value in expected.items():
            close(getattr(r, name), value)
    for r in pd.read_csv(OUT / "routing_burden.csv", encoding="utf-8-sig").itertuples():
        own = p.loc[p.variant == r.variant]
        assert r.hours == len(own) and r.routed_hours == own.routed.sum()
        assert r.routed_up_starts == len(own.loc[(own.routed == 1) & (own.sustained_onset == 1)])
        assert r.routed_other_hours == r.routed_hours - r.routed_up_starts
    decision = json.loads((OUT / "decision.json").read_text(encoding="utf-8"))
    def get(v, cond, period="May-Jun"):
        return metrics.loc[(metrics.variant == v) & (metrics.condition == cond) & (metrics.period == period)].iloc[0]
    for r in decision["comparisons"]:
        v = r["variant"]; a, b = get(v, "up_start"), get("B0", "up_start")
        gain = 1 - a.mae / b.mae; under = 1 - a.under / b.under
        all_change = get(v, "all").mae / get("B0", "all").mae - 1
        peak_change = get(v, "daily_peak_day_equal").mae / get("B0", "daily_peak_day_equal").mae - 1
        wins = sum(get(v, "up_start", m).mae < get("B0", "up_start", m).mae for m in ["5", "6"])
        for field, expected in [("mae_reduction", gain), ("under_reduction", under), ("overall_mae_change", all_change), ("daily_peak_mae_change", peak_change)]:
            close(r[field], expected)
        assert wins == r["monthly_wins"]
        assert r["development_gate_passed"] == (gain >= .05 and under >= .05 and all_change <= .01 and peak_change <= .01 and wins >= 2)
    # Future prediction changes cannot alter any earlier routing or resulting values.
    checks = 0
    for cut in [pd.Timestamp("2021-05-15"), pd.Timestamp("2021-06-15")]:
        earlier = p.loc[p.timestamp < cut]
        trimmed_parent = {key: value for key, value in base.items() if key[1] < cut}
        trimmed_alarms = {key: value for key, value in alarms.items() if key[1] < cut}
        for r in earlier.loc[earlier.variant == "G1"].itertuples():
            used = "B1" if trimmed_alarms[("ml", r.timestamp)] else "B0"
            close(r.prediction, trimmed_parent[(used, r.timestamp)].prediction)
        checks += 1
    g1 = p.loc[(p.variant == "G1") & (p.sustained_onset == 1)]
    hour = p.loc[(p.variant == "hour08") & (p.sustained_onset == 1)]
    assert g1.timestamp.tolist() == hour.timestamp.tolist() and g1.prediction.tolist() == hour.prediction.tolist()
    cases = p.loc[(p.variant == "G1") & (p.routed == 1)].copy()
    cases["B0_absolute_error"] = abs(cases.b0 - cases.actual)
    cases["G1_absolute_error"] = abs(cases.prediction - cases.actual)
    cases.to_csv(OUT / "verified_routed_cases.csv", index=False, encoding="utf-8-sig")
    result = {"status": "passed", "run_sha256": sha(OUT / "run.json"), "verifier_sha256": sha(__file__),
              "predictions_checked": len(p), "daily_rows_checked": len(daily), "metrics_checked": len(metrics),
              "future_removal_checks": checks, "new_fits": 0, "alarm_origin_past_only": True,
              "calendar_routing_event_predictions_identical": True,
              "extra_outputs_sha256": {"verified_routed_cases.csv": sha(OUT / "verified_routed_cases.csv")}}
    (OUT / "independent_verification.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result, ensure_ascii=False), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/regime_route_verify.py`

### `Modeling/scripts/verify_m03_documents.py`

원고 연결: Modeling\04_Modeling_원고.md; Modeling\04_Modeling_원고.md.

코드 설명: Check current M03 documentation links, figures and displayed numeric tables.

주요 함수: `sha()`, `main()`.

In [98]:
%%submission_source Modeling/scripts/verify_m03_documents.py
"""Check current M03 documentation links, figures and displayed numeric tables."""
import csv
import hashlib
import json
import re
from datetime import datetime, timedelta, timezone
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def main():
    documents = [ROOT / name for name in ["README.md", "Modeling/README.md", "Modeling/04_Modeling_원고.md", "Modeling/10.04_M03_마지막값추가와시차입력_실행기록.md"]]
    links = 0
    for path in documents:
        text = path.read_text(encoding="utf-8-sig")
        if path == ROOT / "README.md":
            text = text.split("## 이전 jsw 작업 단계 기록")[0]
        assert not re.search(r"[\u3040-\u30ff]", text), path
        for target in re.findall(r"!?\[[^\]]*\]\(([^)]+)\)", text):
            if target.startswith(("http:", "https:", "#", "app:")):
                continue
            assert (path.parent / target.split("#")[0].strip("<>")).exists(), (path, target)
            links += 1
        assert text.count("```") % 2 == 0, path
    manuscript = documents[2].read_text(encoding="utf-8-sig")
    m03 = manuscript.split("## 4.3 M03:")[1].split("## 4.4 ")[0]
    tables = ROOT / "Modeling/tables/m03/c"
    with (tables / "comparison.csv").open(encoding="utf-8-sig", newline="") as stream:
        comparison = list(csv.DictReader(stream))
    with (tables / "condition_errors.csv").open(encoding="utf-8-sig", newline="") as stream:
        conditions = list(csv.DictReader(stream))
    numeric_rows = 0
    for group in ["A", "B", "C_A", "C_B"]:
        maximum = next(r for r in comparison if r["group"] == group and r["target"] == "target_maximum" and r["split"] == "pooled_development")
        mean = next(r for r in comparison if r["group"] == group and r["target"] == "target_mean" and r["split"] == "pooled_development")
        line = "| " + group + " | " + " | ".join(f"{float(maximum[x]):.2f}" for x in ["MAE", "RMSE", "mean_under", "mean_over"]) + f" | {float(mean['MAE']):.2f} |"
        assert line in m03.replace("**", ""), line
        peak = next(r for r in conditions if r["group"] == group and r["target"] == "target_maximum" and r["condition"] == "daily_maximum")
        line = "| " + group + " | " + " | ".join(f"{float(peak[x]):.2f}" for x in ["MAE", "mean_under", "mean_over"]) + " |"
        assert line in m03.replace("**", ""), line
        numeric_rows += 2
    reduction_rates = {}
    for label, condition, metric in [("overall_MAE", "all", "MAE"),
                                     ("daily_maximum_MAE", "daily_maximum", "MAE"),
                                     ("daily_maximum_under", "daily_maximum", "mean_under")]:
        pair = {group: next(r for r in conditions if r["group"] == group and
                            r["target"] == "target_maximum" and r["condition"] == condition and r["value"] == "all")
                for group in ["A", "B"]}
        rate = (1 - float(pair["B"][metric]) / float(pair["A"][metric])) * 100
        assert f"{rate:.1f}%" in m03, (label, rate)
        reduction_rates[label] = rate
    figures = tables / "manuscript_figures.json"
    metadata = json.loads(figures.read_text(encoding="utf-8"))
    for file, digest in metadata["figures"].items():
        assert sha(ROOT / file) == digest
    metadata["visual_review"] = "passed: figure viewed at 50 percent; axes, legends, monthly comparison, peak under/over stack and denominators checked"
    metadata["manuscript_sha256"] = sha(documents[2])
    metadata["document_validation"] = {"links_checked": links, "M03_numeric_rows_checked": numeric_rows, "scope": "four current documents; historical root jsw section excluded"}
    figures.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    result = {"status": "passed", "verified_at": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="minutes"),
              "links_checked": links, "M03_numeric_rows_checked": numeric_rows,
              "M03_reduction_rates_checked": reduction_rates,
              "documents_sha256": {str(p.relative_to(ROOT)): sha(p) for p in documents},
              "no_fitting": True, "script_sha256": sha(Path(__file__))}
    (tables / "document_verification.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result, ensure_ascii=False), flush=True)


if __name__ == "__main__":
    main()


코드 등록 완료: `Modeling/scripts/verify_m03_documents.py`

원고에 기록된 결과와 해석을 아래에 보존했다. 다음 실행 셀의 결과는 이번 실행에서 계산하거나 검증한 값이다.

# 1. 데이터 확인과 전처리

## 1.1 원본 자료와 EDA 범위

자원 최적화 AI 데이터셋의 `okm_augumented_2021.csv`를 열어 기록 구조, 결측, 시간 오류와 변수의 계산 관계를 점검하였다. 가이드북은 공장 변압기 센서의 전력 기록에 ERP 생산량과 기상·달력·비용 정보를 결합한 자료로 설명한다. **한 행은 한 시간에 대응하며, 그 안에 15분 구간별 전력값 네 개가 있다.**

이 자료로 관찰할 수 있는 것은 **공장 단위의 전력 수요 변화와 이에 동반되는 생산·환경 조건**이다. 본 분석은 이러한 조건에 따른 전력 패턴을 확인하고, 관측을 마친 과거 정보를 이용해 다음 한 시간의 최대전력을 예측하는 것을 목적으로 한다. CSV에는 공정 단계와 설비 식별자가 없어 개별 설비의 가동·정지 상태나 특정 공정의 전력 사용 원인은 현 자료에서 식별되지 않는다.

| 항목 | 원본 확인 결과 |
|---|---|
| 규모·기간 | 6,168행 × 18열, 2021년 1월 1일 ~ 9월 14일, 257일 |
| 전력 | `15분·30분·45분·60분` 및 `평균` |
| 생산·기상 | 생산량, 기온, 풍속, 습도, 강수량 |
| 시간·기타 | 날짜, 시간, 달력 변수, 전기요금(계절), 공장인원, 인건비 |

이후 EDA의 분석 기간은 **1 ~ 8월**로 정하였다. 이 기간의 5,832행에서 시간 오류 48행을 제외한 **5,784시간·241일**을 분석 자료로 구성하였다. 9월의 336행은 이번 EDA 범위에 포함하지 않으며, 오류 때문에 제외한 자료는 아니다. 원본 전체를 점검한 수치와 EDA 분석 자료의 수치는 아래에서 구분하였다.

**관련 Python 코드:** [EDA/scripts/data_overview.py](EDA/scripts/data_overview.py)의 `main()`에서 원본 규모·기간과 기간별 행 수를 확인한다.

## 1.2 변수의 의미와 사용 기준

주요 변수의 정의, 관찰 가능한 상태·조건과 분석 목적의 연결은 다음과 같다.

| 주요 변수 | 정의 | 나타내는 상태·조건 | 분석 목적과의 연결 |
|---|---|---|---|
| `15분·30분·45분·60분` | 한 시간 안의 네 15분 구간 전력 기록 | 공장 전력 수요의 수준과 시간 내 변동 | 높은 전력 구간을 파악하고, 다음 시간의 네 값 중 최댓값을 예측 대상으로 정의한다. |
| `평균` | 네 전력값의 산술평균을 정수로 반올림한 값 | 해당 시간의 평균적인 전력 수준 | 시간대·월별 수준과 생산량·기상 변수의 관계를 비교한다. |
| `생산량` | ERP에서 결합된 생산량 정보 | 전력 기록과 함께 비교할 생산 규모 정보 | 생산량의 크기·증감에 따라 전력 수준과 변동이 달라지는지 확인한다. |
| 기온·풍속·습도·강수량 | 해당 시점에 결합된 기상 정보 | 공장 운영과 함께 관측된 외부 환경 조건 | 생산·시간 조건을 고려해 기상과 전력의 관계를 검토한다. |
| 날짜·시간·`day`·`d`·`m` | 관측 날짜·시간과 요일·일자·월 정보 | 시간대·평일과 주말·계절적 조건 | 반복 패턴을 비교하고, 과거 기록과 예측 대상 시점을 연결한다. |

생산량의 계획·실적 구분과 일자별 ERP 값을 시간별로 배정한 방식은 현 자료에서 식별되지 않는다. 따라서 생산량 0을 휴무로, 낮은 전력을 설비 정지로 단정하지 않는다. 기상과 전력의 관계도 특정 설비의 냉난방 부하를 직접 측정한 결과로 해석하지 않는다. 위 표의 관계 탐색과 예측 입력은 구분하며, 예측에는 관측을 마친 과거 정보와 미리 알 수 있는 목표 시간의 달력 정보만 사용한다. 목표 시간의 실제 전력·생산량·날씨는 입력에 포함하지 않는다.

원본 `평균`은 네 전력값의 산술평균을 정수로 반올림한 값이다. 전체 6,168행에서 다음 관계가 성립하였다.

`평균 = floor((15분 + 30분 + 45분 + 60분) / 4 + 0.5)`

시간대·월별 전력 수준과 다른 변수의 관계를 살펴볼 때는 이 `평균`을 사용한다. 한 시간 안의 네 구간을 비교할 때는 **반올림 전 산술평균**을 기준으로 각 구간의 차이를 계산한다. 반올림에 따른 차이가 구간별 차이에 섞이지 않도록 구분한 것이다. CSV의 원단위와 계측 집계식은 현 자료에서 식별되지 않아 kW·kWh 또는 실제 요금으로 환산하지 않는다.

`공장인원`은 비결측 6,151행 모두에서 다음 관계와 소수점 저장 오차 수준으로 일치하였다.

`공장인원 = 생산량 / (15분 + 30분 + 45분 + 60분)`

따라서 이를 실제 근무 인원으로 해석하거나, 생산량·전력과 별개의 외부 조건으로 사용하지 않는다. 이후 EDA의 관계 탐색에는 생산량과 기온·풍속·습도·강수량을 사용한다.

**관련 Python 코드:** [EDA/scripts/data_overview.py](EDA/scripts/data_overview.py)의 `main()`에서 평균의 반올림 관계와 공장인원 계산 관계를 검증한다.

## 1.3 결측 확인과 처리

원본의 결측은 **21셀**이며 풍속·강수량·공장인원에만 있다. 전력 네 값, 평균 전력, 생산량에는 결측이 없다. 21셀은 모두 EDA 분석 기간의 정상 시간 행에 있으므로, 시간 오류를 제외한 뒤에도 남는다.

| 변수 | 결측 수와 위치 | EDA 처리 기준 |
|---|---|---|
| 풍속 | 3셀: 6월 1일 01·02시, 7월 4일 20시 | 보간하거나 0으로 채우지 않는다. 풍속을 사용하는 계산에서만 해당 기록을 제외한다. |
| 강수량 | 1셀: 1월 24일 00시 | 결측을 강수량 0으로 간주하지 않는다. 강수량을 사용하는 계산에서만 제외한다. |
| 공장인원 | 17셀: 8월 28일 18시 ~ 29일 10시 | 생산량과 네 전력값이 모두 0으로, 위 비율식의 0/0에 해당한다. 공장인원은 분석 변수에서 제외하되 해당 행의 다른 기록은 유지한다. |

상관분석은 **비교하는 두 변수에 모두 값이 있는 행**으로 계산한다. 따라서 평균 전력과 풍속의 비교에는 5,781행, 강수량과의 비교에는 5,783행을 사용할 수 있다. 기상 결측이 있는 시간도 전력·생산량의 시간대별 집계에는 포함한다. 전체 행을 일괄 삭제하거나 결측을 채운 단일 완전 자료로 바꾸지 않았다.

**관련 Python 코드:** 결측 위치·개수는 [EDA/scripts/data_overview.py](EDA/scripts/data_overview.py)의 `main()`, 비교 변수별 결측 제외는 [EDA/scripts/eda_variable_relations.py](EDA/scripts/eda_variable_relations.py)의 `relations()`에서 확인할 수 있다.

## 1.4 시간 오류·극단값·0값의 처리

시간 범위를 벗어난 명확한 오류는 제외하였다. 값이 크거나 0이라는 이유만으로 관측값을 삭제하지는 않았다.

| 점검 항목 | 확인 결과 | 처리와 판단 근거 |
|---|---|---|
| 시간 오류 | 7월 13일·15일의 각 24행, 총 48행에서 시간이 70 ~ 188로 기록됨 | 실제 시각을 식별할 수 없어 EDA에서 제외하였다. 행 순서나 중앙값으로 시각을 복원하지 않았다. |
| 생산량 극단값 | 1 ~ 8월 5,832행 중 533행이 1.5×IQR 기준 밖이며 최댓값은 9,830 | 큰 값을 오류로 볼 독립적인 근거가 없어 원래 값을 유지하였다. |
| 강수량 분포 | 1 ~ 8월 비결측 5,831행 중 75.27%가 0. IQR이 0이어서 양수 1,442행 모두가 기준 밖으로 표시됨 | 모든 양수 강수를 제거하게 되는 IQR 기준을 삭제 규칙으로 사용하지 않았다. |
| 전력 0 | 네 전력값이 모두 0인 연속 17행 | 결측과 구분해 유지하였다. 정지·정전·계측 문제 중 원인은 현 자료에서 식별되지 않는다. |

IQR 기준은 `Q1−1.5×IQR` 미만 또는 `Q3+1.5×IQR` 초과다. 위 생산량·강수량 점검은 시간 오류 제외 전의 변수 분포를 확인한 것으로, EDA에 사용할 5,784행과 분모가 다르다. 높은 전력도 크기만을 이유로 제거하거나 일정 값으로 제한하지 않았다.

**관련 Python 코드:** [EDA/scripts/data_overview.py](EDA/scripts/data_overview.py)의 `main()`에서 시간 오류, IQR 기준 밖 기록과 전력 0값을 확인한다.

## 1.5 반복 기록 점검과 EDA 자료 구성

원본 전체에서 완전히 동일한 중복 행은 없었다. 다만 시간 오류를 제외한 241일의 하루 96개 전력값을 비교하면, 서로 다른 배열은 **126개**이고 **160일이 반복 그룹에 속했다.** 다른 날짜의 생산량·기상까지 동일한 중복 행은 아니므로 날짜를 일괄 삭제하지 않았다. 반복 원인은 현 자료에서 식별되지 않으며, 이후 패턴을 해석할 때 같은 전력 배열이 여러 날짜에 등장할 수 있음을 고려한다.

원본 전체에서 생산량이 0인 2,657행 중 2,640행에는 양수의 평균 전력이 기록되어 있었다. **생산량 0을 결측이나 실제 휴무로 바꾸지 않고 유지하였다.** 또한 정상 241일 중 181일에는 하루 안에 여러 생산량 값이 있다. 가이드북의 ERP 일자별 생산량을 시간별로 배정한 방식은 식별되지 않아, 시간별 기록의 합을 검증된 공식 일일 생산실적으로 간주하지 않는다.

최종적으로 EDA에는 **1 ~ 8월 정상 시간 5,784행·241일**을 사용한다. 날짜에서 요일을 계산하면 월 ~ 금요일 171일·4,104시간과 토·일요일 70일·1,680시간으로 구성된다. 공휴일은 해당 요일에 포함한다. 분석 질문에 따라 평일만 비교하더라도 공통 출발 자료는 이 5,784행이며, 기상 결측의 제외 범위는 사용하는 변수에 따라 달라진다.

원본 CSV는 보존하였다. 별도의 전처리 완료 CSV를 저장해 공통 입력으로 읽는 방식은 아니며, **각 EDA 코드가 원본에서 같은 기간·시간 필터를 적용하고 필요한 변수를 선택한다.** 따라서 전처리는 원본을 수정하는 작업이 아니라 분석에 사용할 행과 변수, 결측 처리 기준을 정하는 작업이다.

**관련 Python 코드:** [EDA/scripts/data_overview.py](EDA/scripts/data_overview.py)의 `main()`에서 반복 배열·생산량 0·날짜별 생산값을 확인한다. 실제 EDA 입력 구성은 [EDA/scripts/eda_daily_pattern.py](EDA/scripts/eda_daily_pattern.py)의 `main()`, [EDA/scripts/eda_daily_repetition.py](EDA/scripts/eda_daily_repetition.py)의 `main()`, [EDA/scripts/eda_variable_relations.py](EDA/scripts/eda_variable_relations.py)의 `source_frame()`, [EDA/scripts/eda_slot_time_patterns.py](EDA/scripts/eda_slot_time_patterns.py)의 `load_data()`에서 확인할 수 있다.

---

<details>
<summary>팀원 확인용: 근거 파일과 비교 원고</summary>

이 문서는 EDA에 앞선 원본 점검과 처리 기준을 설명한다. 이후 EDA의 패턴·상관 결과나 모델링 결과는 포함하지 않았으며, 플롯 이미지는 사용하지 않았다.

- **비교 원고:** [0.5페이지용 데이터 압축안](EDA/01_데이터_0.5페이지용_압축본.md)
- **후속 문서:** [EDA 상세 원고](EDA/02_EDA_원고.md), [EDA 보고서 압축안](EDA/02_EDA_3페이지용_압축본.md)
- **원본:** [data/origin/okm_augumented_2021.csv](EDA/../data/origin/okm_augumented_2021.csv)
- **변수 정의·가이드북 대조:** [data/origin/데이터_변수표.md](EDA/../data/origin/데이터_변수표.md), [data/origin/Guidebook_자원 최적화 AI 데이터셋_내용 정리.md](EDA/../data/origin/Guidebook_자원 최적화 AI 데이터셋_내용 정리.md)
- **분석 목적·예측 시점 연결:** [Modeling/04_Modeling_원고.md](EDA/../Modeling/04_Modeling_원고.md)의 4.1. 다음 시간 최대전력의 정의와 과거 입력·목표 시간 달력의 사용 범위를 따른다.
- **수치 근거:** [EDA/tables/data_overview_manifest.json](EDA/tables/data_overview_manifest.json), [EDA/tables/data_missing_records.csv](EDA/tables/data_missing_records.csv), [EDA/tables/data_invalid_times.csv](EDA/tables/data_invalid_times.csv), [EDA/tables/data_distribution.csv](EDA/tables/data_distribution.csv)
- **현재 EDA 입력 대조:** [tmp/current_eda_input_audit.json](EDA/../tmp/current_eda_input_audit.json). 원본으로 재계산한 값과 EDA 2.1 ~ 2.5의 주요 결과표가 일치함을 확인한 검증 기록이다.

</details>

### 1.6 데이터 확인 코드 실행

6168행의 원본에서 1 ~ 8월 정상 5784시간을 확인하고, 평균 반올림·공장인원 파생 관계·결측·시간 오류·반복 배열을 검증한다. 이어 M01의 정확한 시차와 학습·평가 대상 행을 재구성한다.

In [99]:
with computation_paths():
    execute_step('01_데이터확인',MODULES['data_overview'].main)
    execute_step('02_시차자료구성',MODULES['m01_prepare'].main)
    execute_step('03_시차독립검증',MODULES['m01_verify'].main)
facts=json.loads((WORK/'EDA/tables/data_overview_manifest.json').read_text(encoding='utf-8'))
display(Markdown('### 데이터 실행 결과

'+pd.DataFrame([facts]).to_markdown(index=False)))

SyntaxError: unterminated string literal (detected at line 6) (1745670020.py, line 6)

원고에 기록된 결과와 해석을 아래에 보존했다. 다음 실행 셀의 결과는 이번 실행에서 계산하거나 검증한 값이다.

# 2. EDA

## 2.1 평일과 주말의 생산량·전력 하루 패턴

하루 중 생산량과 전력이 언제 높아지고 낮아지는지, 그 패턴이 평일과 주말에 어떻게 다른지 비교했다. 2021년 1 ~ 8월의 정상 기록 241일을 평일 171일과 주말 70일로 나누고, 각 시간대의 평균을 계산했다. 평일은 월 ~ 금요일, 주말은 토·일요일이며 공휴일도 해당 요일에 포함했다.

![평일과 주말의 시간대별 평균 생산량 및 평균 전력](EDA/figures/restart_01_daily_pattern_clean.png)

이미지 파일: [EDA/figures/restart_01_daily_pattern_clean.png](EDA/figures/restart_01_daily_pattern_clean.png). 생성 코드: `EDA/scripts/eda_daily_pattern.py`의 `main()`.

*위는 생산량, 아래는 전력이다. 파란 실선은 평일, 주황 점선은 주말을 나타낸다. 예를 들어 평일 08시의 값은 평일 171일의 08시 기록을 평균한 값이다. 생산량과 전력의 단위가 달라 세로축은 각각 표시했다.*

**평일에는 12시에 두 값이 함께 낮아졌다.** 생산량과 전력 모두 11시보다 12시에 낮았고, 13시에 다시 높아졌다. 하루 안에서도 수준이 일정하게 유지되지는 않았다.

**생산량이 가장 많은 시간과 전력이 가장 높은 시간은 달랐다.** 평일의 시간대별 평균 생산량은 19시에 가장 높았지만, 평균 전력은 08시에 가장 높았다. 생산량과 전력의 하루 패턴이 그대로 일치하지는 않았다.

**주말에는 생산량이 낮아도 전력 기록이 이어졌다.** 08시 이후 주말의 평균 생산량은 거의 0이었지만, 평균 전력은 0으로 내려가지 않았다. 예를 들어 19시에는 주말 생산량 기록이 모두 0이었는데 평균 전력은 41.11이었다. 생산량 기록만으로 전력의 하루 패턴을 설명하기 어려운 부분이다.

이 비교에서 평일·주말의 차이와 시간대에 따른 변화가 함께 확인됐다. 생산량과 전력의 관계를 살펴볼 때는 같은 시간대인지, 평일인지 주말인지를 함께 구분할 필요가 있다.

*생산량은 CSV의 ‘생산량’ 열, 전력은 ‘평균’ 열을 사용했다. ‘평균’은 한 시간 안의 네 전력값을 평균해 정수로 반올림한 값이다. 수치는 원자료 기록값 단위이며, 위 패턴을 실제 점심시간·휴무·설비 운전의 원인으로 확정한 것은 아니다.*

근거: [시간대별 평균](EDA/tables/daily_pattern_restart.csv)

**관련 Python 코드**

- [EDA/scripts/eda_daily_pattern.py](EDA/scripts/eda_daily_pattern.py): 평일·주말의 시간대별 생산량·전력 평균을 원본과 대조하고, 이 절의 선그래프를 생성한다. 선 색·축·범례는 `main()`의 그림 생성 부분에서 수정할 수 있다.
- [EDA/scripts/data_overview.py](EDA/scripts/data_overview.py): 원본 품질을 점검하고 위 코드가 읽는 `hourly_overview.csv`를 생성한다.
- [EDA/scripts/eda_figures.py](EDA/scripts/eda_figures.py): 공통 글꼴·색상·이미지 저장 설정이다.

## 2.2 전력의 하루 패턴은 날짜와 월에 따라 어떻게 달라지는가?

2.1의 평균 선만으로는 같은 패턴이 매일 나타나는지 알기 어렵다. 같은 241일의 시간별 평균 전력을 날짜별로 비교하고, 평일·주말을 구분한 상태에서 월별 수준도 살펴봤다.

![평일과 주말의 시간대별 전력 분포](EDA/figures/daily_repetition/04_daily_pattern_simple.png)

이미지 파일: [EDA/figures/daily_repetition/04_daily_pattern_simple.png](EDA/figures/daily_repetition/04_daily_pattern_simple.png). 생성 코드: `EDA/scripts/plot_daily_repetition_simple.py`의 `main()`.

*실선은 날짜별 값의 중앙값, 점선은 산술평균이다. 음영은 각 시간대에서 날짜별 전력값의 10 ~ 90백분위 범위이며 신뢰구간이 아니다. 평일 171일, 주말 70일을 각각 요약했다.*

**주말의 낮은 평균이 모든 날짜의 낮은 전력을 뜻하지는 않았다.** 주말 08시 이후에는 중앙값이 낮게 유지되지만 평균은 그보다 높았고, 음영의 위쪽도 높은 값까지 이어졌다. 같은 주말 시간대에 낮은 전력과 높은 전력의 날짜가 함께 포함돼 있어, 평균 선 하나만으로 날짜별 차이를 설명하기 어렵다.

**12시 하락·13시 상승은 평일 대부분에서 나타났지만, 모든 날짜의 공통 규칙은 아니었다.** 같은 날짜의 12시 전력이 11시보다 낮고, 13시 전력이 12시보다 높은지를 확인했다. 이 조건에 해당하는 날짜는 평일 147/171일(86.0%), 주말 13/70일(18.6%)이었다. 작은 변화도 포함한 방향 비교이며, 고전력이나 피크를 분류하는 기준은 아니다.

날짜별 11→12시 변화의 중앙값은 평일 −64, 주말 −1이었고, 12→13시 변화의 중앙값은 각각 +61, 0이었다. 평일에서는 변화 폭도 컸다. 다만 하락 후 상승한 평일 147일 중 13시 전력이 11시 수준 이상으로 올라온 날짜는 51일이어서, 상승을 이전 수준으로의 완전한 회복으로 해석하지 않았다.

**예외는 생산량 기록과 함께 살펴볼 필요가 있었다.** 위 방향에 해당하지 않은 평일 24일 중 20일은 하루 동안 생산량이 모두 0이었다. 하루에 한 번이라도 양수 생산량이 기록된 평일에서는 145/149일(97.3%)에 하락 후 상승이 나타났다. 반면 주말은 양수 생산량이 있는 날짜 6/32일(18.8%), 없는 날짜 7/38일(18.4%)로 비슷했다. 생산량 기록의 유무만으로 평일·주말의 차이를 모두 설명할 수는 없었다.

### 월별 평균 전력 비교

날짜별 차이가 확인되어 전체 기간의 평균을 월별로 나눴다. 2.1과 동일한 평균 수준을 비교하기 위해 산술평균을 사용했다. 각 칸은 해당 월·평일 또는 주말·같은 시간대에 기록된 CSV ‘평균’ 값을 다시 평균한 것이다. 생산량이 0인 날짜도 포함했다.

![평일과 주말의 월별 시간대별 평균 전력](EDA/figures/daily_repetition/07_monthly_power_mean_heatmap.png)

이미지 파일: [EDA/figures/daily_repetition/07_monthly_power_mean_heatmap.png](EDA/figures/daily_repetition/07_monthly_power_mean_heatmap.png). 생성 코드: `EDA/scripts/plot_monthly_power_heatmap.py`의 `main()`이며 `--statistic mean`을 사용한다.

*두 패널은 같은 0 ~ 210 색 척도를 사용한다. 각 패널의 행은 1 ~ 8월, 열은 0 ~ 23시다. 월별 평일은 20 ~ 23일, 주말은 8 ~ 10일이며 공휴일도 해당 요일에 포함했다.*

**평일의 시간대별 형태가 비슷해도 전력 수준은 월마다 달랐다.** 예를 들어 평일 08시 평균은 7월 185.05, 8월 142.68이었다. 주말의 같은 시간 평균은 각각 23.78, 21.33이었다. 따라서 7·8월 주말의 낮은 색은 실제 기록의 평균이며, 평일의 높은 전력 기록이나 월 안의 최댓값과 모순되지 않는다.

8월 평일 08시 평균이 낮아진 데에는 날짜 구성도 포함돼 있었다. 8월 평일 22일 중 5일은 하루 동안 생산량이 모두 0이었고, 이 날짜들의 08시 평균 전력은 21.00이었다. 양수 생산량이 있는 나머지 17일의 08시 평균은 178.47이었다. 7월 평일 20일에는 모두 양수 생산량 기록이 있었다. 낮은 날짜를 제외한 결과는 구성 차이를 확인하는 보조 비교이며, 히트맵의 전체 평균을 대체하지 않는다.

### 실제 시간순 전력과 낮은 구간의 지속

월별 평균에는 전력이 낮게 이어진 기간과 다시 높아진 시점이 섞여 있다. 이를 구분하기 위해 같은 5,784시간을 실제 날짜·시각 순서로 연결하고, 시간 평균과 시간 최대값을 함께 표시했다. 이 그림의 평균은 네 15분 값의 **반올림 전 산술평균**, 최대값은 네 값의 최댓값이다. 앞의 CSV 정수 ‘평균’을 집계한 그림과 계산 정밀도를 구분했다. 정상 시각이 없는 2월 1일·7월 15일은 선을 끊었고, 빈 구간을 보간하거나 평활하지 않았다.

![1월부터 8월까지 실제 시간순 평균·최대 전력](EDA/../Modeling/figures/observed_power_monthly_2021_01_08.png)

이미지 파일: [Modeling/figures/observed_power_monthly_2021_01_08.png](EDA/../Modeling/figures/observed_power_monthly_2021_01_08.png). 생성 코드: [Modeling/scripts/plot_observed_power_timeline.py](EDA/../Modeling/scripts/plot_observed_power_timeline.py)의 `main()` 내 월별 `level` 패널. 기존에 검증한 관측 그림을 재사용했다.

*여덟 패널은 1 ~ 8월이며 가로축은 실제 날짜, 세로축은 전력값이다. 파란 선은 시간 평균, 주황 선은 시간 최대값으로 모든 월에 같은 세로축 범위를 적용했다. 두 선의 간격은 같은 시간 안의 평균·최대 차이이며 직전 시간 대비 변화량이 아니다.*

**전력은 낮은 수준이 이어지는 기간과 그보다 높은 값이 반복되는 기간을 오갔다.** 1월에는 한 날짜 안에서도 큰 상승·하락이 나타났고, 8월 초에는 낮은 값이 길게 이어졌다. 그림의 선이 빽빽한 정도만으로 월별 급변 빈도를 판단하지 않고, 기존 2.4의 CSV ‘평균’ 20 ~ 26 구간을 전체 기간에 적용해 지속시간을 확인했다.

| 기록 구분 | 평일 | 주말 | 전체 |
|---|---:|---:|---:|
| 네 전력값 모두 0 | 0시간 | 17시간 | 17시간 |
| 양수이며 CSV ‘평균’이 20 미만 | 0시간 | 7시간 | 7시간 |
| CSV ‘평균’이 20 ~ 26 | 747시간 | 1,189시간 | 1,936시간 |
| CSV ‘평균’이 26 초과 | 3,357시간 | 467시간 | 3,824시간 |
| 합계 | 4,104시간 | 1,680시간 | 5,784시간 |

*20 ~ 26은 평일 분포에서 발견한 탐색적 구간을 그대로 재사용한 것이다. 주말까지 포함하면 양수이면서 20 미만인 7시간이 있어 별도로 표시했다. 26 초과를 고전력·피크 또는 특정 설비의 운전 상태로 정의한 것은 아니다.*

20 ~ 26이 정확히 한 시간 간격으로 이어진 구간은 71개였고, 각 구간의 지속시간은 중앙값 24시간, 최소 1시간, 최대 223시간이었다. 가장 긴 구간은 **7월 31일 00시부터 8월 9일 06시까지**였으며 이 기간의 생산량은 모두 0이었다. 모든 구간의 앞뒤 정상 기록을 확인했으므로 자료 시작·종료나 결측 때문에 잘린 지속시간은 없었다. 다만 19와 20 사이를 오가도 구간이 끊기는 정의이므로, 71개를 실제 가동·정지 횟수로 해석하지 않았다. 날짜별 반복 배열도 있어 각 구간을 독립적인 실험으로 간주하지 않는다.

### 전력 0 구간과 다시 높아지는 과정

![17시간 연속 전력 0과 전후 시간순 변화](EDA/../Modeling/figures/observed_power_zero_interval.png)

이미지 파일: [Modeling/figures/observed_power_zero_interval.png](EDA/../Modeling/figures/observed_power_zero_interval.png). 생성 코드: [Modeling/scripts/plot_observed_power_timeline.py](EDA/../Modeling/scripts/plot_observed_power_timeline.py)의 `main()` 내 `zoom` 패널.

*위 패널은 평균·최대 전력, 아래 패널은 각각의 직전 시간 대비 차이다. 파란 선은 평균, 주황 선은 최대이며 회색 음영은 네 값이 모두 0인 17시간이다. 0 구간 앞뒤 24시간을 함께 표시했다.*

**전력 0은 8월 28일 18시부터 29일 10시까지 이어진 하나의 연속 사례였다.** 17개의 서로 독립적인 사건이 아니다. 진입 직전 시간 평균은 22.00, 최대값은 47이었고, 0 구간 직후인 29일 11시에는 평균 16.50·최대값 41이 기록됐다. 바로 100이나 200으로 상승한 것이 아니라 낮은 값이 이어진 뒤 다시 크게 높아졌다. 낮은 시간 평균에 최대값 47이 포함될 수 있다는 점도 평균과 최대를 함께 확인할 이유다.

따라서 생산량 0, 평균 20 ~ 26, 네 전력값 모두 0을 같은 상태로 묶지 않았다. 실제 정지·전원 차단·계측 이상 중 어떤 원인인지는 현 자료에서 미식별이며, 이 한 사례로 0 발생의 일반 규칙이나 예측 가능성을 주장하지 않는다. 기록은 보존했다. 반면 반복되는 전력 구간의 진입·이탈은 시간 연결을 통해 비교할 수 있으므로, **전력 자체의 전환이 생산량 전환과 얼마나 겹치는지**를 Analysis 3.1에서 확인한다.

이 비교는 하루 패턴이 많은 평일에서 반복되지만 날짜·월에 따라 수준과 지속시간이 달라진다는 것을 보여준다. 달력만으로 실제 운전 상태나 날씨의 영향을 확정할 수는 없다. 다음 절에서는 생산량과 날씨의 관계를 함께 살펴보고, 전체 관계와 월별 관계가 어떻게 다른지 비교한다.

시간순 분석 근거: [구간별 시간 수](EDA/../Analysis/tables/a06_power_state_transitions/state_counts.csv) · [연속 구간과 지속시간](EDA/../Analysis/tables/a06_power_state_transitions/episodes.csv) · [0 구간 전후 값](EDA/../Modeling/tables/observed_timeline/zero_interval.csv) · [원자료 전수 대조·정의](EDA/../Analysis/tables/a06_power_state_transitions/summary.json).

근거: [날짜별 전력·변화](EDA/tables/daily_repetition/daily_values.csv) · [평일·주말 집계](EDA/tables/daily_repetition/group_summary.csv) · [생산 기록별 비교](EDA/tables/daily_repetition/production_context.csv) · [월·요일·시간별 평균과 표본 수](EDA/tables/daily_repetition/independent_monthly_cells.csv) · [7·8월 생산 기록 구성](EDA/tables/daily_repetition/july_august_production_composition.csv)

**관련 Python 코드**

- [EDA/scripts/eda_daily_repetition.py](EDA/scripts/eda_daily_repetition.py): 날짜별 11→12시·12→13시 변화, 평일·주말의 빈도, 생산량 기록별 차이와 시간대별 분포를 계산한다.
- [EDA/scripts/plot_daily_repetition_simple.py](EDA/scripts/plot_daily_repetition_simple.py): 위 결과표로 평균·중앙값·10 ~ 90백분위 범위의 선그래프를 생성한다. `main()`의 선·음영·축 설정에서 그림을 수정할 수 있다.
- [EDA/scripts/verify_monthly_power_heatmap.py](EDA/scripts/verify_monthly_power_heatmap.py): 원본에서 월·평일/주말·시간별 평균과 중앙값을 직접 계산해 결과표를 검증하고, 7·8월 생산량 기록 구성도 비교한다.
- [EDA/scripts/plot_monthly_power_heatmap.py](EDA/scripts/plot_monthly_power_heatmap.py): 월별 평균 히트맵을 생성한다. `main()`의 색상·색 척도·축 설정에서 그림을 수정할 수 있다. 기본값과 본문 그림은 `--statistic mean`이다.
- [Analysis/scripts/a06_power_state_transitions.py](EDA/../Analysis/scripts/a06_power_state_transitions.py): `load()`에서 원본의 정상 1 ~ 8월 기록과 정확한 시간 연결을 구성하고, `episodes()`에서 연속 구간을 집계한다. `independent_verify()`가 원본을 별도로 순회해 5,784시간과 5,781개 시간차를 대조한다. 이 절과 Analysis 3.1이 같은 집계를 사용한다.
- [Modeling/scripts/plot_observed_power_timeline.py](EDA/../Modeling/scripts/plot_observed_power_timeline.py): 검증된 전체 관측 기록으로 월별 선그래프와 0 구간 확대 그림을 생성한다. 학습·검증 표본 제한이나 모델 학습 없이 전체 5,784시간을 사용하며, `main()`에서 선·축·패널을 수정한다.

## 2.3 생산량·날씨와 전력의 관계

2.1에서는 생산량과 전력의 하루 패턴이 완전히 일치하지 않았고, 2.2에서는 같은 시간대라도 월과 날짜 구성에 따라 전력 수준이 달랐다. 이번에는 생산량·날씨가 달라질 때 전력도 함께 달라지는지 살펴봤다. 날씨 변수끼리, 날씨와 생산량 사이의 관계도 함께 비교했다.

**이 절의 상관분석에는 Spearman 순위상관을 사용했다.** 생산량 기록의 43.34%, 강수량 기록의 75.06%가 0이었고, 두 변수는 일부 큰 값 쪽으로 분포가 치우쳐 있었다. 이러한 분포를 고려하면서, 생산량·날씨의 상대적인 크기가 전력의 크기와 함께 움직이는 경향을 확인하고자 순위상관을 선택했다.

생산량·날씨는 수치형 자료이며, 각 값을 작은 값부터 순위로 바꿔 계산했다. 같은 값에는 평균 순위를 부여했다. 모든 정상 기록과 극단값을 포함했고, 결측은 비교하는 두 변수에 필요한 만큼만 제외했다. 상관계수는 관측 관계의 크기와 방향을 설명하며, 시간별 기록의 반복·연속성을 고려하지 않은 p값으로 유의성을 주장하지 않았다.

![생산량 날씨 변수 사이 및 평균 전력과의 Spearman 상관](EDA/figures/restart_02_variable_relations.png)

이미지 파일: [EDA/figures/restart_02_variable_relations.png](EDA/figures/restart_02_variable_relations.png). 생성 코드: `EDA/scripts/eda_variable_relations.py`의 `figures()`.

*숫자는 두 변수의 순위상관이다. +1에 가까울수록 함께 높아지는 관계, −1에 가까울수록 한쪽이 높을 때 다른 쪽은 낮아지는 관계가 강하다. 같은 변수의 중복 비교는 생략했다.*

**생산량과 전력은 함께 높아지는 관계가 있었다.** 두 변수의 순위상관은 0.752였다. 다만 2.1·2.2의 평일·주말, 시간대와 월별 차이도 포함된 전체 관계이므로, 생산량 증가가 전력을 얼마나 증가시키는지 나타내는 수치는 아니다.

**날씨와 전력의 관계는 전체 기간을 합치면 작았다.** 평균 전력과의 순위상관은 기온 0.058, 풍속 0.137, 습도 −0.119, 강수량 −0.014였다. 한편 날씨 변수끼리는 풍속·습도가 −0.436, 습도·강수량이 0.457이었다. 날씨 조건들은 함께 달라지므로 한 변수만 따로 보고 전력의 변화를 설명하기 어렵다.

날씨와 생산량 사이의 전체 순위상관도 기온 0.114, 풍속 0.089, 습도 −0.085, 강수량 −0.006으로 작았다. 전체 그림에서는 생산량·전력의 관계와 날씨 변수끼리의 관계가 상대적으로 두드러졌지만, 이것만으로 특정 시기나 조건 안에서도 같은 관계가 유지된다고 볼 수는 없다.

**기온과 전력의 관계는 월에 따라 방향이 달랐다.** 기온의 전체 상관이 작다는 이유만으로 관계가 없다고 판단하지 않고, 월별로 같은 비교를 이어갔다. 2월에는 −0.356, 4월에는 0.383으로 서로 다른 방향의 관계가 나타났다.

![월별 생산량 및 기상 변수와 평균 전력의 Spearman 상관](EDA/figures/restart_02_monthly_relations.png)

이미지 파일: [EDA/figures/restart_02_monthly_relations.png](EDA/figures/restart_02_monthly_relations.png). 생성 코드: `EDA/scripts/eda_variable_relations.py`의 `figures()`.

*각 칸은 해당 월의 변수와 평균 전력 사이의 순위상관이다. 모든 칸에 같은 −1 ~ +1 색 척도를 사용했다. 월마다 정상 날짜 수와 결측 수가 달라 표본 수는 결과표에 함께 기록했다.*

이 차이를 이해하기 위해 생산 기록이 함께 달라진 8월을 살펴봤다. 8월 전체의 기온·전력 상관은 −0.076이었지만, 생산량이 양수인 383시간에서는 0.599였다. 8월 2 ~ 8일은 생산량이 모두 0이고 평균 기온은 27.81이었는데, 평균 전력은 21.56으로 낮았다. **높은 기온과 낮은 전력이 함께 기록된 기간이 전체 관계에 포함된 것이다.**

Spearman 상관이 작아도, 중간 구간에서만 전력이 높아지는 관계처럼 방향이 바뀌는 패턴은 남을 수 있다. 따라서 전체 상관만으로 날씨를 전력과 무관하다고 제외하거나, 기온이 전력을 높이는 원인이라고 단정할 수 없다. 생산 여부와 시기를 함께 구분해야 하며, 높은 전력값이 발생하는 조건은 별도로 살펴볼 필요가 있다.

근거: [분포·정규성 진단](EDA/tables/restart_relations_distribution.csv) · [전체 관계](EDA/tables/restart_relations_spearman.csv) · [월별 관계·표본 수](EDA/tables/restart_relations_monthly.csv) · [8월 생산 여부별 비교](EDA/tables/restart_relations_august_context.csv) · [8월 2 ~ 8일 기록](EDA/tables/restart_relations_august_week.csv)



**관련 Python 코드**

- [EDA/scripts/eda_variable_relations.py](EDA/scripts/eda_variable_relations.py): `diagnose()`에서 분포·결측·동률을 점검하고, `relations()`에서 전체·월별 Spearman 상관과 8월 생산 기록별 비교를 계산한다. `figures()`는 이 절의 두 상관 히트맵을 생성하며 색 척도·축·숫자 표시를 수정할 수 있다. 실행 옵션은 `--diagnose --figures`다.
- [EDA/scripts/eda_figures.py](EDA/scripts/eda_figures.py): 두 히트맵에 적용하는 공통 글꼴과 이미지 저장 설정이다.

## 2.4 평일의 전력 분포와 낮은 전력이 나타나는 시간

2.2에서는 같은 평일 시간대라도 날짜의 생산 기록 구성에 따라 전력 수준이 달랐고, 2.3에서는 낮은 전력이 기록된 기간이 날씨와의 전체 관계에 포함돼 있었다. 이번에는 평일 안에서도 서로 다른 전력 수준이 나타나는지 분포를 확인하고, 낮은 전력이 하루 중 어느 시간에 나타나는지 살펴봤다. 2021년 1 ~ 8월의 정상 평일 171일·4,104시간을 모두 사용했다. 평일은 월 ~ 금요일이며 공휴일도 해당 요일에 포함했다. 전력은 앞 절과 같은 CSV ‘평균’ 값이다.

![평일 시간별 평균 전력의 분포](EDA/figures/weekday_power_levels.png)

이미지 파일: [EDA/figures/weekday_power_levels.png](EDA/figures/weekday_power_levels.png). 생성 코드: `EDA/scripts/eda_weekday_power_levels.py`의 `plot_distribution()`.

*가로축은 시간별 평균 전력, 세로축은 해당 기록 수다. 막대는 전력값을 5 간격으로 묶었다. 주황색은 20 ~ 26, 파란색은 49 ~ 208의 기록이며 모든 평일 기록을 포함했다.*

그림에서는 20대 초반에 좁게 모인 주황 막대와, 그 오른쪽의 빈 구간을 지나 넓게 퍼진 파란 막대가 구분된다. 따라서 평일의 전력 분포를 한 가지 평균 수준으로 요약하기보다 낮은 구간의 존재와 그 밖의 변동을 함께 살펴봤다. 빈 구간의 정확한 범위와 기록 수는 막대 폭으로 추정하지 않고 전력값별 빈도표로 확인했다.

**평일에도 20 ~ 26의 낮은 전력 구간이 뚜렷하게 나타났다.** 이 구간에는 747시간(18.20%)이 모였고, 27 ~ 48에는 기록이 없었다. 그다음 관측값은 49였다. 따라서 이번 EDA에서는 실제 분포의 빈 구간을 근거로 20 ~ 26을 ‘낮은 전력 구간’으로 구분했다. 상위·하위 일정 비율을 맞춘 기준이 아니며, 설비의 가동·정지나 전력 피크를 판정하는 기준도 아니다.

**평일 171일의 24시간을 각각 확인했더니, 낮은 전력이 나타나는 시간 조합은 아래 네 형태로만 관측됐다.** 특정 사례를 골라 일반화한 것이 아니라 날짜별로 20 ~ 26에 해당하는 시각을 모두 추출한 뒤 동일한 조합끼리 집계했다.

| 평일의 전력 형태 | 00 ~ 06시 | 07시 | 08 ~ 23시 | 날짜 수 |
|---|---|---|---|---:|
| 07시부터 낮은 구간을 벗어남 | 20 ~ 26 | 49 이상 | 49 이상 | 29일 |
| 08시부터 낮은 구간에 들어감 | 49 이상 | 49 이상 | 20 ~ 26 | 7일 |
| 하루 내내 낮은 구간 | 20 ~ 26 | 20 ~ 26 | 20 ~ 26 | 18일 |
| 하루 내내 49 이상 | 49 이상 | 49 이상 | 49 이상 | 117일 |
| **합계** | | | | **171일** |

*‘49 이상’의 실제 범위는 49 ~ 208이다. 같은 구간에 속해도 시간별 전력값은 달라질 수 있으므로, 하루 종일 49 이상인 날의 전력이 일정했다는 뜻은 아니다. 표의 네 형태는 이번 자료에서 낮은 전력이 나타난 시각의 조합을 요약한 것이다.*

낮은 전력은 평일 54일에서 관측됐다. 이 중 18일은 하루 내내 낮았고, 나머지 36일은 00 ~ 06시 또는 08 ~ 23시에 연속해서 낮은 값을 보였다. 이 결과는 평일·주말의 평균 차이뿐 아니라 평일 안에도 다른 전력 수준의 기록이 함께 존재함을 보여준다.

**낮은 전력을 회사 휴무와 동일하게 해석할 수는 없다.** 낮은 747시간 중 92시간에는 해당 시간의 생산량이 양수였다. 실제 휴무나 설비 운전의 원인은 현 자료에서 식별할 수 없다. 따라서 이 기록을 오류로 삭제하거나 생산량 0만으로 같은 상태를 정의하지 않았다.

평일의 날씨·전력 관계를 살펴볼 때는 이러한 전력 수준과 시간 구성이 함께 포함돼 있다는 점을 고려해야 한다. 생산 기록과 시간대를 고려한 날씨 비교는 Analysis에서 다루고, 다음 절에서는 지금까지 사용한 시간 평균을 구성하는 네 구간의 전력이 어떻게 다른지 확인한다.

근거: [전력값별 빈도](EDA/tables/weekday_power_levels/power_frequency.csv) · [날짜별 전체 시간 조합](EDA/tables/weekday_power_levels/daily_hour_patterns.csv) · [네 형태의 집계](EDA/tables/weekday_power_levels/pattern_summary.csv) · [생산 기록 대조](EDA/tables/weekday_power_levels/production_context.csv)

**관련 Python 코드**

- [EDA/scripts/eda_weekday_power_levels.py](EDA/scripts/eda_weekday_power_levels.py): `main()`에서 평일 전력 분포의 빈 구간과 수치를 확인하고, `summarize_patterns()`에서 171일의 정확한 시간 조합을 전수 집계한다. `plot_distribution()`이 이 절의 히스토그램을 생성하며 막대 간격·색·축·범례를 수정할 수 있다.
- [EDA/scripts/eda_variable_relations.py](EDA/scripts/eda_variable_relations.py): `source_frame()`에서 동일한 원본과 1 ~ 8월 정상 시간 기록을 읽는다.
- [EDA/scripts/eda_figures.py](EDA/scripts/eda_figures.py): 공통 글꼴·색상·이미지 저장 설정이다.

## 2.5 시간대별 15분 구간의 전력 패턴

앞 절들은 시간 평균으로 전력 수준을 비교했다. 이번에는 같은 시간 안의 `15분·30분·45분·60분` 값이 구간 위치에 따라 반복적으로 높거나 낮은지 살펴봤다. 2021년 1 ~ 8월 정상 241일·5,784시간의 네 값 23,136개를 모두 사용하고, 평일 171일과 주말 70일을 구분했다. 낮은 전력과 생산량 0인 기록도 포함했으며 특정 날짜를 골라 대표 사례로 삼지 않았다.

![평일과 주말의 시간대별 15분 구간 평균 대비 차이](EDA/figures/slot_time_patterns.png)

이미지 파일: [EDA/figures/slot_time_patterns.png](EDA/figures/slot_time_patterns.png). 생성 코드: `EDA/scripts/eda_slot_time_patterns.py`의 `plot_centered_heatmap()`.

*왼쪽은 평일, 오른쪽은 주말이다. 행은 00 ~ 23시, 열은 같은 시간 안의 네 전력 구간이다. 공휴일은 해당 요일에 포함했다. 두 패널은 같은 −23 ~ +23 색 척도를 사용한다.*

**숫자는 각 구간이 같은 시간의 평균보다 얼마나 높거나 낮은지를 나타낸다.** 날짜별로 각 구간 전력에서 그 시간의 네 값 산술평균을 뺀 뒤, 같은 시간대의 모든 평일 또는 주말에서 그 차이를 평균했다. 빨간색은 평균보다 높은 구간, 파란색은 낮은 구간이다. 기준 평균은 CSV의 정수 반올림 전 값이며, 차이는 전력 기록값과 같은 단위다.

예를 들어 평일 10시의 네 구간 평균은 120.84·149.14·151.96·151.18이고, 이 네 값의 평균은 143.28이다. 첫 구간의 `120.84−143.28=−22.44`가 히트맵에 **−22.4**로 표시된다. 이는 첫 구간이 10시 평균보다 평균적으로 22.4 낮다는 뜻이다. 이전 시각 대비 변화나 백분율을 나타내는 수치는 아니다.

**평일에는 특정 시간대의 구간 차이가 두드러졌다.** 07시는 앞 두 구간이 낮고 뒤 두 구간이 높았다. 10시는 첫 구간, 12시는 가운데 두 구간이 낮았고, 17시는 두 번째 구간이 특히 낮고 마지막 구간이 높았다. 반면 04시의 차이는 +0.9·+0.3·+0.3·−1.5로 모두 0 근처였다. 하루의 모든 시간대에 하나의 구간 규칙이 나타난 것은 아니다.

평균 그림의 형태가 여러 날짜에서도 나타났는지 확인하기 위해 모든 시간대에 같은 배열 조건을 적용했다. 아래는 차이가 두드러진 시간대의 평일·주말 비교다. 같은 값은 엄격한 부등호 조건에 포함하지 않았으며, 분모는 각 집단의 모든 날짜다.

| 시간대 | 확인한 조건 | 평일 | 주말 |
|---|---|---:|---:|
| 07시 | 60분 값 > 15분 값 | 160/171일(93.6%) | 26/70일(37.1%) |
| 10시 | 15분 값이 나머지 세 값보다 모두 낮음 | 145/171일(84.8%) | 13/70일(18.6%) |
| 12시 | 30분·45분 값이 15분·60분 값보다 모두 낮음 | 129/171일(75.4%) | 10/70일(14.3%) |
| 17시 | 45분·60분 값이 15분·30분 값보다 모두 높음 | 145/171일(84.8%) | 11/70일(15.7%) |

*07시의 마지막 값이 첫 값보다 높다는 조건은 네 값이 모두 차례로 상승한다는 뜻이 아니다. 평일에 `15분<30분<45분<60분`이 성립한 날짜는 79/171일(46.2%)이었다. 날짜가 달라도 같은 배열이 반복돼 이 비율을 독립적인 실험의 결과로 해석하지 않았다. 중복 배열을 한 번씩만 센 보조 점검에서도 위 평일 형태가 다수였으며 자세한 수치는 결과표에 수록했다.*

**주말은 대부분 시간대에서 평균적인 구간 차이가 작았고, 평일의 형태도 적게 나타났다.** 예를 들어 12시 네 칸은 −1.6 ~ +2.0에 머물렀다. 다만 07시는 첫 칸 +8.3, 마지막 칸 −11.3으로 평균상 하락이 보였다. 실제 마지막 값이 첫 값보다 낮은 날은 37/70일(52.9%)이었고, 끝−처음 차이의 평균은 −19.67, 중앙값은 −1이었다. 평균의 하락 폭만으로 큰 하락이 대부분의 주말에 반복됐다고 판단하지 않았다. 옅은 색 역시 서로 다른 날짜의 변화가 상쇄될 수 있어 매일 변동이 작았다는 뜻은 아니다.

이 비교에서 특정 평일 시간대의 구간 위치별 차이가 여러 날짜에서 확인됐다. 시간 평균은 하루의 전력 수준을 요약하지만 같은 시간 안의 높낮이와 변화 순서를 모두 담지는 못한다. 구간별 기록을 고려할 근거는 확인했으며, 이 정보가 예측에 얼마나 도움이 되는지는 이후 Analysis·Modeling에서 평가한다. 현재 기록만으로 설비 운전 원인이나 향후 모든 날짜에 적용되는 규칙을 확정하지 않았다.

근거: [구간별 평균·평균 대비 차이](EDA/tables/slot_time_patterns/hour_slot_summary.csv) · [구간 사이 변화 방향](EDA/tables/slot_time_patterns/within_hour_directions.csv) · [배열 조건별 날짜 수·중복 점검](EDA/tables/slot_time_patterns/within_hour_profiles.csv)

**관련 Python 코드**

- [EDA/scripts/eda_slot_time_patterns.py](EDA/scripts/eda_slot_time_patterns.py): `load_data()`에서 전체 정상 기록을 읽고, `summarize()`에서 시간대·구간별 평균과 날짜 빈도를 계산한다. `verify_independently()`는 원본 CSV를 별도로 순회해 수치를 대조한다. 히트맵의 색·축·숫자·패널 크기는 `plot_centered_heatmap()`에서 수정할 수 있다. 집계를 재사용해 히트맵만 수정하려면 `--figures-only`로 실행한다.
- [EDA/scripts/eda_figures.py](EDA/scripts/eda_figures.py): 공통 글꼴·색상 설정이다.

### 높은 값이 나타나는 구간 수

앞의 히트맵은 같은 시간대의 평균적인 구간 차이를 보여준다. 이어서 높은 값이 관측된 시간 안에서도 한 구간만 높은 경우와 여러 구간이 함께 높은 경우가 얼마나 나타나는지 확인했다. 시간 최대값이 높다는 사실만으로 네 구간 모두 높은 상태였다고 해석할 수 있는지 살펴보기 위한 비교다.

1월 정상 시간의 네 값 중 최대값을 모아 95분위수를 계산한 뒤, 그 값인 **185를 비교 기준으로 고정**했다. 분위수는 선형 보간으로 계산했고 185와 같은 값도 포함했다. 이 기준을 1 ~ 8월 정상 5,784시간에 적용하자, 네 구간 중 적어도 하나가 185 이상인 시간은 360시간이었다. 각 시간에서 기준 이상인 구간 수를 세어 아래와 같이 구분했다.

| 같은 시간 안에서 185 이상인 구간 수 | 시간 수 | 해당 360시간 중 비율 |
|---|---:|---:|
| 1개 | 136 | 37.8% |
| 2개 | 114 | 31.7% |
| 3개 | 82 | 22.8% |
| 4개 | 28 | 7.8% |
| **합계** | **360** | **100.0%** |

*각 행은 서로 겹치지 않는 시간 단위 집계다. 비율은 소수 첫째 자리에서 반올림해 개별 비율의 합이 100.0%와 다를 수 있다. 185는 높은 기록의 형태를 비교하기 위한 기술적 기준이며, 실제 현장의 위험·경보·계약전력 기준이 아니다.*

**한 구간만 높은 경우는 37.8%였고, 네 구간 모두 높은 경우는 7.8%였다.** 높은 최대값이 나타난 시간에도 기준 이상인 구간 수가 달랐다. 따라서 최대값은 해당 시간의 가장 높은 기록을 나타내지만, 높은 값이 네 구간에 얼마나 넓게 나타났는지까지 설명하지는 못한다. 시간 평균·최대값과 네 구간의 기록을 함께 살펴볼 근거다.

이 표는 관측된 형태의 빈도이며 독립적인 사건 발생확률이나 미래 예측 정확도를 뜻하지 않는다. 날짜별 반복 배열을 포함했고, 각 15분 열의 높은 기록이 실제로 몇 분 동안 유지됐는지는 현 자료에서 식별되지 않는다. 구간별 기록을 예측에 활용했을 때의 성능은 별도의 시간순 비교로 판단해야 한다. 이 추가 비교에는 새 이미지를 사용하지 않았다.

근거: [시간별 형태·기준 이상 구간 수](EDA/tables/p01_peak_shapes/hourly_shapes.csv) · [구간 수별·월별 집계](EDA/tables/p01_peak_shapes/shape_counts.csv) · [비교 조건](EDA/tables/p01_peak_shapes/contract.json) · [기준값·전체 집계](EDA/tables/p01_peak_shapes/run.json) · [원자료 독립 대조](EDA/tables/p01_peak_shapes/verification.json)

**관련 Python 코드**

- [EDA/scripts/p01_peak_shapes.py](EDA/scripts/p01_peak_shapes.py): `main()`에서 1월 시간 최대값의 95분위수와 시간별 기준 이상 구간 수를 계산한다. `shape_counts.csv`는 월·최대 위치·기준 이상 구간 수별 시간 수이며, 본문 표는 `high_slots`별로 합산한 값이다. 같은 함수의 원본 CSV 별도 순회가 시간별 최대·평균·범위·최대 동률·기준 이상 구간 수를 독립 대조한다. 기존 완료 결과를 덮어쓰지 않도록 `run.json`이 있으면 실행을 중단하며, 최초 실행은 프로젝트 루트에서 `EDA/scripts/p01_peak_shapes.py`로 수행한다.

### 2.6 EDA 코드 실행

평일·주말 하루 패턴, 날짜별 반복·월별 평균, 생산·날씨 관계, 평일 전력 분포, 15분 구간의 중심화 패턴을 순서대로 계산한다. 원고의 관측 타임라인은 같은 정상 시간으로 다시 생성한다. 이미 검증한 보조 요약표는 함께 대조한다.

In [ ]:
with computation_paths():
    for name in ['eda_daily_pattern','eda_daily_repetition','plot_daily_repetition_simple','plot_monthly_power_heatmap','verify_monthly_power_heatmap']:
        execute_step(name,MODULES[name].main)
    relation=MODULES['eda_variable_relations']
    execute_step('eda_variable_relations_diagnose',lambda:relation.diagnose(relation.source_frame()))
    execute_step('eda_variable_relations_figures',lambda:relation.figures(relation.source_frame()))
    for name in ['eda_weekday_power_levels','eda_slot_time_patterns','plot_report_daily_patterns','plot_observed_power_timeline']:
        execute_step(name,MODULES[name].main)
for relative in ['EDA/tables/daily_repetition/group_summary.csv','EDA/tables/daily_repetition/month_hour_mean.csv']:
    if (WORK/relative).exists():show_table(relative)

원고에 기록된 결과와 해석을 아래에 보존했다. 다음 실행 셀의 결과는 이번 실행에서 계산하거나 검증한 값이다.

# 3. Analysis

## 3.1 생산량 전환에 따른 시간 사이 전력 변화

EDA에서는 평일 12시 전력 하락과 13시 상승이 147/171일에서 반복됐고, 같은 시간대의 전력 수준도 월과 생산량 기록에 따라 달랐다. 이러한 반복 패턴을 고려해, 생산량이 변할 때 전력의 변화 크기도 달라지는지 비교했다. 2021년 1 ~ 8월 정상 기록 241일 중 정확히 1시간 전 기록과 연결되는 5,781시간을 사용했다. 시간별 전력은 네 15분 값의 평균으로 계산하고, 현재와 직전 평균 전력 차이의 절댓값을 ‘시간 사이 전력 변화 크기’로 정의했다. 상승과 하락이 상쇄되지 않도록 변화의 크기를 비교한 것이다.

**생산량 증가·감소·유지만 나누면 생산량 0인 기록의 영향이 섞였다.** 생산량 유지 2,296시간 중 2,292시간(99.83%)은 직전과 현재 생산량이 모두 0이었다. 양수 생산량이 같은 값으로 유지된 기록은 4시간뿐이어서, 증가·감소와 유지의 차이를 생산량 증감 자체의 영향으로 해석하기 어려웠다. 이에 직전과 현재 생산량이 모두 양수인 경우와, 0에서 양수로 바뀐 경우를 구분했다. 여기서 0→양수는 기록상의 전환을 뜻하며 실제 설비 재가동을 의미하지 않는다.

두 집단의 시간대 구성이 다르면 반복되는 전력 상승·하락이 비교 결과에 섞일 수 있다. 따라서 **같은 월·시간대·평일/주말 안에서 두 집단을 비교**했다. 각 집단에 최소 두 날짜가 있는 공통 46조건을 남기고, 조건별 평균에 두 집단이 합쳐진 시간 수를 동일한 가중치로 적용했다. 비교에는 0→양수 전환 150시간과 양수가 이어진 567시간이 포함됐다. 0→양수 전체 215시간 중 69.8%가 포함된 결과이며, 양수 지속 전체 3,060시간의 평균과는 구분된다.

![생산량 기록 조건별 시간 사이 전력 변화 크기](Analysis/figures/a01_production_transition.png)

이미지 파일: [Analysis/figures/a01_production_transition.png](Analysis/figures/a01_production_transition.png). 생성 코드: [plot_a01_production_changes.py](Analysis/scripts/plot_a01_production_changes.py)의 `main()`.

*막대는 같은 달력 조건으로 구성을 맞춘 평균이다. ‘직전·현재 생산량 모두 양수’와 ‘직전 생산량 0→현재 양수’를 비교하며, 세로축은 현재와 직전 평균 전력의 절대 차이를 나타낸다. 전력값은 원자료의 척도를 사용했다.*

**0→양수 전환에서 평균 전력 변화 크기는 37.92로, 양수가 이어진 기록의 30.21보다 7.71, 약 25.5% 컸다.** 같은 비교에서 한 시간 안의 네 15분 값의 최대·최소 차이는 22.90과 21.37로 차이가 1.53이었다. 따라서 이 전환에서 두드러진 특징은 이전 시간과 현재 시간 사이의 전력 수준 변화였다.

직전 전력 수준에 대비한 변화도 함께 확인했다. 각 시간의 절대 변화 크기를 직전 전력으로 나눈 상대 변화율을 계산한 뒤, 동일한 717시간과 달력 조건으로 비교했다. 평균 상대 변화율은 **0→양수 전환 53.70%, 양수 지속 33.25%로 20.45%p 차이**였다. 작은 분모에만 결과가 의존하는지 확인하기 위해 EDA에서 구분한 낮은 전력 구간 20 ~ 26을 직전 전력에 적용해 보조 점검했다. 이 구간을 제외해도 53.75%와 33.53%로 방향이 유지됐다. 기본 비교에는 직전 전력이 0인 기록이 없었다. 절대 변화와 상대 변화에서 모두 차이가 나타났지만, 상대 변화율 계산이 두 집단의 직전 전력 수준을 동일하게 맞춘 것은 아니다.

월을 하나씩 제외한 비교에서도 절대 변화 크기 차이는 +6.54 ~ +9.20으로 유지됐다. 반복되는 하루 전력 배열의 비중을 낮추거나 같은 날짜 안의 직전 기록만 비교해도 방향은 같았다. 반면 직전·현재 생산량이 모두 양수인 기록에서, 달력 조건을 고려한 생산량 변화 크기와 전력 변화 크기의 순위 관계는 −0.006으로 약했다. 양수→0 전환에서도 0→양수와 같은 추가 차이가 일관되게 나타나지 않아, 모든 생산량 전환으로 결과를 확대하지 않았다.

이 결과는 전력 변동을 설명할 때 **생산량 0→양수 전환을 별도 조건으로 살펴볼 근거**가 된다. 예측 단계에서는 해당 전환 시간에 오차와 과소예측이 집중되는지 먼저 확인하고, 예측 시점에 알 수 있는 과거 생산량·전력과 달력 정보로 이를 줄일 수 있는지 평가한다. 목표 시간의 실제 생산량으로 구분한 전환은 사후 평가에 사용하며, 이를 미리 아는 입력으로 넣지 않는다. 이번 분석에서 확인한 것은 관측된 전력 변화의 차이이며, 예측 개선 효과는 시간순 평가에서 별도로 검증한다.

### 전력 자체의 구간 전환과 생산량 전환의 차이

생산량 전환 비교만으로 전력이 크게 오르내리는 모든 조건을 설명할 수는 없다. EDA 2.2의 실제 시간순 그림에는 낮은 전력이 길게 이어지는 구간과 짧은 시간 안의 큰 상승·하락이 함께 나타났다. EDA 2.4의 평균 20 ~ 26 구간을 기준으로, 전력 자체의 진입·이탈이 언제 나타나고 생산량 전환과 얼마나 겹치는지 확인했다. 기존과 같은 정상 5,784시간 중 정확히 한 시간 전 기록이 있는 5,781시간 전체를 사용했다. 모델링용 표본·기간 분할은 적용하지 않았다.

구간은 CSV의 정수 ‘평균’으로 구분하고, 변화량은 네 값의 반올림 전 평균과 최대값 각각에서 직전 시간 값을 뺐다. 처음에는 20 ~ 26과 나머지를 비교했으나, 주말의 20 미만 양수 기록 7시간이 ‘나머지’에 섞여 있었다. 이를 상승 전환으로 잘못 해석하지 않도록 **전력 0·양수이면서 20 미만·20 ~ 26·26 초과**로 구분했다. 기존 경계 20·26은 유지했으며 새 피크 임계값을 정하거나 관측값을 제외하지 않았다.

![월별 실제 평균·최대 전력의 직전 시간 대비 변화](Analysis/../Modeling/figures/observed_power_delta_monthly_2021_01_08.png)

이미지 파일: [Modeling/figures/observed_power_delta_monthly_2021_01_08.png](Analysis/../Modeling/figures/observed_power_delta_monthly_2021_01_08.png). 생성 코드: [Modeling/scripts/plot_observed_power_timeline.py](Analysis/../Modeling/scripts/plot_observed_power_timeline.py)의 `main()` 내 월별 `delta` 패널.

*가로축은 실제 날짜, 세로축은 직전 시간 대비 차이이며 모든 월의 축 범위가 같다. 파란 선은 평균 변화, 주황 선은 최대값 변화다. 양수는 상승, 음수는 하락이다. 전력 0이 이어지면 이 그림의 변화량도 0이지만, 변화량 0 자체가 전력 0을 뜻하지는 않는다. 정상 기록이 없는 시차는 연결하지 않았다.*

| 직전→현재 전력 구간 | 시간 수 | 평균 전력 변화의 중앙값 | 최대전력 변화의 중앙값 |
|---|---:|---:|---:|
| 20 ~ 26→26 초과 | 64 | +39.50 | +72.00 |
| 26 초과→20 ~ 26 | 64 | −47.75 | −68.00 |
| 20 ~ 26 유지 | 1,865 | 0.00 | 0.00 |
| 26 초과 유지 | 3,757 | −2.00 | −2.00 |

*합계 5,750시간이다. 양수 20 미만 구간에 드나든 13시간과 전력 0의 진입·지속·이탈 18시간은 원자료와 전체 전환표에 보존했다. 각 행은 달력 조건을 맞추지 않은 관측 집계이므로 앞의 46조건·717시간 비교와 수치를 직접 비교하지 않는다. 26 초과는 고전력·피크 판정이 아니다.*

**20 ~ 26에서 26 초과로 올라간 64시간 중 생산량도 0→양수로 바뀐 경우는 5시간(7.8%)뿐이었다.** 반대로 26 초과에서 20 ~ 26으로 내려간 64시간 중 생산량이 양수→0으로 바뀐 경우는 35시간(54.7%)이었다. 전력 구간 전환과 생산량 전환은 일치하지 않았다. 실제로 최대전력 상승이 큰 순서의 5시간과 하락이 큰 순서의 5시간은 모두 직전·현재 생산량이 0이었다. 최대 상승은 3월 21일 08시의 26→208(+182), 최대 하락은 1월 16일 09시의 185→22(−163)였다. 상위 사례는 크기를 확인하는 예시이며 전체 발생 빈도를 대표하지 않는다.

전환 시각에는 달력 구성도 나타났다. 평일의 20 ~ 26→26 초과 37시간 중 29시간은 07시, 나머지 8시간은 00시였다. 그러나 주말에는 여러 시각에 분산돼 평일의 시간대를 전체 규칙으로 확대할 수 없었다. 20 ~ 26에서 상승 전환한 시간과 그 구간을 유지한 시간의 ‘직전 마지막 값−직전 평균’ 중앙값은 모두 0.00이었다. 이 단순 요약에서 뚜렷한 선행 상승을 확인하지 못했지만, 시간대 구성을 맞추거나 다른 입력과 결합한 예측 가치를 평가한 결과는 아니다.

**구간 전환만 분석해도 전체 전력 변동을 설명할 수 있는 것은 아니었다.** 26 초과가 이어진 3,757시간에서는 최대전력 변화가 −88 ~ +107이었고, 전체 5,781시간의 절대 변화량 합계 중 81.1%를 차지했다. 이 비중에는 시간 수가 많은 영향도 포함되며, 전환 한 번의 위험도나 예측오차 비중을 뜻하지 않는다. 낮은 구간을 벗어나는 변화뿐 아니라 이미 전력이 높은 쪽에 있을 때의 추가 상승·하락도 함께 평가해야 한다.

이에 생산량 전환을 별도 평가 조건으로 보겠다는 해석은 유지하되, 전력 변동 전체를 대신하는 조건으로 사용하지 않는다. 다음 예측 평가에서는 전력 구간의 상승 전환·하락 전환·구간 유지와 실제 최대전력 수준을 함께 구분해 과소·과대예측을 확인한다. 마지막 15분 값이 전체 관계에 정보를 더한다는 3.4의 결과가 상승 전환에서도 오류 감소로 이어지는지는 아직 미확인이다. 이 구간은 실제 목표값으로 정한 **사후 오류 분석용 조건**이며 예측 입력이 아니다. 관측 전환의 구성과 생산량과의 차이는 확인했으므로, 같은 분류를 반복 탐색하지 않고 예측오차가 생성된 뒤에 그 결과로 후속 순서를 정한다.

전력 전환 근거: [전체 전환 집계](Analysis/tables/a06_power_state_transitions/transition_summary.csv) · [요일·시간대별 빈도](Analysis/tables/a06_power_state_transitions/transition_hours.csv) · [전환 시점의 원자료 값](Analysis/tables/a06_power_state_transitions/boundary_records.csv) · [상승·하락 상위 사례](Analysis/tables/a06_power_state_transitions/largest_changes.csv) · [정의·검증 기록](Analysis/tables/a06_power_state_transitions/summary.json).

근거: [분석 조건·해석·검증 기록](Analysis/10.03_A01_생산량증감과전력변동_분석기록.md) · [생산량 전환별 비교](Analysis/tables/a01_production_changes/transition_matched_contrasts.csv) · [상대 변화율 보조 점검](Analysis/tables/a01_production_changes/relative_contrasts.csv) · [양수 생산량 내 관계](Analysis/tables/a01_production_changes/positive_production_correlations.csv)

**관련 Python 코드**

- [Analysis/scripts/a01_production_changes.py](Analysis/scripts/a01_production_changes.py): `load()`로 정확한 시간 연결과 전력 지표를 구성하고, `contrast()`·`followup()`으로 같은 달력 조건의 전환 비교와 월·반복 배열 보조 점검을 수행한다.
- [Analysis/scripts/a01_relative_changes.py](Analysis/scripts/a01_relative_changes.py): `main()`에서 직전 전력 대비 상대 변화율과 낮은 전력 구간 제외 결과를 비교한다.
- [Analysis/scripts/plot_a01_production_changes.py](Analysis/scripts/plot_a01_production_changes.py): 저장된 전환 비교표로 위 그림을 생성한다.
- [Analysis/scripts/a06_power_state_transitions.py](Analysis/scripts/a06_power_state_transitions.py): `load()`에서 원자료로 전력 구간과 정확한 시차를 구성하고, `summarize()`에서 변화량·생산 전환의 겹침을 집계한다. `episodes()`는 EDA의 지속시간을 계산하며, `independent_verify()`가 별도 CSV 순회로 원자료와 시차 계산을 대조한다. 모델을 학습하지 않는다.
- [Modeling/scripts/plot_observed_power_timeline.py](Analysis/../Modeling/scripts/plot_observed_power_timeline.py): `main()`의 `delta` 패널에서 월별 평균·최대 변화량 그림을 생성한다. 원자료와 같은 전체 5,784시간을 사용함을 위 독립 검증에서 확인했다.

## 3.2 과거 전력 정보와 생산량 전환의 사전 구분

3.1에서는 생산량이 0에서 양수로 바뀐 시간의 전력 변화가 더 컸지만, 전력 자체의 구간 전환과는 일치하지 않았다. 이번에는 그중 **현재 생산량이 0일 때, 이미 관측한 전력으로 다음 시간의 양수 전환을 미리 구분할 수 있는지** 확인했다. 현재 생산량이 0인 기록에서 다음에도 0인 경우와 다음에 양수로 바뀌는 경우를 비교했다. 이는 생산량 전환에 한정된 정보 가치 검토이며, 전력 상승·하락을 모두 예측하는 분석은 아니다.

입력 정보의 가치는 전환 구분용 분류모형의 시간순 검증으로 비교했다. 1 ~ 4월로 학습하고 5 ~ 6월에서 모형과 전환 판정 기준을 정한 뒤, 7 ~ 8월에 그대로 적용했다. 입력은 ① 다음 시간의 달력 정보(시간대·평일/주말·월), ② 여기에 관측을 마친 시간의 평균 전력 추가, ③ 다시 그 시간의 마지막−첫 15분 값 추가로 나눴다. 세 경우에 같은 트리 분류모형을 사용했고, 다음 시간의 실제 생산량·전력은 입력에 넣지 않았다. 이는 분석에서 사용할 정보의 가치를 비교하는 탐색적 검증이며, 다음 시간 전력값을 예측하는 모형의 성능 평가와는 구분된다.

7 ~ 8월 평가 대상은 현재 생산량이 0인 617시간이었고, 이 중 다음 시간에 양수로 전환한 기록은 52시간(8.43%)이었다.

![과거 전력 추가에 따른 생산량 전환 적중과 오탐](Analysis/figures/a02_transition_detection.png)

이미지 파일: [Analysis/figures/a02_transition_detection.png](Analysis/figures/a02_transition_detection.png). 생성 코드: [a02_transition_review.py](Analysis/scripts/a02_transition_review.py)의 `main()`.

*파란 막대는 실제 전환을 찾아낸 시간 수, 주황 막대는 전환으로 잘못 예상한 시간 수다. 점선은 실제 전환 전체 52시간이다. 각 입력의 판정 기준은 5 ~ 6월 검증에서 정하고 7 ~ 8월 평가에서 변경하지 않았다.*

**달력만으로는 실제 전환 52시간 중 35시간을 찾아냈지만, 직전 평균 전력을 추가하면 50시간을 찾아냈다.** 전환으로 잘못 예상한 시간도 51시간에서 25시간으로 줄었다. 실제 전환을 찾아낸 비율인 재현율은 67.3%에서 96.2%로 높아졌고, 전환이라고 판정한 기록 중 실제 전환의 비율인 정밀도는 40.7%에서 66.7%로 높아졌다. 전환을 더 많이 찾아내면서 잘못된 판정도 줄인 결과다. 다만 과거 전력을 사용해도 전환으로 판정한 75시간 중 25시간은 실제로 전환하지 않았다.

확률 순위에 따른 정밀도·재현율을 요약한 AP도 달력만 사용하면 0.377, 직전 평균 전력을 추가하면 0.902였다. 과거 평균 전력을 추가한 AP는 7월 0.906, 8월 0.898로 두 월 모두 달력만 사용한 결과보다 높았다. 반복 전력 배열의 비중을 낮춘 보조 점검에서도 같은 방향이 유지됐다. **전환 구분의 큰 이득은 시간대에 더해 이미 관측한 평균 전력 수준을 사용했을 때 나타났다.**

시간 안의 마지막−첫 값을 더하면 AP는 0.904로 소폭 높아졌지만, 실제 전환 적중 50시간과 오탐 25시간은 같았다. 따라서 이 비교에서 시간 안의 변화가 전환 구분에 큰 추가 정보를 주었다고 해석하지 않았다. 앞서 이 값은 전체 기록의 다음 시간 전력 변화와 연결됐지만, 전력 변화의 크기를 설명하는 정보와 생산량 전환 여부를 구분하는 정보의 유용성은 달랐다.

이 결과는 현재 생산량이 0이라는 기록만으로 다음 시간의 상태를 판단하기보다 **이미 관측한 전력 수준을 함께 사용해 전환 가능성을 표현할 근거**가 된다. 실제 생산량 전환을 미리 안다고 가정하지 않고도 과거 정보로 전환 여부를 구분하는 데 도움을 얻었다. 전환 가능성을 전력 예측에 어떻게 반영할지와 실제 전력 예측오차가 줄어드는지는 Modeling에서 별도로 다룬다. 현재 기간은 EDA에서도 살펴본 자료이므로 시간순 검증 결과를 새로운 독립 시험이나 실제 운영 성능으로 확대하지 않는다.

근거: [전환 구분·검증 조건과 해석](Analysis/10.03_A02_발전_전환사전구분과전력예측_분석기록.md) · [모형·판정 기준 고정](Analysis/tables/a02_transition_forecast/frozen.json) · [시간순 평가 결과](Analysis/tables/a02_transition_forecast/classification_test.csv) · [반복 배열 보조 점검](Analysis/tables/a02_transition_forecast/repetition_sensitivity.csv)

**관련 Python 코드**

- [Analysis/scripts/a02_prior_power_signals.py](Analysis/scripts/a02_prior_power_signals.py): `build()`로 연속 세 시간의 관측·과거 입력을 구성하고, `association()`으로 앞선 전력 변화 관계를 계산한다.
- [Analysis/scripts/a02_signal_diagnostics.py](Analysis/scripts/a02_signal_diagnostics.py): 과거 전력 수준 조정 방식과 같은 수준의 비교 가능한 기록을 점검한다.
- [Analysis/scripts/a02_transition_forecast.py](Analysis/scripts/a02_transition_forecast.py): `class_validation()`에서 입력·모형·판정 기준을 검증하고, `final_evaluation()`에서 고정한 조건으로 7~8월 전환 구분 결과를 계산한다. 같은 파일의 전력 회귀 실험은 이 절의 전환 구분 성과와 별개다.
- [Analysis/scripts/a02_transition_review.py](Analysis/scripts/a02_transition_review.py): 저장된 전환 확률로 반복 배열 보조 점검을 수행하고 적중·오탐 그림을 생성한다.

## 3.3 날씨 관계와 예측 입력의 검토

과거 전력 정보에 더해 날씨를 예측 입력으로 검토할 근거를 확인했다. EDA에서 8월 기온·전력 관계가 생산량 조건에 따라 달랐으므로, 네 기상변수가 모두 관측된 평일 171일·4,102시간에서 월·시간대·생산 기록·다른 기상변수를 고려하기 전후의 순위 관계를 비교했다. 전력은 네 15분 값의 평균을 사용했다.

![동일한 평일 기록에서 조건을 고려하기 전후의 날씨·전력 관계](Analysis/figures/a03_weather_conditions.png)

이미지 파일: [Analysis/figures/a03_weather_conditions.png](Analysis/figures/a03_weather_conditions.png). 생성 코드: [plot_a03_weather.py](Analysis/scripts/plot_a03_weather.py)의 `main()`.

| 기상변수 | 단순 순위 상관 | 조건을 고려한 뒤 남은 관계 |
|---|---:|---:|
| 기온 | +0.227 | −0.004 |
| 풍속 | +0.195 | +0.046 |
| 습도 | −0.102 | +0.100 |
| 강수량 | +0.057 | −0.060 |

*단순 관계는 Spearman 상관, 조건을 고려한 관계는 해당 조건으로 설명되는 순위 차이를 제거한 뒤 계산한 부분 순위 관계다. 두 열은 동일한 4,102시간을 비교한다.*

전체 기간에서는 조건을 고려한 관계가 약했다. 8월의 생산량 양수인 평일 360시간·17일에서는 기온 관계가 +0.302로 남았지만, 날짜별 차이까지 고려하면 +0.014로 약해졌다. 따라서 기온이 높은 날짜에 전력도 높은 경향을 같은 날 안의 기온 상승과 전력 상승 관계로 확대하기 어려웠다.

이 결과만으로 날씨의 예측 가치를 배제할 수는 없다. 다만 단순 상관을 근거로 큰 예측 개선을 기대하기 어려워, 예측에서는 3.1·3.2에서 확인한 과거 전력 정보의 활용을 우선 비교한다. 날씨의 추가 가치는 예측 시점에 이용 가능한 과거 기상정보를 더했을 때 오차가 줄어드는지로 판단한다. 목표 시간의 실측 날씨와 생산량을 미리 아는 입력으로 사용하지 않는다.

근거: [날씨 관계의 조건·월·날짜별 점검](Analysis/10.03_A03_생산조건을고려한날씨전력관계_분석기록.md) · [동일 기록 비교](Analysis/tables/a03_conditioned_weather/common_row_associations.csv) · [날짜별 차이 점검](Analysis/tables/a03_conditioned_weather/temperature_monthly_diagnostics.csv)

**관련 Python 코드**

- [Analysis/scripts/a03_conditioned_weather.py](Analysis/scripts/a03_conditioned_weather.py): `prepare()`·`association()`으로 생산 기록과 달력·기상 조건을 고려한 순위 관계를 계산한다.
- [Analysis/scripts/a03_weather_diagnostics.py](Analysis/scripts/a03_weather_diagnostics.py): `main()`에서 동일한 기록의 전후 관계를 비교하고, `adjusted()`로 날짜별 차이와 생산량 조정 방식의 영향을 점검한다.
- [Analysis/scripts/plot_a03_weather.py](Analysis/scripts/plot_a03_weather.py): 동일한 평일 4,102시간의 비교표로 위 그림을 생성한다.

## 3.4 다음 시간 최대전력과 마지막 15분 값의 관계

3.1에서 확인한 큰 전력 변화가 높은 최대전력도 뜻하는지 살펴봤다. 목표 시간의 네 15분 값 중 최댓값을 ‘시간별 최대전력’으로 정의하고 연속값으로 비교했다. 3.1과 같은 46조건·717시간에서 생산량 0→양수 전환의 최대전력은 135.73, 양수 지속은 135.32로 차이가 작았다. **생산 전환에 따른 큰 변화와 높은 최대전력은 구분해야 했다.**

다음으로 관측을 마친 시간의 어떤 전력값이 다음 시간 최대전력과 연결되는지 확인했다. 정확히 연속된 세 시간의 기록이 있는 5,778시간에서, 목표 시간의 월·시간대·평일/주말과 직전 생산량·평균 전력을 고려한 부분 순위 관계를 비교했다. 목표 시간의 실제 생산량과 평균 전력은 조건에 넣지 않았다.

![직전 네 구간과 다음 시간 최대전력의 관계](Analysis/figures/a04_terminal_slots.png)

이미지 파일: [Analysis/figures/a04_terminal_slots.png](Analysis/figures/a04_terminal_slots.png). 생성 코드: [plot_a04_terminal_slots.py](Analysis/scripts/plot_a04_terminal_slots.py)의 `main()`.

*네 막대는 동일한 5,778시간에서 계산한 부분 순위 관계다. 월·시간대·평일/주말, 직전 생산 기록과 평균 수준을 고려했다. 값은 예측 정확도나 오차 개선률을 뜻하지 않는다.*

**마지막 15분 값의 관계는 +0.434로, 다른 세 구간보다 컸다.** 직전 최대값까지 고려해도 +0.409로 남았고, 이 비교의 기간별 관계는 1 ~ 4월 +0.394, 5 ~ 6월 +0.430, 7 ~ 8월 +0.390이었다. 반복 전력 배열의 비중을 낮춘 결과도 +0.374로 양의 방향이 유지됐다. 평균·최대 수준만으로 설명되지 않는 마지막 값의 관계가 관측된 것이다.

다만 과거 평균값과 달력·생산량 0 여부를 정확히 맞춘 비교에는 1,039시간만 남았고, 이 중 71.5%는 EDA의 낮은 전력 구간이었다. 해당 비교의 1 ~ 4월 관계는 −0.009로 거의 없었다. 비교 가능한 기록의 구성도 달라졌으므로, 마지막 값이 모든 조건에서 같은 정보를 준다고 일반화하지 않았다.

이 결과를 바탕으로 Modeling에서는 **과거 평균·최대에 마지막 15분 값을 추가해 다음 시간 최대전력을 직접 예측하는 방법**을 비교한다. 전체 오차와 실제 일별 최대전력 시간의 과소·과대예측을 함께 평가하고, 3.1의 전력 상승 전환·하락 전환·구간 유지에서도 추가 정보가 오류를 줄이는지 확인한다. 실제 목표값으로 구분한 구간은 사후 평가에만 사용한다. 과대예측을 크게 늘려 과소예측만 줄이는 결과는 개선으로 채택하지 않는다. 이번 분석은 예측 입력 후보의 근거이며, 실제 최대전력의 과소예측 감소는 시간순 예측 검증에서 확인한다.

근거: [최대전력 조건·선행정보와 예외 점검](Analysis/10.03_A04_다음시간최대전력과선행정보_분석기록.md) · [생산조건별 최대값 비교](Analysis/tables/a04_maximum_conditions/matched_contrasts.csv) · [평균·최대 고려 후 관계](Analysis/tables/a04_maximum_conditions/terminal_unique_information.csv) · [엄격 비교의 표본 구성](Analysis/tables/a04_maximum_conditions/exact_mean_support.csv)

**관련 Python 코드**

- [Analysis/scripts/a04_maximum_conditions.py](Analysis/scripts/a04_maximum_conditions.py): `build()`로 다음 시간 최대값과 과거 입력을 구성하고, `matched()`·`association()`으로 생산 전환별 최대값과 선행 관계를 비교한다.
- [Analysis/scripts/a04_maximum_diagnostics.py](Analysis/scripts/a04_maximum_diagnostics.py): `daily_priorities()`·`maximum_context()`로 일별 최대 시간과 평균·최대 차이를 점검하고, `slot_group_associations()`로 네 구간의 관계를 비교한다.
- [Analysis/scripts/a04_terminal_unique.py](Analysis/scripts/a04_terminal_unique.py): `compare()`로 과거 평균·최대까지 고려한 마지막 값의 추가 관계와 기간별 안정성을 확인한다.
- [Analysis/scripts/a04_support_diagnostics.py](Analysis/scripts/a04_support_diagnostics.py): 같은 과거 평균을 정확히 맞춘 비교의 표본 구성과 초기 기간의 예외를 점검한다.
- [Analysis/scripts/plot_a04_terminal_slots.py](Analysis/scripts/plot_a04_terminal_slots.py): 저장된 네 구간 비교표로 위 그림을 생성한다.

### 3.5 Analysis 코드 실행

생산량 전환과 전력 변화, 과거 전력의 전환 구분, 조건을 고려한 날씨 관계, 마지막 15분 값의 추가 관계를 다시 계산한다. A02의 고정 분류·회귀 설정과 혼합 진단은 같은 시간순 조건으로 실행한다. 전력 자체의 구간 전환을 확인한 뒤 P01의 시간 안 최대 형태를 연결한다.

In [ ]:
with computation_paths():
    for name in ['a01_production_changes','a01_relative_changes','a02_prior_power_signals','a02_signal_diagnostics']:
        execute_step(name,MODULES[name].main)
    for stage in ['validation','mixture','final']:
        execute_step('a02_transition_forecast_'+stage,lambda stage=stage:cli('a02_transition_forecast',stage))
    for name in ['a02_transition_review','a03_conditioned_weather','a03_weather_diagnostics','a04_maximum_conditions','a04_maximum_diagnostics','a04_terminal_unique','a04_support_diagnostics','a06_power_state_transitions','plot_a01_production_changes','plot_a03_weather','plot_a04_terminal_slots','plot_analysis_transition_summary']:
        execute_step(name,MODULES[name].main)
    peak=MODULES['p01_peak_shapes']
    peak.OUT=WORK/'EDA/tables/p01_recalculated'
    execute_step('p01_peak_shapes',peak.main)
show_table('Analysis/tables/a02_transition_forecast/classification_test.csv')
show_table('Analysis/tables/a04_maximum_diagnostics/slot_group_associations.csv') if (WORK/'Analysis/tables/a04_maximum_diagnostics/slot_group_associations.csv').exists() else None

원고에 기록된 결과와 해석을 아래에 보존했다. 다음 실행 셀의 결과는 이번 실행에서 계산하거나 검증한 값이다.

# 4. Modeling

EDA에서는 생산량과 전력의 하루 패턴이 완전히 일치하지 않았고, 낮은 전력이 이어지는 구간과 그 안팎의 큰 변화가 함께 관측됐다. Analysis에서는 생산량 전환과 전력 자체의 상승·하락 전환을 구분해야 한다는 점을 확인했다. 또한 직전 평균·최대 전력을 고려한 뒤에도 마지막 15분 값과 다음 시간 최대전력의 관계가 남았다. 이 결과를 바탕으로, 관측을 마친 과거 정보로 다음 한 시간의 최대전력을 예측하고 그 크기를 작게 예상하는 문제를 줄이는 방법을 검토했다.

이번 원고는 M01의 자료·평가 설계에서 M02 모델 비교, M03 입력 비교, M04 오류분석, M05 고정 후반기 평가와 후속 개발까지 이어진다. 4.9에는 기존 M05의 고정 후반기 결과를 보존하고, 4.11에는 과소예측 검토 우선순위의 보조 활용을 기록했다. 현재 차별화 성과는 4.12의 지속 상승 예측과 선택적 HGB 보완에 정리한다. 이 절은 개발 성과와 이미 관측한 7~8월에서의 탐색적 개선을 연결하며, 각 과거 절의 당시 판단과 구분한다. 수치는 원자료의 전력 척도로 제시하고 실제 전력·비용 절감량으로 환산하지 않는다.

<!-- MODELING_CLOSE_BEGIN -->
![모델링의 개발·적용 흐름](Modeling/figures/modeling_close/pipeline.png)

*전체 흐름: 자료 진단·시간순 평가·모델 선택을 거쳐, 매시간 지속 상승 경고에 따라 기본 또는 보완 HGB를 선택한다. 해석과 재현 점검은 4.13에 정리했다.*
<!-- MODELING_CLOSE_END -->

## 4.1 M01: 다음 시간 최대전력의 예측 설계와 자료 검증

### 예측할 값과 예측 시점

Analysis 3.4에서 확인한 마지막 값의 부분 순위 관계 +0.409는 새로운 입력을 비교할 근거였지만, 그 값이 실제 예측오차를 줄이는지는 아직 알 수 없었다. 먼저 단순 기준·기본 모델·추가 입력을 **같은 정보 시점과 같은 평가 기록에서 비교할 수 있도록** 목표와 시차를 정했다.

주 목표는 목표 시간 안에 기록된 네 값 `15분·30분·45분·60분`의 최댓값이다. 한 시간의 평균만 예측하면 같은 시간 안에 나타난 높은 값을 작게 요약할 수 있으므로, 최대전력을 직접 예측하도록 했다. 보조 목표는 네 값의 산술평균이며 CSV의 정수 반올림 ‘평균’ 대신 반올림 전 값을 사용했다. 최대와 평균은 각각 별도의 연속값 예측 문제로 평가한다.

예측 시점은 **직전 시간의 관측을 마친 뒤, 다음 한 시간의 관측이 시작되기 전**으로 정했다. 예를 들어 08시 기록을 확인한 뒤 09시의 네 값 중 최댓값을 예측한다. 목표 시간의 달력은 미리 알 수 있지만 목표 시간의 실제 생산량·전력·날씨는 알 수 없는 것으로 취급한다. 직전 기록을 보고 지연 없이 이용할 수 있다는 가정은 현 자료에서 검증할 수 없어 적용 조건으로 명시했다.

| 구분 | 정의와 사용 범위 |
|---|---|
| 주 목표 | 다음 한 시간의 네 전력값 중 최댓값 |
| 보조 목표 | 다음 한 시간의 네 전력값의 반올림 전 산술평균 |
| 달력 입력 | 목표 시간의 시간대 사인·코사인, 월, 평일/주말, 요일 더미 |
| 기본 과거 입력 | 직전 생산량·생산량 0 여부, 1·2시간 전 평균 및 최대 전력 |
| 이후 비교할 입력 | 직전 마지막 15분 값, 24·168시간 전 평균·최대, 이용 가능한 과거 날씨 |
| 사후 진단 | 목표 생산량과 전력 구간의 전환, 실제 일별 최대 시간, 하루 배열의 학습·평가 겹침 |

기본 입력 A는 달력 11개와 과거 입력 6개, 총 17개다. B는 여기에 직전 마지막 15분 값을 추가한다. C는 전날·전주 전력의 추가 가치를 비교할 구성이다. 날씨 D는 개발에서 선택한 입력 구성에 추가할 후보로 두고, 실제 비교할 부모 구성을 실행 전에 기록한다. 과거 평균·최대와 마지막 값의 효과를 다른 입력 변경과 섞지 않기 위한 구분이다.

저장 자료에는 목표 생산량이나 배열 식별자 등 사후 진단 열도 들어 있다. 이 열들을 자료 전체와 함께 모델에 넣지 않고, 입력 목록으로 허용한 변수만 선택한다. 특히 목표 시간의 실측 생산량·날씨, 실제 하루 최대 시각, 공장인원은 입력에서 제외했다. 기록 생성 방식과 독립성이 미식별인 부분을 임의로 채우지 않았다.

### 정상 기록과 정확한 시간 연결

자료 범위는 기존 EDA·Analysis와 같은 2021년 1 ~ 8월이다. 해당 기간 5,832행에서 시각 오류 48행을 제외한 정상 241일·5,784시간을 사용했다. 9월 336행은 이번 분석·시차·학습에 편입하지 않았다. 원본 파일을 보존하고 해시로 동일 자료인지 확인했다.

**과거 입력은 이전 행이 아니라 정확히 해당 시각의 기록으로 연결했다.** 정상 시각이 없는 날짜를 건너뛴 직전 행을 한 시간 전으로 취급하면 실제보다 긴 간격의 전력을 사용하게 된다. 따라서 날짜·시각으로 만든 키에서 정확히 1·2·24·168시간 전을 조회했다.

| 필요한 시차 | 연결 가능한 시간 | 정상 시간 중 제외 수 |
|---|---:|---:|
| 1시간 전 | 5,781 | 3 |
| 2시간 전 | 5,778 | 6 |
| 24시간 전 | 5,712 | 72 |
| 168시간 전 | 5,568 | 216 |
| 기본 1·2시간 모두 존재 | 5,778 | 6 |
| 1·2·24·168시간 모두 존재 | 5,520 | 264 |

*제외 수의 분모는 정상 5,784시간이다. 시차별 누락이 서로 겹치므로 제외 수를 더하지 않는다. 범위 시작 이전의 시각과 시각 오류 날짜의 영향을 각각 기록했다.*

주 비교에는 모든 시차가 존재하는 **공통 표본(common) 5,520시간**을 사용한다. 기본 모델도 학습·평가 양쪽에서 같은 표본을 사용하므로, 나중에 전날·전주 입력을 추가했을 때 표본이 바뀌는 효과를 줄일 수 있다. 기본 1·2시간만 필요한 더 넓은 표본(core) 5,778시간은 보조 적용 범위로 관리한다. 두 표본의 점수를 같은 조건의 모델 비교처럼 섞지 않는다.

필수 전력·생산 시차가 없는 시간은 해당 비교에서 제외하며 과거값을 보간하지 않는다. 날씨에는 직전 풍속 3시간과 강수량 1시간의 결측이 있어, D를 실행할 때 학습 자료의 중앙값과 결측 표시를 사용할 계획이다. 학습에서 전부 결측인 변수는 제외 사유를 기록하고 미래 관측으로 채우지 않는다.

공통 표본 밖의 단순 기준 가용성도 확인했다. 직전 기준은 1시간, 전날 기준은 24시간이 없으면 1시간, 전주 기준은 168시간이 없으면 24시간과 1시간 순으로 대체하도록 정했다. 각각 허용한 시차가 모두 없을 때만 예측을 보류한다. 이러한 대체 규칙은 적용 범위를 위한 보조 조건이며 M02의 공통 표본 성능에 섞지 않았다.

### 시간순 평가와 최대 시간의 가중치

평가 기록이 학습보다 뒤에 오도록 세 번의 개발 비교와 후반기 평가를 정했다. 각 분할에서는 한 번 학습한 모델을 유지하고, 평가 중 매시간 관측을 마친 입력만 갱신한다. 미래 목표값을 사용한 재학습이나 무작위 행 분할은 하지 않는다.

| 역할 | 학습 기간 | 평가 기간 | 공통 학습 시간 | 공통 평가 시간 |
|---|---|---|---:|---:|
| 개발 1 | 1 ~ 3월 | 4월 | 1,992 | 720 |
| 개발 2 | 1 ~ 4월 | 5월 | 2,712 | 744 |
| 개발 3 | 1 ~ 5월 | 6월 | 3,456 | 720 |
| 개발 선택 후 후반기 평가 | 1 ~ 6월 | 7 ~ 8월 | 4,176 | 1,344 |

M05의 후반기 평가는 7월과 8월을 하나의 평가 분할로 묶는다. **1~6월 자료로 각 모델을 한 번 학습한 뒤, 같은 모델을 고정해 7~8월을 평가한다.** 개발 비교처럼 학습 구간을 월별로 늘려 1~7월로 다시 학습하고 8월을 예측하는 방식은 아니다. 다만 매시간 직전 관측이 완료되면 그 전력·생산량 등을 입력에 반영해 다음 한 시간을 예측한다. 따라서 모델은 두 달 동안 고정되지만 입력값은 매시간 갱신되며, 6월 말에 7~8월의 모든 시각을 한꺼번에 예측하는 것도 아니다. 예측 시점에 아직 관측되지 않은 값은 사용하지 않는다.

개발 평가에는 91일·2,184시간이 포함된다. 후반기의 정상 기록은 60일·1,440시간이지만 공통 시차 조건을 적용하면 56일·1,344시간이다. 7월 14·16·20·22일의 96시간이 제외되는 것은 입력 가용성의 차이이며 성능 차이가 아니다. 기본 시차 표본은 1,436시간으로 더 넓지만, 불완전한 두 날짜의 남은 22시간 중 최대값을 실제 하루 최대로 대신하지 않는다.

전체 시간의 평균 오차와 함께 **실제 일별 최대전력이 기록된 시간의 오차**를 평가한다. 하루 24시간이 모두 평가 가능한 날짜만 사용하고, 한 날짜에 최대값이 여러 번 나타나면 해당 시간들의 가중치 합을 1로 맞춘다. 여러 번 같은 최대값이 나타난 날이 더 큰 비중을 갖지 않도록 한 것이다. 실제 일별 최대는 낮은 전력의 날에도 존재하므로 최대값 자체의 크기와 표본 수도 함께 제시한다.

### 반복 배열을 확인한 뒤 보완한 해석

정상 241일의 하루 96개 전력값 배열은 126종이며, 160일이 반복 배열 그룹에 속한다. 시간순 분할을 적용해도 같은 하루 전력 배열이 학습과 평가에 함께 나타날 수 있으므로 그 겹침을 확인했다.

| 평가 기간 | 평가 날짜 | 학습과 같은 배열인 날짜 | 같은 배열 시간 | 새로운 배열 시간 |
|---|---:|---:|---:|---:|
| 4월 | 30일 | 0일 | 0 | 720 |
| 5월 | 31일 | 17일 | 408 | 336 |
| 6월 | 30일 | 26일 | 624 | 96 |
| 7 ~ 8월 | 56일 | 0일 | 0 | 1,344 |

*공통 표본의 학습·평가 대상 날짜를 기준으로 비교했다. 기본 시차 표본은 포함 날짜가 달라 겹침 수도 다르다. 위 수치만으로 자료 생성의 독립성을 입증하지 않는다.*

**5·6월의 전체 점수에는 학습과 같은 배열이 많은 구성의 영향이 포함된다.** 따라서 시간순 평가와 관측값은 유지하되, 전체·월별 점수에 학습과 겹치는 배열과 새로운 배열의 결과를 병기하도록 했다. 평가 안에서 같은 배열에 속한 날짜 수의 역수로 가중한 보조 결과도 확인한다. 이 가중치는 사후 평가에만 적용하며 학습 입력이나 학습 가중치로 사용하지 않는다. 특히 6월의 새로운 배열은 4일뿐이므로 해당 월의 새 배열 성능만으로 안정성을 주장하지 않는다.

7 ~ 8월은 이미 EDA에서 관찰한 내부 후반기 기간이다. 개발 선택이 끝난 뒤 고정한 방법의 결과를 확인하는 용도로 두며, 새로운 독립 시험으로 표현하지 않는다. 후반기에서 학습과 직접 같은 배열이 없다는 사실 역시 독립성의 증명이 아니다.

M01에서는 자료 범위·정확한 시차·공통 표본·목표와 입력 경계·평가 가중치를 확인했다. 원본을 별도로 순회해 5,784시간의 목표와 과거 입력값 91,356개를 대조했고, 저장된 8개 분할·표본 조합도 독립 검증했다. **같은 조건에서 모델과 입력을 비교할 준비가 됐다는 판단으로 M02에 연결했다.** M01 자체에는 모델의 예측 성능 결과가 없다. 이 절은 표를 사용하며 별도의 이미지는 사용하지 않았다.

근거: [M01 조건](Modeling/config/m01_contract.json) · [입력 변수표](Modeling/tables/m01/feature_dictionary.csv) · [시차 가용성](Modeling/tables/m01/lag_coverage.csv) · [시차 누락 원인](Modeling/tables/m01/unavailable_lags.csv) · [분할별 표본](Modeling/tables/m01/split_counts.csv) · [일별 가용성](Modeling/tables/m01/daily_coverage.csv) · [배열 겹침](Modeling/tables/m01/profile_overlap.csv) · [평가 가중치](Modeling/tables/m01/evaluation_diagnostics.csv) · [독립 검증](Modeling/tables/m01/independent_verification.json).

**관련 Python 코드**

- [Modeling/scripts/m01_prepare.py](Modeling/scripts/m01_prepare.py): `load_frame()`에서 원자료·목표·정확한 시차를 구성하고, `feature_columns()`에서 허용 입력 목록을 지정한다. `independent_check()`는 원본의 목표·시차값을 별도로 대조하며, `main()`은 분할·배열 겹침·가중치를 저장한다.
- [Modeling/scripts/m01_verify.py](Modeling/scripts/m01_verify.py): 저장된 자료를 표준 CSV·시간 계산으로 독립 대조한다. 전처리에서 확인한 조건과 후속 사용 파일이 일치하는지 검증한다.
- 상세 실행 조건과 회차 기록은 [M01 검증 기록](Modeling/10.04_M01_자료시차평가조건_검증기록.md)에 있다. 당시 초기 모델 후보는 Ridge·HGB였으며 M02에서 승인된 네 계열 비교로 확대했다.

## 4.2 M02: 단순 기준과 네 모델 계열의 성능 비교

M01에서 평가 조건을 확인했으므로, 이번에는 기본 과거 정보만으로 어느 정도 예측할 수 있는지 비교했다. 마지막 15분 값의 효과를 확인하기 전에 **새 입력이 넘어야 할 예측 기준과 기본 모델을 정하는 것**이 목적이다. 공통 개발 표본의 같은 목표 시각에서 단순 기준 3개와 Ridge·Elastic Net·SVR·HGB를 비교했다.

### 모델을 비교한 이유와 동일 조건

직전·전날·전주 단순 기준은 각각 해당 과거 시간의 전력값을 다음 시간의 예측값으로 그대로 사용한다. 직전 기준으로 최근 수준의 지속을, 전날·전주 기준으로 시간대·주간 반복의 가치를 확인한다. 모델이 이 기준보다 좋아야 학습의 추가 효과를 설명할 수 있다.

학습 모델은 기본 A 입력 17개를 모두 동일하게 사용했다. Ridge는 과거 평균·최대처럼 정보가 겹칠 수 있는 변수들의 계수를 축소하는 선형 기준이다. Elastic Net은 계수 축소와 변수 선택을 함께 하는 경우를 비교한다. RBF SVR은 커널을 이용한 비선형 회귀, HGB는 시간대와 과거 전력의 조합에 따라 달라지는 관계를 트리 부스팅으로 학습하는 후보다. 어떤 계열이 더 좋을지는 실행 결과로 판단했다.

| 계열 | 후보 설정 | 학습 자료 안의 전처리 |
|---|---|---|
| Ridge | alpha 10·1, 2개 | 입력 표준화 |
| Elastic Net | alpha 0.1·0.01·0.001 × L1 비율 0.1·0.5·0.9, 9개 | 입력·목표 표준화 후 예측을 원척도로 복원 |
| RBF SVR | C 1·10·100 × gamma scale·0.03 × epsilon 0.1·0.05, 12개 | 입력·목표 표준화 후 예측을 원척도로 복원 |
| HGB | 최대 잎 수 7·15, 2개 | 원척도 입력·목표 |

HGB의 나머지 설정은 학습률 0.05, 부스팅 반복 150회, 최소 잎 표본 20, L2 규제 1, 시드 42이며 조기 종료를 껐다. Elastic Net은 최대 반복 20,000·허용 오차 0.0001을 사용했다. Elastic Net의 alpha와 SVR의 C·epsilon은 표준화한 목표 척도에서 적용한 값이다. 모든 표준화는 각 분할의 학습 자료로만 적합하고 평가 자료에 그대로 적용했다.

총 25개 후보 설정을 두 목표·세 개발 분할에서 학습해 150회 적합했다. 각 계열에서는 3개월 전체의 MAE를 최소화하는 설정을 확인하고, 최저값의 1% 안에서는 미리 기록한 설정 순서를 우선했다. 따라서 아래 선택 설정의 점수가 계열 내 절대 최저점과 조금 다를 수 있다. 1%는 수치상 유사한 후보를 고르는 규칙이며 현장 비용의 허용 기준은 아니다. 후보 수가 계열마다 다르고 탐색 범위도 제한적이므로 모든 설정에서의 알고리즘 우열을 주장하지 않는다.

모든 학습 모델은 같은 학습 행·평가 행·A 입력을 사용했다. 단순 기준도 같은 평가 시각을 사용하지만 각각 조회하는 과거 정보는 다르다. 전날·전주 전력은 단순 기준에서는 사용하고 이번 A 학습 입력에는 포함하지 않았다. 이 시차를 학습 입력에 더하는 효과는 이후 C 비교의 질문이다. 마지막 15분 값과 날씨도 이번 학습에 포함하지 않았다.

### 전체 성능과 월별 차이

MAE는 시간별 절대오차의 평균으로 주 지표이며, 큰 오류에 더 민감한 RMSE를 보조 지표로 함께 제시한다. 과소예측량은 `max(실제−예측, 0)`, 과대예측량은 `max(예측−실제, 0)`으로 계산했다. 각각 해당 방향의 오류가 없는 시간에도 0을 포함해 전체 평가 시간에서 평균했다. 발생 비율과 오류가 발생한 시간의 평균 크기는 별도 결과표에 저장했다.

![기본 모델의 최대전력 예측오차와 월별 비교](Modeling/figures/m02/model_comparison.png)

이미지 파일: [Modeling/figures/m02/model_comparison.png](Modeling/figures/m02/model_comparison.png). 생성 코드: [Modeling/scripts/plot_m02_manuscript.py](Modeling/scripts/plot_m02_manuscript.py)의 `plot_comparison()`.

*왼쪽은 동일한 개발 2,184시간에서의 최대전력 예측 MAE다. 회색 막대는 단순 기준과 나머지 모델, 파란 막대는 HGB다. 오른쪽은 직전 기준·전주 기준·HGB의 월별 MAE다. MAE는 작을수록 좋으며 오른쪽 점을 잇는 선은 월별 점수의 연결이지 실제 시간순 전력 곡선이 아니다.*

| 방법·선택 설정 | MAE | RMSE | 평균 과소예측량 | 평균 과대예측량 |
|---|---:|---:|---:|---:|
| 직전 시간 | 13.79 | 23.04 | 6.92 | 6.88 |
| 전날 같은 시간 | 43.36 | 68.64 | 22.33 | 21.03 |
| 전주 같은 시간 | 19.78 | 40.27 | 10.63 | 9.16 |
| Ridge alpha 10 | 14.46 | 21.50 | 7.22 | 7.24 |
| Elastic Net alpha 0.01·L1 비율 0.5 | 14.42 | 21.69 | 7.20 | 7.22 |
| SVR C 1·gamma 0.03·epsilon 0.05 | 14.26 | 20.92 | 8.55 | 5.71 |
| HGB 최대 잎 15 | **6.74** | **10.95** | **3.43** | **3.31** |

**이번 개발 비교에서 HGB의 MAE는 가장 좋은 단순 기준인 직전 시간보다 51.1% 작았다.** 계산은 `(13.7930−6.7380)/13.7930`이며, 예측 정확도가 51.1%p 높아졌다는 뜻은 아니다. 평균 과소·과대예측량도 함께 줄어 전체에서 과대예측만 늘려 얻은 과소예측 감소는 아니었다.

Ridge·Elastic Net·SVR의 선택 설정은 MAE에서 직전 기준을 넘지 못했다. 계열 내 MAE만 최저인 Ridge 14.31, Elastic Net 14.29도 같은 결론이었다. 반면 세 계열의 RMSE는 직전 기준보다 낮아, 평균 절대오차와 큰 오류를 줄이는 효과는 다를 수 있었다. 따라서 모든 지표에서 단순 기준보다 나쁘다고 해석하지 않았다.

| 평가 월 | 직전 시간 MAE | 전주 MAE | HGB MAE |
|---|---:|---:|---:|
| 4월 | 14.02 | 24.54 | 8.31 |
| 5월 | 13.05 | 26.94 | 6.13 |
| 6월 | 14.34 | 7.64 | 5.80 |

HGB는 세 달 모두 단순 기준보다 낮았고, 날짜별 MAE도 91일 중 80일에서 직전 기준보다 낮았다. 나머지 11일에서는 직전 기준보다 높아 모든 날짜에서 개선된 것은 아니다. 전주 기준은 6월 MAE 7.64로 크게 좋아졌으므로, 전체 평균만으로 주간 시차의 가치를 제외하지 않는다. 6월의 배열 반복 구성도 함께 고려해야 한다.

보조 목표인 시간 평균도 동일한 절차로 별도 학습했다. 계열별 선택 설정은 최대 목표와 다를 수 있으며 최대 예측에서 평균을 환산한 결과가 아니다.

| 방법 | 시간 평균 MAE | 시간 평균 RMSE |
|---|---:|---:|
| 직전 시간 | 15.11 | 25.17 |
| 전날 | 39.74 | 63.68 |
| 전주 | 17.96 | 37.23 |
| Ridge | 14.16 | 20.40 |
| Elastic Net | 14.17 | 20.47 |
| SVR | 14.01 | 20.09 |
| HGB | **5.97** | **9.19** |

HGB가 두 목표 모두에서 가장 낮은 개발 오차를 보였으므로, 최대전력 직접 예측의 기본 후보로 HGB 최대 잎 15를 선택했다. 다만 과거 전력과 달력의 조합 중 어느 정보가 개선을 만들었는지는 모델 계열 비교만으로 식별하지 못했다. 특히 마지막 15분 값은 아직 입력에 없으므로 이번 개선을 그 값의 기여로 설명하지 않는다.

### 실제 일별 최대 시간에서의 과소예측

전체 평균 성능이 좋아도 높은 전력이 나타난 시간의 과소예측이 남을 수 있다. 완전한 91일의 실제 최대 시간 137개를 선택해 하루당 가중치 합이 1이 되도록 평가했다. 평가한 실제 일별 최대값의 가중 평균은 150.81이었다. 낮은 전력의 날도 포함된 상대적 일별 최대이며, 모든 시간에 절대적 피크 위험이 있었다는 뜻은 아니다.

| 방법 | 최대 시간 MAE | 평균 과소예측량 | 평균 과대예측량 |
|---|---:|---:|---:|
| 직전 시간 | 36.59 | 36.59 | 0.00 |
| 전주 | 19.68 | 15.66 | 4.02 |
| HGB | **11.75** | **11.26** | 0.49 |

**HGB의 최대 시간 평균 과소예측량은 직전 기준보다 69.2%, 전주 기준보다 28.1% 작았다.** 전주 기준은 전체 MAE에서는 직전 기준보다 나빴지만 최대 시간의 과소예측은 더 작아, 주 목표와 중요한 조건의 지표를 함께 볼 필요가 있었다. HGB는 이 두 기준보다 최대 시간 MAE도 낮았다.

직전 기준의 최대 시간 과대예측량이 0인 것은 기준값이 같은 하루의 실제 최대값을 넘기 어렵다는 평가 조건과도 연결된다. HGB는 0.49의 과대예측량이 있어 해당 항목만 개선됐다고 주장하지 않았다. 다만 전체 평균 과대예측은 6.88에서 3.31로 감소했다. 실제 최대 시각을 사후에 골라 오차를 확인한 것이므로, 하루 최대 시간을 미리 선택했다는 성과나 현장 경보 성능으로 확대하지 않는다.

### 새로운 배열에서도 개선이 유지되는가?

M01에서 5·6월의 학습·평가 배열 겹침이 많았으므로 같은 구분으로 성능을 확인했다.

| 평가 조건 | 시간 수 | 직전 기준 MAE | HGB MAE |
|---|---:|---:|---:|
| 학습과 같은 배열 없음 | 1,152 | 13.08 | 7.56 |
| 학습과 같은 배열 있음 | 1,032 | 14.59 | 5.82 |
| 평가 내 반복 배열 비중을 낮춘 보조 결과 | 2,184 | 13.67 | 6.39 |

*마지막 행은 평가에서 같은 하루 배열에 속한 날짜 수의 역수로 가중한 결과다. 위 두 행과 별도의 보조 요약이며 세 행을 합산하지 않는다. 전체 자료를 다시 학습하거나 반복 기록을 삭제한 결과가 아니다.*

학습과 직접 같은 배열이 없는 기록에서도 개선 방향이 유지됐다. 따라서 개선이 직접 같은 배열에만 의존한다는 결과는 아니었다. 그러나 배열이 겹치는 집단과 겹치지 않는 집단은 월·날짜 구성도 달라, 두 집단 점수 차이를 반복의 인과효과로 해석하지 않는다. 새로운 배열이 4일뿐인 6월의 한계와 이미 관찰한 내부 기간이라는 조건도 유지한다.

### 전력 상승·하락과 구간 유지에서 남은 오류

Analysis에서는 전력 자체의 구간 전환이 생산량 전환과 대부분 일치하지 않았다. 관측 변동이 큰 조건이 실제 예측 실패조건이기도 한지 이번 예측으로 확인했다. A06과 동일하게 CSV의 정수 ‘평균’ 20 ~ 26을 낮은 구간으로 구분하고, 전력 0·양수 20 미만·20 ~ 26·26 초과를 나눴다. 26 초과를 현장의 고전력이나 피크 임계값으로 정의하지 않았다.

![전력 구간별 오류와 실제 일별 최대 시간의 과소·과대예측](Modeling/figures/m02/condition_errors.png)

이미지 파일: [Modeling/figures/m02/condition_errors.png](Modeling/figures/m02/condition_errors.png). 생성 코드: [Modeling/scripts/plot_m02_manuscript.py](Modeling/scripts/plot_m02_manuscript.py)의 `plot_conditions()`.

*왼쪽은 전력 구간 전환·유지별 MAE이며 회색은 직전 기준, 파란색은 HGB다. 괄호는 해당 조건의 시간 수다. 오른쪽은 실제 일별 최대 시간의 평균 과소예측량(파랑)·과대예측량(주황)을 쌓아 표시했다. 오른쪽 막대 전체 높이는 해당 조건의 MAE이며, 위 숫자도 합계다. 왼쪽은 시간별 평균, 오른쪽은 하루 최대 동률에 하루당 가중치 합 1을 적용한 평균이어서 두 패널의 분모는 다르다.*

| 전력 조건 | 시간 수 | 직전 기준 MAE | HGB MAE | 남은 오류 |
|---|---:|---:|---:|---|
| 20 ~ 26→26 초과 | 20 | 75.40 | 47.87 | 20시간 모두 과소예측 |
| 26 초과→20 ~ 26 | 19 | 68.11 | 28.39 | 평균 과대예측량 28.08 |
| 20 ~ 26 유지 | 625 | **2.79** | 3.77 | 평균 과대예측량 3.28 |
| 26 초과 유지 | 1,510 | 16.93 | **7.18** | 전체 절대오차에서 큰 비중 |

*합계 2,174시간이다. 나머지 10시간은 양수 20 미만 구간과의 전환이며 전체 결과표에 포함했다. 이번 개발 기간에는 전력 0 사례가 없다. 구간은 목표 시간의 실제 전력으로 정한 사후 진단이며 예측 입력이 아니다.*

**상승 전환에서는 오차가 줄었지만 과소예측이 크게 남았다.** MAE 47.87은 전체 HGB MAE 6.74보다 훨씬 컸고 해당 20시간은 모두 과소예측이었다. 하락 전환에는 반대로 과대예측이 남았다. 낮은 구간이 유지되는 조건에서는 HGB가 단순 기준보다 나빴으므로, 복잡한 모델이 모든 조건에서 유리하다고 볼 수 없었다.

오류 크기와 전체 기여도도 구분했다. 상승 전환 20시간은 HGB 전체 절대오차의 6.5%, 하락 전환 19시간은 3.7%였다. 반면 26 초과가 유지된 1,510시간은 73.7%를 차지했다. 이 비중에는 시간 수가 많은 영향도 포함돼 한 시간의 위험도나 실패 확률을 뜻하지 않는다. 전체 오류를 줄이려면 전환뿐 아니라 구간 안의 변동도 함께 봐야 한다. 생산 전환·월·시간대와 겹치는 구분의 기여율을 더하지 않았다.

생산량 전환도 별도로 확인했다. 전력 구간 분류와 서로 다른 질문이므로 전력 상승의 성과를 생산량 전환의 성과로 대신하지 않았다.

| 직전→현재 생산량 기록 | 시간 수 | 직전 기준 MAE | HGB MAE |
|---|---:|---:|---:|
| 0→양수 | 92 | 29.70 | 9.26 |
| 양수→0 | 92 | 27.10 | 8.82 |
| 양수 지속 | 1,311 | 15.95 | 7.25 |
| 0 지속 | 689 | 5.80 | 5.15 |

생산량 조건별로는 모두 개선됐지만, 전력 구간으로 나누면 낮은 구간 유지에서 악화가 있었다. 따라서 **생산량 조건의 개선만으로 전력의 모든 상태에서 개선됐다고 결론 내릴 수 없었다.** 목표 생산량으로 정한 전환은 사후 평가이며 실제 가동·정지의 원인으로 확정하지 않는다. 조건별 수치는 [전체 오류표](Modeling/tables/m02/condition_errors.csv)의 월·요일·시간대 결과와 함께 보존했다.

### 기본 모델 선택과 다음 입력 비교

확인된 사실은 현재 A 입력과 후보 설정에서 HGB가 전체·각 월·실제 일별 최대 시간의 오차를 줄였고, 학습과 직접 같은 배열이 없는 기록에서도 개선 방향을 유지했다는 것이다. 이에 HGB 최대 잎 15를 다음 입력 비교의 기본 모델로 선택했다. Ridge·Elastic Net·SVR의 결과도 보존해 선택 이유를 확인할 수 있게 했다.

동시에 상승 전환의 과소예측, 하락 전환의 과대예측, 낮은 구간 유지의 악화가 남았다. 따라서 다음 질문은 **같은 HGB 설정·학습 기간·평가 행에서 마지막 15분 값 하나를 추가하면 어떤 오류가 줄고 어떤 오류가 늘어나는가**다. Analysis 3.4의 관계가 실제 오류 감소로 이어지는지 전체·월별·최대 시간·전력 전환/유지 조건의 동일 시각 오차 차이로 확인한다. 입력 효과와 모델 설정 변경 효과를 섞지 않는다.

M02 시점에는 전날·전주 입력의 필요성과 날씨 비교의 순서를 남은 오류에 따라 갱신하기로 했다. 이어진 4.3에서는 입력 비교를, 4.9에서는 기존 후보의 고정 후반기 평가를 수행했다. 현재의 지속 상승 보완 성과는 4.12에서 별도로 설명한다.

### 재현성과 검증 범위

정규 CPython 3.13.1과 numpy 2.2.6·pandas 3.0.1·scikit-learn 1.7.1·joblib 1.5.1에서 실행했다. 150회 학습은 수렴 경고 없이 종료 코드 0이었다. 원자료·조건·입력 파일·코드·결과표의 해시를 저장했으며, 각 모델의 표준화 통계와 표본 수가 학습 자료로만 계산됐는지 대조했다. 선택된 24개 모델을 저장하고 다시 불러와 같은 평가 예측이 나오는지도 확인했다.

독립 검증은 저장된 122,304개 후보 예측행의 목표값·단순 기준·오차·진단 구분을 확인하고, 224개 성능 요약과 14개 최대 시간 지표를 재계산해 종료 코드 0을 확인했다. 모든 방법·목표가 같은 2,184개 평가 시각을 사용하고 7 ~ 8월이 포함되지 않은 것도 검증했다. 계열별 선택 방법의 음수 예측은 없었고, 예측값을 자동으로 0에 절단하지 않았다.

M02 결과는 개발 기간에서 후보 설정을 선택한 점수다. M02만으로 마지막 값의 추가 효과를 입증하지 않으며, 그 효과는 다음 M03에서 별도로 확인한다. 최종 후반기 성능, 실제 운영의 경보 성과나 전력 절감량은 아직 입증하지 않았다.

근거: [M02 조건](Modeling/config/m02_contract.json) · [모델 비교표](Modeling/tables/m02/comparison.csv) · [모든 후보 지표](Modeling/tables/m02/all_candidate_metrics.csv) · [계열별 선택](Modeling/tables/m02/family_selection.csv) · [조건별 오류](Modeling/tables/m02/condition_errors.csv) · [날짜별 오류](Modeling/tables/m02/daily_errors.csv) · [선택 방법의 시간별 예측](Modeling/tables/m02/selected_predictions.csv) · [학습 검증](Modeling/tables/m02/training_audit.csv) · [모델 파일 목록](Modeling/tables/m02/model_manifest.csv) · [실행환경·해시](Modeling/tables/m02/run.json) · [독립 검증](Modeling/tables/m02/independent_verification.json).

**관련 Python 코드**

- [Modeling/scripts/m02_compare.py](Modeling/scripts/m02_compare.py): `candidates()`에서 계열별 설정과 전처리를 구성하고, `metric()`에서 공통 오차지표를 계산한다. `main()`이 시간순 학습·예측·설정 선택·조건별 진단과 모델 저장·재로딩 대조를 수행한다.
- [Modeling/scripts/m02_verify.py](Modeling/scripts/m02_verify.py): 표준 CSV·수학 계산으로 예측·시차 기준·입력 해시·사후 조건·성능과 최대 시간 가중치를 독립 검증한다.
- [Modeling/scripts/plot_m02_manuscript.py](Modeling/scripts/plot_m02_manuscript.py): `load_verified()`가 저장 결과표의 해시와 검증 상태를 확인한다. `plot_comparison()`은 전체·월별 성능, `plot_conditions()`는 조건별 오류와 실제 최대 시간 그림을 생성한다. 색·축·범례·패널은 해당 함수에서 수정할 수 있다. 학습이나 새로운 모델 선택을 수행하지 않는다.
- 후보의 상세 설정·판단·재현 순서는 [M02 실행 기록](Modeling/10.04_M02_기본모형비교_실행기록.md)과 [Modeling README](Modeling/README.md)에 있다. 두 그림은 각각 가로·세로 50% 크기의 접촉 시트로 확인했다.

## 4.3 M03: 마지막 값의 추가 효과와 계절 시차의 선택

직전 시간의 마지막 15분 값을 추가한 HGB는 기본 입력 모델보다 전체 최대전력 MAE를 **8.1%**, 실제 일별 최대 시간 MAE를 **35.4%**, 최대 시간의 평균 과소예측량을 **37.9%** 줄였다. 세 수치는 같은 개발 기록에서 입력 하나를 추가한 전후 비교다. 이 절에서는 마지막 값을 선택한 이유, 동일 조건의 비교 방법, 개선이 나타난 조건과 남은 실패조건을 연결한다.

### 관계 분석에서 입력 비교로 이어진 질문

Analysis 3.4의 마지막 값 관계는 예측에 쓸 정보를 찾은 결과였다. M02에서는 마지막 값을 넣지 않은 HGB가 단순 기준보다 좋았지만, 실제 일별 최대 시간에 평균 11.26의 과소예측이 남았다. 전력이 낮은 구간에서 올라가는 시간과 낮은 구간이 유지되는 시간에도 서로 다른 오류가 남았다. 따라서 **모델 설정을 그대로 둔 채 마지막 값 하나가 어떤 오류를 줄이는지** 비교했다.

한 시간의 평균과 최대가 같더라도 마지막에 기록된 값은 다를 수 있다. 평균·최대만으로는 네 값의 마지막 수준을 구분할 수 없으므로, 마지막 값은 직전 시간 끝부분의 전력 수준을 추가로 알려 주는 입력이다. 이 정보가 다음 시간의 최대전력을 예측하는 데 도움이 되는지는 실제 예측오차의 변화로 판단했다. 기록 순서에서 얻은 단서를 이용한 비교이며, 특정 설비의 가동 원인이나 전력 변화의 물리적 원리를 확인한 것으로 해석하지 않는다.

모델은 M02에서 선택한 HGB 최대 잎 15로 고정했다. 학습 기간과 평가 행도 같게 유지했으며 달력·직전 생산량·1 ~ 2시간 전 평균과 최대를 쓰는 A에 직전 마지막 값만 추가해 B를 만들었다. A는 17개, B는 18개 입력이다. 마지막 값은 목표 시간의 마지막 값이 아니라 관측을 마친 직전 시간의 ‘60분’ 값이다. 실제 미래 생산량·날씨·전력은 입력에 넣지 않았다.

고정한 설정은 최대 잎 수 15, 반복 횟수 150, 학습률 0.05, 잎의 최소 표본 수 20, L2 규제 1.0이며 난수 시드는 42, 조기 종료는 미사용이다. 1~3월 학습→4월 평가, 1~4월→5월, 1~5월→6월의 동일한 시간순 조건에서 비교했다. 각 평가월의 모델은 고정하고 관측을 마친 시차 입력만 매시간 갱신했다. 이후 4.4의 확장 M02에서도 같은 HGB와 설정을 유지했으므로, 본 절의 A/B·C 비교를 재학습 없이 기존 검증 결과로 이어서 사용한다.

첫 비교 후 B가 전체와 최대 시간의 오차를 줄였지만 상승·낮은 구간 유지의 오류는 악화된 것을 확인했다. M02에서 전주 기준이 6월에 상대적으로 좋았던 결과와 연결해, 24·168시간 전 평균·최대를 추가한 C_A·C_B를 다음 비교로 선택했다. 이를 통해 마지막 값의 효과가 단순히 계절 시차 정보를 빠뜨린 데서 생겼는지 확인했다.

| 구성 | 입력 | 입력 수 | 비교할 질문 |
|---|---|---:|---|
| A | M02 기본 입력 | 17 | 마지막 값이 없는 기준 |
| B | A + 직전 마지막 값 | 18 | 마지막 값 하나의 추가 가치 |
| C_A | A + 전날·전주 평균 및 최대 | 21 | 계절 시차의 추가 가치 |
| C_B | C_A + 직전 마지막 값 | 22 | 계절 시차를 넣은 뒤에도 마지막 값의 이득이 남는가 |

*모든 구성은 동일한 common 학습 행과 개발 4 ~ 6월 2,184시간에서 비교했다. 하이퍼파라미터를 다시 선택하거나 예측값을 절단하지 않았다. 입력 구성별로 다시 학습하므로 트리 구조도 달라지며, 이를 마지막 값 자체의 인과효과나 변수 중요도로 해석하지 않는다.*

### 마지막 값의 이득은 전체와 실제 최대 시간에서 확인됐다

| 입력 | 최대전력 MAE | RMSE | 전체 평균 과소예측 | 전체 평균 과대예측 | 보조 평균 목표 MAE |
|---|---:|---:|---:|---:|---:|
| A | 6.74 | 10.95 | 3.43 | 3.31 | 5.97 |
| B | **6.19** | **9.78** | 2.87 | 3.32 | **5.49** |
| C_A | 7.05 | 11.74 | 3.52 | 3.53 | 6.27 |
| C_B | 6.23 | 10.02 | **2.85** | 3.38 | 5.59 |

A→B에서 최대전력 MAE는 **8.1% 감소**했다. C_A→C_B에서도 **11.6% 감소**해 마지막 값의 추가 가치는 전날·전주 입력이 있을 때도 남았다. 보조 평균 목표 역시 두 비교에서 개선됐다. 반면 전날·전주 입력 자체는 전체 오차를 안정적으로 줄이지 못했다. C_A는 A보다, C_B는 B보다 전체 MAE가 컸다.

실제 일별 최대 시간에서도 마지막 값의 추가 효과를 확인했다. M02와 같은 91일의 최대 시간 137개 행에 동률 가중치를 적용했으며, 실제 최대값의 하루 평균은 약 150.81이었다. 이 집계에는 낮은 전력의 날의 상대적 최대도 들어 있으므로 모든 최대 시간을 운영상 위험한 피크로 간주하지 않는다.

| 입력 | 최대 시간 MAE | 평균 과소예측량 | 평균 과대예측량 |
|---|---:|---:|---:|
| A | 11.75 | 11.26 | 0.49 |
| B | **7.59** | **6.99** | 0.60 |
| C_A | 11.78 | 11.05 | 0.73 |
| C_B | 8.18 | 7.05 | 1.13 |

B의 최대 시간 MAE는 A보다 **35.4% 감소**했고 평균 과소예측은 **37.9% 감소**했다. 대신 평균 과대예측은 0.49→0.60으로 증가했고, 전체 시간의 평균 과대예측도 3.309→3.318로 조금 늘었다. 따라서 과소예측을 줄였다는 근거와 과대예측의 증가를 함께 제시한다. 평균 과소예측량은 해당 방향의 오류가 없는 시간에도 0을 포함한 평균이며, 발생 횟수의 감소율이 아니다. 감소율은 표에 반올림해 표시한 값이 아니라 저장된 원래 지표에서 계산했다.

![마지막 값·계절 시차 입력의 월별 성능과 실제 최대 시간 오류](Modeling/figures/m03/last_slot_ablation.png)

이미지 파일: [Modeling/figures/m03/last_slot_ablation.png](Modeling/figures/m03/last_slot_ablation.png). 생성 코드: [Modeling/scripts/plot_m03_manuscript.py](Modeling/scripts/plot_m03_manuscript.py)의 `plot_ablation()`.

*왼쪽은 4·5·6월 최대전력 MAE다. 실선 A/B와 점선 C_A/C_B를 비교하면 마지막 값 추가의 월별 방향과 계절 시차의 상충을 볼 수 있다. 오른쪽은 실제 일별 최대 시간의 평균 과소예측(파랑)·과대예측(주황)이며 막대 위 숫자는 합계 MAE, 파란 막대 안 숫자는 평균 과소예측량이다. 왼쪽은 시간별 평균, 오른쪽은 하루 최대 동률의 날짜 가중 평균이므로 분모가 다르다.*

### 월별 개선과 반복 배열의 영향을 구분했다

| 입력 | 4월 MAE | 5월 MAE | 6월 MAE | 학습과 같은 배열 없음 | 같은 배열 있음 | 반복 배열 역빈도 가중 |
|---|---:|---:|---:|---:|---:|---:|
| A | 8.31 | 6.13 | 5.80 | 7.56 | 5.82 | 6.39 |
| B | **7.14** | 5.77 | 5.67 | **6.75** | 5.56 | 5.96 |
| C_A | 9.44 | 6.52 | 5.20 | 8.34 | 5.61 | 6.69 |
| C_B | 8.21 | **5.65** | **4.86** | 7.14 | **5.22** | **5.95** |

*같은 배열이 없는 기록은 1,152시간, 있는 기록은 1,032시간이다. 역빈도 가중은 평가 안의 같은 하루 배열에 속한 날짜 수의 역수로 가중한 보조 결과이며 다른 열과 합산하지 않는다.*

A→B와 C_A→C_B 모두 각 개발 월에서 MAE가 작아졌다. A→B의 개선율은 4월 14.0%, 5월 5.9%, 6월 2.2%로 월마다 차이가 컸다. 날짜별로는 58일에서 좋아지고 33일에서 나빠졌으며, 시간별 절대오차는 1,100시간 개선·1,084시간 악화였다. 따라서 전체 평균 개선을 모든 시간에 안정적으로 적용되는 개선으로 확대하지 않는다.

학습과 직접 같은 배열이 없는 기록에서도 마지막 값의 이득이 남았다. 이는 직접 같은 배열만 맞혀 생긴 개선이라는 설명을 지지하지 않는다. 다만 배열 구분별 월·날짜 구성은 다르고 6월의 새로운 배열은 4일뿐이다. 독립성이나 새로운 운영 기간의 안정성을 입증한 결과로 표현하지 않는다.

C_B의 반복 배열 가중 MAE는 B보다 약 0.012 작지만 차이는 약 0.2%였다. 전체와 최대 시간에서는 B가 더 좋고, 가중치에 따라 작은 순위 차이가 생기는 점을 남겼다. 한 보조 지표만 골라 계절 시차를 채택하지 않았다.

### 최대 시간 개선과 급상승 개선은 같은 결과가 아니었다

| 전력 조건 | 시간 수 | A MAE | B MAE | C_A MAE | C_B MAE |
|---|---:|---:|---:|---:|---:|
| 낮은 구간→26 초과 | 20 | 47.87 | 48.77 | 45.18 | 44.77 |
| 26 초과→낮은 구간 | 19 | 28.39 | 15.64 | 35.88 | 13.79 |
| 낮은 구간 유지 | 625 | 3.77 | 4.11 | 4.47 | 4.62 |
| 26 초과 유지 | 1,510 | 7.18 | 6.40 | 7.27 | 6.31 |

*낮은 구간은 기존 A06의 정수 평균 20 ~ 26이다. 26 초과는 현장의 고전력 기준이 아니며 목표 시간의 실제 상태를 이용한 사후 진단이다. 나머지 10시간은 양수 20 미만과의 전환이다.*

마지막 값 추가는 하락 전환과 26 초과 유지에서 오차를 줄였다. 그러나 낮은 구간에서 올라간 20시간은 B에서도 모두 과소예측이고 오차가 47.87→48.77로 늘었다. 낮은 구간 유지도 3.77→4.11로 나빠졌다. C_B는 상승 조건을 일부 개선했지만 여전히 모두 과소예측이며 낮은 구간 유지에서는 더 나빴다. **일별 최대 시간의 과소예측 개선을 낮은 전력에서 급상승하는 모든 시간의 개선으로 대신할 수 없었다.**

B의 전체 절대오차 중 26 초과 유지가 71.5%, 낮은 구간 유지가 19.0%, 상승이 7.2%, 하락이 2.2%였다. 전환의 한 시간당 오차는 크지만 전체 오류에는 유지 조건도 크게 기여했다. 기여율에는 시간 수가 반영되며 위험도나 운영 손실 비중이 아니다. 월·생산량 전환 등 겹치는 구분의 기여를 합산하지 않았다.

전력 0은 이번 개발 기간에 없었다. 따라서 EDA에서 확인한 8월의 17시간 연속 0과 그 전후의 예측성능은 이번 비교에서 평가하지 않았다. 미래의 실제 상태로 B와 단순 기준을 바꿔 쓰는 방식도 사용하지 않았다.

최대 시간의 개선과 급상승의 악화가 함께 나타난 이유를 해석하기 위해 저장된 예측으로 두 집합의 겹침을 확인했다. **급상승 20시간은 20개의 서로 다른 날짜에 있었고, 실제 일별 최대 시간과 겹친 시간은 0개였다.** 최대 시간 137개 행은 모두 구간 유지에 속했다. 따라서 두 결과는 같은 사건을 다르게 평가한 것이 아니었다.

| 실제 일별 최대 시간의 전력 조건 | 최대 시간 행 수 | 날짜 가중치 합 | A 평균 과소예측 | B 평균 과소예측 |
|---|---:|---:|---:|---:|
| 26 초과 유지 | 84 | 79 | 12.69 | 7.75 |
| 낮은 구간 유지 | 53 | 12 | 1.87 | 1.96 |

*최대가 동률인 하루가 있어 행 수와 날짜 가중치 합은 다르다. 두 조건의 가중치 합은 전체 91일이며, 위 수치는 조건 안의 날짜 가중 평균이다.*

최대 시간의 과소예측 개선은 주로 26 초과가 유지되는 날의 최대 시간에서 나타났다. 낮은 구간에 머문 날의 상대적 최대에서는 과소예측이 조금 늘었다. 이 결과에 따라 필살기의 설명은 ‘급상승을 해결한 예측’보다 ‘주로 26 초과 유지 조건의 일별 최대 시간 과소예측을 줄인 개발 결과’로 구체화한다. 26 초과라는 상태 구분을 절대적인 위험 기준으로 바꾸지는 않는다.

예측 시점에 아는 직전 상태로만 나눠도 한계가 남았다. 직전 낮은 구간 650시간의 MAE는 A 5.12→B 5.47로 악화됐고, 직전 26 초과 1,529시간에서는 7.45→6.51로 개선됐다. 직전 낮은 상태에는 다음 시간의 유지와 상승이 함께 포함되므로 평균 편향만 보정하면 두 조건을 동시에 해결할 수 있다고 결론 내리지 않는다. 이 최소 진단은 재학습 없이 수행했고 [직전 상태 오류](Modeling/tables/m03/diagnostics/prior_state_errors.csv)·[최대 시간의 전력 조건](Modeling/tables/m03/diagnostics/maximum_transition_errors.csv)·[급상승 사례](Modeling/tables/m03/diagnostics/rise_cases.csv)에 보존했다.

### 입력 선택과 후속 계획의 변경

M03에서 선택한 개발 후보는 **B: 기본 입력에 마지막 15분 값만 추가한 HGB**였다. 같은 설정에서 전체·월별·최대 시간의 개선이 확인됐고 C_B보다 입력 수가 적으며 전체와 최대 시간의 오차도 작았다. 계절 시차는 6월에 유리했지만 4월과 학습에 없는 배열에서 B보다 나빠 기본 구성에 추가하지 않았다. 이는 당시 입력·설정에서의 선택이며, 4.9의 후반기 결과와 4.12의 새 비교 조건까지 포함한 최종 우위를 뜻하지 않는다.

M03에서 확인한 성과는 **직전 마지막 값을 추가했을 때 다음 최대전력의 전체 오차와 실제 일별 최대 시간의 과소예측이 줄어든 개발 결과**다. 이후 4.9에서는 이 효과가 후반기 최대 시간까지 유지되지 않았음을 확인했다. 따라서 마지막 값 하나의 효과를 현재 보고서의 주된 차별성으로 제시하지 않고, 기본 입력을 선택한 근거와 남은 오류를 설명하는 결과로 사용한다.

M03은 전력 크기의 기본 예측을 구성하는 단계다. M04에서는 상승 전환과 낮은 상태에서 남은 오류를 분석하고 보완 규칙을 비교했으며, M05에서는 당시 후보를 고정해 후반기를 확인했다. 이 과정에서 전력 크기의 오차 감소와 상승 시작을 미리 찾는 능력을 구분할 필요가 드러났다. 그 질문을 이어받은 지속 상승 정의·분류·선택적 HGB 보완은 4.12에 정리한다.

과거 날씨 D는 이번 회차에 넣지 않았다. Analysis 3.3의 조건부 관계가 약했고, 현재 오류 결과에서 날씨 추가가 답할 구체적인 새 질문은 확인되지 않았다. 날씨의 예측 가치가 없다고 검증한 것은 아니며, 근거 없이 후보를 늘리는 대신 상승 과소예측과 낮은 구간 과대예측의 진단을 우선한다.

직전 상태와 최대 시간의 겹침 질문은 위 최소 진단으로 확인했다. 이어진 M04의 질문은 **직전 낮은 구간에서만 직전 기준을 쓰는 규칙이 유지 오차를 줄이면서 상승 과소예측을 얼마나 늘리는가**였다. 4.6에서 이 규칙을 실제 비교하고 상충 때문에 기각했다. 이후에도 가동 원인을 추정으로 채우지 않고 관측 가능한 전력 전환을 정의하는 방향으로 후속 분석을 진행했다.

제출 보고서 제2장에는 동일 조건의 입력 비교와 당시 B 선택 이유를, 제3장에는 상승·낮은 유지에서 남은 오류와 최대 시간 개선의 조건을 연결한다. 제5장의 현재 차별성은 4.12의 지속 상승 경고와 선택적 보완을 중심으로 설명한다. M03의 개발 개선률을 후속 방법의 개선률과 합산하지 않는다.

### 재현과 검증 근거

A/B에서 B 6회, C 비교에서 C_A·C_B 12회로 새 학습 18회를 수행했다. A는 M02 모델을, C 비교의 A/B는 첫 회차의 저장 모델을 재사용했다. 모든 재사용 모델은 해시·설정·입력 이름을 확인하고 동일 예측을 전수 대조했으며, 새 모델도 저장 후 다시 불러와 예측을 확인했다.

독립 검증은 표준 CSV·수학 계산으로 A/B의 8,736예측행·4,368비교행·568지표행, C의 17,472예측행·17,472비교행·1,136지표행을 재계산했다. 각 회차에서 원본과 20,880시차값, 동일 2,184평가 시각, 91일의 최대 시간 동률 가중치, 결과표·모델 해시를 확인했다. 모두 종료 코드 0이며, 7 ~ 8월은 학습·평가에 사용하지 않았다. C 통합표의 A/B는 재사용 결과여서 두 회차를 독립 표본처럼 합산하지 않는다.

근거: [A/B 조건](Modeling/config/m03_ab_contract.json) · [C 조건](Modeling/config/m03_c_contract.json) · [전체·월별 비교](Modeling/tables/m03/c/comparison.csv) · [조건별 오류](Modeling/tables/m03/c/condition_errors.csv) · [동일 시간 예측 비교](Modeling/tables/m03/c/paired_predictions.csv) · [조건별 변화](Modeling/tables/m03/c/paired_deltas.csv) · [날짜별 변화](Modeling/tables/m03/c/daily_deltas.csv) · [학습 점검](Modeling/tables/m03/c/training_audit.csv) · [모델 목록](Modeling/tables/m03/c/model_manifest.csv) · [실행환경·해시](Modeling/tables/m03/c/run.json) · [독립 검증](Modeling/tables/m03/c/independent_verification.json).

**관련 Python 코드**

- [Modeling/scripts/m03_ablation.py](Modeling/scripts/m03_ablation.py): `main()`이 회차별 조건·부모 해시를 확인하고 고정 HGB의 입력 비교를 수행한다. `subsets()`는 M02와 같은 전체·월·전력/생산 전환·배열·최대 시간 진단을 구성한다. 같은 시각의 절대오차·과소·과대예측 차이를 함께 저장한다. 지표는 M02의 `metric()`을 재사용한다.
- [Modeling/scripts/m03_verify.py](Modeling/scripts/m03_verify.py): 원본에서 마지막 값·전날·전주 입력을 다시 조회하고, 목표값·같은 평가 시각·최대 시간 가중치·오차지표와 전후 차이를 독립 재계산한다.
- [Modeling/scripts/m03_error_followup.py](Modeling/scripts/m03_error_followup.py): 저장된 예측으로 직전 상태별 오류와 상승·실제 최대 시간의 겹침을 확인한다. 추가 학습 없이 산술 검증과 10개 지표의 독립 재계산을 수행하며 [진단 기록](Modeling/tables/m03/diagnostics/summary.json)에 남긴다.
- [Modeling/scripts/plot_m03_manuscript.py](Modeling/scripts/plot_m03_manuscript.py): `load_verified()`가 검증 상태·해시를 확인하고 `plot_ablation()`이 월별·최대 시간 그림을 만든다. 학습을 다시 수행하지 않으며 그림은 50% 크기로 실제 확인했다.
- 공통 모듈 [m01_prepare.py](Modeling/scripts/m01_prepare.py)의 `ROOT`, `sha()`, `read_contract()`, `feature_columns()`와 [m02_compare.py](Modeling/scripts/m02_compare.py)의 `metric()`, `state()`, 초기 후보 구성도 M03 실행에 필요하다. 주 실행 파일과 함께 전체 소스를 보존한다.
- [verify_m03_documents.py](Modeling/scripts/verify_m03_documents.py)는 본문의 비교표·세 핵심 감소율·그림 해시·링크를 확인한다. 전체 제출 코드와 보존 파일·실행 순서는 본 원고 4.5와 [보존 목록](Modeling/submission_manifest.json)에 연결했다.
- 회차별 선택 이유·종료 근거·재현 순서는 [M03 실행 기록](Modeling/10.04_M03_마지막값추가와시차입력_실행기록.md)에 있다. 상세 원고만 확장했으며 압축본은 작성하지 않았다.


## 4.4 M02 확장 재실행: Optuna·시간순 검증·앙상블 비교 후의 판단

### 기존 HGB를 다른 모델과 다시 비교한 이유

4.2의 M02에서는 HGB가 기본 A 입력에서 가장 좋았고, 4.3의 M03에서는 마지막 값을 추가한 B의 개선을 확인했다. 그러나 이 비교만으로 XGBoost·LightGBM·CatBoost나 앙상블을 포함한 후보 중에서도 HGB를 유지할 수 있는지는 확인하지 못했다. 또한 상승 전환과 낮은 구간 유지의 오류가 남아 있었으므로, 모델을 바꾸면 그 오류가 줄어드는지도 함께 확인할 필요가 있었다.

사용자 요청에 따라 M02 자체를 확장 재실행했다. 모델 차이와 입력 효과를 분리하기 위해 **마지막 값이 없는 기본 A 입력에서** 다시 비교했고, 선택 모델이나 설정이 바뀌면 그 설정으로 M03을 반복하도록 정했다. 이전 M02·M03은 당시의 결과와 판단을 보여주는 근거로 유지한다. 이미 개발 결과를 보고 정한 확장 설계이므로 이번 점수를 새로운 독립 시험의 성적으로 표현하지 않는다.

### 학습과 예측 구간: 모든 적격 시간의 순차 비교

| 외부 개발 평가 | 학습 기간 | 예측·평가 기간 | 학습 시간 | 평가 시간 |
|---|---|---|---:|---:|
| 1차 | 1~3월 | 4월 전체 | 1,992 | 720 |
| 2차 | 1~4월 | 5월 전체 | 2,712 | 744 |
| 3차 | 1~5월 | 6월 전체 | 3,456 | 720 |

이 표의 시간 수는 정확한 1·2·24·168시간 시차가 모두 있는 common 조건의 수다. 평가월의 유리한 날만 고른 것이 아니라 조건을 충족하는 모든 시각 2,184개를 같은 순서로 비교했다. 1~3월은 학습과 내부 검증에 쓰이고, 4~6월은 모델·설정의 개발 선택에 쓰인다. 후속 M05에는 1~6월 학습→7~8월 평가를 남겼다. 7~8월은 이번 실행의 튜닝·선택에 쓰지 않았지만 과거 관찰 이력이 있어 완전히 보지 않은 독립 시험으로 부르지 않는다. 9월은 기존 분석 범위 밖이므로 편입하지 않는다.

각 외부 평가월의 설정은 그 이전 자료 안에서만 선택했다. 예를 들어 4월 평가의 Optuna는 `1월 학습→2월 검증`과 `1~2월 학습→3월 검증`을 사용한다. 5월 평가에는 4월 내부 검증이, 6월 평가에는 5월 내부 검증이 추가된다. 각 내부 학습 시각은 검증 시각보다 이르고 내부 검증의 마지막 시각은 외부 평가보다 이르다. 시각 오류로 생긴 공백 때문에 단순한 행 개수 분할 대신 실제 달력 경계를 사용했다.

외부 월이 시작되면 선택한 설정으로 이전 자료 전체를 학습하고 그 월 동안 모델을 고정한다. 관측이 끝난 입력은 매시간 갱신한다. 따라서 이 결과는 **매시간 다음 한 시간을 예측한 결과**이며, 월 전체를 월초에 한 번에 예측하거나 하루 최대 시각을 미리 지정한 결과가 아니다.

### 모델과 튜닝 범위

핵심 튜닝 대상은 HGB·XGBoost·LightGBM·CatBoost·ExtraTrees의 5계열이다. 부스팅의 서로 다른 학습 방식과 무작위 분할 트리 평균을 비교한다. 기존 Ridge·Elastic Net·RBF SVR·HGB는 초기 결과의 참조로 유지하며 직전·전날·전주 단순 기준을 포함한다. 선형·커널 참조 모델을 이번 Optuna에서 다시 튜닝하지는 않았다. 따라서 모든 계열에 동일한 탐색량을 부여한 대규모 최적화라는 주장은 하지 않는다.

각 계열과 외부 월 조합에 기본 30회, 내부 선택 점수가 최초 20회의 최선보다 1% 이상 개선된 경우에만 50회 탐색했다. 학습률·트리 수·깊이 또는 잎 수·규제 범위는 [실행 전 조건](Modeling/config/m02_rerun_contract.json)에 고정했다. 외부 결과를 보고 탐색 범위나 횟수를 늘리지 않았다. 각 시도는 모든 내부 월을 검증했고, 외부 평가값으로 조기 종료하거나 예측값을 사후 보정하지 않았다.

앙상블은 튜닝한 5계열의 동일 가중 평균과 내부 검증 점수 상위 3계열의 비음수 가중 평균을 비교했다. 가중 합은 1이고 구성원·가중치는 외부 평가 전에 내부 예측만으로 선택했다. 내부 예측은 설정·가중치 선택에 사용했으므로 독립적인 성능 추정치로 해석하지 않는다. 복잡한 스태킹은 이번 비교에 포함하지 않았다.

실제 실행은 Optuna 총 700회, 내부 모델 학습 1,630회, 외부 모델 학습 15회였다. 모든 시도는 정규 CPython 3.13에서 수행됐고 총 실행 시간은 약 5.6분이었다. 기존 참조 예측은 해시를 대조한 후 재사용했다.

### 전체 정확도와 최대 시간 정확도의 공동 평가

전체 시간 MAE와 **실제 일별 최대 시간 MAE**를 두 주지표로 사용했다. 후자는 예측이 끝난 뒤 실제 일별 최대가 나타난 시각의 수치 오차를 평가하며, 같은 날 최대가 여러 번이면 그 날의 합계 가중치를 1로 나눠 91일을 같은 비중으로 비교한다. 내부 검증에서는 적격 24시간이 모두 있는 날짜만 이 지표에 포함한다. 실제 최대 시각은 사후 평가용이며 입력에 들어가지 않는다.

두 목적값을 Optuna에 각각 전달하고 어느 한 지표를 개선하려면 다른 지표를 양보해야 하는 파레토 후보를 보존했다. 최종 한 후보를 정할 때는 사전에 다음 규칙을 기록했다.

`선택 점수 = max(전체 MAE / 직전 기준 전체 MAE, 최대 시간 MAE / 직전 기준 최대 시간 MAE)`

이 규칙은 두 상대오차 중 더 큰 쪽을 줄이는 절충이다. 분모는 항상 같은 기록의 직전 값 기준이다. 현장 손실비용을 알 수 없는 상태에서 임의의 비용 가중치를 만들지 않고, 두 원래 지표도 각각 공개했다. 최종 개발 선택에서는 다른 방법에 두 지표 모두 뒤지는 후보를 제외하고, 최선 점수의 1% 이내이면 기존 방법과 단일 모델을 우선한다. 실측한 운영 손실함수나 최적의 경제적 판단을 뜻하지는 않는다.

주 목표는 전력 크기의 연속값이므로 MAE와 RMSE로 크기 오차를 평가한다. AUROC·AUPRC는 고전력 사건의 정의와 예측 점수가 있는 분류·순위 문제에서 따로 평가할 수 있다. 이번에는 분류 기준을 새로 만들지 않았으며, 최대 시간 MAE가 낮다는 사실을 최대 시각 탐지 성과로 바꾸어 해석하지 않는다.

### 전체 비교 결과와 HGB 유지 판단

| 방법 | 전체 MAE | 실제 일별 최대 시간 MAE | 선택 점수 |
|---|---:|---:|---:|
| 직전 값 | 13.793 | 36.589 | 1.0000 |
| 전날 같은 시각 | 43.360 | 37.903 | 3.1436 |
| 전주 같은 시각 | 19.784 | 19.683 | 1.4344 |
| 기존 Ridge | 14.455 | 36.130 | 1.0480 |
| 기존 Elastic Net | 14.419 | 37.395 | 1.0454 |
| 기존 RBF SVR | 14.260 | 32.000 | 1.0338 |
| 기존 HGB | 6.738 | 11.751 | 0.4885 |
| Optuna HGB | 6.963 | 13.251 | 0.5048 |
| XGBoost | 7.131 | 12.864 | 0.5170 |
| LightGBM | 7.494 | 15.756 | 0.5433 |
| CatBoost | 7.480 | 17.794 | 0.5423 |
| ExtraTrees | 6.825 | 11.540 | 0.4948 |
| 5모델 동일 평균 | 6.905 | 14.091 | 0.5006 |
| 내부 상위 3모델 가중 평균 | 6.872 | 13.035 | 0.4982 |

*학습 모델은 기본 A 입력을 사용하며, 단순 기준을 포함한 모든 방법은 같은 2,184시간을 평가한다. 최대 시간 지표는 동률을 나눈 91일 가중 평균이다. 기존 모델 행은 과거 개발에서 선택한 설정의 보존 참조이며, 새 5계열은 각 외부 월 이전 내부 검증으로 설정을 선택했다.*

정한 규칙의 최선은 **기존 HGB**였다. ExtraTrees는 전체 MAE가 6.825로 기존 6.738보다 약 1.3% 컸지만 최대 시간 MAE는 11.540으로 기존 11.751보다 약 1.8% 작았다. 두 방법만 서로 한쪽 지표에서 앞서는 파레토 후보로 남았다. 기존 HGB의 선택 점수 0.4885가 ExtraTrees의 0.4948보다 작아 기존 모델과 설정을 유지했다. 이 자료에서는 두 방법 모두 최대 시간의 상대오차보다 전체 상대오차가 커서, 최종 절충 점수의 결정 요인은 전체 MAE였다. **최대 시간만을 가장 우선한다면 ExtraTrees를 선호할 수 있으므로 목적에 따른 차이를 함께 밝힌다.**

Optuna HGB·XGBoost·LightGBM·CatBoost와 두 앙상블은 이번 범위에서 기존 HGB보다 두 주지표가 모두 컸다. 따라서 알고리즘의 이름이나 복잡성을 이유로 교체하지 않았다. 이는 이번 입력·기간·탐색 범위의 결론이며, 해당 모델의 모든 설정이나 입력에서 기존 HGB가 더 좋다는 뜻은 아니다.

### HGB의 하이퍼파라미터 비교와 유지할 설정

이번 기본 A 입력·개발 기간·선택 규칙을 종합하면 **기존 HGB를 유지하는 것이 가장 적절했다.** 전체 MAE 6.738과 RMSE 10.954는 비교 방법 중 가장 작았다. 다만 일별 최대 시간 MAE만 보면 ExtraTrees가 11.540으로 기존 HGB 11.751보다 조금 작았으므로, HGB가 모든 지표에서 가장 좋았다고 표현하지 않는다.

HGB의 설정 선택은 두 차례로 구분한다. 초기 M02에서는 최대 잎 수 7과 15의 두 후보를 비교했고 나머지 설정은 고정했다. 이 제한된 후보 비교에서 잎 수 15를 선택했다. 이후 확장 M02에서는 HGB도 Optuna 대상에 포함해 학습률·반복 횟수·최대 잎 수·잎의 최소 표본 수·L2 규제를 함께 탐색했다. 각 외부 평가월 이전의 시간순 내부 검증을 사용했고, 실제 HGB 탐색은 4·5·6월용 각각 30회씩 총 90회였다.

| 설정 선택 단계 | 전체 MAE | 일별 최대 시간 MAE | 판단 |
|---|---:|---:|---|
| 초기 두 후보 비교에서 선택한 기존 HGB | 6.738 | 11.751 | 기존 설정 유지 |
| 시간순 내부 검증으로 Optuna 튜닝한 HGB | 6.963 | 13.251 | 이번 개발 비교에서 교체하지 않음 |

유지할 기존 HGB의 설정은 다음과 같다.

| 하이퍼파라미터 | 유지할 값 |
|---|---:|
| 최대 잎 수 (`max_leaf_nodes`) | 15 |
| 반복 횟수 (`max_iter`) | 150 |
| 학습률 (`learning_rate`) | 0.05 |
| 잎의 최소 표본 수 (`min_samples_leaf`) | 20 |
| L2 규제 (`l2_regularization`) | 1.0 |

재현 조건은 난수 시드 42, 조기 종료 미사용이다. 기존 설정은 초기 4~6월 개발 결과로 선택했고, Optuna 설정은 각 평가월 이전의 내부 검증으로 선택했다. 이 선택 이력의 차이를 함께 밝히며, 튜닝의 우열이나 HGB의 일반적인 우수성을 독립적으로 입증한 결과로 표현하지 않는다. 현재 결론은 **개발 후보와 설정을 유지한다**는 것이며 후반기 확인은 남아 있다. 근거는 [초기 설정](Modeling/config/m02_contract.json), [HGB를 포함한 모든 Optuna 시도](Modeling/tables/m02_rerun/trials.csv), [월별 선택 설정](Modeling/tables/m02_rerun/selected_configurations.json)에 있다.

### 월별 변화와 남은 실패조건

| 방법 | 4월 MAE | 5월 MAE | 6월 MAE | 학습에 없는 배열 MAE |
|---|---:|---:|---:|---:|
| 기존 HGB | 8.308 | 6.129 | 5.797 | 7.558 |
| ExtraTrees | 8.521 | 6.118 | 5.861 | 7.679 |
| 내부 상위 3모델 가중 평균 | 8.974 | 5.990 | 5.681 | 7.957 |

가중 앙상블은 5·6월 전체 MAE를 기존 HGB보다 낮췄지만 4월에는 8.974로 기존 8.308보다 컸다. 학습에 없는 하루 배열 1,152시간에서도 7.957로 기존 7.558보다 컸다. 4월은 모든 평가 날짜가 학습에 없는 배열이라는 M01 결과와 함께 보면, 후반 개발월의 개선만으로 전체 안정성이 좋아졌다고 주장하기 어렵다. 이 차이를 배열 반복이 오류의 원인이라는 인과 설명으로 단정하지는 않는다.

| 조건 | 평가 시간 | 기존 HGB MAE | ExtraTrees MAE |
|---|---:|---:|---:|
| 낮은 구간→26 초과 | 20 | 47.866 | 60.550 |
| 26 초과→낮은 구간 | 19 | 28.394 | 32.245 |
| 낮은 구간 유지 | 625 | 3.766 | 5.162 |
| 26 초과 유지 | 1510 | 7.183 | 6.477 |

ExtraTrees는 26 초과 유지에서 MAE를 낮췄으나 상승 전환 20시간에서는 60.550으로 기존 47.866보다 컸고 낮은 구간 유지 625시간에서도 5.162로 기존 3.766보다 컸다. 최대 시간에서 평균 과소예측은 기존 11.260→10.897로 줄었지만 평균 과대예측은 0.491→0.643으로 늘었다. 최대 시간 MAE의 작은 개선만으로 급상승·낮은 유지 문제까지 해결했다고 판단할 수 없다.

### 기존 M03과의 연결 및 다음 질문

모델과 설정을 바꾸는 조건이 충족되지 않아 **M03을 다시 실행하지 않았다.** 기존 HGB의 A→B·C 비교는 같은 모델·설정·표본의 검증으로 계속 유효하다. 현재 후보는 그 비교에서 선택한 B이며, 최대전력 MAE 6.191과 최대 시간 MAE 7.589라는 기존 결과를 유지한다. 이번 확장 M02는 A 입력의 모델 비교이므로 모든 모델에 마지막 값을 추가했을 때의 순위를 검증한 결과는 아니다. 특히 ExtraTrees B가 HGB B보다 나쁘다는 결론은 낼 수 없다.

후속 우선순위는 기존 M03의 남은 질문으로 돌아간다. 직전 낮은 상태는 예측 시점에 알 수 있고, B에서는 해당 650시간의 MAE가 5.466으로 A 5.117보다 컸다. 다음 최소 M04는 그 조건에서 직전 값 기준을 사용하는 규칙이 낮은 유지 오차를 줄이는 대신 상승 과소예측을 얼마나 늘리는지 비교하는 것이다. 규칙과 허용 범위를 먼저 정하고 같은 시간의 전체·최대 시간·상승 오류로 채택 또는 기각한다. 이번 회차에서는 규칙과 M05를 실행하지 않았다. 새로운 근거 없이 모델 탐색만 계속 늘리지 않는다.

### 실행 코드와 검증 근거

- [M02 확장 실행 기록](Modeling/10.04_M02_확장모델비교_실행기록.md), [실행 전 조건](Modeling/config/m02_rerun_contract.json)
- [학습·시간순 내부 검증·두 목적 튜닝·앙상블 실행 코드](Modeling/scripts/m02_rerun.py), [모델 생성과 탐색 보조 코드](Modeling/scripts/m022_compare.py)
- [전체 비교표](Modeling/tables/m02_rerun/comparison.csv), [두 주지표와 선택 점수](Modeling/tables/m02_rerun/selection_metrics.csv), [조건별 오류](Modeling/tables/m02_rerun/condition_errors.csv)
- [월별 설정과 앙상블 가중치](Modeling/tables/m02_rerun/selected_configurations.json), [Optuna 모든 시도](Modeling/tables/m02_rerun/trials.csv), [시간순 분할 감사](Modeling/tables/m02_rerun/fold_audit.csv)
- [시각별 예측](Modeling/tables/m02_rerun/predictions.csv), [개발 선택 판단](Modeling/tables/m02_rerun/selection.json), [실행 환경·해시](Modeling/tables/m02_rerun/run.json)
- [독립 검증 코드](Modeling/scripts/m02_rerun_verify.py), [독립 검증 결과](Modeling/tables/m02_rerun/independent_verification.json)

독립 검증은 표준 라이브러리로 원본 시차값 33,408개, 예측 30,576행, 지표 1,988행, 튜닝 700회, 설정·앙상블 21개를 대조했다. 시간순 학습 경계, 동일 2,184시각, 91일 최대 가중치, 저장 모델 해시·앙상블 계산·최종 선택의 일치가 확인됐고 종료 코드는 0이었다. 원고 압축과 후반기 평가는 별도 단계로 남긴다.

## 4.5 제출 코드와 재현 파일의 보존

이 원고에 사용한 Python 파일의 전체 소스는 아래 링크의 실제 `.py` 파일에 보존한다. 최종 제출에는 공통 모듈·원본 조회·검증 절차까지 같은 폴더 구조로 포함한다. 아래 표는 M01~M05와 초기 재개발의 코드이며, 후속 분석의 추가 코드는 각 절과 4.12의 코드표에 연결했다. 현재 전체 보존 범위와 의존 관계는 `submission_manifest.json`을 기준으로 확인한다.

### 사용한 Python 코드 전체 목록

| 파일 | 원고와의 연결·역할 | 주요 함수 또는 실행 방식 |
|---|---|---|
| [m01_prepare.py](Modeling/scripts/m01_prepare.py) | 4.1 원본 범위·목표·정확한 시차·공통 표본·평가 가중치 구성. 다른 학습 코드도 이 모듈을 불러온다. | `load_frame()`, `feature_columns()`, `read_contract()`, `independent_check()`, `main()` |
| [m01_verify.py](Modeling/scripts/m01_verify.py) | 4.1 저장 전처리 자료·원본·시차·분할·해시의 독립 검증 | `rows()`, `main()` |
| [m02_compare.py](Modeling/scripts/m02_compare.py) | 4.2 기본 모형 비교·학습 전처리·예측·오류·저장 모델. M03과 확장 비교의 공통 지표도 제공한다. | `candidates()`, `state()`, `metric()`, `main()` |
| [m02_verify.py](Modeling/scripts/m02_verify.py) | 4.2 초기 후보 예측·지표·원본 단순 기준·최대 시간 가중치·모델 해시 검증 | `main()` |
| [m03_ablation.py](Modeling/scripts/m03_ablation.py) | 4.3 같은 HGB 설정의 A/B·C_A/C_B 입력 비교·전후 오차. 확장 M02도 오류 조건과 시간 함수를 불러온다. | `now()`, `subsets()`, `main()`; `--stage ab` 또는 `--stage c` |
| [m03_verify.py](Modeling/scripts/m03_verify.py) | 4.3 입력·목표·같은 시각·전후 차이의 독립 검증. 확장 검증과 문서 생성·검증도 이 모듈의 CSV·산술 함수를 불러온다. | `rows()`, `sha()`, `close()`, `select()`, `verify_metric()`, `main()` |
| [m03_error_followup.py](Modeling/scripts/m03_error_followup.py) | 4.3 직전 상태 오류와 상승·실제 최대 시간의 겹침 확인·10개 지표 재계산 | `main()`; 추가 학습 없음 |
| [plot_observed_power_timeline.py](Modeling/scripts/plot_observed_power_timeline.py) | M01 이후 관측 전력·시간 변화량·0 구간 그림 3개. EDA·Analysis 원고에 연결된 관측 근거의 생성 코드 | `main()`; M01 저장 자료만 사용 |
| [plot_m02_manuscript.py](Modeling/scripts/plot_m02_manuscript.py) | 4.2 모델 비교·조건 오류 그림 2개와 50% 검토 이미지 생성 | `load_verified()`, `plot_comparison()`, `plot_conditions()`, `main()` |
| [plot_m03_manuscript.py](Modeling/scripts/plot_m03_manuscript.py) | 4.3 마지막 값·계절 시차 비교 그림 1개와 50% 검토 이미지 생성 | `load_verified()`, `plot_ablation()`, `main()` |
| [m022_environment.py](Modeling/scripts/m022_environment.py) | 확장 논의 당시 정규 Python·패키지 import 점검에 사용한 코드. 현재 확장 실행은 자체 환경 정보를 별도로 기록한다. | 모듈 실행으로 import 결과 저장; 모델 학습 없음 |
| [m022_compare.py](Modeling/scripts/m022_compare.py) | 4.4에서 실제 불러오는 5계열 모델 생성·탐색 범위·오류 요약 공통 모듈 | `make_model()`, `suggest()`, `add_errors()`, `summarize()`를 import한다. 내부의 과거 B 입력 `main()`은 현재 재현 순서에서 실행하지 않는다. |
| [m02_rerun.py](Modeling/scripts/m02_rerun.py) | 4.4 A 입력·시간순 중첩 검증·두 목적 Optuna·앙상블·최종 개발 선택 | `peak_weights()`, `objectives()`, `balance()`, `choose_trial()`, `selection_table()`, `select_method()`, `main()` |
| [m02_rerun_verify.py](Modeling/scripts/m02_rerun_verify.py) | 4.4 원본 시차·예측·두 지표·튜닝 선택·앙상블·시간순 경계의 독립 검증 | `two_metrics()`, `score()`, `check_peak_weights()`, `main()` |
| [m02_rerun_documents.py](Modeling/scripts/m02_rerun_documents.py) | 검증된 확장 결과를 원고·실행 기록·README·프로젝트 메모리에 처음 통합한 코드 | `main()`; 초기 문서 생성 이력용이며 현재 원고를 반복 덮어쓰는 실행 단계로 사용하지 않는다. |
| [verify_m03_documents.py](Modeling/scripts/verify_m03_documents.py) | 기존 원고 수치·링크·그림 해시 확인. 확장 문서 검증도 이 파일을 불러온다. | `main()`; 학습 없음 |
| [m02_rerun_document_verify.py](Modeling/scripts/m02_rerun_document_verify.py) | 4.4 비교·월별·조건별 표 수치와 문서 링크·기존 M03 표 보존 확인 | `main()`; 학습 없음 |
| [m04_compare.py](Modeling/scripts/m04_compare.py) | 4.6 직전 낮은 상태 규칙·조건별 오류 기여·예측 전 입력·일별 전후 비교와 선택 | `subsets()`, `paired()`, `main()`; 학습 없음 |
| [m04_verify.py](Modeling/scripts/m04_verify.py) | 4.6 원본 시차·규칙·지표·전후/일별 오류·동일 입력·선택 판단의 독립 검증과 문서 검증 | `main()`, `documents()`; 문서는 `--documents` |
| [m04_rise_compare.py](Modeling/scripts/m04_rise_compare.py) | 4.7 변화 입력·상승 가중·두 분류 결합의 시간순 튜닝과 예측 | `build_frame()`, `stats()`, `score()`, `main()` |
| [m04_rise_verify.py](Modeling/scripts/m04_rise_verify.py) | 4.7 원본 27입력·AP·모든 지표·튜닝 선택·저장 모델 검증 | `classification()`, `stats()`, `main()` |
| [m04_rise_gate.py](Modeling/scripts/m04_rise_gate.py) | 4.7 과거 내부 예측에서 상승 중심 결합 점수만 변경 | `main()`; 새 학습 없음 |
| [m04_rise_gate_verify.py](Modeling/scripts/m04_rise_gate_verify.py) | 4.7 결합식·300회 점수·지표·변경 없는 채택 요건 독립 검증 | `gate_rows()`, `objective()`, `main()` |
| [m04_rise_documents.py](Modeling/scripts/m04_rise_documents.py) | 4.7 검증된 표의 상세 원고·기록·README 통합 및 수치·링크 검증 | `--write` 최초 작성, 이후 `--verify` |
| [m04_surge_diagnose.py](Modeling/scripts/m04_surge_diagnose.py) | 4.8 증가량 경계 고정·기존 7방법 재평가·4월 순위/임계값/전문가 진단 | 전체 파일 보존; `main()` |
| [m04_surge_verify.py](Modeling/scripts/m04_surge_verify.py) | 4.8 원본 정확한 시차·분위수·급등 오차·AP/혼동행렬 독립 검증 | 전체 파일 보존; `main()` |
| [m04_surge_risk.py](Modeling/scripts/m04_surge_risk.py) | 4.8 전체 상태 분류의 120회 Optuna·Q90 보조 학습·위험 출력 비교 | 전체 파일 보존; `main()` |
| [m04_surge_risk_verify.py](Modeling/scripts/m04_surge_risk_verify.py) | 4.8 120회 내부 OOF·시간 경계·모델 재로딩·모든 선택 요건 검증 | 전체 파일 보존; `main()` |
| [m04_surge_calibration.py](Modeling/scripts/m04_surge_calibration.py) | 4.8 고정된 내부 음성 꼬리 경계로 한 번의 경보 보정; 새 학습 없음 | 전체 파일 보존; `main()` |
| [m04_surge_calibration_verify.py](Modeling/scripts/m04_surge_calibration_verify.py) | 4.8 순서통계량 경계·복사 예측·경보·변경 없는 선택 요건 독립 검증 | 전체 파일 보존; `main()` |
| [m04_surge_documents.py](Modeling/scripts/m04_surge_documents.py) | 4.8 상세 원고·기록·README 작성과 표/링크 검증; --write 또는 --verify | 전체 파일 보존; `main()` |
| [m05_evaluate.py](Modeling/scripts/m05_evaluate.py) | 4.9 개발 설정·경보 경계 고정 후 후반기 학습·추론·평가 | `--stage freeze` → `--stage run` |
| [m05_verify.py](Modeling/scripts/m05_verify.py) | 4.9 원본 입력·시간 경계·모델 재로딩·전체 지표와 판단 독립 검증 | `main()`; 새 학습 없음 |
| [m05_documents.py](Modeling/scripts/m05_documents.py) | 4.9 검증 결과의 상세 원고·기록·README 반영과 문서 검증 | `--write` 또는 `--verify` |
| [modeling_submission_audit.py](Modeling/scripts/modeling_submission_audit.py) | 본 목록의 누락·프로젝트 내부 import·원본/결과/모델 해시·보존 파일 점검 | `local_imports()`, `source_and_result_checks()`, `main()`; `--record` 또는 `--verify` |

`m02_rerun.py → m022_compare.py → m03_ablation.py → m02_compare.py → m01_prepare.py`처럼 직접 실행하지 않는 파일도 연결되어 있다. 확장 검증은 `m03_verify.py`, 확장 문서 검증은 `verify_m03_documents.py`에도 의존한다. 따라서 실행 진입점만 남기거나 원고에서 이름을 눈으로 찾는 방법만으로 삭제 대상을 정하지 않는다. 보존 목록은 Python의 import 구문과 문자열로 지정한 `__import__()`까지 점검한다.

### 코드와 함께 유지할 파일

| 대상 | 보존 경로와 이유 |
|---|---|
| 원자료 | [okm_augumented_2021.csv](Modeling/../data/origin/okm_augumented_2021.csv). 목표·시차 재구성과 원본 독립 검증에 필요하다. |
| 현재 실행 조건 | [M01](Modeling/config/m01_contract.json), [초기 M02](Modeling/config/m02_contract.json), [M03 A/B](Modeling/config/m03_ab_contract.json), [M03 C](Modeling/config/m03_c_contract.json), [확장 M02](Modeling/config/m02_rerun_contract.json), [M04](Modeling/config/m04_contract.json), [상승 비교](Modeling/config/m04_rise_contract.json), [결합 점수 수정](Modeling/config/m04_rise_gate_contract.json), [일반 급등 진단](Modeling/config/m04_surge_contract.json), [위험 출력](Modeling/config/m04_surge_risk_contract.json), [경보 보정](Modeling/config/m04_surge_calibration_contract.json), [M05](Modeling/config/m05_contract.json). 각 실행 코드가 해당 JSON을 읽는다. |
| 과거 확장 조건·환경 기록 | [m022 조건](Modeling/config/m022_contract.json), [환경 점검](Modeling/tables/m022/environment.json). 중단한 B 입력 실행의 이력을 보존하며 현재 모델 선택 결과로 사용하지 않는다. |
| 전처리와 결과·검증 자료 | `Modeling/tables/m01/`, `m02/`, `m03/ab/`, `m03/c/`, `m03/diagnostics/`, `m02_rerun/`, `m04/`, `m04_rise/`, `m04_rise_gate/`, `m04_surge/`, `m04_surge_risk/`, `m04_surge_calibration/`, `m05/`, `observed_timeline/`의 파일. 원고에 직접 링크하지 않은 CSV도 부모 결과 재사용이나 출력 해시 전수 검증에 필요하다. M04의 `implementation_attempt1/`은 INVALID 표시가 있는 수정 전 이력이며 유효 결과와 구분한다. |
| 저장 학습 모델 | `Modeling/models/m02/`, `m03/ab/`, `m03/c/`, `m02_rerun/`, `m04_rise/`, `m04_surge_risk/`, `m05/`의 파일. 모델 목록이 가리키는 실제 파일과 부모 모델 재사용·재로딩 검증을 보존한다. |
| 그림 | `Modeling/figures/`의 파일. 본문 그림과 기존 관측 그림의 근거를 보존한다. 그림 코드는 검토용 축소 이미지를 필요할 때 다시 생성한다. |
| 라이브러리 버전 | [requirements.txt](Modeling/requirements.txt). 정규 CPython 3.13에서 현재 설치된 직접 사용 라이브러리 버전을 기록한다. 표준 라이브러리는 별도 패키지로 설치하지 않는다. |
| 기계 판독 보존 목록 | [submission_manifest.json](Modeling/submission_manifest.json). 코드와 내부 import 관계, 파일별 경로·크기·SHA-256, 실행 환경을 기록한다. 목록 파일 자체도 함께 보존한다. |

이 목록의 범위는 **현재 Modeling 원고의 실행·검증·그림과 직접 참고 파일**이다. EDA·Analysis 고유의 재현 파일 전체나 최종 제출 요건 전체를 감사한 목록은 아니다. 해당 단계의 코드는 각 원고의 보존 목록과 함께 판단해야 한다. 현재 목록의 파일을 유지한다는 것과 최종 제출이 완료됐다는 것은 구분한다.

### 실행 순서와 정리 전 점검

프로젝트 루트의 상대 폴더 구조를 유지하고 정규 CPython 3.13을 사용한다. 원본 결과를 보존한 새 작업 사본에서 처음 계산할 때의 순서는 아래와 같다. `m02_rerun.py`는 이미 실행 기록이 있는 같은 결과 폴더를 덮어쓰지 않도록 막으므로, 현재 완료 폴더에서 다시 학습 명령을 실행하지 않는다.

| 순서 | Python 진입점·인수 | 목적 |
|---|---|---|
| 1 | `Modeling/scripts/m01_prepare.py` → `Modeling/scripts/m01_verify.py` | 원자료에서 입력·평가 조건을 만들고 독립 검증 |
| 2 | `Modeling/scripts/m02_compare.py` → `Modeling/scripts/m02_verify.py` | 초기 M02 학습·예측·비교 및 검증 |
| 3 | `Modeling/scripts/m03_ablation.py --stage ab` → `Modeling/scripts/m03_verify.py --stage ab` | 부모 HGB를 확인하고 마지막 값 비교 |
| 4 | `Modeling/scripts/m03_ablation.py --stage c` → `Modeling/scripts/m03_verify.py --stage c` | 검증된 A/B를 재사용해 계절 시차 비교 |
| 5 | `Modeling/scripts/m03_error_followup.py` | 직전 상태·상승·최대 시간 오류 진단 |
| 6 | `Modeling/scripts/plot_observed_power_timeline.py`, `Modeling/scripts/plot_m02_manuscript.py`, `Modeling/scripts/plot_m03_manuscript.py` | 검증된 표에서 관측·모델·입력 비교 그림 생성 |
| 7 | `Modeling/scripts/m02_rerun.py` → `Modeling/scripts/m02_rerun_verify.py` | 확장 M02의 튜닝·예측·앙상블·선택 및 독립 검증 |
| 8 | `Modeling/scripts/m04_compare.py` → `Modeling/scripts/m04_verify.py` | M03 예측의 규칙 비교·오류 진단과 원본 독립 검증 |
| 9 | `Modeling/scripts/m04_rise_compare.py` → `Modeling/scripts/m04_rise_verify.py` | 변화 입력·상승 가중·분류 결합의 추가 학습과 독립 검증 |
| 10 | `Modeling/scripts/m04_rise_gate.py` → `Modeling/scripts/m04_rise_gate_verify.py` | 검증된 과거 내부 예측의 결합 점수 수정·독립 검증; 새 학습 없음 |
| 11 | `Modeling/scripts/m04_surge_diagnose.py` → `Modeling/scripts/m04_surge_verify.py` | 증가량 정의·기존 예측 재평가·4월 진단 |
| 12 | `Modeling/scripts/m04_surge_risk.py` → `Modeling/scripts/m04_surge_risk_verify.py` | 전체 상태 분류·Q90 보조 학습과 검증 |
| 13 | `Modeling/scripts/m04_surge_calibration.py` → `Modeling/scripts/m04_surge_calibration_verify.py` | 저장 내부 예측의 음성 꼬리 경보 보정; 새 학습 없음 |
| 14 | `Modeling/scripts/m05_evaluate.py --stage freeze` → `--stage run` → `Modeling/scripts/m05_verify.py` | 후반기 조건 고정·학습·추론·독립 검증 |
| 15 | `Modeling/scripts/m02_rerun_document_verify.py`, `Modeling/scripts/m04_verify.py --documents`, `Modeling/scripts/m04_rise_documents.py --verify`, `Modeling/scripts/m04_surge_documents.py --verify`, `Modeling/scripts/m05_documents.py --verify` | 기존·추가 원고 수치와 링크 검증 |
| 16 | `Modeling/scripts/modeling_submission_audit.py --record` → `Modeling/scripts/modeling_submission_audit.py --verify` | 최신 상태를 보존 목록에 기록하고 의존성 확인 |

정리 전에는 학습 없이 아래 명령으로 현재 목록을 검증할 수 있다. PowerShell에서 프로젝트 루트를 작업 디렉터리로 실행한다.

```powershell
$p = Start-Process -FilePath 'C:\Program Files\Python313\python.exe' -ArgumentList 'Modeling/scripts/modeling_submission_audit.py','--verify' -WorkingDirectory (Get-Location).Path -Wait -PassThru -NoNewWindow
exit $p.ExitCode
```

Codex에서 정규 Python을 실행할 때는 프로젝트 AGENTS.md에 따라 첫 시도부터 `sandbox_permissions="require_escalated"`로 위 Start-Process 경로를 사용한다. 현재 모델 결과는 기존 정규 Python 실행·독립 검증에 근거하며, 이번 코드 목록 점검에서 새로운 환경의 전체 재학습까지 수행한 것은 아니다. 이후 원고·코드·결과를 의도적으로 갱신하면 해당 결과를 검증한 뒤 보존 목록을 다시 기록한다. `--verify`는 누락·변경을 발견하면 실패하며 파일을 삭제하지 않는다.

### 재개발에 추가한 코드

| 파일 | 역할 |
|---|---|
| [redevelopment_r01.py](Modeling/scripts/redevelopment_r01.py) | R01 기존 A/B의 조건별 오차와 학습 자료 기준 구간 경계 계산 |
| [redevelopment_compare.py](Modeling/scripts/redevelopment_compare.py) | R02 모델×입력·Optuna 비교, 조건부 R03 변화량 목표, 적격 후보 R04 후반기 평가 |
| [redevelopment_verify.py](Modeling/scripts/redevelopment_verify.py) | 원자료 시차·지표·튜닝 선택·모델 추론·채택 요건 독립 검증 |
| [redevelopment_documents.py](Modeling/scripts/redevelopment_documents.py) | 상세 원고·실행 기록 생성, 사용 코드와 문서 링크 검증 |
| [redevelopment_peak.py](Modeling/scripts/redevelopment_peak.py) | R02/R03 실패 근거를 반영한 일별 최대 시간 학습 가중 비교·조건부 후반기 평가 |
| [redevelopment_peak_verify.py](Modeling/scripts/redevelopment_peak_verify.py) | 학습 가중·내부 선택·가중 모델 3개 독립 재학습 및 평가 검증 |

## 4.6 M04: 개선 조건과 실패조건, 낮은 상태 규칙의 기각

M03의 마지막 값 추가는 전체 오차와 실제 일별 최대 시간의 과소예측을 줄였다. 그러나 낮은 구간 유지 625시간에서 B의 MAE는 4.110으로 A의 3.766과 직전 기준의 2.786보다 컸다. 낮은 상태에서 26초과 상태로 바뀐 20시간도 B가 모두 작게 예측했다. 이 결과에서 '예측 시점에 아는 직전 낮은 상태에서는 직전 기준을 사용하는 편이 나은가'라는 질문을 도출했다. [M04 실행 기록](Modeling/10.05_M04_조건별오류와낮은전력규칙_실행기록.md)에는 사전 조건·실행·구현 수정·검증과 판단 근거를 남겼다.

### 사후 오류 조건과 사용할 수 있는 규칙의 구분

기존 전력 상태 정의를 유지했다. 최대값이 0이면 zero이며, 나머지는 평균을 `floor(mean+0.5)`로 반올림한 값이 20 미만이면 below20, 20~26이면 low, 그 밖이면 above26이다. 따라서 low는 최대값 20~26이나 정확한 평균 20~26을 의미하지 않는다. low→low·low→above26은 실제 다음 상태를 이용한 사후 진단이며 예측 규칙의 입력으로 사용할 수 없다.

이번 규칙은 직전 상태가 low이면 관측된 직전 최대값을 예측으로 쓰고, 그 밖이면 기존 B를 쓴다. 사용 정보는 직전 평균·최대값과 B 예측이다. HGB를 다시 학습하거나 입력·설정을 바꾸지 않았고, M03과 동일한 4~6월 2,184시간에서 비교했다. 실제 일별 최대 137시간은 91일의 동률 가중치 합이 하루 1이 되도록 평가했다. 과거 상태·시간순 경계·최대값 계산은 M01~M03의 정의를 이어받았다.

계산 전에 [M04 조건](Modeling/config/m04_contract.json)에 채택 기준을 기록했다. 전체 MAE가 감소하면서 실제 일별 최대 시간 MAE와 평균 과소예측, 상승 전환 평균 과소예측이 모두 악화되지 않아야 채택한다. 이 기준은 전력 상승을 크게 놓치는 문제를 중요하게 보려는 개발 선택 기준이며, 경제적 비용 최적화로 해석하지 않는다. 7~8월은 이번 비교에 사용하지 않았다.

### 평균 오차 개선과 중요한 조건의 악화

| 평가 항목 | 기존 B | 낮은 상태에 직전 기준 적용 |
|---|---:|---:|
| 전체 MAE | 6.191 | 6.061 |
| 전체 RMSE | 9.783 | 10.578 |
| 실제 일별 최대 시간 MAE | 7.589 | 7.875 |
| 실제 일별 최대 시간 평균 과소예측 | 6.987 | 7.340 |
| 낮은 구간 유지 MAE | 4.110 | 2.786 |
| 상승 전환 평균 과소예측 | 48.772 | 75.400 |
| 낮은 유지 중 일별 최대 시간 MAE | 2.465 | 4.633 |
| 전체 평균 과소예측 | 2.872 | 3.359 |

650시간에 규칙을 적용했고 나머지 1,534시간의 예측은 유지했다. 낮은 유지 전체의 MAE는 줄었지만 상승 20시간의 과소예측이 증가했고, 전체 RMSE도 커졌다. 실제 일별 최대 시간 중 low→low인 53행·날짜가중치 12에서도 MAE가 2.465→4.633으로 악화됐다. 상승 20시간과 일별 최대 137시간의 겹침은 0이므로, 최대 시간 악화는 상승 전환 자체가 아니라 낮은 상태의 상대적 최대를 작게 예상한 데서 나타났다. 같은 '낮은 구간'이라도 구간 전체와 그 안의 최대 시간은 평가 질문이 다르다.

전체 MAE는 약 2.1% 작아졌지만 사전 요건 네 개 중 세 개를 충족하지 못해 규칙을 기각했다. [선택 판단](Modeling/tables/m04/selection.json)에 따라 기존 B를 개발 후보로 유지한다. 전체 평균 점수의 개선만으로 최대전력 관리 목적에 적합한 개선이라고 판단하지 않았다.

월별 MAE는 4월 7.144→6.865, 5월 5.770→5.792, 6월 5.671→5.534였다. 새 배열에서는 6.754→6.418로 개선됐지만 학습과 겹치는 배열에서는 5.562→5.662로 악화됐다. [일별 비교](Modeling/tables/m04/daily_errors.csv)는 개선 20일·악화 27일·동일 44일이었다. 일부 날짜의 큰 이득이 전체 MAE를 낮출 수 있으므로 전체 개선을 대부분 날짜의 개선으로 표현하지 않는다.

### 기존 B에서 어디에 오류가 남는가

아래 기여율은 해당 조건의 절대오차 합을 B의 전체 절대오차 합으로 나눈 값이다. 이 표의 전력 전환은 서로 배타적인 구분이다. 생산 전환·시간대·평일 여부는 별도로 평가하며 이 표와 기여율을 합산하지 않는다. [조건별 원표](Modeling/tables/m04/metrics.csv)에 모든 분모와 평균 과소·과대예측, 발생 비율을 보존했다.

| 사후 전력 조건 | 시간 수 | B MAE | 전체 절대오차 기여율 |
|---|---:|---:|---:|
| 26초과 상태 유지 | 1,510 | 6.400 | 71.5% |
| 낮은 구간 유지 | 625 | 4.110 | 19.0% |
| 낮은 구간→26초과 | 20 | 48.772 | 7.2% |
| 26초과→낮은 구간 | 19 | 15.638 | 2.2% |
| 20미만→낮은 구간 | 5 | 1.262 | 0.05% |
| 낮은 구간→20미만 | 5 | 1.708 | 0.06% |

상승 20시간은 표본의 약 0.9%지만 절대오차의 7.2%가 모였고 모두 과소예측이었다. 급상승의 작은 표본을 무시할 수는 없지만, 전체 오류의 가장 큰 몫은 26초과가 유지되는 1,510시간에서 발생했다. 마지막 값 추가로 그 조건의 MAE가 A의 7.183에서 B의 6.400으로 줄었어도 잔여 오류는 남는다. 변화가 큰 조건, 평균 오차가 큰 조건, 전체 오류의 기여가 큰 조건을 구분해야 한다.

### 예측 전 입력에서 확인한 단서와 한계

규칙이 실패한 이유를 보기 위해 직전 낮은 상태 650시간의 관측 입력을 비교했다. 상승 20시간의 직전 마지막 값은 19~23으로 낮은 유지의 19~28 범위와 겹쳤다. 직전 마지막 값과 그 이전 마지막 값의 차이는 상승에서 -5~2, 중앙값 0이었다. 이번 상승 사례에서는 마지막 값이 항상 미리 치솟는 패턴을 확인하지 못했다. [입력 요약](Modeling/tables/m04/origin_feature_summary.csv)은 원인이나 모든 상승의 예측 불가능성을 증명하는 결과가 아니다.

상승은 07시 18시간·00시 2시간에 나타났고, 07시 전체 91시간의 B MAE는 20.759였다. 시간대는 이미 모델 입력에 들어간 정보이며 이 발견은 오류가 모이는 조건의 진단이다. 관찰된 사례를 피하는 시간대 규칙을 사후로 만들면 개발 결과에 맞춘 추가 선택이 되므로 별도의 성과로 주장하지 않는다.

같은 B 입력에서 결과가 다른 사례도 확인했다. 2021-04-12·19·26의 00시는 모두 월·요일·시간, 직전 생산량, 직전/그 이전 평균·최대값과 직전 마지막 값이 같았다. 같은 4월 모델은 세 시간 모두 45.024를 예측했지만 실제 최대값은 각각 71·21·68이었다. 이 입력 구성 안의 같은 상태에서도 다음 결과가 달라질 수 있으며, 단일 예측값이 세 결과를 모두 정확히 맞출 수는 없다. 이는 다른 입력을 쓰는 모든 모델의 한계나 현장 원인을 규명한 결과가 아니다. [같은 입력의 상충](Modeling/tables/m04/origin_signature_collisions.csv), [개별 사례](Modeling/tables/m04/prior_low_cases.csv), [시간대별 사례 수](Modeling/tables/m04/origin_hour_counts.csv)를 함께 보존한다.

### 결과에 따른 판단 갱신과 재현성

M03의 마지막 값 추가 효과는 유지한다. M04에서 확인한 것은 직전 낮은 상태를 일괄적으로 직전 기준으로 대체하면 유지의 평균 오차는 줄어도 상승 및 상대적 최대 시간의 과소예측이 커진다는 점이다. 해당 보완 규칙은 기각으로 종료하고, 현재 B를 다음 후반기 확인의 후보로 유지한다. 관찰된 실패를 피하도록 임의의 경계를 반복 조정하지 않는다. 새로운 방법은 현재 결과에서 구체적인 질문과 검증 조건이 생길 때 선택한다.

[m04_compare.py](Modeling/scripts/m04_compare.py)는 저장된 예측의 규칙 적용·조건별 오류·전후 비교·입력 요약·선택을 수행하며 `m01_prepare.py`, `m02_compare.py`에 의존한다. [m04_verify.py](Modeling/scripts/m04_verify.py)는 원본에서 정확한 과거 시각을 따로 조회하고 규칙·모든 지표·전후 차이·동률 가중·선택을 대조하며 `m03_verify.py`의 산술 검사를 사용한다. 전체 코드와 의존 파일은 4.5에 포함했다. [최종 실행](Modeling/tables/m04/run.json)과 [독립 검증](Modeling/tables/m04/independent_verification.json)은 8,736예측·208지표·4,368전후 비교·91일별 집계·과거값 52,416개의 대조를 기록한다.

첫 구현에서 날짜별 표가 갱신 전 오차를 참조한 오류는 수정하고 영향받는 표를 재계산했다. 수정 전 이력은 INVALID 표시와 함께 보존하며 유효 결과로 사용하지 않는다. 최종 표의 수치·선택·원본 정렬은 독립 검증을 통과했다. 새 학습·튜닝이나 압축본·그림 생성은 이번 실행에 포함하지 않았다.

제3장에는 개선 조건과 규칙 기각의 근거를 연결한다. 낮은 상태에서 직전 값을 사용하는 규칙은 상승 과소예측을 악화해 채택하지 않았다. 4.9의 고정 후반기 평가에서도 마지막 값의 최대 시간 개선은 안정적으로 유지되지 않았다. 이 결과가 4.12에서 현재 상태만으로 일괄 보정하지 않고 다음 시간의 지속 상승을 별도로 예측해 보완 여부를 정한 배경이다.

## 4.7 M04 추가 개발: 상승 예측 결합의 개선과 채택 한계

M03에서 마지막 값 추가는 전체·일별 최대 시간 오차를 줄였지만, 낮은 상태에서 26초과 상태로 바뀐 20시간을 모두 과소예측했다. 4.6의 직전 값 대체도 그 과소예측을 48.772에서 75.400으로 늘려 기각했다. 이 결과에서 '과거 변화 정보나 상승 사례의 학습 비중을 늘리면 상승 오차를 줄일 수 있는가, 그리고 상승 확률로 적용 대상을 고르면 낮은 유지의 오경보 부담을 줄일 수 있는가'를 후속 질문으로 정했다. [추가 실행 기록](Modeling/10.05_M04_상승예측결합_실행기록.md)과 [첫 비교 조건](Modeling/config/m04_rise_contract.json)에 질문·비교 범위·판단 기준을 기록했다.

### 상승의 정의와 시간순 비교

이번 상승은 기존 low→above26 전환이다. 직전 최대값이 양수이고 직전 정확한 평균을 `floor(mean+0.5)`로 반올림한 값이 20~26이며, 다음 시간의 제공 평균값이 26을 넘는 경우를 뜻한다. 모든 큰 증가나 0 이후 재가동을 포괄하는 정의가 아니다. 그 범위의 성과는 별도로 확인해야 한다. 목표는 다음 시간 네 15분 값의 최댓값이며 단위가 확인되지 않은 값에 임의로 kW를 붙이지 않는다.

외부 개발 비교는 1~3월 학습→4월, 1~4월→5월, 1~5월→6월의 같은 2,184시간·91일이다. 학습 상승 사례는 각각 31·39·47시간, 평가 상승은 8·8·4시간이다. 내부 검증은 1월→2월부터 과거 월만 순차 확장해 외부 평가 월 이전에 끝낸다. 직전 상태는 예측 시점에 관측할 수 있고, 다음 상태·일별 최대 시간·다음 생산량은 입력에 넣지 않는다. 7~8월은 이번 튜닝과 비교에 사용하지 않았다. 다만 후반기 자료의 과거 관찰 이력은 있으므로 완전히 처음 보는 독립 시험으로 표현하지 않는다.

전체 MAE와 일별 최대 시간 MAE, 낮은 유지 MAE는 B 대비 악화를 1% 이내로 제한하고, 낮은 유지 평균 과대예측은 5% 이내로 제한했다. 상승 MAE와 평균 과소예측은 각각 10% 이상 줄고 각 월의 상승 MAE도 악화되지 않아야 한다. 분류 결합의 오경보율은 직전 낮은 상태의 비상승 사례 중 5% 이내로 정했다. 이는 실행 전에 기록한 개발 허용 범위이며 현장 비용에서 추정한 최적값이 아니다. 외부 개발 결과로 채택 여부를 확인하되 그 결과에 맞춰 임계값을 직접 조정하지 않았다.

### 변화 입력·상승 가중 학습·분류 결합의 구분

B의 18개 입력에 과거 변화 입력 9개를 추가해 총 27개를 비교했다. 추가 입력은 직전 네 구간의 마지막값-첫값, 범위, 선형 기울기, 직전과 그 이전 시간의 마지막값·평균·최대·생산량 차이, 최대 6시간의 낮은 상태 연속 길이와 그 조회 중 시간 누락 여부다. 모든 값은 t-1 이전 관측으로 계산하고 누락 시각을 건너뛰어 연결하지 않는다. 기존 날씨 후보 D와 혼동하지 않도록 이번 `D_dynamic`은 변화 입력 HGB로 표기한다.

변화 입력 HGB는 기존 B의 설정을 고정한다. 상승 가중 HGB는 같은 변화 입력에서 학습 자료의 상승 정답에만 가중치 1~12를 부여하고 Optuna로 선택한다. 미래 상승 정답은 학습 가중치에 쓰며 추론 때 주어지는 정보로 사용하지 않는다. HGB 설정은 잎 15, 반복 150, 학습률 0.05, 최소 잎 표본 20, L2 규제 1, 조기 종료 사용 안 함, 난수 시드 42다. 변화 입력만 추가한 효과와 상승 학습 비중의 효과를 구분할 수 있도록 순차 비교했다.

분류기는 직전 상태가 낮은 학습 행만 이용해 다음 시간 상승 확률을 추정한다. Logistic은 학습 구간에서만 표준화하며 C를 0.01~100에서 찾고, HGB 분류기는 잎 3·7·15, 최소 잎 표본 10·20·40, L2 0.1~10을 비교했다. 희소 상승을 상위로 구분하는 능력은 내부 AP로 선택하고 Brier를 보조 확인했다. 여기서 AP는 비보간 평균 정밀도이며 사다리꼴 PR 면적과 같은 수치로 표기하지 않는다. AUROC·정밀도·재현율·오경보도 보존했다. 분류 지표는 상승 여부, 회귀 MAE는 전력 크기를 평가하므로 서로 대체하지 않는다.

결합은 `B + I(직전 낮은 상태이고 상승확률≥임계값) × 결합비율 × (상승 가중 HGB-B)`다. 조건 밖에서는 B를 그대로 사용한다. 확률 임계값 0.01~0.8과 결합 비율 0~1은 과거 내부 검증의 예측만으로 선택했다. 첫 회차는 전체·최대 시간·상승의 B 대비 상대 MAE 중 최댓값과 허용 범위 초과 벌점을 최소화했다. 가중치·두 분류기·두 결합을 외부 월마다 각 30회 탐색해 총 Optuna 450회, 학습 840회를 실행했다. 이 중 외부 최종 학습은 12회이며 저장 모델 12개를 재로딩해 예측을 대조했다.

### 같은 개발 시간에서의 결과

| 방법 | 전체 MAE | 일별 최대 시간 MAE | 상승 MAE | 상승 평균 과소 | 낮은 유지 MAE | 낮은 유지 평균 과대 |
|---|---:|---:|---:|---:|---:|---:|
| 기존 B | 6.191 | 7.589 | 48.772 | 48.772 | 4.110 | 3.554 |
| 변화 입력 HGB | 6.330 | 8.140 | 50.836 | 50.836 | 4.339 | 3.858 |
| 상승 가중 HGB | 6.868 | 8.912 | 35.682 | 35.527 | 6.240 | 5.429 |
| Logistic 분류 결합 | 6.191 | 7.589 | 48.772 | 48.772 | 4.110 | 3.554 |
| HGB 분류 결합 | 6.146 | 7.589 | 40.217 | 40.179 | 4.228 | 3.673 |
| Logistic 결합·상승 중심 선택 | 6.185 | 7.589 | 48.507 | 48.507 | 4.099 | 3.543 |
| HGB 결합·상승 중심 선택 | 6.148 | 7.589 | 40.471 | 40.321 | 4.226 | 3.670 |

상승 가중 HGB는 상승 MAE를 35.682로 줄였지만 전체·최대 시간·낮은 유지 오차를 모두 키웠다. 변화 입력만 추가한 모형도 기존 B보다 좋아지지 않았다. Logistic 결합의 첫 내부 선택은 결합 비율을 0으로 정해 회귀 예측이 B와 같았다. 분류기를 붙였다는 사실만으로 상승 예측이 개선되는 것은 아니었다.

HGB 분류 결합은 상승 MAE를 17.5%, 평균 과소예측을 17.6% 줄였고 전체 MAE도 6.191에서 6.146으로 약 0.7% 줄였다. 그러나 낮은 유지 MAE는 4.110에서 4.228로 2.9% 늘어 사전 허용 범위 1%를 넘었다. 다른 채택 요건은 통과했지만 이 요건을 충족하지 못해 최종 교체 후보로 채택하지 않았다. 일별 최대 시간 MAE가 같은 이유는 이 결합이 바꾼 예측이 해당 최대 시간과 겹치지 않았기 때문이다. 이번 상승 개선을 일별 최대 시간 개선으로 표현하지 않는다.

### 어느 월에서 상승을 찾았고 어디서 오경보가 남았는가

다음 표는 첫 HGB 결합의 월별 결과다. AP의 분모는 직전 낮은 상태의 200·262·188시간이고 상승은 8·8·4시간이다. TP는 상승에서 결합 조건을 충족한 수, FP는 비상승에서 그 조건을 충족한 수다. 각 월의 확률·비율은 해당 월 이전 내부 검증으로 정했다.

| 개발 월 | 상승 수 | B 상승 MAE | HGB 결합 상승 MAE | AP | TP / FP / FN / TN | 임계값 / 결합 비율 |
|---|---:|---:|---:|---:|---|---|
| 4월 | 8 | 60.744 | 60.744 | 0.188 | 0 / 1 / 8 / 191 | 0.7386 / 0.0013 |
| 5월 | 8 | 53.287 | 35.411 | 0.796 | 5 / 2 / 3 / 252 | 0.4246 / 0.8736 |
| 6월 | 4 | 15.795 | 8.775 | 1.000 | 4 / 3 / 0 / 181 | 0.3738 / 0.8224 |

세 달을 합치면 상승 20시간 중 9시간에서 조건을 충족해 재현율 45.0%였고, 총 15번 중 9번이 상승이므로 정밀도는 60.0%였다. 오경보 6시간은 비상승 630시간의 약 0.95%다. 비상승에는 낮은 유지 625시간과 낮은 상태에서 20미만으로 내려간 5시간이 포함된다. 오경보율이 작아도 예측값을 크게 높인 오경보 한 번이 회귀 오차를 늘릴 수 있다. 이 수치는 현장 경보를 실제 운영한 성과가 아닌 개발 자료의 결합 조건 평가다.

예를 들어 6월 6일 08시는 실제 최대값 22를 90.295로, 6월 27일 07시는 실제 23을 87.341로 예측했다. 이처럼 낮은 유지에서 상승 전문가가 과도하게 반영된 사례가 남았다. 4월에는 상승 8시간을 모두 놓쳤고 결합 비율도 약 0.0013이어서 수정 효과가 거의 없었다. 반면 5·6월은 일부 상승을 찾아 그 크기 오차를 줄였다. 월별 AP의 0.188·0.796·1.000 차이와 6월 상승 4시간의 작은 분모를 함께 제시해야 하며 후반기에도 같은 성능이 나온다고 단정하지 않는다.

### 결과에 따른 한 번의 선택 점수 수정

첫 내부 점수는 최대 시간 오차가 B와 같으면 최댓값 항이 1에 머물러, 다른 조건이 허용 범위 안에 있는 상승 개선을 충분히 구분하지 못할 수 있었다. 이 문제만 확인하기 위해 [후속 조건](Modeling/config/m04_rise_gate_contract.json)을 기록하고 최댓값 항을 상승 상대 MAE로 바꿨다. 나머지 벌점·채택 허용 범위·분류기·회귀모델·입력·분할은 고정했다. 저장된 과거 내부 예측으로 결합 임계값과 비율만 두 계열·세 외부 월에 각 50회, 총 300회 탐색했으며 새 모델 학습은 없었다.

HGB의 상승 중심 선택은 상승 MAE 40.471, 낮은 유지 MAE 4.226으로 첫 결합보다 충분한 개선을 보이지 않았고 낮은 유지 요건도 계속 실패했다. Logistic의 상승 중심 선택은 낮은 유지 오차를 줄였으나 상승 MAE 감소가 약 0.5%에 그쳐 10% 요건을 충족하지 못했다. 합계 750회의 개발 탐색 이후 모든 요건을 통과한 추가 후보는 없었고 기존 B를 유지했다. 이번에 정한 비교와 한 번의 점수 수정은 여기서 종료하며, 외부 결과에 맞춰 허용 범위를 늘리거나 선택 경계를 계속 바꾸지 않는다.

### 보고서 연결과 다음 판단

확인된 사실은 마지막 값 추가의 기존 효과와 별개로, 상승 분류와 가중 회귀의 결합이 일부 상승의 과소예측을 줄일 수 있었다는 점이다. 유지할 판단은 기존 B를 현재 모델 선택의 개발 후보로 유지한다는 점이며, 수정할 주장은 '급상승을 전반적으로 잘 예측한다'는 표현이다. 낮은 유지의 과대예측과 4월 상승 미탐지가 남았고, 운영 원인과 실제 절감 효과는 현 자료에서 식별하거나 실측하지 못했다.

제2장에는 동일 조건의 추가 비교와 당시 B 유지 이유, 제3장에는 상승 개선·오경보·월별 실패를 연결한다. 이번 17.5% 상승 오차 감소는 당시 허용 요건을 통과하지 못한 보완 실험의 부분 성과로 보존한다. 이후 4.9에서 후반기를 확인했고, 현재 보고서의 차별화 결과는 지속 상승을 새로 정의한 4.12에 제시한다. 두 실험의 상승 정의와 평가 조건이 다르므로 개선률을 같은 성과로 합산하지 않는다.

### 전체 코드와 독립 검증

[학습·분류·가중 회귀·결합 코드](Modeling/scripts/m04_rise_compare.py), [원본 입력·모델 재로딩·독립 검증 코드](Modeling/scripts/m04_rise_verify.py), [저장 내부 예측의 선택 점수 수정 코드](Modeling/scripts/m04_rise_gate.py), [300회 선택·결합 독립 검증 코드](Modeling/scripts/m04_rise_gate_verify.py), [원고 생성·검증 코드](Modeling/scripts/m04_rise_documents.py)를 4.5의 전체 코드 보존 목록에 포함했다. 공통 모듈도 해당 목록과 함께 보존한다.

[첫 전체 지표](Modeling/tables/m04_rise/metrics.csv), [첫 월별 설정](Modeling/tables/m04_rise/selected_configurations.json), [분류 지표](Modeling/tables/m04_rise/classification.csv), [개별 예측](Modeling/tables/m04_rise/predictions.csv), [첫 독립 검증](Modeling/tables/m04_rise/independent_verification.json), [최종 비교표](Modeling/tables/m04_rise_gate/selection_metrics.csv), [최종 선택](Modeling/tables/m04_rise_gate/selection.json), [후속 독립 검증](Modeling/tables/m04_rise_gate/independent_verification.json)에 근거를 남겼다. 첫 회차는 과거 입력 112,752값·10,920예측행·260지표행·450시도와 12개 모델을, 후속은 15,288예측행·364지표행·300시도의 점수·AP·혼동행렬과 변경 없는 채택 요건을 대조했다. 두 검증은 정규 CPython 3.13에서 종료 코드 0으로 통과했다. 원자료와 기존 M01~M04 결과를 유지했고 압축본·PDF·새 그림·파일 삭제는 수행하지 않았다.

## 4.8 M04 후속: 일반 급등의 재평가와 별도 위험 출력

4.7의 결합은 낮은 상태에서 26초과로 바뀐 20시간의 오차를 일부 줄였지만, 그 정의는 시간당 큰 전력 증가 자체와 달랐다.
이번 질문은 '직전 상태가 낮지 않아도 발생하는 큰 증가에서 개선이 남는가, 4월 미탐지는 어디에서 생기는가, 전력 크기 예측을 유지하면서 별도 위험 출력으로 더 잘 찾을 수 있는가'다. [실행 기록](Modeling/10.05_M04_증가량급등과위험출력_실행기록.md)에 진단→최소 재학습→한 번의 경보 보정 순서를 남겼다. 앞선 실패를 삭제하거나 허용 범위를 늘려 통과시킨 작업은 아니다.

### 급등을 학습 자료에서 먼저 정의

급등은 `다음 시간 최대값 - 정확히 직전 시간 최대값 ≥ 93`으로 정했다. 1월 공통 비교 576시간에서 양의 증가 233건의 95분위수가 93이었다. 같은 학습 자료의 90분위수 61.6은 민감도 점검에만 사용했다. 1월 공통 표본은 시차 168시간 때문에 1월8일부터 시작한다. 시간 누락을 건너뛰어 차이를 만들지 않으며 임의 단위를 붙이지 않는다. [진단 조건](Modeling/config/m04_surge_contract.json)은 계산 전에 기록했다.

같은 개발 2,184시간·91일에서 주 급등은 8시간·8일(4월2·5월4·6월2)이었고, 민감도 급등은 38시간(11·13·14)이었다. 8시간 중 기존 low→above26 전환과 겹치는 것은 3시간뿐이며 5시간은 직전 낮은 상태가 아니다. 38시간에서는 17시간이 겹치고 21시간은 직전 낮은 상태 밖이다. 따라서 prior-low에서만 학습한 분류기는 범위상 모든 급등을 다루기 어렵다. 1~3월 학습의 주 급등도 15시간·7일에 불과하다. 1~6월 공통 표본에는 직전 최대값이 0인 사례가 없어 이번 결과는 0 이후 재가동의 성능을 입증하지 않는다. [사건 수](Modeling/tables/m04_surge/event_counts.csv)를 함께 보존한다.

### 기존 예측을 새 정의에서 재평가

이번 표는 모델을 다시 학습하지 않고 기존 7방법의 같은 예측을 사용한 결과다. 주 급등 8건과 민감도 급등 38건의 분모를 섞지 않는다.

| 기존 방법 | 증가량≥93의 MAE | 증가량≥61.6의 MAE |
|---|---:|---:|
| B | 38.586 | 30.444 |
| D_dynamic | 37.626 | 30.978 |
| W_rise_weighted | 34.817 | 25.167 |
| G_Logistic | 38.586 | 30.444 |
| G_HGB | 31.125 | 25.941 |
| G_Logistic_rise_focus | 38.586 | 30.277 |
| G_HGB_rise_focus | 33.466 | 26.075 |

B에서 주 급등 8건은 모두 과소예측되었고 MAE와 평균 과소예측은 38.586이었다. 기존 HGB 분류 결합은 둘 다 31.125로 19.3% 줄였으며, 민감도 급등 MAE도 30.444→25.941로 14.8% 줄였다. 이는 급등 크기의 오차 개선이다. 예측 증가가 93 이상일 때만 경보를 내는 직접 규칙에서는 B와 기존 결합 모두 TP2·FP2·FN6으로 같았다. 크기를 덜 작게 예상해도 경계를 넘지 않으면 탐지 수는 늘지 않는다. 또한 기존 결합은 4.7의 낮은 유지 MAE 허용 요건에 실패했으므로 이 결과만으로 정상 전력 예측을 교체하지 않는다. [전체 오류표](Modeling/tables/m04_surge/errors.csv)와 [직접/보정 경보표](Modeling/tables/m04_surge/alarm_metrics.csv)를 연결한다.

### 4월 미탐지를 순위·경계·전문가로 구분

4월의 기존 상승 8시간은 prior-low 200시간 중 확률 순위 9~34위였다. 확률은 0.00377~0.06329로 선택 경계 0.73860보다 모두 낮았고, 결합 비율도 약 0.00129였다. 이 8시간 중 새 주 급등 정의를 만족한 것은 4월9일07시 한 건이었다. 해당 시간은 실제 증가 104인데 B 최대 예측은 32.474, 상승 전문가 예측은 34.951이었다. 경보 경계만 낮춰도 전문가가 실제 최대값을 충분히 높게 예측하지 못하는 문제가 남는다. [8건의 순위·확률·전문가 상세](Modeling/tables/m04_surge/april_legacy_cases.csv)를 보존했다.

이 진단으로 '기존 낮은 상태 분류기의 경계를 계속 조절'하는 대신, 전체 상태에서 급등 여부를 학습하고 정상 예측과 위험 출력을 분리하는 비교를 선택했다. 4월의 모든 상승을 새 급등으로 표현하지 않으며 관측상 원인을 설비 기동 등으로 단정하지 않는다.

### 전체 상태 분류와 Q90 보조 출력의 시간순 비교

[추가 학습 조건](Modeling/config/m04_surge_risk_contract.json)은 외부 1~3월→4월, 1~4월→5월, 1~5월→6월과 내부 과거 월 확장 검증을 유지했다. 전체 상태 Logistic/HGB 분류기는 검증된 과거 변화 포함 27입력으로 증가량≥93 여부를 학습했다. Logistic은 학습 구간에서만 표준화하고 C 0.001~100을 탐색했다. HGB는 잎3·7·15, 최소 잎 표본10·20·40, L2 0.1~10, 반복150·학습률0.05·조기종료 안 함으로 비교했다. 외부 월마다 각20회 Optuna, 총120회다. 내부 AP를 최대화하고 동률은 Brier·시도 순서로 구분했다.

AP는 비보간 평균 정밀도다. 분류 순위의 주 비교는 각 월 AP를 그 월 사건 수로 가중한 평균이며, 월별 점수 척도가 달라질 수 있어 합친 AP는 보조값으로 보존했다. AUROC도 남겼지만 전체 정상 시간이 대부분인 자료에서 그 수치만으로 희소 급등 탐지를 평가하지 않았다. 전력 크기는 여전히 MAE·과소/과대로 따로 평가한다.

Q90은 B의 18입력·기존 HGB 설정에 분위수 손실 0.9만 사용한 보조 상단 예측이다. 분위수를 새로 탐색하지 않았다. 분류 내부360회·분위수 내부9회·외부9회, 총378회 학습하고 저장 모델9개를 재로딩했다. 기존 B와 기존 HGB 결합의 위험 점수는 저장 예측에서 `예측 최대값-직전 최대값`으로 재사용했다. 모든 방법에서 정상 전력 예측은 B로 유지해 위험 출력 때문에 낮은 유지의 정상 예측이 악화되지 않게 했다. 이 구조는 현장 경보 비용이 사라졌다는 뜻은 아니다.

### 경보 경계의 한 번의 보정과 결과

첫 경보 선택은 과거 내부 예측에서 오경보 허용률 1%(보조5%) 아래 TP를 최대화하고 동률이면 FP가 적은 경계를 골랐다. 초기 내부 자료에 주 급등이 매우 적어 허용 경계에서 TP가 모두 0이면 일부 방법은 경보를 전혀 내지 않는 선택으로 귀결됐다. 이를 확인한 뒤 [후속 조건](Modeling/config/m04_surge_calibration_contract.json)에 음성 꼬리 경계를 한 번만 적용하기로 기록했다. 새 모델·입력·급등 정의·AP·채택 기준은 바꾸지 않았고 새 학습도 없었다.

보정 경계는 과거 내부 비급등 점수만 내림차순 정렬해 `k=floor(허용률×음성 수)`로 정하고, 인덱스 k의 점수 바로 위 부동소수점 값을 사용한다. 같은 점수의 동률은 보수적으로 제외한다. 미래 평가 월의 점수나 정답으로 경계를 고르지 않는다. 아래는 주 허용률1% 결과다. 허용률은 내부 선택 조건이며 외부에서 같은 오경보율을 보장하는 값이 아니다.

| 위험 출력 | 월별 AP의 사건 수 가중 평균 | 첫 경계 TP/FP | 음성 꼬리 경계 TP/FP | 재현율 | 정밀도 | 실제 오경보율 |
|---|---:|---:|---:|---:|---:|---:|
| B 증가 점수 | 0.413 | 4/3 | 5/8 | 62.5% | 38.5% | 0.368% |
| 기존 HGB 결합 증가 점수 | 0.496 | 4/3 | 6/10 | 75.0% | 37.5% | 0.460% |
| 전체 상태 HGB 분류 | 0.535 | 1/1 | 3/4 | 37.5% | 42.9% | 0.184% |
| 전체 상태 Logistic 분류 | 0.179 | 0/0 | 0/0 | 0.0% | 0.0% | 0.000% |
| Q90 상단 증가 점수 | 0.585 | 0/0 | 0/0 | 0.0% | 0.0% | 0.000% |

| 월 | 급등 수 | B 점수 TP/FP | 기존 HGB 결합 점수 TP/FP | 전체 상태 HGB TP/FP |
|---|---:|---:|---:|---:|
| 4월 | 2 | 1/4 | 1/4 | 0/0 |
| 5월 | 4 | 2/2 | 3/2 | 1/2 |
| 6월 | 2 | 2/2 | 2/4 | 2/2 |

기존 결합 증가 점수는 B 점수의 5/8건 탐지에서 6/8건으로 늘었다. 대신 오경보가 8→10건으로 늘어 정밀도는 38.5%→37.5%였다. 이는 '동일 오경보 수' 비교가 아니라 '같은 내부 허용률에서 정한 경계'의 비교다. 실제 오경보율은 각각 8/2,176=0.368%, 10/2,176=0.460%다. 전체 상태 HGB는 순위 AP가 더 높아도 고정된 경계에서 3건만 찾았고 4월 주 급등 2건을 모두 놓쳤다. AP의 개선과 실제 운용 경계의 탐지 개선을 구분해야 한다. [최종 경보표](Modeling/tables/m04_surge_calibration/alarm_metrics.csv), [전체 비교표](Modeling/tables/m04_surge_calibration/selection_metrics.csv)에 FN/TN·5% 조건·민감도 결과도 보존했다.

### 상단 예측의 성과와 한계

Q90의 주 급등 MAE는 14.616, 평균 과소예측은 14.371로 B보다 작았다. 그러나 상단을 정상 예측으로 대체하면 전체 MAE가 6.191에서 16.661로 커진다. 전체 관측이 상단 이하인 비율은 4~6월 각각 92.78%·93.15%·79.44%, 합계 88.51%였다. 주 급등에서의 포함률은 12.5%로 8건 중1건뿐이다. 따라서 이를 급등의 90% 보장 상단이나 신뢰구간이라고 표현할 수 없다. [상단 오류·포함률](Modeling/tables/m04_surge_risk/regression_metrics.csv)은 보조 분석으로 남기고 최종 정상 예측이나 위험 채널로 채택하지 않았다.

### 채택 판단과 다음 회차의 변경 근거

사전 추가 후보 목록은 전체 상태 Logistic/HGB와 Q90이었다. 채택하려면 B 대비 가중 AP가 5% 이상 높고, 합계 TP와 각 월 TP가 작지 않으며, 실제 합계 오경보율1% 이내이고 TP 증가 또는 FP 감소가 있어야 했다. 이 새 후보들은 모든 요건을 함께 충족하지 못해 [자동 선택](Modeling/tables/m04_surge_calibration/selection.json)은 B 점수를 유지했다. 기존 HGB 결합 점수는 수치 요건을 모두 통과했지만 사전에 '기존 결과 재평가용'으로 제외한 방법이다. 결과를 본 뒤 후보 목록을 바꿔 자동 채택했다고 쓰지 않는다.

다만 '기존 결합을 정상 예측 교체가 아닌 위험 점수로 사용'하는 방향은 이번 결과에서 얻은 후속 후보다. 정상 예측 B를 유지하면서 6/8건을 찾은 성과와 추가 오경보2건을 함께 보여줄 수 있다. 이것은 같은 개발 자료에서 찾은 탐색적 후보이며 8건의 작은 분모를 가진다. 보고서 제3장에는 급등 크기 오차19.3% 감소와 위험 출력의 이득·오경보 부담, 4월 한계·미평가 0재가동을 연결한다. 제5장의 주요 근거는 M03 마지막 값의 기여를 유지하고, 이번 결과는 그 뒤에 확인한 보완 가능성으로 제시한다. '모든 급등을 미리 정확히 맞춘다'는 주장은 하지 않는다.

선언한 새 모델 비교와 한 번의 경보 보정은 여기서 종료한다. 다음 M05에서는 정상 예측 A/B와 별도 위험 출력 B 점수/기존 HGB 결합 점수의 비교 조건을 먼저 고정한다. M01에서 정한 대로 1~6월 자료로 각 모델을 한 번 학습하고, 같은 모델을 고정해 7~8월 전체를 평가한다. 증가 경계93과 음성 꼬리 보정식을 유지하며 모델 설정·결합 규칙·경보 경계는 6월 말까지의 자료로 확정한다. 7월 정답을 추가해 모델을 다시 학습하거나 8월용 경보 경계를 새로 정하지 않는다. 평가 중에는 관측을 마친 직전 시간의 전력·생산량 등을 입력에 반영해 다음 한 시간을 예측하며, 이 입력 갱신은 모델 재학습과 구분한다. 7~8월 결과를 보고 재튜닝하지 않으며, 후반기 자료의 과거 관찰 이력이 있으므로 완전히 처음 보는 시험이라고 쓰지 않는다. 이번 회차는 M05·현장 경보 운영·전력 절감 실측·압축본·파일 삭제를 실행하지 않았다.

### 코드와 재현 근거

[진단 실행](Modeling/scripts/m04_surge_diagnose.py)→[진단 독립 검증](Modeling/scripts/m04_surge_verify.py)→[위험 출력 학습](Modeling/scripts/m04_surge_risk.py)→[학습 독립 검증](Modeling/scripts/m04_surge_risk_verify.py)→[경보 보정](Modeling/scripts/m04_surge_calibration.py)→[보정 독립 검증](Modeling/scripts/m04_surge_calibration_verify.py)→[문서 작성·검증](Modeling/scripts/m04_surge_documents.py)의 순서다. 전체 코드는 4.5의 32개 파일 목록에 포함한다.

추가 조건3개, `tables/m04_surge/`, `tables/m04_surge_risk/`, `tables/m04_surge_calibration/`, `models/m04_surge_risk/`의 모델9개를 함께 보존한다. `trial_oof_predictions.npz`는 120회 선택을 독립 검증하는 배열이므로 본문에 직접 인용하지 않더라도 필요하다. 진단 15,288행·추가 위험 출력 10,920행·내부 예측32,160행·후속 경보10,920행의 검증을 마쳤다. 새 그림이나 별도 압축 보고서는 만들지 않고 상세 원고와 두 README의 기존 흐름에 연결했다.

## 4.9 M05: 고정 후반기 평가와 성과 범위의 갱신

M03에서 마지막 값 추가 B는 개발 전체 MAE와 실제 일별 최대 시간 과소예측을 줄였지만, M04에서는 상승 전환·낮은 유지의 상충이 남았다. 일반 급등에서 기존 결합의 크기 오차가 줄고 위험 점수도 일부 탐지를 늘려 후속 후보로 연결했다. M05는 이 두 효과가 후반기에도 유지되는지 확인하기 위해 실행했다. [실행 조건](Modeling/config/m05_contract.json)은 2026-10-05T15:57+09:00에 기록했으며, 그 뒤 모델을 학습하고 후반기 예측을 생성했다.

### 학습·평가와 고정한 정보

1~6월 공통 4,176시간으로 A·B HGB, 상승 가중 전문가, 직전 낮은 상태 HGB 분류기를 각각 한 번 학습했다. 분류기의 실제 학습 대상은 직전 낮은 상태 1,305시간이며 가장 늦은 대상은 6월28일07시다. 모델4개를 저장·재로딩했다. 평가 중에는 매시간 t-1 관측을 입력에 반영하되 모델·결합 규칙·경보 경계를 고정했다. 7월 정답을 추가한 8월 재학습이나 후반기 Optuna 탐색은 없었다.

주 비교는 단순 기준3개와 A/B를 같은 공통 1,344시간·56일에서 평가했다. 7월600시간·8월744시간이며 실제 일별 최대 시간은77행·날짜 가중치 합56이다. 기본 시차만 필요한1,436시간은 같은 A/B 모델의 추론 범위를 확장한 보조 평가다. 보조 표본에서 다시 학습하지 않았고 두 표본 점수를 섞지 않았다. 후반기는 이전 EDA 관찰 이력이 있어 완전히 처음 보는 독립 시험으로 표현하지 않는다.

A는17입력, B는 직전 마지막 값을 더한18입력이고 기존 HGB 잎15·반복150·학습률0.05·최소 잎 표본20·L2 1·조기종료 안 함·시드42를 고정했다. 위험 후보는 마지막 개발월6월의 설정을 선택한 뒤 1~6월로 재적합했다. 상승 학습 가중치1.730905, 분류기 잎15·최소 잎 표본20·L2 0.155280, 결합 확률 경계0.373766·비율0.822355를 사용했다. 이는 후반기 점수가 가장 좋은 설정을 고른 결과가 아니다.

급등 정답 경계는1월 양의 증가95분위수93을 유지했다. 경보 수치 경계는 기존4~6월의 시간순 외부 예측에서 비급등 점수의 음성 꼬리1%로 산정했다. B 위험 점수 경계는 56.267189, 기존 결합 위험 점수는 58.080088이며 두 평가 월에 같은 값을 썼다. 급등 정답 경계93, 결합 확률 경계, 경보 점수 경계는 각각 다른 역할이다. 학습 자료에 재대입한 예측이나7~8월 정답으로 경계를 고르지 않았다. [경보 산정표](Modeling/tables/m05/alarm_calibration.csv)를 보존한다.

### 전체 및 실제 최대 시간의 비교

| 방법 | 전체 MAE | 실제 일별 최대 시간 MAE | 최대 시간 평균 과소 | 최대 시간 평균 과대 |
|---|---:|---:|---:|---:|
| lag1 | 12.678 | 19.830 | 19.830 | 0.000 |
| lag24 | 38.661 | 38.588 | 31.503 | 7.085 |
| lag168 | 29.504 | 38.371 | 25.737 | 12.634 |
| A | 6.683 | 13.376 | 12.936 | 0.439 |
| B | 6.621 | 13.803 | 13.341 | 0.462 |
| G_aux | 6.591 | 13.803 | 13.341 | 0.462 |

G_aux는 위험 점수를 만들기 위한 보조 결합값이며 정상 전력 출력은 B다. 해당 행의 회귀 오차가 조금 낮더라도 이를 최종 전력 모델 교체로 해석하지 않는다.

B의 전체 MAE는 직전 기준12.678보다 47.8% 작았다. 실제 최대 시간 MAE도19.830→13.803으로 감소했다. A 역시 단순 기준보다 낮은 오차를 보였다. 따라서 과거 정보와 달력을 사용하는 HGB의 기본 예측 성과는 후반기에서도 확인됐다. 이는 마지막 값 한 변수의 추가 효과와 구분되는 모델 성과다.

A→B 전체 MAE는6.683→6.621로 0.92% 감소했으나, 실제 최대 시간 MAE는13.376→13.803으로 3.19% 증가했다. 최대 시간 평균 과소도12.936→13.341로 증가했다. 개발에서 나타난 전체8.1%·최대 시간 과소37.9% 감소가 후반기에서 같은 방향과 크기로 유지된 것은 아니다.

실행 전 확인 기준은 A 대비 전체·최대 시간 MAE 각각1% 이상 감소, 최대 시간 평균 과소 감소, 직전 기준보다 낮은 전체·최대 시간 MAE, 각 월의 A 대비 MAE 악화1% 이내, 배열 재가중 전체 MAE 비악화였다. 1%는 기존 수치상 동률 규칙을 이어받은 것이며 현장 비용의 최적 허용값이 아니다. [판단 파일](Modeling/tables/m05/conclusion.json)에서 전체·최대 시간·과소·월별 요건이 실패했다. 단순 기준 개선과 배열 재가중 요건은 통과했지만, 마지막 값의 광범위한 효과는 확인되지 않았다.

### 월별·전력 상태·적용 범위의 차이

| 월 | A 전체 MAE | B 전체 MAE | A 최대 시간 MAE | B 최대 시간 MAE |
|---|---:|---:|---:|---:|
| 7월 | 6.842 | 6.893 | 15.801 | 16.670 |
| 8월 | 6.555 | 6.402 | 11.420 | 11.491 |

7월은 전체·최대 시간 모두 B가 A보다 나빠졌고, 8월은 전체는 개선했지만 최대 시간 MAE가 소폭 증가했다. 후반기 공통 표본의 학습과 동일한 하루 배열은0일이었다. 전체 점수 하나만으로 개발의 효과가 새로운 배열에서도 안정적으로 유지됐다고 주장하지 않는다.

26초과 유지795시간의 MAE는 A8.041→B7.852, 26초과에서 낮은 상태로 내려간10시간은14.012→9.125로 감소했다. 반면 낮은 유지508시간은3.418→3.610, 상승 전환10시간은28.400→29.482로 증가했다. 전력 크기나 변화 방향에 따라 입력 추가의 이득이 달랐다. [조건별 결과](Modeling/tables/m05/metrics.csv)와 [같은 시간의 A/B 오차 차이](Modeling/tables/m05/paired_predictions.csv)를 남겼다.

보조1,436시간에서도 전체 MAE는 6.785→6.707로 줄었지만 최대 시간은 13.967→14.311로 늘었다. 주 표본에서 제외된 시간으로 범위를 넓혀도 최대 시간의 상충은 남았다. 좋은 점수의 표본만 선택해 결론을 바꾸지 않았다.

8월28일18시~29일10시의0값17시간에서 B는 평균24.047을 과대예측했다. 그 뒤 첫 양수 시간의 실제 최대값41은23.559로 예측해17.441만큼 작게 예상했다. 이 사례의 증가41은 주 급등 경계93에 미달하므로 급등 미탐지6건과 같은 사건으로 세지 않는다. 학습에는 최대값0 기록이 없었고, 실제0을 입력으로 받았을 때의 성능은 별도 한계로 확인됐다. 원인이나 설비 휴무 여부는 현 자료에서 식별하지 못했다. [0값 전후 사례](Modeling/tables/m05/zero_cases.csv)를 보존한다.

### 급등 위험 출력의 탐지와 오경보

후반기 주 급등은6시간·6일이며7·8월 각각3건이다. 이 중4건은 직전 낮은 상태,2건은26초과 상태였다. 경보1% 조건의 결과는 다음과 같다.

| 기간 | 위험 점수 | 급등 수 | TP / FP / FN / TN | 재현율 | 정밀도 | 실제 오경보율 | AP |
|---|---|---:|---|---:|---:|---:|---:|
| 7~8월 | B_score | 6 | 3 / 16 / 3 / 1322 | 50.0% | 15.8% | 1.196% | 0.309 |
| 7~8월 | G_HGB_score | 6 | 4 / 14 / 2 / 1324 | 66.7% | 22.2% | 1.046% | 0.479 |
| 7월 | B_score | 3 | 1 / 8 / 2 / 589 | 33.3% | 11.1% | 1.340% | 0.200 |
| 7월 | G_HGB_score | 3 | 3 / 7 / 0 / 590 | 100.0% | 30.0% | 1.173% | 0.533 |
| 8월 | B_score | 3 | 2 / 8 / 1 / 733 | 66.7% | 20.0% | 1.080% | 0.455 |
| 8월 | G_HGB_score | 3 | 1 / 7 / 2 / 734 | 33.3% | 12.5% | 0.945% | 0.442 |

월별 AP의 사건 수 가중 평균은 B0.328→기존 결합0.488로 증가했다. 급등6건 중 탐지는3→4건, 오경보는16→14건으로 개선됐으나 실제 오경보율은1.196%→1.046%로 후보도 사전1% 요건을 넘었다. 정밀도는15.8%→22.2%로, 후보 경보18건 중 실제 급등은4건이다. 재현율66.7%만 제시하면 경보 부담을 판단할 수 없다.

7월에는 B1건→후보3건을 찾았지만8월에는 B2건→후보1건으로 감소했다. 8월9일08시는 두 점수가 모두57.887이었다. 직전 상태가26초과여서 결합값은 B와 같았지만, B 경계56.267은 넘고 후보 경계58.080은 넘지 못했다. 따라서 이 한 건의 추가 미탐지는 분류기의 순위 악화가 아니라 방법별 고정 경보 경계 차이에서 생겼다. 8월16일07시는 두 점수가 모두 경계 아래여서 두 방법 모두 놓쳤다. [급등6건](Modeling/tables/m05/surge_cases.csv)·[검증된 점수/경보 사례](Modeling/tables/m05/confirmed_diagnostics.json)를 함께 확인했다. 이 사례를 본 뒤 경계를 다시 낮추지는 않았다.

위험 후보는 가중 AP5% 이상 개선·합계 탐지 비악화·각 월 탐지 비악화·실제 오경보율1% 이내·탐지 증가 또는 오경보 감소를 함께 만족해야 했다. 합계 순위·탐지·오경보 수는 개선했지만 월별 탐지와 실제 오경보율이 실패해 최종 위험 채널의 안정적 우위를 확인하지 못했다. 주 급등 크기 MAE 자체는 B38.417→보조 결합32.100으로 16.4% 감소했다. 이는 보조 결합값의 부분 성과이며 최종 채택을 의미하지 않는다.

증가61.6 이상은25시간·17일이었다. 보조 경계에서는 순위·정밀도 개선도 있었으나 사전에 민감도 점검으로 정한 범위다. 또한5% 경보 허용에서는 B가6건 모두 찾는 대신 오경보47건을 냈다. 이 조건의 좋은 탐지율로 주1% 평가를 대체하거나 '급등을 모두 잘 찾았다'고 주장하지 않는다. [전체 위험 지표](Modeling/tables/m05/risk_metrics.csv)에 모든 조건을 보존했다.

### 보고서와 다음 작업의 판단 갱신

확인된 성과는 후반기에서도 HGB가 직전·전날·전주 기준보다 전체·실제 최대 시간 오차를 줄였다는 점이다. 유지할 근거는 M02의 동일 조건 모델 비교와 M03의 개발 입력 기여 결과이며, 수정할 주장은 마지막 값과 급등 결합이 후반기까지 안정적으로 최대 시간의 과소예측을 줄였다는 표현이다. 현재 결과에서는 그 강한 주장을 입증하지 못했다.

최종 보고서 제2장에는 기본·확장 모델 선택과 이 절의 단순 기준 대비 HGB 성과를 제시하고, 이후 4.12의 동일 조건 지속 상승 보완 비교를 연결한다. 제3장에는 선택 후보의 상승·낮은 구간·최대 시간 오류와 후속 오경보 억제를 설명한다. 제5장의 차별성은 4.12에서 확인한 탐색적 선택 보완 효과를 중심으로 제시한다. 이 M05의 미채택 후보를 그 성과로 바꾸어 설명하지 않는다.

이 M05는 선언한 고정 확인 평가로 종료했다. 당시 후보의 고정 결과를 보존하고, 이후 별도 개발에서는 이미 확인한 7~8월을 탐색에 이용한 기간으로 명시했다. 4.10의 재개발과 4.11의 검토 우선순위를 거쳐, 4.12에서는 지속 상승 시작을 예측하고 보완 HGB 적용을 제한하는 방법을 평가했다. 후속 성과가 나왔더라도 이 절의 기존 결과와 학습 조건은 변경하지 않는다.

### 실행 코드와 검증

[m05_evaluate.py](Modeling/scripts/m05_evaluate.py)의 `--stage freeze`→`--stage run`으로 조건·경보 경계를 먼저 기록하고 학습·추론·결과를 생성했다. [m05_verify.py](Modeling/scripts/m05_verify.py)는 원본에서149,040입력값,10,936예측행,1,000지표행,24경보행을 대조하고4모델을 재로딩했다. [m05_documents.py](Modeling/scripts/m05_documents.py)는 검증된 결과를 상세 원고와 연결 기록에 반영하고 표·문서 링크를 검증한다. 새 Optuna 탐색은0회, 학습은4회이며 원자료·기존M01~M04 결과를 보존했다.

[실행 기록](Modeling/tables/m05/run.json)·[독립 검증](Modeling/tables/m05/independent_verification.json)·[저장 모델 목록](Modeling/tables/m05/model_manifest.csv)·[학습 범위](Modeling/tables/m05/training_audit.csv)·[전체 예측](Modeling/tables/m05/predictions.csv)·[별도 위험 출력](Modeling/tables/m05/risk_predictions.csv)·[일별 오류](Modeling/tables/m05/daily_errors.csv)·[큰 오류](Modeling/tables/m05/largest_errors.csv)를 함께 보존한다. 원고4.5의 전체 Python35개 목록과 제출 보존 목록도 갱신한다. 압축본·최종 제출·파일 삭제·실제 절감 실측은 이번 실행에 포함되지 않았다.

## 4.10 후반기 오류를 반영한 예측 설계 재개발

### 이전 결과와 새 질문

M03에서 마지막 값 추가로 전체 MAE가 8.1%, 실제 일별 최대 시간 MAE가 35.4% 감소했지만 M05에서는 전체 0.92% 개선·최대 시간 3.19% 악화로 달라졌다. M02는 A 입력으로 모델을 선택했으므로 B에서도 HGB가 가장 적합한지는 미확인이었다. 사용자의 차별성 입증 요구에 따라 기존 오류 진단 → 모델×입력 비교 → 조건부 변화량 목표 → 적격 후보 후반기 재평가를 실행했다. 원고 4.1~4.9와 기존 결과는 보존했다.

### R01: 마지막 값의 이득과 손해

학습 없이 개발 2,184시간·후반기 1,344시간의 같은 시각 A/B 예측 3,528쌍을 비교했다. 직전 전력 수준, 직전 네 전력값의 변동 폭·상승/하강, 마지막 값과 최대값 차이로 구분하고 새 경계는 각 평가 시점 이전 학습 자료에서 정했다. 실제 하루 최대 여부는 평가용 가중으로만 사용했다.

| 기간 | 직전 관측 조건 | 최대 시간 행 수 | 날짜 가중 합 | A MAE | B MAE | B−A |

|---|---|---:|---:|---:|---:|---:|

| development | prior_state:above26 | 84 | 79.000 | 13.148 | 8.367 | -4.781 |

| development | prior_trend:up | 91 | 73.848 | 13.342 | 8.334 | -5.007 |

| later | prior_state:above26 | 44 | 40.000 | 16.905 | 17.587 | +0.682 |

| later | prior_trend:up | 51 | 41.333 | 15.624 | 16.177 | +0.553 |

가중 합은 완전한 날짜의 실제 최대 동률을 나눈 합이며 독립 사례 수가 아니다. 개발에서 개선되던 직전 조건이 후반기에서 악화해 이 조건만으로 B를 선택하는 규칙을 만들 근거가 부족했다. 전력 수준별 편향도 남아 있어 변화량 목표의 제한 비교 근거로 삼았다. 관측 조건이 오류와 관련된다는 사실만으로 운영 원인을 확정하지 않았다.

근거: [R01 계약](Modeling/config/redevelopment_r01_contract.json), [예측쌍](Modeling/tables/redevelopment/r01/paired.csv), [조건별 결과](Modeling/tables/redevelopment/r01/conditions.csv), [학습 구간 경계](Modeling/tables/redevelopment/r01/thresholds.csv), [날짜별 오류](Modeling/tables/redevelopment/r01/daily_errors.csv), [검증](Modeling/tables/redevelopment/r01/independent_verification.json).

### R02: 모델과 입력의 교차 비교

HGB·ExtraTrees × A·B를 비교했다. A는 17입력이고 B는 마지막 값을 더한 18입력이다. 고정 비교에는 HGB의 기존 잎 15 설정, ExtraTrees의 기존 외부 월별 A 선택 설정을 A/B에 동일하게 적용했다. 재튜닝 비교는 모델×입력×외부 월마다 Optuna 30회·seed 42로 동일 예산을 적용했다. 기존 모델은 해시와 입력·설정을 확인해 재로딩했다.

외부 비교는 1~3월→4월, 1~4월→5월, 1~5월→6월의 모든 적격 2,184시간이다. 내부는 각 외부 학습 구간 안에서 1월→2월부터 월 단위 확장 검증을 한다. 외부 평가 월에는 모델을 고정하고 매시간 입력만 갱신한다. 전체·날짜 균등 최대 시간 MAE 두 목표를 직전 기준으로 정규화한 상대오차 최댓값으로 설정을 선택했다. 고정 A/B로 입력의 기여를 확인하며 재튜닝의 기여와 구분한다.

Optuna 360회·새 학습 1095회를 수행했다. trial 수와 fit 수는 서로 다르다. 실행 전 import 누락을 수정했으며 최초 계약은 [보존본](Modeling/tables/redevelopment/r02/preflight_contract.json)에 남겼다. 데이터·검색 공간·평가 기준은 바뀌지 않았다.

| 방법 | 전체 MAE | 최대 시간 MAE | 최대 시간 과소 | 최대 시간 과대 | 요건 |
|---|---:|---:|---:|---:|---|
| fixed_ExtraTrees_A | 6.825 | 11.540 | 10.897 | 0.643 | 미충족 |
| fixed_ExtraTrees_B | 6.343 | 9.019 | 8.226 | 0.793 | 미충족 |
| fixed_HGB_A | 6.738 | 11.751 | 11.260 | 0.491 | 미충족 |
| fixed_HGB_B | 6.191 | 7.589 | 6.987 | 0.602 | 미충족 |
| lag1 | 13.793 | 36.589 | 36.589 | 0.000 | 미충족 |
| tuned_ExtraTrees_A | 6.825 | 11.540 | 10.897 | 0.643 | 미충족 |
| tuned_ExtraTrees_B | 6.259 | 9.075 | 8.365 | 0.711 | 미충족 |
| tuned_HGB_A | 6.963 | 13.251 | 12.705 | 0.546 | 미충족 |
| tuned_HGB_B | 6.531 | 9.789 | 9.160 | 0.629 | 미충족 |

새 후보 채택에는 기존 HGB A/B 각각보다 최대 시간 MAE·평균 과소가 5% 이상 작고, 전체 MAE는 각각 1% 이내 악화여야 한다. 최대 시간 과대 증가량은 각 기준 최대 시간 MAE의 5% 이내, 각 월의 전체·최대 MAE는 기존 B보다 5% 이내 악화여야 한다. 날짜 하나씩 제외해도 최대 시간 개선 방향이 유지돼야 한다. 1%·5%는 사전 수치 비교 목표이며 통계적 유의성이나 경제적 손익 기준이 아니다.

개발 선택: 새 적격 후보 없음.

근거: [R02 계약·검색 범위](Modeling/config/redevelopment_r02_contract.json), [360회 trial](Modeling/tables/redevelopment/r02/trials.csv), [내부 월별 점수](Modeling/tables/redevelopment/r02/trial_fold_scores.csv), [시간 경계](Modeling/tables/redevelopment/r02/time_audit.csv), [선택 설정](Modeling/tables/redevelopment/r02/configurations.json), [예측](Modeling/tables/redevelopment/r02/predictions.csv), [지표](Modeling/tables/redevelopment/r02/metrics.csv), [판정](Modeling/tables/redevelopment/r02/selection.json), [검증](Modeling/tables/redevelopment/r02/independent_verification.json).

### R03: 변화량 목표의 조건부 비교

R02에 적격 후보가 없고 전력 수준별 편향이 남아 fixed_HGB_B를 동일 설정의 B 직접 예측 비교 상대로 선택했다. 모델 종류·입력·외부 월별 설정을 유지하고 학습 목표만 '다음 최대값−직전 마지막 값'으로 바꿨다. 추론 시 관측 마지막 값에 예측 변화량을 더하며 실제 미래 전력을 사용하지 않는다. 추가 Optuna 없이 새 학습 3회다. 출력은 직접 예측처럼 사후 절단하지 않았다.

| 방법 | 전체 MAE | 최대 시간 MAE | 최대 시간 과소 | 최대 시간 과대 | 요건 |
|---|---:|---:|---:|---:|---|
| delta_fixed_HGB_B | 6.181 | 8.953 | 8.319 | 0.634 | 미충족 |
| fixed_HGB_A | 6.738 | 11.751 | 11.260 | 0.491 | 미충족 |
| fixed_HGB_B | 6.191 | 7.589 | 6.987 | 0.602 | 미충족 |
| lag1 | 13.793 | 36.589 | 36.589 | 0.000 | 미충족 |

근거: [R03 조건](Modeling/tables/redevelopment/r03/contract.json), [설정](Modeling/tables/redevelopment/r03/configurations.json), [예측](Modeling/tables/redevelopment/r03/predictions.csv), [판정](Modeling/tables/redevelopment/r03/selection.json).

R03 [실행 또는 생략 기록](Modeling/tables/redevelopment/r03/run.json), [검증](Modeling/tables/redevelopment/r03/independent_verification.json).

### R03b: 실제 최대 시간의 학습 비중

모델×입력과 변화량 목표 비교가 모두 새 후보를 만들지 못해, 평가의 핵심인 최대 시간과 학습의 초점이 일치하는지를 추가로 확인했다. 이전 M04의 급상승 전환 가중과 달리, 이번에는 과거 학습 자료의 실제 일별 최대 시간에 비중을 줬다. HGB B의 입력·트리 설정은 고정하고 한 가지 학습 가중만 바꿨다. 새 이유·범위·7개 가중과 내부 선택 규칙을 실행 전에 별도 계약으로 기록했다.

학습 가중은 1+(lambda−1)×날짜 균등 최대 가중이다. 완전한 학습 날짜의 실제 최대 동률에는 1/동률 수를 적용하고 다른 시간에는 추가 가중을 주지 않는다. 평가 날짜의 미래 최대 여부는 학습·추론에 사용하지 않는다. lambda는 1, 1.25, 1.5, 2, 3, 4, 8로 제한했다. 각 외부 학습 구간의 과거 내부 예측에서 전체·과대·월별 악화를 제한하고 최대 시간 MAE와 과소의 상대오차 최댓값으로 선택했다. 새 Optuna 없이 내부 21개 설정 비교·새 학습 75회·독립 검증용 재학습 3회를 수행했다.

| 방법 | 전체 MAE | 최대 시간 MAE | 최대 시간 과소 | 최대 시간 과대 | 요건 |
|---|---:|---:|---:|---:|---|
| fixed_HGB_A | 6.738 | 11.751 | 11.260 | 0.491 | 미충족 |
| fixed_HGB_B | 6.191 | 7.589 | 6.987 | 0.602 | 미충족 |
| lag1 | 13.793 | 36.589 | 36.589 | 0.000 | 미충족 |
| peak_weighted_HGB_B | 6.183 | 7.551 | 6.966 | 0.584 | 미충족 |

4월·5월에는 lambda=1, 6월에는 1.5가 선택됐다. 최대 시간 MAE 7.589→7.551의 약 0.5% 개선과 평균 과소 6.987→6.966의 약 0.3% 개선은 새 채택 기준의 5%에 미달했다. 기존 기준은 완화하지 않았고 이 방법도 새 최종 후보로 채택하지 않았다.

근거: [가중 계약](Modeling/config/redevelopment_peak_contract.json), [내부 예측](Modeling/tables/redevelopment/r03b/inner_predictions.csv), [21개 비교](Modeling/tables/redevelopment/r03b/weight_trials.csv), [학습 가중 기록](Modeling/tables/redevelopment/r03b/training_weight_audit.csv), [선택 설정](Modeling/tables/redevelopment/r03b/configurations.json), [외부 예측](Modeling/tables/redevelopment/r03b/predictions.csv), [판정](Modeling/tables/redevelopment/r03b/selection.json), [독립 검증](Modeling/tables/redevelopment/r03b/independent_verification.json).

### R04: 후반기 재개발 평가와 판단 갱신

R02·R03·R03b에 새 적격 후보가 없어 R04를 생략했다. 이번 모델×입력 및 변화량 비교에서는 강한 차별성을 입증할 새 방법을 채택하지 못했다. 선언한 비교는 종료하고 차별성 목표는 미해결로 남긴다. 같은 탐색을 늘리지 않고 새 근거가 있을 때 후속 가설을 검토한다.

R04 [실행 또는 생략 기록](Modeling/tables/redevelopment/r04/run.json), [검증](Modeling/tables/redevelopment/r04/independent_verification.json).

4~6월은 반복 선택에 사용한 개발 기간, 7~8월은 이번 가설 형성에 사용한 후반기 기간이다. 시간순 검증을 적용해도 선택·열람 이력이 사라지지 않으므로 새로운 독립 시험으로 표현하지 않는다. 기존 M05 결과는 유지한다. 급등 탐지 주장은 별도 경보 규칙과 AP·TP/FP/FN·실제 오경보율 검증이 필요하며 연속값 오차로 대신하지 않는다. 실제 전력·비용 절감, 현장 적용, 최종 제출은 하지 않았다.

### 사용 코드와 재현 순서

- [redevelopment_r01.py](Modeling/scripts/redevelopment_r01.py): R01 기존 A/B의 조건별 오차와 학습 자료 기준 구간 경계 계산.

- [redevelopment_compare.py](Modeling/scripts/redevelopment_compare.py): R02 모델×입력·Optuna 비교, 조건부 R03 변화량 목표, 적격 후보 R04 후반기 평가.

- [redevelopment_verify.py](Modeling/scripts/redevelopment_verify.py): 원자료 시차·지표·튜닝 선택·모델 추론·채택 요건 독립 검증.

- [redevelopment_documents.py](Modeling/scripts/redevelopment_documents.py): 상세 원고·실행 기록 생성, 사용 코드와 문서 링크 검증.

- [redevelopment_peak.py](Modeling/scripts/redevelopment_peak.py): R02/R03 실패 근거를 반영한 일별 최대 시간 학습 가중 비교·조건부 후반기 평가.

- [redevelopment_peak_verify.py](Modeling/scripts/redevelopment_peak_verify.py): 학습 가중·내부 선택·가중 모델 3개 독립 재학습 및 평가 검증.

공통 입력 코드 [m01_prepare.py](Modeling/scripts/m01_prepare.py)를 함께 보존한다. R01은 paired()/summary(), 비교는 compare()/assess()/delta_stage()/later_stage(), 검증은 r01()/stages(), 문서는 section()/verify()를 사용한다.

아래는 최초 작성·실행 때의 단계 순서다. 정규 CPython 3.13 및 루트 AGENTS.md의 승격 Start-Process 경로를 사용한다. 기존 run.json이 있으면 덮어쓰기를 거부하므로 새로운 실행은 별도 버전 경로를 사용한다.

~~~text
redevelopment_r01.py --stage freeze
redevelopment_r01.py --stage run
redevelopment_verify.py --stage r01
redevelopment_compare.py --stage freeze
redevelopment_compare.py --stage r02
redevelopment_verify.py --stage r02
redevelopment_compare.py --stage r03
redevelopment_verify.py --stage r03
redevelopment_peak.py --stage freeze
redevelopment_peak.py --stage run
redevelopment_peak_verify.py --stage r03b
redevelopment_peak.py --stage later
redevelopment_peak_verify.py --stage r04
redevelopment_documents.py --write
redevelopment_documents.py --verify
~~~

모든 코드·부모 해시·모델은 [재개발 결과](Modeling/tables/redevelopment)와 [재개발 모델](Modeling/models/redevelopment)에 보존한다. 원고 압축·파일 삭제는 하지 않았다.

## 4.11 하루 2회 검토를 위한 과소예측 우선순위

### 예측오류 분석에서 검토 우선순위로 연결

기존 HGB B는 다음 시간의 최대값을 예측한다. 이후 오류분석에서는 이 예측이 실제보다 크게 낮아질 가능성을 보조 점수로 계산했다. 고정 경고 경계의 성과는 월별로 달랐지만, 사후에 같은 수의 시간을 골랐을 때 보조 점수의 선별력이 높아지는 단서가 있었다. 이에 실제 확인 횟수를 제한하고 미래 점수를 보지 않아도 개선이 남는지 비교했다.

이번 비교의 목표는 **하루 2회라는 동일한 확인량에서 큰 과소예측을 더 많이 선별하는 것**이다. 점 예측값 자체를 수정하거나 급등의 발생 여부를 다시 정의하지 않았다. 정상 예측과 검토 우선순위를 함께 제시하는 보완 방식으로 설계했다.

### 사건 정의와 시간순 보조 점수

큰 과소예측은 실제 다음 시간 최대값이 B 예측보다 9.738515 이상 큰 경우다. 이 경계는 2월의 학습 밖 시간순 예측 오차에서 정한 과소예측량 90분위수로 고정했다. 현장의 위험 한계나 비용 기준은 아니며, 이전의 시간당 증가량 93 이상인 급등과 구분한다. 4 ~ 6월 2,184시간 중 해당 사건은 184건이었다.

보조 모델의 입력은 기존 B의 18개 변수, B 예측값, 과거 유사 상태의 다음 값 표준편차·범위·20번째 이웃 거리·이웃 평균과 B 예측의 차이로 총 23개다. 유사 상태는 해당 평가월보다 앞선 자료에서만 표준화하고 가까운 20개를 선택했다. 보조 모델의 정답도 점 예측 모델의 학습 밖 오차를 사용했다.

4월 평가는 2월 오차로 보조 모델을 학습하고 3월 점수로 모델과 기준을 정했다. 5월은 2 ~ 3월 학습·4월 선택, 6월은 2 ~ 4월 학습·5월 선택으로 확장했다. Logistic과 HGB 중 이전 월 AP가 높은 모델을 선택한 결과 세 달 모두 HGB였다. 높은 B 예측값·직전 최대값·유사 상태 변동의 세 단순 점수와, 이들 중 이전 월 AP로 고른 기준도 함께 비교했다. 이번 선택 규칙 평가에서는 기존 점수와 모델을 그대로 사용했고 신규 학습·튜닝은 하지 않았다.

### 미래 점수를 보지 않는 하루 2회 선택

매시간 다음 시간의 점수가 생성되는 시점에 바로 검토 여부를 정했다. 하루의 24개 예측 시점을 예정된 확인 기회로 두고, 남은 확인 횟수를 b, 현재를 포함한 남은 시간을 r로 표시했다. 이전 달 점수 분포의 1−b/r 분위수를 현재 점수가 넘으면 한 번을 배정했다. 이미 두 번 확인했으면 더 선택하지 않았고, 남은 시간과 확인 횟수가 같으면 남은 시간을 선택했다. 이 규칙으로 모든 방법이 매일 정확히 2회, 91일간 182시간을 선택했다.

예측 시점에는 현재까지 관측한 입력·이전 달 점수·그날 남은 확인 횟수만 사용했다. 아직 오지 않은 시간의 점수나 실제 전력은 사용하지 않았다. 하루 2회와 24시간 확인 가능 조건은 비교를 위한 운영 가정이며, 실제 사업장의 인력·응답 시간으로 검증한 값은 아니다.

결과를 보기 전에 보고서에 남길 개발 성과의 기준을 정했다. 전체 선별 건수가 가장 좋은 단순 기준보다 최소 5건·20% 이상 많고, 이전 월에서 선택한 단순 기준보다 세 달 중 두 달 이상에서 많으면 제한된 기여를 인정하기로 했다. 모든 달에서 이겨야 한다는 조건은 두지 않았으며, 월별 차이와 과소예측량은 함께 제시한다. 이는 자동 경고 후보의 이전 채택 기준을 변경한 것이 아니라 동일 확인량이라는 별도 활용 목적의 비교다.

### 동일 확인량에서 확인한 개선

| 검토 우선순위 점수 | 확인 시간 | 큰 과소예측 선별 | 확인 중 적중 비율 | 전체 사건 재현율 |
|---|---:|---:|---:|---:|
| 예측값이 높은 시간 | 182 | 12 | 6.6% | 6.5% |
| 직전 최대값이 높은 시간 | 182 | 11 | 6.0% | 6.0% |
| 과거 유사 상태의 변동이 큰 시간 | 182 | 20 | 11.0% | 10.9% |
| 이전 월에서 선택한 단순 점수 | 182 | 14 | 7.7% | 7.6% |
| 과소예측 보조 모델 | 182 | 27 | 14.8% | 14.7% |

가장 좋은 단순 기준인 유사 상태 변동 점수는 20건, 보조 모델은 27건을 선별했다. 같은 182시간에서 **7건, 35.0% 증가**했으며 확인 중 적중 비율은 11.0%에서 14.8%로 높아졌다. 높은 예측값만 보는 기준의 12건과 이전 월에서 고른 단순 기준의 14건보다도 많았다. 단순히 예상 전력이 큰 시간을 확인하는 것에 더해, 예측이 크게 낮아질 가능성을 별도 점수로 제공한 기여다. 개별 이웃 변수의 기여를 분리한 실험은 아니므로 특정 입력 하나의 효과로 해석하지 않는다.

월별 확인량은 4월 60회, 5월 62회, 6월 60회로 각 방법에 동일했다.

| 월 | 높은 예측값 기준 | 유사 상태 변동 기준 | 이전 월 선택 기준 | 보조 모델 |
|---|---:|---:|---:|---:|
| 4월 | 1 | 6 | 6 | 7 |
| 5월 | 2 | 8 | 2 | 6 |
| 6월 | 9 | 6 | 6 | 14 |

보조 모델은 이전 월 선택 기준보다 세 달 모두 많이 선별했다. 유사 상태 변동 기준과 비교하면 4월 6→7건, 5월 8→6건, 6월 6→14건으로 월별 차이가 남았다. 선택된 시간의 과소예측량 합은 유사 상태 변동 기준 586.723에서 보조 모델 667.693으로 13.8% 증가했다. 이는 확인 대상으로 포함한 예측오차의 합이며 실제로 보정하거나 절감한 전력량이 아니다.

### 성과의 적용 범위

이 결과는 **동일한 하루 2회 확인 조건에서 선별 효율을 높인 탐색적 개발 성과**다. 전체 큰 과소예측 184건 중 157건은 선택하지 않았으므로 자동 경고나 전체 오류 방지 성능으로 확대하지 않는다. 보조 모델의 선택 182회 중 33회는 남은 확인 횟수를 소진하는 규칙에 따라 배정됐고 이 중 큰 과소예측은 없었다. 이 비효율도 동일 확인량 규칙의 적용 결과로 남긴다.

날짜별 결과를 달력 주 단위로 묶어 동일한 주를 함께 재표집한 2,000회 민감도 분석에서, 유사 상태 변동 기준 대비 선별 증가 7건의 95% 백분위 범위는 −6 ~ 19건이었다. 표본 구성에 따라 우위가 바뀔 여지가 있다. 4 ~ 6월은 여러 개발 실험에서 사용한 기간이고 반복 배열도 있어 이 범위를 독립 시험의 통계적 입증으로 해석하지 않는다. 이번 35%는 해당 개발 조건에서 관측한 개선률이다.

현장 활용안은 다음 시간의 최대값 예측과 함께 검토 필요 표시를 제시하고, 선택된 시간에 담당자가 이미 확인 가능한 가동 계획과 관측 상태를 추가 점검하는 것이다. 실제 조치의 효과는 측정하지 않았다. 결과보고서에서는 제3장의 오류분석에 선별 성과를, 제4장에 하루 확인량을 제한한 활용 절차를 연결하고, 제5장에는 예측값과 과소예측 검토 우선순위를 함께 제공한 기여를 기술할 수 있다.

### 전체 Python 코드와 재현 근거

| 코드 | 역할 |
|---|---|
| [error_warning.py](Modeling/scripts/error_warning.py) | diagnose: B·6시간 입력의 동일 상태 진단, 과거 이웃과 보조 입력 구성 |
| [error_warning_fit.py](Modeling/scripts/error_warning_fit.py) | freeze·run: 사건 경계·시간순 분할 고정, 보조 모델 6개 학습, 월별 점수·이전 월 선택 저장 |
| [error_warning_verify.py](Modeling/scripts/error_warning_verify.py) | 기존 입력·이웃·모델·점수·사건 정의와 HGB 독립 재학습 검증 |
| [priority_review.py](Modeling/scripts/priority_review.py) | freeze·run: 하루 2회 순차 선택, 동일 확인량 비교, 일별·월별 지표와 주별 민감도 |
| [priority_review_verify.py](Modeling/scripts/priority_review_verify.py) | 별도 순차 계산으로 선택·분위수·지표 검증, 미래 점수 변조로 과거 선택 불변 확인 |
| [priority_review_documents.py](Modeling/scripts/priority_review_documents.py) | 기존 원고 보존, 결과표 기반 상세 절 생성, 새 절·README·코드 링크 검증 |

부모 입력의 공통 구성은 [profile_history.py](Modeling/scripts/profile_history.py)와 그 [검증 코드](Modeling/scripts/profile_history_verify.py)를 따른다. 계약과 점수의 해시로 기존 [보조 모델 결과](Modeling/tables/error_warning/run.json), [모델 목록](Modeling/tables/error_warning/model_manifest.csv), [부모 검증](Modeling/tables/error_warning/independent_verification.json)을 연결했다. 첫 진단 파일의 초기 freeze·run 초안은 학습 진입점으로 사용하지 않으며 실제 학습은 별도 fit 파일을 사용한다.

이번 [고정 계약](Modeling/tables/priority_review/contract.json), [전체 비교표](Modeling/tables/priority_review/metrics.csv), [매시간 선택](Modeling/tables/priority_review/predictions.csv), [날짜별 결과](Modeling/tables/priority_review/daily.csv), [주별 민감도](Modeling/tables/priority_review/sensitivity.csv), [판정](Modeling/tables/priority_review/decision.json), [실행 기록](Modeling/tables/priority_review/run.json), [독립 검증](Modeling/tables/priority_review/independent_verification.json)을 함께 보존한다.

최초 실행 순서는 프로젝트 루트에서 정규 CPython 3.13으로 다음과 같다. Windows에서는 루트 AGENTS.md의 Start-Process 경로를 사용한다. 부모 보조 점수는 위 error_warning 코드로 먼저 재현한다.

~~~text
Modeling/scripts/priority_review.py freeze
Modeling/scripts/priority_review.py run
Modeling/scripts/priority_review_verify.py
Modeling/scripts/priority_review_documents.py --write
Modeling/scripts/priority_review_documents.py --verify
~~~

완료된 계약·결과의 덮어쓰기는 막으며 현재 저장 결과는 두 검증 명령으로 확인한다. 독립 검증은 10,920개 시간별 선택, 455개 방법·날짜별 할당, 지표 20행과 2,000회 주별 재표집을 대조했다. 미래 점수만 변경한 45개 시험에서 앞선 선택이 유지됐고, 기존 모델 6개와 부모 결과의 해시도 확인했다. 이번 비교의 신규 학습은 0회다. 상세 원고만 추가하며 압축본과 기존 M05 결과는 변경하지 않았다.

## 4.12 지속 상승 예측과 선택적 HGB 보완

### 중요한 상승 구간의 오차를 줄이기 위한 예측 설계

기본 HGB는 전체 전력 예측에서 단순 시차 기준보다 낮은 오차를 보였지만, 큰 변화가 시작되는 시간의 과소예측은 남았다. 따라서 전력 크기 예측과 함께 **평소 수준보다 높은 상태가 여러 시간 이어지는 시작을 한 시간 전에 예측하고, 그 경고가 나온 시간에만 보완 HGB를 적용하는 방법**을 검토했다. 이는 4.11의 예측오류 검토 우선순위와는 별도로 실제 전력 수준의 상승 시작을 정답으로 학습하는 방법이다.

관측 기간의 적용 결과, 수정한 경고는 공통 평가의 지속 상승 13건 중 7건을 선별했고 오경보는 0건이었다. 이 시간에 보완 HGB를 사용하면 기본 HGB 대비 상승 시작 MAE는 **9.1%**, 상승 과소예측은 **15.8%**, 실제 일별 최대 시간 MAE는 **4.6%** 감소했다. 전체 MAE도 **0.28%** 줄었다. 아래에서는 이 성과가 나온 사건 정의·평가 조건과 남은 탐지 범위를 설명한다.

이 절은 기존 M05를 새 수치로 바꾼 결과가 아니다. 4.9 M05 이후 전력 수준의 지속 전환을 별도 대상으로 정의한 S01~S06의 후속 연구다. 회귀 학습 구간도 달라졌으므로 4.9의 B 점수를 이 절의 B0 점수와 직접 비교하지 않는다. 특히 7~8월은 이미 확인한 자료이며 경과시간 입력 수정의 가설도 그 오류에서 나왔으므로, **후속 수치는 탐색적 재평가이고 새 독립 시험이 아니다.**

### 한 시간의 높은 값과 지속 상승의 시작을 구분했다

전력 수준은 한 시간 안의 네 15분 전력값을 산술평균해 정의했다. 네 값 중 최대값은 기존 HGB의 예측 목표로 유지했다. 따라서 지속 상승 분류는 평균 수준이 바뀌는 시점을 찾고, 회귀 모델은 그 시간의 최대 전력 크기를 예측한다. 한 구간의 일시적인 높은 값과 여러 시간 유지되는 높은 수준을 같은 사건으로 취급하지 않았다.

| 항목 | 고정한 정의 |
|---|---|
| 예측 시점 | t-1 시간의 관측을 마친 직후 다음 t 시간 예측 |
| 상승 크기 기준 | 1월의 양의 시간 평균 증가량 90분위수인 59.5 |
| 상승 시작 조건 | 직전 시간 대비 증가량과 과거 6시간 중앙값 대비 상승폭이 모두 59.5 이상이고, 중앙값 대비 상승폭이 과거 6시간 IQR의 1.5배 이상 |
| 지속 조건 | t·t+1·t+2의 평균이 시작 직전의 과거 중앙값보다 기준의 절반 이상 높은 상태 유지 |
| 중복 시작 방지 | 시작 당시 기준값을 고정해 같은 방향의 진행 중인 상태에서 시작을 반복 집계하지 않음 |
| 사용 가능한 입력 | 예측 전에 확인된 과거 전력·생산량·상태와 달력 정보 |

지속 여부는 미래 두 시간까지 관측된 뒤 정답으로 확정한다. 이를 과거 학습의 정답으로 사용하되, t 시점의 평균·최대값이나 미래 지속시간은 예측 입력에 넣지 않았다. 월별 학습에서도 다음 월 시작 전에 t+2 정답까지 확인된 기록만 사용했다. 59.5는 자료에서 정한 분석 기준이며 현장의 물리적 위험 한계값은 아니다.

1~6월 진단에서는 상승 시작 34건 중 28건이 3시간 지속 조건을 충족했다. 하락은 시작 112건 중 43건이 해당 방향으로 지속됐다. 이 진단을 바탕으로 상승·하락을 별도로 예측했으며, 아래 결합 성과는 개발 근거가 남은 **지속 상승**에 해당한다. 현재의 지속 정의는 시작 기준선으로부터 높은 수준에 머무는 조건이며, 일정한 폭의 박스권이나 일정한 전력값을 보장하지 않는다. [사건 진단](Modeling/tables/regime_diagnosis/event_summary.csv), [개별 사건](Modeling/tables/regime_diagnosis/events.csv), [정답·시점 조건](Modeling/tables/regime_diagnosis/contract.json)을 보존한다.

### 상승 경고에 따른 두 HGB의 선택

기본 HGB B0는 기존 B 구성의 18개 입력을 사용한다. 보완 HGB B1은 여기에 과거 6시간의 중앙값·변동 폭·변화 방향 등 상태 정보를 더한 31개 입력을 사용한다. 두 모델은 같은 행에서 학습하고 HGB 설정도 같게 유지했다. 잎 수 15, 반복 150, 학습률 0.05, 최소 잎 표본 20, L2 1, 조기 종료 미사용, 난수 시드 42다.

초기 S04에서는 전환 확률을 모든 시간의 HGB 입력에 추가했으나 전체·최대 시간 오차가 악화했다. 이후 보완 HGB가 필요한 시간을 분류 경고로 제한하는 방식을 검토했다. 적용 규칙은 다음과 같다.

~~~text
지속 상승 경고가 있으면: 보완 HGB B1의 다음 시간 최대 전력 예측 사용
지속 상승 경고가 없으면: 기본 HGB B0의 다음 시간 최대 전력 예측 사용
~~~

상승 분류기는 과거 전력·생산량·상태와 달력을 사용하는 Logistic 모델이다. 표준화는 각 학습 자료에서만 계산했고 C=1, 최대 반복 3,000, 클래스 가중치 미사용, 난수 시드 42를 유지했다. 경고 경계는 이전 월의 모델이 낸 예측과 당시 확인된 정답으로 산정했다. 과거 음성 오경보 비율 1% 이내에서 적중 수가 가장 많은 경계를 고르고, 동률이면 오경보가 적고 경계가 높은 경우를 우선했다. 같은 학습 자료에 재대입한 점수로 경계를 정하지 않았다.

### 개발 평가와 고정 후속 평가의 연결

| 구분 | 학습·경계 산정 | 평가와 해석 |
|---|---|---|
| 4~6월 개발 | 각 월 이전의 확인된 정답으로 누적 학습. 경고 경계는 이전 월의 예측에서 산정 | 4월은 순위 평가, 5~6월은 자동 경고와 회귀 결합 평가 |
| 7~8월 후속 | 분류기는 1~6월 정답 확정 4,336시간, 회귀는 2~6월 공통 3,598시간으로 한 번씩 학습. 경계는 4~6월 OOF에서 산정 | 같은 모델·경계를 두 달 동안 유지. 매시간 입력은 갱신하고 7월 정답으로 8월을 다시 학습하지 않음 |

1월 자료 전체로 사건 기준을 정했으므로, 회귀 결합의 이전 월 분류 예측은 2월부터 사용했다. 이 조건에 맞춰 B0·B1도 동일한 2월 이후 행으로 학습했다. 초기 분류 표본이 부족한 시기의 대체와 분류기 가용 표시도 같은 조건으로 관리했다. 기존 M05처럼 회귀를 1월부터 학습한 점수를 이번 비교의 기준으로 재사용하지 않았다.

5~6월 개발 평가에서는 상승 12건을 모두 탐지하고 오경보 0건을 기록했다. 경고 때 B1을 적용한 결과 상승 시작 MAE는 10.046→8.661로 13.8%, 상승 과소예측은 8.002→7.137로 10.8% 줄었다. 경고 밖의 1,450시간은 기본 HGB의 예측을 유지했다. 전체 MAE 감소는 약 0.2%였다. [개발 탐지](Modeling/tables/regime_age_ablation/development_detection.csv), [개발 회귀](Modeling/tables/regime_age_ablation/development_metrics.csv)에 원래 분류기와 수정 분류기의 결과를 함께 남겼다.

### 후속 오류 진단에서 경과시간 입력을 수정했다

초기 S05 후속 결과에서는 상승 7건을 찾았지만 8월에 오경보 38건이 발생했다. 경고 때 보완 HGB를 사용하는 조합의 전체 MAE는 기본 HGB 6.335에서 6.680으로 증가했다. 개별 입력의 기여를 확인하니, 오경보 38건 모두 과거 상태의 경과시간이 180~227시간으로 학습 최대값 64시간을 넘었다. 이 변수는 38건 모두에서 상승 확률을 높이는 가장 큰 양의 선형 기여를 보였다.

이 근거로 S06에서는 상승 분류기에서 경과시간 입력 하나만 제거해 24개에서 23개 입력으로 줄였다. HGB B0·B1과 그 예측값은 그대로 사용했으므로, 경고 대상을 바꾼 효과를 분리할 수 있었다. B1의 경과시간 입력까지 제거한 실험은 아니다. 개발 평가에서 12건 적중·오경보 0건과 회귀 개선이 유지된 뒤에만 후속 기간을 재평가했다.

변경된 점수의 경고 경계 0.4554612321은 같은 과거 전용 알고리즘으로 4~6월 OOF 2,182시간에서 정했다. 이 자료의 상승 18건을 찾고 오경보는 0건이었다. 7~8월의 정답에 맞춰 경계를 다시 조정하거나 경과시간이 긴 평가 기록을 삭제하지 않았다. [오경보 조건](Modeling/tables/regime_followup/verified_alarm_conditions.csv), [단일 입력 제거 조건](Modeling/tables/regime_age_ablation/contract.json), [과거 경계 산정](Modeling/tables/regime_age_ablation/calibration.csv)을 함께 보존한다.

### 중요한 시간의 오차 감소와 불필요한 보완 적용의 억제

7~8월 회귀 비교는 동일한 1,344시간을 사용했다. 그중 지속 정답까지 확인할 수 있는 1,340시간에서 분류 성능을 계산했다. 미래 정답이 없는 네 시간도 과거 입력과 현재 최대값이 있으므로 회귀 평가에는 유지했다. 일별 최대 시간 평가는 하루 24시간 공통 예측이 모두 있는 56일을 사용하고, 최대값 동률에는 하루 가중치 합이 1이 되도록 나눴다.

| 방법 | 전체 MAE | 상승 시작 MAE | 상승 평균 과소예측 | 일별 최대 시간 MAE |
|---|---:|---:|---:|---:|
| 기본 HGB B0 | 6.335 | 20.037 | 19.419 | 13.179 |
| 지속 상승 경고 때 보완 HGB 적용 | 6.317 | 18.205 | 16.342 | 12.574 |
| 모든 08시에 보완 HGB 적용 | 6.419 | 18.782 | 16.918 | 12.671 |

상승 평균 과소예측은 상승 시작에 속한 모든 평가 시간에서 `max(실제값-예측값, 0)`을 평균한 값이다. 과소예측한 사례만 따로 골라 평균한 수치가 아니다. 실제 상승이나 일별 최대 시간은 평가 조건을 구분하는 데 사용했으며, 예측 대상을 선택하는 입력은 관측 전의 분류 경고였다.

기본 HGB 대비 상승 시작 MAE는 20.037→18.205, 평균 과소예측은 19.419→16.342로 감소했다. 일별 최대 시간 MAE는 13.179→12.574로 감소했다. 전체 MAE는 6.335→6.317로 변화해 전체 평균에 대한 추가 이득은 작았지만, 중요한 구간의 개선을 얻으면서 전체 오차 악화를 피했다. 월별 전체 MAE도 7월 6.528→6.498, 8월 6.180→6.171로 감소했다.

| 지속 상승 탐지 | 경과시간 입력 포함 | 경과시간 입력 제거 |
|---|---:|---:|
| 공통 평가의 실제 상승 | 13 | 13 |
| 적중 / 미탐지 | 7 / 6 | 7 / 6 |
| 오경보 | 38 | 0 |
| 재현율 | 53.8% | 53.8% |
| 정밀도 | 15.6% | 100% (7/7건) |

경과시간 제거는 탐지한 상승 수를 늘리기보다 불필요한 경고를 억제했다. 보완 HGB 적용은 45회에서 7회로 줄었고, 나머지 1,337시간은 기본 HGB 예측과 같다. 오경보 38건을 없애면서 기존의 7개 적중을 유지한 것이 전체 MAE 악화를 해소한 직접적인 변화다. 이 표의 정밀도 100%는 해당 기간 경고 7건의 결과이며, 이후 기간의 무오경보를 보장하지 않는다. [후속 탐지](Modeling/tables/regime_age_ablation/followup_detection.csv), [동일 시간 회귀 지표](Modeling/tables/regime_age_ablation/followup_metrics.csv), [보완 적용 횟수](Modeling/tables/regime_age_ablation/followup_burden.csv)를 근거로 한다.

<!-- MODELING_CLOSE_BEGIN -->
![전체·상승·최대 시간 성능과 오경보 개선](Modeling/figures/modeling_close/performance_and_alarms.png)

*7~8월 탐색적 재평가. 회귀는 동일 1,344시간, 경고는 정답이 확인된 1,340시간이다. 일별 최대는 완전한 56일을 같은 가중치로 평가했다.*
<!-- MODELING_CLOSE_END -->

### 탐지 범위와 단순 주간 규칙에 대한 해석

탐지한 7건은 모두 월요일 08시였다. 결과를 확인한 뒤 월요일 08시에 항상 보완 HGB를 쓰는 규칙도 대조했다. 이 사후 규칙은 상승 8건을 찾는 대신 오경보 1건이 발생했고, 수정 모델은 7건을 찾고 오경보 0건을 기록했다. 단순 규칙의 전체 MAE는 6.320, 상승 시작 MAE는 18.434로 모델의 6.317·18.205보다 조금 컸으며 일별 최대 시간 MAE 12.574는 같았다. 따라서 관측된 개선을 복잡한 분류기만의 큰 우위라고 설명하지 않는다. [주간 규칙의 사후 조건](Modeling/tables/regime_age_ablation/calendar_control_contract.json), [탐지](Modeling/tables/regime_age_ablation/calendar_control_detection.csv), [회귀](Modeling/tables/regime_age_ablation/calendar_control_metrics.csv)를 남겼다.

공통 평가에서는 13건 중 6건을 놓쳤다. 과거 6시간 이력만 요구하는 전체 분류 범위에서는 상승 14건 중 7건을 찾아 재현율 50%였고, 상승 한 건은 회귀 공통 시차 조건을 충족하지 않아 공통 비교에서 제외됐다. 따라서 현재 모델은 모든 상승을 포괄하는 탐지기로 설명하기보다, 관측된 일부 상승에서 오경보를 억제하며 예측값을 보완한 후보로 해석한다.

사전에 정한 분류 기준은 재현율 80%·정밀도 80%·음성 오경보 비율 1% 이하였다. 수정 모델은 재현율 기준에 미달했다. 반면 상승 MAE·과소예측 각각 5% 이상 개선, 전체·일별 최대 MAE 악화 1% 이내라는 회귀 결합 기준은 통과했다. 두 판단을 유지하면서 보고서에서는 확인된 오경보 억제와 오차 감소를 주된 성과로 제시한다. 이는 다른 팀의 결과와 비교한 우위나 실제 운영에서 검증한 절감 효과는 아니다.

### 현장 활용과 보고서의 연결

활용 흐름은 직전 시간의 관측 완료 → 다음 시간의 지속 상승 경고와 최대 전력 예측 생성 → 경고가 있으면 보완 HGB 결과 확인 → 담당자가 예정된 가동과 동시 사용 조건을 점검하는 순서다. 경고가 없을 때도 기본 HGB의 예측은 계속 제공한다. 경고가 없는 시간을 안전하다고 판정하는 체계로 사용하면 현재 미탐지 범위를 설명할 수 없으므로, 담당자의 확인을 보조하는 용도로 제안한다. 실제 가동 조정이나 전력·비용 절감량은 측정하지 않았다.

최종 결과보고서에서는 제1장에 지속 전환의 정의와 데이터 진단 근거를 짧게 연결하고, 제2장에 동일 조건의 HGB 결합과 성능표를 제시한다. 제3장에는 경과시간 외삽에 따른 오경보·입력 제거 효과·미탐지와 주간 규칙 대조를 배치한다. 제4장은 예측값을 확인할 시점과 조치 검토 흐름, 제5장은 **데이터에서 찾은 실패조건을 반영해 필요한 시간에만 예측을 보완한 기여**를 요약한다. 제6장에는 아래 코드·조건·저장 결과의 재현 경로를 제시한다. S01~S06 번호는 내부 작업 이력이며 최종 보고서의 장 번호로 사용할 필요가 없다.

### 사용 코드와 검증·제출 보존

| 단계 | Python 코드와 역할 |
|---|---|
| S01~S02 | [regime_diagnose.py](Modeling/scripts/regime_diagnose.py): 사건 기준·과거 상태·선행 조건 진단. [regime_diagnose_verify.py](Modeling/scripts/regime_diagnose_verify.py): 원자료에서 시점·사건·조건 재구성 |
| S03 | [regime_forecast.py](Modeling/scripts/regime_forecast.py): 방향별 시간순 분류·과거 경계·입력 구성. [regime_forecast_verify.py](Modeling/scripts/regime_forecast_verify.py): AP·경계·시간순 조건·저장 모델 검증 |
| S04 | [regime_integrate.py](Modeling/scripts/regime_integrate.py), [regime_integrate_verify.py](Modeling/scripts/regime_integrate_verify.py): 동일 행 HGB 입력 비교와 독립 검증. [regime_route.py](Modeling/scripts/regime_route.py), [regime_route_verify.py](Modeling/scripts/regime_route_verify.py): 경고 시간에만 적용하는 결합·대조 |
| S05 | [regime_followup.py](Modeling/scripts/regime_followup.py), [regime_followup_verify.py](Modeling/scripts/regime_followup_verify.py): 6월 말 고정 모델의 후속 평가·원자료와 예측의 대조 |
| S06 | [regime_age_ablation.py](Modeling/scripts/regime_age_ablation.py): 경과시간 입력 제거·개발 통과 뒤 조건부 후속 평가. [regime_age_ablation_verify.py](Modeling/scripts/regime_age_ablation_verify.py): 독립 재학습·경계·오차 검증 |
| 사후 주간 대조 | [regime_age_calendar_control.py](Modeling/scripts/regime_age_calendar_control.py): 월요일 08시 규칙을 새 학습 없이 대조 |
| 입력·문서·보존 | [m01_prepare.py](Modeling/scripts/m01_prepare.py): 공통 시차와 HGB 입력. [regime_manuscript_documents.py](Modeling/scripts/regime_manuscript_documents.py): 원고 보존·절 추가·수치와 문서 검증. [modeling_submission_audit.py](Modeling/scripts/modeling_submission_audit.py): 의존 코드·모델·표·문서 보존 검사 |

이 코드들의 전체 `.py` 소스와 공통 의존 파일을 같은 폴더 구조로 제출 보존한다. 4.5의 기존 코드표에 더해 위 코드들이 이번 4.12의 실행 근거다. [S01~S02 결과](Modeling/tables/regime_diagnosis), [S03 결과](Modeling/tables/regime_forecast), [S04 입력 비교](Modeling/tables/regime_integration), [S04 선택 결합](Modeling/tables/regime_routing), [S05 결과](Modeling/tables/regime_followup), [S06 결과](Modeling/tables/regime_age_ablation)와 각 모델 목록도 함께 유지한다.

S06의 모델 4개를 저장 후 다시 불러오고 별도 코드로 모두 재학습했다. 학습 구간 안의 표준화·정답 확정 시점·과거 OOF 경계, 분류 지표 24개와 회귀 지표 150개를 검증했다. 원자료의 정상 시각 5,784시간에서 최대값을 대조하고, 기존 HGB 예측의 재사용·경고에 따른 선택·일별 최대와 시작 후 3시간 오차를 확인했다. 모든 검증이 통과했으며, S06의 새 회귀 학습은 0회다. [모델 목록](Modeling/tables/regime_age_ablation/model_manifest.csv), [실행](Modeling/tables/regime_age_ablation/run.json), [독립 검증](Modeling/tables/regime_age_ablation/independent_verification.json), [판단](Modeling/tables/regime_age_ablation/decision.json)을 보존한다.

아래는 최초 분석 실행 순서다. 정규 CPython 3.13에서 루트 AGENTS.md의 승격 Start-Process 경로를 사용한다. 완료된 분석은 기존 실행 파일의 덮어쓰기를 거부하므로 현재 제출 자료의 보존 검사는 마지막 명령으로 수행한다. S06 분석 당시의 원고 상태는 [추가 전 원고](Modeling/tables/regime_manuscript/manuscript_before.md)에 보존했다. 분석 단계의 검증 기록과 이후 원고 편집의 검증 기록을 구분한다.

~~~text
Modeling/scripts/regime_diagnose.py probe
Modeling/scripts/regime_diagnose.py freeze
Modeling/scripts/regime_diagnose.py run
Modeling/scripts/regime_diagnose_verify.py
Modeling/scripts/regime_forecast.py freeze
Modeling/scripts/regime_forecast.py run
Modeling/scripts/regime_forecast_verify.py
Modeling/scripts/regime_integrate.py freeze
Modeling/scripts/regime_integrate.py run
Modeling/scripts/regime_integrate_verify.py
Modeling/scripts/regime_route.py freeze
Modeling/scripts/regime_route.py run
Modeling/scripts/regime_route_verify.py
Modeling/scripts/regime_followup.py freeze
Modeling/scripts/regime_followup.py run
Modeling/scripts/regime_followup_verify.py
Modeling/scripts/regime_age_ablation.py freeze
Modeling/scripts/regime_age_ablation.py run
Modeling/scripts/regime_age_ablation_verify.py
Modeling/scripts/regime_age_calendar_control.py
Modeling/scripts/regime_manuscript_documents.py --write
Modeling/scripts/regime_manuscript_documents.py --verify
Modeling/scripts/modeling_submission_audit.py --verify
~~~

이번 원고 편집에서는 새로운 학습·예측이나 압축본을 만들지 않았다. M03 이후의 잠정 결론과 후속 연결 문장을 현재 결과에 맞게 수정했고, 기존 M05를 포함한 성능표·수치·실험 조건은 유지했다. 편집 전 원고 전체를 별도로 보존했다. 재현 파일의 무결성 검증과 깨끗한 새 환경에서의 전체 재실행은 구분하며, 최종 제출 완료를 주장하지 않는다.

## 4.13 보고서 시각화·영향요인 해석과 모델링 종료

최종 설명의 중심은 4.12의 지속 상승 경고와 선택적 HGB 보완이다. 추가 모델 탐색은 종료하고, 보고서용 그림·해석·활용과 현재 고정 방법의 재현 점검을 마쳤다. 전체 분석 흐름도는 4.1 앞에 두고, 성능·경고 비교는 4.12에 연결했다. 아래 그림은 제3장의 사례·영향요인 근거다.

![지속 상승의 개선·미탐지 사례](Modeling/figures/modeling_close/transition_cases.png)

개선 사례는 공통 상승 중 경고가 있고 절대오차가 줄어든 4건을 개선폭 순으로 정렬한 중앙 순위의 7월 12일 08시다. 실제 최대값 198에 대해 기본 예측 181.129가 보완 후 192.522로 가까워졌다. 미탐지 사례는 놓친 6건의 실제 최대값 중앙 순위인 8월 13일 08시다. 실제 207에 비해 두 예측 모두 182.404로 낮았다. 시작 전 4시간부터 이후 5시간까지 공통 예측이 완전한 사례만 후보로 삼았고, 두 후보군 모두 해당 조건을 충족했다. [선정 규칙·결과](Modeling/tables/modeling_close/case_selection.csv)와 [전체 사례 창](Modeling/tables/modeling_close/case_windows.csv)을 보존했다. 경고 7건 중 개별 절대오차가 줄어든 것은 4건이고 3건은 늘었다. 사례 그림의 개선폭을 전체 평균 성과로 일반화하지 않는다.

![Logistic의 변수군별 기여](Modeling/figures/modeling_close/logit_contributions.png)

추가 해석의 질문은 “경고가 월요일 08시에 집중되는 이유가 달력뿐인가, 과거 관측 상태도 판단에 기여하는가”였다. 현재 분류기는 표준화 뒤 선형 Logistic이므로, 입력별 표준화값에 계수를 곱한 항과 절편을 직접 합산해 모델의 로짓을 정확하게 복원했다. SHAP을 추가 실행하지 않았으며 위 값은 확률 백분율이나 인과효과가 아니다. 변수 간 상관 때문에 각 항을 독립적인 공정 영향력으로 해석하지 않는다.

| 공통 평가 조건 | 건수 | 과거 전력 평균 기여 | 달력 평균 기여 | 평균 로짓 |
|---|---:|---:|---:|---:|
| 상승 적중 | 7 | 13.857 | 0.300 | 4.008 |
| 상승 미탐지 | 6 | 4.159 | -1.220 | -7.339 |
| 월요일 08시 비사건 | 1 | 2.719 | 0.150 | -6.813 |

공통 절편은 -10.104이며 생산량·추적 상태 항도 합산에 포함했다. 과거 전력 입력도 점수에 기여했지만, 이 분해가 달력보다 독립적인 예측 가치가 크다는 실험은 아니다. 경고는 여전히 월요일 08시에 한정되고 단순 주간 규칙과의 성능 차이는 작다. 한 건뿐인 월요일 비사건의 해석도 해당 사례 범위로 제한한다. [전체 입력 계수](Modeling/tables/modeling_close/coefficient_table.csv), [1,428시간 기여](Modeling/tables/modeling_close/logit_contributions.csv), [조건별 요약](Modeling/tables/modeling_close/contribution_summary.csv), [상승 13건](Modeling/tables/modeling_close/event_explanations.csv)을 보존한다.

### 현장 활용과 종료 범위

제4장의 제안은 직전 관측 완료 → 다음 최대 전력·지속 상승 경고 확인 → 담당자가 계획된 가동·동시 사용 조건과 함께 보완 예측을 검토하는 절차다. 무경고를 안전으로 판단하지 않으며, 필수 시차가 없는 경우 보완 예측을 보류하고 자료 상태를 표시한다. 가동 조정·자동 제어·절감량은 실측하지 않았다. 제5장에는 전력 수준의 지속 전환을 별도 정의하고, 오류 진단에서 찾은 입력 문제를 수정해 불필요한 보완 적용을 줄인 기여를 연결한다.

현재 방법은 제한된 탐색 성과를 보고하는 후보로 고정한다. 원자료 진단·시간순 모델 비교·입력 효과·상승 보완·오류 해석·시각화·현재 환경 재현 점검이 끝났으므로 모델링의 추가 성능 탐색을 종료한다. 남은 작업은 최종 한글 양식에 각 장을 배치하고 EDA·Analysis와 분량을 조정하는 제출 편집이다. 전체 급등·급락 탐지가 완성되었다는 의미로 종료를 표현하지 않는다.

### 원자료부터 현재 선택 방법까지 재현

[modeling_close_analysis.py](Modeling/scripts/modeling_close_analysis.py)는 기존 조건을 읽어 원자료에서 M01 시차와 지속 사건을 다시 만들고, 분류 학습 4,336시간·회귀 학습 3,598시간을 재구성했다. 4~6월 OOF 분류 3개와 6월 말 고정 분류기·B0·B1을 다시 학습한 뒤, 과거 경계 0.4554612321·분류 1,428개·회귀 4,032개 예측을 대조했다. 모두 일치했다. 새 후보나 튜닝을 추가한 재학습은 아니다.

최초 재현에서는 원자료에서 만든 실수값과 기존 CSV를 읽은 값 사이의 미세한 수치 차이로 HGB 예측이 일치하지 않았다. 기존 입력 파일과 합친 학습표의 CSV 저장·읽기 단계를 동일하게 적용하자 학습 입력이 정확히 일치했고 예측 대조도 통과했다. 실행 조건과 결과를 바꾸지 않고 수치 저장 경계를 코드에 명시했으며, 수정 전 코드·계약도 보존했다. 이 점은 다른 파서·환경에서 재실행할 때 확인해야 할 재현 조건이다.

평가자용 [modeling_reproduce_selected.py](Modeling/scripts/modeling_reproduce_selected.py)는 새 출력 폴더를 지정해 원자료부터 고정 방법의 학습·추론·기존 결과 대조까지 한 번에 수행한다. 실제로 `Modeling/tables/modeling_replay`에서 실행했고 [실행 기록](Modeling/tables/modeling_replay/run.json)의 분류·회귀 대조가 통과했다. 이미 사용한 폴더는 덮어쓰지 않으므로 재실행 때는 새로운 폴더명을 지정한다. 현재 방법의 재현에 필요한 기존 정의·설정·참조 결과를 함께 제출한다.

다음 명령을 프로젝트 루트의 정규 CPython 3.13·승격 Start-Process 경로에서 실행한다. 첫 줄은 평가자용 재현 경로이며, 뒤의 명령은 최초 분석·그림·문서 생성 이력이다. 현재 자료의 무결성은 verify 명령으로 확인한다.

~~~text
Modeling/scripts/modeling_reproduce_selected.py --output-dir Modeling/reproduction_new
Modeling/scripts/modeling_close_analysis.py freeze
Modeling/scripts/modeling_close_analysis.py run
Modeling/scripts/modeling_close_figures.py
Modeling/scripts/modeling_close_documents.py --update
Modeling/scripts/modeling_close_verify.py
Modeling/scripts/modeling_submission_audit.py --verify
~~~

[modeling_close_figures.py](Modeling/scripts/modeling_close_figures.py)는 저장 결과에서 PNG·SVG 그림을 만든다. [modeling_close_documents.py](Modeling/scripts/modeling_close_documents.py)는 상세 원고를 보존하고 [3페이지용 압축본](Modeling/04_Modeling_3페이지용_압축본.md)을 마크다운으로 생성한다. 압축본은 평가·모델 선택, 지속 상승 보완, 오류·활용·재현성의 세 절로 재구성했다. 성능·경고 수·개선 및 미탐지 사례를 [통합 그림](Modeling/figures/modeling_close/compact_evidence.png) 하나로 묶어 흐름도와 함께 두 그림만 싣는다. 변수 기여의 전체 그림·표와 과거 실험별 결과·전체 코드 연결은 이 상세 원고에 남겼다. [modeling_close_verify.py](Modeling/scripts/modeling_close_verify.py)는 기여 합·예측 오차·그림 근거·문서 보존·링크를 검증한다. 3페이지는 편집 목표이며 마크다운 자체의 실제 페이지 수를 검증했다는 뜻은 아니다.

원본 상세 원고는 [시각화 추가 전 사본](Modeling/tables/modeling_close/manuscript_before.md)에 보존했다. [실행 기록](Modeling/tables/modeling_close/run.json)과 [원자료부터 재현한 예측](Modeling/tables/modeling_close/reproduced_predictions.csv)을 제출 보존 목록에 추가한다. 현재 환경에서 선택 방법을 재실행한 결과이며, 깨끗한 새 환경의 설치 검증이나 기존 Optuna 전체 탐색을 다시 수행한 것은 아니다.

### 4.14 고정된 과거 결과의 재사용과 검증

과거 모델 비교·입력 추가·상승 결합·재개발·오류 검토의 결과 파일을 해시로 확인한다. 저장 모델의 예측과 MAE·일별 최대 오차를 원고 검증 코드로 다시 대조한다. 새로운 모델 선택이나 Optuna 재탐색은 수행하지 않는다. 아래 원문 코드에는 과거 탐색 구현도 모두 보존되어 있다.

In [ ]:
def verify_historical():
    rows=[]
    for p in sorted((PACKAGE/'Modeling/tables').rglob('run.json')):
        if 'implementation_attempt' in str(p):continue
        r=json.loads(p.read_text(encoding='utf-8'))
        checks=0
        for name,h in r.get('outputs_sha256',{}).items():
            q=p.parent/name
            if not q.exists():q=PACKAGE/name
            assert q.exists() and digest(q)==h,(p,name)
            checks+=1
        rows.append({'experiment':p.parent.relative_to(PACKAGE/'Modeling/tables').as_posix(),'stored_status':r.get('status','recorded'),'output_hashes_checked':checks})
    pd.DataFrame(rows).to_csv(RUN/'과거결과_해시검증.csv',index=False,encoding='utf-8-sig')
    display(Markdown('### 이번 실행의 과거 결과 해시 검증

'+pd.DataFrame(rows).to_markdown(index=False)))
execute_step('과거결과해시검증',verify_historical)
# 원고 검증 함수가 쓰는 보고서는 실행 사본에만 생성한다.
with computation_paths():
    # EDA/Analysis 재계산과 독립적으로 고정된 모델 입력을 복원한다.
    shutil.copytree(PACKAGE/'Modeling/tables',WORK/'Modeling/tables',dirs_exist_ok=True)
    for name,args in [('m01_verify',()),('m02_verify',()),('m02_rerun_verify',()),('m03_verify',('--stage','ab')),('m03_verify',('--stage','c')),('m04_verify',()),('m04_rise_verify',()),('m04_rise_gate_verify',()),('m04_surge_verify',()),('m04_surge_risk_verify',()),('m04_surge_calibration_verify',()),('m05_verify',())]:
        execute_step(name+'_'+('_'.join(args) or 'results'),lambda name=name,args=args:cli(name,*args))

### 4.15 현재 선택 방법을 원자료에서 재현

과거 OOF 분류 3개와 6월 말 고정 분류기·B0·B1을 다시 학습한다. 경계 0.4554612321, 분류 1428개·회귀 4032개 예측을 참조 결과와 대조한다. 로짓 기여의 합과 예측 오차, 선택한 사례도 다시 계산한다.

In [ ]:
current=MODULES['modeling_close_analysis']
original_out=current.OUT
current.OUT=RUN/'현재모델'
try:
    execute_step('현재모델정의고정',current.freeze)
    execute_step('현재모델원자료재학습대조',current.run)
finally:current.OUT=original_out
replay=json.loads((RUN/'현재모델/run.json').read_text(encoding='utf-8'))
display(Markdown('### 현재 모델 재현 결과

'+pd.DataFrame([{k:replay[k] for k in ['status','raw_valid_hours','class_train','regression_train','classification_predictions','regression_predictions_checked','refits','oof_threshold','exact_logit_reconstruction']}]).to_markdown(index=False)))
prediction=pd.read_csv(RUN/'현재모델/reproduced_predictions.csv',encoding='utf-8-sig')
result=[]
for variant,g in prediction.groupby('variant'):
    for condition,gg in [('all',g),('up_start',g.loc[g.sustained_onset.eq(1)])]:
        result.append({'variant':variant,'condition':condition,'n':len(gg),'MAE':(gg.prediction-gg.actual).abs().mean(),'underprediction':(gg.actual-gg.prediction).clip(lower=0).mean()})
display(Markdown('### 이번 실행에서 계산한 회귀 오차

'+pd.DataFrame(result).to_markdown(index=False)))
pd.DataFrame(result).to_csv(RUN/'현재모델/재계산_회귀오차.csv',index=False,encoding='utf-8-sig')
events=pd.read_csv(RUN/'현재모델/event_explanations.csv',encoding='utf-8-sig')
assert len(events)==13 and events.alarm.sum()==7
display(Markdown(f'### 이번 실행의 상승 탐지

공통 지속 상승 {len(events)}건 중 {int(events.alarm.sum())}건을 탐지했다. 적중 중 값오차 개선은 {int(((events.alarm==1)&(events.absolute_error_gain>0)).sum())}건, 악화는 {int(((events.alarm==1)&(events.absolute_error_gain<0)).sum())}건이다.'))

## 5. 전체 실행 확인

입력·참조 파일 보존과 모든 실행 단계의 정상 종료를 확인한다. 실행 단계·소요 시간·현재 모델 재현 결과를 JSON으로 저장한다. 이 기록은 현재 환경에서의 실행 검증이며 운영 배포·절감 실측을 뜻하지 않는다.

In [ ]:
assert all(digest(PACKAGE/r['path'])==r['sha256'] for r in inventory['files']), '참조 자료 보존 실패'
summary={'status':'passed','source_code_count':inventory['code_count'],'input_files_checked':len(inventory['files']),'steps':RECORDS,'selected_model_reproduction':replay,'historical_optuna_search_rerun':False,'outputs':RUN.relative_to(PACKAGE).as_posix()}
(RUN/'실행검증.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
display(Markdown('### 전체 실행 결과

'+pd.DataFrame(RECORDS).to_markdown(index=False)+f"

모든 {len(RECORDS)}단계가 정상 종료했다. 입력 {len(inventory['files'])}개의 해시를 다시 확인했다. 실행 기록: `{summary['outputs']}/실행검증.json`."))